<a href="https://colab.research.google.com/github/SANGHATI23/genomic-evidence-reliability/blob/main/02_GES_temporal_validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# STEP 1: Connect Google Drive and clone the GitHub repository
# Project: GES Temporal Validation
# ============================================================

from google.colab import drive
from pathlib import Path
import subprocess
import os

# Mount Google Drive
drive.mount("/content/drive")

# ------------------------------------------------------------
# Project locations
# ------------------------------------------------------------

REPO_URL = "https://github.com/SANGHATI23/genomic-evidence-reliability.git"

# Code will run from Colab's local storage
REPO_DIR = Path("/content/genomic-evidence-reliability")

# Large datasets and persistent results will stay in Google Drive
PROJECT_DATA_DIR = Path(
    "/content/drive/MyDrive/GES_RAG_Temporal_Study"
)

# ------------------------------------------------------------
# Clone or update the GitHub repository
# ------------------------------------------------------------

if REPO_DIR.exists():
    print("Repository already exists. Updating from GitHub...")
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "pull", "--ff-only"],
        check=True
    )
else:
    print("Cloning GitHub repository...")
    subprocess.run(
        ["git", "clone", REPO_URL, str(REPO_DIR)],
        check=True
    )

# ------------------------------------------------------------
# Create persistent study folders in Google Drive
# ------------------------------------------------------------

study_folders = [
    "data_raw",
    "data_interim",
    "data_processed",
    "models",
    "configs",
    "outputs/tables",
    "outputs/figures",
    "outputs/logs",
    "outputs/quality_checks"
]

for folder in study_folders:
    (PROJECT_DATA_DIR / folder).mkdir(
        parents=True,
        exist_ok=True
    )

# Move into the cloned repository
os.chdir(REPO_DIR)

# ------------------------------------------------------------
# Confirm setup
# ------------------------------------------------------------

print("\nSETUP COMPLETED")
print("=" * 60)
print("GitHub repository:", REPO_DIR)
print("Persistent study data:", PROJECT_DATA_DIR)
print("Current working directory:", Path.cwd())

print("\nRepository files:")
for item in sorted(REPO_DIR.iterdir()):
    print(" -", item.name)

print("\nGit status:")
subprocess.run(["git", "status", "--short"])

Mounted at /content/drive
Cloning GitHub repository...

SETUP COMPLETED
GitHub repository: /content/genomic-evidence-reliability
Persistent study data: /content/drive/MyDrive/GES_RAG_Temporal_Study
Current working directory: /content/genomic-evidence-reliability

Repository files:
 - .git
 - .gitignore
 - 01_evidence_drift_full_pipeline.ipynb
 - LICENSE
 - README.md
 - figure1_stability_distribution.png
 - figure2_review_confidence.png
 - figure3_gene_stability.png
 - figure4_risk_groups.png

Git status:


CompletedProcess(args=['git', 'status', '--short'], returncode=0)

In [2]:
# ============================================================
# STEP 2: Create and freeze the initial experiment protocol
# Project: GES Temporal Validation
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

# ------------------------------------------------------------
# Confirm project locations
# ------------------------------------------------------------

REPO_DIR = Path("/content/genomic-evidence-reliability")
PROJECT_DATA_DIR = Path(
    "/content/drive/MyDrive/GES_RAG_Temporal_Study"
)

if not REPO_DIR.exists():
    raise FileNotFoundError(
        "GitHub repository was not found. Run Step 1 again."
    )

if not PROJECT_DATA_DIR.exists():
    raise FileNotFoundError(
        "Google Drive project directory was not found. Run Step 1 again."
    )

# ------------------------------------------------------------
# Prespecified experimental protocol
# ------------------------------------------------------------

protocol = {
    "study_id": "GES-RAG",
    "experiment": "Experiment 1: Temporal Validation of GES",
    "protocol_version": "1.0.0",
    "protocol_created_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "research_question": (
        "Does a lower genomic evidence-stability score calculated "
        "from an earlier ClinVar release predict future evidence "
        "instability?"
    ),

    "primary_unit_of_analysis": {
        "level": "RCV",
        "description": (
            "ClinVar variant-condition aggregate record"
        ),
        "primary_identifier": "RCV accession",
        "supporting_identifiers": [
            "VariationID",
            "VCV accession",
            "AlleleID",
            "normalized HGVS",
            "condition identifiers"
        ]
    },

    "release_design": {
        "baseline_label": "T0",
        "baseline_expected_date": "2023-01-05",
        "baseline_target_month": "2023-01",

        "followup_label": "T1",
        "followup_expected_date": "2026-01-01",
        "followup_target_month": "2026-01",

        "minimum_followup_months": 24,

        "release_date_status": (
            "Pending confirmation from the official ClinVar archive"
        ),

        "important_rule": (
            "No T1 information may be used to construct, normalize, "
            "fit, tune, or threshold the T0 GES model."
        )
    },

    "population": {
        "assembly": "GRCh38",
        "primary_classification_type": "germline",

        "primary_genes": [
            "BRCA1",
            "BRCA2",
            "MLH1"
        ],

        "exploratory_genes": [
            "EGFR"
        ],

        "egfr_policy": (
            "Analyze separately because germline and somatic "
            "interpretation processes may differ."
        )
    },

    "baseline_features_t0_only": [
        "time_since_last_evaluation",
        "submitter_count",
        "submitter_structure",
        "review_status",
        "review_star_level",
        "conflict_status",
        "classification_entropy"
    ],

    "models": {
        "full_ges": {
            "model_type": "logistic_regression",
            "uses_review_status": True
        },

        "no_star_ges": {
            "model_type": "logistic_regression",
            "uses_review_status": False
        }
    },

    "primary_future_instability_outcome": {
        "type": "binary",
        "event_value": 1,
        "non_event_value": 0,

        "event_occurs_if_any": [
            (
                "Aggregate classification changes between clinically "
                "meaningful groups: Benign/Likely Benign, VUS, or "
                "Pathogenic/Likely Pathogenic."
            ),
            (
                "A new unresolved classification conflict appears "
                "at T1."
            ),
            (
                "A previously conflicted record resolves into a "
                "materially different classification."
            )
        ],

        "not_primary_event": [
            "Review-star change alone",
            "Simple record-version change",
            "Formatting-only classification change",
            "Unmatched or censored record"
        ]
    },

    "comparators": [
        "review_stars_only",
        "conflict_only",
        "recency_only",
        "submitter_count_only",
        "unweighted_additive_risk_score",
        "combined_metadata_heuristic",
        "no_star_ges",
        "full_ges"
    ],

    "primary_metric": "AUPRC",

    "secondary_metrics": [
        "AUROC",
        "Brier score",
        "calibration intercept",
        "calibration slope",
        "sensitivity",
        "specificity",
        "positive predictive value",
        "negative predictive value"
    ],

    "subgroup_analyses": [
        "same-review-star analysis",
        "risk enrichment in lowest 5 percent",
        "risk enrichment in lowest 10 percent",
        "risk enrichment in lowest 20 percent",
        "leave-one-gene-out analysis",
        "EGFR exploratory analysis"
    ],

    "missing_record_policy": {
        "rule": (
            "A T0 record not matched at T1 will not automatically "
            "be classified as stable."
        ),
        "status": "unmatched_or_censored",
        "report_separately": True
    },

    "reproducibility": {
        "random_seed": 42,
        "bootstrap_samples_planned": 2000,
        "confidence_level": 0.95,
        "raw_data_in_github": False,
        "raw_data_location": "Google Drive",
        "code_location": "GitHub"
    }
}

# ------------------------------------------------------------
# Save protocol in both GitHub and Google Drive
# ------------------------------------------------------------

repo_config_dir = REPO_DIR / "configs"
drive_config_dir = PROJECT_DATA_DIR / "configs"

repo_config_dir.mkdir(parents=True, exist_ok=True)
drive_config_dir.mkdir(parents=True, exist_ok=True)

repo_protocol_path = (
    repo_config_dir / "temporal_validation_protocol_v1.json"
)

drive_protocol_path = (
    drive_config_dir / "temporal_validation_protocol_v1.json"
)

protocol_text = json.dumps(
    protocol,
    indent=2,
    ensure_ascii=False
)

repo_protocol_path.write_text(
    protocol_text,
    encoding="utf-8"
)

drive_protocol_path.write_text(
    protocol_text,
    encoding="utf-8"
)

# ------------------------------------------------------------
# Generate protocol checksum
# ------------------------------------------------------------

protocol_sha256 = hashlib.sha256(
    protocol_text.encode("utf-8")
).hexdigest()

checksum_text = (
    f"{protocol_sha256}  "
    f"{repo_protocol_path.name}\n"
)

repo_checksum_path = (
    repo_config_dir /
    "temporal_validation_protocol_v1.sha256"
)

drive_checksum_path = (
    drive_config_dir /
    "temporal_validation_protocol_v1.sha256"
)

repo_checksum_path.write_text(
    checksum_text,
    encoding="utf-8"
)

drive_checksum_path.write_text(
    checksum_text,
    encoding="utf-8"
)

# ------------------------------------------------------------
# Display confirmation
# ------------------------------------------------------------

print("PROTOCOL CREATED SUCCESSFULLY")
print("=" * 65)
print("Protocol version:", protocol["protocol_version"])
print(
    "Primary unit:",
    protocol["primary_unit_of_analysis"]["level"]
)
print(
    "Primary genes:",
    ", ".join(protocol["population"]["primary_genes"])
)
print(
    "Exploratory gene:",
    ", ".join(protocol["population"]["exploratory_genes"])
)
print(
    "Expected T0 release:",
    protocol["release_design"]["baseline_expected_date"]
)
print(
    "Expected T1 release:",
    protocol["release_design"]["followup_expected_date"]
)
print("Primary metric:", protocol["primary_metric"])

print("\nSaved to GitHub repository:")
print(repo_protocol_path)

print("\nSaved to Google Drive:")
print(drive_protocol_path)

print("\nProtocol SHA-256:")
print(protocol_sha256)

print("\nIMPORTANT:")
print(
    "Any later protocol change must be saved as a new version "
    "rather than silently overwriting Version 1."
)

PROTOCOL CREATED SUCCESSFULLY
Protocol version: 1.0.0
Primary unit: RCV
Primary genes: BRCA1, BRCA2, MLH1
Exploratory gene: EGFR
Expected T0 release: 2023-01-05
Expected T1 release: 2026-01-01
Primary metric: AUPRC

Saved to GitHub repository:
/content/genomic-evidence-reliability/configs/temporal_validation_protocol_v1.json

Saved to Google Drive:
/content/drive/MyDrive/GES_RAG_Temporal_Study/configs/temporal_validation_protocol_v1.json

Protocol SHA-256:
60ea29236abddec303b624f85b9a47ad7accf5b0eff47295c772147b9c5f1c43

IMPORTANT:
Any later protocol change must be saved as a new version rather than silently overwriting Version 1.


In [3]:
# ============================================================
# STEP 3: Download and verify archived ClinVar variant summaries
# T0 = January 2023
# T1 = January 2026
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import requests
import hashlib
import gzip
import json
import shutil

# ------------------------------------------------------------
# Project directories
# ------------------------------------------------------------

REPO_DIR = Path("/content/genomic-evidence-reliability")
PROJECT_DATA_DIR = Path(
    "/content/drive/MyDrive/GES_RAG_Temporal_Study"
)

RAW_DATA_DIR = PROJECT_DATA_DIR / "data_raw" / "clinvar"
MANIFEST_DIR = PROJECT_DATA_DIR / "configs"
REPO_CONFIG_DIR = REPO_DIR / "configs"

RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)
REPO_CONFIG_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Official archived ClinVar files
# ------------------------------------------------------------

releases = {
    "T0": {
        "release_month": "2023-01",
        "url": (
            "https://ftp.ncbi.nlm.nih.gov/pub/clinvar/"
            "tab_delimited/archive/"
            "variant_summary_2023-01.txt.gz"
        ),
        "filename": "variant_summary_2023-01.txt.gz"
    },
    "T1": {
        "release_month": "2026-01",
        "url": (
            "https://ftp.ncbi.nlm.nih.gov/pub/clinvar/"
            "tab_delimited/archive/"
            "variant_summary_2026-01.txt.gz"
        ),
        "filename": "variant_summary_2026-01.txt.gz"
    }
}

# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------

def sha256_file(file_path, chunk_size=1024 * 1024):
    """Calculate SHA-256 without loading the whole file into memory."""
    digest = hashlib.sha256()

    with open(file_path, "rb") as file_handle:
        while True:
            chunk = file_handle.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def test_gzip_integrity(file_path, chunk_size=1024 * 1024):
    """Read through the gzip stream to detect truncation or corruption."""
    decompressed_bytes = 0

    with gzip.open(file_path, "rb") as gzip_handle:
        while True:
            chunk = gzip_handle.read(chunk_size)

            if not chunk:
                break

            decompressed_bytes += len(chunk)

    return decompressed_bytes


def download_file(url, destination):
    """Download to a temporary file, then rename after success."""
    temporary_path = destination.with_suffix(
        destination.suffix + ".part"
    )

    if temporary_path.exists():
        temporary_path.unlink()

    print(f"\nChecking URL:\n{url}")

    with requests.get(
        url,
        stream=True,
        timeout=(30, 300),
        headers={"User-Agent": "GES-RAG-research/1.0"}
    ) as response:

        response.raise_for_status()

        response_headers = {
            "content_length_header": response.headers.get(
                "Content-Length"
            ),
            "last_modified_header": response.headers.get(
                "Last-Modified"
            ),
            "etag_header": response.headers.get("ETag"),
            "content_type_header": response.headers.get(
                "Content-Type"
            )
        }

        expected_size = response.headers.get("Content-Length")

        if expected_size is not None:
            expected_size = int(expected_size)

        downloaded_size = 0

        with open(temporary_path, "wb") as output_handle:
            for chunk in response.iter_content(
                chunk_size=1024 * 1024
            ):
                if chunk:
                    output_handle.write(chunk)
                    downloaded_size += len(chunk)

                    print(
                        f"\rDownloaded: "
                        f"{downloaded_size / (1024**2):,.1f} MB",
                        end=""
                    )

    print()

    if expected_size is not None and downloaded_size != expected_size:
        temporary_path.unlink(missing_ok=True)

        raise IOError(
            f"Download size mismatch for {url}. "
            f"Expected {expected_size:,} bytes but received "
            f"{downloaded_size:,} bytes."
        )

    temporary_path.replace(destination)

    return response_headers


# ------------------------------------------------------------
# Download and verify each release
# ------------------------------------------------------------

manifest = {
    "study_id": "GES-RAG",
    "experiment": "Temporal Validation of GES",
    "manifest_version": "1.0.0",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "source": "NCBI ClinVar archived tab-delimited releases",
    "files": {}
}

for timepoint, release_info in releases.items():

    destination = RAW_DATA_DIR / release_info["filename"]

    print("\n" + "=" * 70)
    print(
        f"{timepoint}: ClinVar release "
        f"{release_info['release_month']}"
    )
    print("=" * 70)

    # Use an existing complete file rather than downloading it twice.
    if destination.exists() and destination.stat().st_size > 0:
        print("Existing file found. It will be verified again.")
        response_headers = {
            "content_length_header": None,
            "last_modified_header": None,
            "etag_header": None,
            "content_type_header": None,
            "download_skipped_existing_file": True
        }
    else:
        response_headers = download_file(
            release_info["url"],
            destination
        )
        response_headers[
            "download_skipped_existing_file"
        ] = False

    compressed_size = destination.stat().st_size

    print("Calculating SHA-256...")
    file_sha256 = sha256_file(destination)

    print("Testing gzip integrity...")
    decompressed_size = test_gzip_integrity(destination)

    manifest["files"][timepoint] = {
        "release_month": release_info["release_month"],
        "url": release_info["url"],
        "local_path": str(destination),
        "filename": destination.name,
        "compressed_bytes": compressed_size,
        "compressed_megabytes": round(
            compressed_size / (1024**2),
            2
        ),
        "decompressed_bytes": decompressed_size,
        "decompressed_megabytes": round(
            decompressed_size / (1024**2),
            2
        ),
        "sha256": file_sha256,
        "gzip_integrity": "passed",
        "http_metadata": response_headers
    }

    print(f"File: {destination.name}")
    print(
        f"Compressed size: "
        f"{compressed_size / (1024**2):,.2f} MB"
    )
    print(
        f"Decompressed size: "
        f"{decompressed_size / (1024**2):,.2f} MB"
    )
    print(f"SHA-256: {file_sha256}")
    print("Gzip integrity: PASSED")

# ------------------------------------------------------------
# Save the provenance manifest
# ------------------------------------------------------------

manifest_text = json.dumps(
    manifest,
    indent=2,
    ensure_ascii=False
)

drive_manifest_path = (
    MANIFEST_DIR /
    "clinvar_temporal_release_manifest_v1.json"
)

repo_manifest_path = (
    REPO_CONFIG_DIR /
    "clinvar_temporal_release_manifest_v1.json"
)

drive_manifest_path.write_text(
    manifest_text,
    encoding="utf-8"
)

repo_manifest_path.write_text(
    manifest_text,
    encoding="utf-8"
)

manifest_sha256 = hashlib.sha256(
    manifest_text.encode("utf-8")
).hexdigest()

drive_manifest_checksum_path = (
    MANIFEST_DIR /
    "clinvar_temporal_release_manifest_v1.sha256"
)

repo_manifest_checksum_path = (
    REPO_CONFIG_DIR /
    "clinvar_temporal_release_manifest_v1.sha256"
)

checksum_text = (
    f"{manifest_sha256}  "
    f"{drive_manifest_path.name}\n"
)

drive_manifest_checksum_path.write_text(
    checksum_text,
    encoding="utf-8"
)

repo_manifest_checksum_path.write_text(
    checksum_text,
    encoding="utf-8"
)

# ------------------------------------------------------------
# Final confirmation
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("ARCHIVED RELEASE DOWNLOAD COMPLETED")
print("=" * 70)

for timepoint, file_info in manifest["files"].items():
    print(
        f"{timepoint}: {file_info['filename']} | "
        f"{file_info['compressed_megabytes']:,.2f} MB | "
        f"gzip={file_info['gzip_integrity']}"
    )

print("\nManifest saved to Google Drive:")
print(drive_manifest_path)

print("\nManifest saved to GitHub repository:")
print(repo_manifest_path)

print("\nManifest SHA-256:")
print(manifest_sha256)

print("\nNo ClinVar records have been analyzed yet.")


T0: ClinVar release 2023-01

Checking URL:
https://ftp.ncbi.nlm.nih.gov/pub/clinvar/tab_delimited/archive/variant_summary_2023-01.txt.gz


HTTPError: 404 Client Error: Not Found for url: https://ftp.ncbi.nlm.nih.gov/pub/clinvar/tab_delimited/archive/variant_summary_2023-01.txt.gz

In [4]:
# ============================================================
# STEP 3A: Verify corrected ClinVar archive URLs
# No files are downloaded in this step
# ============================================================

import requests

releases = {
    "T0": {
        "release_month": "2023-01",
        "release_date": "2023-01-05",
        "url": (
            "https://ftp.ncbi.nlm.nih.gov/pub/clinvar/"
            "tab_delimited/archive/2023/"
            "variant_summary_2023-01.txt.gz"
        ),
        "filename": "variant_summary_2023-01.txt.gz"
    },
    "T1": {
        "release_month": "2026-01",
        "release_date": "2026-01-01",
        "url": (
            "https://ftp.ncbi.nlm.nih.gov/pub/clinvar/"
            "tab_delimited/archive/"
            "variant_summary_2026-01.txt.gz"
        ),
        "filename": "variant_summary_2026-01.txt.gz"
    }
}

print("VERIFYING OFFICIAL CLINVAR ARCHIVE FILES")
print("=" * 70)

all_urls_valid = True

for timepoint, info in releases.items():

    response = requests.head(
        info["url"],
        allow_redirects=True,
        timeout=60,
        headers={"User-Agent": "GES-RAG-research/1.0"}
    )

    status_code = response.status_code
    content_length = response.headers.get("Content-Length")
    last_modified = response.headers.get("Last-Modified")

    print(f"\n{timepoint}: {info['release_month']}")
    print("URL:", info["url"])
    print("HTTP status:", status_code)
    print("Content-Length:", content_length)
    print("Last-Modified:", last_modified)

    if status_code != 200:
        all_urls_valid = False
        print("Result: FAILED")
    else:
        print("Result: AVAILABLE")

print("\n" + "=" * 70)

if not all_urls_valid:
    raise RuntimeError(
        "At least one ClinVar URL could not be verified."
    )

print("BOTH ARCHIVED RELEASE FILES ARE AVAILABLE")
print("No data have been downloaded or analyzed yet.")

VERIFYING OFFICIAL CLINVAR ARCHIVE FILES

T0: 2023-01
URL: https://ftp.ncbi.nlm.nih.gov/pub/clinvar/tab_delimited/archive/2023/variant_summary_2023-01.txt.gz
HTTP status: 200
Content-Length: 148245222
Last-Modified: Thu, 05 Jan 2023 05:05:01 GMT
Result: AVAILABLE

T1: 2026-01
URL: https://ftp.ncbi.nlm.nih.gov/pub/clinvar/tab_delimited/archive/variant_summary_2026-01.txt.gz
HTTP status: 200
Content-Length: 412880955
Last-Modified: Thu, 01 Jan 2026 05:05:02 GMT
Result: AVAILABLE

BOTH ARCHIVED RELEASE FILES ARE AVAILABLE
No data have been downloaded or analyzed yet.


In [5]:
# ============================================================
# STEP 3B: Download, validate, and archive ClinVar T0 and T1
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import subprocess
import hashlib
import gzip
import json
import shutil
import os

# ------------------------------------------------------------
# Project locations
# ------------------------------------------------------------

REPO_DIR = Path("/content/genomic-evidence-reliability")

PROJECT_DATA_DIR = Path(
    "/content/drive/MyDrive/GES_RAG_Temporal_Study"
)

RAW_DATA_DIR = (
    PROJECT_DATA_DIR /
    "data_raw" /
    "clinvar"
)

DRIVE_CONFIG_DIR = PROJECT_DATA_DIR / "configs"
REPO_CONFIG_DIR = REPO_DIR / "configs"

RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_CONFIG_DIR.mkdir(parents=True, exist_ok=True)
REPO_CONFIG_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Verified official ClinVar releases
# ------------------------------------------------------------

releases = {
    "T0": {
        "release_month": "2023-01",
        "release_date": "2023-01-05",
        "url": (
            "https://ftp.ncbi.nlm.nih.gov/pub/clinvar/"
            "tab_delimited/archive/2023/"
            "variant_summary_2023-01.txt.gz"
        ),
        "filename": "variant_summary_2023-01.txt.gz",
        "expected_compressed_bytes": 148245222
    },

    "T1": {
        "release_month": "2026-01",
        "release_date": "2026-01-01",
        "url": (
            "https://ftp.ncbi.nlm.nih.gov/pub/clinvar/"
            "tab_delimited/archive/"
            "variant_summary_2026-01.txt.gz"
        ),
        "filename": "variant_summary_2026-01.txt.gz",
        "expected_compressed_bytes": 412880955
    }
}

# ------------------------------------------------------------
# Utility functions
# ------------------------------------------------------------

def sha256_file(file_path, chunk_size=8 * 1024 * 1024):
    """
    Calculate a file SHA-256 checksum without loading the
    complete file into memory.
    """

    digest = hashlib.sha256()

    with open(file_path, "rb") as file_handle:
        while True:
            chunk = file_handle.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def validate_gzip(file_path, chunk_size=8 * 1024 * 1024):
    """
    Read the complete gzip stream to verify that the archive
    is not truncated or corrupted.
    """

    decompressed_bytes = 0

    with gzip.open(file_path, "rb") as gzip_handle:
        while True:
            chunk = gzip_handle.read(chunk_size)

            if not chunk:
                break

            decompressed_bytes += len(chunk)

    return decompressed_bytes


def download_with_resume(url, destination):
    """
    Download using wget with resume support.

    A temporary .part file is used so that an incomplete file
    cannot be mistaken for a completed research archive.
    """

    temporary_path = Path(str(destination) + ".part")

    print("Downloading:")
    print(url)

    command = [
        "wget",
        "--continue",
        "--tries=5",
        "--timeout=60",
        "--read-timeout=300",
        "--show-progress",
        "--output-document",
        str(temporary_path),
        url
    ]

    result = subprocess.run(command)

    if result.returncode != 0:
        raise RuntimeError(
            f"Download failed for {url}. "
            f"The partial file remains at {temporary_path}, "
            "so the download can resume when this cell is rerun."
        )

    temporary_path.replace(destination)


# ------------------------------------------------------------
# Confirm available Google Drive space
# ------------------------------------------------------------

drive_usage = shutil.disk_usage(PROJECT_DATA_DIR)

print("GOOGLE DRIVE STORAGE CHECK")
print("=" * 70)
print(
    f"Free space visible to Colab: "
    f"{drive_usage.free / (1024**3):,.2f} GB"
)

required_download_bytes = sum(
    item["expected_compressed_bytes"]
    for item in releases.values()
)

print(
    f"Required compressed download space: "
    f"{required_download_bytes / (1024**3):,.2f} GB"
)

if drive_usage.free < required_download_bytes * 2:
    raise OSError(
        "Insufficient visible free space for safe download "
        "and temporary-file handling."
    )

# ------------------------------------------------------------
# Download and validate releases
# ------------------------------------------------------------

manifest = {
    "study_id": "GES-RAG",
    "experiment": "Experiment 1: Temporal Validation of GES",
    "manifest_version": "1.0.0",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "source": "NCBI ClinVar archived tab-delimited releases",
    "archive_verification": {
        "T0_http_status_verified": 200,
        "T1_http_status_verified": 200,
        "verification_completed_before_download": True
    },
    "files": {}
}

for timepoint, release in releases.items():

    destination = RAW_DATA_DIR / release["filename"]

    print("\n" + "=" * 70)
    print(
        f"{timepoint}: ClinVar {release['release_month']} "
        f"release"
    )
    print("=" * 70)

    existing_file_valid = False

    if destination.exists():

        existing_size = destination.stat().st_size

        if existing_size == release["expected_compressed_bytes"]:
            print(
                "A complete-size local file already exists. "
                "The download will be skipped."
            )
            existing_file_valid = True

        else:
            print(
                "An incomplete or unexpected-size local file exists."
            )
            print(
                f"Observed size: {existing_size:,} bytes"
            )
            print(
                "It will be removed before a clean download."
            )
            destination.unlink()

    if not existing_file_valid:
        download_with_resume(
            release["url"],
            destination
        )

    # --------------------------------------------------------
    # Exact compressed-size validation
    # --------------------------------------------------------

    observed_compressed_bytes = destination.stat().st_size
    expected_compressed_bytes = release[
        "expected_compressed_bytes"
    ]

    print("\nValidating compressed file size...")

    if observed_compressed_bytes != expected_compressed_bytes:
        raise IOError(
            f"{timepoint} size mismatch. "
            f"Expected {expected_compressed_bytes:,} bytes, "
            f"but found {observed_compressed_bytes:,} bytes."
        )

    print(
        f"Compressed size confirmed: "
        f"{observed_compressed_bytes:,} bytes"
    )

    # --------------------------------------------------------
    # SHA-256 checksum
    # --------------------------------------------------------

    print("Calculating SHA-256 checksum...")

    file_sha256 = sha256_file(destination)

    print("SHA-256:", file_sha256)

    # --------------------------------------------------------
    # Gzip-integrity validation
    # --------------------------------------------------------

    print("Testing complete gzip stream...")

    decompressed_bytes = validate_gzip(destination)

    print(
        f"Gzip integrity passed. "
        f"Decompressed bytes read: {decompressed_bytes:,}"
    )

    # --------------------------------------------------------
    # Manifest record
    # --------------------------------------------------------

    manifest["files"][timepoint] = {
        "release_month": release["release_month"],
        "release_date": release["release_date"],
        "filename": release["filename"],
        "official_url": release["url"],
        "local_path": str(destination),
        "expected_compressed_bytes": expected_compressed_bytes,
        "observed_compressed_bytes": observed_compressed_bytes,
        "compressed_megabytes": round(
            observed_compressed_bytes / (1024**2),
            2
        ),
        "decompressed_bytes": decompressed_bytes,
        "decompressed_megabytes": round(
            decompressed_bytes / (1024**2),
            2
        ),
        "sha256": file_sha256,
        "gzip_integrity": "passed",
        "record_content_examined": False
    }

# ------------------------------------------------------------
# Save provenance manifest
# ------------------------------------------------------------

manifest_text = json.dumps(
    manifest,
    indent=2,
    ensure_ascii=False
)

manifest_filename = (
    "clinvar_temporal_release_manifest_v1.json"
)

drive_manifest_path = (
    DRIVE_CONFIG_DIR /
    manifest_filename
)

repo_manifest_path = (
    REPO_CONFIG_DIR /
    manifest_filename
)

drive_manifest_path.write_text(
    manifest_text,
    encoding="utf-8"
)

repo_manifest_path.write_text(
    manifest_text,
    encoding="utf-8"
)

# ------------------------------------------------------------
# Save manifest checksum
# ------------------------------------------------------------

manifest_sha256 = hashlib.sha256(
    manifest_text.encode("utf-8")
).hexdigest()

checksum_filename = (
    "clinvar_temporal_release_manifest_v1.sha256"
)

checksum_text = (
    f"{manifest_sha256}  {manifest_filename}\n"
)

(
    DRIVE_CONFIG_DIR /
    checksum_filename
).write_text(
    checksum_text,
    encoding="utf-8"
)

(
    REPO_CONFIG_DIR /
    checksum_filename
).write_text(
    checksum_text,
    encoding="utf-8"
)

# ------------------------------------------------------------
# Final confirmation
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("ARCHIVED RELEASE DOWNLOAD COMPLETED")
print("=" * 70)

for timepoint, file_info in manifest["files"].items():

    print(
        f"{timepoint}: "
        f"{file_info['filename']}"
    )

    print(
        f"  Release date: "
        f"{file_info['release_date']}"
    )

    print(
        f"  Compressed size: "
        f"{file_info['compressed_megabytes']:,.2f} MB"
    )

    print(
        f"  Decompressed size: "
        f"{file_info['decompressed_megabytes']:,.2f} MB"
    )

    print(
        f"  SHA-256: "
        f"{file_info['sha256']}"
    )

    print(
        f"  Gzip integrity: "
        f"{file_info['gzip_integrity'].upper()}"
    )

print("\nManifest saved in Google Drive:")
print(drive_manifest_path)

print("\nManifest saved in GitHub repository:")
print(repo_manifest_path)

print("\nManifest SHA-256:")
print(manifest_sha256)

print("\nNo ClinVar record content has been analyzed yet.")

GOOGLE DRIVE STORAGE CHECK
Free space visible to Colab: 1.07 GB
Required compressed download space: 0.52 GB

T0: ClinVar 2023-01 release
Downloading:
https://ftp.ncbi.nlm.nih.gov/pub/clinvar/tab_delimited/archive/2023/variant_summary_2023-01.txt.gz

Validating compressed file size...
Compressed size confirmed: 148,245,222 bytes
Calculating SHA-256 checksum...
SHA-256: 0acea623ca1b30fcdf8f460c9d70981e03e56c7e5dc566cae6c9a2fbb1ef9830
Testing complete gzip stream...
Gzip integrity passed. Decompressed bytes read: 1,382,458,833

T1: ClinVar 2026-01 release
Downloading:
https://ftp.ncbi.nlm.nih.gov/pub/clinvar/tab_delimited/archive/variant_summary_2026-01.txt.gz

Validating compressed file size...
Compressed size confirmed: 412,880,955 bytes
Calculating SHA-256 checksum...
SHA-256: 37bffab7594d26d74882dd21721300911ce1d625033ff1b77fb7b6a35b50d74c
Testing complete gzip stream...
Gzip integrity passed. Decompressed bytes read: 3,687,468,861

ARCHIVED RELEASE DOWNLOAD COMPLETED
T0: variant_summ

In [6]:
# ============================================================
# STEP 4: Inspect and freeze the T0 and T1 file schemas
# Only header rows are read; no ClinVar records are analyzed
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import gzip
import json
import hashlib

# ------------------------------------------------------------
# Project locations
# ------------------------------------------------------------

REPO_DIR = Path("/content/genomic-evidence-reliability")

PROJECT_DATA_DIR = Path(
    "/content/drive/MyDrive/GES_RAG_Temporal_Study"
)

RAW_DATA_DIR = (
    PROJECT_DATA_DIR /
    "data_raw" /
    "clinvar"
)

DRIVE_CONFIG_DIR = PROJECT_DATA_DIR / "configs"
REPO_CONFIG_DIR = REPO_DIR / "configs"

T0_PATH = RAW_DATA_DIR / "variant_summary_2023-01.txt.gz"
T1_PATH = RAW_DATA_DIR / "variant_summary_2026-01.txt.gz"

for required_path in [T0_PATH, T1_PATH]:
    if not required_path.exists():
        raise FileNotFoundError(
            f"Required ClinVar file was not found:\n{required_path}"
        )

# ------------------------------------------------------------
# Read only the first line of each gzip file
# ------------------------------------------------------------

def read_header_only(file_path):
    """
    Read only the first text line from a gzip-compressed TSV.
    No data records are loaded.
    """

    with gzip.open(
        file_path,
        mode="rt",
        encoding="utf-8",
        errors="replace"
    ) as file_handle:
        header_line = file_handle.readline().rstrip("\n\r")

    columns = header_line.split("\t")

    # ClinVar commonly prefixes the first column with '#'.
    normalized_columns = [
        column.lstrip("#").strip()
        for column in columns
    ]

    return {
        "raw_header": header_line,
        "columns": columns,
        "normalized_columns": normalized_columns,
        "column_count": len(columns)
    }


schemas = {
    "T0": read_header_only(T0_PATH),
    "T1": read_header_only(T1_PATH)
}

t0_columns = schemas["T0"]["normalized_columns"]
t1_columns = schemas["T1"]["normalized_columns"]

common_columns = sorted(
    set(t0_columns).intersection(t1_columns)
)

t0_only_columns = sorted(
    set(t0_columns).difference(t1_columns)
)

t1_only_columns = sorted(
    set(t1_columns).difference(t0_columns)
)

# ------------------------------------------------------------
# Check fields needed for the temporal study
# ------------------------------------------------------------

required_field_candidates = {
    "allele_identifier": [
        "AlleleID"
    ],
    "variation_identifier": [
        "VariationID"
    ],
    "rcv_accession": [
        "RCVaccession",
        "RCVAccession"
    ],
    "gene_symbol": [
        "GeneSymbol"
    ],
    "clinical_significance": [
        "ClinicalSignificance"
    ],
    "last_evaluated": [
        "LastEvaluated"
    ],
    "review_status": [
        "ReviewStatus"
    ],
    "submitter_count": [
        "NumberSubmitters"
    ],
    "phenotype_identifiers": [
        "PhenotypeIDS",
        "PhenotypeIDs"
    ],
    "phenotype_names": [
        "PhenotypeList"
    ],
    "assembly": [
        "Assembly"
    ],
    "origin": [
        "Origin",
        "OriginSimple"
    ]
}


def locate_candidate_field(columns, candidates):
    for candidate in candidates:
        if candidate in columns:
            return candidate

    return None


field_availability = {}

for conceptual_field, candidates in required_field_candidates.items():

    t0_match = locate_candidate_field(
        t0_columns,
        candidates
    )

    t1_match = locate_candidate_field(
        t1_columns,
        candidates
    )

    field_availability[conceptual_field] = {
        "candidate_names": candidates,
        "T0_column": t0_match,
        "T1_column": t1_match,
        "available_in_both": (
            t0_match is not None and
            t1_match is not None
        )
    }

# ------------------------------------------------------------
# Save schema manifest
# ------------------------------------------------------------

schema_manifest = {
    "study_id": "GES-RAG",
    "experiment": "Experiment 1: Temporal Validation of GES",
    "schema_manifest_version": "1.0.0",
    "created_utc": datetime.now(timezone.utc).isoformat(),

    "scope": (
        "Header-only inspection. No ClinVar data records "
        "were read or analyzed."
    ),

    "files": {
        "T0": {
            "filename": T0_PATH.name,
            "column_count": schemas["T0"]["column_count"],
            "normalized_columns": t0_columns
        },
        "T1": {
            "filename": T1_PATH.name,
            "column_count": schemas["T1"]["column_count"],
            "normalized_columns": t1_columns
        }
    },

    "schema_comparison": {
        "common_columns": common_columns,
        "T0_only_columns": t0_only_columns,
        "T1_only_columns": t1_only_columns
    },

    "required_field_availability": field_availability
}

schema_text = json.dumps(
    schema_manifest,
    indent=2,
    ensure_ascii=False
)

schema_filename = (
    "clinvar_variant_summary_schema_manifest_v1.json"
)

drive_schema_path = (
    DRIVE_CONFIG_DIR /
    schema_filename
)

repo_schema_path = (
    REPO_CONFIG_DIR /
    schema_filename
)

drive_schema_path.write_text(
    schema_text,
    encoding="utf-8"
)

repo_schema_path.write_text(
    schema_text,
    encoding="utf-8"
)

schema_sha256 = hashlib.sha256(
    schema_text.encode("utf-8")
).hexdigest()

checksum_filename = (
    "clinvar_variant_summary_schema_manifest_v1.sha256"
)

checksum_text = (
    f"{schema_sha256}  {schema_filename}\n"
)

(
    DRIVE_CONFIG_DIR /
    checksum_filename
).write_text(
    checksum_text,
    encoding="utf-8"
)

(
    REPO_CONFIG_DIR /
    checksum_filename
).write_text(
    checksum_text,
    encoding="utf-8"
)

# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print("CLINVAR SCHEMA INSPECTION COMPLETED")
print("=" * 72)

print(
    f"T0 column count: "
    f"{schemas['T0']['column_count']}"
)

print(
    f"T1 column count: "
    f"{schemas['T1']['column_count']}"
)

print(
    f"Columns shared by T0 and T1: "
    f"{len(common_columns)}"
)

print("\nREQUIRED FIELD AVAILABILITY")
print("-" * 72)

all_required_available = True

for conceptual_field, result in field_availability.items():

    status = (
        "AVAILABLE"
        if result["available_in_both"]
        else "MISSING"
    )

    if not result["available_in_both"]:
        all_required_available = False

    print(
        f"{conceptual_field:25s} | "
        f"T0={str(result['T0_column']):20s} | "
        f"T1={str(result['T1_column']):20s} | "
        f"{status}"
    )

print("\nT1-ONLY COLUMNS")
print("-" * 72)

if t1_only_columns:
    for column in t1_only_columns:
        print(" -", column)
else:
    print("None")

print("\nT0-ONLY COLUMNS")
print("-" * 72)

if t0_only_columns:
    for column in t0_only_columns:
        print(" -", column)
else:
    print("None")

print("\n" + "=" * 72)

if all_required_available:
    print(
        "ALL REQUIRED SUMMARY FIELDS ARE AVAILABLE "
        "IN BOTH RELEASES"
    )
else:
    print(
        "ONE OR MORE REQUIRED SUMMARY FIELDS ARE MISSING"
    )

print("\nSchema manifest saved to Google Drive:")
print(drive_schema_path)

print("\nSchema manifest saved to GitHub repository:")
print(repo_schema_path)

print("\nSchema manifest SHA-256:")
print(schema_sha256)

print("\nNo ClinVar data records were analyzed.")

CLINVAR SCHEMA INSPECTION COMPLETED
T0 column count: 34
T1 column count: 43
Columns shared by T0 and T1: 34

REQUIRED FIELD AVAILABILITY
------------------------------------------------------------------------
allele_identifier         | T0=AlleleID             | T1=AlleleID             | AVAILABLE
variation_identifier      | T0=VariationID          | T1=VariationID          | AVAILABLE
rcv_accession             | T0=RCVaccession         | T1=RCVaccession         | AVAILABLE
gene_symbol               | T0=GeneSymbol           | T1=GeneSymbol           | AVAILABLE
clinical_significance     | T0=ClinicalSignificance | T1=ClinicalSignificance | AVAILABLE
last_evaluated            | T0=LastEvaluated        | T1=LastEvaluated        | AVAILABLE
review_status             | T0=ReviewStatus         | T1=ReviewStatus         | AVAILABLE
submitter_count           | T0=NumberSubmitters     | T1=NumberSubmitters     | AVAILABLE
phenotype_identifiers     | T0=PhenotypeIDS         | T1=PhenotypeIDS 

In [7]:
# ============================================================
# STEP 5: Audit the unit of analysis and RCV multiplicity
# No temporal outcomes or model scores are created
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
import pandas as pd
import numpy as np
import hashlib
import json
import re

# ------------------------------------------------------------
# Project locations
# ------------------------------------------------------------

REPO_DIR = Path("/content/genomic-evidence-reliability")

PROJECT_DATA_DIR = Path(
    "/content/drive/MyDrive/GES_RAG_Temporal_Study"
)

RAW_DATA_DIR = (
    PROJECT_DATA_DIR /
    "data_raw" /
    "clinvar"
)

DRIVE_CONFIG_DIR = PROJECT_DATA_DIR / "configs"
REPO_CONFIG_DIR = REPO_DIR / "configs"

T0_PATH = RAW_DATA_DIR / "variant_summary_2023-01.txt.gz"
T1_PATH = RAW_DATA_DIR / "variant_summary_2026-01.txt.gz"

for required_path in [T0_PATH, T1_PATH]:
    if not required_path.exists():
        raise FileNotFoundError(
            f"Required file not found:\n{required_path}"
        )

# ------------------------------------------------------------
# Prespecified genes and selected audit columns
# ------------------------------------------------------------

TARGET_GENES = [
    "BRCA1",
    "BRCA2",
    "MLH1",
    "EGFR"
]

USE_COLUMNS = [
    "Assembly",
    "GeneSymbol",
    "VariationID",
    "RCVaccession",
    "PhenotypeIDS",
    "PhenotypeList",
    "ClinicalSignificance",
    "ReviewStatus"
]

CHUNK_SIZE = 250_000

# Match a target symbol either alone or within a delimited field.
gene_pattern = re.compile(
    r"(?:^|[;,|])\s*("
    + "|".join(TARGET_GENES)
    + r")\s*(?:$|[;,|])",
    flags=re.IGNORECASE
)

# Find valid RCV accessions while allowing optional versions.
rcv_pattern = re.compile(
    r"RCV\d+(?:\.\d+)?",
    flags=re.IGNORECASE
)

# ------------------------------------------------------------
# Audit one archived release
# ------------------------------------------------------------

def audit_release(file_path, timepoint):
    """
    Stream selected fields from one ClinVar release and audit
    RCV multiplicity for GRCh38 records involving target genes.
    """

    total_rows_read = 0
    grch38_target_rows = 0
    rows_without_rcv = 0
    rows_with_one_rcv = 0
    rows_with_multiple_rcv = 0
    multi_gene_symbol_rows = 0
    max_rcv_per_row = 0

    rcv_count_distribution = Counter()
    gene_row_counts = Counter()

    unique_variation_ids = set()
    unique_rcv_accessions = set()

    multi_rcv_samples = []
    no_rcv_samples = []
    multi_gene_samples = []

    print("\n" + "=" * 76)
    print(f"Auditing {timepoint}: {file_path.name}")
    print("=" * 76)

    reader = pd.read_csv(
        file_path,
        sep="\t",
        compression="gzip",
        usecols=USE_COLUMNS,
        dtype="string",
        chunksize=CHUNK_SIZE,
        low_memory=False
    )

    for chunk_number, chunk in enumerate(reader, start=1):

        total_rows_read += len(chunk)

        # Normalize only fields needed for filtering.
        assembly = chunk["Assembly"].fillna("").str.strip()

        gene_symbols = (
            chunk["GeneSymbol"]
            .fillna("")
            .str.strip()
        )

        # Restrict to GRCh38.
        assembly_mask = assembly.str.upper().eq("GRCH38")

        # Include rows where at least one target gene appears.
        gene_mask = gene_symbols.str.contains(
            gene_pattern,
            na=False
        )

        selected = chunk.loc[
            assembly_mask & gene_mask
        ].copy()

        if selected.empty:
            print(
                f"\rChunks processed: {chunk_number:,} | "
                f"Rows read: {total_rows_read:,} | "
                f"Target rows: {grch38_target_rows:,}",
                end=""
            )
            continue

        grch38_target_rows += len(selected)

        # ----------------------------------------------------
        # Identify target genes represented by each row
        # ----------------------------------------------------

        selected["_target_genes"] = (
            selected["GeneSymbol"]
            .fillna("")
            .apply(
                lambda value: sorted(
                    {
                        token.strip().upper()
                        for token in re.split(
                            r"[;,|]",
                            str(value)
                        )
                        if token.strip().upper()
                        in TARGET_GENES
                    }
                )
            )
        )

        for genes in selected["_target_genes"]:
            for gene in genes:
                gene_row_counts[gene] += 1

        multi_gene_mask = selected["_target_genes"].apply(
            lambda genes: len(genes) > 1
        )

        multi_gene_symbol_rows += int(
            multi_gene_mask.sum()
        )

        if (
            multi_gene_mask.any()
            and len(multi_gene_samples) < 8
        ):
            needed = 8 - len(multi_gene_samples)

            sample_columns = [
                "GeneSymbol",
                "VariationID",
                "RCVaccession"
            ]

            multi_gene_samples.extend(
                selected.loc[
                    multi_gene_mask,
                    sample_columns
                ]
                .head(needed)
                .fillna("")
                .to_dict(orient="records")
            )

        # ----------------------------------------------------
        # Extract all RCV accessions from each row
        # ----------------------------------------------------

        selected["_rcv_list"] = (
            selected["RCVaccession"]
            .fillna("")
            .apply(
                lambda value: sorted(
                    set(
                        match.upper()
                        for match in rcv_pattern.findall(
                            str(value)
                        )
                    )
                )
            )
        )

        selected["_rcv_count"] = (
            selected["_rcv_list"]
            .apply(len)
        )

        count_values = selected["_rcv_count"]

        rows_without_rcv += int(
            count_values.eq(0).sum()
        )

        rows_with_one_rcv += int(
            count_values.eq(1).sum()
        )

        rows_with_multiple_rcv += int(
            count_values.gt(1).sum()
        )

        if len(count_values) > 0:
            max_rcv_per_row = max(
                max_rcv_per_row,
                int(count_values.max())
            )

        rcv_count_distribution.update(
            count_values.astype(int).tolist()
        )

        # ----------------------------------------------------
        # Track unique identifiers
        # ----------------------------------------------------

        valid_variation_ids = (
            selected["VariationID"]
            .dropna()
            .astype(str)
            .str.strip()
        )

        unique_variation_ids.update(
            value
            for value in valid_variation_ids
            if value and value != "-1"
        )

        for rcv_list in selected["_rcv_list"]:
            unique_rcv_accessions.update(rcv_list)

        # ----------------------------------------------------
        # Store a few structural examples
        # ----------------------------------------------------

        multiple_mask = count_values.gt(1)

        if (
            multiple_mask.any()
            and len(multi_rcv_samples) < 10
        ):
            needed = 10 - len(multi_rcv_samples)

            sample_columns = [
                "GeneSymbol",
                "VariationID",
                "RCVaccession",
                "PhenotypeIDS",
                "PhenotypeList",
                "ClinicalSignificance",
                "ReviewStatus"
            ]

            sample_frame = selected.loc[
                multiple_mask,
                sample_columns
            ].head(needed)

            multi_rcv_samples.extend(
                sample_frame
                .fillna("")
                .to_dict(orient="records")
            )

        no_rcv_mask = count_values.eq(0)

        if (
            no_rcv_mask.any()
            and len(no_rcv_samples) < 5
        ):
            needed = 5 - len(no_rcv_samples)

            sample_columns = [
                "GeneSymbol",
                "VariationID",
                "RCVaccession",
                "PhenotypeIDS",
                "PhenotypeList"
            ]

            no_rcv_samples.extend(
                selected.loc[
                    no_rcv_mask,
                    sample_columns
                ]
                .head(needed)
                .fillna("")
                .to_dict(orient="records")
            )

        print(
            f"\rChunks processed: {chunk_number:,} | "
            f"Rows read: {total_rows_read:,} | "
            f"Target rows: {grch38_target_rows:,}",
            end=""
        )

    print()

    if grch38_target_rows == 0:
        raise RuntimeError(
            f"No GRCh38 target-gene records were found in "
            f"{file_path.name}."
        )

    multiple_rcv_percentage = (
        rows_with_multiple_rcv /
        grch38_target_rows *
        100
    )

    one_rcv_percentage = (
        rows_with_one_rcv /
        grch38_target_rows *
        100
    )

    no_rcv_percentage = (
        rows_without_rcv /
        grch38_target_rows *
        100
    )

    return {
        "timepoint": timepoint,
        "filename": file_path.name,
        "rows_read_from_complete_file": total_rows_read,
        "grch38_target_gene_rows": grch38_target_rows,
        "gene_row_counts": dict(
            sorted(gene_row_counts.items())
        ),
        "unique_variation_ids": len(
            unique_variation_ids
        ),
        "unique_rcv_accessions": len(
            unique_rcv_accessions
        ),
        "rows_without_rcv": rows_without_rcv,
        "rows_without_rcv_percentage": round(
            no_rcv_percentage,
            4
        ),
        "rows_with_one_rcv": rows_with_one_rcv,
        "rows_with_one_rcv_percentage": round(
            one_rcv_percentage,
            4
        ),
        "rows_with_multiple_rcv": (
            rows_with_multiple_rcv
        ),
        "rows_with_multiple_rcv_percentage": round(
            multiple_rcv_percentage,
            4
        ),
        "maximum_rcv_accessions_in_one_row": (
            max_rcv_per_row
        ),
        "rcv_count_distribution": {
            str(key): value
            for key, value in sorted(
                rcv_count_distribution.items()
            )
        },
        "rows_containing_multiple_target_genes": (
            multi_gene_symbol_rows
        ),
        "multi_rcv_structural_samples": (
            multi_rcv_samples
        ),
        "no_rcv_structural_samples": no_rcv_samples,
        "multi_gene_structural_samples": (
            multi_gene_samples
        )
    }


# ------------------------------------------------------------
# Run the audit
# ------------------------------------------------------------

audit_results = {
    "study_id": "GES-RAG",
    "audit_name": (
        "ClinVar variant_summary unit-of-analysis audit"
    ),
    "audit_version": "1.0.0",
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "scope": {
        "assembly": "GRCh38",
        "genes": TARGET_GENES,
        "temporal_outcomes_created": False,
        "model_training_performed": False,
        "purpose": (
            "Determine whether variant_summary rows map "
            "one-to-one or one-to-many to RCV accessions."
        )
    },
    "releases": {}
}

audit_results["releases"]["T0"] = audit_release(
    T0_PATH,
    "T0"
)

audit_results["releases"]["T1"] = audit_release(
    T1_PATH,
    "T1"
)

# ------------------------------------------------------------
# Determine whether direct RCV-level use is valid
# ------------------------------------------------------------

any_multi_rcv_rows = any(
    result["rows_with_multiple_rcv"] > 0
    for result in audit_results["releases"].values()
)

if any_multi_rcv_rows:
    audit_results["design_interpretation"] = {
        "direct_one_row_one_rcv_assumption_valid": False,
        "decision": (
            "Do not treat each variant_summary row as one "
            "RCV-level variant-condition record."
        ),
        "required_next_action": (
            "Either formally amend the primary analysis to "
            "VCV/VariationID-level or obtain an archived "
            "condition-specific ClinVar source."
        )
    }
else:
    audit_results["design_interpretation"] = {
        "direct_one_row_one_rcv_assumption_valid": True,
        "decision": (
            "The target-gene rows appear one-to-one with RCV "
            "accessions, subject to further identifier checks."
        ),
        "required_next_action": (
            "Proceed to controlled extraction and linkage audit."
        )
    }

# ------------------------------------------------------------
# Save audit
# ------------------------------------------------------------

audit_text = json.dumps(
    audit_results,
    indent=2,
    ensure_ascii=False
)

audit_filename = (
    "clinvar_rcv_multiplicity_audit_v1.json"
)

drive_audit_path = (
    DRIVE_CONFIG_DIR /
    audit_filename
)

repo_audit_path = (
    REPO_CONFIG_DIR /
    audit_filename
)

drive_audit_path.write_text(
    audit_text,
    encoding="utf-8"
)

repo_audit_path.write_text(
    audit_text,
    encoding="utf-8"
)

audit_sha256 = hashlib.sha256(
    audit_text.encode("utf-8")
).hexdigest()

checksum_filename = (
    "clinvar_rcv_multiplicity_audit_v1.sha256"
)

checksum_text = (
    f"{audit_sha256}  {audit_filename}\n"
)

(
    DRIVE_CONFIG_DIR /
    checksum_filename
).write_text(
    checksum_text,
    encoding="utf-8"
)

(
    REPO_CONFIG_DIR /
    checksum_filename
).write_text(
    checksum_text,
    encoding="utf-8"
)

# ------------------------------------------------------------
# Display concise audit results
# ------------------------------------------------------------

print("\n" + "=" * 76)
print("RCV MULTIPLICITY AUDIT COMPLETED")
print("=" * 76)

for timepoint in ["T0", "T1"]:

    result = audit_results["releases"][timepoint]

    print(f"\n{timepoint}: {result['filename']}")

    print(
        "  Complete-file rows read:",
        f"{result['rows_read_from_complete_file']:,}"
    )

    print(
        "  GRCh38 target-gene rows:",
        f"{result['grch38_target_gene_rows']:,}"
    )

    print(
        "  Unique VariationIDs:",
        f"{result['unique_variation_ids']:,}"
    )

    print(
        "  Unique RCV accessions:",
        f"{result['unique_rcv_accessions']:,}"
    )

    print(
        "  Rows with one RCV:",
        f"{result['rows_with_one_rcv']:,}",
        f"({result['rows_with_one_rcv_percentage']:.2f}%)"
    )

    print(
        "  Rows with multiple RCVs:",
        f"{result['rows_with_multiple_rcv']:,}",
        f"({result['rows_with_multiple_rcv_percentage']:.2f}%)"
    )

    print(
        "  Rows without an RCV:",
        f"{result['rows_without_rcv']:,}",
        f"({result['rows_without_rcv_percentage']:.2f}%)"
    )

    print(
        "  Maximum RCVs in one row:",
        result["maximum_rcv_accessions_in_one_row"]
    )

    print(
        "  Gene row counts:",
        result["gene_row_counts"]
    )

print("\nDESIGN INTERPRETATION")
print("-" * 76)

print(
    "Direct one-row/one-RCV assumption valid:",
    audit_results[
        "design_interpretation"
    ][
        "direct_one_row_one_rcv_assumption_valid"
    ]
)

print(
    "Decision:",
    audit_results[
        "design_interpretation"
    ]["decision"]
)

print(
    "Required next action:",
    audit_results[
        "design_interpretation"
    ]["required_next_action"]
)

print("\nAudit saved to Google Drive:")
print(drive_audit_path)

print("\nAudit saved to GitHub repository:")
print(repo_audit_path)

print("\nAudit SHA-256:")
print(audit_sha256)

print("\nNo temporal outcomes or GES scores were created.")


Auditing T0: variant_summary_2023-01.txt.gz


/tmp/ipykernel_6541/1536338811.py:140: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  gene_mask = gene_symbols.str.contains(


Chunks processed: 1 | Rows read: 250,000 | Target rows: 9,156

/tmp/ipykernel_6541/1536338811.py:140: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  gene_mask = gene_symbols.str.contains(


Chunks processed: 2 | Rows read: 500,000 | Target rows: 12,914

...

/tmp/ipykernel_6541/1536338811.py:140: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  gene_mask = gene_symbols.str.contains(


Chunks processed: 32 | Rows read: 8,000,000 | Target rows: 45,505

/tmp/ipykernel_6541/1536338811.py:140: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  gene_mask = gene_symbols.str.contains(


Chunks processed: 34 | Rows read: 8,378,417 | Target rows: 45,600

RCV MULTIPLICITY AUDIT COMPLETED

T0: variant_summary_2023-01.txt.gz
  Complete-file rows read: 3,255,919
  GRCh38 target-gene rows: 35,911
  Unique VariationIDs: 35,911
  Unique RCV accessions: 70,748
  Rows with one RCV: 19,663 (54.75%)
  Rows with multiple RCVs: 16,243 (45.23%)
  Rows without an RCV: 5 (0.01%)
  Maximum RCVs in one row: 15
  Gene row counts: {'BRCA1': 13066, 'BRCA2': 16109, 'EGFR': 2044, 'MLH1': 4692}

T1: variant_summary_2026-01.txt.gz
  Complete-file rows read: 8,378,417
  GRCh38 target-gene rows: 45,600
  Unique VariationIDs: 45,600
  Unique RCV accessions: 99,805
  Rows with one RCV: 21,644 (47.46%)
  Rows with multiple RCVs: 23,951 (52.52%)
  Rows without an RCV: 5 (0.01%)
  Maximum RCVs in one row: 19
  Gene row counts: {'BRCA1': 15076, 'BRCA2': 20750, 'EGFR': 3676, 'MLH1': 6098}

DESIGN INTERPRETATION
----------------------------------------------------------------------------
Direct one-row/o

/tmp/ipykernel_6541/1536338811.py:140: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  gene_mask = gene_symbols.str.contains(


In [8]:
# ============================================================
# STEP 6: Discover official ClinVar RCV XML release files
# No XML datasets are downloaded in this step
# ============================================================

from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import urljoin, urlparse
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import json
import hashlib

# ------------------------------------------------------------
# Project locations
# ------------------------------------------------------------

REPO_DIR = Path("/content/genomic-evidence-reliability")

PROJECT_DATA_DIR = Path(
    "/content/drive/MyDrive/GES_RAG_Temporal_Study"
)

DRIVE_CONFIG_DIR = PROJECT_DATA_DIR / "configs"
REPO_CONFIG_DIR = REPO_DIR / "configs"

DRIVE_CONFIG_DIR.mkdir(parents=True, exist_ok=True)
REPO_CONFIG_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Official ClinVar RCV XML locations
# ------------------------------------------------------------

DIRECTORY_ROOTS = {
    # Current RCV XML format
    "new_rcv_root": (
        "https://ftp.ncbi.nlm.nih.gov/pub/clinvar/"
        "xml/RCV_release/"
    ),

    "new_rcv_archive": (
        "https://ftp.ncbi.nlm.nih.gov/pub/clinvar/"
        "xml/RCV_release/archive/"
    ),

    # Historical RCV XML format
    "old_rcv_root": (
        "https://ftp.ncbi.nlm.nih.gov/pub/clinvar/"
        "xml/RCV_xml_old_format/"
    ),

    "old_rcv_archive": (
        "https://ftp.ncbi.nlm.nih.gov/pub/clinvar/"
        "xml/RCV_xml_old_format/archive/"
    )
}

TARGET_RELEASES = {
    "T0": {
        "year": "2023",
        "month": "01",
        "expected_release_date": "2023-01-05"
    },
    "T1": {
        "year": "2026",
        "month": "01",
        "expected_release_date": "2026-01-01"
    }
}

HEADERS = {
    "User-Agent": "GES-RAG-research/1.0"
}

# ------------------------------------------------------------
# Read one NCBI directory listing
# ------------------------------------------------------------

def read_directory_listing(url):
    """
    Return links and visible listing text from an NCBI FTP
    directory exposed through HTTPS.
    """

    response = requests.get(
        url,
        timeout=90,
        headers=HEADERS
    )

    response.raise_for_status()

    soup = BeautifulSoup(
        response.text,
        "html.parser"
    )

    entries = []

    for link in soup.find_all("a"):

        href = link.get("href")

        if not href:
            continue

        if href in {
            "../",
            "/",
            "?C=N;O=D",
            "?C=M;O=A",
            "?C=S;O=A",
            "?C=D;O=A"
        }:
            continue

        absolute_url = urljoin(url, href)

        # Keep only links within the same NCBI directory tree.
        if (
            urlparse(absolute_url).netloc !=
            "ftp.ncbi.nlm.nih.gov"
        ):
            continue

        parent_text = link.parent.get_text(
            " ",
            strip=True
        )

        entries.append({
            "directory_url": url,
            "name": href.rstrip("/"),
            "href": href,
            "url": absolute_url,
            "is_directory": href.endswith("/"),
            "listing_text": parent_text
        })

    return entries


# ------------------------------------------------------------
# Traverse root and archive folders to limited depth
# ------------------------------------------------------------

def crawl_directory_tree(start_url, max_depth=2):
    """
    Traverse a small number of directory levels while avoiding
    weekly-release folders and unrelated paths.
    """

    discovered = []
    visited = set()
    queue = [(start_url, 0)]

    while queue:

        current_url, depth = queue.pop(0)

        if current_url in visited:
            continue

        visited.add(current_url)

        print(
            f"Reading directory "
            f"(depth {depth}): {current_url}"
        )

        try:
            entries = read_directory_listing(
                current_url
            )

        except Exception as error:

            print(
                "  Could not read:",
                type(error).__name__,
                str(error)
            )

            discovered.append({
                "directory_url": current_url,
                "name": "",
                "href": "",
                "url": current_url,
                "is_directory": True,
                "listing_text": "",
                "crawl_error": (
                    f"{type(error).__name__}: {error}"
                )
            })

            continue

        for entry in entries:

            entry["crawl_depth"] = depth
            entry["crawl_error"] = None
            discovered.append(entry)

            if (
                entry["is_directory"]
                and depth < max_depth
            ):

                directory_name = (
                    entry["name"]
                    .lower()
                    .strip()
                )

                # Monthly/archive discovery only.
                if any(
                    excluded in directory_name
                    for excluded in [
                        "weekly",
                        "documentation",
                        "schema"
                    ]
                ):
                    continue

                queue.append(
                    (
                        entry["url"],
                        depth + 1
                    )
                )

    return discovered


# ------------------------------------------------------------
# Crawl each official location
# ------------------------------------------------------------

all_entries = []

for source_name, root_url in DIRECTORY_ROOTS.items():

    print("\n" + "=" * 76)
    print(source_name)
    print("=" * 76)

    source_entries = crawl_directory_tree(
        root_url,
        max_depth=2
    )

    for entry in source_entries:
        entry["source_name"] = source_name

    all_entries.extend(source_entries)

entries_df = pd.DataFrame(all_entries)

if entries_df.empty:
    raise RuntimeError(
        "No entries were found in the official RCV XML "
        "directories."
    )

# ------------------------------------------------------------
# Identify likely monthly RCV XML files
# ------------------------------------------------------------

def looks_like_rcv_xml_file(name):
    name_lower = str(name).lower()

    return (
        not name_lower.endswith("/")
        and (
            ".xml" in name_lower
            or ".gz" in name_lower
        )
        and (
            "rcv" in name_lower
            or "clinvarfullrelease" in name_lower
            or "clinvarrcvrelease" in name_lower
        )
        and not any(
            excluded in name_lower
            for excluded in [
                ".md5",
                ".sha",
                ".xsd",
                "readme"
            ]
        )
    )


file_candidates = entries_df.loc[
    entries_df["name"]
    .fillna("")
    .apply(looks_like_rcv_xml_file)
].copy()

# Find release-like date strings in the filename or listing.
date_pattern = re.compile(
    r"(20\d{2})[-_]?([01]\d)(?:[-_]?([0-3]\d))?"
)

def extract_date_tokens(row):
    searchable = (
        f"{row.get('name', '')} "
        f"{row.get('listing_text', '')} "
        f"{row.get('url', '')}"
    )

    matches = date_pattern.findall(searchable)

    return [
        {
            "year": match[0],
            "month": match[1],
            "day": match[2] or None
        }
        for match in matches
    ]


if not file_candidates.empty:
    file_candidates["date_tokens"] = (
        file_candidates.apply(
            extract_date_tokens,
            axis=1
        )
    )

# ------------------------------------------------------------
# Match candidates to T0 and T1
# ------------------------------------------------------------

matched_candidates = {
    "T0": [],
    "T1": []
}

for _, row in file_candidates.iterrows():

    row_record = row.to_dict()
    tokens = row_record.get("date_tokens", [])

    searchable_text = (
        f"{row_record.get('name', '')} "
        f"{row_record.get('listing_text', '')} "
        f"{row_record.get('directory_url', '')}"
    ).lower()

    for timepoint, target in TARGET_RELEASES.items():

        year_month_match = any(
            token["year"] == target["year"]
            and token["month"] == target["month"]
            for token in tokens
        )

        year_directory_match = (
            target["year"] in searchable_text
        )

        month_name_match = (
            f"{target['year']}-{target['month']}"
            in searchable_text
            or
            f"{target['year']}_{target['month']}"
            in searchable_text
        )

        if year_month_match or (
            year_directory_match
            and month_name_match
        ):
            matched_candidates[timepoint].append(
                row_record
            )

# ------------------------------------------------------------
# Save discovery manifest
# ------------------------------------------------------------

def clean_records(records):
    cleaned = []

    for record in records:

        cleaned_record = {}

        for key, value in record.items():

            if isinstance(value, float) and pd.isna(value):
                cleaned_record[key] = None
            elif isinstance(value, list):
                cleaned_record[key] = value
            else:
                cleaned_record[key] = str(value)

        cleaned.append(cleaned_record)

    return cleaned


discovery_manifest = {
    "study_id": "GES-RAG",
    "experiment": (
        "Experiment 1: Temporal Validation of GES"
    ),
    "discovery_manifest_version": "1.0.0",
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "purpose": (
        "Identify official condition-specific ClinVar RCV XML "
        "releases without downloading the datasets."
    ),
    "reason": (
        "variant_summary rows are variant-level and frequently "
        "contain multiple RCV accessions."
    ),
    "target_releases": TARGET_RELEASES,
    "directory_roots": DIRECTORY_ROOTS,
    "directories_and_files_examined": len(entries_df),
    "rcv_xml_file_candidates_found": len(
        file_candidates
    ),
    "matched_candidates": {
        timepoint: clean_records(records)
        for timepoint, records
        in matched_candidates.items()
    }
}

manifest_text = json.dumps(
    discovery_manifest,
    indent=2,
    ensure_ascii=False
)

manifest_filename = (
    "clinvar_rcv_xml_release_discovery_v1.json"
)

drive_manifest_path = (
    DRIVE_CONFIG_DIR /
    manifest_filename
)

repo_manifest_path = (
    REPO_CONFIG_DIR /
    manifest_filename
)

drive_manifest_path.write_text(
    manifest_text,
    encoding="utf-8"
)

repo_manifest_path.write_text(
    manifest_text,
    encoding="utf-8"
)

manifest_sha256 = hashlib.sha256(
    manifest_text.encode("utf-8")
).hexdigest()

checksum_filename = (
    "clinvar_rcv_xml_release_discovery_v1.sha256"
)

checksum_text = (
    f"{manifest_sha256}  {manifest_filename}\n"
)

(
    DRIVE_CONFIG_DIR /
    checksum_filename
).write_text(
    checksum_text,
    encoding="utf-8"
)

(
    REPO_CONFIG_DIR /
    checksum_filename
).write_text(
    checksum_text,
    encoding="utf-8"
)

# ------------------------------------------------------------
# Display discovery results
# ------------------------------------------------------------

print("\n" + "=" * 76)
print("RCV XML RELEASE DISCOVERY COMPLETED")
print("=" * 76)

print(
    "Directory entries examined:",
    f"{len(entries_df):,}"
)

print(
    "Potential RCV XML files found:",
    f"{len(file_candidates):,}"
)

for timepoint in ["T0", "T1"]:

    target = TARGET_RELEASES[timepoint]
    candidates = matched_candidates[timepoint]

    print("\n" + "-" * 76)

    print(
        f"{timepoint}: "
        f"{target['year']}-{target['month']}"
    )

    print(
        "Expected release date:",
        target["expected_release_date"]
    )

    print(
        "Candidate files found:",
        len(candidates)
    )

    if candidates:

        for candidate_number, candidate in enumerate(
            candidates,
            start=1
        ):

            print(
                f"\n  Candidate {candidate_number}"
            )

            print(
                "  Source format:",
                candidate.get("source_name")
            )

            print(
                "  Name:",
                candidate.get("name")
            )

            print(
                "  URL:",
                candidate.get("url")
            )

            print(
                "  Directory listing:",
                candidate.get("listing_text")
            )

    else:
        print(
            "  No automatically matched file was found."
        )

print("\nDiscovery manifest saved to Google Drive:")
print(drive_manifest_path)

print("\nDiscovery manifest saved to GitHub repository:")
print(repo_manifest_path)

print("\nDiscovery manifest SHA-256:")
print(manifest_sha256)

print("\nNo RCV XML dataset was downloaded.")


new_rcv_root
Reading directory (depth 0): https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/RCV_release/
Reading directory (depth 1): https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/
Reading directory (depth 1): https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/RCV_release/archive/
Reading directory (depth 2): https://ftp.ncbi.nlm.nih.gov/pub/clinvar/
Reading directory (depth 2): https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/archive/
Reading directory (depth 2): https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/sample_xml/
Reading directory (depth 2): https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/RCV_xml_old_format/
Reading directory (depth 2): https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/VCV_xml_old_format/
Reading directory (depth 2): https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/RCV_release/archive/2024/

new_rcv_archive
Reading directory (depth 0): https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/RCV_release/archive/
Reading directory (depth 1): https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/RCV_release/
Re

In [9]:
# ============================================================
# STEP 7: Verify RCV XML metadata, official MD5 checksums,
# and available storage
#
# No large XML file is downloaded in this step.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import requests
import shutil
import hashlib
import json
import re

# ------------------------------------------------------------
# Project locations
# ------------------------------------------------------------

REPO_DIR = Path(
    "/content/genomic-evidence-reliability"
)

PROJECT_DATA_DIR = Path(
    "/content/drive/MyDrive/GES_RAG_Temporal_Study"
)

DRIVE_CONFIG_DIR = (
    PROJECT_DATA_DIR /
    "configs"
)

REPO_CONFIG_DIR = (
    REPO_DIR /
    "configs"
)

DRIVE_CONFIG_DIR.mkdir(
    parents=True,
    exist_ok=True
)

REPO_CONFIG_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------
# Selected condition-specific RCV XML releases
# ------------------------------------------------------------

xml_releases = {
    "T0": {
        "snapshot_month": "2023-01",
        "format": "historical_RCV_XML",
        "filename": (
            "ClinVarFullRelease_2023-01.xml.gz"
        ),
        "url": (
            "https://ftp.ncbi.nlm.nih.gov/pub/clinvar/"
            "xml/RCV_xml_old_format/archive/2023/"
            "ClinVarFullRelease_2023-01.xml.gz"
        ),
        "md5_url": (
            "https://ftp.ncbi.nlm.nih.gov/pub/clinvar/"
            "xml/RCV_xml_old_format/archive/2023/"
            "ClinVarFullRelease_2023-01.xml.gz.md5"
        ),
        "expected_directory_size_bytes": 2510598470
    },

    "T1": {
        "snapshot_month": "2026-01",
        "format": "current_RCV_XML",
        "filename": (
            "ClinVarRCVRelease_2026-01.xml.gz"
        ),
        "url": (
            "https://ftp.ncbi.nlm.nih.gov/pub/clinvar/"
            "xml/RCV_release/"
            "ClinVarRCVRelease_2026-01.xml.gz"
        ),
        "md5_url": (
            "https://ftp.ncbi.nlm.nih.gov/pub/clinvar/"
            "xml/RCV_release/"
            "ClinVarRCVRelease_2026-01.xml.gz.md5"
        ),
        "expected_directory_size_bytes": 5434707247
    }
}

HEADERS = {
    "User-Agent": "GES-RAG-research/1.0"
}

GIB = 1024 ** 3

# ------------------------------------------------------------
# Helper: parse NCBI MD5 response
# ------------------------------------------------------------

def parse_md5_text(md5_text):
    """
    Extract a 32-character hexadecimal MD5 value from the
    official NCBI checksum file.
    """

    match = re.search(
        r"\b[a-fA-F0-9]{32}\b",
        md5_text
    )

    if not match:
        raise ValueError(
            "Could not identify an MD5 checksum in:\n"
            + md5_text
        )

    return match.group(0).lower()


# ------------------------------------------------------------
# Verify remote metadata without downloading XML content
# ------------------------------------------------------------

verification_results = {}

print("VERIFYING OFFICIAL RCV XML FILES")
print("=" * 76)

for timepoint, release in xml_releases.items():

    print(
        f"\n{timepoint}: "
        f"{release['snapshot_month']}"
    )

    print(
        "Format:",
        release["format"]
    )

    print(
        "Filename:",
        release["filename"]
    )

    # HEAD retrieves HTTP metadata only.
    head_response = requests.head(
        release["url"],
        allow_redirects=True,
        timeout=90,
        headers=HEADERS
    )

    head_response.raise_for_status()

    content_length_header = (
        head_response.headers.get(
            "Content-Length"
        )
    )

    remote_size_bytes = (
        int(content_length_header)
        if content_length_header
        else None
    )

    last_modified = (
        head_response.headers.get(
            "Last-Modified"
        )
    )

    etag = head_response.headers.get(
        "ETag"
    )

    # MD5 files are tiny text files.
    md5_response = requests.get(
        release["md5_url"],
        timeout=90,
        headers=HEADERS
    )

    md5_response.raise_for_status()

    official_md5_text = (
        md5_response.text.strip()
    )

    official_md5 = parse_md5_text(
        official_md5_text
    )

    expected_size = release[
        "expected_directory_size_bytes"
    ]

    size_matches_listing = (
        remote_size_bytes == expected_size
        if remote_size_bytes is not None
        else None
    )

    verification_results[timepoint] = {
        "snapshot_month": (
            release["snapshot_month"]
        ),
        "format": release["format"],
        "filename": release["filename"],
        "url": release["url"],
        "md5_url": release["md5_url"],
        "http_status": (
            head_response.status_code
        ),
        "remote_size_bytes": (
            remote_size_bytes
        ),
        "remote_size_gib": (
            round(
                remote_size_bytes / GIB,
                3
            )
            if remote_size_bytes is not None
            else None
        ),
        "expected_directory_size_bytes": (
            expected_size
        ),
        "size_matches_directory_listing": (
            size_matches_listing
        ),
        "last_modified_header": (
            last_modified
        ),
        "etag_header": etag,
        "official_md5": official_md5,
        "official_md5_file_text": (
            official_md5_text
        )
    }

    print(
        "HTTP status:",
        head_response.status_code
    )

    print(
        "Remote size:",
        (
            f"{remote_size_bytes:,} bytes "
            f"({remote_size_bytes / GIB:.3f} GiB)"
            if remote_size_bytes is not None
            else "Not supplied by server"
        )
    )

    print(
        "Matches directory listing:",
        size_matches_listing
    )

    print(
        "Last-Modified:",
        last_modified
    )

    print(
        "Official MD5:",
        official_md5
    )

# ------------------------------------------------------------
# Check temporary Colab and persistent Drive storage
# ------------------------------------------------------------

local_usage = shutil.disk_usage(
    "/content"
)

drive_usage = shutil.disk_usage(
    PROJECT_DATA_DIR
)

total_xml_bytes = sum(
    release[
        "expected_directory_size_bytes"
    ]
    for release in xml_releases.values()
)

largest_xml_bytes = max(
    release[
        "expected_directory_size_bytes"
    ]
    for release in xml_releases.values()
)

# Allow 2 GiB beyond the largest compressed XML for temporary
# extraction files and operational safety.
local_safety_requirement = (
    largest_xml_bytes +
    (2 * GIB)
)

drive_archive_requirement = (
    total_xml_bytes +
    (1 * GIB)
)

local_sequential_processing_feasible = (
    local_usage.free >=
    local_safety_requirement
)

drive_full_archive_feasible = (
    drive_usage.free >=
    drive_archive_requirement
)

if local_sequential_processing_feasible:

    selected_processing_strategy = (
        "Download one RCV XML release at a time to "
        "/content, verify MD5, stream-parse the four "
        "target genes, save compact extracts to Google "
        "Drive, and then delete the temporary raw XML."
    )

else:

    selected_processing_strategy = (
        "Stream the compressed XML directly from NCBI "
        "without retaining the complete raw XML locally. "
        "Save only the four-gene extracts and provenance."
    )

storage_assessment = {
    "local_colab": {
        "free_bytes": local_usage.free,
        "free_gib": round(
            local_usage.free / GIB,
            3
        ),
        "total_bytes": local_usage.total,
        "total_gib": round(
            local_usage.total / GIB,
            3
        )
    },

    "google_drive": {
        "free_bytes": drive_usage.free,
        "free_gib": round(
            drive_usage.free / GIB,
            3
        ),
        "total_bytes": drive_usage.total,
        "total_gib": round(
            drive_usage.total / GIB,
            3
        )
    },

    "xml_storage_requirements": {
        "T0_gib": round(
            xml_releases["T0"][
                "expected_directory_size_bytes"
            ] / GIB,
            3
        ),
        "T1_gib": round(
            xml_releases["T1"][
                "expected_directory_size_bytes"
            ] / GIB,
            3
        ),
        "both_files_gib": round(
            total_xml_bytes / GIB,
            3
        ),
        "largest_file_plus_safety_gib": round(
            local_safety_requirement / GIB,
            3
        )
    },

    "local_sequential_processing_feasible": (
        local_sequential_processing_feasible
    ),

    "drive_full_archive_feasible": (
        drive_full_archive_feasible
    ),

    "selected_processing_strategy": (
        selected_processing_strategy
    )
}

# ------------------------------------------------------------
# Save verification manifest
# ------------------------------------------------------------

verification_manifest = {
    "study_id": "GES-RAG",
    "experiment": (
        "Experiment 1: Temporal Validation of GES"
    ),
    "manifest_version": "1.0.0",
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "purpose": (
        "Verify selected historical and current RCV XML "
        "files, official MD5 checksums, and storage "
        "requirements before downloading large datasets."
    ),

    "files": verification_results,
    "storage_assessment": storage_assessment,

    "large_xml_downloaded": False,
    "record_content_examined": False
}

manifest_text = json.dumps(
    verification_manifest,
    indent=2,
    ensure_ascii=False
)

manifest_filename = (
    "clinvar_rcv_xml_verification_and_storage_v1.json"
)

drive_manifest_path = (
    DRIVE_CONFIG_DIR /
    manifest_filename
)

repo_manifest_path = (
    REPO_CONFIG_DIR /
    manifest_filename
)

drive_manifest_path.write_text(
    manifest_text,
    encoding="utf-8"
)

repo_manifest_path.write_text(
    manifest_text,
    encoding="utf-8"
)

manifest_sha256 = hashlib.sha256(
    manifest_text.encode("utf-8")
).hexdigest()

checksum_filename = (
    "clinvar_rcv_xml_verification_and_storage_v1.sha256"
)

checksum_text = (
    f"{manifest_sha256}  "
    f"{manifest_filename}\n"
)

(
    DRIVE_CONFIG_DIR /
    checksum_filename
).write_text(
    checksum_text,
    encoding="utf-8"
)

(
    REPO_CONFIG_DIR /
    checksum_filename
).write_text(
    checksum_text,
    encoding="utf-8"
)

# ------------------------------------------------------------
# Final output
# ------------------------------------------------------------

print("\n" + "=" * 76)
print("RCV XML VERIFICATION AND STORAGE CHECK COMPLETED")
print("=" * 76)

print(
    "\nTemporary Colab storage free:",
    f"{storage_assessment['local_colab']['free_gib']:.3f} GiB"
)

print(
    "Google Drive storage free:",
    f"{storage_assessment['google_drive']['free_gib']:.3f} GiB"
)

print(
    "Combined compressed XML size:",
    (
        f"{storage_assessment['xml_storage_requirements']"
        "['both_files_gib']:.3f} GiB"
    )
)

print(
    "\nCan Google Drive archive both XML files:",
    drive_full_archive_feasible
)

print(
    "Can Colab process one XML at a time:",
    local_sequential_processing_feasible
)

print("\nSELECTED PROCESSING STRATEGY")
print("-" * 76)
print(selected_processing_strategy)

print("\nVerification manifest saved to Google Drive:")
print(drive_manifest_path)

print("\nVerification manifest saved to GitHub:")
print(repo_manifest_path)

print("\nManifest SHA-256:")
print(manifest_sha256)

print("\nNo large XML file was downloaded.")
print("No ClinVar XML records were examined.")

SyntaxError: f-string: expecting '}' (2457452963.py, line 510)

In [10]:
# ============================================================
# STEP 7: Verify RCV XML files, official MD5 checksums,
# and available storage
#
# No large XML file is downloaded in this step.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import requests
import shutil
import hashlib
import json
import re

# ------------------------------------------------------------
# Project locations
# ------------------------------------------------------------

REPO_DIR = Path(
    "/content/genomic-evidence-reliability"
)

PROJECT_DATA_DIR = Path(
    "/content/drive/MyDrive/GES_RAG_Temporal_Study"
)

DRIVE_CONFIG_DIR = PROJECT_DATA_DIR / "configs"
REPO_CONFIG_DIR = REPO_DIR / "configs"

DRIVE_CONFIG_DIR.mkdir(
    parents=True,
    exist_ok=True
)

REPO_CONFIG_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------
# Selected RCV-level ClinVar XML releases
# ------------------------------------------------------------

xml_releases = {
    "T0": {
        "snapshot_month": "2023-01",
        "xml_format": "historical_RCV_XML",
        "filename": "ClinVarFullRelease_2023-01.xml.gz",
        "url": (
            "https://ftp.ncbi.nlm.nih.gov/pub/clinvar/"
            "xml/RCV_xml_old_format/archive/2023/"
            "ClinVarFullRelease_2023-01.xml.gz"
        ),
        "md5_url": (
            "https://ftp.ncbi.nlm.nih.gov/pub/clinvar/"
            "xml/RCV_xml_old_format/archive/2023/"
            "ClinVarFullRelease_2023-01.xml.gz.md5"
        )
    },

    "T1": {
        "snapshot_month": "2026-01",
        "xml_format": "current_RCV_XML",
        "filename": "ClinVarRCVRelease_2026-01.xml.gz",
        "url": (
            "https://ftp.ncbi.nlm.nih.gov/pub/clinvar/"
            "xml/RCV_release/"
            "ClinVarRCVRelease_2026-01.xml.gz"
        ),
        "md5_url": (
            "https://ftp.ncbi.nlm.nih.gov/pub/clinvar/"
            "xml/RCV_release/"
            "ClinVarRCVRelease_2026-01.xml.gz.md5"
        )
    }
}

HEADERS = {
    "User-Agent": "GES-RAG-research/1.0"
}

GIB = 1024 ** 3

# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------

def extract_md5(text):
    """
    Extract a 32-character MD5 checksum from an NCBI
    checksum-file response.
    """

    match = re.search(
        r"\b[a-fA-F0-9]{32}\b",
        text
    )

    if match is None:
        raise ValueError(
            "No valid MD5 checksum was found in:\n"
            + text
        )

    return match.group(0).lower()


def verify_remote_release(timepoint, release):
    """
    Verify HTTP availability, file size, modification date,
    and official MD5 without downloading the large XML file.
    """

    print("\n" + "-" * 76)
    print(
        f"{timepoint}: "
        f"{release['snapshot_month']}"
    )
    print("-" * 76)

    print("Format:", release["xml_format"])
    print("Filename:", release["filename"])

    head_response = requests.head(
        release["url"],
        allow_redirects=True,
        timeout=90,
        headers=HEADERS
    )

    head_response.raise_for_status()

    content_length = head_response.headers.get(
        "Content-Length"
    )

    remote_size_bytes = (
        int(content_length)
        if content_length is not None
        else None
    )

    last_modified = head_response.headers.get(
        "Last-Modified"
    )

    etag = head_response.headers.get(
        "ETag"
    )

    md5_response = requests.get(
        release["md5_url"],
        timeout=90,
        headers=HEADERS
    )

    md5_response.raise_for_status()

    official_md5_text = md5_response.text.strip()
    official_md5 = extract_md5(
        official_md5_text
    )

    print(
        "HTTP status:",
        head_response.status_code
    )

    if remote_size_bytes is not None:
        print(
            "Remote compressed size:",
            f"{remote_size_bytes:,} bytes "
            f"({remote_size_bytes / GIB:.3f} GiB)"
        )
    else:
        print(
            "Remote compressed size:",
            "Content-Length was not returned"
        )

    print(
        "Last-Modified:",
        last_modified
    )

    print(
        "Official MD5:",
        official_md5
    )

    return {
        "snapshot_month": release["snapshot_month"],
        "xml_format": release["xml_format"],
        "filename": release["filename"],
        "url": release["url"],
        "md5_url": release["md5_url"],
        "http_status": head_response.status_code,
        "remote_size_bytes": remote_size_bytes,
        "remote_size_gib": (
            round(
                remote_size_bytes / GIB,
                3
            )
            if remote_size_bytes is not None
            else None
        ),
        "last_modified_header": last_modified,
        "etag_header": etag,
        "official_md5": official_md5,
        "official_md5_file_text": official_md5_text
    }


# ------------------------------------------------------------
# Verify both official XML releases
# ------------------------------------------------------------

print("VERIFYING OFFICIAL RCV XML RELEASES")
print("=" * 76)

verification_results = {}

for timepoint, release in xml_releases.items():
    verification_results[timepoint] = (
        verify_remote_release(
            timepoint,
            release
        )
    )

# ------------------------------------------------------------
# Storage assessment
# ------------------------------------------------------------

local_usage = shutil.disk_usage(
    "/content"
)

drive_usage = shutil.disk_usage(
    PROJECT_DATA_DIR
)

known_file_sizes = [
    item["remote_size_bytes"]
    for item in verification_results.values()
    if item["remote_size_bytes"] is not None
]

if len(known_file_sizes) != len(xml_releases):
    raise RuntimeError(
        "A remote file size was not returned. "
        "Storage safety cannot yet be confirmed."
    )

total_xml_bytes = sum(
    known_file_sizes
)

largest_xml_bytes = max(
    known_file_sizes
)

# Require the largest compressed file plus 2 GiB of
# operational safety space.
local_required_bytes = (
    largest_xml_bytes +
    (2 * GIB)
)

# Require both compressed files plus 1 GiB if they were
# to be permanently archived in Drive.
drive_required_bytes = (
    total_xml_bytes +
    (1 * GIB)
)

local_sequential_processing_feasible = (
    local_usage.free >= local_required_bytes
)

drive_full_archive_feasible = (
    drive_usage.free >= drive_required_bytes
)

if local_sequential_processing_feasible:

    selected_strategy = (
        "Download one RCV XML release at a time into "
        "/content, verify its official MD5 checksum, "
        "stream-parse only BRCA1, BRCA2, MLH1, and EGFR, "
        "save the compact extraction to Google Drive, and "
        "delete the temporary raw XML before processing the "
        "next release."
    )

else:

    selected_strategy = (
        "Do not retain the complete XML locally. Stream the "
        "compressed XML directly from NCBI, extract only "
        "BRCA1, BRCA2, MLH1, and EGFR records, and save the "
        "compact extraction plus provenance."
    )

storage_assessment = {
    "local_colab": {
        "free_bytes": local_usage.free,
        "free_gib": round(
            local_usage.free / GIB,
            3
        ),
        "total_bytes": local_usage.total,
        "total_gib": round(
            local_usage.total / GIB,
            3
        )
    },

    "google_drive": {
        "free_bytes": drive_usage.free,
        "free_gib": round(
            drive_usage.free / GIB,
            3
        ),
        "total_bytes": drive_usage.total,
        "total_gib": round(
            drive_usage.total / GIB,
            3
        )
    },

    "xml_requirements": {
        "T0_gib": (
            verification_results["T0"][
                "remote_size_gib"
            ]
        ),
        "T1_gib": (
            verification_results["T1"][
                "remote_size_gib"
            ]
        ),
        "combined_gib": round(
            total_xml_bytes / GIB,
            3
        ),
        "largest_file_plus_safety_gib": round(
            local_required_bytes / GIB,
            3
        )
    },

    "local_sequential_processing_feasible": (
        local_sequential_processing_feasible
    ),

    "drive_full_archive_feasible": (
        drive_full_archive_feasible
    ),

    "selected_processing_strategy": (
        selected_strategy
    )
}

# ------------------------------------------------------------
# Save verification and storage manifest
# ------------------------------------------------------------

manifest = {
    "study_id": "GES-RAG",
    "experiment": (
        "Experiment 1: Temporal Validation of GES"
    ),
    "manifest_version": "1.0.1",
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "purpose": (
        "Verify the selected historical and current "
        "condition-specific ClinVar RCV XML releases and "
        "determine a storage-safe processing strategy."
    ),

    "files": verification_results,
    "storage_assessment": storage_assessment,

    "large_xml_downloaded": False,
    "xml_records_examined": False
}

manifest_text = json.dumps(
    manifest,
    indent=2,
    ensure_ascii=False
)

manifest_filename = (
    "clinvar_rcv_xml_verification_and_storage_v1.json"
)

drive_manifest_path = (
    DRIVE_CONFIG_DIR /
    manifest_filename
)

repo_manifest_path = (
    REPO_CONFIG_DIR /
    manifest_filename
)

drive_manifest_path.write_text(
    manifest_text,
    encoding="utf-8"
)

repo_manifest_path.write_text(
    manifest_text,
    encoding="utf-8"
)

manifest_sha256 = hashlib.sha256(
    manifest_text.encode("utf-8")
).hexdigest()

checksum_filename = (
    "clinvar_rcv_xml_verification_and_storage_v1.sha256"
)

checksum_text = (
    f"{manifest_sha256}  {manifest_filename}\n"
)

(
    DRIVE_CONFIG_DIR /
    checksum_filename
).write_text(
    checksum_text,
    encoding="utf-8"
)

(
    REPO_CONFIG_DIR /
    checksum_filename
).write_text(
    checksum_text,
    encoding="utf-8"
)

# ------------------------------------------------------------
# Final output
# ------------------------------------------------------------

combined_xml_gib = storage_assessment[
    "xml_requirements"
]["combined_gib"]

local_free_gib = storage_assessment[
    "local_colab"
]["free_gib"]

drive_free_gib = storage_assessment[
    "google_drive"
]["free_gib"]

print("\n" + "=" * 76)
print("RCV XML VERIFICATION AND STORAGE CHECK COMPLETED")
print("=" * 76)

print(
    "\nTemporary Colab storage free:",
    f"{local_free_gib:.3f} GiB"
)

print(
    "Google Drive storage free:",
    f"{drive_free_gib:.3f} GiB"
)

print(
    "Combined compressed XML size:",
    f"{combined_xml_gib:.3f} GiB"
)

print(
    "\nCan Google Drive archive both XML files:",
    drive_full_archive_feasible
)

print(
    "Can Colab process one XML at a time:",
    local_sequential_processing_feasible
)

print("\nSELECTED PROCESSING STRATEGY")
print("-" * 76)
print(selected_strategy)

print("\nManifest saved to Google Drive:")
print(drive_manifest_path)

print("\nManifest saved to GitHub repository:")
print(repo_manifest_path)

print("\nManifest SHA-256:")
print(manifest_sha256)

print("\nNo large XML file was downloaded.")
print("No ClinVar XML record was examined.")

VERIFYING OFFICIAL RCV XML RELEASES

----------------------------------------------------------------------------
T0: 2023-01
----------------------------------------------------------------------------
Format: historical_RCV_XML
Filename: ClinVarFullRelease_2023-01.xml.gz
HTTP status: 200
Remote compressed size: 2,510,598,470 bytes (2.338 GiB)
Last-Modified: Thu, 26 Mar 2026 22:35:14 GMT
Official MD5: d00be8862bbbd1d5a8b991c30246d224

----------------------------------------------------------------------------
T1: 2026-01
----------------------------------------------------------------------------
Format: current_RCV_XML
Filename: ClinVarRCVRelease_2026-01.xml.gz
HTTP status: 200
Remote compressed size: 5,434,707,247 bytes (5.061 GiB)
Last-Modified: Thu, 26 Mar 2026 19:43:29 GMT
Official MD5: 5740de7f8f74a49ba8c58e3ec1b8cc26

RCV XML VERIFICATION AND STORAGE CHECK COMPLETED

Temporary Colab storage free: 205.299 GiB
Google Drive storage free: 0.545 GiB
Combined compressed XML size: 7.

In [11]:
# ============================================================
# STEP 8: Download and verify the T0 condition-specific
# ClinVar RCV XML release
#
# The raw XML archive stays temporarily in /content.
# No XML records are parsed in this step.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import subprocess
import requests
import hashlib
import shutil
import json
import re

# ------------------------------------------------------------
# Project locations
# ------------------------------------------------------------

REPO_DIR = Path(
    "/content/genomic-evidence-reliability"
)

PROJECT_DATA_DIR = Path(
    "/content/drive/MyDrive/GES_RAG_Temporal_Study"
)

DRIVE_CONFIG_DIR = PROJECT_DATA_DIR / "configs"
REPO_CONFIG_DIR = REPO_DIR / "configs"

TEMP_RAW_DIR = Path(
    "/content/clinvar_rcv_raw"
)

DRIVE_CONFIG_DIR.mkdir(
    parents=True,
    exist_ok=True
)

REPO_CONFIG_DIR.mkdir(
    parents=True,
    exist_ok=True
)

TEMP_RAW_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------
# Official T0 RCV XML release
# ------------------------------------------------------------

T0_RELEASE = {
    "timepoint": "T0",
    "snapshot_month": "2023-01",
    "xml_format": "ClinVarFullRelease historical RCV XML",
    "filename": "ClinVarFullRelease_2023-01.xml.gz",
    "url": (
        "https://ftp.ncbi.nlm.nih.gov/pub/clinvar/"
        "xml/RCV_xml_old_format/archive/2023/"
        "ClinVarFullRelease_2023-01.xml.gz"
    ),
    "md5_url": (
        "https://ftp.ncbi.nlm.nih.gov/pub/clinvar/"
        "xml/RCV_xml_old_format/archive/2023/"
        "ClinVarFullRelease_2023-01.xml.gz.md5"
    )
}

T0_LOCAL_PATH = (
    TEMP_RAW_DIR /
    T0_RELEASE["filename"]
)

HEADERS = {
    "User-Agent": "GES-RAG-research/1.0"
}

GIB = 1024 ** 3

# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------

def extract_md5(text):
    """
    Extract a 32-character hexadecimal MD5 checksum.
    """

    match = re.search(
        r"\b[a-fA-F0-9]{32}\b",
        text
    )

    if match is None:
        raise ValueError(
            "A valid MD5 checksum was not found in:\n"
            + text
        )

    return match.group(0).lower()


def calculate_md5(
    file_path,
    chunk_size=16 * 1024 * 1024
):
    """
    Calculate MD5 without loading the complete file into RAM.
    """

    digest = hashlib.md5()

    with open(file_path, "rb") as file_handle:

        while True:

            chunk = file_handle.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def calculate_sha256(
    file_path,
    chunk_size=16 * 1024 * 1024
):
    """
    Calculate SHA-256 for an additional local provenance hash.
    """

    digest = hashlib.sha256()

    with open(file_path, "rb") as file_handle:

        while True:

            chunk = file_handle.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


# ------------------------------------------------------------
# Obtain current official metadata
# ------------------------------------------------------------

print("VERIFYING CURRENT OFFICIAL T0 METADATA")
print("=" * 76)

head_response = requests.head(
    T0_RELEASE["url"],
    allow_redirects=True,
    timeout=90,
    headers=HEADERS
)

head_response.raise_for_status()

content_length = head_response.headers.get(
    "Content-Length"
)

if content_length is None:
    raise RuntimeError(
        "NCBI did not return a Content-Length header."
    )

official_size_bytes = int(content_length)

official_last_modified = (
    head_response.headers.get("Last-Modified")
)

official_etag = head_response.headers.get(
    "ETag"
)

md5_response = requests.get(
    T0_RELEASE["md5_url"],
    timeout=90,
    headers=HEADERS
)

md5_response.raise_for_status()

official_md5_file_text = (
    md5_response.text.strip()
)

official_md5 = extract_md5(
    official_md5_file_text
)

print(
    "Official compressed size:",
    f"{official_size_bytes:,} bytes "
    f"({official_size_bytes / GIB:.3f} GiB)"
)

print(
    "Official MD5:",
    official_md5
)

print(
    "Server Last-Modified:",
    official_last_modified
)

# ------------------------------------------------------------
# Confirm sufficient temporary storage
# ------------------------------------------------------------

local_usage = shutil.disk_usage(
    TEMP_RAW_DIR
)

required_free_bytes = (
    official_size_bytes +
    (2 * GIB)
)

print(
    "\nTemporary storage free:",
    f"{local_usage.free / GIB:.3f} GiB"
)

print(
    "Required with safety allowance:",
    f"{required_free_bytes / GIB:.3f} GiB"
)

if local_usage.free < required_free_bytes:
    raise OSError(
        "Insufficient temporary Colab storage for the "
        "T0 RCV XML archive."
    )

# ------------------------------------------------------------
# Download or resume the T0 archive
# ------------------------------------------------------------

print("\n" + "=" * 76)
print("DOWNLOADING T0 RCV XML")
print("=" * 76)

if T0_LOCAL_PATH.exists():

    existing_size = T0_LOCAL_PATH.stat().st_size

    if existing_size == official_size_bytes:
        print(
            "A complete-size local file already exists. "
            "The download will be skipped and the file "
            "will be verified."
        )

    elif existing_size < official_size_bytes:
        print(
            "A partial local file exists."
        )

        print(
            "Current partial size:",
            f"{existing_size:,} bytes"
        )

        print(
            "The download will resume."
        )

    else:
        print(
            "The existing file is larger than the official "
            "file and will be removed."
        )

        T0_LOCAL_PATH.unlink()

if (
    not T0_LOCAL_PATH.exists()
    or T0_LOCAL_PATH.stat().st_size
    != official_size_bytes
):

    download_command = [
        "wget",
        "--continue",
        "--tries=10",
        "--timeout=60",
        "--read-timeout=300",
        "--retry-connrefused",
        "--progress=bar:force:noscroll",
        "--output-document",
        str(T0_LOCAL_PATH),
        T0_RELEASE["url"]
    ]

    subprocess.run(
        download_command,
        check=True
    )

# ------------------------------------------------------------
# Verify exact downloaded size
# ------------------------------------------------------------

observed_size_bytes = (
    T0_LOCAL_PATH.stat().st_size
)

print("\nVERIFYING DOWNLOADED FILE")
print("=" * 76)

print(
    "Observed compressed size:",
    f"{observed_size_bytes:,} bytes "
    f"({observed_size_bytes / GIB:.3f} GiB)"
)

if observed_size_bytes != official_size_bytes:
    raise IOError(
        "Downloaded-size mismatch. "
        f"Expected {official_size_bytes:,} bytes but found "
        f"{observed_size_bytes:,} bytes."
    )

print("Compressed size: PASSED")

# ------------------------------------------------------------
# Verify official MD5
# ------------------------------------------------------------

print("\nCalculating local MD5...")

observed_md5 = calculate_md5(
    T0_LOCAL_PATH
)

print(
    "Observed MD5:",
    observed_md5
)

if observed_md5 != official_md5:
    raise IOError(
        "MD5 verification failed. The downloaded file does "
        "not match the current official NCBI checksum."
    )

print("Official MD5 verification: PASSED")

# ------------------------------------------------------------
# Calculate additional SHA-256
# ------------------------------------------------------------

print("\nCalculating local SHA-256...")

observed_sha256 = calculate_sha256(
    T0_LOCAL_PATH
)

print(
    "Observed SHA-256:",
    observed_sha256
)

# ------------------------------------------------------------
# Save download receipt
# ------------------------------------------------------------

receipt = {
    "study_id": "GES-RAG",
    "experiment": (
        "Experiment 1: Temporal Validation of GES"
    ),
    "receipt_version": "1.0.0",
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "timepoint": "T0",
    "snapshot_month": T0_RELEASE[
        "snapshot_month"
    ],
    "xml_format": T0_RELEASE[
        "xml_format"
    ],

    "official_source": {
        "filename": T0_RELEASE["filename"],
        "url": T0_RELEASE["url"],
        "md5_url": T0_RELEASE["md5_url"],
        "server_last_modified": (
            official_last_modified
        ),
        "etag": official_etag,
        "official_size_bytes": (
            official_size_bytes
        ),
        "official_md5": official_md5,
        "official_md5_file_text": (
            official_md5_file_text
        )
    },

    "local_verification": {
        "temporary_local_path": str(
            T0_LOCAL_PATH
        ),
        "observed_size_bytes": (
            observed_size_bytes
        ),
        "observed_md5": observed_md5,
        "observed_sha256": observed_sha256,
        "size_verification": "passed",
        "md5_verification": "passed"
    },

    "storage_policy": {
        "raw_xml_saved_to_google_drive": False,
        "raw_xml_saved_to_github": False,
        "raw_xml_location": (
            "Temporary Colab storage only"
        ),
        "planned_action": (
            "Stream-parse the four target genes and delete "
            "the raw XML after the compact extraction has "
            "been verified."
        )
    },

    "xml_records_examined": False
}

receipt_text = json.dumps(
    receipt,
    indent=2,
    ensure_ascii=False
)

receipt_filename = (
    "clinvar_t0_rcv_xml_download_receipt_v1.json"
)

drive_receipt_path = (
    DRIVE_CONFIG_DIR /
    receipt_filename
)

repo_receipt_path = (
    REPO_CONFIG_DIR /
    receipt_filename
)

drive_receipt_path.write_text(
    receipt_text,
    encoding="utf-8"
)

repo_receipt_path.write_text(
    receipt_text,
    encoding="utf-8"
)

receipt_sha256 = hashlib.sha256(
    receipt_text.encode("utf-8")
).hexdigest()

checksum_filename = (
    "clinvar_t0_rcv_xml_download_receipt_v1.sha256"
)

checksum_text = (
    f"{receipt_sha256}  {receipt_filename}\n"
)

(
    DRIVE_CONFIG_DIR /
    checksum_filename
).write_text(
    checksum_text,
    encoding="utf-8"
)

(
    REPO_CONFIG_DIR /
    checksum_filename
).write_text(
    checksum_text,
    encoding="utf-8"
)

# ------------------------------------------------------------
# Final confirmation
# ------------------------------------------------------------

print("\n" + "=" * 76)
print("T0 RCV XML DOWNLOAD AND VERIFICATION COMPLETED")
print("=" * 76)

print(
    "Temporary raw file:",
    T0_LOCAL_PATH
)

print(
    "Verified size:",
    f"{observed_size_bytes:,} bytes"
)

print(
    "Official MD5:",
    observed_md5
)

print(
    "Local SHA-256:",
    observed_sha256
)

print(
    "\nDownload receipt saved to Google Drive:"
)

print(drive_receipt_path)

print(
    "\nDownload receipt saved to GitHub repository:"
)

print(repo_receipt_path)

print(
    "\nDownload receipt SHA-256:",
    receipt_sha256
)

print(
    "\nThe T0 XML archive is verified but has not "
    "yet been parsed."
)

VERIFYING CURRENT OFFICIAL T0 METADATA
Official compressed size: 2,510,598,470 bytes (2.338 GiB)
Official MD5: d00be8862bbbd1d5a8b991c30246d224
Server Last-Modified: Thu, 26 Mar 2026 22:35:14 GMT

Temporary storage free: 205.299 GiB
Required with safety allowance: 4.338 GiB

DOWNLOADING T0 RCV XML

VERIFYING DOWNLOADED FILE
Observed compressed size: 2,510,598,470 bytes (2.338 GiB)
Compressed size: PASSED

Calculating local MD5...
Observed MD5: d00be8862bbbd1d5a8b991c30246d224
Official MD5 verification: PASSED

Calculating local SHA-256...
Observed SHA-256: 911c8a58872ea89cc7bb4f1ee3362596d965f5103422b30f456abaf99c41e5e7

T0 RCV XML DOWNLOAD AND VERIFICATION COMPLETED
Temporary raw file: /content/clinvar_rcv_raw/ClinVarFullRelease_2023-01.xml.gz
Verified size: 2,510,598,470 bytes
Official MD5: d00be8862bbbd1d5a8b991c30246d224
Local SHA-256: 911c8a58872ea89cc7bb4f1ee3362596d965f5103422b30f456abaf99c41e5e7

Download receipt saved to Google Drive:
/content/drive/MyDrive/GES_RAG_Temporal_St

In [12]:
# ============================================================
# STEP 9: Inspect embedded T0 release metadata and schema
#
# Purpose:
# 1. Confirm the date embedded inside the historical XML.
# 2. Verify the top-level record structure.
# 3. Locate one BRCA1/BRCA2/MLH1/EGFR example record.
#
# No temporal outcomes or GES scores are created.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from collections import Counter, defaultdict
from lxml import etree
import gzip
import hashlib
import json
import re

# ------------------------------------------------------------
# Project locations
# ------------------------------------------------------------

REPO_DIR = Path(
    "/content/genomic-evidence-reliability"
)

PROJECT_DATA_DIR = Path(
    "/content/drive/MyDrive/GES_RAG_Temporal_Study"
)

DRIVE_CONFIG_DIR = PROJECT_DATA_DIR / "configs"
REPO_CONFIG_DIR = REPO_DIR / "configs"

T0_XML_PATH = Path(
    "/content/clinvar_rcv_raw/"
    "ClinVarFullRelease_2023-01.xml.gz"
)

DRIVE_CONFIG_DIR.mkdir(
    parents=True,
    exist_ok=True
)

REPO_CONFIG_DIR.mkdir(
    parents=True,
    exist_ok=True
)

if not T0_XML_PATH.exists():
    raise FileNotFoundError(
        "The verified T0 XML file was not found:\n"
        f"{T0_XML_PATH}\n"
        "Run Step 8 again."
    )

# ------------------------------------------------------------
# Prespecified study genes
# ------------------------------------------------------------

TARGET_GENES = {
    "BRCA1",
    "BRCA2",
    "MLH1",
    "EGFR"
}

MAX_RECORDS_TO_SEARCH = 250_000

# ------------------------------------------------------------
# XML helper functions
# ------------------------------------------------------------

def local_name(element_or_tag):
    """
    Return an XML tag name without its namespace.
    """

    if hasattr(element_or_tag, "tag"):
        tag = element_or_tag.tag
    else:
        tag = element_or_tag

    if not isinstance(tag, str):
        return ""

    return etree.QName(tag).localname


def clean_text(value):
    """
    Normalize whitespace in XML text.
    """

    if value is None:
        return None

    value = re.sub(
        r"\s+",
        " ",
        str(value)
    ).strip()

    return value or None


def element_path(element):
    """
    Construct a namespace-free path for schema inspection.
    """

    names = []
    current = element

    while current is not None:

        name = local_name(current)

        if name:
            names.append(name)

        current = current.getparent()

    return "/".join(reversed(names))


def extract_rcv_accessions(record):
    """
    Extract RCV accessions from ClinVarAccession elements.
    """

    accessions = []

    for element in record.iter():

        if local_name(element) != "ClinVarAccession":
            continue

        accession = (
            element.get("Acc")
            or element.get("Accession")
        )

        accession = clean_text(accession)

        if accession and accession.upper().startswith("RCV"):
            accessions.append(
                accession.upper()
            )

    return sorted(set(accessions))


def extract_target_genes(record):
    """
    Identify exact target-gene symbols appearing in element
    text or XML attribute values.
    """

    found_genes = set()
    gene_paths = []

    for element in record.iter():

        text = clean_text(element.text)

        if text and text.upper() in TARGET_GENES:

            gene = text.upper()
            found_genes.add(gene)

            gene_paths.append({
                "gene": gene,
                "source": "element_text",
                "element": local_name(element),
                "path": element_path(element),
                "attributes": {
                    str(key): str(value)
                    for key, value
                    in element.attrib.items()
                }
            })

        for attribute_name, attribute_value in (
            element.attrib.items()
        ):

            normalized_value = clean_text(
                attribute_value
            )

            if (
                normalized_value
                and normalized_value.upper()
                in TARGET_GENES
            ):

                gene = normalized_value.upper()
                found_genes.add(gene)

                gene_paths.append({
                    "gene": gene,
                    "source": (
                        f"attribute:{attribute_name}"
                    ),
                    "element": local_name(element),
                    "path": element_path(element),
                    "attributes": {
                        str(key): str(value)
                        for key, value
                        in element.attrib.items()
                    }
                })

    return sorted(found_genes), gene_paths


def collect_section_values(record, section_name):
    """
    Collect structured text values under a named XML section.
    """

    results = []

    for section in record.iter():

        if local_name(section) != section_name:
            continue

        section_result = {
            "section_attributes": {
                str(key): str(value)
                for key, value
                in section.attrib.items()
            },
            "values": []
        }

        for descendant in section.iter():

            value = clean_text(descendant.text)

            if value is None:
                continue

            section_result["values"].append({
                "element": local_name(descendant),
                "value": value,
                "attributes": {
                    str(key): str(attribute_value)
                    for key, attribute_value
                    in descendant.attrib.items()
                }
            })

        # Avoid excessively large diagnostic output.
        section_result["values"] = (
            section_result["values"][:50]
        )

        results.append(section_result)

    return results[:10]


def extract_measure_identifiers(record):
    """
    Collect identifiers from MeasureSet and Measure elements.
    These will later help determine the precise VariationID
    linkage rule.
    """

    identifiers = []

    for element in record.iter():

        name = local_name(element)

        if name not in {
            "MeasureSet",
            "Measure",
            "VariationArchive"
        }:
            continue

        if not element.attrib:
            continue

        identifiers.append({
            "element": name,
            "path": element_path(element),
            "attributes": {
                str(key): str(value)
                for key, value
                in element.attrib.items()
            }
        })

    return identifiers[:50]


def summarize_record(record):
    """
    Produce a compact structural summary of one ClinVarSet.
    """

    genes, gene_paths = extract_target_genes(
        record
    )

    tag_counts = Counter()
    attribute_names = defaultdict(set)

    for element in record.iter():

        name = local_name(element)

        if not name:
            continue

        tag_counts[name] += 1

        for attribute_name in element.attrib:
            attribute_names[name].add(
                str(attribute_name)
            )

    return {
        "record_tag": local_name(record),

        "record_attributes": {
            str(key): str(value)
            for key, value
            in record.attrib.items()
        },

        "rcv_accessions": extract_rcv_accessions(
            record
        ),

        "target_genes": genes,

        "target_gene_locations": gene_paths[:20],

        "measure_identifiers": (
            extract_measure_identifiers(record)
        ),

        "clinical_significance_sections": (
            collect_section_values(
                record,
                "ClinicalSignificance"
            )
        ),

        "trait_sections": (
            collect_section_values(
                record,
                "TraitSet"
            )
        ),

        "tag_counts": dict(
            sorted(tag_counts.items())
        ),

        "attribute_names_by_element": {
            name: sorted(values)
            for name, values
            in sorted(attribute_names.items())
        }
    }

# ------------------------------------------------------------
# Stream-parse the historical XML
# ------------------------------------------------------------

print("INSPECTING T0 HISTORICAL RCV XML")
print("=" * 76)

root_tag = None
root_attributes = {}
root_namespace_map = {}

top_level_record_tags = Counter()
first_record_summary = None
first_target_record_summary = None

records_examined = 0
target_record_position = None

with gzip.open(
    T0_XML_PATH,
    mode="rb"
) as compressed_stream:

    context = etree.iterparse(
        compressed_stream,
        events=("start", "end"),
        huge_tree=True,
        recover=False
    )

    for event, element in context:

        # The first start event corresponds to the XML root.
        if event == "start" and root_tag is None:

            root_tag = local_name(element)

            root_attributes = {
                str(key): str(value)
                for key, value
                in element.attrib.items()
            }

            root_namespace_map = {
                str(key): str(value)
                for key, value
                in (element.nsmap or {}).items()
            }

            print("Root element:", root_tag)
            print("Root attributes:", root_attributes)
            print("Root namespaces:", root_namespace_map)

            continue

        if event != "end":
            continue

        # The historical RCV release uses ClinVarSet records.
        if local_name(element) != "ClinVarSet":
            continue

        records_examined += 1
        top_level_record_tags[
            local_name(element)
        ] += 1

        if first_record_summary is None:
            first_record_summary = summarize_record(
                element
            )

        genes, _ = extract_target_genes(
            element
        )

        if genes and first_target_record_summary is None:

            first_target_record_summary = (
                summarize_record(element)
            )

            target_record_position = (
                records_examined
            )

            print(
                "\nFirst target-gene record located at "
                f"record {records_examined:,}"
            )

            print(
                "Target gene(s):",
                first_target_record_summary[
                    "target_genes"
                ]
            )

            print(
                "RCV accession(s):",
                first_target_record_summary[
                    "rcv_accessions"
                ]
            )

        # Clear processed records to keep memory usage stable.
        element.clear()

        parent = element.getparent()

        if parent is not None:

            while element.getprevious() is not None:
                del parent[0]

        # Stop after finding one target record. If none appears,
        # stop at the prespecified diagnostic limit.
        if first_target_record_summary is not None:
            break

        if records_examined >= MAX_RECORDS_TO_SEARCH:
            break

del context

# ------------------------------------------------------------
# Interpret embedded release metadata
# ------------------------------------------------------------

root_date_candidates = {}

for key, value in root_attributes.items():

    if any(
        token in key.lower()
        for token in [
            "date",
            "dated",
            "release",
            "version"
        ]
    ):
        root_date_candidates[key] = value

embedded_2023_indicator = any(
    "2023" in str(value)
    for value in root_attributes.values()
)

if embedded_2023_indicator:

    provenance_interpretation = (
        "The XML root contains a 2023 indicator, consistent "
        "with the named January 2023 snapshot."
    )

    historical_snapshot_check = "passed"

else:

    provenance_interpretation = (
        "No 2023 indicator was found in the root attributes. "
        "The file should not yet be treated as a confirmed "
        "historical snapshot until additional embedded "
        "metadata is inspected."
    )

    historical_snapshot_check = (
        "requires_additional_review"
    )

# ------------------------------------------------------------
# Save schema and provenance probe
# ------------------------------------------------------------

probe = {
    "study_id": "GES-RAG",

    "experiment": (
        "Experiment 1: Temporal Validation of GES"
    ),

    "probe_version": "1.0.0",

    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "source_file": {
        "path": str(T0_XML_PATH),
        "filename": T0_XML_PATH.name,
        "expected_snapshot": "2023-01",
        "previously_verified_sha256": (
            "911c8a58872ea89cc7bb4f1ee3362596d"
            "965f5103422b30f456abaf99c41e5e7"
        )
    },

    "root_metadata": {
        "root_tag": root_tag,
        "root_attributes": root_attributes,
        "root_namespace_map": root_namespace_map,
        "date_or_version_candidates": (
            root_date_candidates
        )
    },

    "schema_probe": {
        "records_examined": records_examined,
        "top_level_record_tags": dict(
            top_level_record_tags
        ),
        "target_record_found": (
            first_target_record_summary
            is not None
        ),
        "target_record_position": (
            target_record_position
        ),
        "first_record_summary": (
            first_record_summary
        ),
        "first_target_record_summary": (
            first_target_record_summary
        )
    },

    "provenance_interpretation": {
        "historical_snapshot_check": (
            historical_snapshot_check
        ),
        "interpretation": (
            provenance_interpretation
        ),
        "http_last_modified_used_as_snapshot_date": (
            False
        )
    },

    "temporal_outcomes_created": False,
    "ges_features_created": False,
    "model_training_performed": False
}

probe_text = json.dumps(
    probe,
    indent=2,
    ensure_ascii=False
)

probe_filename = (
    "clinvar_t0_historical_xml_schema_probe_v1.json"
)

drive_probe_path = (
    DRIVE_CONFIG_DIR /
    probe_filename
)

repo_probe_path = (
    REPO_CONFIG_DIR /
    probe_filename
)

drive_probe_path.write_text(
    probe_text,
    encoding="utf-8"
)

repo_probe_path.write_text(
    probe_text,
    encoding="utf-8"
)

probe_sha256 = hashlib.sha256(
    probe_text.encode("utf-8")
).hexdigest()

checksum_filename = (
    "clinvar_t0_historical_xml_schema_probe_v1.sha256"
)

checksum_text = (
    f"{probe_sha256}  {probe_filename}\n"
)

(
    DRIVE_CONFIG_DIR /
    checksum_filename
).write_text(
    checksum_text,
    encoding="utf-8"
)

(
    REPO_CONFIG_DIR /
    checksum_filename
).write_text(
    checksum_text,
    encoding="utf-8"
)

# ------------------------------------------------------------
# Final output
# ------------------------------------------------------------

print("\n" + "=" * 76)
print(
    "T0 HISTORICAL XML PROVENANCE AND "
    "SCHEMA PROBE COMPLETED"
)
print("=" * 76)

print(
    "Root element:",
    root_tag
)

print(
    "Root attributes:",
    root_attributes
)

print(
    "Embedded date/version candidates:",
    root_date_candidates
)

print(
    "Records examined:",
    f"{records_examined:,}"
)

print(
    "Target-gene record found:",
    first_target_record_summary is not None
)

if first_target_record_summary is not None:

    print(
        "Target record position:",
        f"{target_record_position:,}"
    )

    print(
        "Target genes:",
        first_target_record_summary[
            "target_genes"
        ]
    )

    print(
        "RCV accessions:",
        first_target_record_summary[
            "rcv_accessions"
        ]
    )

    print(
        "Measure identifiers:",
        first_target_record_summary[
            "measure_identifiers"
        ]
    )

print(
    "\nHistorical snapshot check:",
    historical_snapshot_check.upper()
)

print(
    "Interpretation:",
    provenance_interpretation
)

print("\nProbe saved to Google Drive:")
print(drive_probe_path)

print("\nProbe saved to GitHub repository:")
print(repo_probe_path)

print("\nProbe SHA-256:")
print(probe_sha256)

print(
    "\nNo temporal outcome, GES feature, "
    "or model score was created."
)

INSPECTING T0 HISTORICAL RCV XML
Root element: ReleaseSet
Root attributes: {'Dated': '2022-12-31', 'Type': 'full', '{http://www.w3.org/2001/XMLSchema-instance}noNamespaceSchemaLocation': 'http://ftp.ncbi.nlm.nih.gov/pub/clinvar/xsd_public/clinvar_public_1.69.xsd'}
Root namespaces: {'xsi': 'http://www.w3.org/2001/XMLSchema-instance'}

First target-gene record located at record 10,264
Target gene(s): ['EGFR']
RCV accession(s): ['RCV000052656']

T0 HISTORICAL XML PROVENANCE AND SCHEMA PROBE COMPLETED
Root element: ReleaseSet
Root attributes: {'Dated': '2022-12-31', 'Type': 'full', '{http://www.w3.org/2001/XMLSchema-instance}noNamespaceSchemaLocation': 'http://ftp.ncbi.nlm.nih.gov/pub/clinvar/xsd_public/clinvar_public_1.69.xsd'}
Embedded date/version candidates: {'Dated': '2022-12-31'}
Records examined: 10,264
Target-gene record found: True
Target record position: 10,264
Target genes: ['EGFR']
RCV accessions: ['RCV000052656']
Measure identifiers: [{'element': 'MeasureSet', 'path': 'Release

In [13]:
# ============================================================
# STEP 10: Create Protocol Version 1.1
#
# Purpose:
# Distinguish the ClinVar archive/release date from the
# embedded data-cutoff date.
#
# Protocol Version 1.0 is preserved and not overwritten.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from copy import deepcopy
import hashlib
import json

# ------------------------------------------------------------
# Project locations
# ------------------------------------------------------------

REPO_DIR = Path(
    "/content/genomic-evidence-reliability"
)

PROJECT_DATA_DIR = Path(
    "/content/drive/MyDrive/GES_RAG_Temporal_Study"
)

REPO_CONFIG_DIR = REPO_DIR / "configs"
DRIVE_CONFIG_DIR = PROJECT_DATA_DIR / "configs"

REPO_CONFIG_DIR.mkdir(
    parents=True,
    exist_ok=True
)

DRIVE_CONFIG_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------
# Original frozen protocol
# ------------------------------------------------------------

V1_PROTOCOL_PATH = (
    REPO_CONFIG_DIR /
    "temporal_validation_protocol_v1.json"
)

EXPECTED_V1_SHA256 = (
    "60ea29236abddec303b624f85b9a47ad7"
    "accf5b0eff47295c772147b9c5f1c43"
)

if not V1_PROTOCOL_PATH.exists():
    raise FileNotFoundError(
        "Original Protocol Version 1 was not found:\n"
        f"{V1_PROTOCOL_PATH}"
    )

v1_protocol_text = V1_PROTOCOL_PATH.read_text(
    encoding="utf-8"
)

observed_v1_sha256 = hashlib.sha256(
    v1_protocol_text.encode("utf-8")
).hexdigest()

print("VERIFYING ORIGINAL FROZEN PROTOCOL")
print("=" * 76)

print(
    "Expected Version 1 SHA-256:",
    EXPECTED_V1_SHA256
)

print(
    "Observed Version 1 SHA-256:",
    observed_v1_sha256
)

if observed_v1_sha256 != EXPECTED_V1_SHA256:
    raise RuntimeError(
        "Protocol Version 1 checksum mismatch. "
        "The original frozen protocol may have changed."
    )

print("Original Protocol Version 1 integrity: PASSED")

v1_protocol = json.loads(
    v1_protocol_text
)

# ------------------------------------------------------------
# Create amended protocol without altering Version 1
# ------------------------------------------------------------

protocol_v1_1 = deepcopy(
    v1_protocol
)

protocol_v1_1["protocol_version"] = "1.1.0"

protocol_v1_1["protocol_amended_utc"] = (
    datetime.now(timezone.utc).isoformat()
)

protocol_v1_1["supersedes_protocol_version"] = (
    "1.0.0"
)

protocol_v1_1["superseded_protocol_sha256"] = (
    observed_v1_sha256
)

# ------------------------------------------------------------
# Clarify temporal-date definitions
# ------------------------------------------------------------

release_design = protocol_v1_1[
    "release_design"
]

# Preserve the previously expected value for auditability.
release_design[
    "original_baseline_expected_date"
] = release_design.get(
    "baseline_expected_date"
)

release_design[
    "baseline_release_label"
] = "2023-01"

release_design[
    "baseline_archive_publication_date"
] = "2023-01-05"

release_design[
    "baseline_embedded_data_cutoff_date"
] = "2022-12-31"

release_design[
    "baseline_source_filename"
] = "ClinVarFullRelease_2023-01.xml.gz"

release_design[
    "baseline_source_format"
] = "historical ClinVar RCV XML"

release_design[
    "baseline_xml_root_element"
] = "ReleaseSet"

release_design[
    "baseline_xml_root_dated_attribute"
] = "2022-12-31"

release_design[
    "baseline_xml_schema"
] = "clinvar_public_1.69.xsd"

release_design[
    "baseline_date_status"
] = "confirmed_from_embedded_XML_metadata"

release_design[
    "baseline_date_interpretation"
] = (
    "T0 is the January 2023 archived ClinVar release. "
    "The release was published on January 5, 2023, while "
    "the XML root Dated attribute indicates that its data "
    "snapshot is current through December 31, 2022. "
    "All T0 feature calculations will therefore use "
    "December 31, 2022 as the temporal data cutoff."
)

# Keep T1 unresolved until its XML root is inspected.
release_design[
    "followup_release_label"
] = "2026-01"

release_design[
    "followup_archive_publication_date"
] = "2026-01-01"

release_design[
    "followup_embedded_data_cutoff_date"
] = None

release_design[
    "followup_date_status"
] = (
    "pending_embedded_XML_verification"
)

release_design[
    "date_definitions"
] = {
    "release_label": (
        "The year-month label used in the archived "
        "ClinVar filename."
    ),
    "archive_publication_date": (
        "The date on which the monthly archived release "
        "was published."
    ),
    "embedded_data_cutoff_date": (
        "The Dated value embedded in the XML root and the "
        "date used as the temporal snapshot boundary."
    ),
    "http_last_modified": (
        "File-server maintenance metadata only; it will "
        "not be used as the ClinVar data snapshot date."
    )
}

release_design[
    "temporal_interval_rule"
] = (
    "The T0-to-T1 observation interval will be calculated "
    "using embedded XML data-cutoff dates, not HTTP "
    "Last-Modified dates."
)

release_design[
    "release_date_status"
] = (
    "T0 confirmed; T1 pending embedded XML inspection"
)

# ------------------------------------------------------------
# Record formal amendment history
# ------------------------------------------------------------

protocol_v1_1["amendment_history"] = [
    {
        "amendment_id": "A001",
        "protocol_version": "1.1.0",
        "amendment_type": (
            "provenance clarification before outcome analysis"
        ),
        "reason": (
            "The January 2023 historical RCV XML contains "
            "ReleaseSet/@Dated='2022-12-31'. The protocol "
            "must distinguish release publication from the "
            "embedded data-cutoff date."
        ),
        "changes": [
            (
                "Confirmed T0 as the January 2023 archived "
                "RCV release."
            ),
            (
                "Recorded January 5, 2023 as the archive "
                "publication date."
            ),
            (
                "Recorded December 31, 2022 as the embedded "
                "T0 data-cutoff date."
            ),
            (
                "Specified that HTTP Last-Modified metadata "
                "will not define the temporal snapshot."
            ),
            (
                "Left the T1 embedded cutoff pending until "
                "the T1 XML is inspected."
            )
        ],
        "impact_on_hypotheses": "none",
        "impact_on_primary_outcome": "none",
        "impact_on_model_specification": "none",
        "outcome_data_examined_before_amendment": False,
        "ges_model_fitted_before_amendment": False,
        "amended_utc": protocol_v1_1[
            "protocol_amended_utc"
        ]
    }
]

# ------------------------------------------------------------
# Save Version 1.1
# ------------------------------------------------------------

v1_1_filename = (
    "temporal_validation_protocol_v1_1.json"
)

repo_v1_1_path = (
    REPO_CONFIG_DIR /
    v1_1_filename
)

drive_v1_1_path = (
    DRIVE_CONFIG_DIR /
    v1_1_filename
)

v1_1_text = json.dumps(
    protocol_v1_1,
    indent=2,
    ensure_ascii=False
)

repo_v1_1_path.write_text(
    v1_1_text,
    encoding="utf-8"
)

drive_v1_1_path.write_text(
    v1_1_text,
    encoding="utf-8"
)

# ------------------------------------------------------------
# Save Version 1.1 checksum
# ------------------------------------------------------------

v1_1_sha256 = hashlib.sha256(
    v1_1_text.encode("utf-8")
).hexdigest()

checksum_filename = (
    "temporal_validation_protocol_v1_1.sha256"
)

checksum_text = (
    f"{v1_1_sha256}  {v1_1_filename}\n"
)

(
    REPO_CONFIG_DIR /
    checksum_filename
).write_text(
    checksum_text,
    encoding="utf-8"
)

(
    DRIVE_CONFIG_DIR /
    checksum_filename
).write_text(
    checksum_text,
    encoding="utf-8"
)

# ------------------------------------------------------------
# Save a standalone amendment record
# ------------------------------------------------------------

amendment_record = {
    "study_id": "GES-RAG",
    "amendment_id": "A001",
    "original_protocol_version": "1.0.0",
    "amended_protocol_version": "1.1.0",
    "original_protocol_sha256": observed_v1_sha256,
    "amended_protocol_sha256": v1_1_sha256,
    "original_protocol_preserved": True,
    "outcome_data_examined_before_amendment": False,
    "ges_model_fitted_before_amendment": False,
    "t0_release_label": "2023-01",
    "t0_archive_publication_date": "2023-01-05",
    "t0_embedded_data_cutoff_date": "2022-12-31",
    "reason": (
        "Clarify the distinction between archive publication "
        "date and the embedded XML data-cutoff date."
    ),
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat()
}

amendment_filename = (
    "protocol_amendment_A001.json"
)

amendment_text = json.dumps(
    amendment_record,
    indent=2,
    ensure_ascii=False
)

(
    REPO_CONFIG_DIR /
    amendment_filename
).write_text(
    amendment_text,
    encoding="utf-8"
)

(
    DRIVE_CONFIG_DIR /
    amendment_filename
).write_text(
    amendment_text,
    encoding="utf-8"
)

# ------------------------------------------------------------
# Final output
# ------------------------------------------------------------

print("\n" + "=" * 76)
print("PROTOCOL VERSION 1.1 CREATED SUCCESSFULLY")
print("=" * 76)

print(
    "Original protocol preserved:",
    True
)

print(
    "Original protocol version:",
    v1_protocol["protocol_version"]
)

print(
    "Original protocol SHA-256:",
    observed_v1_sha256
)

print(
    "\nAmended protocol version:",
    protocol_v1_1["protocol_version"]
)

print(
    "T0 release label:",
    release_design["baseline_release_label"]
)

print(
    "T0 archive publication date:",
    release_design[
        "baseline_archive_publication_date"
    ]
)

print(
    "T0 embedded data cutoff:",
    release_design[
        "baseline_embedded_data_cutoff_date"
    ]
)

print(
    "T1 embedded data cutoff:",
    release_design[
        "followup_embedded_data_cutoff_date"
    ]
)

print(
    "\nAmended protocol saved to GitHub repository:"
)

print(repo_v1_1_path)

print(
    "\nAmended protocol saved to Google Drive:"
)

print(drive_v1_1_path)

print(
    "\nProtocol Version 1.1 SHA-256:",
    v1_1_sha256
)

print(
    "\nNo temporal outcome was examined and no GES model "
    "was fitted before this amendment."
)

VERIFYING ORIGINAL FROZEN PROTOCOL
Expected Version 1 SHA-256: 60ea29236abddec303b624f85b9a47ad7accf5b0eff47295c772147b9c5f1c43
Observed Version 1 SHA-256: 60ea29236abddec303b624f85b9a47ad7accf5b0eff47295c772147b9c5f1c43
Original Protocol Version 1 integrity: PASSED

PROTOCOL VERSION 1.1 CREATED SUCCESSFULLY
Original protocol preserved: True
Original protocol version: 1.0.0
Original protocol SHA-256: 60ea29236abddec303b624f85b9a47ad7accf5b0eff47295c772147b9c5f1c43

Amended protocol version: 1.1.0
T0 release label: 2023-01
T0 archive publication date: 2023-01-05
T0 embedded data cutoff: 2022-12-31
T1 embedded data cutoff: None

Amended protocol saved to GitHub repository:
/content/genomic-evidence-reliability/configs/temporal_validation_protocol_v1_1.json

Amended protocol saved to Google Drive:
/content/drive/MyDrive/GES_RAG_Temporal_Study/configs/temporal_validation_protocol_v1_1.json

Protocol Version 1.1 SHA-256: 7db783abbcf920127675ba9965bc3c387b5e5cb81f5c80eec276f4bfd48b1482

No t

In [15]:
# ============================================================
# STEP 11: Stream-extract T0 RCV records for the four genes
#
# Input:
#   ClinVarFullRelease_2023-01.xml.gz
#
# Output:
#   One compact Parquet row per RCV variant-condition record
#
# No T1 information, temporal outcome, or GES model is used.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
from lxml import etree
import pyarrow as pa
import pyarrow.parquet as pq
import pandas as pd
import gzip
import hashlib
import json
import re
import shutil

# ------------------------------------------------------------
# Project locations
# ------------------------------------------------------------

REPO_DIR = Path(
    "/content/genomic-evidence-reliability"
)

PROJECT_DATA_DIR = Path(
    "/content/drive/MyDrive/GES_RAG_Temporal_Study"
)

T0_XML_PATH = Path(
    "/content/clinvar_rcv_raw/"
    "ClinVarFullRelease_2023-01.xml.gz"
)

LOCAL_OUTPUT_DIR = Path(
    "/content/ges_t0_extraction"
)

DRIVE_OUTPUT_DIR = (
    PROJECT_DATA_DIR /
    "data_interim"
)

DRIVE_CONFIG_DIR = (
    PROJECT_DATA_DIR /
    "configs"
)

REPO_CONFIG_DIR = (
    REPO_DIR /
    "configs"
)

for directory in [
    LOCAL_OUTPUT_DIR,
    DRIVE_OUTPUT_DIR,
    DRIVE_CONFIG_DIR,
    REPO_CONFIG_DIR
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )

if not T0_XML_PATH.exists():
    raise FileNotFoundError(
        "Verified T0 XML archive not found:\n"
        f"{T0_XML_PATH}"
    )

# ------------------------------------------------------------
# Frozen study settings
# ------------------------------------------------------------

TARGET_GENES = {
    "BRCA1",
    "BRCA2",
    "MLH1",
    "EGFR"
}

PRIMARY_GENES = {
    "BRCA1",
    "BRCA2",
    "MLH1"
}

EXPLORATORY_GENES = {
    "EGFR"
}

T0_RELEASE_LABEL = "2023-01"
T0_ARCHIVE_PUBLICATION_DATE = "2023-01-05"
T0_DATA_CUTOFF_DATE = "2022-12-31"

SOURCE_SHA256 = (
    "911c8a58872ea89cc7bb4f1ee3362596d"
    "965f5103422b30f456abaf99c41e5e7"
)

BATCH_SIZE = 5_000
PROGRESS_INTERVAL = 100_000

LOCAL_PARQUET_PATH = (
    LOCAL_OUTPUT_DIR /
    "t0_rcv_target_genes_raw.parquet"
)

DRIVE_PARQUET_PATH = (
    DRIVE_OUTPUT_DIR /
    "t0_rcv_target_genes_raw.parquet"
)

# Remove an incomplete local output from a prior failed run.
if LOCAL_PARQUET_PATH.exists():
    LOCAL_PARQUET_PATH.unlink()

# ------------------------------------------------------------
# Utility functions
# ------------------------------------------------------------

def local_name(element_or_tag):
    """Return an XML tag without its namespace."""

    tag = (
        element_or_tag.tag
        if hasattr(element_or_tag, "tag")
        else element_or_tag
    )

    if not isinstance(tag, str):
        return ""

    return etree.QName(tag).localname


def clean_text(value):
    """Normalize whitespace and return None for blank values."""

    if value is None:
        return None

    normalized = re.sub(
        r"\s+",
        " ",
        str(value)
    ).strip()

    return normalized or None


def json_text(value):
    """Serialize nested structures consistently."""

    return json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":")
    )


def first_direct_child(parent, child_name):
    """Return the first direct child having a local tag name."""

    for child in parent:

        if local_name(child) == child_name:
            return child

    return None


def descendants(parent, tag_name):
    """Yield descendants with a given local XML tag."""

    for element in parent.iter():

        if local_name(element) == tag_name:
            yield element


def first_descendant_text(parent, tag_name):
    """Return the first nonblank text under a named tag."""

    if parent is None:
        return None

    for element in descendants(
        parent,
        tag_name
    ):

        value = clean_text(element.text)

        if value is not None:
            return value

    return None


def extract_accession(
    parent,
    prefix
):
    """
    Extract one accession and version from ClinVarAccession
    elements under the supplied parent.
    """

    if parent is None:
        return None, None

    prefix = prefix.upper()

    for element in descendants(
        parent,
        "ClinVarAccession"
    ):

        accession = clean_text(
            element.get("Acc")
            or element.get("Accession")
        )

        if (
            accession
            and accession.upper().startswith(prefix)
        ):

            version = clean_text(
                element.get("Version")
            )

            return accession.upper(), version

    return None, None


def extract_target_genes(measure_set):
    """
    Extract exact target-gene symbols from the reference
    MeasureSet.
    """

    if measure_set is None:
        return []

    genes = set()

    for element in measure_set.iter():

        text = clean_text(element.text)

        if (
            text
            and text.upper() in TARGET_GENES
        ):
            genes.add(text.upper())

        for attribute_value in element.attrib.values():

            value = clean_text(
                attribute_value
            )

            if (
                value
                and value.upper() in TARGET_GENES
            ):
                genes.add(value.upper())

    return sorted(genes)


def extract_trait_information(reference_assertion):
    """
    Extract condition names and identifiers from the
    reference assertion's TraitSet.
    """

    trait_names = set()
    condition_ids = set()
    trait_records = []

    if reference_assertion is None:
        return [], [], []

    trait_set = first_direct_child(
        reference_assertion,
        "TraitSet"
    )

    if trait_set is None:
        return [], [], []

    for trait in trait_set:

        if local_name(trait) != "Trait":
            continue

        current_names = set()
        current_ids = set()

        for element in trait.iter():

            tag = local_name(element)

            if tag == "ElementValue":

                value = clean_text(
                    element.text
                )

                value_type = clean_text(
                    element.get("Type")
                )

                if value and (
                    value_type is None
                    or value_type.lower()
                    in {
                        "preferred",
                        "alternate"
                    }
                ):
                    current_names.add(value)

            elif tag == "XRef":

                database = clean_text(
                    element.get("DB")
                )

                identifier = clean_text(
                    element.get("ID")
                )

                if database and identifier:
                    current_ids.add(
                        f"{database}:{identifier}"
                    )

        trait_names.update(
            current_names
        )

        condition_ids.update(
            current_ids
        )

        trait_records.append({
            "names": sorted(current_names),
            "identifiers": sorted(current_ids)
        })

    return (
        sorted(trait_names),
        sorted(condition_ids),
        trait_records
    )


def extract_clinical_significance(parent):
    """
    Extract classification, review status, and last-evaluated
    date from a reference assertion or SCV assertion.
    """

    clinical_significance = (
        first_direct_child(
            parent,
            "ClinicalSignificance"
        )
        if parent is not None
        else None
    )

    if clinical_significance is None:
        return {
            "classification": None,
            "review_status": None,
            "last_evaluated": None,
            "explanation": None
        }

    classification = first_descendant_text(
        clinical_significance,
        "Description"
    )

    review_status = first_descendant_text(
        clinical_significance,
        "ReviewStatus"
    )

    last_evaluated = clean_text(
        clinical_significance.get(
            "DateLastEvaluated"
        )
    )

    if last_evaluated is None:
        last_evaluated = (
            first_descendant_text(
                clinical_significance,
                "DateLastEvaluated"
            )
        )

    explanation = first_descendant_text(
        clinical_significance,
        "Explanation"
    )

    return {
        "classification": classification,
        "review_status": review_status,
        "last_evaluated": last_evaluated,
        "explanation": explanation
    }


def review_stars(review_status):
    """
    Convert standard ClinVar review-status wording to stars.
    The raw status is also retained.
    """

    if not review_status:
        return None

    status = review_status.lower().strip()

    if "practice guideline" in status:
        return 4

    if "expert panel" in status:
        return 3

    if (
        "multiple submitters" in status
        and (
            "no conflict" in status
            or "no conflicts" in status
        )
    ):
        return 2

    if "criteria provided" in status:
        return 1

    return 0


def normalize_classification_group(
    classification
):
    """
    Map a raw germline classification to a broad clinical
    group without discarding the original value.
    """

    if not classification:
        return "Missing"

    value = classification.lower()

    if "conflict" in value:
        return "Conflicting"

    has_pathogenic = (
        "pathogenic" in value
    )

    has_benign = (
        "benign" in value
    )

    has_vus = (
        "uncertain significance" in value
        or value.strip() == "vus"
    )

    group_count = sum([
        has_pathogenic,
        has_benign,
        has_vus
    ])

    if group_count > 1:
        return "Mixed"

    if has_pathogenic:
        return "Pathogenic/Likely pathogenic"

    if has_benign:
        return "Benign/Likely benign"

    if has_vus:
        return "VUS"

    return "Other"


def extract_submitter(assertion):
    """
    Extract submitter metadata from a ClinVarAssertion.
    """

    for element in descendants(
        assertion,
        "ClinVarSubmissionID"
    ):

        submitter = clean_text(
            element.get("submitter")
            or element.get("Submitter")
        )

        submitter_id = clean_text(
            element.get("submitterID")
            or element.get("SubmitterID")
        )

        return submitter, submitter_id

    return None, None


def extract_origins(assertion):
    """Extract reported observation origins from an SCV."""

    origins = set()

    for element in descendants(
        assertion,
        "Origin"
    ):

        value = clean_text(
            element.text
        )

        if value:
            origins.add(value)

    return sorted(origins)


def extract_assertion_method(assertion):
    """
    Extract assertion-method descriptions where available.
    """

    values = set()

    for element in assertion.iter():

        tag = local_name(element)

        if tag not in {
            "Method",
            "MethodType",
            "Description"
        }:
            continue

        # Restrict generic Description tags to Method ancestry.
        ancestor_names = {
            local_name(ancestor)
            for ancestor in element.iterancestors()
        }

        if (
            tag == "Description"
            and "Method" not in ancestor_names
        ):
            continue

        value = clean_text(
            element.text
        )

        if value:
            values.add(value)

    return sorted(values)


def extract_scv_records(clinvar_set):
    """
    Extract compact metadata for each submitted SCV assertion.
    """

    scv_records = []

    for child in clinvar_set:

        if local_name(child) != "ClinVarAssertion":
            continue

        scv_accession, scv_version = (
            extract_accession(
                child,
                "SCV"
            )
        )

        significance = (
            extract_clinical_significance(
                child
            )
        )

        submitter, submitter_id = (
            extract_submitter(
                child
            )
        )

        scv_records.append({
            "scv_accession": scv_accession,
            "scv_version": scv_version,
            "classification": (
                significance["classification"]
            ),
            "classification_group": (
                normalize_classification_group(
                    significance[
                        "classification"
                    ]
                )
            ),
            "review_status": (
                significance["review_status"]
            ),
            "last_evaluated": (
                significance["last_evaluated"]
            ),
            "submitter": submitter,
            "submitter_id": submitter_id,
            "origins": extract_origins(
                child
            ),
            "assertion_methods": (
                extract_assertion_method(
                    child
                )
            )
        })

    return scv_records


def classification_counts(
    scv_records,
    field_name
):
    """Count SCV classification values or groups."""

    counts = Counter()

    for record in scv_records:

        value = record.get(
            field_name
        )

        if value:
            counts[value] += 1

    return dict(
        sorted(counts.items())
    )


def detect_scv_group_disagreement(
    scv_records
):
    """
    Detect disagreement among the three primary clinical
    classification groups.
    """

    primary_groups = {
        record[
            "classification_group"
        ]
        for record in scv_records
        if record[
            "classification_group"
        ] in {
            "Pathogenic/Likely pathogenic",
            "Benign/Likely benign",
            "VUS"
        }
    }

    return len(primary_groups) > 1


def derive_study_scope(genes):
    """Identify primary, exploratory, or mixed-gene scope."""

    genes = set(genes)

    has_primary = bool(
        genes.intersection(
            PRIMARY_GENES
        )
    )

    has_exploratory = bool(
        genes.intersection(
            EXPLORATORY_GENES
        )
    )

    if has_primary and has_exploratory:
        return "mixed"

    if has_primary:
        return "primary"

    if has_exploratory:
        return "exploratory"

    return "outside_scope"


def parse_one_rcv_record(
    clinvar_set,
    record_index
):
    """Convert one target ClinVarSet into one output row."""

    reference_assertion = (
        first_direct_child(
            clinvar_set,
            "ReferenceClinVarAssertion"
        )
    )

    if reference_assertion is None:
        return None

    measure_set = first_direct_child(
        reference_assertion,
        "MeasureSet"
    )

    genes = extract_target_genes(
        measure_set
    )

    if not genes:
        return None

    rcv_accession, rcv_version = (
        extract_accession(
            reference_assertion,
            "RCV"
        )
    )

    variation_id = (
        clean_text(
            measure_set.get("ID")
        )
        if measure_set is not None
        else None
    )

    vcv_accession = (
        clean_text(
            measure_set.get("Acc")
        )
        if measure_set is not None
        else None
    )

    vcv_version = (
        clean_text(
            measure_set.get("Version")
        )
        if measure_set is not None
        else None
    )

    measure_set_type = (
        clean_text(
            measure_set.get("Type")
        )
        if measure_set is not None
        else None
    )

    measure_types = sorted({
        clean_text(
            element.get("Type")
        )
        for element in (
            measure_set.iter()
            if measure_set is not None
            else []
        )
        if (
            local_name(element) == "Measure"
            and clean_text(
                element.get("Type")
            )
        )
    })

    (
        condition_names,
        condition_ids,
        trait_records
    ) = extract_trait_information(
        reference_assertion
    )

    aggregate = (
        extract_clinical_significance(
            reference_assertion
        )
    )

    scv_records = extract_scv_records(
        clinvar_set
    )

    submitters = sorted({
        record["submitter"]
        for record in scv_records
        if record["submitter"]
    })

    submitter_ids = sorted({
        record["submitter_id"]
        for record in scv_records
        if record["submitter_id"]
    })

    aggregate_review_status = (
        aggregate["review_status"]
    )

    aggregate_classification = (
        aggregate["classification"]
    )

    aggregate_conflict_flag = bool(
        (
            aggregate_review_status
            and "conflict"
            in aggregate_review_status.lower()
        )
        or
        (
            aggregate_classification
            and "conflict"
            in aggregate_classification.lower()
        )
    )

    scv_disagreement_flag = (
        detect_scv_group_disagreement(
            scv_records
        )
    )

    return {
        "timepoint": "T0",
        "release_label": T0_RELEASE_LABEL,
        "archive_publication_date": (
            T0_ARCHIVE_PUBLICATION_DATE
        ),
        "embedded_data_cutoff_date": (
            T0_DATA_CUTOFF_DATE
        ),
        "xml_record_index": record_index,

        "rcv_accession": rcv_accession,
        "rcv_version": rcv_version,
        "variation_id": variation_id,
        "vcv_accession": vcv_accession,
        "vcv_version": vcv_version,

        "target_genes_json": json_text(
            genes
        ),
        "study_scope": derive_study_scope(
            genes
        ),

        "measure_set_type": measure_set_type,
        "measure_types_json": json_text(
            measure_types
        ),

        "condition_names_json": json_text(
            condition_names
        ),
        "condition_ids_json": json_text(
            condition_ids
        ),
        "trait_records_json": json_text(
            trait_records
        ),

        "aggregate_classification": (
            aggregate_classification
        ),
        "aggregate_classification_group": (
            normalize_classification_group(
                aggregate_classification
            )
        ),
        "aggregate_review_status": (
            aggregate_review_status
        ),
        "aggregate_review_stars": (
            review_stars(
                aggregate_review_status
            )
        ),
        "aggregate_last_evaluated": (
            aggregate["last_evaluated"]
        ),
        "aggregate_explanation": (
            aggregate["explanation"]
        ),

        "aggregate_conflict_flag": (
            aggregate_conflict_flag
        ),
        "scv_group_disagreement_flag": (
            scv_disagreement_flag
        ),

        "scv_count_xml": len(
            scv_records
        ),
        "unique_submitter_count_xml": len(
            submitters
        ),
        "submitters_json": json_text(
            submitters
        ),
        "submitter_ids_json": json_text(
            submitter_ids
        ),

        "scv_classification_counts_json": (
            json_text(
                classification_counts(
                    scv_records,
                    "classification"
                )
            )
        ),

        "scv_group_counts_json": (
            json_text(
                classification_counts(
                    scv_records,
                    "classification_group"
                )
            )
        ),

        "scv_records_json": json_text(
            scv_records
        ),

        "source_filename": T0_XML_PATH.name,
        "source_sha256": SOURCE_SHA256
    }


def sha256_file(
    file_path,
    chunk_size=16 * 1024 * 1024
):
    """Calculate SHA-256 for an output file."""

    digest = hashlib.sha256()

    with open(file_path, "rb") as handle:

        while True:

            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


# ------------------------------------------------------------
# Stream parser and Parquet writer
# ------------------------------------------------------------

print("EXTRACTING T0 RCV RECORDS")
print("=" * 76)

records_scanned = 0
records_extracted = 0
records_missing_rcv = 0
records_missing_variation_id = 0
records_with_multiple_target_genes = 0

gene_counts = Counter()
scope_counts = Counter()
classification_counts_aggregate = Counter()
review_status_counts = Counter()

output_buffer = []
parquet_writer = None
output_schema = None

started_utc = datetime.now(
    timezone.utc
)

with gzip.open(
    T0_XML_PATH,
    mode="rb"
) as compressed_stream:

    context = etree.iterparse(
        compressed_stream,
        events=("end",),
        tag="ClinVarSet",
        huge_tree=True,
        recover=False
    )

    for _, clinvar_set in context:

        records_scanned += 1

        output_row = parse_one_rcv_record(
            clinvar_set,
            records_scanned
        )

        if output_row is not None:

            records_extracted += 1

            genes = json.loads(
                output_row[
                    "target_genes_json"
                ]
            )

            for gene in genes:
                gene_counts[gene] += 1

            scope_counts[
                output_row["study_scope"]
            ] += 1

            classification_counts_aggregate[
                output_row[
                    "aggregate_classification_group"
                ]
            ] += 1

            review_status_counts[
                output_row[
                    "aggregate_review_status"
                ]
                or "Missing"
            ] += 1

            if not output_row[
                "rcv_accession"
            ]:
                records_missing_rcv += 1

            if not output_row[
                "variation_id"
            ]:
                records_missing_variation_id += 1

            if len(genes) > 1:
                records_with_multiple_target_genes += 1

            output_buffer.append(
                output_row
            )

        # Write each completed output batch.
        if len(output_buffer) >= BATCH_SIZE:

            batch_df = pd.DataFrame(
                output_buffer
            )

            batch_table = pa.Table.from_pandas(
                batch_df,
                preserve_index=False,
                schema=output_schema
            )

            if parquet_writer is None:

                output_schema = batch_table.schema

                parquet_writer = pq.ParquetWriter(
                    LOCAL_PARQUET_PATH,
                    output_schema,
                    compression="zstd",
                    use_dictionary=True
                )

            parquet_writer.write_table(
                batch_table
            )

            output_buffer.clear()

        if (
            records_scanned %
            PROGRESS_INTERVAL == 0
        ):

            print(
                f"Records scanned: "
                f"{records_scanned:,} | "
                f"Target RCVs extracted: "
                f"{records_extracted:,}"
            )

        # Release parsed XML memory.
        clinvar_set.clear()

        parent = clinvar_set.getparent()

        if parent is not None:

            while (
                clinvar_set.getprevious()
                is not None
            ):
                del parent[0]

del context

# Write the final incomplete batch.
if output_buffer:

    final_df = pd.DataFrame(
        output_buffer
    )

    final_table = pa.Table.from_pandas(
        final_df,
        preserve_index=False,
        schema=output_schema
    )

    if parquet_writer is None:

        output_schema = final_table.schema

        parquet_writer = pq.ParquetWriter(
            LOCAL_PARQUET_PATH,
            output_schema,
            compression="zstd",
            use_dictionary=True
        )

    parquet_writer.write_table(
        final_table
    )

    output_buffer.clear()

if parquet_writer is not None:
    parquet_writer.close()

finished_utc = datetime.now(
    timezone.utc
)

if records_extracted == 0:
    raise RuntimeError(
        "No target-gene RCV records were extracted."
    )

if not LOCAL_PARQUET_PATH.exists():
    raise RuntimeError(
        "The local Parquet output was not created."
    )

# ------------------------------------------------------------
# Validate local Parquet metadata
# ------------------------------------------------------------

parquet_metadata = pq.read_metadata(
    LOCAL_PARQUET_PATH
)

parquet_row_count = (
    parquet_metadata.num_rows
)

if parquet_row_count != records_extracted:
    raise RuntimeError(
        "Parquet row-count mismatch. "
        f"Expected {records_extracted:,}, "
        f"found {parquet_row_count:,}."
    )

local_output_bytes = (
    LOCAL_PARQUET_PATH.stat().st_size
)

local_output_sha256 = sha256_file(
    LOCAL_PARQUET_PATH
)

print("\nLOCAL EXTRACTION COMPLETED")
print("=" * 76)

print(
    "Complete XML records scanned:",
    f"{records_scanned:,}"
)

print(
    "Target RCV records extracted:",
    f"{records_extracted:,}"
)

print(
    "Local Parquet rows:",
    f"{parquet_row_count:,}"
)

print(
    "Local Parquet size:",
    f"{local_output_bytes / (1024**2):,.2f} MB"
)

print(
    "Local Parquet SHA-256:",
    local_output_sha256
)

# ------------------------------------------------------------
# Confirm Drive capacity and copy compact extraction
# ------------------------------------------------------------

drive_usage = shutil.disk_usage(
    DRIVE_OUTPUT_DIR
)

# Preserve at least 100 MiB of visible free Drive space.
required_drive_bytes = (
    local_output_bytes
    + 100 * 1024 * 1024
)

if drive_usage.free < required_drive_bytes:
    raise OSError(
        "The compact extraction was created locally, but "
        "Google Drive does not have enough free space to "
        "copy it safely.\n"
        f"Local output: {LOCAL_PARQUET_PATH}"
    )

shutil.copy2(
    LOCAL_PARQUET_PATH,
    DRIVE_PARQUET_PATH
)

drive_output_sha256 = sha256_file(
    DRIVE_PARQUET_PATH
)

if drive_output_sha256 != local_output_sha256:
    raise IOError(
        "Google Drive copy checksum does not match the "
        "verified local extraction."
    )

# ------------------------------------------------------------
# Save extraction manifest
# ------------------------------------------------------------

manifest = {
    "study_id": "GES-RAG",
    "experiment": (
        "Experiment 1: Temporal Validation of GES"
    ),
    "extraction_version": "1.0.0",
    "started_utc": started_utc.isoformat(),
    "finished_utc": finished_utc.isoformat(),

    "protocol": {
        "version": "1.1.0",
        "sha256": (
            "7db783abbcf920127675ba9965bc3c387"
            "b5e5cb81f5c80eec276f4bfd48b1482"
        )
    },

    "source": {
        "filename": T0_XML_PATH.name,
        "temporary_path": str(
            T0_XML_PATH
        ),
        "release_label": (
            T0_RELEASE_LABEL
        ),
        "archive_publication_date": (
            T0_ARCHIVE_PUBLICATION_DATE
        ),
        "embedded_data_cutoff_date": (
            T0_DATA_CUTOFF_DATE
        ),
        "sha256": SOURCE_SHA256,
        "xml_format": (
            "historical ClinVar RCV XML"
        ),
        "xml_schema": (
            "clinvar_public_1.69.xsd"
        )
    },

    "filter": {
        "target_genes": sorted(
            TARGET_GENES
        ),
        "primary_genes": sorted(
            PRIMARY_GENES
        ),
        "exploratory_genes": sorted(
            EXPLORATORY_GENES
        ),
        "unit_of_analysis": (
            "RCV variant-condition aggregate"
        )
    },

    "counts": {
        "complete_xml_records_scanned": (
            records_scanned
        ),
        "target_rcv_records_extracted": (
            records_extracted
        ),
        "parquet_row_count": (
            parquet_row_count
        ),
        "gene_counts": dict(
            sorted(gene_counts.items())
        ),
        "scope_counts": dict(
            sorted(scope_counts.items())
        ),
        "aggregate_classification_group_counts": (
            dict(
                sorted(
                    classification_counts_aggregate.items()
                )
            )
        ),
        "aggregate_review_status_counts": (
            dict(
                sorted(
                    review_status_counts.items()
                )
            )
        ),
        "records_missing_rcv": (
            records_missing_rcv
        ),
        "records_missing_variation_id": (
            records_missing_variation_id
        ),
        "records_with_multiple_target_genes": (
            records_with_multiple_target_genes
        )
    },

    "output": {
        "local_path": str(
            LOCAL_PARQUET_PATH
        ),
        "google_drive_path": str(
            DRIVE_PARQUET_PATH
        ),
        "format": "Parquet",
        "compression": "Zstandard",
        "bytes": local_output_bytes,
        "megabytes": round(
            local_output_bytes /
            (1024 ** 2),
            3
        ),
        "sha256": (
            local_output_sha256
        )
    },

    "leakage_controls": {
        "t1_information_used": False,
        "temporal_outcome_created": False,
        "ges_model_fitted": False,
        "ges_threshold_selected": False
    },

    "raw_xml_deleted": False
}

manifest_text = json.dumps(
    manifest,
    indent=2,
    ensure_ascii=False
)

manifest_filename = (
    "clinvar_t0_target_gene_rcv_extraction_manifest_v1.json"
)

drive_manifest_path = (
    DRIVE_CONFIG_DIR /
    manifest_filename
)

repo_manifest_path = (
    REPO_CONFIG_DIR /
    manifest_filename
)

drive_manifest_path.write_text(
    manifest_text,
    encoding="utf-8"
)

repo_manifest_path.write_text(
    manifest_text,
    encoding="utf-8"
)

manifest_sha256 = hashlib.sha256(
    manifest_text.encode("utf-8")
).hexdigest()

checksum_filename = (
    "clinvar_t0_target_gene_rcv_extraction_manifest_v1.sha256"
)

checksum_text = (
    f"{manifest_sha256}  {manifest_filename}\n"
)

(
    DRIVE_CONFIG_DIR /
    checksum_filename
).write_text(
    checksum_text,
    encoding="utf-8"
)

(
    REPO_CONFIG_DIR /
    checksum_filename
).write_text(
    checksum_text,
    encoding="utf-8"
)

# ------------------------------------------------------------
# Final summary
# ------------------------------------------------------------

print("\n" + "=" * 76)
print("T0 TARGET-GENE RCV EXTRACTION COMPLETED")
print("=" * 76)

print(
    "Complete XML records scanned:",
    f"{records_scanned:,}"
)

print(
    "RCV records extracted:",
    f"{records_extracted:,}"
)

print(
    "Gene counts:",
    dict(
        sorted(gene_counts.items())
    )
)

print(
    "Study-scope counts:",
    dict(
        sorted(scope_counts.items())
    )
)

print(
    "Missing RCV accessions:",
    f"{records_missing_rcv:,}"
)

print(
    "Missing VariationIDs:",
    f"{records_missing_variation_id:,}"
)

print(
    "Rows containing multiple target genes:",
    f"{records_with_multiple_target_genes:,}"
)

print(
    "\nDrive Parquet output:"
)
print(DRIVE_PARQUET_PATH)

print(
    "\nParquet rows:",
    f"{parquet_row_count:,}"
)

print(
    "Parquet size:",
    f"{local_output_bytes / (1024**2):,.2f} MB"
)

print(
    "Parquet SHA-256:",
    local_output_sha256
)

print(
    "\nExtraction manifest:"
)
print(drive_manifest_path)

print(
    "\nManifest SHA-256:",
    manifest_sha256
)

print(
    "\nIMPORTANT: The raw T0 XML has not been deleted. "
    "We will validate the extracted dataset first."
)

print(
    "No T1 information, temporal outcome, or GES model "
    "was used."
)

EXTRACTING T0 RCV RECORDS
Records scanned: 100,000 | Target RCVs extracted: 53
Records scanned: 200,000 | Target RCVs extracted: 71
Records scanned: 300,000 | Target RCVs extracted: 131
Records scanned: 400,000 | Target RCVs extracted: 201
Records scanned: 500,000 | Target RCVs extracted: 390
Records scanned: 600,000 | Target RCVs extracted: 648
Records scanned: 700,000 | Target RCVs extracted: 670
Records scanned: 800,000 | Target RCVs extracted: 876
Records scanned: 900,000 | Target RCVs extracted: 1,336
Records scanned: 1,000,000 | Target RCVs extracted: 1,587
Records scanned: 1,100,000 | Target RCVs extracted: 1,862
Records scanned: 1,200,000 | Target RCVs extracted: 2,016
Records scanned: 1,300,000 | Target RCVs extracted: 7,592
Records scanned: 1,400,000 | Target RCVs extracted: 15,240
Records scanned: 1,500,000 | Target RCVs extracted: 25,316
Records scanned: 1,600,000 | Target RCVs extracted: 30,033
Records scanned: 1,700,000 | Target RCVs extracted: 34,521
Records scanned: 1,8

In [16]:
# STEP 1: Verify the saved T0 Parquet artifact

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

from pathlib import Path
import hashlib
import pyarrow.parquet as pq

T0_PARQUET = Path(
    "/content/drive/MyDrive/GES_RAG_Temporal_Study/"
    "data_interim/t0_rcv_target_genes_raw.parquet"
)

EXPECTED_ROWS = 71_659
EXPECTED_SHA256 = (
    "663ad4c20ce114a763dc6512ff95951f95c2e8448c20a6c4ccb5818486c25f8d"
)

print("=" * 72)
print("T0 PARQUET ARTIFACT VERIFICATION")
print("=" * 72)
print("Path:", T0_PARQUET)
print("File exists:", T0_PARQUET.exists())

if not T0_PARQUET.exists():
    raise FileNotFoundError(
        f"T0 Parquet file was not found at:\n{T0_PARQUET}"
    )

# Read Parquet metadata without loading the complete dataset into memory
parquet_file = pq.ParquetFile(T0_PARQUET)

observed_rows = parquet_file.metadata.num_rows
observed_columns = parquet_file.metadata.num_columns
observed_row_groups = parquet_file.metadata.num_row_groups
observed_size_mb = T0_PARQUET.stat().st_size / (1024 ** 2)

# Calculate SHA-256
sha256 = hashlib.sha256()

with open(T0_PARQUET, "rb") as file:
    for block in iter(lambda: file.read(1024 * 1024), b""):
        sha256.update(block)

observed_sha256 = sha256.hexdigest()

print("\nPARQUET METADATA")
print("-" * 72)
print(f"Rows:       {observed_rows:,}")
print(f"Columns:    {observed_columns}")
print(f"Row groups: {observed_row_groups}")
print(f"File size:  {observed_size_mb:.3f} MB")

print("\nINTEGRITY CHECKS")
print("-" * 72)
print("Expected rows: ", f"{EXPECTED_ROWS:,}")
print("Observed rows: ", f"{observed_rows:,}")
print("Row-count check:", "PASS" if observed_rows == EXPECTED_ROWS else "FAIL")

print("\nExpected SHA-256:", EXPECTED_SHA256)
print("Observed SHA-256:", observed_sha256)
print(
    "Checksum check:",
    "PASS" if observed_sha256 == EXPECTED_SHA256 else "FAIL"
)

print("\nCOLUMN NAMES")
print("-" * 72)

for number, column in enumerate(parquet_file.schema.names, start=1):
    print(f"{number:02d}. {column}")

print("\nOVERALL STATUS")
print("-" * 72)

if (
    observed_rows == EXPECTED_ROWS
    and observed_sha256 == EXPECTED_SHA256
):
    print("PASS — the exact validated T0 artifact is present.")
else:
    print("STOP — do not delete the raw T0 XML or proceed to T1 yet.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
T0 PARQUET ARTIFACT VERIFICATION
Path: /content/drive/MyDrive/GES_RAG_Temporal_Study/data_interim/t0_rcv_target_genes_raw.parquet
File exists: True

PARQUET METADATA
------------------------------------------------------------------------
Rows:       71,659
Columns:    34
Row groups: 15
File size:  3.317 MB

INTEGRITY CHECKS
------------------------------------------------------------------------
Expected rows:  71,659
Observed rows:  71,659
Row-count check: PASS

Expected SHA-256: 663ad4c20ce114a763dc6512ff95951f95c2e8448c20a6c4ccb5818486c25f8d
Observed SHA-256: 663ad4c20ce114a763dc6512ff95951f95c2e8448c20a6c4ccb5818486c25f8d
Checksum check: PASS

COLUMN NAMES
------------------------------------------------------------------------
01. timepoint
02. release_label
03. archive_publication_date
04. embedded_data_cutoff_date
05. xml_record_index
06. rcv_accessio

In [17]:
# STEP 2: Validate primary keys and core identifiers
# This cell reads only the identifier columns and does not modify the dataset.

from pathlib import Path
import pandas as pd

T0_PARQUET = Path(
    "/content/drive/MyDrive/GES_RAG_Temporal_Study/"
    "data_interim/t0_rcv_target_genes_raw.parquet"
)

EXPECTED_ROWS = 71_659

identifier_columns = [
    "xml_record_index",
    "rcv_accession",
    "rcv_version",
    "variation_id",
    "vcv_accession",
    "vcv_version",
]

df_ids = pd.read_parquet(
    T0_PARQUET,
    columns=identifier_columns
)

print("=" * 78)
print("T0 PRIMARY-KEY AND IDENTIFIER VALIDATION")
print("=" * 78)

# ---------------------------------------------------------------------
# 1. Basic counts
# ---------------------------------------------------------------------

row_count = len(df_ids)

print("\nBASIC COUNTS")
print("-" * 78)
print(f"Rows loaded:                    {row_count:,}")
print(f"Unique RCV accessions:          {df_ids['rcv_accession'].nunique(dropna=True):,}")
print(f"Unique VariationIDs:            {df_ids['variation_id'].nunique(dropna=True):,}")
print(f"Unique VCV accessions:          {df_ids['vcv_accession'].nunique(dropna=True):,}")
print(f"Unique XML record indexes:      {df_ids['xml_record_index'].nunique(dropna=True):,}")

# Repeated VariationIDs are expected because one variant may have
# several condition-specific RCV records.
repeated_variation_rows = df_ids.duplicated(
    subset=["variation_id"],
    keep=False
).sum()

print(
    f"Rows sharing a VariationID:     {repeated_variation_rows:,} "
    "(expected at RCV level)"
)

# ---------------------------------------------------------------------
# 2. Missing mandatory identifiers
# ---------------------------------------------------------------------

mandatory_columns = [
    "xml_record_index",
    "rcv_accession",
    "rcv_version",
    "variation_id",
]

missing_counts = (
    df_ids[mandatory_columns]
    .isna()
    .sum()
    .astype(int)
)

blank_rcv_count = (
    df_ids["rcv_accession"]
    .astype("string")
    .str.strip()
    .eq("")
    .sum()
)

print("\nMISSING MANDATORY IDENTIFIERS")
print("-" * 78)

for column, count in missing_counts.items():
    print(f"{column:25s}: {count:,}")

print(f"{'blank rcv_accession':25s}: {blank_rcv_count:,}")

# ---------------------------------------------------------------------
# 3. Duplicate-key checks
# ---------------------------------------------------------------------

duplicate_rcv_mask = df_ids.duplicated(
    subset=["rcv_accession"],
    keep=False
)

duplicate_rcv_version_mask = df_ids.duplicated(
    subset=["rcv_accession", "rcv_version"],
    keep=False
)

duplicate_xml_index_mask = df_ids.duplicated(
    subset=["xml_record_index"],
    keep=False
)

duplicate_rcv_rows = int(duplicate_rcv_mask.sum())
duplicate_rcv_keys = int(
    df_ids.loc[duplicate_rcv_mask, "rcv_accession"].nunique()
)

duplicate_rcv_version_rows = int(duplicate_rcv_version_mask.sum())
duplicate_xml_index_rows = int(duplicate_xml_index_mask.sum())

print("\nDUPLICATE-KEY CHECKS")
print("-" * 78)
print(f"Rows in duplicated RCV accessions:       {duplicate_rcv_rows:,}")
print(f"Distinct duplicated RCV accessions:      {duplicate_rcv_keys:,}")
print(f"Rows in duplicated RCV-version pairs:    {duplicate_rcv_version_rows:,}")
print(f"Rows in duplicated XML record indexes:   {duplicate_xml_index_rows:,}")

# ---------------------------------------------------------------------
# 4. Identifier-format checks
# ---------------------------------------------------------------------

rcv_string = (
    df_ids["rcv_accession"]
    .astype("string")
    .str.strip()
    .str.upper()
)

vcv_string = (
    df_ids["vcv_accession"]
    .astype("string")
    .str.strip()
    .str.upper()
)

# Standard ClinVar aggregate accessions:
# RCV followed by nine digits
# VCV followed by nine digits
malformed_rcv_mask = (
    rcv_string.notna()
    & ~rcv_string.str.fullmatch(r"RCV\d{9}", na=False)
)

malformed_vcv_mask = (
    vcv_string.notna()
    & vcv_string.ne("")
    & ~vcv_string.str.fullmatch(r"VCV\d{9}", na=False)
)

variation_numeric = pd.to_numeric(
    df_ids["variation_id"],
    errors="coerce"
)

rcv_version_numeric = pd.to_numeric(
    df_ids["rcv_version"],
    errors="coerce"
)

invalid_variation_id_mask = (
    variation_numeric.isna()
    | (variation_numeric <= 0)
    | (variation_numeric % 1 != 0)
)

invalid_rcv_version_mask = (
    rcv_version_numeric.isna()
    | (rcv_version_numeric <= 0)
    | (rcv_version_numeric % 1 != 0)
)

print("\nIDENTIFIER-FORMAT CHECKS")
print("-" * 78)
print(f"Malformed RCV accessions:        {malformed_rcv_mask.sum():,}")
print(f"Malformed nonmissing VCVs:       {malformed_vcv_mask.sum():,}")
print(f"Invalid VariationIDs:            {invalid_variation_id_mask.sum():,}")
print(f"Invalid RCV versions:            {invalid_rcv_version_mask.sum():,}")
print(f"Missing VCV accessions:          {df_ids['vcv_accession'].isna().sum():,}")
print(f"Missing VCV versions:            {df_ids['vcv_version'].isna().sum():,}")

# ---------------------------------------------------------------------
# 5. Show suspicious records only when found
# ---------------------------------------------------------------------

suspicious_mask = (
    duplicate_rcv_mask
    | duplicate_xml_index_mask
    | malformed_rcv_mask
    | invalid_variation_id_mask
    | invalid_rcv_version_mask
)

suspicious = (
    df_ids.loc[suspicious_mask]
    .sort_values(
        ["rcv_accession", "rcv_version"],
        na_position="last"
    )
    .head(20)
)

if len(suspicious) > 0:
    print("\nFIRST 20 SUSPICIOUS RECORDS")
    print("-" * 78)
    print(suspicious.to_string(index=False))
else:
    print("\nNo suspicious mandatory-identifier records were found.")

# ---------------------------------------------------------------------
# 6. Overall acceptance status
# ---------------------------------------------------------------------

critical_failures = {
    "unexpected row count": row_count != EXPECTED_ROWS,
    "missing mandatory identifiers": int(missing_counts.sum()) > 0,
    "blank RCV accessions": int(blank_rcv_count) > 0,
    "duplicated RCV accessions": duplicate_rcv_rows > 0,
    "duplicated XML indexes": duplicate_xml_index_rows > 0,
    "malformed RCV accessions": int(malformed_rcv_mask.sum()) > 0,
    "invalid VariationIDs": int(invalid_variation_id_mask.sum()) > 0,
    "invalid RCV versions": int(invalid_rcv_version_mask.sum()) > 0,
}

failed_checks = [
    name for name, failed in critical_failures.items()
    if failed
]

print("\nOVERALL STATUS")
print("-" * 78)

if not failed_checks:
    print("PASS — mandatory identifiers and RCV-level primary keys are valid.")
    print(
        "Repeated VariationIDs are not treated as errors because the "
        "study unit is the condition-specific RCV record."
    )
else:
    print("FAIL — do not delete the raw T0 XML or proceed to T1.")
    print("Failed checks:")
    for check in failed_checks:
        print(" -", check)

T0 PRIMARY-KEY AND IDENTIFIER VALIDATION

BASIC COUNTS
------------------------------------------------------------------------------
Rows loaded:                    71,659
Unique RCV accessions:          71,659
Unique VariationIDs:            36,807
Unique VCV accessions:          36,807
Unique XML record indexes:      71,659
Rows sharing a VariationID:     51,103 (expected at RCV level)

MISSING MANDATORY IDENTIFIERS
------------------------------------------------------------------------------
xml_record_index         : 0
rcv_accession            : 0
rcv_version              : 0
variation_id             : 0
blank rcv_accession      : 0

DUPLICATE-KEY CHECKS
------------------------------------------------------------------------------
Rows in duplicated RCV accessions:       0
Distinct duplicated RCV accessions:      0
Rows in duplicated RCV-version pairs:    0
Rows in duplicated XML record indexes:   0

IDENTIFIER-FORMAT CHECKS
------------------------------------------------------

In [18]:
# STEP 3: Validate target-gene assignments and primary/exploratory scope

from pathlib import Path
import pandas as pd
import json

T0_PARQUET = Path(
    "/content/drive/MyDrive/GES_RAG_Temporal_Study/"
    "data_interim/t0_rcv_target_genes_raw.parquet"
)

EXPECTED_TOTAL = 71_659

EXPECTED_GENE_COUNTS = {
    "BRCA1": 25_408,
    "BRCA2": 34_915,
    "MLH1": 8_936,
    "EGFR": 2_400,
}

PRIMARY_GENES = {"BRCA1", "BRCA2", "MLH1"}
EXPLORATORY_GENES = {"EGFR"}
ALLOWED_GENES = PRIMARY_GENES | EXPLORATORY_GENES

df_gene = pd.read_parquet(
    T0_PARQUET,
    columns=[
        "rcv_accession",
        "variation_id",
        "target_genes_json",
        "study_scope",
    ],
)

print("=" * 78)
print("T0 TARGET-GENE AND STUDY-SCOPE VALIDATION")
print("=" * 78)

# ---------------------------------------------------------------------
# 1. Safely parse target_genes_json
# ---------------------------------------------------------------------

def parse_gene_list(value):
    """
    Return a normalized list of gene symbols.
    Parsing failures are returned as None.
    """
    if pd.isna(value):
        return None

    if isinstance(value, list):
        parsed = value
    else:
        try:
            parsed = json.loads(str(value))
        except (json.JSONDecodeError, TypeError, ValueError):
            return None

    if not isinstance(parsed, list):
        return None

    normalized = []

    for gene in parsed:
        if gene is None:
            continue

        gene_text = str(gene).strip().upper()

        if gene_text:
            normalized.append(gene_text)

    return sorted(set(normalized))


df_gene["parsed_genes"] = df_gene["target_genes_json"].apply(
    parse_gene_list
)

parse_failure_mask = df_gene["parsed_genes"].isna()

df_gene["gene_count"] = df_gene["parsed_genes"].apply(
    lambda genes: len(genes) if isinstance(genes, list) else None
)

single_gene_mask = df_gene["gene_count"].eq(1)

df_gene["target_gene"] = df_gene["parsed_genes"].apply(
    lambda genes: genes[0]
    if isinstance(genes, list) and len(genes) == 1
    else pd.NA
)

# ---------------------------------------------------------------------
# 2. Structural gene checks
# ---------------------------------------------------------------------

zero_gene_mask = df_gene["gene_count"].eq(0)
multiple_gene_mask = df_gene["gene_count"].gt(1)

unexpected_gene_mask = (
    df_gene["target_gene"].notna()
    & ~df_gene["target_gene"].isin(ALLOWED_GENES)
)

print("\nGENE-STRUCTURE CHECKS")
print("-" * 78)
print(f"Rows loaded:                         {len(df_gene):,}")
print(f"JSON parsing failures:               {parse_failure_mask.sum():,}")
print(f"Rows with zero target genes:         {zero_gene_mask.sum():,}")
print(f"Rows with exactly one target gene:   {single_gene_mask.sum():,}")
print(f"Rows with multiple target genes:     {multiple_gene_mask.sum():,}")
print(f"Rows with unexpected target genes:   {unexpected_gene_mask.sum():,}")

# ---------------------------------------------------------------------
# 3. Gene-specific counts
# ---------------------------------------------------------------------

observed_gene_counts = (
    df_gene["target_gene"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nGENE-SPECIFIC COUNTS")
print("-" * 78)

for gene in sorted(ALLOWED_GENES):
    observed = int(observed_gene_counts.get(gene, 0))
    expected = EXPECTED_GENE_COUNTS[gene]
    status = "PASS" if observed == expected else "FAIL"

    print(
        f"{gene:6s}: observed={observed:>7,} | "
        f"expected={expected:>7,} | {status}"
    )

primary_count = int(
    df_gene["target_gene"].isin(PRIMARY_GENES).sum()
)

exploratory_count = int(
    df_gene["target_gene"].isin(EXPLORATORY_GENES).sum()
)

print("-" * 78)
print(f"Primary-gene records:                {primary_count:,}")
print(f"Exploratory EGFR records:            {exploratory_count:,}")
print(f"Total recognized target records:     {primary_count + exploratory_count:,}")

# ---------------------------------------------------------------------
# 4. Study-scope checks
# ---------------------------------------------------------------------

scope_normalized = (
    df_gene["study_scope"]
    .astype("string")
    .str.strip()
    .str.lower()
)

df_gene["scope_normalized"] = scope_normalized

print("\nOBSERVED STUDY-SCOPE VALUES")
print("-" * 78)
print(
    scope_normalized
    .value_counts(dropna=False)
    .to_string()
)

expected_scope = df_gene["target_gene"].map(
    lambda gene:
        "primary"
        if gene in PRIMARY_GENES
        else "exploratory"
        if gene in EXPLORATORY_GENES
        else pd.NA
)

scope_missing_mask = scope_normalized.isna() | scope_normalized.eq("")

scope_mismatch_mask = (
    expected_scope.notna()
    & scope_normalized.notna()
    & scope_normalized.ne(expected_scope)
)

print("\nSTUDY-SCOPE CONSISTENCY")
print("-" * 78)
print(f"Missing or blank study_scope:        {scope_missing_mask.sum():,}")
print(f"Gene/scope mismatches:               {scope_mismatch_mask.sum():,}")

# ---------------------------------------------------------------------
# 5. Show suspicious records if present
# ---------------------------------------------------------------------

suspicious_mask = (
    parse_failure_mask
    | zero_gene_mask
    | multiple_gene_mask
    | unexpected_gene_mask
    | scope_missing_mask
    | scope_mismatch_mask
)

suspicious = df_gene.loc[
    suspicious_mask,
    [
        "rcv_accession",
        "variation_id",
        "target_genes_json",
        "parsed_genes",
        "study_scope",
        "scope_normalized",
    ],
].head(20)

if len(suspicious) > 0:
    print("\nFIRST 20 SUSPICIOUS RECORDS")
    print("-" * 78)
    print(suspicious.to_string(index=False))
else:
    print("\nNo suspicious gene or study-scope records were found.")

# ---------------------------------------------------------------------
# 6. Overall acceptance status
# ---------------------------------------------------------------------

gene_count_failures = {
    gene: int(observed_gene_counts.get(gene, 0)) != expected
    for gene, expected in EXPECTED_GENE_COUNTS.items()
}

critical_failures = {
    "unexpected total row count": len(df_gene) != EXPECTED_TOTAL,
    "gene JSON parsing failures": int(parse_failure_mask.sum()) > 0,
    "rows without exactly one target gene": int((~single_gene_mask).sum()) > 0,
    "unexpected target genes": int(unexpected_gene_mask.sum()) > 0,
    "gene-count discrepancy": any(gene_count_failures.values()),
    "missing study scope": int(scope_missing_mask.sum()) > 0,
    "gene/scope mismatch": int(scope_mismatch_mask.sum()) > 0,
}

failed_checks = [
    name
    for name, failed in critical_failures.items()
    if failed
]

print("\nOVERALL STATUS")
print("-" * 78)

if not failed_checks:
    print("PASS — gene assignments and study-scope labels are valid.")
    print(
        "The cohort contains 69,259 primary-gene RCVs and "
        "2,400 exploratory EGFR RCVs."
    )
else:
    print("FAIL — do not delete the raw T0 XML or proceed to T1.")
    print("Failed checks:")

    for check in failed_checks:
        print(" -", check)

T0 TARGET-GENE AND STUDY-SCOPE VALIDATION

GENE-STRUCTURE CHECKS
------------------------------------------------------------------------------
Rows loaded:                         71,659
JSON parsing failures:               0
Rows with zero target genes:         0
Rows with exactly one target gene:   71,659
Rows with multiple target genes:     0
Rows with unexpected target genes:   0

GENE-SPECIFIC COUNTS
------------------------------------------------------------------------------
BRCA1 : observed= 25,408 | expected= 25,408 | PASS
BRCA2 : observed= 34,915 | expected= 34,915 | PASS
EGFR  : observed=  2,400 | expected=  2,400 | PASS
MLH1  : observed=  8,936 | expected=  8,936 | PASS
------------------------------------------------------------------------------
Primary-gene records:                69,259
Exploratory EGFR records:            2,400
Total recognized target records:     71,659

OBSERVED STUDY-SCOPE VALUES
--------------------------------------------------------------------

In [19]:
# STEP 4: Validate T0 provenance and temporal metadata
# This cell does not modify the Parquet dataset.

from pathlib import Path
import pandas as pd

T0_PARQUET = Path(
    "/content/drive/MyDrive/GES_RAG_Temporal_Study/"
    "data_interim/t0_rcv_target_genes_raw.parquet"
)

EXPECTED_ROWS = 71_659
EXPECTED_SOURCE_RECORDS = 2_302_323

EXPECTED_VALUES = {
    "timepoint": "T0",
    "release_label": "2023-01",
    "archive_publication_date": "2023-01-05",
    "embedded_data_cutoff_date": "2022-12-31",
    "source_filename": "ClinVarFullRelease_2023-01.xml.gz",
    "source_sha256": (
        "911c8a58872ea89cc7bb4f1ee3362596d965f5103422b30f456abaf99c41e5e7"
    ),
}

columns = [
    "timepoint",
    "release_label",
    "archive_publication_date",
    "embedded_data_cutoff_date",
    "xml_record_index",
    "source_filename",
    "source_sha256",
]

df_prov = pd.read_parquet(
    T0_PARQUET,
    columns=columns,
)

print("=" * 82)
print("T0 PROVENANCE AND TEMPORAL-METADATA VALIDATION")
print("=" * 82)

# ---------------------------------------------------------------------
# 1. Normalization helpers
# ---------------------------------------------------------------------

def normalize_text(series):
    return (
        series.astype("string")
        .str.strip()
    )


def normalize_date(series):
    """
    Convert date-like values to ISO YYYY-MM-DD strings.
    Invalid or missing values become <NA>.
    """
    parsed = pd.to_datetime(series, errors="coerce")

    return parsed.dt.strftime("%Y-%m-%d").astype("string")


# Normalize stored fields without changing the source file
normalized = pd.DataFrame(index=df_prov.index)

normalized["timepoint"] = (
    normalize_text(df_prov["timepoint"])
    .str.upper()
)

normalized["release_label"] = normalize_text(
    df_prov["release_label"]
)

normalized["archive_publication_date"] = normalize_date(
    df_prov["archive_publication_date"]
)

normalized["embedded_data_cutoff_date"] = normalize_date(
    df_prov["embedded_data_cutoff_date"]
)

# Compare only the filename, even if a full path was accidentally stored
normalized["source_filename"] = (
    normalize_text(df_prov["source_filename"])
    .apply(
        lambda value:
            Path(value).name
            if pd.notna(value) and value != ""
            else pd.NA
    )
    .astype("string")
)

normalized["source_sha256"] = (
    normalize_text(df_prov["source_sha256"])
    .str.lower()
)

# ---------------------------------------------------------------------
# 2. Row count and missingness
# ---------------------------------------------------------------------

print("\nBASIC CHECKS")
print("-" * 82)
print(f"Rows loaded:                         {len(df_prov):,}")
print(f"Expected rows:                       {EXPECTED_ROWS:,}")

print("\nMISSING PROVENANCE VALUES")
print("-" * 82)

missing_counts = normalized.isna().sum()

for column in normalized.columns:
    print(f"{column:32s}: {int(missing_counts[column]):,}")

xml_index_numeric = pd.to_numeric(
    df_prov["xml_record_index"],
    errors="coerce",
)

missing_xml_index = int(xml_index_numeric.isna().sum())

print(f"{'xml_record_index':32s}: {missing_xml_index:,}")

# ---------------------------------------------------------------------
# 3. Unique values and expected constants
# ---------------------------------------------------------------------

print("\nOBSERVED PROVENANCE CONSTANTS")
print("-" * 82)

constant_failures = {}

for column, expected in EXPECTED_VALUES.items():
    observed_values = (
        normalized[column]
        .dropna()
        .unique()
        .tolist()
    )

    observed_values_sorted = sorted(
        str(value) for value in observed_values
    )

    exact_match = observed_values_sorted == [expected]

    constant_failures[column] = not exact_match

    print(f"\n{column}")
    print(f"  Expected: {expected}")
    print(f"  Observed unique value count: {len(observed_values_sorted):,}")

    if len(observed_values_sorted) <= 10:
        for value in observed_values_sorted:
            print(f"  Observed: {value}")
    else:
        print("  First 10 observed values:")
        for value in observed_values_sorted[:10]:
            print(f"    {value}")

    print(f"  Status: {'PASS' if exact_match else 'FAIL'}")

# ---------------------------------------------------------------------
# 4. XML source-record index checks
# ---------------------------------------------------------------------

invalid_xml_index_mask = (
    xml_index_numeric.isna()
    | (xml_index_numeric <= 0)
    | (xml_index_numeric % 1 != 0)
    | (xml_index_numeric > EXPECTED_SOURCE_RECORDS)
)

duplicate_xml_index_mask = xml_index_numeric.duplicated(
    keep=False
)

strictly_increasing = (
    xml_index_numeric.notna().all()
    and xml_index_numeric.is_monotonic_increasing
    and not duplicate_xml_index_mask.any()
)

minimum_xml_index = (
    int(xml_index_numeric.min())
    if xml_index_numeric.notna().any()
    else None
)

maximum_xml_index = (
    int(xml_index_numeric.max())
    if xml_index_numeric.notna().any()
    else None
)

print("\nXML RECORD-INDEX CHECKS")
print("-" * 82)
print(f"Minimum retained XML index:          {minimum_xml_index:,}")
print(f"Maximum retained XML index:          {maximum_xml_index:,}")
print(f"Complete source XML record count:    {EXPECTED_SOURCE_RECORDS:,}")
print(f"Invalid or out-of-range indexes:     {invalid_xml_index_mask.sum():,}")
print(f"Rows in duplicated indexes:          {duplicate_xml_index_mask.sum():,}")
print(f"Indexes strictly increasing:         {strictly_increasing}")

# ---------------------------------------------------------------------
# 5. Date-order validation
# ---------------------------------------------------------------------

publication_dates = pd.to_datetime(
    normalized["archive_publication_date"],
    errors="coerce",
)

cutoff_dates = pd.to_datetime(
    normalized["embedded_data_cutoff_date"],
    errors="coerce",
)

invalid_date_order_mask = (
    publication_dates.notna()
    & cutoff_dates.notna()
    & publication_dates.lt(cutoff_dates)
)

followup_gap_days = (
    publication_dates - cutoff_dates
).dt.days

print("\nTEMPORAL RELATIONSHIP CHECK")
print("-" * 82)
print(
    "Rows where publication date precedes embedded cutoff: "
    f"{invalid_date_order_mask.sum():,}"
)

if followup_gap_days.notna().any():
    print(
        "Publication-minus-cutoff interval: "
        f"{int(followup_gap_days.min())} to "
        f"{int(followup_gap_days.max())} days"
    )

# ---------------------------------------------------------------------
# 6. Show suspicious rows only if found
# ---------------------------------------------------------------------

provenance_mismatch_mask = pd.Series(
    False,
    index=df_prov.index,
)

for column, expected in EXPECTED_VALUES.items():
    provenance_mismatch_mask |= (
        normalized[column].isna()
        | normalized[column].ne(expected)
    )

suspicious_mask = (
    provenance_mismatch_mask
    | invalid_xml_index_mask
    | duplicate_xml_index_mask
    | invalid_date_order_mask
)

suspicious_columns = [
    "timepoint",
    "release_label",
    "archive_publication_date",
    "embedded_data_cutoff_date",
    "xml_record_index",
    "source_filename",
    "source_sha256",
]

suspicious = df_prov.loc[
    suspicious_mask,
    suspicious_columns,
].head(20)

if len(suspicious) > 0:
    print("\nFIRST 20 SUSPICIOUS RECORDS")
    print("-" * 82)
    print(suspicious.to_string(index=False))
else:
    print("\nNo suspicious provenance records were found.")

# ---------------------------------------------------------------------
# 7. Overall acceptance status
# ---------------------------------------------------------------------

critical_failures = {
    "unexpected row count":
        len(df_prov) != EXPECTED_ROWS,

    "missing provenance values":
        int(missing_counts.sum()) > 0 or missing_xml_index > 0,

    "incorrect provenance constants":
        any(constant_failures.values()),

    "invalid XML indexes":
        int(invalid_xml_index_mask.sum()) > 0,

    "duplicated XML indexes":
        int(duplicate_xml_index_mask.sum()) > 0,

    "XML indexes not strictly increasing":
        not strictly_increasing,

    "invalid publication/cutoff date order":
        int(invalid_date_order_mask.sum()) > 0,
}

failed_checks = [
    name
    for name, failed in critical_failures.items()
    if failed
]

print("\nOVERALL STATUS")
print("-" * 82)

if not failed_checks:
    print(
        "PASS — T0 provenance, source integrity metadata, "
        "and temporal boundaries are internally consistent."
    )
    print(
        "The embedded cutoff of 2022-12-31 is confirmed as the "
        "reference date for subsequent T0 recency calculations."
    )
else:
    print(
        "FAIL — do not delete the raw T0 XML or proceed to T1."
    )
    print("Failed checks:")

    for check in failed_checks:
        print(" -", check)

T0 PROVENANCE AND TEMPORAL-METADATA VALIDATION

BASIC CHECKS
----------------------------------------------------------------------------------
Rows loaded:                         71,659
Expected rows:                       71,659

MISSING PROVENANCE VALUES
----------------------------------------------------------------------------------
timepoint                       : 0
release_label                   : 0
archive_publication_date        : 0
embedded_data_cutoff_date       : 0
source_filename                 : 0
source_sha256                   : 0
xml_record_index                : 0

OBSERVED PROVENANCE CONSTANTS
----------------------------------------------------------------------------------

timepoint
  Expected: T0
  Observed unique value count: 1
  Observed: T0
  Status: PASS

release_label
  Expected: 2023-01
  Observed unique value count: 1
  Observed: 2023-01
  Status: PASS

archive_publication_date
  Expected: 2023-01-05
  Observed unique value count: 1
  Observed: 2023-0

In [20]:
# STEP 5: Audit aggregate classifications, review statuses, stars,
# and conflict/disagreement flags.
#
# This is a profiling and structural-validation step.
# It does not modify the Parquet dataset.

from pathlib import Path
import pandas as pd

T0_PARQUET = Path(
    "/content/drive/MyDrive/GES_RAG_Temporal_Study/"
    "data_interim/t0_rcv_target_genes_raw.parquet"
)

EXPECTED_ROWS = 71_659

columns = [
    "rcv_accession",
    "aggregate_classification",
    "aggregate_classification_group",
    "aggregate_review_status",
    "aggregate_review_stars",
    "aggregate_conflict_flag",
    "scv_group_disagreement_flag",
]

df_evidence = pd.read_parquet(
    T0_PARQUET,
    columns=columns,
)

print("=" * 88)
print("T0 CLASSIFICATION, REVIEW-STATUS, AND CONFLICT-FIELD AUDIT")
print("=" * 88)

# ---------------------------------------------------------------------
# 1. Normalize text fields for auditing
# ---------------------------------------------------------------------

def normalize_text(series):
    return (
        series.astype("string")
        .str.strip()
        .replace("", pd.NA)
    )


classification = normalize_text(
    df_evidence["aggregate_classification"]
)

classification_group = normalize_text(
    df_evidence["aggregate_classification_group"]
)

review_status = normalize_text(
    df_evidence["aggregate_review_status"]
)

review_stars_numeric = pd.to_numeric(
    df_evidence["aggregate_review_stars"],
    errors="coerce",
)

# ---------------------------------------------------------------------
# 2. Basic missingness
# ---------------------------------------------------------------------

print("\nBASIC COUNTS")
print("-" * 88)
print(f"Rows loaded:                              {len(df_evidence):,}")
print(f"Unique RCV accessions:                    {df_evidence['rcv_accession'].nunique():,}")

print("\nMISSINGNESS")
print("-" * 88)
print(f"Missing aggregate classification:         {classification.isna().sum():,}")
print(f"Missing classification group:             {classification_group.isna().sum():,}")
print(f"Missing aggregate review status:          {review_status.isna().sum():,}")
print(f"Missing or nonnumeric review stars:        {review_stars_numeric.isna().sum():,}")
print(
    f"Missing aggregate conflict flag:          "
    f"{df_evidence['aggregate_conflict_flag'].isna().sum():,}"
)
print(
    f"Missing SCV disagreement flag:            "
    f"{df_evidence['scv_group_disagreement_flag'].isna().sum():,}"
)

# ---------------------------------------------------------------------
# 3. Aggregate classification-group distribution
# ---------------------------------------------------------------------

print("\nAGGREGATE CLASSIFICATION-GROUP DISTRIBUTION")
print("-" * 88)

group_counts = (
    classification_group
    .fillna("<MISSING>")
    .value_counts(dropna=False)
)

print(group_counts.to_string())

print("\nRAW AGGREGATE CLASSIFICATIONS — TOP 30")
print("-" * 88)

raw_classification_counts = (
    classification
    .fillna("<MISSING>")
    .value_counts(dropna=False)
    .head(30)
)

print(raw_classification_counts.to_string())

# ---------------------------------------------------------------------
# 4. Review-status distribution
# ---------------------------------------------------------------------

print("\nAGGREGATE REVIEW-STATUS DISTRIBUTION")
print("-" * 88)

review_status_counts = (
    review_status
    .fillna("<MISSING>")
    .value_counts(dropna=False)
)

print(review_status_counts.to_string())

# ---------------------------------------------------------------------
# 5. Review-star structural checks
# ---------------------------------------------------------------------

invalid_star_mask = (
    review_stars_numeric.notna()
    & (
        (review_stars_numeric < 0)
        | (review_stars_numeric > 4)
        | (review_stars_numeric % 1 != 0)
    )
)

print("\nREVIEW-STAR DISTRIBUTION")
print("-" * 88)

star_counts = (
    review_stars_numeric
    .fillna(-1)
    .value_counts()
    .sort_index()
)

for value, count in star_counts.items():
    label = "<MISSING/NONNUMERIC>" if value == -1 else str(int(value))
    print(f"Stars {label:>20s}: {int(count):,}")

print("\nREVIEW-STAR STRUCTURAL CHECKS")
print("-" * 88)
print(f"Invalid stars outside integer 0–4:         {invalid_star_mask.sum():,}")

# ---------------------------------------------------------------------
# 6. Review-status and star combinations
# ---------------------------------------------------------------------

status_star_table = pd.DataFrame(
    {
        "review_status": review_status.fillna("<MISSING>"),
        "review_stars": review_stars_numeric,
    }
)

status_star_combinations = (
    status_star_table
    .groupby(
        ["review_status", "review_stars"],
        dropna=False,
    )
    .size()
    .reset_index(name="record_count")
    .sort_values(
        ["review_stars", "record_count", "review_status"],
        ascending=[True, False, True],
        na_position="first",
    )
)

print("\nREVIEW-STATUS × STAR COMBINATIONS")
print("-" * 88)
print(status_star_combinations.to_string(index=False))

# Count statuses mapped to more than one star value
stars_per_status = (
    status_star_table
    .dropna(subset=["review_status"])
    .groupby("review_status")["review_stars"]
    .nunique(dropna=False)
)

multi_star_statuses = stars_per_status[
    stars_per_status > 1
]

print("\nSTATUS-MAPPING CONSISTENCY")
print("-" * 88)
print(
    "Review-status values associated with more than one star value: "
    f"{len(multi_star_statuses):,}"
)

if len(multi_star_statuses) > 0:
    print(multi_star_statuses.to_string())

# ---------------------------------------------------------------------
# 7. Boolean-flag auditing
# ---------------------------------------------------------------------

def normalize_boolean_value(value):
    if pd.isna(value):
        return pd.NA

    if isinstance(value, bool):
        return value

    if isinstance(value, (int, float)) and value in (0, 1):
        return bool(value)

    text = str(value).strip().lower()

    mapping = {
        "true": True,
        "false": False,
        "1": True,
        "0": False,
        "yes": True,
        "no": False,
    }

    return mapping.get(text, "INVALID")


aggregate_conflict_normalized = (
    df_evidence["aggregate_conflict_flag"]
    .apply(normalize_boolean_value)
)

scv_disagreement_normalized = (
    df_evidence["scv_group_disagreement_flag"]
    .apply(normalize_boolean_value)
)

invalid_aggregate_conflict = (
    aggregate_conflict_normalized.eq("INVALID").sum()
)

invalid_scv_disagreement = (
    scv_disagreement_normalized.eq("INVALID").sum()
)

print("\nAGGREGATE CONFLICT-FLAG VALUES")
print("-" * 88)
print(
    aggregate_conflict_normalized
    .fillna("<MISSING>")
    .value_counts(dropna=False)
    .to_string()
)

print("\nSCV GROUP-DISAGREEMENT-FLAG VALUES")
print("-" * 88)
print(
    scv_disagreement_normalized
    .fillna("<MISSING>")
    .value_counts(dropna=False)
    .to_string()
)

print("\nBOOLEAN-FIELD STRUCTURAL CHECKS")
print("-" * 88)
print(f"Invalid aggregate conflict values:        {invalid_aggregate_conflict:,}")
print(f"Invalid SCV disagreement values:          {invalid_scv_disagreement:,}")

# ---------------------------------------------------------------------
# 8. Conflict/group consistency checks
# ---------------------------------------------------------------------

group_normalized_lower = (
    classification_group
    .fillna("")
    .str.lower()
)

group_says_conflicting = group_normalized_lower.eq("conflicting")

aggregate_conflict_true = aggregate_conflict_normalized.eq(True)

conflicting_group_but_flag_false = (
    group_says_conflicting
    & aggregate_conflict_normalized.eq(False)
)

conflict_flag_true_but_group_not_conflicting = (
    aggregate_conflict_true
    & ~group_says_conflicting
)

print("\nCONFLICT/GROUP RELATIONSHIP")
print("-" * 88)
print(
    "Classification group is Conflicting but aggregate flag is False: "
    f"{conflicting_group_but_flag_false.sum():,}"
)
print(
    "Aggregate conflict flag is True but group is not Conflicting:     "
    f"{conflict_flag_true_but_group_not_conflicting.sum():,}"
)

# These are reported for investigation and are not automatically
# treated as extraction failures because the two fields may encode
# related but nonidentical concepts.

# ---------------------------------------------------------------------
# 9. Show structurally suspicious records
# ---------------------------------------------------------------------

suspicious_mask = (
    invalid_star_mask
    | aggregate_conflict_normalized.eq("INVALID")
    | scv_disagreement_normalized.eq("INVALID")
)

suspicious = df_evidence.loc[
    suspicious_mask,
    columns,
].head(20)

if len(suspicious) > 0:
    print("\nFIRST 20 STRUCTURALLY SUSPICIOUS RECORDS")
    print("-" * 88)
    print(suspicious.to_string(index=False))
else:
    print("\nNo structurally invalid classification or review records were found.")

# ---------------------------------------------------------------------
# 10. Technical status
# ---------------------------------------------------------------------

critical_failures = {
    "unexpected row count":
        len(df_evidence) != EXPECTED_ROWS,

    "duplicate or missing RCV identity":
        df_evidence["rcv_accession"].nunique(dropna=True) != EXPECTED_ROWS,

    "missing classification groups":
        int(classification_group.isna().sum()) > 0,

    "missing review statuses":
        int(review_status.isna().sum()) > 0,

    "missing or nonnumeric review stars":
        int(review_stars_numeric.isna().sum()) > 0,

    "invalid review-star values":
        int(invalid_star_mask.sum()) > 0,

    "invalid conflict flag values":
        int(invalid_aggregate_conflict) > 0,

    "invalid disagreement flag values":
        int(invalid_scv_disagreement) > 0,
}

failed_checks = [
    name
    for name, failed in critical_failures.items()
    if failed
]

print("\nTECHNICAL STATUS")
print("-" * 88)

if not failed_checks:
    print(
        "PASS — classification groups, review-star values, and "
        "boolean evidence flags are structurally valid."
    )
    print(
        "The review-status × star combinations above should now be "
        "examined before freezing the historical review-status mapping."
    )
else:
    print(
        "REVIEW REQUIRED — do not delete the raw T0 XML yet."
    )
    print("Checks requiring investigation:")

    for check in failed_checks:
        print(" -", check)

T0 CLASSIFICATION, REVIEW-STATUS, AND CONFLICT-FIELD AUDIT

BASIC COUNTS
----------------------------------------------------------------------------------------
Rows loaded:                              71,659
Unique RCV accessions:                    71,659

MISSINGNESS
----------------------------------------------------------------------------------------
Missing aggregate classification:         0
Missing classification group:             0
Missing aggregate review status:          0
Missing or nonnumeric review stars:        0
Missing aggregate conflict flag:          0
Missing SCV disagreement flag:            0

AGGREGATE CLASSIFICATION-GROUP DISTRIBUTION
----------------------------------------------------------------------------------------
aggregate_classification_group
VUS                             26305
Pathogenic/Likely pathogenic    20430
Benign/Likely benign            19483
Other                            3961
Conflicting                      1480

RAW AGGREGATE CLA

In [21]:
# STEP 6: Diagnose the aggregate_conflict_flag logic
#
# This cell does not modify the Parquet dataset.
# It tests whether "no conflicts" was accidentally interpreted as conflict-positive.

from pathlib import Path
import pandas as pd

T0_PARQUET = Path(
    "/content/drive/MyDrive/GES_RAG_Temporal_Study/"
    "data_interim/t0_rcv_target_genes_raw.parquet"
)

columns = [
    "rcv_accession",
    "aggregate_classification",
    "aggregate_classification_group",
    "aggregate_review_status",
    "aggregate_review_stars",
    "aggregate_conflict_flag",
    "scv_group_disagreement_flag",
    "aggregate_explanation",
]

df_conflict = pd.read_parquet(
    T0_PARQUET,
    columns=columns,
)

print("=" * 92)
print("T0 AGGREGATE-CONFLICT FLAG DIAGNOSTIC")
print("=" * 92)

# ---------------------------------------------------------------------
# 1. Normalize relevant fields
# ---------------------------------------------------------------------

def normalize_text(series):
    return (
        series.astype("string")
        .str.strip()
        .str.lower()
        .replace("", pd.NA)
    )


def normalize_boolean(series):
    mapping = {
        True: True,
        False: False,
        1: True,
        0: False,
        "true": True,
        "false": False,
        "1": True,
        "0": False,
        "yes": True,
        "no": False,
    }

    def convert(value):
        if pd.isna(value):
            return pd.NA

        if value in mapping:
            return mapping[value]

        text = str(value).strip().lower()
        return mapping.get(text, pd.NA)

    return series.apply(convert).astype("boolean")


status = normalize_text(
    df_conflict["aggregate_review_status"]
)

classification = normalize_text(
    df_conflict["aggregate_classification"]
)

classification_group = normalize_text(
    df_conflict["aggregate_classification_group"]
)

explanation = normalize_text(
    df_conflict["aggregate_explanation"]
)

observed_flag = normalize_boolean(
    df_conflict["aggregate_conflict_flag"]
)

scv_disagreement = normalize_boolean(
    df_conflict["scv_group_disagreement_flag"]
)

# ---------------------------------------------------------------------
# 2. Define exact semantic categories
# ---------------------------------------------------------------------

status_no_conflicts = status.eq(
    "criteria provided, multiple submitters, no conflicts"
)

status_conflicting = status.eq(
    "criteria provided, conflicting interpretations"
)

group_conflicting = classification_group.eq("conflicting")

raw_classification_conflicting = classification.str.contains(
    "conflicting",
    case=False,
    na=False,
)

status_contains_conflict_word = status.str.contains(
    "conflict",
    case=False,
    na=False,
)

explanation_contains_conflict_word = explanation.str.contains(
    "conflict",
    case=False,
    na=False,
)

flag_true = observed_flag.eq(True)
flag_false = observed_flag.eq(False)

# ---------------------------------------------------------------------
# 3. High-level diagnostic counts
# ---------------------------------------------------------------------

print("\nKEY COUNTS")
print("-" * 92)
print(f"Rows loaded:                                      {len(df_conflict):,}")
print(f"Observed aggregate_conflict_flag = True:          {flag_true.sum():,}")
print(f"Classification group = Conflicting:               {group_conflicting.sum():,}")
print(f"Raw classification contains 'conflicting':        {raw_classification_conflicting.sum():,}")
print(f"Review status = conflicting interpretations:      {status_conflicting.sum():,}")
print(f"Review status = multiple submitters, no conflicts:{status_no_conflicts.sum():,}")
print(f"Review status merely contains word 'conflict':    {status_contains_conflict_word.sum():,}")
print(f"SCV group disagreement = True:                    {scv_disagreement.eq(True).sum():,}")

# ---------------------------------------------------------------------
# 4. Critical no-conflicts test
# ---------------------------------------------------------------------

no_conflicts_flagged_true = (
    status_no_conflicts
    & flag_true
)

no_conflicts_flagged_false = (
    status_no_conflicts
    & flag_false
)

print("\n'NO CONFLICTS' STATUS TEST")
print("-" * 92)
print(
    "Rows with status 'multiple submitters, no conflicts' "
    f"and conflict flag=True:  {no_conflicts_flagged_true.sum():,}"
)
print(
    "Rows with status 'multiple submitters, no conflicts' "
    f"and conflict flag=False: {no_conflicts_flagged_false.sum():,}"
)

# ---------------------------------------------------------------------
# 5. Cross-tabulation of review status and conflict flag
# ---------------------------------------------------------------------

cross_tab = pd.crosstab(
    status.fillna("<missing>"),
    observed_flag.astype("string").fillna("<missing>"),
    margins=True,
)

print("\nREVIEW STATUS × OBSERVED CONFLICT FLAG")
print("-" * 92)
print(cross_tab.to_string())

# ---------------------------------------------------------------------
# 6. Test likely parser rules
# ---------------------------------------------------------------------

# Possible incorrect rule:
# mark conflict when either:
#   a) normalized classification is conflicting, OR
#   b) review status contains the substring "conflict"
#
# This would incorrectly include "no conflicts".

candidate_substring_rule = (
    group_conflicting
    | status_contains_conflict_word
)

substring_rule_mismatch = (
    observed_flag.notna()
    & observed_flag.ne(candidate_substring_rule)
)

print("\nCANDIDATE SUBSTRING-RULE TEST")
print("-" * 92)
print(
    "Expected True under candidate substring rule:  "
    f"{candidate_substring_rule.sum():,}"
)
print(
    "Observed aggregate conflict=True:              "
    f"{flag_true.sum():,}"
)
print(
    "Rows where observed flag differs from rule:    "
    f"{substring_rule_mismatch.sum():,}"
)

if substring_rule_mismatch.sum() == 0:
    print(
        "RESULT: The observed flag exactly matches the rule:\n"
        "        classification_group == 'Conflicting'\n"
        "        OR review_status contains the substring 'conflict'."
    )
else:
    print(
        "RESULT: The observed flag is not fully explained by the "
        "candidate substring rule."
    )

# ---------------------------------------------------------------------
# 7. Test a stricter semantic conflict rule
# ---------------------------------------------------------------------

# Provisional semantic rule:
# - aggregate classification is explicitly Conflicting, OR
# - review status explicitly says conflicting interpretations.
#
# SCV disagreement is retained separately and should not automatically
# be merged into the aggregate conflict flag without a frozen definition.

strict_semantic_conflict = (
    group_conflicting
    | status_conflicting
)

likely_false_positive_mask = (
    flag_true
    & ~strict_semantic_conflict
)

likely_false_negative_mask = (
    flag_false
    & strict_semantic_conflict
)

print("\nPROVISIONAL STRICT-SEMANTIC COMPARISON")
print("-" * 92)
print(
    "True under strict semantic rule:               "
    f"{strict_semantic_conflict.sum():,}"
)
print(
    "Observed True but strict rule=False:            "
    f"{likely_false_positive_mask.sum():,}"
)
print(
    "Observed False but strict rule=True:            "
    f"{likely_false_negative_mask.sum():,}"
)

# ---------------------------------------------------------------------
# 8. Breakdown of observed True records
# ---------------------------------------------------------------------

diagnostic_categories = pd.DataFrame(
    {
        "observed_flag_true": flag_true,
        "group_conflicting": group_conflicting,
        "status_conflicting": status_conflicting,
        "status_no_conflicts": status_no_conflicts,
        "scv_disagreement_true": scv_disagreement.eq(True),
        "explanation_mentions_conflict": explanation_contains_conflict_word,
    }
)

true_breakdown = (
    diagnostic_categories.loc[flag_true]
    .groupby(
        [
            "group_conflicting",
            "status_conflicting",
            "status_no_conflicts",
            "scv_disagreement_true",
        ],
        dropna=False,
    )
    .size()
    .reset_index(name="record_count")
    .sort_values("record_count", ascending=False)
)

print("\nBREAKDOWN OF OBSERVED CONFLICT-FLAG=TRUE RECORDS")
print("-" * 92)
print(true_breakdown.to_string(index=False))

# ---------------------------------------------------------------------
# 9. Display representative suspected false positives
# ---------------------------------------------------------------------

suspected_examples = df_conflict.loc[
    likely_false_positive_mask,
    columns,
].head(20)

print("\nFIRST 20 OBSERVED-TRUE / STRICT-RULE-FALSE RECORDS")
print("-" * 92)

if len(suspected_examples) > 0:
    print(suspected_examples.to_string(index=False))
else:
    print("None found.")

# ---------------------------------------------------------------------
# 10. Diagnostic conclusion
# ---------------------------------------------------------------------

print("\nDIAGNOSTIC STATUS")
print("-" * 92)

if (
    no_conflicts_flagged_true.sum() == status_no_conflicts.sum()
    and status_no_conflicts.sum() > 0
):
    print(
        "LIKELY SEMANTIC BUG — records explicitly labeled "
        "'no conflicts' are marked conflict=True."
    )
    print(
        "Do not freeze the T0 extraction or delete the raw XML yet."
    )
elif likely_false_positive_mask.sum() > 0:
    print(
        "REVIEW REQUIRED — some conflict=True records are not "
        "explained by the strict semantic definition."
    )
else:
    print(
        "PASS — no evidence was found that 'no conflicts' records "
        "were incorrectly marked conflict-positive."
    )

T0 AGGREGATE-CONFLICT FLAG DIAGNOSTIC

KEY COUNTS
--------------------------------------------------------------------------------------------
Rows loaded:                                      71,659
Observed aggregate_conflict_flag = True:          10,708
Classification group = Conflicting:               1,480
Raw classification contains 'conflicting':        1,480
Review status = conflicting interpretations:      1,394
Review status = multiple submitters, no conflicts:9,224
Review status merely contains word 'conflict':    10,618
SCV group disagreement = True:                    2,465

'NO CONFLICTS' STATUS TEST
--------------------------------------------------------------------------------------------
Rows with status 'multiple submitters, no conflicts' and conflict flag=True:  9,224
Rows with status 'multiple submitters, no conflicts' and conflict flag=False: 0

REVIEW STATUS × OBSERVED CONFLICT FLAG
---------------------------------------------------------------------------------

In [22]:
# STEP 7: Locate the code that created aggregate_conflict_flag
# Read-only diagnostic — no files will be modified.

from pathlib import Path
import json
import re

SEARCH_ROOTS = [
    Path("/content/genomic-evidence-reliability"),
    Path("/content/drive/MyDrive/GES_RAG_Temporal_Study"),
    Path("/content/drive/MyDrive/Colab Notebooks"),
]

SEARCH_TERMS = [
    "aggregate_conflict_flag",
    "contains('conflict",
    'contains("conflict',
    "'conflict' in",
    '"conflict" in',
    "conflict_flag",
]

TEXT_EXTENSIONS = {
    ".py",
    ".txt",
    ".md",
    ".json",
    ".yaml",
    ".yml",
}

MAX_SNIPPET_LINES = 12
MAX_MATCHES = 100

print("=" * 96)
print("SEARCHING FOR THE AGGREGATE-CONFLICT PARSER LOGIC")
print("=" * 96)

existing_roots = []

for root in SEARCH_ROOTS:
    if root.exists():
        existing_roots.append(root)
        print(f"Searching: {root}")
    else:
        print(f"Not found: {root}")

if not existing_roots:
    raise FileNotFoundError(
        "None of the expected repository or notebook directories were found."
    )

matches_found = 0

# ---------------------------------------------------------------------
# Helper: print source-code context around matching lines
# ---------------------------------------------------------------------

def print_text_context(file_path, text, matched_line_numbers):
    global matches_found

    lines = text.splitlines()

    for line_number in sorted(set(matched_line_numbers)):
        matches_found += 1

        if matches_found > MAX_MATCHES:
            return

        start = max(0, line_number - 4)
        end = min(len(lines), line_number + 7)

        print("\n" + "=" * 96)
        print(f"FILE: {file_path}")
        print(f"MATCH NEAR LINE: {line_number + 1}")
        print("-" * 96)

        for index in range(start, end):
            marker = ">>" if index == line_number else "  "
            print(f"{marker} {index + 1:05d}: {lines[index]}")


# ---------------------------------------------------------------------
# Search ordinary text/code files
# ---------------------------------------------------------------------

for root in existing_roots:
    for file_path in root.rglob("*"):
        if matches_found >= MAX_MATCHES:
            break

        if not file_path.is_file():
            continue

        if file_path.suffix.lower() not in TEXT_EXTENSIONS:
            continue

        try:
            text = file_path.read_text(
                encoding="utf-8",
                errors="ignore",
            )
        except Exception:
            continue

        lower_text = text.lower()

        if not any(term.lower() in lower_text for term in SEARCH_TERMS):
            continue

        lines = text.splitlines()
        matched_lines = []

        for index, line in enumerate(lines):
            line_lower = line.lower()

            if any(term.lower() in line_lower for term in SEARCH_TERMS):
                matched_lines.append(index)

        print_text_context(
            file_path=file_path,
            text=text,
            matched_line_numbers=matched_lines,
        )

# ---------------------------------------------------------------------
# Search notebook code cells separately
# ---------------------------------------------------------------------

for root in existing_roots:
    for notebook_path in root.rglob("*.ipynb"):
        if matches_found >= MAX_MATCHES:
            break

        try:
            notebook = json.loads(
                notebook_path.read_text(
                    encoding="utf-8",
                    errors="ignore",
                )
            )
        except Exception:
            continue

        for cell_index, cell in enumerate(
            notebook.get("cells", [])
        ):
            if cell.get("cell_type") != "code":
                continue

            source = cell.get("source", [])

            if isinstance(source, list):
                source_text = "".join(source)
            else:
                source_text = str(source)

            source_lower = source_text.lower()

            if not any(
                term.lower() in source_lower
                for term in SEARCH_TERMS
            ):
                continue

            matches_found += 1

            print("\n" + "=" * 96)
            print(f"NOTEBOOK: {notebook_path}")
            print(f"CODE CELL INDEX: {cell_index}")
            print("-" * 96)

            source_lines = source_text.splitlines()

            for line_number, line in enumerate(
                source_lines,
                start=1,
            ):
                marker = (
                    ">>"
                    if any(
                        term.lower() in line.lower()
                        for term in SEARCH_TERMS
                    )
                    else "  "
                )

                print(f"{marker} {line_number:04d}: {line}")

            if matches_found >= MAX_MATCHES:
                break

# ---------------------------------------------------------------------
# Final status
# ---------------------------------------------------------------------

print("\n" + "=" * 96)
print("SEARCH STATUS")
print("-" * 96)

if matches_found == 0:
    print(
        "No matching saved code was found.\n"
        "The extraction cell may exist only in the currently open, "
        "unsaved Colab notebook."
    )
else:
    print(f"Candidate code locations found: {matches_found}")
    print(
        "Do not edit anything yet. Copy the output containing the "
        "aggregate_conflict_flag assignment."
    )

if matches_found >= MAX_MATCHES:
    print(
        f"Search stopped after {MAX_MATCHES} matches to avoid excessive output."
    )

SEARCHING FOR THE AGGREGATE-CONFLICT PARSER LOGIC
Searching: /content/genomic-evidence-reliability
Searching: /content/drive/MyDrive/GES_RAG_Temporal_Study
Searching: /content/drive/MyDrive/Colab Notebooks

NOTEBOOK: /content/genomic-evidence-reliability/01_evidence_drift_full_pipeline.ipynb
CODE CELL INDEX: 5
------------------------------------------------------------------------------------------------
   0001: # ============================================================
   0002: # STEP 5: Feature Engineering
   0003: # ============================================================
   0004: 
   0005: import numpy as np
   0006: from datetime import datetime
   0007: 
   0008: df_feat = df_filtered.copy()
   0009: 
   0010: # -------------------------------
   0011: # 1. Convert LastEvaluated to datetime
   0012: # -------------------------------
   0013: df_feat["LastEvaluated"] = pd.to_datetime(df_feat["LastEvaluated"], errors="coerce")
   0014: 
   0015: # Current date
   0016: cu

In [23]:
# STEP 7A: Find the exact code assignment that creates aggregate_conflict_flag
# Read-only search. No files are modified.

from pathlib import Path
import json
import re

SEARCH_ROOTS = [
    Path("/content/genomic-evidence-reliability"),
    Path("/content/drive/MyDrive/GES_RAG_Temporal_Study"),
    Path("/content/drive/MyDrive/Colab Notebooks"),
]

# Strong assignment patterns only
ASSIGNMENT_PATTERNS = [
    re.compile(r"""aggregate_conflict_flag\s*=""", re.IGNORECASE),
    re.compile(
        r"""["']aggregate_conflict_flag["']\s*:""",
        re.IGNORECASE,
    ),
    re.compile(
        r"""\[\s*["']aggregate_conflict_flag["']\s*\]\s*=""",
        re.IGNORECASE,
    ),
]

# Exclude the diagnostic/search cells we just created
EXCLUDED_MARKERS = [
    "T0 AGGREGATE-CONFLICT FLAG DIAGNOSTIC",
    "SEARCHING FOR THE AGGREGATE-CONFLICT PARSER LOGIC",
    "STEP 7A:",
]

TEXT_EXTENSIONS = {
    ".py",
    ".txt",
    ".md",
    ".json",
    ".yaml",
    ".yml",
}

candidates = []

print("=" * 100)
print("SEARCHING FOR EXACT aggregate_conflict_flag ASSIGNMENTS")
print("=" * 100)

# ---------------------------------------------------------------------
# Search normal source files
# ---------------------------------------------------------------------

for root in SEARCH_ROOTS:
    if not root.exists():
        continue

    for file_path in root.rglob("*"):
        if not file_path.is_file():
            continue

        if file_path.suffix.lower() not in TEXT_EXTENSIONS:
            continue

        try:
            text = file_path.read_text(
                encoding="utf-8",
                errors="ignore",
            )
        except Exception:
            continue

        if any(marker in text for marker in EXCLUDED_MARKERS):
            continue

        lines = text.splitlines()

        for line_index, line in enumerate(lines):
            if any(pattern.search(line) for pattern in ASSIGNMENT_PATTERNS):
                start = max(0, line_index - 15)
                end = min(len(lines), line_index + 21)

                candidates.append(
                    {
                        "type": "text file",
                        "path": str(file_path),
                        "location": f"line {line_index + 1}",
                        "source_lines": lines[start:end],
                        "highlight_index": line_index - start,
                    }
                )

# ---------------------------------------------------------------------
# Search notebook code cells only
# Notebook outputs are deliberately ignored.
# ---------------------------------------------------------------------

for root in SEARCH_ROOTS:
    if not root.exists():
        continue

    for notebook_path in root.rglob("*.ipynb"):
        try:
            notebook = json.loads(
                notebook_path.read_text(
                    encoding="utf-8",
                    errors="ignore",
                )
            )
        except Exception:
            continue

        for cell_index, cell in enumerate(
            notebook.get("cells", [])
        ):
            if cell.get("cell_type") != "code":
                continue

            source = cell.get("source", [])

            if isinstance(source, list):
                source_text = "".join(source)
            else:
                source_text = str(source)

            if any(
                marker in source_text
                for marker in EXCLUDED_MARKERS
            ):
                continue

            source_lines = source_text.splitlines()

            matching_lines = [
                index
                for index, line in enumerate(source_lines)
                if any(
                    pattern.search(line)
                    for pattern in ASSIGNMENT_PATTERNS
                )
            ]

            for line_index in matching_lines:
                start = max(0, line_index - 20)
                end = min(
                    len(source_lines),
                    line_index + 31,
                )

                candidates.append(
                    {
                        "type": "notebook code cell",
                        "path": str(notebook_path),
                        "location": (
                            f"cell {cell_index}, "
                            f"line {line_index + 1}"
                        ),
                        "source_lines": source_lines[start:end],
                        "highlight_index": line_index - start,
                    }
                )

# ---------------------------------------------------------------------
# Deduplicate identical candidates
# ---------------------------------------------------------------------

unique_candidates = []
seen = set()

for candidate in candidates:
    key = (
        candidate["path"],
        candidate["location"],
        "\n".join(candidate["source_lines"]),
    )

    if key not in seen:
        seen.add(key)
        unique_candidates.append(candidate)

# ---------------------------------------------------------------------
# Print results
# ---------------------------------------------------------------------

if not unique_candidates:
    print(
        "\nNo exact saved assignment was found.\n"
        "The extraction code may exist only in the currently open "
        "unsaved Colab notebook."
    )
else:
    print(
        f"\nExact assignment candidates found: "
        f"{len(unique_candidates)}"
    )

    for candidate_number, candidate in enumerate(
        unique_candidates,
        start=1,
    ):
        print("\n" + "=" * 100)
        print(f"CANDIDATE {candidate_number}")
        print(f"TYPE:     {candidate['type']}")
        print(f"FILE:     {candidate['path']}")
        print(f"LOCATION: {candidate['location']}")
        print("-" * 100)

        for index, line in enumerate(
            candidate["source_lines"]
        ):
            marker = (
                ">>>"
                if index == candidate["highlight_index"]
                else "   "
            )

            print(f"{marker} {line}")

print("\n" + "=" * 100)
print("SEARCH COMPLETE")
print("-" * 100)
print(
    "Paste the complete candidate containing the parser or output-row "
    "construction. Do not edit the code yet."
)

SEARCHING FOR EXACT aggregate_conflict_flag ASSIGNMENTS

Exact assignment candidates found: 2

CANDIDATE 1
TYPE:     notebook code cell
FILE:     /content/drive/MyDrive/Colab Notebooks/02_GES_temporal_validation.ipynb
LOCATION: cell 13, line 862
----------------------------------------------------------------------------------------------------
        submitters = sorted({
            record["submitter"]
            for record in scv_records
            if record["submitter"]
        })
    
        submitter_ids = sorted({
            record["submitter_id"]
            for record in scv_records
            if record["submitter_id"]
        })
    
        aggregate_review_status = (
            aggregate["review_status"]
        )
    
        aggregate_classification = (
            aggregate["classification"]
        )
    
>>>     aggregate_conflict_flag = bool(
            (
                aggregate_review_status
                and "conflict"
                in aggregate_review

In [24]:
# STEP 7B: Dry-run the corrected aggregate-conflict definition
#
# READ-ONLY:
# - does not modify the notebook
# - does not modify the original Parquet
# - does not write a replacement file

from pathlib import Path
import pandas as pd

T0_PARQUET = Path(
    "/content/drive/MyDrive/GES_RAG_Temporal_Study/"
    "data_interim/t0_rcv_target_genes_raw.parquet"
)

columns = [
    "rcv_accession",
    "variation_id",
    "aggregate_classification",
    "aggregate_classification_group",
    "aggregate_review_status",
    "aggregate_review_stars",
    "aggregate_conflict_flag",
    "scv_group_disagreement_flag",
]

df_fix = pd.read_parquet(
    T0_PARQUET,
    columns=columns,
)

print("=" * 94)
print("DRY RUN: CORRECTED T0 AGGREGATE-CONFLICT RULE")
print("=" * 94)

# ---------------------------------------------------------------------
# 1. Normalize text and stored boolean values
# ---------------------------------------------------------------------

def normalize_text(series):
    return (
        series.astype("string")
        .str.strip()
        .str.lower()
        .replace("", pd.NA)
    )


def normalize_boolean(value):
    if pd.isna(value):
        return pd.NA

    if isinstance(value, bool):
        return value

    if isinstance(value, (int, float)) and value in (0, 1):
        return bool(value)

    text = str(value).strip().lower()

    mapping = {
        "true": True,
        "false": False,
        "1": True,
        "0": False,
        "yes": True,
        "no": False,
    }

    return mapping.get(text, pd.NA)


classification = normalize_text(
    df_fix["aggregate_classification"]
)

classification_group = normalize_text(
    df_fix["aggregate_classification_group"]
)

review_status = normalize_text(
    df_fix["aggregate_review_status"]
)

old_conflict_flag = (
    df_fix["aggregate_conflict_flag"]
    .apply(normalize_boolean)
    .astype("boolean")
)

# ---------------------------------------------------------------------
# 2. Correct semantic definition
# ---------------------------------------------------------------------
#
# Conflict-positive when either:
#
# A. The normalized aggregate classification group is explicitly
#    "Conflicting"
#
# OR
#
# B. The review status explicitly states
#    "criteria provided, conflicting interpretations"
#
# The phrase "multiple submitters, no conflicts" is therefore False.
#
# SCV disagreement remains a separate field and is not silently merged
# into aggregate_conflict_flag.

classification_is_conflicting = (
    classification_group.eq("conflicting")
)

review_status_is_conflicting = (
    review_status.eq(
        "criteria provided, conflicting interpretations"
    )
)

corrected_conflict_flag = (
    classification_is_conflicting
    | review_status_is_conflicting
).astype("boolean")

# ---------------------------------------------------------------------
# 3. Compare original and corrected flags
# ---------------------------------------------------------------------

changed_mask = (
    old_conflict_flag.notna()
    & corrected_conflict_flag.notna()
    & old_conflict_flag.ne(corrected_conflict_flag)
)

true_to_false_mask = (
    old_conflict_flag.eq(True)
    & corrected_conflict_flag.eq(False)
)

false_to_true_mask = (
    old_conflict_flag.eq(False)
    & corrected_conflict_flag.eq(True)
)

unchanged_true_mask = (
    old_conflict_flag.eq(True)
    & corrected_conflict_flag.eq(True)
)

unchanged_false_mask = (
    old_conflict_flag.eq(False)
    & corrected_conflict_flag.eq(False)
)

print("\nOLD VERSUS CORRECTED COUNTS")
print("-" * 94)
print(
    f"Rows loaded:                              "
    f"{len(df_fix):,}"
)
print(
    f"Original conflict=True:                   "
    f"{old_conflict_flag.eq(True).sum():,}"
)
print(
    f"Corrected conflict=True:                  "
    f"{corrected_conflict_flag.eq(True).sum():,}"
)
print(
    f"Rows whose flag would change:             "
    f"{changed_mask.sum():,}"
)
print(
    f"True → False corrections:                 "
    f"{true_to_false_mask.sum():,}"
)
print(
    f"False → True corrections:                 "
    f"{false_to_true_mask.sum():,}"
)
print(
    f"Unchanged True records:                   "
    f"{unchanged_true_mask.sum():,}"
)
print(
    f"Unchanged False records:                  "
    f"{unchanged_false_mask.sum():,}"
)

# ---------------------------------------------------------------------
# 4. Explain corrected positive records
# ---------------------------------------------------------------------

both_sources = (
    classification_is_conflicting
    & review_status_is_conflicting
)

classification_only = (
    classification_is_conflicting
    & ~review_status_is_conflicting
)

review_status_only = (
    ~classification_is_conflicting
    & review_status_is_conflicting
)

print("\nCORRECTED CONFLICT-POSITIVE SOURCES")
print("-" * 94)
print(
    f"Classification and review status agree:   "
    f"{both_sources.sum():,}"
)
print(
    f"Classification group only:                "
    f"{classification_only.sum():,}"
)
print(
    f"Review status only:                       "
    f"{review_status_only.sum():,}"
)
print(
    f"Total corrected conflict-positive:        "
    f"{corrected_conflict_flag.eq(True).sum():,}"
)

# ---------------------------------------------------------------------
# 5. Confirm the known erroneous category
# ---------------------------------------------------------------------

no_conflicts_status = review_status.eq(
    "criteria provided, multiple submitters, no conflicts"
)

print("\n'NO CONFLICTS' CATEGORY AFTER CORRECTION")
print("-" * 94)
print(
    f"Rows in category:                         "
    f"{no_conflicts_status.sum():,}"
)
print(
    f"Corrected flag=True in category:          "
    f"{(no_conflicts_status & corrected_conflict_flag.eq(True)).sum():,}"
)
print(
    f"Corrected flag=False in category:         "
    f"{(no_conflicts_status & corrected_conflict_flag.eq(False)).sum():,}"
)

# ---------------------------------------------------------------------
# 6. Relationship with SCV disagreement
# ---------------------------------------------------------------------

scv_disagreement = (
    df_fix["scv_group_disagreement_flag"]
    .apply(normalize_boolean)
    .astype("boolean")
)

print("\nSCV DISAGREEMENT RELATIONSHIP")
print("-" * 94)
print(
    f"SCV disagreement=True:                    "
    f"{scv_disagreement.eq(True).sum():,}"
)
print(
    f"Corrected conflict=True and disagreement=True: "
    f"{(corrected_conflict_flag.eq(True) & scv_disagreement.eq(True)).sum():,}"
)
print(
    f"Corrected conflict=False but disagreement=True:"
    f" {(corrected_conflict_flag.eq(False) & scv_disagreement.eq(True)).sum():,}"
)

# ---------------------------------------------------------------------
# 7. Display representative changed records
# ---------------------------------------------------------------------

comparison = df_fix.copy()

comparison["old_conflict_flag"] = old_conflict_flag
comparison["corrected_conflict_flag"] = corrected_conflict_flag

changed_examples = comparison.loc[
    changed_mask,
    [
        "rcv_accession",
        "variation_id",
        "aggregate_classification",
        "aggregate_classification_group",
        "aggregate_review_status",
        "aggregate_review_stars",
        "old_conflict_flag",
        "corrected_conflict_flag",
        "scv_group_disagreement_flag",
    ],
].head(20)

print("\nFIRST 20 RECORDS THAT WOULD CHANGE")
print("-" * 94)

if len(changed_examples) > 0:
    print(changed_examples.to_string(index=False))
else:
    print("No records would change.")

# ---------------------------------------------------------------------
# 8. Dry-run acceptance checks
# ---------------------------------------------------------------------

expected_corrected_true = 1_484
expected_true_to_false = 9_224
expected_false_to_true = 0

checks = {
    "all rows retained":
        len(df_fix) == 71_659,

    "no missing old flags":
        old_conflict_flag.isna().sum() == 0,

    "no missing corrected flags":
        corrected_conflict_flag.isna().sum() == 0,

    "corrected conflict count":
        int(corrected_conflict_flag.eq(True).sum())
        == expected_corrected_true,

    "expected True-to-False corrections":
        int(true_to_false_mask.sum())
        == expected_true_to_false,

    "no False-to-True corrections":
        int(false_to_true_mask.sum())
        == expected_false_to_true,

    "no-conflicts category fully corrected":
        int(
            (
                no_conflicts_status
                & corrected_conflict_flag.eq(True)
            ).sum()
        ) == 0,
}

print("\nDRY-RUN CHECKS")
print("-" * 94)

for check_name, passed in checks.items():
    print(
        f"{check_name:45s}: "
        f"{'PASS' if passed else 'FAIL'}"
    )

failed_checks = [
    name
    for name, passed in checks.items()
    if not passed
]

print("\nDRY-RUN STATUS")
print("-" * 94)

if not failed_checks:
    print(
        "PASS — the corrected semantic rule produces the "
        "expected 1,484 conflict-positive records."
    )
    print(
        "Exactly 9,224 erroneous True values would be changed "
        "to False, with no False-to-True changes."
    )
    print(
        "No file has been modified. The original Parquet remains intact."
    )
else:
    print(
        "STOP — the proposed correction did not reproduce "
        "the expected diagnostic counts."
    )
    print("Failed checks:")

    for check in failed_checks:
        print(" -", check)

DRY RUN: CORRECTED T0 AGGREGATE-CONFLICT RULE

OLD VERSUS CORRECTED COUNTS
----------------------------------------------------------------------------------------------
Rows loaded:                              71,659
Original conflict=True:                   10,708
Corrected conflict=True:                  1,484
Rows whose flag would change:             9,224
True → False corrections:                 9,224
False → True corrections:                 0
Unchanged True records:                   1,484
Unchanged False records:                  60,951

CORRECTED CONFLICT-POSITIVE SOURCES
----------------------------------------------------------------------------------------------
Classification and review status agree:   1,390
Classification group only:                90
Review status only:                       4
Total corrected conflict-positive:        1,484

'NO CONFLICTS' CATEGORY AFTER CORRECTION
----------------------------------------------------------------------------------------

In [25]:
# STEP 7C: Create a corrected, versioned T0 Parquet artifact
#
# The original file is preserved unchanged.
# A new Parquet and correction manifest are created.

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import shutil

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

# ---------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------

STUDY_ROOT = Path(
    "/content/drive/MyDrive/GES_RAG_Temporal_Study"
)

ORIGINAL_PARQUET = (
    STUDY_ROOT
    / "data_interim"
    / "t0_rcv_target_genes_raw.parquet"
)

CORRECTED_PARQUET = (
    STUDY_ROOT
    / "data_interim"
    / "t0_rcv_target_genes_corrected_v1_1.parquet"
)

TEMP_PARQUET = CORRECTED_PARQUET.with_suffix(
    ".parquet.tmp"
)

MANIFEST_PATH = (
    STUDY_ROOT
    / "configs"
    / "clinvar_t0_conflict_flag_correction_manifest_v1_1.json"
)

REPO_CONFIG_PATH = Path(
    "/content/genomic-evidence-reliability/configs/"
    "clinvar_t0_conflict_flag_correction_manifest_v1_1.json"
)

EXPECTED_ORIGINAL_SHA256 = (
    "663ad4c20ce114a763dc6512ff95951f95c2e8448c20a6c4ccb5818486c25f8d"
)

EXPECTED_ROWS = 71_659
EXPECTED_OLD_TRUE = 10_708
EXPECTED_NEW_TRUE = 1_484
EXPECTED_TRUE_TO_FALSE = 9_224
EXPECTED_FALSE_TO_TRUE = 0

print("=" * 96)
print("CREATING CORRECTED T0 PARQUET — VERSION 1.1")
print("=" * 96)

# ---------------------------------------------------------------------
# Helper: SHA-256
# ---------------------------------------------------------------------

def calculate_sha256(path, block_size=1024 * 1024):
    digest = hashlib.sha256()

    with path.open("rb") as file:
        for block in iter(
            lambda: file.read(block_size),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()

# ---------------------------------------------------------------------
# 1. Validate original artifact
# ---------------------------------------------------------------------

if not ORIGINAL_PARQUET.exists():
    raise FileNotFoundError(
        f"Original Parquet was not found:\n{ORIGINAL_PARQUET}"
    )

original_sha256 = calculate_sha256(
    ORIGINAL_PARQUET
)

print("\nORIGINAL ARTIFACT")
print("-" * 96)
print("Path:", ORIGINAL_PARQUET)
print("SHA-256:", original_sha256)

if original_sha256 != EXPECTED_ORIGINAL_SHA256:
    raise RuntimeError(
        "STOP — the original Parquet checksum does not match "
        "the verified extraction artifact."
    )

if CORRECTED_PARQUET.exists():
    raise FileExistsError(
        "A corrected file already exists:\n"
        f"{CORRECTED_PARQUET}\n\n"
        "Do not overwrite it. Stop and inspect the existing file."
    )

if TEMP_PARQUET.exists():
    TEMP_PARQUET.unlink()

# ---------------------------------------------------------------------
# 2. Read original table
# ---------------------------------------------------------------------

table = pq.read_table(
    ORIGINAL_PARQUET
)

if table.num_rows != EXPECTED_ROWS:
    raise RuntimeError(
        f"Unexpected row count: {table.num_rows:,}"
    )

required_columns = {
    "rcv_accession",
    "aggregate_classification_group",
    "aggregate_review_status",
    "aggregate_conflict_flag",
}

missing_columns = (
    required_columns
    - set(table.column_names)
)

if missing_columns:
    raise KeyError(
        f"Missing required columns: {sorted(missing_columns)}"
    )

# ---------------------------------------------------------------------
# 3. Calculate corrected semantic conflict flag
# ---------------------------------------------------------------------

classification_group = (
    table
    .column("aggregate_classification_group")
    .to_pandas()
    .astype("string")
    .str.strip()
    .str.lower()
)

review_status = (
    table
    .column("aggregate_review_status")
    .to_pandas()
    .astype("string")
    .str.strip()
    .str.lower()
)

old_conflict_flag = (
    table
    .column("aggregate_conflict_flag")
    .to_pandas()
    .astype("boolean")
)

classification_is_conflicting = (
    classification_group.eq("conflicting")
)

review_status_is_conflicting = (
    review_status.eq(
        "criteria provided, conflicting interpretations"
    )
)

corrected_conflict_flag = (
    classification_is_conflicting
    | review_status_is_conflicting
).astype("boolean")

true_to_false_mask = (
    old_conflict_flag.eq(True)
    & corrected_conflict_flag.eq(False)
)

false_to_true_mask = (
    old_conflict_flag.eq(False)
    & corrected_conflict_flag.eq(True)
)

changed_mask = (
    old_conflict_flag.ne(
        corrected_conflict_flag
    )
)

old_true_count = int(
    old_conflict_flag.eq(True).sum()
)

new_true_count = int(
    corrected_conflict_flag.eq(True).sum()
)

true_to_false_count = int(
    true_to_false_mask.sum()
)

false_to_true_count = int(
    false_to_true_mask.sum()
)

changed_count = int(
    changed_mask.sum()
)

print("\nCORRECTION COUNTS")
print("-" * 96)
print(f"Rows:                         {table.num_rows:,}")
print(f"Original conflict=True:       {old_true_count:,}")
print(f"Corrected conflict=True:      {new_true_count:,}")
print(f"Rows changed:                 {changed_count:,}")
print(f"True → False:                 {true_to_false_count:,}")
print(f"False → True:                 {false_to_true_count:,}")

prewrite_checks = {
    "expected original True count":
        old_true_count == EXPECTED_OLD_TRUE,

    "expected corrected True count":
        new_true_count == EXPECTED_NEW_TRUE,

    "expected True-to-False count":
        true_to_false_count
        == EXPECTED_TRUE_TO_FALSE,

    "expected False-to-True count":
        false_to_true_count
        == EXPECTED_FALSE_TO_TRUE,

    "no missing corrected values":
        corrected_conflict_flag.isna().sum() == 0,
}

failed_prewrite_checks = [
    name
    for name, passed in prewrite_checks.items()
    if not passed
]

if failed_prewrite_checks:
    raise RuntimeError(
        "STOP — correction counts did not match the "
        "validated dry run:\n- "
        + "\n- ".join(failed_prewrite_checks)
    )

# ---------------------------------------------------------------------
# 4. Replace only the derived conflict column
# ---------------------------------------------------------------------

column_index = table.schema.get_field_index(
    "aggregate_conflict_flag"
)

corrected_arrow_column = pa.array(
    corrected_conflict_flag.tolist(),
    type=pa.bool_(),
)

corrected_table = table.set_column(
    column_index,
    pa.field(
        "aggregate_conflict_flag",
        pa.bool_(),
    ),
    corrected_arrow_column,
)

# Confirm that no column other than the intended one was removed
if corrected_table.column_names != table.column_names:
    raise RuntimeError(
        "Column order changed unexpectedly."
    )

if corrected_table.num_rows != table.num_rows:
    raise RuntimeError(
        "Row count changed unexpectedly."
    )

# ---------------------------------------------------------------------
# 5. Write to a temporary file first
# ---------------------------------------------------------------------

pq.write_table(
    corrected_table,
    TEMP_PARQUET,
    compression="zstd",
    use_dictionary=True,
)

if not TEMP_PARQUET.exists():
    raise RuntimeError(
        "Temporary corrected Parquet was not created."
    )

# ---------------------------------------------------------------------
# 6. Reopen and validate the written file
# ---------------------------------------------------------------------

written_table = pq.read_table(
    TEMP_PARQUET,
    columns=[
        "rcv_accession",
        "aggregate_classification_group",
        "aggregate_review_status",
        "aggregate_conflict_flag",
    ],
)

written_flags = (
    written_table
    .column("aggregate_conflict_flag")
    .to_pandas()
    .astype("boolean")
)

written_rcvs = (
    written_table
    .column("rcv_accession")
    .to_pandas()
)

validation_checks = {
    "row count preserved":
        written_table.num_rows == EXPECTED_ROWS,

    "unique RCV count preserved":
        written_rcvs.nunique(dropna=True)
        == EXPECTED_ROWS,

    "corrected True count preserved":
        int(written_flags.eq(True).sum())
        == EXPECTED_NEW_TRUE,

    "corrected False count":
        int(written_flags.eq(False).sum())
        == EXPECTED_ROWS - EXPECTED_NEW_TRUE,

    "no missing corrected flags":
        int(written_flags.isna().sum()) == 0,

    "column count preserved":
        corrected_table.num_columns
        == table.num_columns,
}

print("\nPOST-WRITE VALIDATION")
print("-" * 96)

for check_name, passed in validation_checks.items():
    print(
        f"{check_name:45s}: "
        f"{'PASS' if passed else 'FAIL'}"
    )

failed_validation_checks = [
    name
    for name, passed in validation_checks.items()
    if not passed
]

if failed_validation_checks:
    TEMP_PARQUET.unlink(missing_ok=True)

    raise RuntimeError(
        "STOP — corrected output failed validation:\n- "
        + "\n- ".join(failed_validation_checks)
    )

# ---------------------------------------------------------------------
# 7. Atomically promote temporary file
# ---------------------------------------------------------------------

TEMP_PARQUET.replace(
    CORRECTED_PARQUET
)

corrected_sha256 = calculate_sha256(
    CORRECTED_PARQUET
)

corrected_size_bytes = (
    CORRECTED_PARQUET.stat().st_size
)

# ---------------------------------------------------------------------
# 8. Create correction manifest
# ---------------------------------------------------------------------

manifest = {
    "manifest_version": "1.1.0",
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "study_phase": (
        "Experiment 1 T0 extraction validation"
    ),

    "unit_of_analysis": (
        "RCV-level variant-condition aggregate"
    ),

    "original_artifact": {
        "path": str(ORIGINAL_PARQUET),
        "sha256": original_sha256,
        "row_count": EXPECTED_ROWS,
        "preserved_unchanged": True,
    },

    "corrected_artifact": {
        "path": str(CORRECTED_PARQUET),
        "sha256": corrected_sha256,
        "row_count": EXPECTED_ROWS,
        "column_count": corrected_table.num_columns,
        "size_bytes": corrected_size_bytes,
    },

    "issue": {
        "field": "aggregate_conflict_flag",
        "type": "semantic substring-matching error",
        "original_rule": (
            "review status or aggregate classification "
            "contained the substring 'conflict'"
        ),
        "problem": (
            "The phrase 'criteria provided, multiple "
            "submitters, no conflicts' was incorrectly "
            "classified as conflict-positive."
        ),
    },

    "corrected_rule": {
        "definition": (
            "aggregate_classification_group equals "
            "'Conflicting' OR aggregate_review_status "
            "equals 'criteria provided, conflicting "
            "interpretations'"
        ),
        "scv_group_disagreement_merged": False,
        "reason": (
            "SCV disagreement remains a separate evidence "
            "field and is not silently combined with the "
            "aggregate conflict indicator."
        ),
    },

    "correction_counts": {
        "original_conflict_true": old_true_count,
        "corrected_conflict_true": new_true_count,
        "true_to_false": true_to_false_count,
        "false_to_true": false_to_true_count,
        "total_rows_changed": changed_count,
    },

    "data_integrity": {
        "rows_added": 0,
        "rows_removed": 0,
        "column_changed": [
            "aggregate_conflict_flag"
        ],
        "raw_evidence_fields_changed": False,
        "identifiers_changed": False,
        "classifications_changed": False,
        "review_statuses_changed": False,
    },

    "validation_checks": {
        key: bool(value)
        for key, value in validation_checks.items()
    },

    "leakage_status": {
        "T1_information_used": False,
        "temporal_outcomes_used": False,
        "GES_model_used": False,
        "threshold_tuning_used": False,
    },
}

MANIFEST_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

MANIFEST_PATH.write_text(
    json.dumps(
        manifest,
        indent=2,
    ),
    encoding="utf-8",
)

manifest_sha256 = calculate_sha256(
    MANIFEST_PATH
)

# Copy the manifest into the cloned repository when available
repo_manifest_copied = False

if REPO_CONFIG_PATH.parent.exists():
    shutil.copy2(
        MANIFEST_PATH,
        REPO_CONFIG_PATH,
    )

    repo_manifest_copied = True

# ---------------------------------------------------------------------
# 9. Final output
# ---------------------------------------------------------------------

print("\nCORRECTED ARTIFACT")
print("-" * 96)
print("Path:", CORRECTED_PARQUET)
print(
    f"Size: {corrected_size_bytes / (1024 ** 2):.3f} MB"
)
print("SHA-256:", corrected_sha256)

print("\nCORRECTION MANIFEST")
print("-" * 96)
print("Path:", MANIFEST_PATH)
print("SHA-256:", manifest_sha256)
print(
    "Copied to repository configs:",
    repo_manifest_copied,
)

print("\nFINAL STATUS")
print("-" * 96)
print(
    "PASS — a corrected, versioned T0 Parquet artifact "
    "has been created and validated."
)
print(
    "The original Parquet remains unchanged as an audit artifact."
)
print(
    "Do not delete the raw T0 XML yet; the saved parser code "
    "must be corrected next."
)

CREATING CORRECTED T0 PARQUET — VERSION 1.1

ORIGINAL ARTIFACT
------------------------------------------------------------------------------------------------
Path: /content/drive/MyDrive/GES_RAG_Temporal_Study/data_interim/t0_rcv_target_genes_raw.parquet
SHA-256: 663ad4c20ce114a763dc6512ff95951f95c2e8448c20a6c4ccb5818486c25f8d

CORRECTION COUNTS
------------------------------------------------------------------------------------------------
Rows:                         71,659
Original conflict=True:       10,708
Corrected conflict=True:      1,484
Rows changed:                 9,224
True → False:                 9,224
False → True:                 0

POST-WRITE VALIDATION
------------------------------------------------------------------------------------------------
row count preserved                          : PASS
unique RCV count preserved                   : PASS
corrected True count preserved               : PASS
corrected False count                        : PASS
no missing 

In [26]:
# STEP 7D: Create a versioned notebook with the conflict-parser bug corrected
#
# The original notebook is preserved unchanged.
# A new notebook named 02_GES_temporal_validation_v1_1.ipynb is created.

from pathlib import Path
from datetime import datetime, timezone
import ast
import hashlib
import json
import shutil

# ---------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------

ORIGINAL_NOTEBOOK = Path(
    "/content/drive/MyDrive/Colab Notebooks/"
    "02_GES_temporal_validation.ipynb"
)

PATCHED_NOTEBOOK = Path(
    "/content/drive/MyDrive/Colab Notebooks/"
    "02_GES_temporal_validation_v1_1.ipynb"
)

STUDY_ROOT = Path(
    "/content/drive/MyDrive/GES_RAG_Temporal_Study"
)

PATCH_MANIFEST = (
    STUDY_ROOT
    / "configs"
    / "t0_parser_conflict_rule_patch_manifest_v1_1.json"
)

REPO_MANIFEST = Path(
    "/content/genomic-evidence-reliability/configs/"
    "t0_parser_conflict_rule_patch_manifest_v1_1.json"
)

print("=" * 100)
print("CREATING VERSIONED NOTEBOOK WITH CORRECTED CONFLICT PARSER")
print("=" * 100)

# ---------------------------------------------------------------------
# Helper: SHA-256
# ---------------------------------------------------------------------

def calculate_sha256(path, block_size=1024 * 1024):
    digest = hashlib.sha256()

    with path.open("rb") as file:
        for block in iter(
            lambda: file.read(block_size),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()

# ---------------------------------------------------------------------
# 1. Validate source notebook
# ---------------------------------------------------------------------

if not ORIGINAL_NOTEBOOK.exists():
    raise FileNotFoundError(
        f"Original notebook was not found:\n{ORIGINAL_NOTEBOOK}"
    )

if PATCHED_NOTEBOOK.exists():
    raise FileExistsError(
        "The patched notebook already exists:\n"
        f"{PATCHED_NOTEBOOK}\n\n"
        "Do not overwrite it. Stop and inspect the existing version."
    )

original_sha256 = calculate_sha256(
    ORIGINAL_NOTEBOOK
)

print("\nORIGINAL NOTEBOOK")
print("-" * 100)
print("Path:", ORIGINAL_NOTEBOOK)
print("SHA-256:", original_sha256)

notebook = json.loads(
    ORIGINAL_NOTEBOOK.read_text(
        encoding="utf-8"
    )
)

# ---------------------------------------------------------------------
# 2. Define the faulty and corrected code blocks
# ---------------------------------------------------------------------

OLD_CODE = """    aggregate_conflict_flag = bool(
        (
            aggregate_review_status
            and "conflict"
            in aggregate_review_status.lower()
        )
        or
        (
            aggregate_classification
            and "conflict"
            in aggregate_classification.lower()
        )
    )
"""

NEW_CODE = """    # Exact semantic matching prevents the phrase
    # "multiple submitters, no conflicts" from being marked positive.
    aggregate_conflict_flag = bool(
        (
            normalize_classification_group(
                aggregate_classification
            ) == "Conflicting"
        )
        or
        (
            isinstance(
                aggregate_review_status,
                str,
            )
            and aggregate_review_status.strip().lower()
            == "criteria provided, conflicting interpretations"
        )
    )
"""

# ---------------------------------------------------------------------
# 3. Locate and patch the exact code block
# ---------------------------------------------------------------------

replacement_locations = []

for cell_index, cell in enumerate(
    notebook.get("cells", [])
):
    if cell.get("cell_type") != "code":
        continue

    source = cell.get("source", [])

    if isinstance(source, list):
        source_text = "".join(source)
    else:
        source_text = str(source)

    occurrence_count = source_text.count(
        OLD_CODE
    )

    if occurrence_count > 0:
        if occurrence_count != 1:
            raise RuntimeError(
                f"Cell {cell_index} contains "
                f"{occurrence_count} copies of the faulty block."
            )

        patched_source = source_text.replace(
            OLD_CODE,
            NEW_CODE,
            1,
        )

        # Confirm that the modified cell remains valid Python
        try:
            ast.parse(patched_source)
        except SyntaxError as error:
            raise RuntimeError(
                "The patched code cell failed Python syntax validation:\n"
                f"{error}"
            ) from error

        cell["source"] = patched_source.splitlines(
            keepends=True
        )

        replacement_locations.append(
            {
                "cell_index": cell_index,
                "old_occurrences": occurrence_count,
            }
        )

if len(replacement_locations) != 1:
    raise RuntimeError(
        "Expected exactly one parser assignment to be patched, "
        f"but found {len(replacement_locations)}."
    )

patched_cell_index = replacement_locations[0][
    "cell_index"
]

print("\nPATCH LOCATION")
print("-" * 100)
print(f"Notebook code-cell index: {patched_cell_index}")
print("Exact faulty blocks replaced: 1")

# ---------------------------------------------------------------------
# 4. Add notebook-level patch metadata
# ---------------------------------------------------------------------

notebook.setdefault("metadata", {})

notebook["metadata"][
    "ges_temporal_validation_patch"
] = {
    "version": "1.1.0",
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "issue": (
        "aggregate_conflict_flag used substring matching and "
        "incorrectly marked 'no conflicts' as conflict-positive"
    ),
    "corrected_definition": (
        "classification group equals Conflicting OR review status "
        "equals criteria provided, conflicting interpretations"
    ),
    "original_notebook": ORIGINAL_NOTEBOOK.name,
}

# ---------------------------------------------------------------------
# 5. Write the patched notebook
# ---------------------------------------------------------------------

PATCHED_NOTEBOOK.write_text(
    json.dumps(
        notebook,
        ensure_ascii=False,
        indent=1,
    ),
    encoding="utf-8",
)

if not PATCHED_NOTEBOOK.exists():
    raise RuntimeError(
        "The patched notebook was not created."
    )

# ---------------------------------------------------------------------
# 6. Reopen and independently validate the saved notebook
# ---------------------------------------------------------------------

saved_notebook = json.loads(
    PATCHED_NOTEBOOK.read_text(
        encoding="utf-8"
    )
)

saved_code_text = "\n".join(
    "".join(cell.get("source", []))
    if isinstance(cell.get("source", []), list)
    else str(cell.get("source", ""))
    for cell in saved_notebook.get("cells", [])
    if cell.get("cell_type") == "code"
)

validation_checks = {
    "faulty exact block removed":
        OLD_CODE not in saved_code_text,

    "corrected classification comparison present":
        '== "Conflicting"' in saved_code_text,

    "exact conflicting-review-status comparison present":
        (
            '== "criteria provided, conflicting interpretations"'
            in saved_code_text
        ),

    "dangerous review-status substring test removed":
        (
            '"conflict"\n            in '
            "aggregate_review_status.lower()"
            not in saved_code_text
        ),

    "notebook remains valid JSON":
        isinstance(saved_notebook, dict),

    "original notebook still exists":
        ORIGINAL_NOTEBOOK.exists(),
}

print("\nSAVED-NOTEBOOK VALIDATION")
print("-" * 100)

for check_name, passed in validation_checks.items():
    print(
        f"{check_name:55s}: "
        f"{'PASS' if passed else 'FAIL'}"
    )

failed_checks = [
    name
    for name, passed in validation_checks.items()
    if not passed
]

if failed_checks:
    PATCHED_NOTEBOOK.unlink(
        missing_ok=True
    )

    raise RuntimeError(
        "Patched notebook failed validation:\n- "
        + "\n- ".join(failed_checks)
    )

patched_sha256 = calculate_sha256(
    PATCHED_NOTEBOOK
)

# ---------------------------------------------------------------------
# 7. Create patch manifest
# ---------------------------------------------------------------------

manifest = {
    "manifest_version": "1.1.0",
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "original_notebook": {
        "path": str(ORIGINAL_NOTEBOOK),
        "sha256": original_sha256,
        "preserved_unchanged": True,
    },

    "patched_notebook": {
        "path": str(PATCHED_NOTEBOOK),
        "sha256": patched_sha256,
        "patched_cell_index": patched_cell_index,
    },

    "issue": {
        "field": "aggregate_conflict_flag",
        "faulty_rule": (
            "Substring 'conflict' found in aggregate review "
            "status or aggregate classification"
        ),
        "false_positive_category": (
            "criteria provided, multiple submitters, no conflicts"
        ),
        "known_false_positive_records": 9224,
    },

    "corrected_rule": {
        "classification_condition": (
            "normalize_classification_group("
            "aggregate_classification) == 'Conflicting'"
        ),
        "review_status_condition": (
            "aggregate_review_status.strip().lower() == "
            "'criteria provided, conflicting interpretations'"
        ),
        "scv_group_disagreement_kept_separate": True,
    },

    "data_changed_by_this_step": False,
    "T1_information_used": False,
    "temporal_outcomes_used": False,
    "GES_model_used": False,

    "validation_checks": {
        name: bool(passed)
        for name, passed in validation_checks.items()
    },
}

PATCH_MANIFEST.parent.mkdir(
    parents=True,
    exist_ok=True,
)

PATCH_MANIFEST.write_text(
    json.dumps(
        manifest,
        indent=2,
    ),
    encoding="utf-8",
)

manifest_sha256 = calculate_sha256(
    PATCH_MANIFEST
)

repo_manifest_copied = False

if REPO_MANIFEST.parent.exists():
    shutil.copy2(
        PATCH_MANIFEST,
        REPO_MANIFEST,
    )
    repo_manifest_copied = True

# ---------------------------------------------------------------------
# 8. Final output
# ---------------------------------------------------------------------

print("\nPATCHED NOTEBOOK")
print("-" * 100)
print("Path:", PATCHED_NOTEBOOK)
print("SHA-256:", patched_sha256)

print("\nPATCH MANIFEST")
print("-" * 100)
print("Path:", PATCH_MANIFEST)
print("SHA-256:", manifest_sha256)
print(
    "Copied to repository configs:",
    repo_manifest_copied,
)

print("\nFINAL STATUS")
print("-" * 100)
print(
    "PASS — the corrected parser has been saved in a new "
    "versioned notebook."
)
print(
    "The original notebook remains unchanged."
)
print(
    "Do not rerun the original extraction cell in the old notebook."
)

CREATING VERSIONED NOTEBOOK WITH CORRECTED CONFLICT PARSER

ORIGINAL NOTEBOOK
----------------------------------------------------------------------------------------------------
Path: /content/drive/MyDrive/Colab Notebooks/02_GES_temporal_validation.ipynb
SHA-256: c8fa3d90c4ec5794636ca51420bb0f4680e7b81689b4436ba9b9b2f0459e41a4

PATCH LOCATION
----------------------------------------------------------------------------------------------------
Notebook code-cell index: 13
Exact faulty blocks replaced: 1

SAVED-NOTEBOOK VALIDATION
----------------------------------------------------------------------------------------------------
faulty exact block removed                             : PASS
corrected classification comparison present            : PASS
exact conflicting-review-status comparison present     : PASS
dangerous review-status substring test removed         : PASS
notebook remains valid JSON                            : PASS
original notebook still exists                       

In [27]:
# STEP 7E: Verify the corrected notebook parser against the corrected T0 Parquet
#
# READ-ONLY:
# - does not modify either notebook
# - does not modify the Parquet
# - does not delete the raw XML

from pathlib import Path
import ast
import hashlib
import json
import re

import pandas as pd

# ---------------------------------------------------------------------
# Paths and expected values
# ---------------------------------------------------------------------

PATCHED_NOTEBOOK = Path(
    "/content/drive/MyDrive/Colab Notebooks/"
    "02_GES_temporal_validation_v1_1.ipynb"
)

CORRECTED_PARQUET = Path(
    "/content/drive/MyDrive/GES_RAG_Temporal_Study/"
    "data_interim/t0_rcv_target_genes_corrected_v1_1.parquet"
)

EXPECTED_ROWS = 71_659
EXPECTED_CONFLICT_TRUE = 1_484
EXPECTED_CONFLICT_FALSE = 70_175

print("=" * 100)
print("CORRECTED PARSER-TO-PARQUET CONSISTENCY VALIDATION")
print("=" * 100)

# ---------------------------------------------------------------------
# Helper
# ---------------------------------------------------------------------

def calculate_sha256(path, block_size=1024 * 1024):
    digest = hashlib.sha256()

    with path.open("rb") as file:
        for block in iter(
            lambda: file.read(block_size),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()


# ---------------------------------------------------------------------
# 1. Confirm both artifacts exist
# ---------------------------------------------------------------------

if not PATCHED_NOTEBOOK.exists():
    raise FileNotFoundError(
        f"Patched notebook not found:\n{PATCHED_NOTEBOOK}"
    )

if not CORRECTED_PARQUET.exists():
    raise FileNotFoundError(
        f"Corrected Parquet not found:\n{CORRECTED_PARQUET}"
    )

notebook_sha256 = calculate_sha256(
    PATCHED_NOTEBOOK
)

parquet_sha256 = calculate_sha256(
    CORRECTED_PARQUET
)

print("\nARTIFACTS")
print("-" * 100)
print("Patched notebook:", PATCHED_NOTEBOOK)
print("Notebook SHA-256:", notebook_sha256)
print()
print("Corrected Parquet:", CORRECTED_PARQUET)
print("Parquet SHA-256:", parquet_sha256)

# ---------------------------------------------------------------------
# 2. Locate the actual parser assignment
# ---------------------------------------------------------------------

notebook = json.loads(
    PATCHED_NOTEBOOK.read_text(
        encoding="utf-8"
    )
)

assignment_pattern = re.compile(
    r"\baggregate_conflict_flag\s*=\s*bool\s*\(",
    re.IGNORECASE,
)

assignment_cells = []

for cell_index, cell in enumerate(
    notebook.get("cells", [])
):
    if cell.get("cell_type") != "code":
        continue

    source = cell.get("source", [])

    if isinstance(source, list):
        source_text = "".join(source)
    else:
        source_text = str(source)

    if assignment_pattern.search(source_text):
        assignment_cells.append(
            {
                "cell_index": cell_index,
                "source": source_text,
            }
        )

print("\nPARSER-CODE LOCATION")
print("-" * 100)
print(
    "Code cells assigning aggregate_conflict_flag:",
    len(assignment_cells),
)

if len(assignment_cells) != 1:
    raise RuntimeError(
        "Expected exactly one parser assignment, "
        f"but found {len(assignment_cells)}."
    )

parser_cell_index = assignment_cells[0][
    "cell_index"
]

parser_source = assignment_cells[0][
    "source"
]

print("Parser code-cell index:", parser_cell_index)

# Confirm the full code cell remains valid Python
try:
    ast.parse(parser_source)
    syntax_valid = True
except SyntaxError as error:
    syntax_valid = False
    print("Syntax error:", error)

# ---------------------------------------------------------------------
# 3. Inspect only the parser assignment cell
# ---------------------------------------------------------------------

has_classification_rule = bool(
    re.search(
        r"""normalize_classification_group\s*\(
             \s*aggregate_classification\s*
             \)\s*==\s*["']Conflicting["']""",
        parser_source,
        flags=re.IGNORECASE | re.VERBOSE,
    )
)

has_exact_review_rule = (
    "criteria provided, conflicting interpretations"
    in parser_source.lower()
)

dangerous_review_substring_rule = bool(
    re.search(
        r"""["']conflict["']\s+
            in\s+
            aggregate_review_status
            \s*\.\s*lower\s*\(\s*\)""",
        parser_source,
        flags=re.IGNORECASE | re.VERBOSE,
    )
)

dangerous_classification_substring_rule = bool(
    re.search(
        r"""["']conflict["']\s+
            in\s+
            aggregate_classification
            \s*\.\s*lower\s*\(\s*\)""",
        parser_source,
        flags=re.IGNORECASE | re.VERBOSE,
    )
)

print("\nPATCHED PARSER-CODE CHECKS")
print("-" * 100)
print(
    f"Parser cell has valid Python syntax:              "
    f"{syntax_valid}"
)
print(
    f"Exact classification-group rule present:          "
    f"{has_classification_rule}"
)
print(
    f"Exact conflicting-review-status rule present:     "
    f"{has_exact_review_rule}"
)
print(
    f"Dangerous review-status substring rule present:   "
    f"{dangerous_review_substring_rule}"
)
print(
    f"Dangerous classification substring rule present:  "
    f"{dangerous_classification_substring_rule}"
)

# ---------------------------------------------------------------------
# 4. Test the intended parser rule with controlled examples
# ---------------------------------------------------------------------

def corrected_parser_rule(
    classification_group,
    review_status,
):
    normalized_group = (
        str(classification_group).strip().lower()
        if classification_group is not None
        else ""
    )

    normalized_status = (
        str(review_status).strip().lower()
        if review_status is not None
        else ""
    )

    return bool(
        normalized_group == "conflicting"
        or normalized_status
        == "criteria provided, conflicting interpretations"
    )


synthetic_tests = {
    "explicit no-conflicts status is False":
        corrected_parser_rule(
            "Benign/Likely benign",
            "criteria provided, multiple submitters, no conflicts",
        ) is False,

    "conflicting classification group is True":
        corrected_parser_rule(
            "Conflicting",
            "no assertion criteria provided",
        ) is True,

    "conflicting review status is True":
        corrected_parser_rule(
            "VUS",
            "criteria provided, conflicting interpretations",
        ) is True,

    "ordinary single-submitter record is False":
        corrected_parser_rule(
            "Pathogenic/Likely pathogenic",
            "criteria provided, single submitter",
        ) is False,
}

print("\nCONTROLLED PARSER TESTS")
print("-" * 100)

for test_name, passed in synthetic_tests.items():
    print(
        f"{test_name:58s}: "
        f"{'PASS' if passed else 'FAIL'}"
    )

# ---------------------------------------------------------------------
# 5. Read corrected Parquet fields
# ---------------------------------------------------------------------

columns = [
    "rcv_accession",
    "aggregate_classification",
    "aggregate_classification_group",
    "aggregate_review_status",
    "aggregate_conflict_flag",
]

df = pd.read_parquet(
    CORRECTED_PARQUET,
    columns=columns,
)

classification_group = (
    df["aggregate_classification_group"]
    .astype("string")
    .str.strip()
    .str.lower()
)

review_status = (
    df["aggregate_review_status"]
    .astype("string")
    .str.strip()
    .str.lower()
)

stored_flag = (
    df["aggregate_conflict_flag"]
    .astype("boolean")
)

# Recalculate using the same frozen semantic rule
recalculated_flag = (
    classification_group.eq("conflicting")
    | review_status.eq(
        "criteria provided, conflicting interpretations"
    )
).astype("boolean")

# ---------------------------------------------------------------------
# 6. Compare every stored value with the parser rule
# ---------------------------------------------------------------------

mismatch_mask = (
    stored_flag.isna()
    | recalculated_flag.isna()
    | stored_flag.ne(recalculated_flag)
)

no_conflicts_status = review_status.eq(
    "criteria provided, multiple submitters, no conflicts"
)

conflicting_status = review_status.eq(
    "criteria provided, conflicting interpretations"
)

conflicting_group = classification_group.eq(
    "conflicting"
)

print("\nFULL-DATASET CONSISTENCY CHECK")
print("-" * 100)
print(f"Rows loaded:                                  {len(df):,}")
print(
    f"Unique RCV accessions:                        "
    f"{df['rcv_accession'].nunique(dropna=True):,}"
)
print(
    f"Missing stored conflict flags:                "
    f"{stored_flag.isna().sum():,}"
)
print(
    f"Stored conflict=True:                         "
    f"{stored_flag.eq(True).sum():,}"
)
print(
    f"Stored conflict=False:                        "
    f"{stored_flag.eq(False).sum():,}"
)
print(
    f"Recalculated conflict=True:                   "
    f"{recalculated_flag.eq(True).sum():,}"
)
print(
    f"Stored-versus-recalculated mismatches:        "
    f"{mismatch_mask.sum():,}"
)

print("\nSEMANTIC CATEGORY CHECKS")
print("-" * 100)
print(
    f"'No conflicts' records:                       "
    f"{no_conflicts_status.sum():,}"
)
print(
    f"'No conflicts' records incorrectly True:      "
    f"{(no_conflicts_status & stored_flag.eq(True)).sum():,}"
)
print(
    f"Conflicting-group records incorrectly False:   "
    f"{(conflicting_group & stored_flag.eq(False)).sum():,}"
)
print(
    f"Conflicting-status records incorrectly False:  "
    f"{(conflicting_status & stored_flag.eq(False)).sum():,}"
)

# ---------------------------------------------------------------------
# 7. Show mismatches only if any exist
# ---------------------------------------------------------------------

if mismatch_mask.any():
    print("\nFIRST 20 PARSER-TO-PARQUET MISMATCHES")
    print("-" * 100)

    mismatch_examples = df.loc[
        mismatch_mask,
        columns,
    ].copy()

    mismatch_examples[
        "recalculated_conflict_flag"
    ] = recalculated_flag.loc[
        mismatch_mask
    ]

    print(
        mismatch_examples
        .head(20)
        .to_string(index=False)
    )
else:
    print(
        "\nNo parser-to-Parquet mismatches were found."
    )

# ---------------------------------------------------------------------
# 8. Final acceptance checks
# ---------------------------------------------------------------------

checks = {
    "one parser assignment found":
        len(assignment_cells) == 1,

    "parser cell syntax valid":
        syntax_valid,

    "correct classification rule present":
        has_classification_rule,

    "correct review-status rule present":
        has_exact_review_rule,

    "dangerous review substring removed":
        not dangerous_review_substring_rule,

    "dangerous classification substring removed":
        not dangerous_classification_substring_rule,

    "all controlled parser tests pass":
        all(synthetic_tests.values()),

    "expected row count":
        len(df) == EXPECTED_ROWS,

    "one unique RCV per row":
        df["rcv_accession"].nunique(
            dropna=True
        ) == EXPECTED_ROWS,

    "no missing stored flags":
        stored_flag.isna().sum() == 0,

    "expected conflict-True count":
        int(stored_flag.eq(True).sum())
        == EXPECTED_CONFLICT_TRUE,

    "expected conflict-False count":
        int(stored_flag.eq(False).sum())
        == EXPECTED_CONFLICT_FALSE,

    "zero parser-output mismatches":
        int(mismatch_mask.sum()) == 0,

    "no-conflicts category remains False":
        int(
            (
                no_conflicts_status
                & stored_flag.eq(True)
            ).sum()
        ) == 0,
}

print("\nFINAL VALIDATION CHECKS")
print("-" * 100)

for check_name, passed in checks.items():
    print(
        f"{check_name:50s}: "
        f"{'PASS' if passed else 'FAIL'}"
    )

failed_checks = [
    name
    for name, passed in checks.items()
    if not passed
]

print("\nFINAL STATUS")
print("-" * 100)

if not failed_checks:
    print(
        "PASS — the patched parser and corrected T0 Parquet "
        "agree across all 71,659 records."
    )
    print(
        "The conflict-field correction is technically validated."
    )
    print(
        "Keep the raw T0 XML until the remaining T0 quality "
        "checks and validation manifest are completed."
    )
else:
    print(
        "STOP — parser-to-Parquet consistency validation failed."
    )
    print("Failed checks:")

    for check in failed_checks:
        print(" -", check)

CORRECTED PARSER-TO-PARQUET CONSISTENCY VALIDATION

ARTIFACTS
----------------------------------------------------------------------------------------------------
Patched notebook: /content/drive/MyDrive/Colab Notebooks/02_GES_temporal_validation_v1_1.ipynb
Notebook SHA-256: 856ca46cedfc10dc17fd1c80b651a742a77827430808c4a862abe1145ba27531

Corrected Parquet: /content/drive/MyDrive/GES_RAG_Temporal_Study/data_interim/t0_rcv_target_genes_corrected_v1_1.parquet
Parquet SHA-256: 99f80e020bf4d8362891d458b0cd758b56732d2d586ee942d7899d61e685aae7

PARSER-CODE LOCATION
----------------------------------------------------------------------------------------------------
Code cells assigning aggregate_conflict_flag: 1
Parser code-cell index: 13

PATCHED PARSER-CODE CHECKS
----------------------------------------------------------------------------------------------------
Parser cell has valid Python syntax:              True
Exact classification-group rule present:          True
Exact conflicting-

In [28]:
# STEP 1: Mount Google Drive and verify the current T0 artifacts

from google.colab import drive
from pathlib import Path
import shutil

drive.mount("/content/drive")

# Main study folders
STUDY_DIR = Path("/content/drive/MyDrive/GES_RAG_Temporal_Study")
DATA_DIR = STUDY_DIR / "data_interim"
CONFIG_DIR = STUDY_DIR / "configs"

# Required corrected artifacts
T0_PARQUET = DATA_DIR / "t0_rcv_target_genes_corrected_v1_1.parquet"
CORRECTION_MANIFEST = (
    CONFIG_DIR / "clinvar_t0_conflict_flag_correction_manifest_v1_1.json"
)
PARSER_PATCH_MANIFEST = (
    CONFIG_DIR / "t0_parser_conflict_rule_patch_manifest_v1_1.json"
)

# Raw XML may disappear whenever the Colab runtime resets
T0_RAW_XML = Path(
    "/content/clinvar_rcv_raw/ClinVarFullRelease_2023-01.xml.gz"
)

files_to_check = {
    "Corrected T0 Parquet": T0_PARQUET,
    "Conflict correction manifest": CORRECTION_MANIFEST,
    "Parser patch manifest": PARSER_PATCH_MANIFEST,
    "Temporary raw T0 XML": T0_RAW_XML,
}

print("=" * 80)
print("GES-RAG TEMPORAL VALIDATION — ENVIRONMENT CHECK")
print("=" * 80)
print(f"Study directory exists: {STUDY_DIR.exists()}")
print(f"Study directory: {STUDY_DIR}")
print()

for label, path in files_to_check.items():
    exists = path.exists()

    if exists:
        size_mb = path.stat().st_size / (1024 ** 2)
        print(f"✅ {label}")
        print(f"   Path: {path}")
        print(f"   Size: {size_mb:,.3f} MB")
    else:
        print(f"❌ {label}")
        print(f"   Expected path: {path}")

    print("-" * 80)

disk = shutil.disk_usage("/content")

print(f"Temporary Colab free space: {disk.free / (1024 ** 3):,.2f} GiB")
print()
print("STEP 1 COMPLETE")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GES-RAG TEMPORAL VALIDATION — ENVIRONMENT CHECK
Study directory exists: True
Study directory: /content/drive/MyDrive/GES_RAG_Temporal_Study

✅ Corrected T0 Parquet
   Path: /content/drive/MyDrive/GES_RAG_Temporal_Study/data_interim/t0_rcv_target_genes_corrected_v1_1.parquet
   Size: 2.864 MB
--------------------------------------------------------------------------------
✅ Conflict correction manifest
   Path: /content/drive/MyDrive/GES_RAG_Temporal_Study/configs/clinvar_t0_conflict_flag_correction_manifest_v1_1.json
   Size: 0.002 MB
--------------------------------------------------------------------------------
✅ Parser patch manifest
   Path: /content/drive/MyDrive/GES_RAG_Temporal_Study/configs/t0_parser_conflict_rule_patch_manifest_v1_1.json
   Size: 0.002 MB
--------------------------------------------------------------------------------
✅ Temporary ra

In [29]:
# STEP 2: Load and verify the corrected T0 Parquet artifact

from pathlib import Path
import pandas as pd
import pyarrow.parquet as pq

T0_PARQUET = Path(
    "/content/drive/MyDrive/GES_RAG_Temporal_Study/"
    "data_interim/t0_rcv_target_genes_corrected_v1_1.parquet"
)

EXPECTED_COLUMNS = [
    "timepoint",
    "release_label",
    "archive_publication_date",
    "embedded_data_cutoff_date",
    "xml_record_index",
    "source_filename",
    "source_sha256",
    "rcv_accession",
    "rcv_version",
    "variation_id",
    "vcv_accession",
    "vcv_version",
    "target_genes_json",
    "study_scope",
    "measure_set_type",
    "measure_types_json",
    "condition_names_json",
    "condition_ids_json",
    "trait_records_json",
    "aggregate_classification",
    "aggregate_classification_group",
    "aggregate_review_status",
    "aggregate_review_stars",
    "aggregate_last_evaluated",
    "aggregate_explanation",
    "aggregate_conflict_flag",
    "scv_group_disagreement_flag",
    "scv_count_xml",
    "unique_submitter_count_xml",
    "submitters_json",
    "submitter_ids_json",
    "scv_classification_counts_json",
    "scv_group_counts_json",
    "scv_records_json",
]

if not T0_PARQUET.exists():
    raise FileNotFoundError(f"Corrected T0 Parquet not found: {T0_PARQUET}")

parquet_file = pq.ParquetFile(T0_PARQUET)
metadata = parquet_file.metadata

df_t0 = pd.read_parquet(T0_PARQUET)

actual_columns = list(df_t0.columns)

missing_columns = sorted(
    set(EXPECTED_COLUMNS) - set(actual_columns)
)

unexpected_columns = sorted(
    set(actual_columns) - set(EXPECTED_COLUMNS)
)

conflict_positive_count = int(
    df_t0["aggregate_conflict_flag"]
    .fillna(False)
    .astype(bool)
    .sum()
)

checks = {
    "Expected row count = 71,659":
        len(df_t0) == 71_659,

    "Expected column count = 34":
        len(actual_columns) == 34,

    "Exact expected column set":
        not missing_columns and not unexpected_columns,

    "Unique RCV accession per row":
        df_t0["rcv_accession"].nunique(dropna=True) == 71_659,

    "No missing RCV accessions":
        df_t0["rcv_accession"].isna().sum() == 0,

    "No missing VariationIDs":
        df_t0["variation_id"].isna().sum() == 0,

    "Corrected conflict-positive count = 1,484":
        conflict_positive_count == 1_484,
}

print("=" * 88)
print("STEP 2 — CORRECTED T0 ARTIFACT STRUCTURE CHECK")
print("=" * 88)

print(f"File: {T0_PARQUET}")
print(f"Rows loaded: {len(df_t0):,}")
print(f"Columns loaded: {len(actual_columns):,}")
print(f"Parquet row groups: {metadata.num_row_groups:,}")
print(
    f"Unique RCV accessions: "
    f"{df_t0['rcv_accession'].nunique(dropna=True):,}"
)
print(
    f"Conflict-positive records: "
    f"{conflict_positive_count:,}"
)
print(
    f"DataFrame memory: "
    f"{df_t0.memory_usage(deep=True).sum() / (1024 ** 2):,.2f} MB"
)

print("\nVALIDATION RESULTS")
print("-" * 88)

for check_name, passed in checks.items():
    status = "✅ PASS" if passed else "❌ FAIL"
    print(f"{status} — {check_name}")

print("\nCOLUMN DIFFERENCES")
print("-" * 88)

print(
    "Missing expected columns:",
    missing_columns if missing_columns else "None"
)

print(
    "Unexpected columns:",
    unexpected_columns if unexpected_columns else "None"
)

print("\nSAMPLE RECORDS")
print("-" * 88)

sample_columns = [
    "rcv_accession",
    "variation_id",
    "target_genes_json",
    "study_scope",
    "aggregate_classification_group",
    "aggregate_review_stars",
    "aggregate_conflict_flag",
    "scv_count_xml",
    "unique_submitter_count_xml",
]

display(df_t0[sample_columns].head(5))

failed_checks = [
    name for name, passed in checks.items()
    if not passed
]

if failed_checks:
    raise AssertionError(
        "STEP 2 FAILED. Do not continue. Failed checks: "
        + "; ".join(failed_checks)
    )

print()
print("✅ STEP 2 COMPLETE")
print("Corrected T0 artifact loaded as: df_t0")
print("No files were modified.")

STEP 2 — CORRECTED T0 ARTIFACT STRUCTURE CHECK
File: /content/drive/MyDrive/GES_RAG_Temporal_Study/data_interim/t0_rcv_target_genes_corrected_v1_1.parquet
Rows loaded: 71,659
Columns loaded: 34
Parquet row groups: 1
Unique RCV accessions: 71,659
Conflict-positive records: 1,484
DataFrame memory: 364.70 MB

VALIDATION RESULTS
----------------------------------------------------------------------------------------
✅ PASS — Expected row count = 71,659
✅ PASS — Expected column count = 34
✅ PASS — Exact expected column set
✅ PASS — Unique RCV accession per row
✅ PASS — No missing RCV accessions
✅ PASS — No missing VariationIDs
✅ PASS — Corrected conflict-positive count = 1,484

COLUMN DIFFERENCES
----------------------------------------------------------------------------------------
Missing expected columns: None
Unexpected columns: None

SAMPLE RECORDS
----------------------------------------------------------------------------------------


,rcv_accession,variation_id,target_genes_json,study_scope,aggregate_classification_group,aggregate_review_stars,aggregate_conflict_flag,scv_count_xml,unique_submitter_count_xml
0,RCV000052656,58866,"[""EGFR""]",exploratory,VUS,1,False,1,1
1,RCV000053439,59596,"[""EGFR""]",exploratory,VUS,1,False,1,1
2,RCV000053440,59597,"[""EGFR""]",exploratory,VUS,1,False,1,1
3,RCV000053441,59598,"[""EGFR""]",exploratory,VUS,1,False,1,1
4,RCV000053532,59680,"[""EGFR""]",exploratory,Pathogenic/Likely pathogenic,1,False,1,1



✅ STEP 2 COMPLETE
Corrected T0 artifact loaded as: df_t0
No files were modified.


In [30]:
# STEP 3: Validate JSON integrity and expected top-level structures

import json
import math
import pandas as pd

if "df_t0" not in globals():
    raise RuntimeError(
        "df_t0 is not loaded. Run Step 2 before running Step 3."
    )

# Each serialized field and its required top-level JSON type
JSON_FIELD_SCHEMA = {
    "target_genes_json": list,
    "measure_types_json": list,
    "condition_names_json": list,
    "condition_ids_json": list,
    "trait_records_json": list,
    "submitters_json": list,
    "submitter_ids_json": list,
    "scv_classification_counts_json": dict,
    "scv_group_counts_json": dict,
    "scv_records_json": list,
}

ALLOWED_TARGET_GENES = {"BRCA1", "BRCA2", "MLH1", "EGFR"}


def is_missing_value(value):
    """Identify missing scalar values without failing on lists or dictionaries."""
    if value is None:
        return True

    if isinstance(value, str):
        return value.strip() == ""

    if isinstance(value, float):
        return math.isnan(value)

    try:
        result = pd.isna(value)
        return bool(result) if isinstance(result, (bool, type(pd.NA))) else False
    except Exception:
        return False


summary_rows = []
error_examples = []

target_gene_invalid_count = 0
target_gene_error_examples = []

print("=" * 96)
print("STEP 3 — JSON FIELD INTEGRITY VALIDATION")
print("=" * 96)

for field, expected_type in JSON_FIELD_SCHEMA.items():

    valid_count = 0
    missing_count = 0
    malformed_count = 0
    wrong_type_count = 0
    empty_count = 0

    for row_index, value in df_t0[field].items():

        if is_missing_value(value):
            missing_count += 1

            if len(error_examples) < 15:
                error_examples.append({
                    "row_index": row_index,
                    "rcv_accession": df_t0.at[row_index, "rcv_accession"],
                    "field": field,
                    "issue": "missing or blank",
                    "value_preview": repr(value)[:150],
                })

            continue

        try:
            # Support either serialized JSON strings or already parsed objects
            if isinstance(value, expected_type):
                parsed_value = value
            elif isinstance(value, str):
                parsed_value = json.loads(value)
            else:
                parsed_value = json.loads(str(value))

        except Exception as exc:
            malformed_count += 1

            if len(error_examples) < 15:
                error_examples.append({
                    "row_index": row_index,
                    "rcv_accession": df_t0.at[row_index, "rcv_accession"],
                    "field": field,
                    "issue": f"malformed JSON: {type(exc).__name__}",
                    "value_preview": repr(value)[:150],
                })

            continue

        if not isinstance(parsed_value, expected_type):
            wrong_type_count += 1

            if len(error_examples) < 15:
                error_examples.append({
                    "row_index": row_index,
                    "rcv_accession": df_t0.at[row_index, "rcv_accession"],
                    "field": field,
                    "issue": (
                        f"wrong top-level type: "
                        f"{type(parsed_value).__name__}; "
                        f"expected {expected_type.__name__}"
                    ),
                    "value_preview": repr(value)[:150],
                })

            continue

        valid_count += 1

        if len(parsed_value) == 0:
            empty_count += 1

        # Additional scientific validation for the target-gene field
        if field == "target_genes_json":
            normalized_genes = {
                str(gene).strip().upper()
                for gene in parsed_value
                if str(gene).strip()
            }

            target_gene_is_valid = (
                len(parsed_value) == 1
                and len(normalized_genes) == 1
                and normalized_genes.issubset(ALLOWED_TARGET_GENES)
            )

            if not target_gene_is_valid:
                target_gene_invalid_count += 1

                if len(target_gene_error_examples) < 10:
                    target_gene_error_examples.append({
                        "row_index": row_index,
                        "rcv_accession": df_t0.at[row_index, "rcv_accession"],
                        "parsed_target_genes": parsed_value,
                    })

    issue_count = missing_count + malformed_count + wrong_type_count

    summary_rows.append({
        "field": field,
        "expected_type": expected_type.__name__,
        "total_rows": len(df_t0),
        "valid_json": valid_count,
        "missing_or_blank": missing_count,
        "malformed_json": malformed_count,
        "wrong_top_level_type": wrong_type_count,
        "empty_valid_structure": empty_count,
        "total_issues": issue_count,
    })

    status = "✅ PASS" if issue_count == 0 else "❌ FAIL"

    print(
        f"{status} — {field}: "
        f"valid={valid_count:,}, "
        f"empty={empty_count:,}, "
        f"missing={missing_count:,}, "
        f"malformed={malformed_count:,}, "
        f"wrong_type={wrong_type_count:,}"
    )

json_validation_summary = pd.DataFrame(summary_rows)

print("\nJSON VALIDATION SUMMARY")
print("-" * 96)
display(json_validation_summary)

print("\nTARGET-GENE STRUCTURE CHECK")
print("-" * 96)
print(
    "Rows not containing exactly one allowed target gene:",
    f"{target_gene_invalid_count:,}"
)

if target_gene_error_examples:
    display(pd.DataFrame(target_gene_error_examples))
else:
    print("✅ Every row contains exactly one of BRCA1, BRCA2, MLH1, or EGFR.")

print("\nJSON ERROR EXAMPLES")
print("-" * 96)

if error_examples:
    display(pd.DataFrame(error_examples))
else:
    print("✅ No missing, malformed, or incorrectly typed JSON values found.")

failed_fields = json_validation_summary.loc[
    json_validation_summary["total_issues"] > 0,
    "field"
].tolist()

if failed_fields or target_gene_invalid_count > 0:
    raise AssertionError(
        "STEP 3 FAILED. Do not continue. "
        f"JSON fields with issues: {failed_fields}; "
        f"invalid target-gene rows: {target_gene_invalid_count:,}"
    )

print()
print("✅ STEP 3 COMPLETE")
print("All serialized JSON fields are syntactically valid.")
print("All fields have the required top-level list or dictionary structure.")
print("No files or dataframe values were modified.")

STEP 3 — JSON FIELD INTEGRITY VALIDATION
✅ PASS — target_genes_json: valid=71,659, empty=0, missing=0, malformed=0, wrong_type=0
✅ PASS — measure_types_json: valid=71,659, empty=0, missing=0, malformed=0, wrong_type=0
✅ PASS — condition_names_json: valid=71,659, empty=0, missing=0, malformed=0, wrong_type=0
✅ PASS — condition_ids_json: valid=71,659, empty=144, missing=0, malformed=0, wrong_type=0
✅ PASS — trait_records_json: valid=71,659, empty=0, missing=0, malformed=0, wrong_type=0
✅ PASS — submitters_json: valid=71,659, empty=0, missing=0, malformed=0, wrong_type=0
✅ PASS — submitter_ids_json: valid=71,659, empty=71,659, missing=0, malformed=0, wrong_type=0
✅ PASS — scv_classification_counts_json: valid=71,659, empty=0, missing=0, malformed=0, wrong_type=0
✅ PASS — scv_group_counts_json: valid=71,659, empty=0, missing=0, malformed=0, wrong_type=0
✅ PASS — scv_records_json: valid=71,659, empty=0, missing=0, malformed=0, wrong_type=0

JSON VALIDATION SUMMARY
--------------------------

,field,expected_type,total_rows,valid_json,missing_or_blank,malformed_json,wrong_top_level_type,empty_valid_structure,total_issues
0,target_genes_json,list,71659,71659,0,0,0,0,0
1,measure_types_json,list,71659,71659,0,0,0,0,0
2,condition_names_json,list,71659,71659,0,0,0,0,0
3,condition_ids_json,list,71659,71659,0,0,0,144,0
4,trait_records_json,list,71659,71659,0,0,0,0,0
5,submitters_json,list,71659,71659,0,0,0,0,0
6,submitter_ids_json,list,71659,71659,0,0,0,71659,0
7,scv_classification_counts_json,dict,71659,71659,0,0,0,0,0
8,scv_group_counts_json,dict,71659,71659,0,0,0,0,0
9,scv_records_json,list,71659,71659,0,0,0,0,0



TARGET-GENE STRUCTURE CHECK
------------------------------------------------------------------------------------------------
Rows not containing exactly one allowed target gene: 0
✅ Every row contains exactly one of BRCA1, BRCA2, MLH1, or EGFR.

JSON ERROR EXAMPLES
------------------------------------------------------------------------------------------------
✅ No missing, malformed, or incorrectly typed JSON values found.

✅ STEP 3 COMPLETE
All serialized JSON fields are syntactically valid.
All fields have the required top-level list or dictionary structure.
No files or dataframe values were modified.


In [31]:
# STEP 4A: Diagnose SCV nested-record structure, counts, and submitter fields
# This is a read-only diagnostic. It does not modify df_t0 or any saved file.

import json
import math
from collections import Counter
import pandas as pd

if "df_t0" not in globals():
    raise RuntimeError(
        "df_t0 is not loaded. Run Step 2 before running Step 4A."
    )


def parse_json_value(value):
    """Return an already parsed value or parse a serialized JSON string."""
    if isinstance(value, str):
        return json.loads(value)
    return value


def normalize_text(value):
    """Normalize a possible text value for nonempty/uniqueness checks."""
    if value is None:
        return None

    text = str(value).strip()

    if not text or text.lower() in {"none", "null", "nan"}:
        return None

    return text


def safe_integer(value):
    """Convert numeric-like values to integer when possible."""
    if value is None:
        return None

    try:
        if pd.isna(value):
            return None
    except Exception:
        pass

    try:
        return int(value)
    except Exception:
        return None


def numeric_dict_sum(dictionary):
    """Sum count values in a JSON dictionary when they are numeric."""
    total = 0
    invalid_values = []

    for key, value in dictionary.items():
        try:
            total += int(value)
        except Exception:
            invalid_values.append((key, value))

    return total, invalid_values


nested_key_presence = Counter()
nested_key_nonempty = Counter()

total_nested_scv_records = 0
rows_processed = 0

scv_length_mismatch_count = 0
submitter_count_mismatch_count = 0
classification_count_mismatch_count = 0
group_count_mismatch_count = 0

invalid_classification_count_values = 0
invalid_group_count_values = 0

scv_length_examples = []
submitter_count_examples = []
classification_count_examples = []
group_count_examples = []

nested_record_samples = []

columns_needed = [
    "rcv_accession",
    "scv_count_xml",
    "unique_submitter_count_xml",
    "submitters_json",
    "submitter_ids_json",
    "scv_classification_counts_json",
    "scv_group_counts_json",
    "scv_records_json",
]

print("=" * 104)
print("STEP 4A — SCV NESTED-RECORD AND COUNT DIAGNOSTIC")
print("=" * 104)

for row in df_t0[columns_needed].itertuples(index=False):

    rows_processed += 1

    rcv = row.rcv_accession
    expected_scv_count = safe_integer(row.scv_count_xml)
    expected_submitter_count = safe_integer(row.unique_submitter_count_xml)

    submitters = parse_json_value(row.submitters_json)
    submitter_ids = parse_json_value(row.submitter_ids_json)
    classification_counts = parse_json_value(
        row.scv_classification_counts_json
    )
    group_counts = parse_json_value(row.scv_group_counts_json)
    scv_records = parse_json_value(row.scv_records_json)

    # ----------------------------------------------------------
    # 1. SCV nested-record count versus scv_count_xml
    # ----------------------------------------------------------
    observed_scv_count = len(scv_records)
    total_nested_scv_records += observed_scv_count

    if observed_scv_count != expected_scv_count:
        scv_length_mismatch_count += 1

        if len(scv_length_examples) < 10:
            scv_length_examples.append({
                "rcv_accession": rcv,
                "scv_count_xml": expected_scv_count,
                "len_scv_records_json": observed_scv_count,
            })

    # ----------------------------------------------------------
    # 2. Submitter-name count versus unique_submitter_count_xml
    # ----------------------------------------------------------
    normalized_submitters = {
        normalized
        for value in submitters
        if (normalized := normalize_text(value)) is not None
    }

    observed_unique_submitters = len(normalized_submitters)

    if observed_unique_submitters != expected_submitter_count:
        submitter_count_mismatch_count += 1

        if len(submitter_count_examples) < 10:
            submitter_count_examples.append({
                "rcv_accession": rcv,
                "unique_submitter_count_xml": expected_submitter_count,
                "unique_names_in_submitters_json":
                    observed_unique_submitters,
                "submitters_json_preview": submitters[:5],
                "submitter_ids_json_preview": submitter_ids[:5],
            })

    # ----------------------------------------------------------
    # 3. Classification-count dictionary versus SCV count
    # ----------------------------------------------------------
    classification_total, invalid_class_values = numeric_dict_sum(
        classification_counts
    )

    invalid_classification_count_values += len(invalid_class_values)

    if classification_total != expected_scv_count:
        classification_count_mismatch_count += 1

        if len(classification_count_examples) < 10:
            classification_count_examples.append({
                "rcv_accession": rcv,
                "scv_count_xml": expected_scv_count,
                "classification_count_sum": classification_total,
                "classification_counts": classification_counts,
            })

    # ----------------------------------------------------------
    # 4. Group-count dictionary versus SCV count
    # ----------------------------------------------------------
    group_total, invalid_group_values = numeric_dict_sum(group_counts)

    invalid_group_count_values += len(invalid_group_values)

    if group_total != expected_scv_count:
        group_count_mismatch_count += 1

        if len(group_count_examples) < 10:
            group_count_examples.append({
                "rcv_accession": rcv,
                "scv_count_xml": expected_scv_count,
                "group_count_sum": group_total,
                "group_counts": group_counts,
            })

    # ----------------------------------------------------------
    # 5. Discover the actual schema of nested SCV records
    # ----------------------------------------------------------
    for record in scv_records:

        if not isinstance(record, dict):
            continue

        if len(nested_record_samples) < 5:
            nested_record_samples.append({
                "rcv_accession": rcv,
                "nested_scv_record": record,
            })

        for key, value in record.items():
            nested_key_presence[key] += 1

            nonempty = False

            if isinstance(value, str):
                nonempty = normalize_text(value) is not None
            elif isinstance(value, (list, dict)):
                nonempty = len(value) > 0
            else:
                try:
                    nonempty = value is not None and not pd.isna(value)
                except Exception:
                    nonempty = value is not None

            if nonempty:
                nested_key_nonempty[key] += 1


key_summary_rows = []

for key in sorted(nested_key_presence):
    present_count = nested_key_presence[key]
    nonempty_count = nested_key_nonempty[key]

    key_summary_rows.append({
        "nested_key": key,
        "records_with_key": present_count,
        "records_with_nonempty_value": nonempty_count,
        "nonempty_percent":
            round((nonempty_count / present_count) * 100, 2)
            if present_count else 0.0,
    })

nested_key_summary = pd.DataFrame(key_summary_rows)

submitter_related_keys = nested_key_summary[
    nested_key_summary["nested_key"]
    .str.contains("submitter|organization|org|id", case=False, regex=True)
].copy()

print(f"Rows processed: {rows_processed:,}")
print(f"Total nested SCV records: {total_nested_scv_records:,}")

print("\nCOUNT-CONSISTENCY RESULTS")
print("-" * 104)

count_results = [
    (
        "SCV nested-record length mismatches",
        scv_length_mismatch_count
    ),
    (
        "Unique submitter-name count mismatches",
        submitter_count_mismatch_count
    ),
    (
        "Classification-count sum mismatches",
        classification_count_mismatch_count
    ),
    (
        "Group-count sum mismatches",
        group_count_mismatch_count
    ),
    (
        "Invalid classification-count values",
        invalid_classification_count_values
    ),
    (
        "Invalid group-count values",
        invalid_group_count_values
    ),
]

for label, count in count_results:
    status = "✅" if count == 0 else "⚠️"
    print(f"{status} {label}: {count:,}")

print("\nNESTED SCV RECORD KEY SUMMARY")
print("-" * 104)
display(nested_key_summary)

print("\nPOSSIBLE SUBMITTER/IDENTIFIER KEYS INSIDE SCV RECORDS")
print("-" * 104)

if len(submitter_related_keys) > 0:
    display(submitter_related_keys)
else:
    print("No submitter- or identifier-related nested keys were detected.")

print("\nSAMPLE NESTED SCV RECORDS")
print("-" * 104)

for number, sample in enumerate(nested_record_samples, start=1):
    print(f"\nSample {number} — {sample['rcv_accession']}")
    print(
        json.dumps(
            sample["nested_scv_record"],
            indent=2,
            ensure_ascii=False
        )[:4000]
    )

diagnostic_examples = [
    ("SCV-length mismatch examples", scv_length_examples),
    ("Submitter-count mismatch examples", submitter_count_examples),
    (
        "Classification-count mismatch examples",
        classification_count_examples
    ),
    ("Group-count mismatch examples", group_count_examples),
]

print("\nMISMATCH EXAMPLES")
print("-" * 104)

any_examples = False

for title, examples in diagnostic_examples:

    if examples:
        any_examples = True
        print(f"\n{title}")
        display(pd.DataFrame(examples))

if not any_examples:
    print("✅ No count mismatch examples were found.")

print()
print("✅ STEP 4A COMPLETE")
print(
    "This was a diagnostic only. No dataframe values or saved files "
    "were modified."
)

STEP 4A — SCV NESTED-RECORD AND COUNT DIAGNOSTIC
Rows processed: 71,659
Total nested SCV records: 100,633

COUNT-CONSISTENCY RESULTS
--------------------------------------------------------------------------------------------------------
✅ SCV nested-record length mismatches: 0
✅ Unique submitter-name count mismatches: 0
✅ Classification-count sum mismatches: 0
✅ Group-count sum mismatches: 0
✅ Invalid classification-count values: 0
✅ Invalid group-count values: 0

NESTED SCV RECORD KEY SUMMARY
--------------------------------------------------------------------------------------------------------


,nested_key,records_with_key,records_with_nonempty_value,nonempty_percent
0,assertion_methods,100633,100633,100.00
1,classification,100633,100633,100.00
2,classification_group,100633,100633,100.00
3,last_evaluated,100633,91286,90.71
4,origins,100633,100633,100.00
5,review_status,100633,100633,100.00
6,scv_accession,100633,100633,100.00
7,scv_version,100633,100633,100.00
8,submitter,100633,100633,100.00
9,submitter_id,100633,0,0.00



POSSIBLE SUBMITTER/IDENTIFIER KEYS INSIDE SCV RECORDS
--------------------------------------------------------------------------------------------------------


,nested_key,records_with_key,records_with_nonempty_value,nonempty_percent
8,submitter,100633,100633,100.0
9,submitter_id,100633,0,0.0



SAMPLE NESTED SCV RECORDS
--------------------------------------------------------------------------------------------------------

Sample 1 — RCV000052656
{
  "assertion_methods": [
    "Microarray",
    "clinical testing"
  ],
  "classification": "Uncertain significance",
  "classification_group": "VUS",
  "last_evaluated": "2011-08-12",
  "origins": [
    "maternal"
  ],
  "review_status": "criteria provided, single submitter",
  "scv_accession": "SCV000080010",
  "scv_version": "5",
  "submitter": "ISCA site 15",
  "submitter_id": null
}

Sample 2 — RCV000053439
{
  "assertion_methods": [
    "Microarray",
    "clinical testing"
  ],
  "classification": "Uncertain significance",
  "classification_group": "VUS",
  "last_evaluated": "2011-08-12",
  "origins": [
    "not provided"
  ],
  "review_status": "criteria provided, single submitter",
  "scv_accession": "SCV000080797",
  "scv_version": "5",
  "submitter": "GeneDx",
  "submitter_id": null
}

Sample 3 — RCV000053440
{
  "assert

In [32]:
# STEP 4B: Diagnose missing condition IDs and submitter IDs
# Read-only diagnostic — no dataframe or saved file is modified.

import json
from collections import Counter, defaultdict
import pandas as pd

if "df_t0" not in globals():
    raise RuntimeError(
        "df_t0 is not loaded. Run Step 2 before running Step 4B."
    )


def parse_json(value):
    """Parse a JSON string or return an already parsed object."""
    if isinstance(value, str):
        return json.loads(value)
    return value


def is_nonempty(value):
    """Determine whether a value contains meaningful information."""
    if value is None:
        return False

    if isinstance(value, str):
        return value.strip().lower() not in {
            "",
            "none",
            "null",
            "nan",
        }

    if isinstance(value, (list, dict)):
        return len(value) > 0

    try:
        return not pd.isna(value)
    except Exception:
        return True


def walk_nested(value, path=""):
    """
    Recursively yield:
    key path, key name, and value
    from nested dictionaries and lists.
    """
    if isinstance(value, dict):
        for key, child in value.items():
            child_path = f"{path}.{key}" if path else str(key)

            yield child_path, str(key), child

            yield from walk_nested(
                child,
                child_path
            )

    elif isinstance(value, list):
        for index, child in enumerate(value):
            child_path = f"{path}[{index}]"

            yield from walk_nested(
                child,
                child_path
            )


# ------------------------------------------------------------
# Part 1: Empty condition_ids_json records
# ------------------------------------------------------------

condition_empty_rows = []

trait_key_presence = Counter()
trait_key_nonempty = Counter()
trait_identifier_paths = Counter()
trait_identifier_examples = defaultdict(list)

# ------------------------------------------------------------
# Part 2: Submitter identifier investigation
# ------------------------------------------------------------

submitter_id_empty_rows = 0

scv_key_presence = Counter()
scv_key_nonempty = Counter()
scv_identifier_paths = Counter()
scv_identifier_examples = defaultdict(list)

rows_with_possible_scv_submitter_id = set()
rows_with_possible_trait_condition_id = set()

print("=" * 108)
print("STEP 4B — CONDITION AND SUBMITTER IDENTIFIER DIAGNOSTIC")
print("=" * 108)

columns_needed = [
    "rcv_accession",
    "condition_names_json",
    "condition_ids_json",
    "trait_records_json",
    "submitters_json",
    "submitter_ids_json",
    "scv_records_json",
]

for row in df_t0[columns_needed].itertuples(index=False):

    rcv = row.rcv_accession

    condition_names = parse_json(row.condition_names_json)
    condition_ids = parse_json(row.condition_ids_json)
    trait_records = parse_json(row.trait_records_json)

    submitters = parse_json(row.submitters_json)
    submitter_ids = parse_json(row.submitter_ids_json)
    scv_records = parse_json(row.scv_records_json)

    # ========================================================
    # CONDITION IDENTIFIER DIAGNOSTIC
    # ========================================================

    condition_ids_empty = len(condition_ids) == 0

    if condition_ids_empty:
        condition_empty_rows.append({
            "rcv_accession": rcv,
            "condition_names": condition_names,
            "trait_record_count": len(trait_records),
            "trait_records_preview":
                json.dumps(
                    trait_records,
                    ensure_ascii=False
                )[:700],
        })

    for path, key, value in walk_nested(trait_records):

        normalized_key = key.lower()

        trait_key_presence[path] += 1

        if is_nonempty(value):
            trait_key_nonempty[path] += 1

        condition_id_keyword = any(
            keyword in normalized_key
            for keyword in [
                "id",
                "identifier",
                "xref",
                "medgen",
                "omim",
                "orphanet",
                "mondo",
                "db",
            ]
        )

        if condition_id_keyword and is_nonempty(value):

            trait_identifier_paths[path] += 1

            if condition_ids_empty:
                rows_with_possible_trait_condition_id.add(rcv)

            if len(trait_identifier_examples[path]) < 3:
                trait_identifier_examples[path].append({
                    "rcv_accession": rcv,
                    "value": value,
                    "condition_ids_json_empty": condition_ids_empty,
                })

    # ========================================================
    # SUBMITTER IDENTIFIER DIAGNOSTIC
    # ========================================================

    if len(submitter_ids) == 0:
        submitter_id_empty_rows += 1

    for path, key, value in walk_nested(scv_records):

        normalized_key = key.lower()

        scv_key_presence[path] += 1

        if is_nonempty(value):
            scv_key_nonempty[path] += 1

        submitter_identifier_keyword = (
            any(
                keyword in normalized_key
                for keyword in [
                    "submitter",
                    "organization",
                    "organisation",
                    "org_id",
                    "orgid",
                ]
            )
            and any(
                keyword in normalized_key
                for keyword in [
                    "id",
                    "identifier",
                    "organization",
                    "organisation",
                ]
            )
        )

        if submitter_identifier_keyword and is_nonempty(value):

            scv_identifier_paths[path] += 1
            rows_with_possible_scv_submitter_id.add(rcv)

            if len(scv_identifier_examples[path]) < 3:
                scv_identifier_examples[path].append({
                    "rcv_accession": rcv,
                    "value": value,
                })


# ------------------------------------------------------------
# Build condition-ID diagnostic tables
# ------------------------------------------------------------

condition_empty_df = pd.DataFrame(condition_empty_rows)

trait_identifier_summary = pd.DataFrame([
    {
        "nested_path": path,
        "records_with_nonempty_value": count,
    }
    for path, count in trait_identifier_paths.most_common()
])

trait_identifier_example_rows = []

for path, examples in trait_identifier_examples.items():
    for example in examples:
        trait_identifier_example_rows.append({
            "nested_path": path,
            **example,
        })

trait_identifier_examples_df = pd.DataFrame(
    trait_identifier_example_rows
)

# ------------------------------------------------------------
# Build submitter-ID diagnostic tables
# ------------------------------------------------------------

scv_identifier_summary = pd.DataFrame([
    {
        "nested_path": path,
        "records_with_nonempty_value": count,
    }
    for path, count in scv_identifier_paths.most_common()
])

scv_identifier_example_rows = []

for path, examples in scv_identifier_examples.items():
    for example in examples:
        scv_identifier_example_rows.append({
            "nested_path": path,
            **example,
        })

scv_identifier_examples_df = pd.DataFrame(
    scv_identifier_example_rows
)


# ------------------------------------------------------------
# Print results
# ------------------------------------------------------------

print("\nCONDITION IDENTIFIER RESULTS")
print("-" * 108)

print(
    "Rows with empty condition_ids_json:",
    f"{len(condition_empty_df):,}"
)

print(
    "Empty-condition-ID rows with a possible identifier "
    "elsewhere in trait_records_json:",
    f"{len(rows_with_possible_trait_condition_id):,}"
)

print(
    "Empty-condition-ID rows without a detected nested identifier:",
    f"{len(condition_empty_df) - len(rows_with_possible_trait_condition_id):,}"
)

print("\nPOSSIBLE CONDITION IDENTIFIER PATHS")
print("-" * 108)

if len(trait_identifier_summary) > 0:
    display(trait_identifier_summary.head(30))
else:
    print("No possible condition-identifier paths were detected.")

print("\nSAMPLE RECORDS WITH EMPTY condition_ids_json")
print("-" * 108)

if len(condition_empty_df) > 0:
    display(
        condition_empty_df[
            [
                "rcv_accession",
                "condition_names",
                "trait_record_count",
                "trait_records_preview",
            ]
        ].head(15)
    )
else:
    print("No empty condition-ID records found.")

print("\nCONDITION IDENTIFIER EXAMPLES")
print("-" * 108)

if len(trait_identifier_examples_df) > 0:
    display(trait_identifier_examples_df.head(30))
else:
    print("No nested condition-identifier examples found.")


print("\nSUBMITTER IDENTIFIER RESULTS")
print("-" * 108)

print(
    "Rows with empty submitter_ids_json:",
    f"{submitter_id_empty_rows:,}"
)

print(
    "Rows with a possible submitter or organization identifier "
    "inside scv_records_json:",
    f"{len(rows_with_possible_scv_submitter_id):,}"
)

print(
    "Rows without any detected nested submitter identifier:",
    f"{len(df_t0) - len(rows_with_possible_scv_submitter_id):,}"
)

print("\nPOSSIBLE SUBMITTER IDENTIFIER PATHS")
print("-" * 108)

if len(scv_identifier_summary) > 0:
    display(scv_identifier_summary.head(30))
else:
    print(
        "No possible submitter or organization identifier paths "
        "were detected inside scv_records_json."
    )

print("\nSUBMITTER IDENTIFIER EXAMPLES")
print("-" * 108)

if len(scv_identifier_examples_df) > 0:
    display(scv_identifier_examples_df.head(30))
else:
    print("No nested submitter identifier examples found.")


print("\nINTERPRETATION FLAGS")
print("-" * 108)

condition_recoverable = (
    len(rows_with_possible_trait_condition_id) > 0
)

submitter_recoverable = (
    len(rows_with_possible_scv_submitter_id) > 0
)

print(
    "Possible condition IDs recoverable from nested traits:",
    "YES" if condition_recoverable else "NO"
)

print(
    "Possible submitter IDs recoverable from nested SCVs:",
    "YES" if submitter_recoverable else "NO"
)

print()
print("✅ STEP 4B COMPLETE")
print(
    "This was a diagnostic only. No dataframe values or saved "
    "artifacts were modified."
)

STEP 4B — CONDITION AND SUBMITTER IDENTIFIER DIAGNOSTIC

CONDITION IDENTIFIER RESULTS
------------------------------------------------------------------------------------------------------------
Rows with empty condition_ids_json: 144
Empty-condition-ID rows with a possible identifier elsewhere in trait_records_json: 0
Empty-condition-ID rows without a detected nested identifier: 144

POSSIBLE CONDITION IDENTIFIER PATHS
------------------------------------------------------------------------------------------------------------


,nested_path,records_with_nonempty_value
0,[0].identifiers,71515
1,[1].identifiers,452
2,[2].identifiers,393
3,[3].identifiers,320
4,[4].identifiers,184
5,[5].identifiers,184
6,[6].identifiers,184
7,[7].identifiers,184
8,[8].identifiers,7



SAMPLE RECORDS WITH EMPTY condition_ids_json
------------------------------------------------------------------------------------------------------------


,rcv_accession,condition_names,trait_record_count,trait_records_preview
0,RCV000052656,[See cases],1,"[{""identifiers"": [], ""names"": [""See cases""]}]"
1,RCV000053439,[See cases],1,"[{""identifiers"": [], ""names"": [""See cases""]}]"
2,RCV000053440,[See cases],1,"[{""identifiers"": [], ""names"": [""See cases""]}]"
3,RCV000053441,[See cases],1,"[{""identifiers"": [], ""names"": [""See cases""]}]"
4,RCV000053532,[See cases],1,"[{""identifiers"": [], ""names"": [""See cases""]}]"
5,RCV000053534,[See cases],1,"[{""identifiers"": [], ""names"": [""See cases""]}]"
6,RCV000136092,[See cases],1,"[{""identifiers"": [], ""names"": [""See cases""]}]"
7,RCV000139830,[See cases],1,"[{""identifiers"": [], ""names"": [""See cases""]}]"
8,RCV000143077,[See cases],1,"[{""identifiers"": [], ""names"": [""See cases""]}]"
9,RCV000445658,[See cases],1,"[{""identifiers"": [], ""names"": [""See cases""]}]"



CONDITION IDENTIFIER EXAMPLES
------------------------------------------------------------------------------------------------------------


,nested_path,rcv_accession,value,condition_ids_json_empty
0,[0].identifiers,RCV000119352,"[Genetic Alliance:Endometrial+cancer/8312, Hum...",False
1,[0].identifiers,RCV000120688,[MedGen:CN169374],False
2,[0].identifiers,RCV000120697,[MedGen:CN169374],False
3,[1].identifiers,RCV001171454,"[CSER _CC_NCGL, University of Washington:UWMG_...",False
4,[1].identifiers,RCV001171455,"[CSER _CC_NCGL, University of Washington:UWMG_...",False
5,[1].identifiers,RCV001535557,"[CSER _CC_NCGL, University of Washington:UWMG_...",False
6,[2].identifiers,RCV000768158,"[GeneReviews:NBK1401, MONDO:MONDO:0011584, Med...",False
7,[2].identifiers,RCV001535782,"[GeneReviews:NBK1401, MONDO:MONDO:0011584, Med...",False
8,[2].identifiers,RCV000515343,"[GeneReviews:NBK1401, MONDO:MONDO:0011584, Med...",False
9,[3].identifiers,RCV000768158,"[Genetic Alliance:Medulloblastoma/4552, Human ...",False



SUBMITTER IDENTIFIER RESULTS
------------------------------------------------------------------------------------------------------------
Rows with empty submitter_ids_json: 71,659
Rows with a possible submitter or organization identifier inside scv_records_json: 0
Rows without any detected nested submitter identifier: 71,659

POSSIBLE SUBMITTER IDENTIFIER PATHS
------------------------------------------------------------------------------------------------------------
No possible submitter or organization identifier paths were detected inside scv_records_json.

SUBMITTER IDENTIFIER EXAMPLES
------------------------------------------------------------------------------------------------------------
No nested submitter identifier examples found.

INTERPRETATION FLAGS
------------------------------------------------------------------------------------------------------------
Possible condition IDs recoverable from nested traits: NO
Possible submitter IDs recoverable from nested SCVs: NO

In [34]:
# STEP 4C: Audit submitter identifiers directly against the raw T0 XML
# Read-only: no source or saved artifact will be modified.

import gzip
import json
from pathlib import Path

import pandas as pd
from lxml import etree

if "df_t0" not in globals():
    raise RuntimeError("Run Step 2 first so that df_t0 is available.")

RAW_XML = Path(
    "/content/clinvar_rcv_raw/ClinVarFullRelease_2023-01.xml.gz"
)

if not RAW_XML.exists():
    raise FileNotFoundError(f"Raw T0 XML not found: {RAW_XML}")


def parse_json(value):
    if isinstance(value, str):
        return json.loads(value)
    return value


def local_name(element):
    return etree.QName(element).localname


def first_accession(element, prefix):
    """Return the first ClinVarAccession beginning with RCV or SCV."""
    for node in element.iter():
        if local_name(node) != "ClinVarAccession":
            continue

        accession = str(
            node.get("Acc")
            or node.get("Accession")
            or ""
        ).strip()

        if accession.upper().startswith(prefix.upper()):
            return accession, dict(node.attrib)

    return None, {}


# ------------------------------------------------------------
# Select a small cross-gene source-audit sample
# ------------------------------------------------------------

audit_df = df_t0[
    [
        "rcv_accession",
        "xml_record_index",
        "target_genes_json",
        "condition_names_json",
        "condition_ids_json",
        "submitters_json",
        "submitter_ids_json",
        "scv_count_xml",
    ]
].copy()

audit_df["gene"] = audit_df["target_genes_json"].apply(
    lambda value: parse_json(value)[0]
)

audit_df["condition_ids_empty"] = audit_df[
    "condition_ids_json"
].apply(lambda value: len(parse_json(value)) == 0)

selected_parts = []

# Earliest record from each gene
for gene in ["EGFR", "BRCA1", "BRCA2", "MLH1"]:
    subset = audit_df[audit_df["gene"] == gene]

    if not subset.empty:
        selected_parts.append(
            subset.nsmallest(1, "xml_record_index")
        )

# Earliest record with missing condition IDs
empty_condition_subset = audit_df[
    audit_df["condition_ids_empty"]
]

if not empty_condition_subset.empty:
    selected_parts.append(
        empty_condition_subset.nsmallest(
            1,
            "xml_record_index"
        )
    )

# Earliest record containing more than one SCV
multi_scv_subset = audit_df[
    audit_df["scv_count_xml"] > 1
]

if not multi_scv_subset.empty:
    selected_parts.append(
        multi_scv_subset.nsmallest(
            1,
            "xml_record_index"
        )
    )

selected_df = (
    pd.concat(selected_parts, ignore_index=True)
    .drop_duplicates(subset=["rcv_accession"])
    .sort_values("xml_record_index")
    .reset_index(drop=True)
)

target_rcvs = set(selected_df["rcv_accession"])

print("=" * 108)
print("STEP 4C — RAW XML SUBMITTER-IDENTIFIER AUDIT")
print("=" * 108)

print("\nSELECTED CROSS-GENE RECORDS")
print("-" * 108)

display(
    selected_df[
        [
            "rcv_accession",
            "gene",
            "xml_record_index",
            "scv_count_xml",
            "condition_ids_empty",
            "submitters_json",
            "submitter_ids_json",
        ]
    ]
)

# ------------------------------------------------------------
# Detect whether the XML uses a namespace
# ------------------------------------------------------------

with gzip.open(RAW_XML, "rb") as xml_stream:
    root_tag = None

    for _, element in etree.iterparse(
        xml_stream,
        events=("start",),
        huge_tree=True,
    ):
        root_tag = element.tag
        break

if root_tag is None:
    raise RuntimeError("Could not read the XML root element.")

if isinstance(root_tag, str) and root_tag.startswith("{"):
    namespace = root_tag.split("}")[0][1:]
    clinvar_set_tag = f"{{{namespace}}}ClinVarSet"
else:
    namespace = None
    clinvar_set_tag = "ClinVarSet"

print("\nXML root tag:", root_tag)
print("ClinVarSet tag used:", clinvar_set_tag)

# ------------------------------------------------------------
# Stream until all selected RCVs are found
# ------------------------------------------------------------

source_results = {}
records_scanned = 0

with gzip.open(RAW_XML, "rb") as xml_stream:

    context = etree.iterparse(
        xml_stream,
        events=("end",),
        tag=clinvar_set_tag,
        huge_tree=True,
        recover=False,
    )

    for _, clinvar_set in context:

        records_scanned += 1

        rcv_accession, rcv_attributes = first_accession(
            clinvar_set,
            "RCV"
        )

        if rcv_accession in target_rcvs:

            scv_records = []

            for node in clinvar_set.iter():

                if local_name(node) != "ClinicalAssertion":
                    continue

                scv_accession, scv_attributes = first_accession(
                    node,
                    "SCV"
                )

                if scv_accession is None:
                    continue

                scv_records.append({
                    "scv_accession": scv_accession,
                    "all_accession_attributes": scv_attributes,
                    "org_id": (
                        scv_attributes.get("OrgID")
                        or scv_attributes.get("OrgId")
                        or scv_attributes.get("OrganizationID")
                    ),
                    "submitter_name": (
                        scv_attributes.get("SubmitterName")
                        or scv_attributes.get("OrgName")
                    ),
                })

            source_results[rcv_accession] = {
                "observed_xml_record_index": records_scanned,
                "rcv_attributes": rcv_attributes,
                "scv_records": scv_records,
            }

            print(
                f"✅ Found {rcv_accession} at XML record "
                f"{records_scanned:,}; SCVs={len(scv_records):,}"
            )

        # Release parsed XML memory
        clinvar_set.clear()

        parent = clinvar_set.getparent()

        if parent is not None:
            while clinvar_set.getprevious() is not None:
                del parent[0]

        if target_rcvs.issubset(source_results.keys()):
            break

        if records_scanned % 250_000 == 0:
            print(
                f"Progress: {records_scanned:,} records scanned; "
                f"{len(source_results)}/{len(target_rcvs)} targets found"
            )

# ------------------------------------------------------------
# Compare source XML with Parquet
# ------------------------------------------------------------

comparison_rows = []

for _, saved_row in selected_df.iterrows():

    rcv = saved_row["rcv_accession"]
    source = source_results.get(rcv)

    if source is None:
        comparison_rows.append({
            "rcv_accession": rcv,
            "gene": saved_row["gene"],
            "found_in_xml": False,
            "saved_scv_count": saved_row["scv_count_xml"],
            "source_scv_count": None,
            "source_org_ids": None,
            "source_submitter_names": None,
            "saved_submitter_ids": parse_json(
                saved_row["submitter_ids_json"]
            ),
        })
        continue

    source_org_ids = sorted({
        str(record["org_id"]).strip()
        for record in source["scv_records"]
        if record["org_id"] is not None
        and str(record["org_id"]).strip()
    })

    source_submitter_names = sorted({
        str(record["submitter_name"]).strip()
        for record in source["scv_records"]
        if record["submitter_name"] is not None
        and str(record["submitter_name"]).strip()
    })

    comparison_rows.append({
        "rcv_accession": rcv,
        "gene": saved_row["gene"],
        "found_in_xml": True,
        "saved_scv_count": int(saved_row["scv_count_xml"]),
        "source_scv_count": len(source["scv_records"]),
        "source_org_ids": source_org_ids,
        "source_submitter_names": source_submitter_names,
        "saved_submitter_ids": parse_json(
            saved_row["submitter_ids_json"]
        ),
    })

comparison_df = pd.DataFrame(comparison_rows)

print("\nSOURCE-TO-PARQUET COMPARISON")
print("-" * 108)
display(comparison_df)

records_with_source_org_ids = int(
    comparison_df["source_org_ids"].apply(
        lambda value: isinstance(value, list) and len(value) > 0
    ).sum()
)

source_scv_count_mismatches = int(
    (
        comparison_df["found_in_xml"]
        & (
            comparison_df["saved_scv_count"]
            != comparison_df["source_scv_count"]
        )
    ).sum()
)

print("\nFINAL SOURCE AUDIT INDICATORS")
print("=" * 108)

print(f"XML records scanned: {records_scanned:,}")
print(
    f"Selected RCVs found: "
    f"{comparison_df['found_in_xml'].sum():,}/"
    f"{len(comparison_df):,}"
)
print(
    "Selected RCVs containing source OrgID values:",
    f"{records_with_source_org_ids:,}"
)
print(
    "Source-versus-saved SCV count mismatches:",
    f"{source_scv_count_mismatches:,}"
)

if records_with_source_org_ids > 0:
    print()
    print("⚠️ PARSER OMISSION DETECTED")
    print(
        "The historical XML contains submitter organization IDs, "
        "but submitter_ids_json is empty in the saved Parquet."
    )
    print(
        "Do not freeze T0 yet. The parser and corrected Parquet "
        "will need a versioned submitter-ID repair."
    )
else:
    print()
    print("ℹ️ No OrgID values were found in this cross-gene source sample.")
    print(
        "The empty submitter_ids_json field may represent historical "
        "source absence, subject to the audit results shown above."
    )

print()
print("✅ STEP 4C COMPLETE")
print("No XML, Parquet, dataframe, or configuration file was modified.")

STEP 4C — RAW XML SUBMITTER-IDENTIFIER AUDIT

SELECTED CROSS-GENE RECORDS
------------------------------------------------------------------------------------------------------------


,rcv_accession,gene,xml_record_index,scv_count_xml,condition_ids_empty,submitters_json,submitter_ids_json
0,RCV000052656,EGFR,10264,1,True,"[""ISCA site 15""]",[]
1,RCV001354365,EGFR,248760,2,False,"[""Department of Pathology and Laboratory Medic...",[]
2,RCV000009926,BRCA2,1250678,1,False,"[""OMIM""]",[]
3,RCV000030989,BRCA1,1251192,1,False,"[""Sharing Clinical Reports Project (SCRP)""]",[]
4,RCV000051097,MLH1,1252512,1,True,"[""ISCA site 4""]",[]



XML root tag: ReleaseSet
ClinVarSet tag used: ClinVarSet
✅ Found RCV000052656 at XML record 10,264; SCVs=0
✅ Found RCV001354365 at XML record 248,760; SCVs=0
Progress: 250,000 records scanned; 2/5 targets found
Progress: 500,000 records scanned; 2/5 targets found
Progress: 750,000 records scanned; 2/5 targets found
Progress: 1,000,000 records scanned; 2/5 targets found
Progress: 1,250,000 records scanned; 2/5 targets found
✅ Found RCV000009926 at XML record 1,250,678; SCVs=0
✅ Found RCV000030989 at XML record 1,251,192; SCVs=0
✅ Found RCV000051097 at XML record 1,252,512; SCVs=0

SOURCE-TO-PARQUET COMPARISON
------------------------------------------------------------------------------------------------------------


,rcv_accession,gene,found_in_xml,saved_scv_count,source_scv_count,source_org_ids,source_submitter_names,saved_submitter_ids
0,RCV000052656,EGFR,True,1,0,[],[],[]
1,RCV001354365,EGFR,True,2,0,[],[],[]
2,RCV000009926,BRCA2,True,1,0,[],[],[]
3,RCV000030989,BRCA1,True,1,0,[],[],[]
4,RCV000051097,MLH1,True,1,0,[],[],[]



FINAL SOURCE AUDIT INDICATORS
XML records scanned: 1,252,512
Selected RCVs found: 5/5
Selected RCVs containing source OrgID values: 0
Source-versus-saved SCV count mismatches: 5

ℹ️ No OrgID values were found in this cross-gene source sample.
The empty submitter_ids_json field may represent historical source absence, subject to the audit results shown above.

✅ STEP 4C COMPLETE
No XML, Parquet, dataframe, or configuration file was modified.


In [35]:
# STEP 4C.1: Inspect the exact historical XML structure for one RCV
# Read-only diagnostic. No files or dataframe values are modified.

import gzip
from pathlib import Path
from lxml import etree

RAW_XML = Path(
    "/content/clinvar_rcv_raw/ClinVarFullRelease_2023-01.xml.gz"
)

TARGET_RCV = "RCV000052656"

if not RAW_XML.exists():
    raise FileNotFoundError(f"Raw T0 XML not found: {RAW_XML}")


def local_name(element):
    """Return an XML tag without any namespace."""
    return etree.QName(element).localname


def find_rcv_accession(clinvar_set):
    """Find the first RCV ClinVarAccession in a ClinVarSet."""
    for node in clinvar_set.iter():
        if local_name(node) != "ClinVarAccession":
            continue

        accession = str(
            node.get("Acc")
            or node.get("Accession")
            or ""
        ).strip()

        if accession.startswith("RCV"):
            return accession

    return None


print("=" * 110)
print("STEP 4C.1 — HISTORICAL XML STRUCTURE INSPECTION")
print("=" * 110)
print(f"Target RCV: {TARGET_RCV}")

records_scanned = 0
target_found = False

with gzip.open(RAW_XML, "rb") as xml_stream:

    context = etree.iterparse(
        xml_stream,
        events=("end",),
        tag="ClinVarSet",
        huge_tree=True,
        recover=False,
    )

    for _, clinvar_set in context:

        records_scanned += 1

        rcv_accession = find_rcv_accession(clinvar_set)

        if rcv_accession == TARGET_RCV:

            target_found = True

            print(
                f"\n✅ Found {TARGET_RCV} at XML record "
                f"{records_scanned:,}"
            )

            print("\nDIRECT CHILD ELEMENTS OF ClinVarSet")
            print("-" * 110)

            for child_number, child in enumerate(
                clinvar_set,
                start=1
            ):
                print(
                    f"{child_number}. "
                    f"{local_name(child)} "
                    f"attributes={dict(child.attrib)}"
                )

            print("\nALL ASSERTION-RELATED ELEMENT NAMES")
            print("-" * 110)

            assertion_tags = sorted({
                local_name(node)
                for node in clinvar_set.iter()
                if any(
                    keyword in local_name(node).lower()
                    for keyword in [
                        "assertion",
                        "submission",
                        "submitter",
                    ]
                )
            })

            for tag_name in assertion_tags:
                print(tag_name)

            print("\nALL ClinVarAccession ELEMENTS")
            print("-" * 110)

            accession_number = 0

            for node in clinvar_set.iter():

                if local_name(node) != "ClinVarAccession":
                    continue

                accession_number += 1

                print(
                    f"{accession_number}. "
                    f"attributes={dict(node.attrib)}"
                )

            print("\nELEMENTS WITH ORGANIZATION OR SUBMITTER ATTRIBUTES")
            print("-" * 110)

            relevant_attribute_count = 0

            for node in clinvar_set.iter():

                attributes = dict(node.attrib)

                relevant_attributes = {
                    key: value
                    for key, value in attributes.items()
                    if any(
                        keyword in key.lower()
                        for keyword in [
                            "org",
                            "submit",
                            "organization",
                            "organisation",
                        ]
                    )
                }

                if relevant_attributes:
                    relevant_attribute_count += 1

                    print(
                        f"{local_name(node)}: "
                        f"{relevant_attributes}"
                    )

            if relevant_attribute_count == 0:
                print(
                    "No organization- or submitter-related "
                    "attributes were detected in this record."
                )

            print("\nFULL TAG FREQUENCY FOR THIS RECORD")
            print("-" * 110)

            tag_counts = {}

            for node in clinvar_set.iter():
                tag_name = local_name(node)
                tag_counts[tag_name] = (
                    tag_counts.get(tag_name, 0) + 1
                )

            for tag_name, count in sorted(
                tag_counts.items(),
                key=lambda item: (-item[1], item[0])
            ):
                print(f"{tag_name}: {count}")

            break

        clinvar_set.clear()

        parent = clinvar_set.getparent()

        if parent is not None:
            while clinvar_set.getprevious() is not None:
                del parent[0]

        if records_scanned % 5_000 == 0:
            print(
                f"Progress: {records_scanned:,} records scanned"
            )

if not target_found:
    raise AssertionError(
        f"{TARGET_RCV} was not found in the historical XML."
    )

print()
print("✅ STEP 4C.1 COMPLETE")
print("No XML, Parquet, dataframe, or configuration file was modified.")

STEP 4C.1 — HISTORICAL XML STRUCTURE INSPECTION
Target RCV: RCV000052656
Progress: 5,000 records scanned
Progress: 10,000 records scanned

✅ Found RCV000052656 at XML record 10,264

DIRECT CHILD ELEMENTS OF ClinVarSet
--------------------------------------------------------------------------------------------------------------
1. RecordStatus attributes={}
2. Title attributes={}
3. ReferenceClinVarAssertion attributes={'ID': '140125', 'DateLastUpdated': '2022-04-23', 'DateCreated': '2013-08-03'}
4. ClinVarAssertion attributes={'ID': '135979', 'SubmissionName': 'nstd101'}

ALL ASSERTION-RELATED ELEMENT NAMES
--------------------------------------------------------------------------------------------------------------
AdditionalSubmitters
Assertion
ClinVarAssertion
ClinVarSubmissionID
ReferenceClinVarAssertion
SubmitterDescription

ALL ClinVarAccession ELEMENTS
--------------------------------------------------------------------------------------------------------------
1. attributes={'A

In [36]:
# STEP 4C.2: Corrected source audit using historical ClinVarAssertion elements
# Read-only. This does not modify the XML, Parquet, dataframe, or manifests.

import gzip
import json
from pathlib import Path

import pandas as pd
from lxml import etree

if "df_t0" not in globals():
    raise RuntimeError("Run Step 2 first so df_t0 is available.")

RAW_XML = Path(
    "/content/clinvar_rcv_raw/ClinVarFullRelease_2023-01.xml.gz"
)

if not RAW_XML.exists():
    raise FileNotFoundError(f"Raw XML not found: {RAW_XML}")


def parse_json(value):
    if isinstance(value, str):
        return json.loads(value)
    return value


def local_name(element):
    return etree.QName(element).localname


def clean_text(value):
    if value is None:
        return None

    value = str(value).strip()
    return value if value else None


def find_accession(element, prefix):
    """
    Find the first ClinVarAccession descendant with the requested
    accession prefix and return both accession and attributes.
    """
    for node in element.iter():

        if local_name(node) != "ClinVarAccession":
            continue

        accession = clean_text(
            node.get("Acc")
            or node.get("Accession")
        )

        if accession and accession.upper().startswith(prefix.upper()):
            return accession, dict(node.attrib)

    return None, {}


def find_primary_submission_name(clinvar_assertion):
    """
    Historical schema stores the submitted organization/name in
    ClinVarSubmissionID/@submitter.
    """
    for node in clinvar_assertion.iter():

        if local_name(node) != "ClinVarSubmissionID":
            continue

        name = clean_text(
            node.get("submitter")
            or node.get("Submitter")
        )

        if name:
            return name

    return None


def collect_additional_submitters(clinvar_assertion):
    """
    Keep additional submitter metadata separate from the primary
    SCV organization identifier.
    """
    additional = []

    for node in clinvar_assertion.iter():

        if local_name(node) != "SubmitterDescription":
            continue

        additional.append({
            "submitter_name": clean_text(
                node.get("SubmitterName")
                or node.get("submitterName")
            ),
            "org_id": clean_text(
                node.get("OrgID")
                or node.get("OrgId")
            ),
            "all_attributes": dict(node.attrib),
        })

    return additional


# ------------------------------------------------------------------
# Select reproducible records:
# earliest per gene, one multi-SCV record, one missing-condition-ID row
# ------------------------------------------------------------------

audit_frame = df_t0[
    [
        "rcv_accession",
        "xml_record_index",
        "target_genes_json",
        "condition_ids_json",
        "submitters_json",
        "submitter_ids_json",
        "scv_count_xml",
        "scv_records_json",
    ]
].copy()

audit_frame["gene"] = audit_frame[
    "target_genes_json"
].apply(lambda value: parse_json(value)[0])

audit_frame["condition_ids_empty"] = audit_frame[
    "condition_ids_json"
].apply(lambda value: len(parse_json(value)) == 0)

selected_parts = []

for gene in ["EGFR", "BRCA1", "BRCA2", "MLH1"]:

    subset = audit_frame[
        audit_frame["gene"] == gene
    ].sort_values("xml_record_index")

    if not subset.empty:
        selected_parts.append(subset.head(1))

multi_scv_subset = audit_frame[
    audit_frame["scv_count_xml"] > 1
].sort_values("xml_record_index")

if not multi_scv_subset.empty:
    selected_parts.append(multi_scv_subset.head(1))

missing_condition_subset = audit_frame[
    audit_frame["condition_ids_empty"]
].sort_values("xml_record_index")

if not missing_condition_subset.empty:
    selected_parts.append(missing_condition_subset.head(1))

selected_df = (
    pd.concat(selected_parts, ignore_index=True)
    .drop_duplicates(subset=["rcv_accession"])
    .sort_values("xml_record_index")
    .reset_index(drop=True)
)

target_rcvs = set(selected_df["rcv_accession"])

print("=" * 112)
print("STEP 4C.2 — CORRECTED HISTORICAL XML SOURCE AUDIT")
print("=" * 112)

print("\nSELECTED RECORDS")
print("-" * 112)

display(
    selected_df[
        [
            "rcv_accession",
            "gene",
            "xml_record_index",
            "scv_count_xml",
            "condition_ids_empty",
            "submitters_json",
            "submitter_ids_json",
        ]
    ]
)

# ------------------------------------------------------------------
# Stream the XML and correctly inspect direct ClinVarAssertion children
# ------------------------------------------------------------------

source_results = {}
records_scanned = 0

with gzip.open(RAW_XML, "rb") as xml_stream:

    context = etree.iterparse(
        xml_stream,
        events=("end",),
        tag="ClinVarSet",
        huge_tree=True,
        recover=False,
    )

    for _, clinvar_set in context:

        records_scanned += 1

        rcv_accession, _ = find_accession(
            clinvar_set,
            "RCV"
        )

        if rcv_accession in target_rcvs:

            source_scv_records = []

            # Historical XML uses direct ClinVarAssertion children
            for child in clinvar_set:

                if local_name(child) != "ClinVarAssertion":
                    continue

                scv_accession, scv_attributes = find_accession(
                    child,
                    "SCV"
                )

                source_scv_records.append({
                    "scv_accession": scv_accession,
                    "primary_submitter_name":
                        find_primary_submission_name(child),
                    "primary_org_id": clean_text(
                        scv_attributes.get("OrgID")
                        or scv_attributes.get("OrgId")
                    ),
                    "scv_accession_attributes":
                        scv_attributes,
                    "additional_submitters":
                        collect_additional_submitters(child),
                })

            source_results[rcv_accession] = {
                "observed_xml_record_index": records_scanned,
                "scv_records": source_scv_records,
            }

            print(
                f"✅ Found {rcv_accession} at XML record "
                f"{records_scanned:,}; "
                f"ClinVarAssertion/SCV records="
                f"{len(source_scv_records):,}"
            )

        clinvar_set.clear()

        parent = clinvar_set.getparent()

        if parent is not None:
            while clinvar_set.getprevious() is not None:
                del parent[0]

        if target_rcvs.issubset(source_results.keys()):
            break

        if records_scanned % 250_000 == 0:
            print(
                f"Progress: {records_scanned:,} records scanned; "
                f"{len(source_results)}/{len(target_rcvs)} targets found"
            )


# ------------------------------------------------------------------
# Compare corrected source extraction with saved Parquet fields
# ------------------------------------------------------------------

comparison_rows = []
detailed_source_records = []

for _, saved_row in selected_df.iterrows():

    rcv = saved_row["rcv_accession"]
    source = source_results.get(rcv)

    saved_submitters = sorted({
        str(value).strip()
        for value in parse_json(saved_row["submitters_json"])
        if str(value).strip()
    })

    saved_submitter_ids = sorted({
        str(value).strip()
        for value in parse_json(saved_row["submitter_ids_json"])
        if str(value).strip()
    })

    if source is None:
        comparison_rows.append({
            "rcv_accession": rcv,
            "gene": saved_row["gene"],
            "found_in_xml": False,
            "saved_scv_count": int(saved_row["scv_count_xml"]),
            "source_scv_count": None,
            "saved_submitters": saved_submitters,
            "source_primary_submitters": None,
            "submitter_names_match": False,
            "saved_submitter_ids": saved_submitter_ids,
            "source_primary_org_ids": None,
            "submitter_ids_match": False,
        })
        continue

    source_primary_submitters = sorted({
        record["primary_submitter_name"]
        for record in source["scv_records"]
        if record["primary_submitter_name"]
    })

    source_primary_org_ids = sorted({
        record["primary_org_id"]
        for record in source["scv_records"]
        if record["primary_org_id"]
    })

    comparison_rows.append({
        "rcv_accession": rcv,
        "gene": saved_row["gene"],
        "found_in_xml": True,
        "saved_scv_count": int(saved_row["scv_count_xml"]),
        "source_scv_count": len(source["scv_records"]),
        "saved_submitters": saved_submitters,
        "source_primary_submitters": source_primary_submitters,
        "submitter_names_match":
            saved_submitters == source_primary_submitters,
        "saved_submitter_ids": saved_submitter_ids,
        "source_primary_org_ids": source_primary_org_ids,
        "submitter_ids_match":
            saved_submitter_ids == source_primary_org_ids,
    })

    for source_record in source["scv_records"]:
        detailed_source_records.append({
            "rcv_accession": rcv,
            "gene": saved_row["gene"],
            "scv_accession":
                source_record["scv_accession"],
            "primary_submitter_name":
                source_record["primary_submitter_name"],
            "primary_org_id":
                source_record["primary_org_id"],
            "additional_submitters":
                source_record["additional_submitters"],
        })

comparison_df = pd.DataFrame(comparison_rows)
source_detail_df = pd.DataFrame(detailed_source_records)

print("\nSOURCE-TO-PARQUET COMPARISON")
print("-" * 112)
display(comparison_df)

print("\nSOURCE SCV DETAILS")
print("-" * 112)
display(source_detail_df)

found_count = int(comparison_df["found_in_xml"].sum())

scv_count_mismatches = int(
    (
        comparison_df["found_in_xml"]
        & (
            comparison_df["saved_scv_count"]
            != comparison_df["source_scv_count"]
        )
    ).sum()
)

submitter_name_mismatches = int(
    (
        comparison_df["found_in_xml"]
        & ~comparison_df["submitter_names_match"]
    ).sum()
)

records_with_source_org_ids = int(
    comparison_df["source_primary_org_ids"].apply(
        lambda value:
            isinstance(value, list) and len(value) > 0
    ).sum()
)

submitter_id_mismatches = int(
    (
        comparison_df["found_in_xml"]
        & ~comparison_df["submitter_ids_match"]
    ).sum()
)

print("\nFINAL CORRECTED SOURCE AUDIT INDICATORS")
print("=" * 112)

print(f"XML records scanned: {records_scanned:,}")
print(
    f"Selected RCVs found: "
    f"{found_count:,}/{len(comparison_df):,}"
)
print(
    "Source-versus-saved SCV count mismatches:",
    f"{scv_count_mismatches:,}"
)
print(
    "Source-versus-saved primary submitter-name mismatches:",
    f"{submitter_name_mismatches:,}"
)
print(
    "Selected RCVs with primary source OrgID values:",
    f"{records_with_source_org_ids:,}"
)
print(
    "Source-versus-saved submitter-ID mismatches:",
    f"{submitter_id_mismatches:,}"
)

if (
    scv_count_mismatches == 0
    and submitter_name_mismatches == 0
    and records_with_source_org_ids > 0
    and submitter_id_mismatches > 0
):
    print()
    print("⚠️ VERSIONED PARSER REPAIR REQUIRED")
    print(
        "SCV counts and primary submitter names match the source, "
        "but primary SCV OrgID values were omitted from "
        "submitter_ids_json."
    )
    print(
        "The appropriate repair source is "
        "ClinVarAssertion/ClinVarAccession[@Type='SCV']/@OrgID."
    )
else:
    print()
    print(
        "Review the comparison table before deciding whether "
        "a parser repair is required."
    )

print()
print("✅ STEP 4C.2 COMPLETE")
print("No XML, Parquet, dataframe, or manifest was modified.")

STEP 4C.2 — CORRECTED HISTORICAL XML SOURCE AUDIT

SELECTED RECORDS
----------------------------------------------------------------------------------------------------------------


,rcv_accession,gene,xml_record_index,scv_count_xml,condition_ids_empty,submitters_json,submitter_ids_json
0,RCV000052656,EGFR,10264,1,True,"[""ISCA site 15""]",[]
1,RCV001354365,EGFR,248760,2,False,"[""Department of Pathology and Laboratory Medic...",[]
2,RCV000009926,BRCA2,1250678,1,False,"[""OMIM""]",[]
3,RCV000030989,BRCA1,1251192,1,False,"[""Sharing Clinical Reports Project (SCRP)""]",[]
4,RCV000051097,MLH1,1252512,1,True,"[""ISCA site 4""]",[]


✅ Found RCV000052656 at XML record 10,264; ClinVarAssertion/SCV records=1
✅ Found RCV001354365 at XML record 248,760; ClinVarAssertion/SCV records=2
Progress: 250,000 records scanned; 2/5 targets found
Progress: 500,000 records scanned; 2/5 targets found
Progress: 750,000 records scanned; 2/5 targets found
Progress: 1,000,000 records scanned; 2/5 targets found
Progress: 1,250,000 records scanned; 2/5 targets found
✅ Found RCV000009926 at XML record 1,250,678; ClinVarAssertion/SCV records=1
✅ Found RCV000030989 at XML record 1,251,192; ClinVarAssertion/SCV records=1
✅ Found RCV000051097 at XML record 1,252,512; ClinVarAssertion/SCV records=1

SOURCE-TO-PARQUET COMPARISON
----------------------------------------------------------------------------------------------------------------


,rcv_accession,gene,found_in_xml,saved_scv_count,source_scv_count,saved_submitters,source_primary_submitters,submitter_names_match,saved_submitter_ids,source_primary_org_ids,submitter_ids_match
0,RCV000052656,EGFR,True,1,1,[ISCA site 15],[ISCA site 15],True,[],[505241],False
1,RCV001354365,EGFR,True,2,2,[Department of Pathology and Laboratory Medici...,[Department of Pathology and Laboratory Medici...,True,[],"[26957, 506152]",False
2,RCV000009926,BRCA2,True,1,1,[OMIM],[OMIM],True,[],[3],False
3,RCV000030989,BRCA1,True,1,1,[Sharing Clinical Reports Project (SCRP)],[Sharing Clinical Reports Project (SCRP)],True,[],[500037],False
4,RCV000051097,MLH1,True,1,1,[ISCA site 4],[ISCA site 4],True,[],[505240],False



SOURCE SCV DETAILS
----------------------------------------------------------------------------------------------------------------


,rcv_accession,gene,scv_accession,primary_submitter_name,primary_org_id,additional_submitters
0,RCV000052656,EGFR,SCV000080010,ISCA site 15,505241,[{'submitter_name': 'International Standards F...
1,RCV001354365,EGFR,SCV001548964,Department of Pathology and Laboratory Medicin...,506152,"[{'submitter_name': 'Franklin by Genoox', 'org..."
2,RCV001354365,EGFR,SCV001784565,GeneDx,26957,[]
3,RCV000009926,BRCA2,SCV000030147,OMIM,3,[]
4,RCV000030989,BRCA1,SCV000053581,Sharing Clinical Reports Project (SCRP),500037,[]
5,RCV000051097,MLH1,SCV000078437,ISCA site 4,505240,[{'submitter_name': 'International Standards F...



FINAL CORRECTED SOURCE AUDIT INDICATORS
XML records scanned: 1,252,512
Selected RCVs found: 5/5
Source-versus-saved SCV count mismatches: 0
Source-versus-saved primary submitter-name mismatches: 0
Selected RCVs with primary source OrgID values: 5
Source-versus-saved submitter-ID mismatches: 5

⚠️ VERSIONED PARSER REPAIR REQUIRED
SCV counts and primary submitter names match the source, but primary SCV OrgID values were omitted from submitter_ids_json.
The appropriate repair source is ClinVarAssertion/ClinVarAccession[@Type='SCV']/@OrgID.

✅ STEP 4C.2 COMPLETE
No XML, Parquet, dataframe, or manifest was modified.


In [37]:
# STEP 4D: Extract primary SCV OrgIDs for the complete 71,659-record T0 cohort
# Creates a new versioned source-audit mapping and manifest.
# Does NOT modify the existing corrected v1.1 Parquet.

import gzip
import hashlib
import json
import re
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from lxml import etree

if "df_t0" not in globals():
    raise RuntimeError(
        "df_t0 is not loaded. Run Step 2 before Step 4D."
    )

RAW_XML = Path(
    "/content/clinvar_rcv_raw/ClinVarFullRelease_2023-01.xml.gz"
)

STUDY_DIR = Path(
    "/content/drive/MyDrive/GES_RAG_Temporal_Study"
)

DATA_DIR = STUDY_DIR / "data_interim"
CONFIG_DIR = STUDY_DIR / "configs"

ORGID_MAP_PATH = (
    DATA_DIR / "t0_rcv_submitter_orgid_source_map_v1_2.parquet"
)

ORGID_MANIFEST_PATH = (
    CONFIG_DIR /
    "clinvar_t0_submitter_orgid_source_audit_manifest_v1_2.json"
)

if not RAW_XML.exists():
    raise FileNotFoundError(
        f"Raw historical XML not found: {RAW_XML}"
    )

if ORGID_MAP_PATH.exists() or ORGID_MANIFEST_PATH.exists():
    raise FileExistsError(
        "A Step 4D v1.2 output already exists. "
        "Do not overwrite a versioned artifact. "
        "Share the existing file status before rerunning."
    )

DATA_DIR.mkdir(parents=True, exist_ok=True)
CONFIG_DIR.mkdir(parents=True, exist_ok=True)


def parse_json(value):
    """Parse serialized JSON or return an existing Python object."""
    if isinstance(value, str):
        return json.loads(value)
    return value


def local_name(element):
    """Return the XML local tag name without a namespace."""
    return etree.QName(element).localname


def normalize_text(value):
    """Normalize whitespace while preserving the original wording."""
    if value is None:
        return None

    normalized = re.sub(
        r"\s+",
        " ",
        str(value)
    ).strip()

    return normalized if normalized else None


def find_accession(element, accession_type):
    """
    Find a ClinVarAccession descendant having Type='RCV' or Type='SCV'.
    Returns the accession and all source attributes.
    """
    for node in element.iter():

        if local_name(node) != "ClinVarAccession":
            continue

        node_type = normalize_text(node.get("Type"))

        accession = normalize_text(
            node.get("Acc")
            or node.get("Accession")
        )

        if (
            node_type == accession_type
            or (
                accession is not None
                and accession.startswith(accession_type)
            )
        ):
            return accession, dict(node.attrib)

    return None, {}


def find_primary_submitter_name(clinvar_assertion):
    """
    Historical ClinVar XML stores the submitted name in
    ClinVarSubmissionID/@submitter.
    """
    for node in clinvar_assertion.iter():

        if local_name(node) != "ClinVarSubmissionID":
            continue

        submitter_name = normalize_text(
            node.get("submitter")
            or node.get("Submitter")
        )

        if submitter_name:
            return submitter_name

    return None


def sha256_file(path, chunk_size=8 * 1024 * 1024):
    """Calculate SHA-256 without loading the complete file into memory."""
    digest = hashlib.sha256()

    with open(path, "rb") as file_handle:

        while True:
            block = file_handle.read(chunk_size)

            if not block:
                break

            digest.update(block)

    return digest.hexdigest()


# ------------------------------------------------------------------
# Prepare frozen saved-record lookup
# ------------------------------------------------------------------

required_columns = [
    "rcv_accession",
    "xml_record_index",
    "scv_count_xml",
    "submitters_json",
    "submitter_ids_json",
    "source_sha256",
]

missing_required_columns = [
    column
    for column in required_columns
    if column not in df_t0.columns
]

if missing_required_columns:
    raise AssertionError(
        f"Required columns are missing: {missing_required_columns}"
    )

saved_lookup = {}

for row in df_t0[required_columns].itertuples(index=False):

    rcv = normalize_text(row.rcv_accession)

    if rcv in saved_lookup:
        raise AssertionError(
            f"Duplicate saved RCV encountered: {rcv}"
        )

    saved_submitters = sorted({
        normalized
        for value in parse_json(row.submitters_json)
        if (normalized := normalize_text(value)) is not None
    })

    saved_submitter_ids = sorted({
        normalized
        for value in parse_json(row.submitter_ids_json)
        if (normalized := normalize_text(value)) is not None
    })

    saved_lookup[rcv] = {
        "expected_xml_record_index":
            int(row.xml_record_index),

        "saved_scv_count":
            int(row.scv_count_xml),

        "saved_submitters":
            saved_submitters,

        "saved_submitter_ids":
            saved_submitter_ids,
    }

target_rcvs = set(saved_lookup)

if len(target_rcvs) != 71_659:
    raise AssertionError(
        f"Expected 71,659 target RCVs, found {len(target_rcvs):,}."
    )

source_sha_values = sorted({
    normalize_text(value)
    for value in df_t0["source_sha256"]
    if normalize_text(value)
})

if len(source_sha_values) != 1:
    raise AssertionError(
        "Expected exactly one frozen source SHA-256 value, "
        f"found {len(source_sha_values)}."
    )

expected_source_sha256 = source_sha_values[0]

print("=" * 112)
print("STEP 4D — FULL T0 SUBMITTER ORGID SOURCE EXTRACTION")
print("=" * 112)

print(f"Raw XML: {RAW_XML}")
print(f"Target RCVs: {len(target_rcvs):,}")
print(f"Output mapping: {ORGID_MAP_PATH}")
print(f"Output manifest: {ORGID_MANIFEST_PATH}")
print()
print("Beginning complete streaming scan...")


# ------------------------------------------------------------------
# Stream the historical XML
# ------------------------------------------------------------------

mapping_rows = []

records_scanned = 0
target_records_found = 0

xml_index_mismatch_examples = []
scv_count_mismatch_examples = []
submitter_name_mismatch_examples = []
missing_orgid_examples = []

total_source_scvs = 0
source_scvs_with_orgid = 0
source_scvs_without_orgid = 0

with gzip.open(RAW_XML, "rb") as xml_stream:

    context = etree.iterparse(
        xml_stream,
        events=("end",),
        tag="ClinVarSet",
        huge_tree=True,
        recover=False,
    )

    for _, clinvar_set in context:

        records_scanned += 1

        rcv_accession, _ = find_accession(
            clinvar_set,
            "RCV"
        )

        if rcv_accession in target_rcvs:

            target_records_found += 1

            saved = saved_lookup[rcv_accession]

            source_scv_records = []

            for child in clinvar_set:

                if local_name(child) != "ClinVarAssertion":
                    continue

                scv_accession, scv_attributes = find_accession(
                    child,
                    "SCV"
                )

                primary_submitter_name = (
                    find_primary_submitter_name(child)
                )

                primary_org_id = normalize_text(
                    scv_attributes.get("OrgID")
                    or scv_attributes.get("OrgId")
                    or scv_attributes.get("OrganizationID")
                )

                source_scv_records.append({
                    "scv_accession": scv_accession,
                    "primary_submitter_name":
                        primary_submitter_name,
                    "primary_org_id":
                        primary_org_id,
                })

                total_source_scvs += 1

                if primary_org_id:
                    source_scvs_with_orgid += 1
                else:
                    source_scvs_without_orgid += 1

                    if len(missing_orgid_examples) < 20:
                        missing_orgid_examples.append({
                            "rcv_accession": rcv_accession,
                            "scv_accession": scv_accession,
                            "primary_submitter_name":
                                primary_submitter_name,
                        })

            source_primary_submitters = sorted({
                record["primary_submitter_name"]
                for record in source_scv_records
                if record["primary_submitter_name"]
            })

            source_primary_org_ids = sorted({
                record["primary_org_id"]
                for record in source_scv_records
                if record["primary_org_id"]
            })

            observed_scv_count = len(source_scv_records)

            xml_index_matches = (
                records_scanned
                == saved["expected_xml_record_index"]
            )

            scv_count_matches = (
                observed_scv_count
                == saved["saved_scv_count"]
            )

            submitter_names_match = (
                source_primary_submitters
                == saved["saved_submitters"]
            )

            if (
                not xml_index_matches
                and len(xml_index_mismatch_examples) < 20
            ):
                xml_index_mismatch_examples.append({
                    "rcv_accession": rcv_accession,
                    "expected_xml_record_index":
                        saved["expected_xml_record_index"],
                    "observed_xml_record_index":
                        records_scanned,
                })

            if (
                not scv_count_matches
                and len(scv_count_mismatch_examples) < 20
            ):
                scv_count_mismatch_examples.append({
                    "rcv_accession": rcv_accession,
                    "saved_scv_count":
                        saved["saved_scv_count"],
                    "source_scv_count":
                        observed_scv_count,
                })

            if (
                not submitter_names_match
                and len(submitter_name_mismatch_examples) < 20
            ):
                submitter_name_mismatch_examples.append({
                    "rcv_accession": rcv_accession,
                    "saved_submitters":
                        saved["saved_submitters"],
                    "source_primary_submitters":
                        source_primary_submitters,
                })

            missing_orgid_scv_count = sum(
                record["primary_org_id"] is None
                for record in source_scv_records
            )

            mapping_rows.append({
                "rcv_accession":
                    rcv_accession,

                "expected_xml_record_index":
                    saved["expected_xml_record_index"],

                "observed_xml_record_index":
                    records_scanned,

                "xml_record_index_matches":
                    xml_index_matches,

                "saved_scv_count":
                    saved["saved_scv_count"],

                "source_scv_count":
                    observed_scv_count,

                "scv_count_matches":
                    scv_count_matches,

                "saved_submitters_json":
                    json.dumps(
                        saved["saved_submitters"],
                        ensure_ascii=False,
                        separators=(",", ":"),
                    ),

                "source_primary_submitters_json":
                    json.dumps(
                        source_primary_submitters,
                        ensure_ascii=False,
                        separators=(",", ":"),
                    ),

                "submitter_names_match":
                    submitter_names_match,

                "saved_submitter_ids_json":
                    json.dumps(
                        saved["saved_submitter_ids"],
                        ensure_ascii=False,
                        separators=(",", ":"),
                    ),

                "source_primary_org_ids_json":
                    json.dumps(
                        source_primary_org_ids,
                        ensure_ascii=False,
                        separators=(",", ":"),
                    ),

                "source_scv_submitter_org_map_json":
                    json.dumps(
                        source_scv_records,
                        ensure_ascii=False,
                        separators=(",", ":"),
                    ),

                "source_orgid_count":
                    len(source_primary_org_ids),

                "source_scvs_missing_orgid":
                    missing_orgid_scv_count,
            })

        clinvar_set.clear()

        parent = clinvar_set.getparent()

        if parent is not None:
            while clinvar_set.getprevious() is not None:
                del parent[0]

        if records_scanned % 100_000 == 0:
            print(
                f"Progress: {records_scanned:,} XML records scanned; "
                f"{target_records_found:,}/{len(target_rcvs):,} "
                "target RCVs found"
            )


# ------------------------------------------------------------------
# Validate complete extraction before saving
# ------------------------------------------------------------------

mapping_df = pd.DataFrame(mapping_rows)

found_rcvs = set(mapping_df["rcv_accession"])
missing_target_rcvs = sorted(target_rcvs - found_rcvs)
unexpected_rcvs = sorted(found_rcvs - target_rcvs)

duplicate_mapping_rcvs = int(
    mapping_df["rcv_accession"].duplicated().sum()
)

xml_index_mismatch_count = int(
    (~mapping_df["xml_record_index_matches"]).sum()
)

scv_count_mismatch_count = int(
    (~mapping_df["scv_count_matches"]).sum()
)

submitter_name_mismatch_count = int(
    (~mapping_df["submitter_names_match"]).sum()
)

rcvs_with_at_least_one_orgid = int(
    (mapping_df["source_orgid_count"] > 0).sum()
)

rcvs_without_any_orgid = int(
    (mapping_df["source_orgid_count"] == 0).sum()
)

rcvs_with_partial_orgid_missingness = int(
    (
        (mapping_df["source_orgid_count"] > 0)
        & (mapping_df["source_scvs_missing_orgid"] > 0)
    ).sum()
)

all_critical_checks_pass = all([
    records_scanned == 2_302_323,
    len(mapping_df) == 71_659,
    not missing_target_rcvs,
    not unexpected_rcvs,
    duplicate_mapping_rcvs == 0,
    xml_index_mismatch_count == 0,
    scv_count_mismatch_count == 0,
    submitter_name_mismatch_count == 0,
])

print("\nFULL SOURCE-EXTRACTION VALIDATION")
print("=" * 112)

validation_results = [
    (
        "Complete XML record count = 2,302,323",
        records_scanned == 2_302_323,
        records_scanned,
    ),
    (
        "All 71,659 target RCVs extracted",
        len(mapping_df) == 71_659,
        len(mapping_df),
    ),
    (
        "No target RCVs missing",
        len(missing_target_rcvs) == 0,
        len(missing_target_rcvs),
    ),
    (
        "No unexpected RCVs",
        len(unexpected_rcvs) == 0,
        len(unexpected_rcvs),
    ),
    (
        "No duplicate mapping RCVs",
        duplicate_mapping_rcvs == 0,
        duplicate_mapping_rcvs,
    ),
    (
        "All XML record indexes match",
        xml_index_mismatch_count == 0,
        xml_index_mismatch_count,
    ),
    (
        "All source/saved SCV counts match",
        scv_count_mismatch_count == 0,
        scv_count_mismatch_count,
    ),
    (
        "All source/saved submitter-name sets match",
        submitter_name_mismatch_count == 0,
        submitter_name_mismatch_count,
    ),
]

for label, passed, observed in validation_results:
    status = "✅ PASS" if passed else "❌ FAIL"
    print(f"{status} — {label}; observed={observed:,}")

print("\nORGID COVERAGE")
print("-" * 112)

print(f"Total source SCVs: {total_source_scvs:,}")
print(f"Source SCVs with primary OrgID: {source_scvs_with_orgid:,}")
print(f"Source SCVs without primary OrgID: {source_scvs_without_orgid:,}")
print(f"RCVs with at least one primary OrgID: {rcvs_with_at_least_one_orgid:,}")
print(f"RCVs without any primary OrgID: {rcvs_without_any_orgid:,}")
print(
    "RCVs with partial OrgID missingness:",
    f"{rcvs_with_partial_orgid_missingness:,}"
)

if not all_critical_checks_pass:

    print("\nDIAGNOSTIC EXAMPLES")
    print("-" * 112)

    example_groups = [
        ("XML-index mismatches", xml_index_mismatch_examples),
        ("SCV-count mismatches", scv_count_mismatch_examples),
        (
            "Submitter-name mismatches",
            submitter_name_mismatch_examples,
        ),
    ]

    for title, examples in example_groups:

        if examples:
            print(f"\n{title}")
            display(pd.DataFrame(examples))

    raise AssertionError(
        "STEP 4D FAILED. The mapping was not saved because one or "
        "more critical source-consistency checks failed."
    )


# ------------------------------------------------------------------
# Save immutable source mapping and manifest
# ------------------------------------------------------------------

mapping_df = mapping_df.sort_values(
    "expected_xml_record_index"
).reset_index(drop=True)

mapping_df.to_parquet(
    ORGID_MAP_PATH,
    index=False,
    compression="zstd",
)

mapping_sha256 = sha256_file(ORGID_MAP_PATH)

saved_mapping_df = pd.read_parquet(ORGID_MAP_PATH)

saved_mapping_valid = (
    len(saved_mapping_df) == 71_659
    and saved_mapping_df["rcv_accession"].nunique() == 71_659
    and sha256_file(ORGID_MAP_PATH) == mapping_sha256
)

if not saved_mapping_valid:
    raise AssertionError(
        "The saved OrgID mapping failed post-write validation."
    )

manifest = {
    "artifact_type":
        "T0 primary SCV submitter OrgID source-audit mapping",

    "version":
        "1.2.0",

    "created_at_utc":
        datetime.now(timezone.utc).isoformat(),

    "source_xml_path":
        str(RAW_XML),

    "source_xml_sha256_expected":
        expected_source_sha256,

    "source_schema":
        "historical ClinVarFullRelease ClinVarSet/ClinVarAssertion",

    "primary_orgid_source":
        (
            "ClinVarSet/ClinVarAssertion/"
            "ClinVarAccession[@Type='SCV']/@OrgID"
        ),

    "primary_submitter_name_source":
        (
            "ClinVarSet/ClinVarAssertion/"
            "ClinVarSubmissionID/@submitter"
        ),

    "xml_records_scanned":
        records_scanned,

    "target_rcvs":
        len(target_rcvs),

    "mapping_rows":
        len(mapping_df),

    "total_source_scvs":
        total_source_scvs,

    "source_scvs_with_primary_orgid":
        source_scvs_with_orgid,

    "source_scvs_without_primary_orgid":
        source_scvs_without_orgid,

    "rcvs_with_at_least_one_primary_orgid":
        rcvs_with_at_least_one_orgid,

    "rcvs_without_any_primary_orgid":
        rcvs_without_any_orgid,

    "rcvs_with_partial_orgid_missingness":
        rcvs_with_partial_orgid_missingness,

    "xml_index_mismatches":
        xml_index_mismatch_count,

    "scv_count_mismatches":
        scv_count_mismatch_count,

    "submitter_name_mismatches":
        submitter_name_mismatch_count,

    "mapping_artifact_path":
        str(ORGID_MAP_PATH),

    "mapping_artifact_sha256":
        mapping_sha256,

    "mapping_artifact_bytes":
        ORGID_MAP_PATH.stat().st_size,

    "original_t0_parquet_modified":
        False,

    "repair_status":
        (
            "Source mapping complete; corrected T0 Parquet v1.2 "
            "not yet created."
        ),
}

with open(
    ORGID_MANIFEST_PATH,
    "w",
    encoding="utf-8",
) as manifest_file:
    json.dump(
        manifest,
        manifest_file,
        indent=2,
        ensure_ascii=False,
        sort_keys=True,
    )

manifest_sha256 = sha256_file(ORGID_MANIFEST_PATH)

print("\nVERSIONED SOURCE-AUDIT ARTIFACTS")
print("=" * 112)

print(f"✅ Mapping saved: {ORGID_MAP_PATH}")
print(f"   Rows: {len(saved_mapping_df):,}")
print(f"   SHA-256: {mapping_sha256}")

print(f"\n✅ Manifest saved: {ORGID_MANIFEST_PATH}")
print(f"   SHA-256: {manifest_sha256}")

print()
print("✅ STEP 4D COMPLETE")
print(
    "The full source-derived OrgID mapping passed all critical "
    "RCV, XML-index, SCV-count, and submitter-name checks."
)
print(
    "The corrected T0 v1.1 Parquet remains unchanged. "
    "Do not delete the raw XML yet."
)

STEP 4D — FULL T0 SUBMITTER ORGID SOURCE EXTRACTION
Raw XML: /content/clinvar_rcv_raw/ClinVarFullRelease_2023-01.xml.gz
Target RCVs: 71,659
Output mapping: /content/drive/MyDrive/GES_RAG_Temporal_Study/data_interim/t0_rcv_submitter_orgid_source_map_v1_2.parquet
Output manifest: /content/drive/MyDrive/GES_RAG_Temporal_Study/configs/clinvar_t0_submitter_orgid_source_audit_manifest_v1_2.json

Beginning complete streaming scan...
Progress: 100,000 XML records scanned; 53/71,659 target RCVs found
Progress: 200,000 XML records scanned; 71/71,659 target RCVs found
Progress: 300,000 XML records scanned; 131/71,659 target RCVs found
Progress: 400,000 XML records scanned; 201/71,659 target RCVs found
Progress: 500,000 XML records scanned; 390/71,659 target RCVs found
Progress: 600,000 XML records scanned; 648/71,659 target RCVs found
Progress: 700,000 XML records scanned; 670/71,659 target RCVs found
Progress: 800,000 XML records scanned; 876/71,659 target RCVs found
Progress: 900,000 XML record

In [38]:
# STEP 4E: Create corrected T0 Parquet v1.2 with submitter OrgIDs
# The original v1.1 artifact remains immutable.
# Only submitter_ids_json is replaced from the verified source mapping.

import hashlib
import json
import shutil
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq

if "df_t0" not in globals():
    raise RuntimeError(
        "df_t0 is unavailable. Run Step 2 before Step 4E."
    )


# ------------------------------------------------------------------
# Paths
# ------------------------------------------------------------------

STUDY_DIR = Path(
    "/content/drive/MyDrive/GES_RAG_Temporal_Study"
)

DATA_DIR = STUDY_DIR / "data_interim"
CONFIG_DIR = STUDY_DIR / "configs"

INPUT_T0_V1_1 = (
    DATA_DIR /
    "t0_rcv_target_genes_corrected_v1_1.parquet"
)

ORGID_MAP_PATH = (
    DATA_DIR /
    "t0_rcv_submitter_orgid_source_map_v1_2.parquet"
)

ORGID_SOURCE_MANIFEST_PATH = (
    CONFIG_DIR /
    "clinvar_t0_submitter_orgid_source_audit_manifest_v1_2.json"
)

OUTPUT_T0_V1_2 = (
    DATA_DIR /
    "t0_rcv_target_genes_corrected_v1_2.parquet"
)

REPAIR_MANIFEST_PATH = (
    CONFIG_DIR /
    "clinvar_t0_submitter_id_repair_manifest_v1_2.json"
)

TEMP_OUTPUT = Path(
    "/content/t0_rcv_target_genes_corrected_v1_2.parquet"
)


# ------------------------------------------------------------------
# Safety checks
# ------------------------------------------------------------------

required_inputs = [
    INPUT_T0_V1_1,
    ORGID_MAP_PATH,
    ORGID_SOURCE_MANIFEST_PATH,
]

for required_path in required_inputs:
    if not required_path.exists():
        raise FileNotFoundError(
            f"Required input is missing: {required_path}"
        )

if OUTPUT_T0_V1_2.exists():
    raise FileExistsError(
        f"The versioned output already exists:\n{OUTPUT_T0_V1_2}\n"
        "Do not overwrite it. Share its file status before rerunning."
    )

if REPAIR_MANIFEST_PATH.exists():
    raise FileExistsError(
        f"The versioned repair manifest already exists:\n"
        f"{REPAIR_MANIFEST_PATH}\n"
        "Do not overwrite it."
    )

if TEMP_OUTPUT.exists():
    TEMP_OUTPUT.unlink()


# ------------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------------

def sha256_file(path, chunk_size=8 * 1024 * 1024):
    """Calculate SHA-256 without reading the whole file at once."""
    digest = hashlib.sha256()

    with open(path, "rb") as file_handle:
        while True:
            block = file_handle.read(chunk_size)

            if not block:
                break

            digest.update(block)

    return digest.hexdigest()


def parse_json_list(value):
    """Parse and validate a JSON list."""
    if isinstance(value, str):
        parsed = json.loads(value)
    else:
        parsed = value

    if not isinstance(parsed, list):
        raise TypeError(
            f"Expected JSON list, found {type(parsed).__name__}"
        )

    return parsed


def canonical_string_list_json(value):
    """
    Validate, clean, deduplicate, and serialize an identifier list
    deterministically.
    """
    parsed = parse_json_list(value)

    cleaned = sorted({
        str(item).strip()
        for item in parsed
        if item is not None and str(item).strip()
    })

    return json.dumps(
        cleaned,
        ensure_ascii=False,
        separators=(",", ":"),
    )


# ------------------------------------------------------------------
# Load and verify source-audit manifest
# ------------------------------------------------------------------

with open(
    ORGID_SOURCE_MANIFEST_PATH,
    "r",
    encoding="utf-8",
) as manifest_file:
    orgid_source_manifest = json.load(manifest_file)

expected_mapping_sha256 = orgid_source_manifest.get(
    "mapping_artifact_sha256"
)

observed_mapping_sha256 = sha256_file(ORGID_MAP_PATH)

if expected_mapping_sha256 != observed_mapping_sha256:
    raise AssertionError(
        "The OrgID mapping checksum does not match its source-audit "
        "manifest. Do not continue."
    )

if orgid_source_manifest.get("mapping_rows") != 71_659:
    raise AssertionError(
        "The source-audit manifest does not report 71,659 mapping rows."
    )

if (
    orgid_source_manifest.get(
        "source_scvs_without_primary_orgid"
    ) != 0
):
    raise AssertionError(
        "The source-audit manifest reports SCVs without OrgID values."
    )


# ------------------------------------------------------------------
# Load source v1.1 and verified OrgID mapping
# ------------------------------------------------------------------

source_df = pd.read_parquet(INPUT_T0_V1_1)
orgid_map_df = pd.read_parquet(ORGID_MAP_PATH)

if len(source_df) != 71_659:
    raise AssertionError(
        f"Expected 71,659 source rows, found {len(source_df):,}."
    )

if len(orgid_map_df) != 71_659:
    raise AssertionError(
        f"Expected 71,659 mapping rows, "
        f"found {len(orgid_map_df):,}."
    )

required_mapping_columns = {
    "rcv_accession",
    "source_primary_org_ids_json",
    "source_orgid_count",
    "source_scvs_missing_orgid",
    "scv_count_matches",
    "submitter_names_match",
    "xml_record_index_matches",
}

missing_mapping_columns = sorted(
    required_mapping_columns - set(orgid_map_df.columns)
)

if missing_mapping_columns:
    raise AssertionError(
        f"Mapping columns are missing: {missing_mapping_columns}"
    )


# ------------------------------------------------------------------
# Validate mapping integrity again before repair
# ------------------------------------------------------------------

mapping_checks = {
    "One mapping row per RCV":
        orgid_map_df["rcv_accession"].nunique() == 71_659,

    "No duplicate mapping RCVs":
        orgid_map_df["rcv_accession"].duplicated().sum() == 0,

    "All XML indexes matched":
        bool(orgid_map_df["xml_record_index_matches"].all()),

    "All SCV counts matched":
        bool(orgid_map_df["scv_count_matches"].all()),

    "All submitter names matched":
        bool(orgid_map_df["submitter_names_match"].all()),

    "Every RCV has at least one OrgID":
        bool((orgid_map_df["source_orgid_count"] > 0).all()),

    "No SCV lacks a primary OrgID":
        int(
            orgid_map_df["source_scvs_missing_orgid"].sum()
        ) == 0,
}

failed_mapping_checks = [
    name
    for name, passed in mapping_checks.items()
    if not passed
]

if failed_mapping_checks:
    raise AssertionError(
        "Mapping validation failed: "
        + "; ".join(failed_mapping_checks)
    )


# ------------------------------------------------------------------
# Match mapping to source artifact in exact source-row order
# ------------------------------------------------------------------

mapping_series = (
    orgid_map_df
    .set_index("rcv_accession")[
        "source_primary_org_ids_json"
    ]
)

mapped_orgid_values = source_df[
    "rcv_accession"
].map(mapping_series)

missing_mapping_values = int(
    mapped_orgid_values.isna().sum()
)

if missing_mapping_values != 0:
    raise AssertionError(
        f"{missing_mapping_values:,} source rows lack an OrgID mapping."
    )

canonical_orgid_values = mapped_orgid_values.apply(
    canonical_string_list_json
)

empty_repaired_id_lists = int(
    canonical_orgid_values.apply(
        lambda value: len(json.loads(value)) == 0
    ).sum()
)

if empty_repaired_id_lists != 0:
    raise AssertionError(
        f"{empty_repaired_id_lists:,} repaired rows still have "
        "empty submitter-ID lists."
    )


# ------------------------------------------------------------------
# Confirm the original v1.1 field is empty before replacing it
# ------------------------------------------------------------------

original_nonempty_submitter_id_rows = int(
    source_df["submitter_ids_json"].apply(
        lambda value: len(parse_json_list(value)) > 0
    ).sum()
)

if original_nonempty_submitter_id_rows != 0:
    raise AssertionError(
        "The v1.1 source contains nonempty submitter-ID rows. "
        "The repair assumptions are no longer valid."
    )


# ------------------------------------------------------------------
# Create v1.2 in memory
# ------------------------------------------------------------------

repaired_df = source_df.copy(deep=True)

repaired_df["submitter_ids_json"] = (
    canonical_orgid_values.to_numpy()
)

other_columns = [
    column
    for column in source_df.columns
    if column != "submitter_ids_json"
]

all_nonrepair_columns_unchanged = (
    source_df[other_columns].equals(
        repaired_df[other_columns]
    )
)

rcv_order_unchanged = (
    source_df["rcv_accession"].tolist()
    == repaired_df["rcv_accession"].tolist()
)

schema_columns_unchanged = (
    list(source_df.columns)
    == list(repaired_df.columns)
)

conflict_positive_before = int(
    source_df["aggregate_conflict_flag"]
    .fillna(False)
    .astype(bool)
    .sum()
)

conflict_positive_after = int(
    repaired_df["aggregate_conflict_flag"]
    .fillna(False)
    .astype(bool)
    .sum()
)

repaired_nonempty_rows = int(
    repaired_df["submitter_ids_json"].apply(
        lambda value: len(json.loads(value)) > 0
    ).sum()
)

exact_mapping_matches = int(
    (
        repaired_df["submitter_ids_json"]
        == canonical_orgid_values.to_numpy()
    ).sum()
)

unique_org_ids = set()

total_rcv_level_orgid_entries = 0

for value in repaired_df["submitter_ids_json"]:
    parsed_ids = json.loads(value)

    total_rcv_level_orgid_entries += len(parsed_ids)
    unique_org_ids.update(parsed_ids)


# ------------------------------------------------------------------
# Pre-write repair validation
# ------------------------------------------------------------------

repair_checks = {
    "Row count remains 71,659":
        len(repaired_df) == 71_659,

    "Column count remains 34":
        len(repaired_df.columns) == 34,

    "Column names and order unchanged":
        schema_columns_unchanged,

    "RCV row order unchanged":
        rcv_order_unchanged,

    "All columns except submitter_ids_json unchanged":
        all_nonrepair_columns_unchanged,

    "Every repaired row has at least one OrgID":
        repaired_nonempty_rows == 71_659,

    "Every repaired value matches the source mapping":
        exact_mapping_matches == 71_659,

    "Conflict-positive count remains 1,484":
        (
            conflict_positive_before == 1_484
            and conflict_positive_after == 1_484
        ),

    "Unique RCV accessions remain 71,659":
        repaired_df["rcv_accession"].nunique() == 71_659,
}

failed_repair_checks = [
    name
    for name, passed in repair_checks.items()
    if not passed
]

print("=" * 112)
print("STEP 4E — CREATE CORRECTED T0 PARQUET v1.2")
print("=" * 112)

print("\nPRE-WRITE REPAIR VALIDATION")
print("-" * 112)

for name, passed in repair_checks.items():
    print(
        f"{'✅ PASS' if passed else '❌ FAIL'} — {name}"
    )

print("\nSUBMITTER-ID REPAIR COVERAGE")
print("-" * 112)

print(
    "Rows with nonempty submitter IDs before repair:",
    f"{original_nonempty_submitter_id_rows:,}"
)

print(
    "Rows with nonempty submitter IDs after repair:",
    f"{repaired_nonempty_rows:,}"
)

print(
    "Total RCV-level OrgID entries after deduplication:",
    f"{total_rcv_level_orgid_entries:,}"
)

print(
    "Unique primary organization IDs represented:",
    f"{len(unique_org_ids):,}"
)

if failed_repair_checks:
    raise AssertionError(
        "STEP 4E FAILED BEFORE WRITING. "
        + "; ".join(failed_repair_checks)
    )


# ------------------------------------------------------------------
# Write locally, verify, then copy to Drive
# ------------------------------------------------------------------

repaired_df.to_parquet(
    TEMP_OUTPUT,
    index=False,
    compression="zstd",
)

temporary_sha256 = sha256_file(TEMP_OUTPUT)

temporary_metadata = pq.ParquetFile(
    TEMP_OUTPUT
).metadata

if temporary_metadata.num_rows != 71_659:
    raise AssertionError(
        "Temporary output has an incorrect Parquet row count."
    )

shutil.copy2(
    TEMP_OUTPUT,
    OUTPUT_T0_V1_2,
)

output_sha256 = sha256_file(
    OUTPUT_T0_V1_2
)

if output_sha256 != temporary_sha256:
    raise AssertionError(
        "The Drive copy checksum differs from the temporary output."
    )


# ------------------------------------------------------------------
# Full post-write readback validation
# ------------------------------------------------------------------

readback_df = pd.read_parquet(
    OUTPUT_T0_V1_2
)

readback_exactly_matches_repaired = (
    readback_df.equals(repaired_df)
)

readback_nonrepair_columns_unchanged = (
    readback_df[other_columns].equals(
        source_df[other_columns]
    )
)

readback_id_mapping_matches = (
    readback_df["submitter_ids_json"].tolist()
    == repaired_df["submitter_ids_json"].tolist()
)

post_write_checks = {
    "Drive checksum equals temporary checksum":
        output_sha256 == temporary_sha256,

    "Readback exactly matches repaired dataframe":
        readback_exactly_matches_repaired,

    "Readback nonrepair columns match v1.1":
        readback_nonrepair_columns_unchanged,

    "Readback submitter IDs match verified mapping":
        readback_id_mapping_matches,

    "Readback has 71,659 rows":
        len(readback_df) == 71_659,

    "Readback has 34 columns":
        len(readback_df.columns) == 34,

    "Readback conflict count remains 1,484":
        int(
            readback_df["aggregate_conflict_flag"]
            .fillna(False)
            .astype(bool)
            .sum()
        ) == 1_484,
}

failed_post_write_checks = [
    name
    for name, passed in post_write_checks.items()
    if not passed
]

print("\nPOST-WRITE VALIDATION")
print("-" * 112)

for name, passed in post_write_checks.items():
    print(
        f"{'✅ PASS' if passed else '❌ FAIL'} — {name}"
    )

if failed_post_write_checks:
    raise AssertionError(
        "STEP 4E POST-WRITE VALIDATION FAILED. "
        + "; ".join(failed_post_write_checks)
    )


# ------------------------------------------------------------------
# Create versioned repair manifest
# ------------------------------------------------------------------

input_v1_1_sha256 = sha256_file(
    INPUT_T0_V1_1
)

repair_manifest = {
    "artifact_type":
        "T0 corrected RCV cohort submitter-ID repair",

    "version":
        "1.2.0",

    "created_at_utc":
        datetime.now(timezone.utc).isoformat(),

    "repair_reason":
        (
            "The historical parser retained primary submitter names "
            "and SCV counts but omitted primary organization IDs "
            "from submitter_ids_json."
        ),

    "source_xml_field":
        (
            "ClinVarSet/ClinVarAssertion/"
            "ClinVarAccession[@Type='SCV']/@OrgID"
        ),

    "source_submitter_name_field":
        (
            "ClinVarSet/ClinVarAssertion/"
            "ClinVarSubmissionID/@submitter"
        ),

    "input_artifact_path":
        str(INPUT_T0_V1_1),

    "input_artifact_sha256":
        input_v1_1_sha256,

    "source_mapping_path":
        str(ORGID_MAP_PATH),

    "source_mapping_sha256":
        observed_mapping_sha256,

    "source_audit_manifest_path":
        str(ORGID_SOURCE_MANIFEST_PATH),

    "output_artifact_path":
        str(OUTPUT_T0_V1_2),

    "output_artifact_sha256":
        output_sha256,

    "output_artifact_bytes":
        OUTPUT_T0_V1_2.stat().st_size,

    "rows":
        len(readback_df),

    "columns":
        len(readback_df.columns),

    "changed_columns": [
        "submitter_ids_json"
    ],

    "unchanged_columns_count":
        len(other_columns),

    "rows_with_nonempty_submitter_ids_before":
        original_nonempty_submitter_id_rows,

    "rows_with_nonempty_submitter_ids_after":
        repaired_nonempty_rows,

    "total_rcv_level_orgid_entries":
        total_rcv_level_orgid_entries,

    "unique_primary_orgids":
        len(unique_org_ids),

    "source_scvs":
        orgid_source_manifest.get(
            "total_source_scvs"
        ),

    "source_scvs_without_primary_orgid":
        orgid_source_manifest.get(
            "source_scvs_without_primary_orgid"
        ),

    "rcv_order_unchanged":
        rcv_order_unchanged,

    "schema_unchanged":
        schema_columns_unchanged,

    "all_nonrepair_columns_unchanged":
        readback_nonrepair_columns_unchanged,

    "conflict_positive_count_before":
        conflict_positive_before,

    "conflict_positive_count_after":
        conflict_positive_after,

    "t1_information_used":
        False,

    "temporal_outcomes_used":
        False,

    "ges_model_output_used":
        False,

    "original_v1_1_modified":
        False,

    "raw_xml_deletion_authorized":
        False,

    "remaining_action":
        (
            "Patch and version the historical XML parser notebook, "
            "then complete remaining date and condition validation "
            "before freezing the consolidated T0 manifest."
        ),
}

with open(
    REPAIR_MANIFEST_PATH,
    "w",
    encoding="utf-8",
) as manifest_file:
    json.dump(
        repair_manifest,
        manifest_file,
        indent=2,
        ensure_ascii=False,
        sort_keys=True,
    )

repair_manifest_sha256 = sha256_file(
    REPAIR_MANIFEST_PATH
)

if TEMP_OUTPUT.exists():
    TEMP_OUTPUT.unlink()


# Keep repaired dataframe available for the next notebook step
df_t0_v1_2 = readback_df


# ------------------------------------------------------------------
# Final output
# ------------------------------------------------------------------

print("\nVERSIONED REPAIR ARTIFACTS")
print("=" * 112)

print(f"✅ Corrected T0 v1.2: {OUTPUT_T0_V1_2}")
print(f"   Rows: {len(readback_df):,}")
print(f"   Columns: {len(readback_df.columns):,}")
print(
    f"   Size: "
    f"{OUTPUT_T0_V1_2.stat().st_size / (1024 ** 2):,.3f} MB"
)
print(f"   SHA-256: {output_sha256}")

print(f"\n✅ Repair manifest: {REPAIR_MANIFEST_PATH}")
print(f"   SHA-256: {repair_manifest_sha256}")

print("\nFINAL STATUS")
print("=" * 112)

print("✅ T0 v1.2 was created and fully readback-validated.")
print("✅ Only submitter_ids_json changed from v1.1.")
print("✅ All 71,659 RCVs now contain source-derived primary OrgIDs.")
print("✅ Conflict correction and every other saved field were preserved.")
print("✅ The original v1.1 Parquet remains unchanged.")
print("⚠️ Do not delete the raw T0 XML yet.")
print("⚠️ The parser notebook still requires a versioned v1.2 patch.")

print()
print("✅ STEP 4E COMPLETE")
print("The repaired dataframe is available as: df_t0_v1_2")

STEP 4E — CREATE CORRECTED T0 PARQUET v1.2

PRE-WRITE REPAIR VALIDATION
----------------------------------------------------------------------------------------------------------------
✅ PASS — Row count remains 71,659
✅ PASS — Column count remains 34
✅ PASS — Column names and order unchanged
✅ PASS — RCV row order unchanged
✅ PASS — All columns except submitter_ids_json unchanged
✅ PASS — Every repaired row has at least one OrgID
✅ PASS — Every repaired value matches the source mapping
✅ PASS — Conflict-positive count remains 1,484
✅ PASS — Unique RCV accessions remain 71,659

SUBMITTER-ID REPAIR COVERAGE
----------------------------------------------------------------------------------------------------------------
Rows with nonempty submitter IDs before repair: 0
Rows with nonempty submitter IDs after repair: 71,659
Total RCV-level OrgID entries after deduplication: 100,614
Unique primary organization IDs represented: 262

POST-WRITE VALIDATION
--------------------------------------

In [39]:
# STEP 4F.1: Locate the exact parser cells that require the OrgID patch
# Read-only inspection — no notebook or data file is modified.

import hashlib
import json
from pathlib import Path

import pandas as pd


NOTEBOOK_V1_1 = Path(
    "/content/drive/MyDrive/Colab Notebooks/"
    "02_GES_temporal_validation_v1_1.ipynb"
)


def sha256_file(path, chunk_size=8 * 1024 * 1024):
    """Calculate SHA-256 without loading the whole file into memory."""
    digest = hashlib.sha256()

    with open(path, "rb") as file_handle:
        while True:
            block = file_handle.read(chunk_size)

            if not block:
                break

            digest.update(block)

    return digest.hexdigest()


if not NOTEBOOK_V1_1.exists():
    raise FileNotFoundError(
        f"Notebook v1.1 was not found:\n{NOTEBOOK_V1_1}"
    )


with open(
    NOTEBOOK_V1_1,
    "r",
    encoding="utf-8",
) as notebook_file:
    notebook = json.load(notebook_file)


cells = notebook.get("cells", [])

if not isinstance(cells, list) or len(cells) == 0:
    raise AssertionError(
        "The notebook contains no readable cells."
    )


SEARCH_TERMS = [
    "submitter_ids_json",
    "submitters_json",
    "ClinVarSubmissionID",
    "ClinVarAccession",
    "ClinVarAssertion",
    "OrgID",
    "OrgId",
    "scv_records_json",
    "scv_records",
    "unique_submitter_count_xml",
    "primary_submitter",
    "aggregate_conflict_flag",
]


candidate_rows = []
matched_code_cells = []

for cell_index, cell in enumerate(cells):

    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))

    matched_terms = [
        term
        for term in SEARCH_TERMS
        if term.lower() in source.lower()
    ]

    if not matched_terms:
        continue

    candidate_rows.append({
        "cell_index": cell_index,
        "execution_count": cell.get("execution_count"),
        "matched_term_count": len(matched_terms),
        "matched_terms": ", ".join(matched_terms),
        "source_lines": len(source.splitlines()),
        "source_characters": len(source),
    })

    matched_code_cells.append({
        "cell_index": cell_index,
        "source": source,
        "matched_terms": matched_terms,
    })


candidate_summary = pd.DataFrame(candidate_rows)

print("=" * 112)
print("STEP 4F.1 — HISTORICAL PARSER NOTEBOOK INSPECTION")
print("=" * 112)

print(f"Notebook: {NOTEBOOK_V1_1}")
print(f"Notebook SHA-256: {sha256_file(NOTEBOOK_V1_1)}")
print(f"Total notebook cells: {len(cells):,}")
print(f"Matching code cells: {len(matched_code_cells):,}")

print("\nCANDIDATE CELL SUMMARY")
print("-" * 112)

if candidate_summary.empty:
    raise AssertionError(
        "No parser-related cells were found using the expected search terms."
    )

display(
    candidate_summary.sort_values(
        ["matched_term_count", "cell_index"],
        ascending=[False, True],
    ).reset_index(drop=True)
)


# Prioritize cells that directly define or write submitter information
priority_terms = {
    "submitter_ids_json",
    "ClinVarSubmissionID",
    "OrgID",
    "OrgId",
}

priority_cells = [
    item
    for item in matched_code_cells
    if priority_terms.intersection(item["matched_terms"])
]

# If no direct matches exist, show the strongest related parser cells
if not priority_cells:
    priority_cells = sorted(
        matched_code_cells,
        key=lambda item: len(item["matched_terms"]),
        reverse=True,
    )[:8]


print("\nFULL CODE OF PRIORITY CANDIDATE CELLS")
print("=" * 112)

for item in priority_cells:

    print()
    print("#" * 112)
    print(
        f"NOTEBOOK CELL INDEX: {item['cell_index']} | "
        f"MATCHED TERMS: {', '.join(item['matched_terms'])}"
    )
    print("#" * 112)

    source_lines = item["source"].splitlines()

    for line_number, line in enumerate(
        source_lines,
        start=1,
    ):
        print(f"{line_number:04d}: {line}")


print()
print("✅ STEP 4F.1 COMPLETE")
print("No notebook, Parquet, XML, or manifest was modified.")

STEP 4F.1 — HISTORICAL PARSER NOTEBOOK INSPECTION
Notebook: /content/drive/MyDrive/Colab Notebooks/02_GES_temporal_validation_v1_1.ipynb
Notebook SHA-256: 856ca46cedfc10dc17fd1c80b651a742a77827430808c4a862abe1145ba27531
Total notebook cells: 24
Matching code cells: 8

CANDIDATE CELL SUMMARY
----------------------------------------------------------------------------------------------------------------


,cell_index,execution_count,matched_term_count,matched_terms,source_lines,source_characters
0,13,15,9,"submitter_ids_json, submitters_json, ClinVarSu...",1584,32672
1,11,12,1,ClinVarAccession,763,16781
2,18,20,1,aggregate_conflict_flag,399,10564
3,19,21,1,aggregate_conflict_flag,355,9822
4,20,22,1,aggregate_conflict_flag,217,5731
5,21,23,1,aggregate_conflict_flag,224,6292
6,22,24,1,aggregate_conflict_flag,385,9570
7,23,25,1,aggregate_conflict_flag,561,13538



FULL CODE OF PRIORITY CANDIDATE CELLS

################################################################################################################
NOTEBOOK CELL INDEX: 13 | MATCHED TERMS: submitter_ids_json, submitters_json, ClinVarSubmissionID, ClinVarAccession, ClinVarAssertion, scv_records_json, scv_records, unique_submitter_count_xml, aggregate_conflict_flag
################################################################################################################
0001: # ============================================================
0002: # STEP 11: Stream-extract T0 RCV records for the four genes
0003: #
0004: # Input:
0005: #   ClinVarFullRelease_2023-01.xml.gz
0006: #
0007: # Output:
0008: #   One compact Parquet row per RCV variant-condition record
0009: #
0010: # No T1 information, temporal outcome, or GES model is used.
0011: # ============================================================
0012: 
0013: from pathlib import Path
0014: from datetime import datetime, time

In [40]:
# STEP 4F.1A: Export exact parser candidate cells to a readable text file
# Read-only inspection of notebook v1.1.

import json
from pathlib import Path

NOTEBOOK_V1_1 = Path(
    "/content/drive/MyDrive/Colab Notebooks/"
    "02_GES_temporal_validation_v1_1.ipynb"
)

OUTPUT_REPORT = Path(
    "/content/drive/MyDrive/GES_RAG_Temporal_Study/"
    "outputs/logs/parser_candidate_cells_v1_1.txt"
)

OUTPUT_REPORT.parent.mkdir(
    parents=True,
    exist_ok=True
)

if not NOTEBOOK_V1_1.exists():
    raise FileNotFoundError(
        f"Notebook not found: {NOTEBOOK_V1_1}"
    )

with open(
    NOTEBOOK_V1_1,
    "r",
    encoding="utf-8",
) as notebook_file:
    notebook = json.load(notebook_file)

SEARCH_TERMS = [
    "submitter_ids_json",
    "submitters_json",
    "ClinVarSubmissionID",
    "ClinVarAccession",
    "ClinVarAssertion",
    "OrgID",
    "OrgId",
    "scv_records_json",
    "unique_submitter_count_xml",
]

matched_cells = []

for cell_index, cell in enumerate(
    notebook.get("cells", [])
):
    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))

    matched_terms = [
        term
        for term in SEARCH_TERMS
        if term.lower() in source.lower()
    ]

    if matched_terms:
        matched_cells.append({
            "cell_index": cell_index,
            "execution_count": cell.get(
                "execution_count"
            ),
            "matched_terms": matched_terms,
            "source": source,
        })

if not matched_cells:
    raise AssertionError(
        "No relevant parser cells were found."
    )

report_lines = []

report_lines.append(
    "GES-RAG HISTORICAL PARSER CANDIDATE CELLS\n"
)
report_lines.append(
    "=" * 100 + "\n"
)
report_lines.append(
    f"Notebook: {NOTEBOOK_V1_1}\n"
)
report_lines.append(
    f"Matching code cells: {len(matched_cells)}\n\n"
)

report_lines.append("CANDIDATE CELL SUMMARY\n")
report_lines.append("-" * 100 + "\n")

for item in matched_cells:
    report_lines.append(
        f"Cell index: {item['cell_index']} | "
        f"Execution count: {item['execution_count']} | "
        f"Matched terms: {', '.join(item['matched_terms'])}\n"
    )

report_lines.append("\n")
report_lines.append(
    "FULL CODE OF MATCHING CELLS\n"
)
report_lines.append(
    "=" * 100 + "\n"
)

for item in matched_cells:

    report_lines.append("\n")
    report_lines.append("#" * 100 + "\n")
    report_lines.append(
        f"NOTEBOOK CELL INDEX: {item['cell_index']}\n"
    )
    report_lines.append(
        f"MATCHED TERMS: "
        f"{', '.join(item['matched_terms'])}\n"
    )
    report_lines.append("#" * 100 + "\n")

    for line_number, line in enumerate(
        item["source"].splitlines(),
        start=1,
    ):
        report_lines.append(
            f"{line_number:04d}: {line}\n"
        )

with open(
    OUTPUT_REPORT,
    "w",
    encoding="utf-8",
) as output_file:
    output_file.writelines(report_lines)

print("=" * 100)
print("STEP 4F.1A — PARSER CELL EXPORT")
print("=" * 100)

print(f"Matching code cells found: {len(matched_cells)}")

print("\nCANDIDATE CELL INDEXES")
print("-" * 100)

for item in matched_cells:
    print(
        f"Cell {item['cell_index']} — "
        f"{', '.join(item['matched_terms'])}"
    )

print("\nREPORT CREATED")
print("-" * 100)
print(OUTPUT_REPORT)
print(
    f"Size: "
    f"{OUTPUT_REPORT.stat().st_size / 1024:,.2f} KB"
)

print()
print("✅ STEP 4F.1A COMPLETE")
print("No notebook or dataset was modified.")

STEP 4F.1A — PARSER CELL EXPORT
Matching code cells found: 2

CANDIDATE CELL INDEXES
----------------------------------------------------------------------------------------------------
Cell 11 — ClinVarAccession
Cell 13 — submitter_ids_json, submitters_json, ClinVarSubmissionID, ClinVarAccession, ClinVarAssertion, scv_records_json, unique_submitter_count_xml

REPORT CREATED
----------------------------------------------------------------------------------------------------
/content/drive/MyDrive/GES_RAG_Temporal_Study/outputs/logs/parser_candidate_cells_v1_1.txt
Size: 63.43 KB

✅ STEP 4F.1A COMPLETE
No notebook or dataset was modified.


In [41]:
# STEP 4F.1B: Print focused parser snippets around submitter-ID logic
# Read-only. No notebook or data artifact is modified.

import json
from pathlib import Path

NOTEBOOK_V1_1 = Path(
    "/content/drive/MyDrive/Colab Notebooks/"
    "02_GES_temporal_validation_v1_1.ipynb"
)

if not NOTEBOOK_V1_1.exists():
    raise FileNotFoundError(
        f"Notebook not found: {NOTEBOOK_V1_1}"
    )

with open(
    NOTEBOOK_V1_1,
    "r",
    encoding="utf-8",
) as notebook_file:
    notebook = json.load(notebook_file)


FOCUS_TERMS = [
    "submitter_ids_json",
    "submitters_json",
    "ClinVarSubmissionID",
    "ClinVarAssertion",
    "OrgID",
    "OrgId",
    "unique_submitter_count_xml",
    "scv_records_json",
]

CONTEXT_LINES_BEFORE = 12
CONTEXT_LINES_AFTER = 18

focused_matches = []

for cell_index, cell in enumerate(
    notebook.get("cells", [])
):
    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))
    lines = source.splitlines()

    matched_line_indexes = []

    for line_index, line in enumerate(lines):

        matched_terms = [
            term
            for term in FOCUS_TERMS
            if term.lower() in line.lower()
        ]

        if matched_terms:
            matched_line_indexes.append(
                {
                    "line_index": line_index,
                    "matched_terms": matched_terms,
                }
            )

    if not matched_line_indexes:
        continue

    # Merge nearby matches into compact code windows
    windows = []

    for match in matched_line_indexes:

        start = max(
            0,
            match["line_index"] - CONTEXT_LINES_BEFORE
        )

        end = min(
            len(lines),
            match["line_index"] + CONTEXT_LINES_AFTER + 1
        )

        if windows and start <= windows[-1]["end"] + 3:
            windows[-1]["end"] = max(
                windows[-1]["end"],
                end
            )

            windows[-1]["matched_terms"].update(
                match["matched_terms"]
            )
        else:
            windows.append({
                "start": start,
                "end": end,
                "matched_terms": set(
                    match["matched_terms"]
                ),
            })

    focused_matches.append({
        "cell_index": cell_index,
        "execution_count": cell.get(
            "execution_count"
        ),
        "lines": lines,
        "windows": windows,
    })


print("=" * 112)
print("STEP 4F.1B — FOCUSED HISTORICAL PARSER SNIPPETS")
print("=" * 112)

print(
    f"Notebook cells containing focused terms: "
    f"{len(focused_matches):,}"
)

for item in focused_matches:

    print()
    print("#" * 112)
    print(
        f"NOTEBOOK CELL INDEX: {item['cell_index']} | "
        f"EXECUTION COUNT: {item['execution_count']}"
    )
    print("#" * 112)

    for window_number, window in enumerate(
        item["windows"],
        start=1,
    ):
        print()
        print(
            f"--- Window {window_number} | "
            f"terms: "
            f"{', '.join(sorted(window['matched_terms']))} | "
            f"lines {window['start'] + 1}–{window['end']} ---"
        )

        for line_index in range(
            window["start"],
            window["end"],
        ):
            marker = ">>> " if any(
                term.lower()
                in item["lines"][line_index].lower()
                for term in FOCUS_TERMS
            ) else "    "

            print(
                f"{marker}"
                f"{line_index + 1:04d}: "
                f"{item['lines'][line_index]}"
            )

if not focused_matches:
    raise AssertionError(
        "No focused parser code was found."
    )

print()
print("✅ STEP 4F.1B COMPLETE")
print("No notebook, XML, Parquet, or manifest was modified.")

STEP 4F.1B — FOCUSED HISTORICAL PARSER SNIPPETS
Notebook cells containing focused terms: 1

################################################################################################################
NOTEBOOK CELL INDEX: 13 | EXECUTION COUNT: 15
################################################################################################################

--- Window 1 | terms: ClinVarAssertion, ClinVarSubmissionID | lines 504–539 ---
    0504: 
    0505:     if has_benign:
    0506:         return "Benign/Likely benign"
    0507: 
    0508:     if has_vus:
    0509:         return "VUS"
    0510: 
    0511:     return "Other"
    0512: 
    0513: 
    0514: def extract_submitter(assertion):
    0515:     """
>>> 0516:     Extract submitter metadata from a ClinVarAssertion.
    0517:     """
    0518: 
    0519:     for element in descendants(
    0520:         assertion,
>>> 0521:         "ClinVarSubmissionID"
    0522:     ):
    0523: 
    0524:         submitter = clean_text(

In [42]:
# STEP 4F.2: Create versioned notebook v1.2 with corrected submitter OrgID parsing
# Preserves notebook v1.1 unchanged.
# Patches only extract_submitter() in the historical parser cell.

import copy
import hashlib
import json
from datetime import datetime, timezone
from pathlib import Path


# ------------------------------------------------------------------
# Paths
# ------------------------------------------------------------------

NOTEBOOK_V1_1 = Path(
    "/content/drive/MyDrive/Colab Notebooks/"
    "02_GES_temporal_validation_v1_1.ipynb"
)

NOTEBOOK_V1_2 = Path(
    "/content/drive/MyDrive/Colab Notebooks/"
    "02_GES_temporal_validation_v1_2.ipynb"
)

PATCH_MANIFEST = Path(
    "/content/drive/MyDrive/GES_RAG_Temporal_Study/"
    "configs/t0_parser_submitter_orgid_patch_manifest_v1_2.json"
)


# ------------------------------------------------------------------
# Safety checks
# ------------------------------------------------------------------

if not NOTEBOOK_V1_1.exists():
    raise FileNotFoundError(
        f"Source notebook not found:\n{NOTEBOOK_V1_1}"
    )

if NOTEBOOK_V1_2.exists():
    raise FileExistsError(
        f"Versioned notebook already exists:\n{NOTEBOOK_V1_2}\n"
        "Do not overwrite it."
    )

if PATCH_MANIFEST.exists():
    raise FileExistsError(
        f"Versioned patch manifest already exists:\n"
        f"{PATCH_MANIFEST}\n"
        "Do not overwrite it."
    )

PATCH_MANIFEST.parent.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------------
# Helpers
# ------------------------------------------------------------------

def sha256_file(path, chunk_size=8 * 1024 * 1024):
    """Calculate the SHA-256 checksum of a file."""
    digest = hashlib.sha256()

    with open(path, "rb") as file_handle:
        while True:
            block = file_handle.read(chunk_size)

            if not block:
                break

            digest.update(block)

    return digest.hexdigest()


def sha256_text(text):
    """Calculate SHA-256 for a text block."""
    return hashlib.sha256(
        text.encode("utf-8")
    ).hexdigest()


# ------------------------------------------------------------------
# Load the immutable v1.1 notebook
# ------------------------------------------------------------------

input_sha256_before = sha256_file(
    NOTEBOOK_V1_1
)

with open(
    NOTEBOOK_V1_1,
    "r",
    encoding="utf-8",
) as notebook_file:
    notebook_v1_1 = json.load(notebook_file)

notebook_v1_2 = copy.deepcopy(
    notebook_v1_1
)


# ------------------------------------------------------------------
# Locate the exact extract_submitter function
# ------------------------------------------------------------------

function_start_marker = (
    "def extract_submitter(assertion):"
)

next_function_marker = (
    "def extract_origins(assertion):"
)

matching_cells = []

for cell_index, cell in enumerate(
    notebook_v1_2.get("cells", [])
):
    if cell.get("cell_type") != "code":
        continue

    source = "".join(
        cell.get("source", [])
    )

    if function_start_marker in source:
        matching_cells.append(
            {
                "cell_index": cell_index,
                "source": source,
            }
        )

if len(matching_cells) != 1:
    raise AssertionError(
        "Expected exactly one extract_submitter() definition, "
        f"but found {len(matching_cells)}."
    )

parser_cell_index = matching_cells[0][
    "cell_index"
]

original_source = matching_cells[0][
    "source"
]

function_start = original_source.index(
    function_start_marker
)

try:
    next_function_start = original_source.index(
        next_function_marker,
        function_start,
    )
except ValueError as exc:
    raise AssertionError(
        "Could not locate extract_origins() after "
        "extract_submitter()."
    ) from exc

original_function = original_source[
    function_start:next_function_start
].rstrip()


# ------------------------------------------------------------------
# Verify that the original function contains the known defect
# ------------------------------------------------------------------

required_original_fragments = [
    'element.get("submitterID")',
    'element.get("SubmitterID")',
    "return submitter, submitter_id",
]

missing_original_fragments = [
    fragment
    for fragment in required_original_fragments
    if fragment not in original_function
]

if missing_original_fragments:
    raise AssertionError(
        "The source function does not match the expected v1.1 "
        "implementation. Missing fragments: "
        f"{missing_original_fragments}"
    )


# ------------------------------------------------------------------
# Corrected historical parser function
# ------------------------------------------------------------------

corrected_function = '''def extract_submitter(assertion):
    """
    Extract the primary submitter name and primary organization ID
    from a historical ClinVarAssertion.

    Historical ClinVar XML stores:
    - the submitted organization name in
      ClinVarSubmissionID/@submitter;
    - the primary organization identifier in
      ClinVarAccession[@Type="SCV"]/@OrgID.

    Additional SubmitterDescription OrgID values are not substituted
    for the primary SCV organization identifier.
    """

    submitter = None
    submitter_id = None

    # Primary submitted organization name
    for element in descendants(
        assertion,
        "ClinVarSubmissionID"
    ):

        submitter = clean_text(
            element.get("submitter")
            or element.get("Submitter")
        )

        if submitter:
            break

    # Primary organization identifier attached to the SCV accession
    for element in descendants(
        assertion,
        "ClinVarAccession"
    ):

        accession_type = clean_text(
            element.get("Type")
        )

        accession = clean_text(
            element.get("Acc")
            or element.get("Accession")
        )

        is_scv_accession = (
            accession_type == "SCV"
            or (
                accession is not None
                and accession.startswith("SCV")
            )
        )

        if not is_scv_accession:
            continue

        submitter_id = clean_text(
            element.get("OrgID")
            or element.get("OrgId")
            or element.get("OrganizationID")
        )

        break

    return submitter, submitter_id'''


# ------------------------------------------------------------------
# Patch only the targeted function
# ------------------------------------------------------------------

patched_source = (
    original_source[:function_start]
    + corrected_function
    + "\n\n\n"
    + original_source[next_function_start:]
)

if patched_source == original_source:
    raise AssertionError(
        "The parser source did not change."
    )

# Syntax validation before saving
compile(
    patched_source,
    f"<notebook-cell-{parser_cell_index}>",
    "exec",
)

notebook_v1_2["cells"][
    parser_cell_index
]["source"] = patched_source.splitlines(
    keepends=True
)


# ------------------------------------------------------------------
# Add version metadata without changing analysis cells
# ------------------------------------------------------------------

metadata = notebook_v1_2.setdefault(
    "metadata",
    {}
)

metadata["ges_rag_parser_version"] = (
    "1.2.0"
)

metadata["ges_rag_parser_patch"] = {
    "patched_function":
        "extract_submitter",

    "patched_cell_index":
        parser_cell_index,

    "reason":
        (
            "Recover primary SCV organization identifiers from "
            "ClinVarAccession[@Type='SCV']/@OrgID."
        ),

    "source_notebook":
        NOTEBOOK_V1_1.name,

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "t1_information_used":
        False,

    "temporal_outcomes_used":
        False,

    "ges_model_output_used":
        False,
}


# ------------------------------------------------------------------
# Confirm only one code cell changed
# ------------------------------------------------------------------

changed_cell_indexes = []

for cell_index, (
    original_cell,
    patched_cell,
) in enumerate(
    zip(
        notebook_v1_1["cells"],
        notebook_v1_2["cells"],
    )
):
    if original_cell != patched_cell:
        changed_cell_indexes.append(
            cell_index
        )

if changed_cell_indexes != [
    parser_cell_index
]:
    raise AssertionError(
        "Unexpected notebook cells changed: "
        f"{changed_cell_indexes}"
    )


# ------------------------------------------------------------------
# Write versioned notebook
# ------------------------------------------------------------------

with open(
    NOTEBOOK_V1_2,
    "w",
    encoding="utf-8",
) as output_file:
    json.dump(
        notebook_v1_2,
        output_file,
        ensure_ascii=False,
        indent=1,
    )

output_sha256 = sha256_file(
    NOTEBOOK_V1_2
)

input_sha256_after = sha256_file(
    NOTEBOOK_V1_1
)

if input_sha256_before != input_sha256_after:
    raise AssertionError(
        "The v1.1 source notebook changed unexpectedly."
    )


# ------------------------------------------------------------------
# Readback verification
# ------------------------------------------------------------------

with open(
    NOTEBOOK_V1_2,
    "r",
    encoding="utf-8",
) as output_file:
    readback_notebook = json.load(
        output_file
    )

readback_source = "".join(
    readback_notebook["cells"][
        parser_cell_index
    ]["source"]
)

readback_checks = {
    "Corrected function exists once":
        readback_source.count(
            function_start_marker
        ) == 1,

    "Historical submitterID lookup removed":
        'element.get("submitterID")'
        not in readback_source,

    "Historical SubmitterID lookup removed":
        'element.get("SubmitterID")'
        not in readback_source,

    "SCV OrgID lookup present":
        'element.get("OrgID")'
        in readback_source,

    "SCV accession-type check present":
        'accession_type == "SCV"'
        in readback_source,

    "Additional submitter IDs not substituted":
        "SubmitterDescription"
        in readback_source,

    "Patched cell compiles":
        True,

    "Only parser cell changed":
        changed_cell_indexes
        == [parser_cell_index],

    "v1.1 checksum unchanged":
        input_sha256_before
        == input_sha256_after,
}

failed_readback_checks = [
    name
    for name, passed in readback_checks.items()
    if not passed
]

print("=" * 112)
print("STEP 4F.2 — CREATE PATCHED PARSER NOTEBOOK v1.2")
print("=" * 112)

print("\nPATCH TARGET")
print("-" * 112)

print(
    f"Parser cell index: "
    f"{parser_cell_index}"
)

print(
    "Patched function: "
    "extract_submitter"
)

print(
    "Primary OrgID source: "
    "ClinVarAssertion/"
    "ClinVarAccession[@Type='SCV']/@OrgID"
)

print("\nREADBACK VALIDATION")
print("-" * 112)

for check_name, passed in readback_checks.items():
    print(
        f"{'✅ PASS' if passed else '❌ FAIL'} "
        f"— {check_name}"
    )

if failed_readback_checks:
    raise AssertionError(
        "Notebook patch validation failed: "
        + "; ".join(failed_readback_checks)
    )


# ------------------------------------------------------------------
# Create patch manifest
# ------------------------------------------------------------------

patch_manifest = {
    "artifact_type":
        "Historical T0 parser submitter OrgID patch",

    "version":
        "1.2.0",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "input_notebook_path":
        str(NOTEBOOK_V1_1),

    "input_notebook_sha256":
        input_sha256_before,

    "output_notebook_path":
        str(NOTEBOOK_V1_2),

    "output_notebook_sha256":
        output_sha256,

    "output_notebook_bytes":
        NOTEBOOK_V1_2.stat().st_size,

    "patched_cell_index":
        parser_cell_index,

    "patched_function":
        "extract_submitter",

    "original_function_sha256":
        sha256_text(
            original_function
        ),

    "corrected_function_sha256":
        sha256_text(
            corrected_function
        ),

    "original_behavior":
        (
            "Attempted to read submitterID or SubmitterID from "
            "ClinVarSubmissionID, producing empty IDs."
        ),

    "corrected_behavior":
        (
            "Reads the primary submitter name from "
            "ClinVarSubmissionID/@submitter and the primary "
            "organization identifier from the SCV "
            "ClinVarAccession/@OrgID."
        ),

    "source_xml_path":
        (
            "ClinVarSet/ClinVarAssertion/"
            "ClinVarAccession[@Type='SCV']/@OrgID"
        ),

    "additional_submitter_policy":
        (
            "SubmitterDescription OrgID values are retained as "
            "separate source metadata and are not substituted for "
            "the primary SCV organization ID."
        ),

    "changed_code_cells": [
        parser_cell_index
    ],

    "source_notebook_modified":
        False,

    "t1_information_used":
        False,

    "temporal_outcomes_used":
        False,

    "ges_model_output_used":
        False,

    "validation_status":
        (
            "Static code and notebook readback validation passed; "
            "runtime source-record validation remains pending."
        ),

    "raw_xml_deletion_authorized":
        False,
}

with open(
    PATCH_MANIFEST,
    "w",
    encoding="utf-8",
) as manifest_file:
    json.dump(
        patch_manifest,
        manifest_file,
        indent=2,
        ensure_ascii=False,
        sort_keys=True,
    )

manifest_sha256 = sha256_file(
    PATCH_MANIFEST
)


# ------------------------------------------------------------------
# Final output
# ------------------------------------------------------------------

print("\nVERSIONED PARSER ARTIFACTS")
print("=" * 112)

print(
    f"✅ Patched notebook:\n"
    f"   {NOTEBOOK_V1_2}"
)

print(
    f"   SHA-256: "
    f"{output_sha256}"
)

print(
    f"   Size: "
    f"{NOTEBOOK_V1_2.stat().st_size / (1024 ** 2):,.3f} MB"
)

print(
    f"\n✅ Patch manifest:\n"
    f"   {PATCH_MANIFEST}"
)

print(
    f"   SHA-256: "
    f"{manifest_sha256}"
)

print("\nFINAL STATUS")
print("=" * 112)

print(
    "✅ Notebook v1.2 was created without overwriting v1.1."
)

print(
    "✅ Only extract_submitter() in the parser code cell changed."
)

print(
    "✅ The patched code reads primary SCV OrgID values from "
    "the correct historical XML location."
)

print(
    "✅ Static syntax and readback checks passed."
)

print(
    "⚠️ Do not delete the raw T0 XML yet."
)

print(
    "⚠️ Runtime validation of the patched function is still required."
)

print()
print("✅ STEP 4F.2 COMPLETE")

STEP 4F.2 — CREATE PATCHED PARSER NOTEBOOK v1.2

PATCH TARGET
----------------------------------------------------------------------------------------------------------------
Parser cell index: 13
Patched function: extract_submitter
Primary OrgID source: ClinVarAssertion/ClinVarAccession[@Type='SCV']/@OrgID

READBACK VALIDATION
----------------------------------------------------------------------------------------------------------------
✅ PASS — Corrected function exists once
✅ PASS — Historical submitterID lookup removed
✅ PASS — Historical SubmitterID lookup removed
✅ PASS — SCV OrgID lookup present
✅ PASS — SCV accession-type check present
✅ PASS — Additional submitter IDs not substituted
✅ PASS — Patched cell compiles
✅ PASS — Only parser cell changed
✅ PASS — v1.1 checksum unchanged

VERSIONED PARSER ARTIFACTS
✅ Patched notebook:
   /content/drive/MyDrive/Colab Notebooks/02_GES_temporal_validation_v1_2.ipynb
   SHA-256: a6c283113cbb8720428fd4a05934381c79a7a17c0b0666d3d05841dd1c0

In [44]:
# STEP 4F.2: Create versioned notebook v1.2 with corrected submitter OrgID parsing
# Preserves notebook v1.1 unchanged.
# Patches only extract_submitter() in the historical parser cell.

import copy
import hashlib
import json
from datetime import datetime, timezone
from pathlib import Path


# ------------------------------------------------------------------
# Paths
# ------------------------------------------------------------------

NOTEBOOK_V1_1 = Path(
    "/content/drive/MyDrive/Colab Notebooks/"
    "02_GES_temporal_validation_v1_1.ipynb"
)

NOTEBOOK_V1_2 = Path(
    "/content/drive/MyDrive/Colab Notebooks/"
    "02_GES_temporal_validation_v1_2.ipynb"
)

PATCH_MANIFEST = Path(
    "/content/drive/MyDrive/GES_RAG_Temporal_Study/"
    "configs/t0_parser_submitter_orgid_patch_manifest_v1_2.json"
)


# ------------------------------------------------------------------
# Safety checks
# ------------------------------------------------------------------

if not NOTEBOOK_V1_1.exists():
    raise FileNotFoundError(
        f"Source notebook not found:\n{NOTEBOOK_V1_1}"
    )

if NOTEBOOK_V1_2.exists():
    raise FileExistsError(
        f"Versioned notebook already exists:\n{NOTEBOOK_V1_2}\n"
        "Do not overwrite it."
    )

if PATCH_MANIFEST.exists():
    raise FileExistsError(
        f"Versioned patch manifest already exists:\n"
        f"{PATCH_MANIFEST}\n"
        "Do not overwrite it."
    )

PATCH_MANIFEST.parent.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------------
# Helpers
# ------------------------------------------------------------------

def sha256_file(path, chunk_size=8 * 1024 * 1024):
    """Calculate the SHA-256 checksum of a file."""
    digest = hashlib.sha256()

    with open(path, "rb") as file_handle:
        while True:
            block = file_handle.read(chunk_size)

            if not block:
                break

            digest.update(block)

    return digest.hexdigest()


def sha256_text(text):
    """Calculate SHA-256 for a text block."""
    return hashlib.sha256(
        text.encode("utf-8")
    ).hexdigest()


# ------------------------------------------------------------------
# Load the immutable v1.1 notebook
# ------------------------------------------------------------------

input_sha256_before = sha256_file(
    NOTEBOOK_V1_1
)

with open(
    NOTEBOOK_V1_1,
    "r",
    encoding="utf-8",
) as notebook_file:
    notebook_v1_1 = json.load(notebook_file)

notebook_v1_2 = copy.deepcopy(
    notebook_v1_1
)


# ------------------------------------------------------------------
# Locate the exact extract_submitter function
# ------------------------------------------------------------------

function_start_marker = (
    "def extract_submitter(assertion):"
)

next_function_marker = (
    "def extract_origins(assertion):"
)

matching_cells = []

for cell_index, cell in enumerate(
    notebook_v1_2.get("cells", [])
):
    if cell.get("cell_type") != "code":
        continue

    source = "".join(
        cell.get("source", [])
    )

    if function_start_marker in source:
        matching_cells.append(
            {
                "cell_index": cell_index,
                "source": source,
            }
        )

if len(matching_cells) != 1:
    raise AssertionError(
        "Expected exactly one extract_submitter() definition, "
        f"but found {len(matching_cells)}."
    )

parser_cell_index = matching_cells[0][
    "cell_index"
]

original_source = matching_cells[0][
    "source"
]

function_start = original_source.index(
    function_start_marker
)

try:
    next_function_start = original_source.index(
        next_function_marker,
        function_start,
    )
except ValueError as exc:
    raise AssertionError(
        "Could not locate extract_origins() after "
        "extract_submitter()."
    ) from exc

original_function = original_source[
    function_start:next_function_start
].rstrip()


# ------------------------------------------------------------------
# Verify that the original function contains the known defect
# ------------------------------------------------------------------

required_original_fragments = [
    'element.get("submitterID")',
    'element.get("SubmitterID")',
    "return submitter, submitter_id",
]

missing_original_fragments = [
    fragment
    for fragment in required_original_fragments
    if fragment not in original_function
]

if missing_original_fragments:
    raise AssertionError(
        "The source function does not match the expected v1.1 "
        "implementation. Missing fragments: "
        f"{missing_original_fragments}"
    )


# ------------------------------------------------------------------
# Corrected historical parser function
# ------------------------------------------------------------------

corrected_function = '''def extract_submitter(assertion):
    """
    Extract the primary submitter name and primary organization ID
    from a historical ClinVarAssertion.

    Historical ClinVar XML stores:
    - the submitted organization name in
      ClinVarSubmissionID/@submitter;
    - the primary organization identifier in
      ClinVarAccession[@Type="SCV"]/@OrgID.

    Additional SubmitterDescription OrgID values are not substituted
    for the primary SCV organization identifier.
    """

    submitter = None
    submitter_id = None

    # Primary submitted organization name
    for element in descendants(
        assertion,
        "ClinVarSubmissionID"
    ):

        submitter = clean_text(
            element.get("submitter")
            or element.get("Submitter")
        )

        if submitter:
            break

    # Primary organization identifier attached to the SCV accession
    for element in descendants(
        assertion,
        "ClinVarAccession"
    ):

        accession_type = clean_text(
            element.get("Type")
        )

        accession = clean_text(
            element.get("Acc")
            or element.get("Accession")
        )

        is_scv_accession = (
            accession_type == "SCV"
            or (
                accession is not None
                and accession.startswith("SCV")
            )
        )

        if not is_scv_accession:
            continue

        submitter_id = clean_text(
            element.get("OrgID")
            or element.get("OrgId")
            or element.get("OrganizationID")
        )

        break

    return submitter, submitter_id'''


# ------------------------------------------------------------------
# Patch only the targeted function
# ------------------------------------------------------------------

patched_source = (
    original_source[:function_start]
    + corrected_function
    + "\n\n\n"
    + original_source[next_function_start:]
)

if patched_source == original_source:
    raise AssertionError(
        "The parser source did not change."
    )

# Syntax validation before saving
compile(
    patched_source,
    f"<notebook-cell-{parser_cell_index}>",
    "exec",
)

notebook_v1_2["cells"][
    parser_cell_index
]["source"] = patched_source.splitlines(
    keepends=True
)


# ------------------------------------------------------------------
# Add version metadata without changing analysis cells
# ------------------------------------------------------------------

metadata = notebook_v1_2.setdefault(
    "metadata",
    {}
)

metadata["ges_rag_parser_version"] = (
    "1.2.0"
)

metadata["ges_rag_parser_patch"] = {
    "patched_function":
        "extract_submitter",

    "patched_cell_index":
        parser_cell_index,

    "reason":
        (
            "Recover primary SCV organization identifiers from "
            "ClinVarAccession[@Type='SCV']/@OrgID."
        ),

    "source_notebook":
        NOTEBOOK_V1_1.name,

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "t1_information_used":
        False,

    "temporal_outcomes_used":
        False,

    "ges_model_output_used":
        False,
}


# ------------------------------------------------------------------
# Confirm only one code cell changed
# ------------------------------------------------------------------

changed_cell_indexes = []

for cell_index, (
    original_cell,
    patched_cell,
) in enumerate(
    zip(
        notebook_v1_1["cells"],
        notebook_v1_2["cells"],
    )
):
    if original_cell != patched_cell:
        changed_cell_indexes.append(
            cell_index
        )

if changed_cell_indexes != [
    parser_cell_index
]:
    raise AssertionError(
        "Unexpected notebook cells changed: "
        f"{changed_cell_indexes}"
    )


# ------------------------------------------------------------------
# Write versioned notebook
# ------------------------------------------------------------------

with open(
    NOTEBOOK_V1_2,
    "w",
    encoding="utf-8",
) as output_file:
    json.dump(
        notebook_v1_2,
        output_file,
        ensure_ascii=False,
        indent=1,
    )

output_sha256 = sha256_file(
    NOTEBOOK_V1_2
)

input_sha256_after = sha256_file(
    NOTEBOOK_V1_1
)

if input_sha256_before != input_sha256_after:
    raise AssertionError(
        "The v1.1 source notebook changed unexpectedly."
    )


# ------------------------------------------------------------------
# Readback verification
# ------------------------------------------------------------------

with open(
    NOTEBOOK_V1_2,
    "r",
    encoding="utf-8",
) as output_file:
    readback_notebook = json.load(
        output_file
    )

readback_source = "".join(
    readback_notebook["cells"][
        parser_cell_index
    ]["source"]
)

readback_checks = {
    "Corrected function exists once":
        readback_source.count(
            function_start_marker
        ) == 1,

    "Historical submitterID lookup removed":
        'element.get("submitterID")'
        not in readback_source,

    "Historical SubmitterID lookup removed":
        'element.get("SubmitterID")'
        not in readback_source,

    "SCV OrgID lookup present":
        'element.get("OrgID")'
        in readback_source,

    "SCV accession-type check present":
        'accession_type == "SCV"'
        in readback_source,

    "Additional submitter IDs not substituted":
        "SubmitterDescription"
        in readback_source,

    "Patched cell compiles":
        True,

    "Only parser cell changed":
        changed_cell_indexes
        == [parser_cell_index],

    "v1.1 checksum unchanged":
        input_sha256_before
        == input_sha256_after,
}

failed_readback_checks = [
    name
    for name, passed in readback_checks.items()
    if not passed
]

print("=" * 112)
print("STEP 4F.2 — CREATE PATCHED PARSER NOTEBOOK v1.2")
print("=" * 112)

print("\nPATCH TARGET")
print("-" * 112)

print(
    f"Parser cell index: "
    f"{parser_cell_index}"
)

print(
    "Patched function: "
    "extract_submitter"
)

print(
    "Primary OrgID source: "
    "ClinVarAssertion/"
    "ClinVarAccession[@Type='SCV']/@OrgID"
)

print("\nREADBACK VALIDATION")
print("-" * 112)

for check_name, passed in readback_checks.items():
    print(
        f"{'✅ PASS' if passed else '❌ FAIL'} "
        f"— {check_name}"
    )

if failed_readback_checks:
    raise AssertionError(
        "Notebook patch validation failed: "
        + "; ".join(failed_readback_checks)
    )


# ------------------------------------------------------------------
# Create patch manifest
# ------------------------------------------------------------------

patch_manifest = {
    "artifact_type":
        "Historical T0 parser submitter OrgID patch",

    "version":
        "1.2.0",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "input_notebook_path":
        str(NOTEBOOK_V1_1),

    "input_notebook_sha256":
        input_sha256_before,

    "output_notebook_path":
        str(NOTEBOOK_V1_2),

    "output_notebook_sha256":
        output_sha256,

    "output_notebook_bytes":
        NOTEBOOK_V1_2.stat().st_size,

    "patched_cell_index":
        parser_cell_index,

    "patched_function":
        "extract_submitter",

    "original_function_sha256":
        sha256_text(
            original_function
        ),

    "corrected_function_sha256":
        sha256_text(
            corrected_function
        ),

    "original_behavior":
        (
            "Attempted to read submitterID or SubmitterID from "
            "ClinVarSubmissionID, producing empty IDs."
        ),

    "corrected_behavior":
        (
            "Reads the primary submitter name from "
            "ClinVarSubmissionID/@submitter and the primary "
            "organization identifier from the SCV "
            "ClinVarAccession/@OrgID."
        ),

    "source_xml_path":
        (
            "ClinVarSet/ClinVarAssertion/"
            "ClinVarAccession[@Type='SCV']/@OrgID"
        ),

    "additional_submitter_policy":
        (
            "SubmitterDescription OrgID values are retained as "
            "separate source metadata and are not substituted for "
            "the primary SCV organization ID."
        ),

    "changed_code_cells": [
        parser_cell_index
    ],

    "source_notebook_modified":
        False,

    "t1_information_used":
        False,

    "temporal_outcomes_used":
        False,

    "ges_model_output_used":
        False,

    "validation_status":
        (
            "Static code and notebook readback validation passed; "
            "runtime source-record validation remains pending."
        ),

    "raw_xml_deletion_authorized":
        False,
}

with open(
    PATCH_MANIFEST,
    "w",
    encoding="utf-8",
) as manifest_file:
    json.dump(
        patch_manifest,
        manifest_file,
        indent=2,
        ensure_ascii=False,
        sort_keys=True,
    )

manifest_sha256 = sha256_file(
    PATCH_MANIFEST
)


# ------------------------------------------------------------------
# Final output
# ------------------------------------------------------------------

print("\nVERSIONED PARSER ARTIFACTS")
print("=" * 112)

print(
    f"✅ Patched notebook:\n"
    f"   {NOTEBOOK_V1_2}"
)

print(
    f"   SHA-256: "
    f"{output_sha256}"
)

print(
    f"   Size: "
    f"{NOTEBOOK_V1_2.stat().st_size / (1024 ** 2):,.3f} MB"
)

print(
    f"\n✅ Patch manifest:\n"
    f"   {PATCH_MANIFEST}"
)

print(
    f"   SHA-256: "
    f"{manifest_sha256}"
)

print("\nFINAL STATUS")
print("=" * 112)

print(
    "✅ Notebook v1.2 was created without overwriting v1.1."
)

print(
    "✅ Only extract_submitter() in the parser code cell changed."
)

print(
    "✅ The patched code reads primary SCV OrgID values from "
    "the correct historical XML location."
)

print(
    "✅ Static syntax and readback checks passed."
)

print(
    "⚠️ Do not delete the raw T0 XML yet."
)

print(
    "⚠️ Runtime validation of the patched function is still required."
)

print()
print("✅ STEP 4F.2 COMPLETE")

FileExistsError: Versioned notebook already exists:
/content/drive/MyDrive/Colab Notebooks/02_GES_temporal_validation_v1_2.ipynb
Do not overwrite it.

In [ ]:
# STEP 4F.2A: Audit the existing parser notebook v1.2
# Read-only. Does not modify, delete, or overwrite any file.

import hashlib
import json
from pathlib import Path


NOTEBOOK_V1_1 = Path(
    "/content/drive/MyDrive/Colab Notebooks/"
    "02_GES_temporal_validation_v1_1.ipynb"
)

NOTEBOOK_V1_2 = Path(
    "/content/drive/MyDrive/Colab Notebooks/"
    "02_GES_temporal_validation_v1_2.ipynb"
)

PATCH_MANIFEST = Path(
    "/content/drive/MyDrive/GES_RAG_Temporal_Study/"
    "configs/t0_parser_submitter_orgid_patch_manifest_v1_2.json"
)


def sha256_file(path, chunk_size=8 * 1024 * 1024):
    """Calculate a file SHA-256 checksum."""
    digest = hashlib.sha256()

    with open(path, "rb") as file_handle:
        while True:
            block = file_handle.read(chunk_size)

            if not block:
                break

            digest.update(block)

    return digest.hexdigest()


def load_notebook(path):
    """Load and minimally validate a Jupyter notebook."""
    with open(path, "r", encoding="utf-8") as notebook_file:
        notebook = json.load(notebook_file)

    if not isinstance(notebook.get("cells"), list):
        raise AssertionError(
            f"Notebook has no valid cells list: {path}"
        )

    return notebook


def find_function_source(notebook, function_name):
    """
    Locate one function definition and return its cell index,
    entire cell source, and function block.
    """
    marker = f"def {function_name}("

    matches = []

    for cell_index, cell in enumerate(
        notebook.get("cells", [])
    ):
        if cell.get("cell_type") != "code":
            continue

        source = "".join(cell.get("source", []))

        if marker not in source:
            continue

        start = source.index(marker)

        remaining = source[start:].splitlines(
            keepends=True
        )

        function_lines = []

        for line_number, line in enumerate(remaining):

            if line_number == 0:
                function_lines.append(line)
                continue

            # Stop at the next top-level function or class
            if (
                line.startswith("def ")
                or line.startswith("class ")
            ):
                break

            function_lines.append(line)

        matches.append({
            "cell_index": cell_index,
            "cell_source": source,
            "function_source": "".join(
                function_lines
            ).rstrip(),
        })

    if len(matches) != 1:
        raise AssertionError(
            f"Expected one {function_name} definition, "
            f"found {len(matches)}."
        )

    return matches[0]


# ------------------------------------------------------------------
# File existence and JSON readability
# ------------------------------------------------------------------

for required_path in [
    NOTEBOOK_V1_1,
    NOTEBOOK_V1_2,
]:
    if not required_path.exists():
        raise FileNotFoundError(
            f"Required notebook is missing: {required_path}"
        )

v1_1 = load_notebook(NOTEBOOK_V1_1)
v1_2 = load_notebook(NOTEBOOK_V1_2)

v1_1_sha256 = sha256_file(NOTEBOOK_V1_1)
v1_2_sha256 = sha256_file(NOTEBOOK_V1_2)


# ------------------------------------------------------------------
# Compare notebook structures and cells
# ------------------------------------------------------------------

same_cell_count = (
    len(v1_1["cells"]) == len(v1_2["cells"])
)

changed_cell_indexes = []

maximum_cells = max(
    len(v1_1["cells"]),
    len(v1_2["cells"]),
)

for cell_index in range(maximum_cells):

    cell_v1_1 = (
        v1_1["cells"][cell_index]
        if cell_index < len(v1_1["cells"])
        else None
    )

    cell_v1_2 = (
        v1_2["cells"][cell_index]
        if cell_index < len(v1_2["cells"])
        else None
    )

    if cell_v1_1 != cell_v1_2:
        changed_cell_indexes.append(cell_index)


# ------------------------------------------------------------------
# Inspect extract_submitter() in both notebooks
# ------------------------------------------------------------------

function_v1_1 = find_function_source(
    v1_1,
    "extract_submitter"
)

function_v1_2 = find_function_source(
    v1_2,
    "extract_submitter"
)

v1_2_function_source = function_v1_2[
    "function_source"
]

# Syntax-check the complete patched code cell
compile(
    function_v1_2["cell_source"],
    f"<v1.2-cell-{function_v1_2['cell_index']}>",
    "exec",
)

expected_patch_checks = {
    "v1.1 and v1.2 have the same number of cells":
        same_cell_count,

    "extract_submitter remains in cell 13":
        (
            function_v1_1["cell_index"] == 13
            and function_v1_2["cell_index"] == 13
        ),

    "extract_submitter changed from v1.1":
        (
            function_v1_1["function_source"]
            != function_v1_2["function_source"]
        ),

    "Old submitterID lookup removed":
        'element.get("submitterID")'
        not in v1_2_function_source,

    "Old SubmitterID lookup removed":
        'element.get("SubmitterID")'
        not in v1_2_function_source,

    "ClinVarSubmissionID name lookup retained":
        "ClinVarSubmissionID"
        in v1_2_function_source,

    "SCV ClinVarAccession lookup present":
        "ClinVarAccession"
        in v1_2_function_source,

    "OrgID lookup present":
        'element.get("OrgID")'
        in v1_2_function_source,

    "SCV type or accession check present":
        (
            'accession_type == "SCV"'
            in v1_2_function_source
            or 'startswith("SCV")'
            in v1_2_function_source
        ),

    "Patched parser code cell compiles":
        True,
}


# ------------------------------------------------------------------
# Inspect optional patch metadata and manifest
# ------------------------------------------------------------------

patch_metadata = v1_2.get(
    "metadata",
    {}
).get(
    "ges_rag_parser_patch"
)

manifest_exists = PATCH_MANIFEST.exists()
manifest_data = None
manifest_sha256 = None
manifest_output_checksum_matches = None

if manifest_exists:

    with open(
        PATCH_MANIFEST,
        "r",
        encoding="utf-8",
    ) as manifest_file:
        manifest_data = json.load(manifest_file)

    manifest_sha256 = sha256_file(
        PATCH_MANIFEST
    )

    manifest_output_checksum_matches = (
        manifest_data.get(
            "output_notebook_sha256"
        ) == v1_2_sha256
    )


# ------------------------------------------------------------------
# Print audit results
# ------------------------------------------------------------------

print("=" * 112)
print("STEP 4F.2A — EXISTING NOTEBOOK v1.2 AUDIT")
print("=" * 112)

print("\nFILE STATUS")
print("-" * 112)

print(f"v1.1 notebook: {NOTEBOOK_V1_1}")
print(
    f"   Size: "
    f"{NOTEBOOK_V1_1.stat().st_size / (1024 ** 2):,.3f} MB"
)
print(f"   SHA-256: {v1_1_sha256}")

print(f"\nv1.2 notebook: {NOTEBOOK_V1_2}")
print(
    f"   Size: "
    f"{NOTEBOOK_V1_2.stat().st_size / (1024 ** 2):,.3f} MB"
)
print(f"   SHA-256: {v1_2_sha256}")

print("\nNOTEBOOK COMPARISON")
print("-" * 112)

print(f"v1.1 cell count: {len(v1_1['cells']):,}")
print(f"v1.2 cell count: {len(v1_2['cells']):,}")
print(f"Changed cell indexes: {changed_cell_indexes}")

print("\nPATCH VALIDATION")
print("-" * 112)

for check_name, passed in expected_patch_checks.items():
    print(
        f"{'✅ PASS' if passed else '❌ FAIL'} — "
        f"{check_name}"
    )

print("\nEXISTING v1.2 extract_submitter() FUNCTION")
print("-" * 112)

for line_number, line in enumerate(
    v1_2_function_source.splitlines(),
    start=1,
):
    print(f"{line_number:03d}: {line}")


print("\nPATCH METADATA")
print("-" * 112)

if patch_metadata is None:
    print("⚠️ No ges_rag_parser_patch metadata found.")
else:
    print(
        json.dumps(
            patch_metadata,
            indent=2,
            ensure_ascii=False,
        )
    )


print("\nPATCH MANIFEST STATUS")
print("-" * 112)

print(f"Manifest exists: {manifest_exists}")

if manifest_exists:
    print(f"Manifest path: {PATCH_MANIFEST}")
    print(f"Manifest SHA-256: {manifest_sha256}")
    print(
        "Manifest output-notebook checksum matches v1.2:",
        manifest_output_checksum_matches,
    )

    print(
        "Manifest validation status:",
        manifest_data.get("validation_status"),
    )
else:
    print(
        "No v1.2 parser patch manifest currently exists."
    )


failed_patch_checks = [
    name
    for name, passed in expected_patch_checks.items()
    if not passed
]

print("\nAUDIT DECISION")
print("=" * 112)

if failed_patch_checks:
    print("❌ EXISTING v1.2 CANNOT YET BE ACCEPTED")
    print(
        "Failed checks:",
        "; ".join(failed_patch_checks),
    )
else:
    print("✅ EXISTING v1.2 CONTAINS THE EXPECTED STATIC PATCH")

    if (
        manifest_exists
        and manifest_output_checksum_matches
    ):
        print(
            "✅ Its patch manifest also matches the notebook checksum."
        )
    elif manifest_exists:
        print(
            "⚠️ A manifest exists, but its output checksum does "
            "not match the notebook."
        )
    else:
        print(
            "⚠️ The notebook is patched, but a versioned patch "
            "manifest still needs to be created."
        )

print()
print("✅ STEP 4F.2A COMPLETE")
print("No notebook, dataset, XML, or manifest was modified.")

In [45]:
# STEP 4F.2A: Audit the existing parser notebook v1.2
# Read-only. Does not modify, delete, or overwrite any file.

import hashlib
import json
from pathlib import Path


NOTEBOOK_V1_1 = Path(
    "/content/drive/MyDrive/Colab Notebooks/"
    "02_GES_temporal_validation_v1_1.ipynb"
)

NOTEBOOK_V1_2 = Path(
    "/content/drive/MyDrive/Colab Notebooks/"
    "02_GES_temporal_validation_v1_2.ipynb"
)

PATCH_MANIFEST = Path(
    "/content/drive/MyDrive/GES_RAG_Temporal_Study/"
    "configs/t0_parser_submitter_orgid_patch_manifest_v1_2.json"
)


def sha256_file(path, chunk_size=8 * 1024 * 1024):
    """Calculate a file SHA-256 checksum."""
    digest = hashlib.sha256()

    with open(path, "rb") as file_handle:
        while True:
            block = file_handle.read(chunk_size)

            if not block:
                break

            digest.update(block)

    return digest.hexdigest()


def load_notebook(path):
    """Load and minimally validate a Jupyter notebook."""
    with open(path, "r", encoding="utf-8") as notebook_file:
        notebook = json.load(notebook_file)

    if not isinstance(notebook.get("cells"), list):
        raise AssertionError(
            f"Notebook has no valid cells list: {path}"
        )

    return notebook


def find_function_source(notebook, function_name):
    """
    Locate one function definition and return its cell index,
    entire cell source, and function block.
    """
    marker = f"def {function_name}("

    matches = []

    for cell_index, cell in enumerate(
        notebook.get("cells", [])
    ):
        if cell.get("cell_type") != "code":
            continue

        source = "".join(cell.get("source", []))

        if marker not in source:
            continue

        start = source.index(marker)

        remaining = source[start:].splitlines(
            keepends=True
        )

        function_lines = []

        for line_number, line in enumerate(remaining):

            if line_number == 0:
                function_lines.append(line)
                continue

            # Stop at the next top-level function or class
            if (
                line.startswith("def ")
                or line.startswith("class ")
            ):
                break

            function_lines.append(line)

        matches.append({
            "cell_index": cell_index,
            "cell_source": source,
            "function_source": "".join(
                function_lines
            ).rstrip(),
        })

    if len(matches) != 1:
        raise AssertionError(
            f"Expected one {function_name} definition, "
            f"found {len(matches)}."
        )

    return matches[0]


# ------------------------------------------------------------------
# File existence and JSON readability
# ------------------------------------------------------------------

for required_path in [
    NOTEBOOK_V1_1,
    NOTEBOOK_V1_2,
]:
    if not required_path.exists():
        raise FileNotFoundError(
            f"Required notebook is missing: {required_path}"
        )

v1_1 = load_notebook(NOTEBOOK_V1_1)
v1_2 = load_notebook(NOTEBOOK_V1_2)

v1_1_sha256 = sha256_file(NOTEBOOK_V1_1)
v1_2_sha256 = sha256_file(NOTEBOOK_V1_2)


# ------------------------------------------------------------------
# Compare notebook structures and cells
# ------------------------------------------------------------------

same_cell_count = (
    len(v1_1["cells"]) == len(v1_2["cells"])
)

changed_cell_indexes = []

maximum_cells = max(
    len(v1_1["cells"]),
    len(v1_2["cells"]),
)

for cell_index in range(maximum_cells):

    cell_v1_1 = (
        v1_1["cells"][cell_index]
        if cell_index < len(v1_1["cells"])
        else None
    )

    cell_v1_2 = (
        v1_2["cells"][cell_index]
        if cell_index < len(v1_2["cells"])
        else None
    )

    if cell_v1_1 != cell_v1_2:
        changed_cell_indexes.append(cell_index)


# ------------------------------------------------------------------
# Inspect extract_submitter() in both notebooks
# ------------------------------------------------------------------

function_v1_1 = find_function_source(
    v1_1,
    "extract_submitter"
)

function_v1_2 = find_function_source(
    v1_2,
    "extract_submitter"
)

v1_2_function_source = function_v1_2[
    "function_source"
]

# Syntax-check the complete patched code cell
compile(
    function_v1_2["cell_source"],
    f"<v1.2-cell-{function_v1_2['cell_index']}>",
    "exec",
)

expected_patch_checks = {
    "v1.1 and v1.2 have the same number of cells":
        same_cell_count,

    "extract_submitter remains in cell 13":
        (
            function_v1_1["cell_index"] == 13
            and function_v1_2["cell_index"] == 13
        ),

    "extract_submitter changed from v1.1":
        (
            function_v1_1["function_source"]
            != function_v1_2["function_source"]
        ),

    "Old submitterID lookup removed":
        'element.get("submitterID")'
        not in v1_2_function_source,

    "Old SubmitterID lookup removed":
        'element.get("SubmitterID")'
        not in v1_2_function_source,

    "ClinVarSubmissionID name lookup retained":
        "ClinVarSubmissionID"
        in v1_2_function_source,

    "SCV ClinVarAccession lookup present":
        "ClinVarAccession"
        in v1_2_function_source,

    "OrgID lookup present":
        'element.get("OrgID")'
        in v1_2_function_source,

    "SCV type or accession check present":
        (
            'accession_type == "SCV"'
            in v1_2_function_source
            or 'startswith("SCV")'
            in v1_2_function_source
        ),

    "Patched parser code cell compiles":
        True,
}


# ------------------------------------------------------------------
# Inspect optional patch metadata and manifest
# ------------------------------------------------------------------

patch_metadata = v1_2.get(
    "metadata",
    {}
).get(
    "ges_rag_parser_patch"
)

manifest_exists = PATCH_MANIFEST.exists()
manifest_data = None
manifest_sha256 = None
manifest_output_checksum_matches = None

if manifest_exists:

    with open(
        PATCH_MANIFEST,
        "r",
        encoding="utf-8",
    ) as manifest_file:
        manifest_data = json.load(manifest_file)

    manifest_sha256 = sha256_file(
        PATCH_MANIFEST
    )

    manifest_output_checksum_matches = (
        manifest_data.get(
            "output_notebook_sha256"
        ) == v1_2_sha256
    )


# ------------------------------------------------------------------
# Print audit results
# ------------------------------------------------------------------

print("=" * 112)
print("STEP 4F.2A — EXISTING NOTEBOOK v1.2 AUDIT")
print("=" * 112)

print("\nFILE STATUS")
print("-" * 112)

print(f"v1.1 notebook: {NOTEBOOK_V1_1}")
print(
    f"   Size: "
    f"{NOTEBOOK_V1_1.stat().st_size / (1024 ** 2):,.3f} MB"
)
print(f"   SHA-256: {v1_1_sha256}")

print(f"\nv1.2 notebook: {NOTEBOOK_V1_2}")
print(
    f"   Size: "
    f"{NOTEBOOK_V1_2.stat().st_size / (1024 ** 2):,.3f} MB"
)
print(f"   SHA-256: {v1_2_sha256}")

print("\nNOTEBOOK COMPARISON")
print("-" * 112)

print(f"v1.1 cell count: {len(v1_1['cells']):,}")
print(f"v1.2 cell count: {len(v1_2['cells']):,}")
print(f"Changed cell indexes: {changed_cell_indexes}")

print("\nPATCH VALIDATION")
print("-" * 112)

for check_name, passed in expected_patch_checks.items():
    print(
        f"{'✅ PASS' if passed else '❌ FAIL'} — "
        f"{check_name}"
    )

print("\nEXISTING v1.2 extract_submitter() FUNCTION")
print("-" * 112)

for line_number, line in enumerate(
    v1_2_function_source.splitlines(),
    start=1,
):
    print(f"{line_number:03d}: {line}")


print("\nPATCH METADATA")
print("-" * 112)

if patch_metadata is None:
    print("⚠️ No ges_rag_parser_patch metadata found.")
else:
    print(
        json.dumps(
            patch_metadata,
            indent=2,
            ensure_ascii=False,
        )
    )


print("\nPATCH MANIFEST STATUS")
print("-" * 112)

print(f"Manifest exists: {manifest_exists}")

if manifest_exists:
    print(f"Manifest path: {PATCH_MANIFEST}")
    print(f"Manifest SHA-256: {manifest_sha256}")
    print(
        "Manifest output-notebook checksum matches v1.2:",
        manifest_output_checksum_matches,
    )

    print(
        "Manifest validation status:",
        manifest_data.get("validation_status"),
    )
else:
    print(
        "No v1.2 parser patch manifest currently exists."
    )


failed_patch_checks = [
    name
    for name, passed in expected_patch_checks.items()
    if not passed
]

print("\nAUDIT DECISION")
print("=" * 112)

if failed_patch_checks:
    print("❌ EXISTING v1.2 CANNOT YET BE ACCEPTED")
    print(
        "Failed checks:",
        "; ".join(failed_patch_checks),
    )
else:
    print("✅ EXISTING v1.2 CONTAINS THE EXPECTED STATIC PATCH")

    if (
        manifest_exists
        and manifest_output_checksum_matches
    ):
        print(
            "✅ Its patch manifest also matches the notebook checksum."
        )
    elif manifest_exists:
        print(
            "⚠️ A manifest exists, but its output checksum does "
            "not match the notebook."
        )
    else:
        print(
            "⚠️ The notebook is patched, but a versioned patch "
            "manifest still needs to be created."
        )

print()
print("✅ STEP 4F.2A COMPLETE")
print("No notebook, dataset, XML, or manifest was modified.")

STEP 4F.2A — EXISTING NOTEBOOK v1.2 AUDIT

FILE STATUS
----------------------------------------------------------------------------------------------------------------
v1.1 notebook: /content/drive/MyDrive/Colab Notebooks/02_GES_temporal_validation_v1_1.ipynb
   Size: 0.583 MB
   SHA-256: 856ca46cedfc10dc17fd1c80b651a742a77827430808c4a862abe1145ba27531

v1.2 notebook: /content/drive/MyDrive/Colab Notebooks/02_GES_temporal_validation_v1_2.ipynb
   Size: 0.585 MB
   SHA-256: a6c283113cbb8720428fd4a05934381c79a7a17c0b0666d3d05841dd1c07a966

NOTEBOOK COMPARISON
----------------------------------------------------------------------------------------------------------------
v1.1 cell count: 24
v1.2 cell count: 24
Changed cell indexes: [13]

PATCH VALIDATION
----------------------------------------------------------------------------------------------------------------
✅ PASS — v1.1 and v1.2 have the same number of cells
✅ PASS — extract_submitter remains in cell 13
✅ PASS — extract_submitter

In [46]:
# STEP 4F.3: Full runtime validation of the patched extract_submitter()
# Uses the actual function stored in notebook v1.2.
# Compares all 71,659 RCVs and 100,633 SCVs against:
#   1. the verified source mapping, and
#   2. the repaired T0 Parquet v1.2.
# No existing artifact is modified.

import ast
import gzip
import hashlib
import json
import re
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from lxml import etree


# ------------------------------------------------------------------
# Paths
# ------------------------------------------------------------------

RAW_XML = Path(
    "/content/clinvar_rcv_raw/ClinVarFullRelease_2023-01.xml.gz"
)

NOTEBOOK_V1_2 = Path(
    "/content/drive/MyDrive/Colab Notebooks/"
    "02_GES_temporal_validation_v1_2.ipynb"
)

PATCH_MANIFEST_PATH = Path(
    "/content/drive/MyDrive/GES_RAG_Temporal_Study/"
    "configs/t0_parser_submitter_orgid_patch_manifest_v1_2.json"
)

T0_V1_2_PATH = Path(
    "/content/drive/MyDrive/GES_RAG_Temporal_Study/"
    "data_interim/t0_rcv_target_genes_corrected_v1_2.parquet"
)

REPAIR_MANIFEST_PATH = Path(
    "/content/drive/MyDrive/GES_RAG_Temporal_Study/"
    "configs/clinvar_t0_submitter_id_repair_manifest_v1_2.json"
)

ORGID_MAP_PATH = Path(
    "/content/drive/MyDrive/GES_RAG_Temporal_Study/"
    "data_interim/t0_rcv_submitter_orgid_source_map_v1_2.parquet"
)

ORGID_SOURCE_MANIFEST_PATH = Path(
    "/content/drive/MyDrive/GES_RAG_Temporal_Study/"
    "configs/clinvar_t0_submitter_orgid_source_audit_manifest_v1_2.json"
)

RUNTIME_MANIFEST_PATH = Path(
    "/content/drive/MyDrive/GES_RAG_Temporal_Study/"
    "configs/t0_parser_submitter_orgid_runtime_validation_manifest_v1_2.json"
)


# ------------------------------------------------------------------
# Safety and existence checks
# ------------------------------------------------------------------

required_paths = [
    RAW_XML,
    NOTEBOOK_V1_2,
    PATCH_MANIFEST_PATH,
    T0_V1_2_PATH,
    REPAIR_MANIFEST_PATH,
    ORGID_MAP_PATH,
    ORGID_SOURCE_MANIFEST_PATH,
]

for required_path in required_paths:
    if not required_path.exists():
        raise FileNotFoundError(
            f"Required file is missing:\n{required_path}"
        )

if RUNTIME_MANIFEST_PATH.exists():
    raise FileExistsError(
        f"Runtime-validation manifest already exists:\n"
        f"{RUNTIME_MANIFEST_PATH}\n"
        "Do not overwrite it."
    )


# ------------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------------

def sha256_file(path, chunk_size=8 * 1024 * 1024):
    """Calculate SHA-256 without loading a complete file into memory."""
    digest = hashlib.sha256()

    with open(path, "rb") as file_handle:
        while True:
            block = file_handle.read(chunk_size)

            if not block:
                break

            digest.update(block)

    return digest.hexdigest()


def parse_json_list(value):
    """Return a validated JSON list."""
    parsed = json.loads(value) if isinstance(value, str) else value

    if not isinstance(parsed, list):
        raise TypeError(
            f"Expected list, found {type(parsed).__name__}"
        )

    return parsed


def normalized_optional_text(value):
    """Normalize a value for deterministic comparison."""
    if value is None:
        return None

    text = re.sub(r"\s+", " ", str(value)).strip()

    return text if text else None


def normalize_record(record):
    """Normalize one SCV submitter record for exact comparison."""
    return {
        "scv_accession":
            normalized_optional_text(
                record.get("scv_accession")
            ),

        "primary_submitter_name":
            normalized_optional_text(
                record.get("primary_submitter_name")
            ),

        "primary_org_id":
            normalized_optional_text(
                record.get("primary_org_id")
            ),
    }


def sort_scv_records(records):
    """Normalize and sort submitted SCV records by accession."""
    return sorted(
        [normalize_record(record) for record in records],
        key=lambda record: (
            record["scv_accession"] or "",
            record["primary_submitter_name"] or "",
            record["primary_org_id"] or "",
        ),
    )


def find_scv_accession(assertion):
    """Find the submitted SCV accession inside one assertion."""
    for node in assertion.iter():

        if etree.QName(node).localname != "ClinVarAccession":
            continue

        accession = normalized_optional_text(
            node.get("Acc")
            or node.get("Accession")
        )

        accession_type = normalized_optional_text(
            node.get("Type")
        )

        if (
            accession_type == "SCV"
            or (
                accession is not None
                and accession.startswith("SCV")
            )
        ):
            return accession

    return None


def find_rcv_accession(clinvar_set):
    """Find the aggregate RCV accession in one ClinVarSet."""
    for node in clinvar_set.iter():

        if etree.QName(node).localname != "ClinVarAccession":
            continue

        accession = normalized_optional_text(
            node.get("Acc")
            or node.get("Accession")
        )

        accession_type = normalized_optional_text(
            node.get("Type")
        )

        if (
            accession_type == "RCV"
            or (
                accession is not None
                and accession.startswith("RCV")
            )
        ):
            return accession

    return None


# ------------------------------------------------------------------
# Verify checksums recorded by existing manifests
# ------------------------------------------------------------------

with open(
    PATCH_MANIFEST_PATH,
    "r",
    encoding="utf-8",
) as file_handle:
    patch_manifest = json.load(file_handle)

with open(
    REPAIR_MANIFEST_PATH,
    "r",
    encoding="utf-8",
) as file_handle:
    repair_manifest = json.load(file_handle)

with open(
    ORGID_SOURCE_MANIFEST_PATH,
    "r",
    encoding="utf-8",
) as file_handle:
    source_audit_manifest = json.load(file_handle)

notebook_sha256 = sha256_file(NOTEBOOK_V1_2)
parquet_sha256 = sha256_file(T0_V1_2_PATH)
mapping_sha256 = sha256_file(ORGID_MAP_PATH)

checksum_checks = {
    "Notebook matches parser-patch manifest":
        notebook_sha256
        == patch_manifest.get("output_notebook_sha256"),

    "T0 v1.2 matches repair manifest":
        parquet_sha256
        == repair_manifest.get("output_artifact_sha256"),

    "OrgID mapping matches source-audit manifest":
        mapping_sha256
        == source_audit_manifest.get("mapping_artifact_sha256"),
}

failed_checksum_checks = [
    name
    for name, passed in checksum_checks.items()
    if not passed
]

if failed_checksum_checks:
    raise AssertionError(
        "Checksum validation failed: "
        + "; ".join(failed_checksum_checks)
    )


# ------------------------------------------------------------------
# Extract the actual helper functions from notebook v1.2
# ------------------------------------------------------------------

with open(
    NOTEBOOK_V1_2,
    "r",
    encoding="utf-8",
) as notebook_file:
    notebook = json.load(notebook_file)


FUNCTIONS_REQUIRED = [
    "local_name",
    "clean_text",
    "descendants",
    "extract_submitter",
]

function_sources = {}


for cell_index, cell in enumerate(
    notebook.get("cells", [])
):
    if cell.get("cell_type") != "code":
        continue

    cell_source = "".join(cell.get("source", []))

    try:
        parsed_cell = ast.parse(cell_source)
    except SyntaxError:
        continue

    cell_lines = cell_source.splitlines(
        keepends=True
    )

    for node in parsed_cell.body:

        if not isinstance(
            node,
            (ast.FunctionDef, ast.AsyncFunctionDef),
        ):
            continue

        if node.name not in FUNCTIONS_REQUIRED:
            continue

        if node.end_lineno is None:
            raise AssertionError(
                f"Cannot determine source boundary for {node.name}."
            )

        source = "".join(
            cell_lines[
                node.lineno - 1:
                node.end_lineno
            ]
        )

        function_sources[node.name] = {
            "cell_index": cell_index,
            "source": source,
        }


missing_functions = [
    function_name
    for function_name in FUNCTIONS_REQUIRED
    if function_name not in function_sources
]

if missing_functions:
    raise AssertionError(
        "Required notebook functions were not found: "
        f"{missing_functions}"
    )


# Execute the exact notebook function definitions
runtime_namespace = {
    "re": re,
    "etree": etree,
}

for function_name in FUNCTIONS_REQUIRED:

    source = function_sources[
        function_name
    ]["source"]

    compile(
        source,
        f"<notebook-function-{function_name}>",
        "exec",
    )

    exec(
        source,
        runtime_namespace,
    )


runtime_extract_submitter = runtime_namespace[
    "extract_submitter"
]

runtime_local_name = runtime_namespace[
    "local_name"
]


# ------------------------------------------------------------------
# Load expected mapping and repaired T0 v1.2
# ------------------------------------------------------------------

mapping_df = pd.read_parquet(
    ORGID_MAP_PATH
)

t0_v1_2_df = pd.read_parquet(
    T0_V1_2_PATH
)


if len(mapping_df) != 71_659:
    raise AssertionError(
        f"Expected 71,659 mapping rows, "
        f"found {len(mapping_df):,}."
    )

if len(t0_v1_2_df) != 71_659:
    raise AssertionError(
        f"Expected 71,659 repaired rows, "
        f"found {len(t0_v1_2_df):,}."
    )


mapping_lookup = {}

for row in mapping_df[
    [
        "rcv_accession",
        "source_scv_submitter_org_map_json",
    ]
].itertuples(index=False):

    mapping_lookup[row.rcv_accession] = sort_scv_records(
        json.loads(
            row.source_scv_submitter_org_map_json
        )
    )


parquet_lookup = {}

for row in t0_v1_2_df[
    [
        "rcv_accession",
        "scv_count_xml",
        "submitters_json",
        "submitter_ids_json",
    ]
].itertuples(index=False):

    parquet_lookup[row.rcv_accession] = {
        "scv_count":
            int(row.scv_count_xml),

        "submitters":
            sorted({
                normalized
                for value in parse_json_list(
                    row.submitters_json
                )
                if (
                    normalized :=
                    normalized_optional_text(value)
                ) is not None
            }),

        "submitter_ids":
            sorted({
                normalized
                for value in parse_json_list(
                    row.submitter_ids_json
                )
                if (
                    normalized :=
                    normalized_optional_text(value)
                ) is not None
            }),
    }


target_rcvs = set(mapping_lookup)

if target_rcvs != set(parquet_lookup):
    raise AssertionError(
        "The mapping and repaired Parquet do not contain "
        "the same RCV accession set."
    )


# ------------------------------------------------------------------
# Full XML runtime validation
# ------------------------------------------------------------------

records_scanned = 0
target_rcvs_found = 0
runtime_scvs_processed = 0

observed_rcvs = set()
duplicate_target_rcvs = 0

scv_count_mismatch_count = 0
mapping_record_mismatch_count = 0
parquet_submitter_name_mismatch_count = 0
parquet_submitter_id_mismatch_count = 0
runtime_exception_count = 0

scv_count_examples = []
mapping_mismatch_examples = []
submitter_name_examples = []
submitter_id_examples = []
runtime_exception_examples = []


print("=" * 116)
print("STEP 4F.3 — FULL PATCHED-PARSER RUNTIME VALIDATION")
print("=" * 116)

print(f"Patched notebook: {NOTEBOOK_V1_2}")
print(f"Patched function cell: {function_sources['extract_submitter']['cell_index']}")
print(f"Target RCVs: {len(target_rcvs):,}")
print("Beginning full streaming XML validation...")


with gzip.open(RAW_XML, "rb") as xml_stream:

    context = etree.iterparse(
        xml_stream,
        events=("end",),
        tag="ClinVarSet",
        huge_tree=True,
        recover=False,
    )

    for _, clinvar_set in context:

        records_scanned += 1

        rcv_accession = find_rcv_accession(
            clinvar_set
        )

        if rcv_accession in target_rcvs:

            if rcv_accession in observed_rcvs:
                duplicate_target_rcvs += 1
            else:
                observed_rcvs.add(
                    rcv_accession
                )

            target_rcvs_found += 1

            runtime_records = []

            for child in clinvar_set:

                if runtime_local_name(
                    child
                ) != "ClinVarAssertion":
                    continue

                scv_accession = (
                    find_scv_accession(child)
                )

                try:
                    submitter_name, submitter_id = (
                        runtime_extract_submitter(
                            child
                        )
                    )
                except Exception as exc:
                    runtime_exception_count += 1

                    if len(runtime_exception_examples) < 20:
                        runtime_exception_examples.append({
                            "rcv_accession":
                                rcv_accession,

                            "scv_accession":
                                scv_accession,

                            "exception_type":
                                type(exc).__name__,

                            "exception_message":
                                str(exc),
                        })

                    continue

                runtime_records.append({
                    "scv_accession":
                        scv_accession,

                    "primary_submitter_name":
                        submitter_name,

                    "primary_org_id":
                        submitter_id,
                })

                runtime_scvs_processed += 1

            runtime_records = sort_scv_records(
                runtime_records
            )

            expected_mapping_records = (
                mapping_lookup[
                    rcv_accession
                ]
            )

            parquet_record = (
                parquet_lookup[
                    rcv_accession
                ]
            )

            runtime_submitters = sorted({
                record["primary_submitter_name"]
                for record in runtime_records
                if record["primary_submitter_name"]
            })

            runtime_submitter_ids = sorted({
                record["primary_org_id"]
                for record in runtime_records
                if record["primary_org_id"]
            })

            if (
                len(runtime_records)
                != parquet_record["scv_count"]
            ):
                scv_count_mismatch_count += 1

                if len(scv_count_examples) < 20:
                    scv_count_examples.append({
                        "rcv_accession":
                            rcv_accession,

                        "runtime_scv_count":
                            len(runtime_records),

                        "parquet_scv_count":
                            parquet_record["scv_count"],
                    })

            if (
                runtime_records
                != expected_mapping_records
            ):
                mapping_record_mismatch_count += 1

                if len(mapping_mismatch_examples) < 20:
                    mapping_mismatch_examples.append({
                        "rcv_accession":
                            rcv_accession,

                        "runtime_records":
                            runtime_records,

                        "expected_mapping_records":
                            expected_mapping_records,
                    })

            if (
                runtime_submitters
                != parquet_record["submitters"]
            ):
                parquet_submitter_name_mismatch_count += 1

                if len(submitter_name_examples) < 20:
                    submitter_name_examples.append({
                        "rcv_accession":
                            rcv_accession,

                        "runtime_submitters":
                            runtime_submitters,

                        "parquet_submitters":
                            parquet_record["submitters"],
                    })

            if (
                runtime_submitter_ids
                != parquet_record["submitter_ids"]
            ):
                parquet_submitter_id_mismatch_count += 1

                if len(submitter_id_examples) < 20:
                    submitter_id_examples.append({
                        "rcv_accession":
                            rcv_accession,

                        "runtime_submitter_ids":
                            runtime_submitter_ids,

                        "parquet_submitter_ids":
                            parquet_record["submitter_ids"],
                    })


        clinvar_set.clear()

        parent = clinvar_set.getparent()

        if parent is not None:
            while clinvar_set.getprevious() is not None:
                del parent[0]

        if records_scanned % 100_000 == 0:
            print(
                f"Progress: {records_scanned:,} XML records scanned; "
                f"{len(observed_rcvs):,}/{len(target_rcvs):,} "
                f"unique target RCVs validated"
            )


# ------------------------------------------------------------------
# Final validation
# ------------------------------------------------------------------

missing_target_rcvs = sorted(
    target_rcvs - observed_rcvs
)

unexpected_observed_rcvs = sorted(
    observed_rcvs - target_rcvs
)


runtime_checks = {
    "Complete XML record count = 2,302,323":
        records_scanned == 2_302_323,

    "All 71,659 target RCVs found":
        len(observed_rcvs) == 71_659,

    "No target RCVs missing":
        len(missing_target_rcvs) == 0,

    "No unexpected target RCVs":
        len(unexpected_observed_rcvs) == 0,

    "No duplicate target RCV records":
        duplicate_target_rcvs == 0,

    "All 100,633 source SCVs processed":
        runtime_scvs_processed == 100_633,

    "No runtime function exceptions":
        runtime_exception_count == 0,

    "No runtime-versus-Parquet SCV count mismatches":
        scv_count_mismatch_count == 0,

    "No runtime-versus-source mapping mismatches":
        mapping_record_mismatch_count == 0,

    "No runtime-versus-Parquet submitter-name mismatches":
        parquet_submitter_name_mismatch_count == 0,

    "No runtime-versus-Parquet submitter-ID mismatches":
        parquet_submitter_id_mismatch_count == 0,
}


print("\nFULL RUNTIME VALIDATION RESULTS")
print("=" * 116)

for check_name, passed in runtime_checks.items():
    print(
        f"{'✅ PASS' if passed else '❌ FAIL'} — "
        f"{check_name}"
    )


failed_runtime_checks = [
    name
    for name, passed in runtime_checks.items()
    if not passed
]


if failed_runtime_checks:

    print("\nMISMATCH EXAMPLES")
    print("-" * 116)

    example_groups = [
        (
            "Runtime exception examples",
            runtime_exception_examples,
        ),
        (
            "SCV-count mismatch examples",
            scv_count_examples,
        ),
        (
            "Per-SCV mapping mismatch examples",
            mapping_mismatch_examples,
        ),
        (
            "Submitter-name mismatch examples",
            submitter_name_examples,
        ),
        (
            "Submitter-ID mismatch examples",
            submitter_id_examples,
        ),
    ]

    for title, examples in example_groups:

        if examples:
            print(f"\n{title}")
            display(pd.DataFrame(examples))

    raise AssertionError(
        "STEP 4F.3 FAILED. "
        + "; ".join(failed_runtime_checks)
    )


# ------------------------------------------------------------------
# Save runtime-validation manifest
# ------------------------------------------------------------------

runtime_manifest = {
    "artifact_type":
        "Historical T0 parser submitter OrgID runtime validation",

    "version":
        "1.2.0",

    "created_at_utc":
        datetime.now(timezone.utc).isoformat(),

    "patched_notebook_path":
        str(NOTEBOOK_V1_2),

    "patched_notebook_sha256":
        notebook_sha256,

    "patched_function":
        "extract_submitter",

    "patched_function_cell_index":
        function_sources[
            "extract_submitter"
        ]["cell_index"],

    "runtime_dependency_functions": {
        function_name: {
            "cell_index":
                function_sources[
                    function_name
                ]["cell_index"],

            "source_sha256":
                hashlib.sha256(
                    function_sources[
                        function_name
                    ]["source"].encode("utf-8")
                ).hexdigest(),
        }
        for function_name in FUNCTIONS_REQUIRED
    },

    "raw_xml_path":
        str(RAW_XML),

    "raw_xml_sha256_expected":
        source_audit_manifest.get(
            "source_xml_sha256_expected"
        ),

    "xml_records_scanned":
        records_scanned,

    "target_rcvs_validated":
        len(observed_rcvs),

    "source_scvs_processed":
        runtime_scvs_processed,

    "runtime_exceptions":
        runtime_exception_count,

    "duplicate_target_rcvs":
        duplicate_target_rcvs,

    "missing_target_rcvs":
        len(missing_target_rcvs),

    "scv_count_mismatches":
        scv_count_mismatch_count,

    "source_mapping_record_mismatches":
        mapping_record_mismatch_count,

    "parquet_submitter_name_mismatches":
        parquet_submitter_name_mismatch_count,

    "parquet_submitter_id_mismatches":
        parquet_submitter_id_mismatch_count,

    "source_mapping_path":
        str(ORGID_MAP_PATH),

    "source_mapping_sha256":
        mapping_sha256,

    "repaired_t0_parquet_path":
        str(T0_V1_2_PATH),

    "repaired_t0_parquet_sha256":
        parquet_sha256,

    "all_runtime_checks_passed":
        True,

    "t1_information_used":
        False,

    "temporal_outcomes_used":
        False,

    "ges_model_output_used":
        False,

    "raw_xml_deletion_authorized":
        False,

    "remaining_validation":
        (
            "Complete aggregate and SCV date parseability, "
            "temporal-boundary checks, condition-field acceptance, "
            "and the consolidated T0 validation manifest."
        ),
}

with open(
    RUNTIME_MANIFEST_PATH,
    "w",
    encoding="utf-8",
) as manifest_file:
    json.dump(
        runtime_manifest,
        manifest_file,
        indent=2,
        ensure_ascii=False,
        sort_keys=True,
    )

runtime_manifest_sha256 = sha256_file(
    RUNTIME_MANIFEST_PATH
)


print("\nRUNTIME-VALIDATION ARTIFACT")
print("=" * 116)

print(f"✅ Manifest: {RUNTIME_MANIFEST_PATH}")
print(f"   SHA-256: {runtime_manifest_sha256}")
print(
    f"   Size: "
    f"{RUNTIME_MANIFEST_PATH.stat().st_size / 1024:,.2f} KB"
)


print("\nFINAL STATUS")
print("=" * 116)

print(
    "✅ The actual patched extract_submitter() function was run "
    "against the complete historical XML."
)

print(
    "✅ All 71,659 RCVs and all 100,633 submitted SCVs matched "
    "the verified source mapping."
)

print(
    "✅ Runtime submitter names and OrgIDs matched the repaired "
    "T0 Parquet v1.2."
)

print(
    "✅ The parser repair is now statically and operationally validated."
)

print(
    "⚠️ Do not delete the raw T0 XML yet; date and condition-field "
    "validation remain."
)

print()
print("✅ STEP 4F.3 COMPLETE")

STEP 4F.3 — FULL PATCHED-PARSER RUNTIME VALIDATION
Patched notebook: /content/drive/MyDrive/Colab Notebooks/02_GES_temporal_validation_v1_2.ipynb
Patched function cell: 13
Target RCVs: 71,659
Beginning full streaming XML validation...
Progress: 100,000 XML records scanned; 53/71,659 unique target RCVs validated
Progress: 200,000 XML records scanned; 71/71,659 unique target RCVs validated
Progress: 300,000 XML records scanned; 131/71,659 unique target RCVs validated
Progress: 400,000 XML records scanned; 201/71,659 unique target RCVs validated
Progress: 500,000 XML records scanned; 390/71,659 unique target RCVs validated
Progress: 600,000 XML records scanned; 648/71,659 unique target RCVs validated
Progress: 700,000 XML records scanned; 670/71,659 unique target RCVs validated
Progress: 800,000 XML records scanned; 876/71,659 unique target RCVs validated
Progress: 900,000 XML records scanned; 1,336/71,659 unique target RCVs validated
Progress: 1,000,000 XML records scanned; 1,587/71,659 

In [ ]:
# STEP 5A: Validate aggregate and nested SCV date fields
# Checks parsing, format, missingness, and the T0 temporal boundary.
# No dataframe or saved artifact is modified.

import json
import re
from collections import Counter, defaultdict
from datetime import date
from pathlib import Path

import pandas as pd


# ------------------------------------------------------------------
# Load the accepted working artifact
# ------------------------------------------------------------------

T0_V1_2_PATH = Path(
    "/content/drive/MyDrive/GES_RAG_Temporal_Study/"
    "data_interim/t0_rcv_target_genes_corrected_v1_2.parquet"
)

if not T0_V1_2_PATH.exists():
    raise FileNotFoundError(
        f"Corrected T0 v1.2 was not found:\n{T0_V1_2_PATH}"
    )

if "df_t0_v1_2" in globals():
    date_df = df_t0_v1_2
else:
    date_df = pd.read_parquet(T0_V1_2_PATH)

EXPECTED_ROWS = 71_659
EXPECTED_SCVS = 100_633
EXPECTED_CUTOFF = date(2022, 12, 31)

if len(date_df) != EXPECTED_ROWS:
    raise AssertionError(
        f"Expected {EXPECTED_ROWS:,} rows, found {len(date_df):,}."
    )


# ------------------------------------------------------------------
# Helpers
# ------------------------------------------------------------------

def parse_json(value):
    """Parse a serialized JSON value or return a parsed object."""
    return json.loads(value) if isinstance(value, str) else value


def normalize_text(value):
    """Normalize a scalar value and identify source missingness."""
    if value is None:
        return None

    try:
        if pd.isna(value):
            return None
    except Exception:
        pass

    text = re.sub(r"\s+", " ", str(value)).strip()

    if text.lower() in {"", "none", "null", "nan", "nat"}:
        return None

    return text


def date_format_category(text):
    """Classify the observed date representation."""
    if re.fullmatch(r"\d{4}-\d{2}-\d{2}", text):
        return "YYYY-MM-DD"

    if re.fullmatch(
        r"\d{4}-\d{2}-\d{2}[T ]\d{2}:\d{2}(:\d{2}(\.\d+)?)?"
        r"(Z|[+-]\d{2}:?\d{2})?",
        text,
    ):
        return "ISO datetime"

    if re.fullmatch(r"\d{4}-\d{2}", text):
        return "YYYY-MM"

    if re.fullmatch(r"\d{4}", text):
        return "YYYY"

    return "Other"


def parse_date_text(text):
    """
    Parse a date into a Python date.
    Returns None when the nonempty value is not parseable.
    """
    try:
        parsed = pd.to_datetime(
            text,
            errors="coerce",
        )
    except Exception:
        return None

    if pd.isna(parsed):
        return None

    try:
        return parsed.date()
    except Exception:
        return None


def update_min_max(stats, parsed_date):
    """Update minimum and maximum parsed dates."""
    if (
        stats["minimum_date"] is None
        or parsed_date < stats["minimum_date"]
    ):
        stats["minimum_date"] = parsed_date

    if (
        stats["maximum_date"] is None
        or parsed_date > stats["maximum_date"]
    ):
        stats["maximum_date"] = parsed_date


# ------------------------------------------------------------------
# Validate the frozen embedded cutoff
# ------------------------------------------------------------------

cutoff_values = [
    normalize_text(value)
    for value in date_df["embedded_data_cutoff_date"]
]

unique_cutoff_values = sorted({
    value
    for value in cutoff_values
    if value is not None
})

parsed_cutoffs = {
    value: parse_date_text(value)
    for value in unique_cutoff_values
}

cutoff_is_valid = (
    len(unique_cutoff_values) == 1
    and parsed_cutoffs.get(
        unique_cutoff_values[0]
    ) == EXPECTED_CUTOFF
)


# ------------------------------------------------------------------
# Aggregate last-evaluated validation
# ------------------------------------------------------------------

aggregate_stats = {
    "total_rows": len(date_df),
    "missing": 0,
    "nonmissing": 0,
    "parsed": 0,
    "malformed": 0,
    "after_cutoff": 0,
    "minimum_date": None,
    "maximum_date": None,
}

aggregate_format_counts = Counter()
aggregate_issue_examples = []

for row in date_df[
    [
        "rcv_accession",
        "aggregate_last_evaluated",
    ]
].itertuples(index=False):

    text = normalize_text(
        row.aggregate_last_evaluated
    )

    if text is None:
        aggregate_stats["missing"] += 1
        continue

    aggregate_stats["nonmissing"] += 1
    aggregate_format_counts[
        date_format_category(text)
    ] += 1

    parsed_date = parse_date_text(text)

    if parsed_date is None:
        aggregate_stats["malformed"] += 1

        if len(aggregate_issue_examples) < 20:
            aggregate_issue_examples.append({
                "rcv_accession": row.rcv_accession,
                "field": "aggregate_last_evaluated",
                "issue": "nonempty but unparseable",
                "value": text,
            })

        continue

    aggregate_stats["parsed"] += 1
    update_min_max(
        aggregate_stats,
        parsed_date,
    )

    if parsed_date > EXPECTED_CUTOFF:
        aggregate_stats["after_cutoff"] += 1

        if len(aggregate_issue_examples) < 20:
            aggregate_issue_examples.append({
                "rcv_accession": row.rcv_accession,
                "field": "aggregate_last_evaluated",
                "issue": "date after T0 cutoff",
                "value": text,
                "parsed_date": parsed_date.isoformat(),
            })


# ------------------------------------------------------------------
# Nested SCV date validation
# ------------------------------------------------------------------

def new_date_key_stats():
    return {
        "records_with_key": 0,
        "missing": 0,
        "nonmissing": 0,
        "parsed": 0,
        "malformed": 0,
        "after_cutoff": 0,
        "nonscalar": 0,
        "minimum_date": None,
        "maximum_date": None,
        "format_counts": Counter(),
    }


scv_date_stats = defaultdict(
    new_date_key_stats
)

total_scv_records = 0
non_dictionary_scv_records = 0
scv_records_without_date_key = 0

scv_date_issue_examples = []
rcvs_with_scv_date_issues = set()

for row in date_df[
    [
        "rcv_accession",
        "scv_records_json",
    ]
].itertuples(index=False):

    scv_records = parse_json(
        row.scv_records_json
    )

    for scv_position, scv_record in enumerate(
        scv_records
    ):
        total_scv_records += 1

        if not isinstance(scv_record, dict):
            non_dictionary_scv_records += 1
            rcvs_with_scv_date_issues.add(
                row.rcv_accession
            )

            if len(scv_date_issue_examples) < 20:
                scv_date_issue_examples.append({
                    "rcv_accession": row.rcv_accession,
                    "scv_position": scv_position,
                    "issue": "SCV record is not a dictionary",
                    "value_type": type(scv_record).__name__,
                })

            continue

        scv_accession = (
            scv_record.get("scv_accession")
            or scv_record.get("accession")
        )

        date_keys = [
            key
            for key in scv_record
            if (
                "date" in str(key).lower()
                or "evaluat" in str(key).lower()
            )
        ]

        if not date_keys:
            scv_records_without_date_key += 1
            rcvs_with_scv_date_issues.add(
                row.rcv_accession
            )

            if len(scv_date_issue_examples) < 20:
                scv_date_issue_examples.append({
                    "rcv_accession": row.rcv_accession,
                    "scv_accession": scv_accession,
                    "scv_position": scv_position,
                    "issue": "no date-like key in SCV record",
                    "available_keys": sorted(
                        scv_record.keys()
                    ),
                })

            continue

        for key in date_keys:
            value = scv_record.get(key)
            stats = scv_date_stats[str(key)]

            stats["records_with_key"] += 1

            if isinstance(value, (list, dict)):
                stats["nonscalar"] += 1
                rcvs_with_scv_date_issues.add(
                    row.rcv_accession
                )

                if len(scv_date_issue_examples) < 20:
                    scv_date_issue_examples.append({
                        "rcv_accession": row.rcv_accession,
                        "scv_accession": scv_accession,
                        "field": key,
                        "issue": "date value is not scalar",
                        "value_type": type(value).__name__,
                    })

                continue

            text = normalize_text(value)

            if text is None:
                stats["missing"] += 1
                continue

            stats["nonmissing"] += 1
            stats["format_counts"][
                date_format_category(text)
            ] += 1

            parsed_date = parse_date_text(text)

            if parsed_date is None:
                stats["malformed"] += 1
                rcvs_with_scv_date_issues.add(
                    row.rcv_accession
                )

                if len(scv_date_issue_examples) < 20:
                    scv_date_issue_examples.append({
                        "rcv_accession": row.rcv_accession,
                        "scv_accession": scv_accession,
                        "field": key,
                        "issue": "nonempty but unparseable",
                        "value": text,
                    })

                continue

            stats["parsed"] += 1
            update_min_max(
                stats,
                parsed_date,
            )

            if parsed_date > EXPECTED_CUTOFF:
                stats["after_cutoff"] += 1
                rcvs_with_scv_date_issues.add(
                    row.rcv_accession
                )

                if len(scv_date_issue_examples) < 20:
                    scv_date_issue_examples.append({
                        "rcv_accession": row.rcv_accession,
                        "scv_accession": scv_accession,
                        "field": key,
                        "issue": "date after T0 cutoff",
                        "value": text,
                        "parsed_date": parsed_date.isoformat(),
                    })


# ------------------------------------------------------------------
# Build readable summaries
# ------------------------------------------------------------------

aggregate_summary_df = pd.DataFrame([{
    **{
        key: value
        for key, value in aggregate_stats.items()
        if key not in {
            "minimum_date",
            "maximum_date",
        }
    },
    "minimum_date": (
        aggregate_stats["minimum_date"].isoformat()
        if aggregate_stats["minimum_date"]
        else None
    ),
    "maximum_date": (
        aggregate_stats["maximum_date"].isoformat()
        if aggregate_stats["maximum_date"]
        else None
    ),
}])

aggregate_formats_df = pd.DataFrame([
    {
        "format": format_name,
        "count": count,
    }
    for format_name, count
    in aggregate_format_counts.most_common()
])

scv_summary_rows = []

for key, stats in sorted(
    scv_date_stats.items()
):
    scv_summary_rows.append({
        "date_key": key,
        "records_with_key":
            stats["records_with_key"],
        "missing":
            stats["missing"],
        "nonmissing":
            stats["nonmissing"],
        "parsed":
            stats["parsed"],
        "malformed":
            stats["malformed"],
        "after_cutoff":
            stats["after_cutoff"],
        "nonscalar":
            stats["nonscalar"],
        "minimum_date": (
            stats["minimum_date"].isoformat()
            if stats["minimum_date"]
            else None
        ),
        "maximum_date": (
            stats["maximum_date"].isoformat()
            if stats["maximum_date"]
            else None
        ),
        "format_distribution":
            dict(stats["format_counts"]),
    })

scv_date_summary_df = pd.DataFrame(
    scv_summary_rows
)

total_scv_malformed = sum(
    stats["malformed"]
    for stats in scv_date_stats.values()
)

total_scv_after_cutoff = sum(
    stats["after_cutoff"]
    for stats in scv_date_stats.values()
)

total_scv_nonscalar = sum(
    stats["nonscalar"]
    for stats in scv_date_stats.values()
)


# ------------------------------------------------------------------
# Critical acceptance checks
# ------------------------------------------------------------------

critical_checks = {
    "T0 embedded cutoff is exactly 2022-12-31":
        cutoff_is_valid,

    "Corrected artifact contains 71,659 RCV rows":
        len(date_df) == EXPECTED_ROWS,

    "Nested records contain 100,633 SCVs":
        total_scv_records == EXPECTED_SCVS,

    "All nested SCV records are dictionaries":
        non_dictionary_scv_records == 0,

    "At least one SCV date-like field was detected":
        len(scv_date_stats) > 0,

    "Every SCV record contains a date-like key":
        scv_records_without_date_key == 0,

    "No malformed aggregate dates":
        aggregate_stats["malformed"] == 0,

    "No aggregate dates occur after the T0 cutoff":
        aggregate_stats["after_cutoff"] == 0,

    "No malformed nested SCV dates":
        total_scv_malformed == 0,

    "No nested SCV dates occur after the T0 cutoff":
        total_scv_after_cutoff == 0,

    "No nested SCV date value is a list or dictionary":
        total_scv_nonscalar == 0,
}


# ------------------------------------------------------------------
# Output
# ------------------------------------------------------------------

print("=" * 116)
print("STEP 5A — T0 DATE PARSEABILITY AND TEMPORAL-BOUNDARY VALIDATION")
print("=" * 116)

print("\nT0 CUTOFF VALIDATION")
print("-" * 116)

print(
    "Unique embedded cutoff values:",
    unique_cutoff_values,
)

print(
    "Parsed cutoff values:",
    {
        key: (
            value.isoformat()
            if value is not None
            else None
        )
        for key, value in parsed_cutoffs.items()
    },
)

print("\nAGGREGATE DATE SUMMARY")
print("-" * 116)
display(aggregate_summary_df)

print("\nAGGREGATE DATE FORMAT DISTRIBUTION")
print("-" * 116)
display(aggregate_formats_df)

print("\nSCV DATE-KEY SUMMARY")
print("-" * 116)

if scv_date_summary_df.empty:
    print("No SCV date-like fields were detected.")
else:
    display(scv_date_summary_df)

print("\nSCV STRUCTURE COUNTS")
print("-" * 116)

print(f"Total nested SCV records: {total_scv_records:,}")
print(
    "Non-dictionary SCV records:",
    f"{non_dictionary_scv_records:,}",
)
print(
    "SCV records without any date-like key:",
    f"{scv_records_without_date_key:,}",
)
print(
    "RCVs with at least one SCV date issue:",
    f"{len(rcvs_with_scv_date_issues):,}",
)

print("\nCRITICAL VALIDATION RESULTS")
print("=" * 116)

for check_name, passed in critical_checks.items():
    print(
        f"{'✅ PASS' if passed else '❌ FAIL'} — "
        f"{check_name}"
    )

print("\nDATE ISSUE EXAMPLES")
print("-" * 116)

all_issue_examples = (
    aggregate_issue_examples
    + scv_date_issue_examples
)

if all_issue_examples:
    display(
        pd.DataFrame(
            all_issue_examples
        )
    )
else:
    print(
        "✅ No malformed, nonscalar, missing-key, or "
        "post-cutoff date issues were found."
    )

failed_checks = [
    check_name
    for check_name, passed
    in critical_checks.items()
    if not passed
]

if failed_checks:
    raise AssertionError(
        "STEP 5A REQUIRES INVESTIGATION. "
        + "; ".join(failed_checks)
    )

print()
print("✅ STEP 5A COMPLETE")
print(
    "Aggregate and SCV date fields passed parseability and "
    "T0 temporal-boundary validation."
)
print(
    "Missing date values, when present, were counted but were not "
    "silently imputed."
)
print("No dataframe or saved artifact was modified.")

In [47]:
# STEP 5A: Validate aggregate and nested SCV date fields
# Checks parsing, format, missingness, and the T0 temporal boundary.
# No dataframe or saved artifact is modified.

import json
import re
from collections import Counter, defaultdict
from datetime import date
from pathlib import Path

import pandas as pd


# ------------------------------------------------------------------
# Load the accepted working artifact
# ------------------------------------------------------------------

T0_V1_2_PATH = Path(
    "/content/drive/MyDrive/GES_RAG_Temporal_Study/"
    "data_interim/t0_rcv_target_genes_corrected_v1_2.parquet"
)

if not T0_V1_2_PATH.exists():
    raise FileNotFoundError(
        f"Corrected T0 v1.2 was not found:\n{T0_V1_2_PATH}"
    )

if "df_t0_v1_2" in globals():
    date_df = df_t0_v1_2
else:
    date_df = pd.read_parquet(T0_V1_2_PATH)

EXPECTED_ROWS = 71_659
EXPECTED_SCVS = 100_633
EXPECTED_CUTOFF = date(2022, 12, 31)

if len(date_df) != EXPECTED_ROWS:
    raise AssertionError(
        f"Expected {EXPECTED_ROWS:,} rows, found {len(date_df):,}."
    )


# ------------------------------------------------------------------
# Helpers
# ------------------------------------------------------------------

def parse_json(value):
    """Parse a serialized JSON value or return a parsed object."""
    return json.loads(value) if isinstance(value, str) else value


def normalize_text(value):
    """Normalize a scalar value and identify source missingness."""
    if value is None:
        return None

    try:
        if pd.isna(value):
            return None
    except Exception:
        pass

    text = re.sub(r"\s+", " ", str(value)).strip()

    if text.lower() in {"", "none", "null", "nan", "nat"}:
        return None

    return text


def date_format_category(text):
    """Classify the observed date representation."""
    if re.fullmatch(r"\d{4}-\d{2}-\d{2}", text):
        return "YYYY-MM-DD"

    if re.fullmatch(
        r"\d{4}-\d{2}-\d{2}[T ]\d{2}:\d{2}(:\d{2}(\.\d+)?)?"
        r"(Z|[+-]\d{2}:?\d{2})?",
        text,
    ):
        return "ISO datetime"

    if re.fullmatch(r"\d{4}-\d{2}", text):
        return "YYYY-MM"

    if re.fullmatch(r"\d{4}", text):
        return "YYYY"

    return "Other"


def parse_date_text(text):
    """
    Parse a date into a Python date.
    Returns None when the nonempty value is not parseable.
    """
    try:
        parsed = pd.to_datetime(
            text,
            errors="coerce",
        )
    except Exception:
        return None

    if pd.isna(parsed):
        return None

    try:
        return parsed.date()
    except Exception:
        return None


def update_min_max(stats, parsed_date):
    """Update minimum and maximum parsed dates."""
    if (
        stats["minimum_date"] is None
        or parsed_date < stats["minimum_date"]
    ):
        stats["minimum_date"] = parsed_date

    if (
        stats["maximum_date"] is None
        or parsed_date > stats["maximum_date"]
    ):
        stats["maximum_date"] = parsed_date


# ------------------------------------------------------------------
# Validate the frozen embedded cutoff
# ------------------------------------------------------------------

cutoff_values = [
    normalize_text(value)
    for value in date_df["embedded_data_cutoff_date"]
]

unique_cutoff_values = sorted({
    value
    for value in cutoff_values
    if value is not None
})

parsed_cutoffs = {
    value: parse_date_text(value)
    for value in unique_cutoff_values
}

cutoff_is_valid = (
    len(unique_cutoff_values) == 1
    and parsed_cutoffs.get(
        unique_cutoff_values[0]
    ) == EXPECTED_CUTOFF
)


# ------------------------------------------------------------------
# Aggregate last-evaluated validation
# ------------------------------------------------------------------

aggregate_stats = {
    "total_rows": len(date_df),
    "missing": 0,
    "nonmissing": 0,
    "parsed": 0,
    "malformed": 0,
    "after_cutoff": 0,
    "minimum_date": None,
    "maximum_date": None,
}

aggregate_format_counts = Counter()
aggregate_issue_examples = []

for row in date_df[
    [
        "rcv_accession",
        "aggregate_last_evaluated",
    ]
].itertuples(index=False):

    text = normalize_text(
        row.aggregate_last_evaluated
    )

    if text is None:
        aggregate_stats["missing"] += 1
        continue

    aggregate_stats["nonmissing"] += 1
    aggregate_format_counts[
        date_format_category(text)
    ] += 1

    parsed_date = parse_date_text(text)

    if parsed_date is None:
        aggregate_stats["malformed"] += 1

        if len(aggregate_issue_examples) < 20:
            aggregate_issue_examples.append({
                "rcv_accession": row.rcv_accession,
                "field": "aggregate_last_evaluated",
                "issue": "nonempty but unparseable",
                "value": text,
            })

        continue

    aggregate_stats["parsed"] += 1
    update_min_max(
        aggregate_stats,
        parsed_date,
    )

    if parsed_date > EXPECTED_CUTOFF:
        aggregate_stats["after_cutoff"] += 1

        if len(aggregate_issue_examples) < 20:
            aggregate_issue_examples.append({
                "rcv_accession": row.rcv_accession,
                "field": "aggregate_last_evaluated",
                "issue": "date after T0 cutoff",
                "value": text,
                "parsed_date": parsed_date.isoformat(),
            })


# ------------------------------------------------------------------
# Nested SCV date validation
# ------------------------------------------------------------------

def new_date_key_stats():
    return {
        "records_with_key": 0,
        "missing": 0,
        "nonmissing": 0,
        "parsed": 0,
        "malformed": 0,
        "after_cutoff": 0,
        "nonscalar": 0,
        "minimum_date": None,
        "maximum_date": None,
        "format_counts": Counter(),
    }


scv_date_stats = defaultdict(
    new_date_key_stats
)

total_scv_records = 0
non_dictionary_scv_records = 0
scv_records_without_date_key = 0

scv_date_issue_examples = []
rcvs_with_scv_date_issues = set()

for row in date_df[
    [
        "rcv_accession",
        "scv_records_json",
    ]
].itertuples(index=False):

    scv_records = parse_json(
        row.scv_records_json
    )

    for scv_position, scv_record in enumerate(
        scv_records
    ):
        total_scv_records += 1

        if not isinstance(scv_record, dict):
            non_dictionary_scv_records += 1
            rcvs_with_scv_date_issues.add(
                row.rcv_accession
            )

            if len(scv_date_issue_examples) < 20:
                scv_date_issue_examples.append({
                    "rcv_accession": row.rcv_accession,
                    "scv_position": scv_position,
                    "issue": "SCV record is not a dictionary",
                    "value_type": type(scv_record).__name__,
                })

            continue

        scv_accession = (
            scv_record.get("scv_accession")
            or scv_record.get("accession")
        )

        date_keys = [
            key
            for key in scv_record
            if (
                "date" in str(key).lower()
                or "evaluat" in str(key).lower()
            )
        ]

        if not date_keys:
            scv_records_without_date_key += 1
            rcvs_with_scv_date_issues.add(
                row.rcv_accession
            )

            if len(scv_date_issue_examples) < 20:
                scv_date_issue_examples.append({
                    "rcv_accession": row.rcv_accession,
                    "scv_accession": scv_accession,
                    "scv_position": scv_position,
                    "issue": "no date-like key in SCV record",
                    "available_keys": sorted(
                        scv_record.keys()
                    ),
                })

            continue

        for key in date_keys:
            value = scv_record.get(key)
            stats = scv_date_stats[str(key)]

            stats["records_with_key"] += 1

            if isinstance(value, (list, dict)):
                stats["nonscalar"] += 1
                rcvs_with_scv_date_issues.add(
                    row.rcv_accession
                )

                if len(scv_date_issue_examples) < 20:
                    scv_date_issue_examples.append({
                        "rcv_accession": row.rcv_accession,
                        "scv_accession": scv_accession,
                        "field": key,
                        "issue": "date value is not scalar",
                        "value_type": type(value).__name__,
                    })

                continue

            text = normalize_text(value)

            if text is None:
                stats["missing"] += 1
                continue

            stats["nonmissing"] += 1
            stats["format_counts"][
                date_format_category(text)
            ] += 1

            parsed_date = parse_date_text(text)

            if parsed_date is None:
                stats["malformed"] += 1
                rcvs_with_scv_date_issues.add(
                    row.rcv_accession
                )

                if len(scv_date_issue_examples) < 20:
                    scv_date_issue_examples.append({
                        "rcv_accession": row.rcv_accession,
                        "scv_accession": scv_accession,
                        "field": key,
                        "issue": "nonempty but unparseable",
                        "value": text,
                    })

                continue

            stats["parsed"] += 1
            update_min_max(
                stats,
                parsed_date,
            )

            if parsed_date > EXPECTED_CUTOFF:
                stats["after_cutoff"] += 1
                rcvs_with_scv_date_issues.add(
                    row.rcv_accession
                )

                if len(scv_date_issue_examples) < 20:
                    scv_date_issue_examples.append({
                        "rcv_accession": row.rcv_accession,
                        "scv_accession": scv_accession,
                        "field": key,
                        "issue": "date after T0 cutoff",
                        "value": text,
                        "parsed_date": parsed_date.isoformat(),
                    })


# ------------------------------------------------------------------
# Build readable summaries
# ------------------------------------------------------------------

aggregate_summary_df = pd.DataFrame([{
    **{
        key: value
        for key, value in aggregate_stats.items()
        if key not in {
            "minimum_date",
            "maximum_date",
        }
    },
    "minimum_date": (
        aggregate_stats["minimum_date"].isoformat()
        if aggregate_stats["minimum_date"]
        else None
    ),
    "maximum_date": (
        aggregate_stats["maximum_date"].isoformat()
        if aggregate_stats["maximum_date"]
        else None
    ),
}])

aggregate_formats_df = pd.DataFrame([
    {
        "format": format_name,
        "count": count,
    }
    for format_name, count
    in aggregate_format_counts.most_common()
])

scv_summary_rows = []

for key, stats in sorted(
    scv_date_stats.items()
):
    scv_summary_rows.append({
        "date_key": key,
        "records_with_key":
            stats["records_with_key"],
        "missing":
            stats["missing"],
        "nonmissing":
            stats["nonmissing"],
        "parsed":
            stats["parsed"],
        "malformed":
            stats["malformed"],
        "after_cutoff":
            stats["after_cutoff"],
        "nonscalar":
            stats["nonscalar"],
        "minimum_date": (
            stats["minimum_date"].isoformat()
            if stats["minimum_date"]
            else None
        ),
        "maximum_date": (
            stats["maximum_date"].isoformat()
            if stats["maximum_date"]
            else None
        ),
        "format_distribution":
            dict(stats["format_counts"]),
    })

scv_date_summary_df = pd.DataFrame(
    scv_summary_rows
)

total_scv_malformed = sum(
    stats["malformed"]
    for stats in scv_date_stats.values()
)

total_scv_after_cutoff = sum(
    stats["after_cutoff"]
    for stats in scv_date_stats.values()
)

total_scv_nonscalar = sum(
    stats["nonscalar"]
    for stats in scv_date_stats.values()
)


# ------------------------------------------------------------------
# Critical acceptance checks
# ------------------------------------------------------------------

critical_checks = {
    "T0 embedded cutoff is exactly 2022-12-31":
        cutoff_is_valid,

    "Corrected artifact contains 71,659 RCV rows":
        len(date_df) == EXPECTED_ROWS,

    "Nested records contain 100,633 SCVs":
        total_scv_records == EXPECTED_SCVS,

    "All nested SCV records are dictionaries":
        non_dictionary_scv_records == 0,

    "At least one SCV date-like field was detected":
        len(scv_date_stats) > 0,

    "Every SCV record contains a date-like key":
        scv_records_without_date_key == 0,

    "No malformed aggregate dates":
        aggregate_stats["malformed"] == 0,

    "No aggregate dates occur after the T0 cutoff":
        aggregate_stats["after_cutoff"] == 0,

    "No malformed nested SCV dates":
        total_scv_malformed == 0,

    "No nested SCV dates occur after the T0 cutoff":
        total_scv_after_cutoff == 0,

    "No nested SCV date value is a list or dictionary":
        total_scv_nonscalar == 0,
}


# ------------------------------------------------------------------
# Output
# ------------------------------------------------------------------

print("=" * 116)
print("STEP 5A — T0 DATE PARSEABILITY AND TEMPORAL-BOUNDARY VALIDATION")
print("=" * 116)

print("\nT0 CUTOFF VALIDATION")
print("-" * 116)

print(
    "Unique embedded cutoff values:",
    unique_cutoff_values,
)

print(
    "Parsed cutoff values:",
    {
        key: (
            value.isoformat()
            if value is not None
            else None
        )
        for key, value in parsed_cutoffs.items()
    },
)

print("\nAGGREGATE DATE SUMMARY")
print("-" * 116)
display(aggregate_summary_df)

print("\nAGGREGATE DATE FORMAT DISTRIBUTION")
print("-" * 116)
display(aggregate_formats_df)

print("\nSCV DATE-KEY SUMMARY")
print("-" * 116)

if scv_date_summary_df.empty:
    print("No SCV date-like fields were detected.")
else:
    display(scv_date_summary_df)

print("\nSCV STRUCTURE COUNTS")
print("-" * 116)

print(f"Total nested SCV records: {total_scv_records:,}")
print(
    "Non-dictionary SCV records:",
    f"{non_dictionary_scv_records:,}",
)
print(
    "SCV records without any date-like key:",
    f"{scv_records_without_date_key:,}",
)
print(
    "RCVs with at least one SCV date issue:",
    f"{len(rcvs_with_scv_date_issues):,}",
)

print("\nCRITICAL VALIDATION RESULTS")
print("=" * 116)

for check_name, passed in critical_checks.items():
    print(
        f"{'✅ PASS' if passed else '❌ FAIL'} — "
        f"{check_name}"
    )

print("\nDATE ISSUE EXAMPLES")
print("-" * 116)

all_issue_examples = (
    aggregate_issue_examples
    + scv_date_issue_examples
)

if all_issue_examples:
    display(
        pd.DataFrame(
            all_issue_examples
        )
    )
else:
    print(
        "✅ No malformed, nonscalar, missing-key, or "
        "post-cutoff date issues were found."
    )

failed_checks = [
    check_name
    for check_name, passed
    in critical_checks.items()
    if not passed
]

if failed_checks:
    raise AssertionError(
        "STEP 5A REQUIRES INVESTIGATION. "
        + "; ".join(failed_checks)
    )

print()
print("✅ STEP 5A COMPLETE")
print(
    "Aggregate and SCV date fields passed parseability and "
    "T0 temporal-boundary validation."
)
print(
    "Missing date values, when present, were counted but were not "
    "silently imputed."
)
print("No dataframe or saved artifact was modified.")

STEP 5A — T0 DATE PARSEABILITY AND TEMPORAL-BOUNDARY VALIDATION

T0 CUTOFF VALIDATION
--------------------------------------------------------------------------------------------------------------------
Unique embedded cutoff values: ['2022-12-31']
Parsed cutoff values: {'2022-12-31': '2022-12-31'}

AGGREGATE DATE SUMMARY
--------------------------------------------------------------------------------------------------------------------


,total_rows,missing,nonmissing,parsed,malformed,after_cutoff,minimum_date,maximum_date
0,71659,5889,65770,65770,0,0,1994-03-17,2022-12-23



AGGREGATE DATE FORMAT DISTRIBUTION
--------------------------------------------------------------------------------------------------------------------


,format,count
0,YYYY-MM-DD,65770



SCV DATE-KEY SUMMARY
--------------------------------------------------------------------------------------------------------------------


,date_key,records_with_key,missing,nonmissing,parsed,malformed,after_cutoff,nonscalar,minimum_date,maximum_date,format_distribution
0,last_evaluated,100633,9347,91286,91286,0,0,0,1994-03-17,2022-12-23,{'YYYY-MM-DD': 91286}



SCV STRUCTURE COUNTS
--------------------------------------------------------------------------------------------------------------------
Total nested SCV records: 100,633
Non-dictionary SCV records: 0
SCV records without any date-like key: 0
RCVs with at least one SCV date issue: 0

CRITICAL VALIDATION RESULTS
✅ PASS — T0 embedded cutoff is exactly 2022-12-31
✅ PASS — Corrected artifact contains 71,659 RCV rows
✅ PASS — Nested records contain 100,633 SCVs
✅ PASS — All nested SCV records are dictionaries
✅ PASS — At least one SCV date-like field was detected
✅ PASS — Every SCV record contains a date-like key
✅ PASS — No malformed aggregate dates
✅ PASS — No aggregate dates occur after the T0 cutoff
✅ PASS — No malformed nested SCV dates
✅ PASS — No nested SCV dates occur after the T0 cutoff
✅ PASS — No nested SCV date value is a list or dictionary

DATE ISSUE EXAMPLES
--------------------------------------------------------------------------------------------------------------------
✅

In [2]:
# STEP 5B: Validate condition names, identifiers, and nested trait records
# Confirms that the flattened condition fields match trait_records_json.
# No dataframe or saved artifact is modified.

import json
import re
from collections import Counter
from pathlib import Path

import pandas as pd


# ------------------------------------------------------------------
# Load corrected T0 v1.2
# ------------------------------------------------------------------

T0_V1_2_PATH = Path(
    "/content/drive/MyDrive/GES_RAG_Temporal_Study/"
    "data_interim/t0_rcv_target_genes_corrected_v1_2.parquet"
)

if not T0_V1_2_PATH.exists():
    raise FileNotFoundError(
        f"Corrected T0 v1.2 was not found:\n{T0_V1_2_PATH}"
    )

if "df_t0_v1_2" in globals():
    condition_df = df_t0_v1_2
else:
    condition_df = pd.read_parquet(T0_V1_2_PATH)

EXPECTED_ROWS = 71_659
EXPECTED_EMPTY_ID_ROWS = 144

if len(condition_df) != EXPECTED_ROWS:
    raise AssertionError(
        f"Expected {EXPECTED_ROWS:,} rows, "
        f"found {len(condition_df):,}."
    )


# ------------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------------

def parse_json_list(value, field_name):
    """Parse a serialized JSON list and validate its top-level type."""
    parsed = json.loads(value) if isinstance(value, str) else value

    if not isinstance(parsed, list):
        raise TypeError(
            f"{field_name} must be a list, "
            f"found {type(parsed).__name__}"
        )

    return parsed


def normalize_text(value):
    """Normalize whitespace and convert blank-like values to None."""
    if value is None:
        return None

    try:
        if pd.isna(value):
            return None
    except Exception:
        pass

    text = re.sub(
        r"\s+",
        " ",
        str(value),
    ).strip()

    if text.lower() in {
        "",
        "none",
        "null",
        "nan",
        "nat",
    }:
        return None

    return text


def normalized_unique(values):
    """Return sorted unique nonblank normalized strings."""
    return sorted({
        normalized
        for value in values
        if (
            normalized := normalize_text(value)
        ) is not None
    })


# ------------------------------------------------------------------
# Validation counters
# ------------------------------------------------------------------

rows_processed = 0

json_parse_errors = 0
non_dictionary_trait_records = 0
trait_records_missing_names_key = 0
trait_records_missing_identifiers_key = 0
trait_names_not_lists = 0
trait_identifiers_not_lists = 0

rows_without_condition_names = 0
rows_without_trait_records = 0

condition_name_mismatch_count = 0
condition_id_mismatch_count = 0

empty_condition_id_rows = 0
empty_nested_identifier_rows = 0
empty_id_source_inconsistency_count = 0

blank_condition_name_entries = 0
blank_condition_id_entries = 0
blank_trait_name_entries = 0
blank_trait_id_entries = 0

rows_with_duplicate_condition_names = 0
rows_with_duplicate_condition_ids = 0

empty_id_rows_named_see_cases = 0

condition_name_frequency = Counter()
condition_id_prefix_frequency = Counter()
trait_count_frequency = Counter()

issue_examples = []


# ------------------------------------------------------------------
# Validate every RCV record
# ------------------------------------------------------------------

columns_needed = [
    "rcv_accession",
    "condition_names_json",
    "condition_ids_json",
    "trait_records_json",
]

for row in condition_df[
    columns_needed
].itertuples(index=False):

    rows_processed += 1
    rcv = row.rcv_accession

    try:
        condition_names_raw = parse_json_list(
            row.condition_names_json,
            "condition_names_json",
        )

        condition_ids_raw = parse_json_list(
            row.condition_ids_json,
            "condition_ids_json",
        )

        trait_records = parse_json_list(
            row.trait_records_json,
            "trait_records_json",
        )

    except Exception as exc:
        json_parse_errors += 1

        if len(issue_examples) < 25:
            issue_examples.append({
                "rcv_accession": rcv,
                "issue": "JSON parsing or top-level type error",
                "details": f"{type(exc).__name__}: {exc}",
            })

        continue

    normalized_condition_names = normalized_unique(
        condition_names_raw
    )

    normalized_condition_ids = normalized_unique(
        condition_ids_raw
    )

    blank_condition_name_entries += sum(
        normalize_text(value) is None
        for value in condition_names_raw
    )

    blank_condition_id_entries += sum(
        normalize_text(value) is None
        for value in condition_ids_raw
    )

    if len(condition_names_raw) != len(
        normalized_condition_names
    ):
        raw_nonblank_name_count = sum(
            normalize_text(value) is not None
            for value in condition_names_raw
        )

        if raw_nonblank_name_count > len(
            normalized_condition_names
        ):
            rows_with_duplicate_condition_names += 1

    if len(condition_ids_raw) != len(
        normalized_condition_ids
    ):
        raw_nonblank_id_count = sum(
            normalize_text(value) is not None
            for value in condition_ids_raw
        )

        if raw_nonblank_id_count > len(
            normalized_condition_ids
        ):
            rows_with_duplicate_condition_ids += 1

    if not normalized_condition_names:
        rows_without_condition_names += 1

    if not trait_records:
        rows_without_trait_records += 1

    trait_count_frequency[len(trait_records)] += 1

    nested_names_raw = []
    nested_ids_raw = []

    structural_error = False

    for trait_position, trait_record in enumerate(
        trait_records
    ):

        if not isinstance(trait_record, dict):
            non_dictionary_trait_records += 1
            structural_error = True

            if len(issue_examples) < 25:
                issue_examples.append({
                    "rcv_accession": rcv,
                    "trait_position": trait_position,
                    "issue": "Trait record is not a dictionary",
                    "value_type": type(trait_record).__name__,
                })

            continue

        if "names" not in trait_record:
            trait_records_missing_names_key += 1
            structural_error = True

            if len(issue_examples) < 25:
                issue_examples.append({
                    "rcv_accession": rcv,
                    "trait_position": trait_position,
                    "issue": "Trait record missing names key",
                    "available_keys": sorted(
                        trait_record.keys()
                    ),
                })

            trait_names = []

        else:
            trait_names = trait_record["names"]

        if "identifiers" not in trait_record:
            trait_records_missing_identifiers_key += 1
            structural_error = True

            if len(issue_examples) < 25:
                issue_examples.append({
                    "rcv_accession": rcv,
                    "trait_position": trait_position,
                    "issue": "Trait record missing identifiers key",
                    "available_keys": sorted(
                        trait_record.keys()
                    ),
                })

            trait_ids = []

        else:
            trait_ids = trait_record["identifiers"]

        if not isinstance(trait_names, list):
            trait_names_not_lists += 1
            structural_error = True

            if len(issue_examples) < 25:
                issue_examples.append({
                    "rcv_accession": rcv,
                    "trait_position": trait_position,
                    "issue": "Trait names value is not a list",
                    "value_type": type(trait_names).__name__,
                })

            trait_names = []

        if not isinstance(trait_ids, list):
            trait_identifiers_not_lists += 1
            structural_error = True

            if len(issue_examples) < 25:
                issue_examples.append({
                    "rcv_accession": rcv,
                    "trait_position": trait_position,
                    "issue": "Trait identifiers value is not a list",
                    "value_type": type(trait_ids).__name__,
                })

            trait_ids = []

        blank_trait_name_entries += sum(
            normalize_text(value) is None
            for value in trait_names
        )

        blank_trait_id_entries += sum(
            normalize_text(value) is None
            for value in trait_ids
        )

        nested_names_raw.extend(trait_names)
        nested_ids_raw.extend(trait_ids)

    normalized_nested_names = normalized_unique(
        nested_names_raw
    )

    normalized_nested_ids = normalized_unique(
        nested_ids_raw
    )

    names_match = (
        normalized_condition_names
        == normalized_nested_names
    )

    ids_match = (
        normalized_condition_ids
        == normalized_nested_ids
    )

    if not names_match:
        condition_name_mismatch_count += 1

        if len(issue_examples) < 25:
            issue_examples.append({
                "rcv_accession": rcv,
                "issue": (
                    "condition_names_json does not match "
                    "trait_records_json names"
                ),
                "condition_names":
                    normalized_condition_names,
                "nested_trait_names":
                    normalized_nested_names,
            })

    if not ids_match:
        condition_id_mismatch_count += 1

        if len(issue_examples) < 25:
            issue_examples.append({
                "rcv_accession": rcv,
                "issue": (
                    "condition_ids_json does not match "
                    "trait_records_json identifiers"
                ),
                "condition_ids":
                    normalized_condition_ids,
                "nested_trait_ids":
                    normalized_nested_ids,
            })

    top_level_ids_empty = (
        len(normalized_condition_ids) == 0
    )

    nested_ids_empty = (
        len(normalized_nested_ids) == 0
    )

    if top_level_ids_empty:
        empty_condition_id_rows += 1

    if nested_ids_empty:
        empty_nested_identifier_rows += 1

    if top_level_ids_empty != nested_ids_empty:
        empty_id_source_inconsistency_count += 1

        if len(issue_examples) < 25:
            issue_examples.append({
                "rcv_accession": rcv,
                "issue": (
                    "Top-level and nested condition-ID "
                    "missingness disagree"
                ),
                "condition_ids_empty":
                    top_level_ids_empty,
                "nested_ids_empty":
                    nested_ids_empty,
            })

    if (
        top_level_ids_empty
        and normalized_condition_names == ["See cases"]
    ):
        empty_id_rows_named_see_cases += 1

    for condition_name in normalized_condition_names:
        condition_name_frequency[
            condition_name
        ] += 1

    for condition_id in normalized_condition_ids:
        prefix = (
            condition_id.split(":", 1)[0]
            if ":" in condition_id
            else "NO_PREFIX"
        )

        condition_id_prefix_frequency[
            prefix
        ] += 1


# ------------------------------------------------------------------
# Summary tables
# ------------------------------------------------------------------

validation_summary = pd.DataFrame([{
    "rows_processed":
        rows_processed,

    "json_or_top_level_type_errors":
        json_parse_errors,

    "non_dictionary_trait_records":
        non_dictionary_trait_records,

    "trait_records_missing_names_key":
        trait_records_missing_names_key,

    "trait_records_missing_identifiers_key":
        trait_records_missing_identifiers_key,

    "trait_names_not_lists":
        trait_names_not_lists,

    "trait_identifiers_not_lists":
        trait_identifiers_not_lists,

    "rows_without_condition_names":
        rows_without_condition_names,

    "rows_without_trait_records":
        rows_without_trait_records,

    "condition_name_mismatches":
        condition_name_mismatch_count,

    "condition_id_mismatches":
        condition_id_mismatch_count,

    "empty_condition_id_rows":
        empty_condition_id_rows,

    "empty_nested_identifier_rows":
        empty_nested_identifier_rows,

    "empty_id_source_inconsistencies":
        empty_id_source_inconsistency_count,
}])

empty_id_summary = pd.DataFrame([{
    "total_rcvs":
        rows_processed,

    "rcvs_with_nonempty_condition_ids":
        rows_processed - empty_condition_id_rows,

    "rcvs_with_empty_condition_ids":
        empty_condition_id_rows,

    "expected_empty_condition_id_rows":
        EXPECTED_EMPTY_ID_ROWS,

    "empty_id_rows_with_empty_nested_identifiers":
        (
            empty_condition_id_rows
            - empty_id_source_inconsistency_count
        ),

    "empty_id_rows_named_exactly_see_cases":
        empty_id_rows_named_see_cases,
}])

blank_and_duplicate_summary = pd.DataFrame([{
    "blank_condition_name_entries":
        blank_condition_name_entries,

    "blank_condition_id_entries":
        blank_condition_id_entries,

    "blank_nested_trait_name_entries":
        blank_trait_name_entries,

    "blank_nested_trait_id_entries":
        blank_trait_id_entries,

    "rows_with_duplicate_condition_names":
        rows_with_duplicate_condition_names,

    "rows_with_duplicate_condition_ids":
        rows_with_duplicate_condition_ids,
}])

id_prefix_summary = pd.DataFrame([
    {
        "identifier_prefix": prefix,
        "count": count,
    }
    for prefix, count
    in condition_id_prefix_frequency.most_common(25)
])

trait_count_summary = pd.DataFrame([
    {
        "trait_records_per_rcv": count,
        "rcv_count": frequency,
    }
    for count, frequency
    in sorted(trait_count_frequency.items())
])


# ------------------------------------------------------------------
# Critical checks
# ------------------------------------------------------------------

critical_checks = {
    "All 71,659 RCV rows were processed":
        rows_processed == EXPECTED_ROWS,

    "All condition JSON fields have valid list structures":
        json_parse_errors == 0,

    "Every nested trait record is a dictionary":
        non_dictionary_trait_records == 0,

    "Every trait record contains a names list":
        (
            trait_records_missing_names_key == 0
            and trait_names_not_lists == 0
        ),

    "Every trait record contains an identifiers list":
        (
            trait_records_missing_identifiers_key == 0
            and trait_identifiers_not_lists == 0
        ),

    "Every RCV contains at least one condition name":
        rows_without_condition_names == 0,

    "Every RCV contains at least one trait record":
        rows_without_trait_records == 0,

    "Top-level condition names match nested trait names":
        condition_name_mismatch_count == 0,

    "Top-level condition IDs match nested trait identifiers":
        condition_id_mismatch_count == 0,

    "Exactly 144 RCVs have no structured condition ID":
        empty_condition_id_rows == EXPECTED_EMPTY_ID_ROWS,

    "All empty condition-ID rows are source-consistent":
        empty_id_source_inconsistency_count == 0,

    "No blank condition or trait entries are present":
        (
            blank_condition_name_entries == 0
            and blank_condition_id_entries == 0
            and blank_trait_name_entries == 0
            and blank_trait_id_entries == 0
        ),
}


# ------------------------------------------------------------------
# Output
# ------------------------------------------------------------------

print("=" * 116)
print("STEP 5B — CONDITION AND TRAIT FIELD VALIDATION")
print("=" * 116)

print("\nCONDITION VALIDATION SUMMARY")
print("-" * 116)
display(validation_summary)

print("\nEMPTY CONDITION-ID ACCEPTANCE SUMMARY")
print("-" * 116)
display(empty_id_summary)

print("\nBLANK AND DUPLICATE VALUE SUMMARY")
print("-" * 116)
display(blank_and_duplicate_summary)

print("\nTRAIT-RECORD COUNT DISTRIBUTION")
print("-" * 116)
display(trait_count_summary)

print("\nTOP CONDITION IDENTIFIER PREFIXES")
print("-" * 116)
display(id_prefix_summary)

print("\nCRITICAL VALIDATION RESULTS")
print("=" * 116)

for check_name, passed in critical_checks.items():
    print(
        f"{'✅ PASS' if passed else '❌ FAIL'} — "
        f"{check_name}"
    )

print("\nCONDITION ISSUE EXAMPLES")
print("-" * 116)

if issue_examples:
    display(pd.DataFrame(issue_examples))
else:
    print(
        "✅ No condition-name, identifier, trait-structure, "
        "or source-consistency issues were found."
    )

failed_checks = [
    check_name
    for check_name, passed in critical_checks.items()
    if not passed
]

if failed_checks:
    raise AssertionError(
        "STEP 5B REQUIRES INVESTIGATION. "
        + "; ".join(failed_checks)
    )

print()
print("✅ STEP 5B COMPLETE")
print(
    "Condition names and identifiers exactly match the nested "
    "trait records."
)
print(
    "The 144 records without structured condition identifiers "
    "are accepted as documented source missingness, not imputed."
)
print("No dataframe or saved artifact was modified.")

FileNotFoundError: Corrected T0 v1.2 was not found:
/content/drive/MyDrive/GES_RAG_Temporal_Study/data_interim/t0_rcv_target_genes_corrected_v1_2.parquet

In [3]:
# STEP 5B.0: Confirm Drive access and locate the corrected T0 v1.2 artifact
# Read-only diagnostic. No file is created, deleted, or modified.

from pathlib import Path
from google.colab import drive

DRIVE_ROOT = Path("/content/drive")
MYDRIVE = DRIVE_ROOT / "MyDrive"

STUDY_DIR = (
    MYDRIVE /
    "GES_RAG_Temporal_Study"
)

DATA_DIR = (
    STUDY_DIR /
    "data_interim"
)

EXPECTED_FILE = (
    DATA_DIR /
    "t0_rcv_target_genes_corrected_v1_2.parquet"
)

REPAIR_MANIFEST = (
    STUDY_DIR /
    "configs" /
    "clinvar_t0_submitter_id_repair_manifest_v1_2.json"
)


# ------------------------------------------------------------
# Mount Drive if the current runtime is not connected
# ------------------------------------------------------------

if not MYDRIVE.exists():
    print("Google Drive is not mounted in this runtime.")
    print("Mounting Drive now...")
    drive.mount("/content/drive")
else:
    print("✅ Google Drive is already mounted.")


print("=" * 100)
print("STEP 5B.0 — LOCATE CORRECTED T0 v1.2")
print("=" * 100)

print(f"MyDrive exists: {MYDRIVE.exists()}")
print(f"Study directory exists: {STUDY_DIR.exists()}")
print(f"Data directory exists: {DATA_DIR.exists()}")

print("\nEXPECTED ARTIFACT")
print("-" * 100)

print(f"Expected path: {EXPECTED_FILE}")
print(f"Exists: {EXPECTED_FILE.exists()}")

if EXPECTED_FILE.exists():
    print(
        f"Size: "
        f"{EXPECTED_FILE.stat().st_size / (1024 ** 2):,.3f} MB"
    )


print("\nREPAIR MANIFEST")
print("-" * 100)

print(f"Expected path: {REPAIR_MANIFEST}")
print(f"Exists: {REPAIR_MANIFEST.exists()}")


print("\nFILES CURRENTLY IN data_interim")
print("-" * 100)

if DATA_DIR.exists():

    files_in_data_dir = sorted(
        path
        for path in DATA_DIR.iterdir()
        if path.is_file()
    )

    if files_in_data_dir:
        for path in files_in_data_dir:
            print(
                f"{path.name} | "
                f"{path.stat().st_size / (1024 ** 2):,.3f} MB"
            )
    else:
        print("The data_interim directory is empty.")

else:
    print("The expected data_interim directory does not exist.")


# ------------------------------------------------------------
# Search MyDrive only if the expected path is still absent
# ------------------------------------------------------------

matching_files = []

if not EXPECTED_FILE.exists():

    print("\nSEARCHING MYDRIVE FOR THE EXACT v1.2 FILENAME")
    print("-" * 100)

    matching_files = list(
        MYDRIVE.rglob(
            "t0_rcv_target_genes_corrected_v1_2.parquet"
        )
    )

    if matching_files:
        for match in matching_files:
            print(
                f"✅ Found: {match}\n"
                f"   Size: "
                f"{match.stat().st_size / (1024 ** 2):,.3f} MB"
            )
    else:
        print("❌ No file with the exact v1.2 filename was found.")


print("\nDIAGNOSTIC DECISION")
print("=" * 100)

if EXPECTED_FILE.exists():
    print("✅ The expected T0 v1.2 artifact is available.")
    print("We can proceed directly to Step 5B.")

elif matching_files:
    print("⚠️ The artifact exists, but under a different directory.")
    print("Use the located path in the next validation cell.")

else:
    print("⚠️ The v1.2 artifact is currently not visible in MyDrive.")
    print(
        "Do not rerun the repair yet; first inspect whether the "
        "repair manifest or source mapping still exists."
    )

print()
print("✅ STEP 5B.0 COMPLETE")
print("No file was modified.")

Google Drive is not mounted in this runtime.
Mounting Drive now...
Mounted at /content/drive
STEP 5B.0 — LOCATE CORRECTED T0 v1.2
MyDrive exists: True
Study directory exists: True
Data directory exists: True

EXPECTED ARTIFACT
----------------------------------------------------------------------------------------------------
Expected path: /content/drive/MyDrive/GES_RAG_Temporal_Study/data_interim/t0_rcv_target_genes_corrected_v1_2.parquet
Exists: True
Size: 2.909 MB

REPAIR MANIFEST
----------------------------------------------------------------------------------------------------
Expected path: /content/drive/MyDrive/GES_RAG_Temporal_Study/configs/clinvar_t0_submitter_id_repair_manifest_v1_2.json
Exists: True

FILES CURRENTLY IN data_interim
----------------------------------------------------------------------------------------------------
t0_rcv_submitter_orgid_source_map_v1_2.parquet | 1.682 MB
t0_rcv_target_genes_corrected_v1_1.parquet | 2.864 MB
t0_rcv_target_genes_corrected_v

In [4]:
# STEP 5B: Validate condition names, identifiers, and nested trait records
# Read-only validation. No dataframe or saved artifact is modified.

import json
import re
from collections import Counter
from pathlib import Path

import pandas as pd


# ------------------------------------------------------------------
# Load corrected T0 v1.2
# ------------------------------------------------------------------

T0_V1_2_PATH = Path(
    "/content/drive/MyDrive/GES_RAG_Temporal_Study/"
    "data_interim/t0_rcv_target_genes_corrected_v1_2.parquet"
)

EXPECTED_ROWS = 71_659
EXPECTED_EMPTY_ID_ROWS = 144

if not T0_V1_2_PATH.exists():
    raise FileNotFoundError(
        f"Corrected T0 v1.2 was not found:\n{T0_V1_2_PATH}"
    )

condition_df = pd.read_parquet(T0_V1_2_PATH)

if len(condition_df) != EXPECTED_ROWS:
    raise AssertionError(
        f"Expected {EXPECTED_ROWS:,} rows, "
        f"found {len(condition_df):,}."
    )


# ------------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------------

def parse_json_list(value, field_name):
    """Parse JSON and require a list as the top-level structure."""
    parsed = json.loads(value) if isinstance(value, str) else value

    if not isinstance(parsed, list):
        raise TypeError(
            f"{field_name} must contain a list, "
            f"found {type(parsed).__name__}."
        )

    return parsed


def normalize_text(value):
    """Normalize whitespace and convert blank-like values to None."""
    if value is None:
        return None

    try:
        if pd.isna(value):
            return None
    except Exception:
        pass

    text = re.sub(
        r"\s+",
        " ",
        str(value),
    ).strip()

    if text.lower() in {
        "",
        "none",
        "null",
        "nan",
        "nat",
    }:
        return None

    return text


def normalized_unique(values):
    """Return sorted, unique, nonblank normalized values."""
    return sorted({
        normalized
        for value in values
        if (
            normalized := normalize_text(value)
        ) is not None
    })


# ------------------------------------------------------------------
# Validation counters
# ------------------------------------------------------------------

rows_processed = 0

json_structure_errors = 0
non_dictionary_trait_records = 0
trait_records_missing_names_key = 0
trait_records_missing_identifiers_key = 0
trait_names_not_lists = 0
trait_identifiers_not_lists = 0

rows_without_condition_names = 0
rows_without_trait_records = 0

condition_name_mismatches = 0
condition_id_mismatches = 0

empty_condition_id_rows = 0
empty_nested_identifier_rows = 0
empty_id_source_inconsistencies = 0
empty_id_rows_named_see_cases = 0

blank_condition_name_entries = 0
blank_condition_id_entries = 0
blank_trait_name_entries = 0
blank_trait_id_entries = 0

rows_with_duplicate_condition_names = 0
rows_with_duplicate_condition_ids = 0

trait_count_frequency = Counter()
condition_id_prefix_frequency = Counter()

issue_examples = []


# ------------------------------------------------------------------
# Validate every RCV
# ------------------------------------------------------------------

columns_needed = [
    "rcv_accession",
    "condition_names_json",
    "condition_ids_json",
    "trait_records_json",
]

for row in condition_df[
    columns_needed
].itertuples(index=False):

    rows_processed += 1
    rcv = row.rcv_accession

    try:
        condition_names_raw = parse_json_list(
            row.condition_names_json,
            "condition_names_json",
        )

        condition_ids_raw = parse_json_list(
            row.condition_ids_json,
            "condition_ids_json",
        )

        trait_records = parse_json_list(
            row.trait_records_json,
            "trait_records_json",
        )

    except Exception as exc:
        json_structure_errors += 1

        if len(issue_examples) < 25:
            issue_examples.append({
                "rcv_accession": rcv,
                "issue": "JSON or top-level structure error",
                "details": f"{type(exc).__name__}: {exc}",
            })

        continue

    condition_names = normalized_unique(
        condition_names_raw
    )

    condition_ids = normalized_unique(
        condition_ids_raw
    )

    blank_condition_name_entries += sum(
        normalize_text(value) is None
        for value in condition_names_raw
    )

    blank_condition_id_entries += sum(
        normalize_text(value) is None
        for value in condition_ids_raw
    )

    nonblank_condition_name_count = sum(
        normalize_text(value) is not None
        for value in condition_names_raw
    )

    nonblank_condition_id_count = sum(
        normalize_text(value) is not None
        for value in condition_ids_raw
    )

    if nonblank_condition_name_count > len(condition_names):
        rows_with_duplicate_condition_names += 1

    if nonblank_condition_id_count > len(condition_ids):
        rows_with_duplicate_condition_ids += 1

    if not condition_names:
        rows_without_condition_names += 1

    if not trait_records:
        rows_without_trait_records += 1

    trait_count_frequency[
        len(trait_records)
    ] += 1

    nested_names_raw = []
    nested_ids_raw = []

    for trait_position, trait_record in enumerate(
        trait_records
    ):

        if not isinstance(trait_record, dict):
            non_dictionary_trait_records += 1

            if len(issue_examples) < 25:
                issue_examples.append({
                    "rcv_accession": rcv,
                    "trait_position": trait_position,
                    "issue": "Trait record is not a dictionary",
                    "value_type": type(trait_record).__name__,
                })

            continue

        if "names" not in trait_record:
            trait_records_missing_names_key += 1
            trait_names = []

            if len(issue_examples) < 25:
                issue_examples.append({
                    "rcv_accession": rcv,
                    "trait_position": trait_position,
                    "issue": "Trait record missing names key",
                    "available_keys": sorted(
                        trait_record.keys()
                    ),
                })

        else:
            trait_names = trait_record["names"]

        if "identifiers" not in trait_record:
            trait_records_missing_identifiers_key += 1
            trait_ids = []

            if len(issue_examples) < 25:
                issue_examples.append({
                    "rcv_accession": rcv,
                    "trait_position": trait_position,
                    "issue": "Trait record missing identifiers key",
                    "available_keys": sorted(
                        trait_record.keys()
                    ),
                })

        else:
            trait_ids = trait_record["identifiers"]

        if not isinstance(trait_names, list):
            trait_names_not_lists += 1
            trait_names = []

            if len(issue_examples) < 25:
                issue_examples.append({
                    "rcv_accession": rcv,
                    "trait_position": trait_position,
                    "issue": "Trait names value is not a list",
                })

        if not isinstance(trait_ids, list):
            trait_identifiers_not_lists += 1
            trait_ids = []

            if len(issue_examples) < 25:
                issue_examples.append({
                    "rcv_accession": rcv,
                    "trait_position": trait_position,
                    "issue": "Trait identifiers value is not a list",
                })

        blank_trait_name_entries += sum(
            normalize_text(value) is None
            for value in trait_names
        )

        blank_trait_id_entries += sum(
            normalize_text(value) is None
            for value in trait_ids
        )

        nested_names_raw.extend(trait_names)
        nested_ids_raw.extend(trait_ids)

    nested_names = normalized_unique(
        nested_names_raw
    )

    nested_ids = normalized_unique(
        nested_ids_raw
    )

    if condition_names != nested_names:
        condition_name_mismatches += 1

        if len(issue_examples) < 25:
            issue_examples.append({
                "rcv_accession": rcv,
                "issue": (
                    "condition_names_json does not match "
                    "trait_records_json"
                ),
                "condition_names": condition_names,
                "nested_names": nested_names,
            })

    if condition_ids != nested_ids:
        condition_id_mismatches += 1

        if len(issue_examples) < 25:
            issue_examples.append({
                "rcv_accession": rcv,
                "issue": (
                    "condition_ids_json does not match "
                    "trait_records_json"
                ),
                "condition_ids": condition_ids,
                "nested_ids": nested_ids,
            })

    top_level_ids_empty = len(condition_ids) == 0
    nested_ids_empty = len(nested_ids) == 0

    if top_level_ids_empty:
        empty_condition_id_rows += 1

    if nested_ids_empty:
        empty_nested_identifier_rows += 1

    if top_level_ids_empty != nested_ids_empty:
        empty_id_source_inconsistencies += 1

    if (
        top_level_ids_empty
        and condition_names == ["See cases"]
    ):
        empty_id_rows_named_see_cases += 1

    for condition_id in condition_ids:
        prefix = (
            condition_id.split(":", 1)[0]
            if ":" in condition_id
            else "NO_PREFIX"
        )

        condition_id_prefix_frequency[
            prefix
        ] += 1


# ------------------------------------------------------------------
# Build summary tables
# ------------------------------------------------------------------

validation_summary = pd.DataFrame([{
    "rows_processed":
        rows_processed,

    "json_structure_errors":
        json_structure_errors,

    "non_dictionary_trait_records":
        non_dictionary_trait_records,

    "trait_records_missing_names_key":
        trait_records_missing_names_key,

    "trait_records_missing_identifiers_key":
        trait_records_missing_identifiers_key,

    "trait_names_not_lists":
        trait_names_not_lists,

    "trait_identifiers_not_lists":
        trait_identifiers_not_lists,

    "rows_without_condition_names":
        rows_without_condition_names,

    "rows_without_trait_records":
        rows_without_trait_records,

    "condition_name_mismatches":
        condition_name_mismatches,

    "condition_id_mismatches":
        condition_id_mismatches,

    "empty_condition_id_rows":
        empty_condition_id_rows,

    "empty_nested_identifier_rows":
        empty_nested_identifier_rows,

    "empty_id_source_inconsistencies":
        empty_id_source_inconsistencies,
}])

empty_id_summary = pd.DataFrame([{
    "total_rcvs":
        rows_processed,

    "rcvs_with_nonempty_condition_ids":
        rows_processed - empty_condition_id_rows,

    "rcvs_with_empty_condition_ids":
        empty_condition_id_rows,

    "expected_empty_condition_id_rows":
        EXPECTED_EMPTY_ID_ROWS,

    "empty_id_rows_with_empty_nested_identifiers":
        empty_nested_identifier_rows,

    "empty_id_rows_named_exactly_see_cases":
        empty_id_rows_named_see_cases,
}])

blank_duplicate_summary = pd.DataFrame([{
    "blank_condition_name_entries":
        blank_condition_name_entries,

    "blank_condition_id_entries":
        blank_condition_id_entries,

    "blank_nested_trait_name_entries":
        blank_trait_name_entries,

    "blank_nested_trait_id_entries":
        blank_trait_id_entries,

    "rows_with_duplicate_condition_names":
        rows_with_duplicate_condition_names,

    "rows_with_duplicate_condition_ids":
        rows_with_duplicate_condition_ids,
}])

trait_count_summary = pd.DataFrame([
    {
        "trait_records_per_rcv": trait_count,
        "rcv_count": frequency,
    }
    for trait_count, frequency
    in sorted(trait_count_frequency.items())
])

id_prefix_summary = pd.DataFrame([
    {
        "identifier_prefix": prefix,
        "count": count,
    }
    for prefix, count
    in condition_id_prefix_frequency.most_common(25)
])


# ------------------------------------------------------------------
# Critical acceptance checks
# ------------------------------------------------------------------

critical_checks = {
    "All 71,659 RCV rows were processed":
        rows_processed == EXPECTED_ROWS,

    "All condition fields contain valid JSON lists":
        json_structure_errors == 0,

    "Every nested trait record is a dictionary":
        non_dictionary_trait_records == 0,

    "Every trait record contains a names list":
        (
            trait_records_missing_names_key == 0
            and trait_names_not_lists == 0
        ),

    "Every trait record contains an identifiers list":
        (
            trait_records_missing_identifiers_key == 0
            and trait_identifiers_not_lists == 0
        ),

    "Every RCV has at least one condition name":
        rows_without_condition_names == 0,

    "Every RCV has at least one trait record":
        rows_without_trait_records == 0,

    "Top-level condition names match nested trait names":
        condition_name_mismatches == 0,

    "Top-level condition IDs match nested trait identifiers":
        condition_id_mismatches == 0,

    "Exactly 144 RCVs lack structured condition IDs":
        empty_condition_id_rows == EXPECTED_EMPTY_ID_ROWS,

    "Empty condition-ID records are source-consistent":
        empty_id_source_inconsistencies == 0,

    "No blank condition or trait entries exist":
        (
            blank_condition_name_entries == 0
            and blank_condition_id_entries == 0
            and blank_trait_name_entries == 0
            and blank_trait_id_entries == 0
        ),
}


# ------------------------------------------------------------------
# Output
# ------------------------------------------------------------------

print("=" * 116)
print("STEP 5B — CONDITION AND TRAIT FIELD VALIDATION")
print("=" * 116)

print("\nCONDITION VALIDATION SUMMARY")
print("-" * 116)
display(validation_summary)

print("\nEMPTY CONDITION-ID ACCEPTANCE SUMMARY")
print("-" * 116)
display(empty_id_summary)

print("\nBLANK AND DUPLICATE VALUE SUMMARY")
print("-" * 116)
display(blank_duplicate_summary)

print("\nTRAIT-RECORD COUNT DISTRIBUTION")
print("-" * 116)
display(trait_count_summary)

print("\nTOP CONDITION IDENTIFIER PREFIXES")
print("-" * 116)
display(id_prefix_summary)

print("\nCRITICAL VALIDATION RESULTS")
print("=" * 116)

for check_name, passed in critical_checks.items():
    print(
        f"{'✅ PASS' if passed else '❌ FAIL'} — "
        f"{check_name}"
    )

print("\nCONDITION ISSUE EXAMPLES")
print("-" * 116)

if issue_examples:
    display(pd.DataFrame(issue_examples))
else:
    print(
        "✅ No condition-name, identifier, trait-structure, "
        "or source-consistency issues were found."
    )

failed_checks = [
    name
    for name, passed in critical_checks.items()
    if not passed
]

if failed_checks:
    raise AssertionError(
        "STEP 5B REQUIRES INVESTIGATION. "
        + "; ".join(failed_checks)
    )

print()
print("✅ STEP 5B COMPLETE")
print(
    "Condition names and identifiers exactly match the nested "
    "trait records."
)
print(
    "The 144 records without structured condition identifiers "
    "are accepted as documented source missingness."
)
print("No dataframe or saved artifact was modified.")

STEP 5B — CONDITION AND TRAIT FIELD VALIDATION

CONDITION VALIDATION SUMMARY
--------------------------------------------------------------------------------------------------------------------


,rows_processed,json_structure_errors,non_dictionary_trait_records,trait_records_missing_names_key,trait_records_missing_identifiers_key,trait_names_not_lists,trait_identifiers_not_lists,rows_without_condition_names,rows_without_trait_records,condition_name_mismatches,condition_id_mismatches,empty_condition_id_rows,empty_nested_identifier_rows,empty_id_source_inconsistencies
0,71659,0,0,0,0,0,0,0,0,0,0,144,144,0



EMPTY CONDITION-ID ACCEPTANCE SUMMARY
--------------------------------------------------------------------------------------------------------------------


,total_rcvs,rcvs_with_nonempty_condition_ids,rcvs_with_empty_condition_ids,expected_empty_condition_id_rows,empty_id_rows_with_empty_nested_identifiers,empty_id_rows_named_exactly_see_cases
0,71659,71515,144,144,144,80



BLANK AND DUPLICATE VALUE SUMMARY
--------------------------------------------------------------------------------------------------------------------


...


TRAIT-RECORD COUNT DISTRIBUTION
--------------------------------------------------------------------------------------------------------------------


,trait_records_per_rcv,rcv_count
0,1,71207
1,2,59
2,3,73
3,4,136
4,8,177
5,9,7



TOP CONDITION IDENTIFIER PREFIXES
--------------------------------------------------------------------------------------------------------------------


,identifier_prefix,count
0,Genetic Testing Registry (GTR),1939723
1,MedGen,73423
2,MONDO,56153
3,OMIM,45571
4,MeSH,37794
5,Genetic Alliance,37613
6,GeneReviews,34780
7,Orphanet,33409
8,SNOMED CT,21807
9,"CSER _CC_NCGL, University of Washington",16917



CRITICAL VALIDATION RESULTS
✅ PASS — All 71,659 RCV rows were processed
✅ PASS — All condition fields contain valid JSON lists
✅ PASS — Every nested trait record is a dictionary
✅ PASS — Every trait record contains a names list
✅ PASS — Every trait record contains an identifiers list
✅ PASS — Every RCV has at least one condition name
✅ PASS — Every RCV has at least one trait record
✅ PASS — Top-level condition names match nested trait names
✅ PASS — Top-level condition IDs match nested trait identifiers
✅ PASS — Exactly 144 RCVs lack structured condition IDs
✅ PASS — Empty condition-ID records are source-consistent
✅ PASS — No blank condition or trait entries exist

CONDITION ISSUE EXAMPLES
--------------------------------------------------------------------------------------------------------------------
✅ No condition-name, identifier, trait-structure, or source-consistency issues were found.

✅ STEP 5B COMPLETE
Condition names and identifiers exactly match the nested trait records.

In [5]:
# STEP 5B: Validate condition names, identifiers, and nested trait records
# Read-only validation. No dataframe or saved artifact is modified.

import json
import re
from collections import Counter
from pathlib import Path

import pandas as pd


# ------------------------------------------------------------------
# Load corrected T0 v1.2
# ------------------------------------------------------------------

T0_V1_2_PATH = Path(
    "/content/drive/MyDrive/GES_RAG_Temporal_Study/"
    "data_interim/t0_rcv_target_genes_corrected_v1_2.parquet"
)

EXPECTED_ROWS = 71_659
EXPECTED_EMPTY_ID_ROWS = 144

if not T0_V1_2_PATH.exists():
    raise FileNotFoundError(
        f"Corrected T0 v1.2 was not found:\n{T0_V1_2_PATH}"
    )

condition_df = pd.read_parquet(T0_V1_2_PATH)

if len(condition_df) != EXPECTED_ROWS:
    raise AssertionError(
        f"Expected {EXPECTED_ROWS:,} rows, "
        f"found {len(condition_df):,}."
    )


# ------------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------------

def parse_json_list(value, field_name):
    """Parse JSON and require a list as the top-level structure."""
    parsed = json.loads(value) if isinstance(value, str) else value

    if not isinstance(parsed, list):
        raise TypeError(
            f"{field_name} must contain a list, "
            f"found {type(parsed).__name__}."
        )

    return parsed


def normalize_text(value):
    """Normalize whitespace and convert blank-like values to None."""
    if value is None:
        return None

    try:
        if pd.isna(value):
            return None
    except Exception:
        pass

    text = re.sub(
        r"\s+",
        " ",
        str(value),
    ).strip()

    if text.lower() in {
        "",
        "none",
        "null",
        "nan",
        "nat",
    }:
        return None

    return text


def normalized_unique(values):
    """Return sorted, unique, nonblank normalized values."""
    return sorted({
        normalized
        for value in values
        if (
            normalized := normalize_text(value)
        ) is not None
    })


# ------------------------------------------------------------------
# Validation counters
# ------------------------------------------------------------------

rows_processed = 0

json_structure_errors = 0
non_dictionary_trait_records = 0
trait_records_missing_names_key = 0
trait_records_missing_identifiers_key = 0
trait_names_not_lists = 0
trait_identifiers_not_lists = 0

rows_without_condition_names = 0
rows_without_trait_records = 0

condition_name_mismatches = 0
condition_id_mismatches = 0

empty_condition_id_rows = 0
empty_nested_identifier_rows = 0
empty_id_source_inconsistencies = 0
empty_id_rows_named_see_cases = 0

blank_condition_name_entries = 0
blank_condition_id_entries = 0
blank_trait_name_entries = 0
blank_trait_id_entries = 0

rows_with_duplicate_condition_names = 0
rows_with_duplicate_condition_ids = 0

trait_count_frequency = Counter()
condition_id_prefix_frequency = Counter()

issue_examples = []


# ------------------------------------------------------------------
# Validate every RCV
# ------------------------------------------------------------------

columns_needed = [
    "rcv_accession",
    "condition_names_json",
    "condition_ids_json",
    "trait_records_json",
]

for row in condition_df[
    columns_needed
].itertuples(index=False):

    rows_processed += 1
    rcv = row.rcv_accession

    try:
        condition_names_raw = parse_json_list(
            row.condition_names_json,
            "condition_names_json",
        )

        condition_ids_raw = parse_json_list(
            row.condition_ids_json,
            "condition_ids_json",
        )

        trait_records = parse_json_list(
            row.trait_records_json,
            "trait_records_json",
        )

    except Exception as exc:
        json_structure_errors += 1

        if len(issue_examples) < 25:
            issue_examples.append({
                "rcv_accession": rcv,
                "issue": "JSON or top-level structure error",
                "details": f"{type(exc).__name__}: {exc}",
            })

        continue

    condition_names = normalized_unique(
        condition_names_raw
    )

    condition_ids = normalized_unique(
        condition_ids_raw
    )

    blank_condition_name_entries += sum(
        normalize_text(value) is None
        for value in condition_names_raw
    )

    blank_condition_id_entries += sum(
        normalize_text(value) is None
        for value in condition_ids_raw
    )

    nonblank_condition_name_count = sum(
        normalize_text(value) is not None
        for value in condition_names_raw
    )

    nonblank_condition_id_count = sum(
        normalize_text(value) is not None
        for value in condition_ids_raw
    )

    if nonblank_condition_name_count > len(condition_names):
        rows_with_duplicate_condition_names += 1

    if nonblank_condition_id_count > len(condition_ids):
        rows_with_duplicate_condition_ids += 1

    if not condition_names:
        rows_without_condition_names += 1

    if not trait_records:
        rows_without_trait_records += 1

    trait_count_frequency[
        len(trait_records)
    ] += 1

    nested_names_raw = []
    nested_ids_raw = []

    for trait_position, trait_record in enumerate(
        trait_records
    ):

        if not isinstance(trait_record, dict):
            non_dictionary_trait_records += 1

            if len(issue_examples) < 25:
                issue_examples.append({
                    "rcv_accession": rcv,
                    "trait_position": trait_position,
                    "issue": "Trait record is not a dictionary",
                    "value_type": type(trait_record).__name__,
                })

            continue

        if "names" not in trait_record:
            trait_records_missing_names_key += 1
            trait_names = []

            if len(issue_examples) < 25:
                issue_examples.append({
                    "rcv_accession": rcv,
                    "trait_position": trait_position,
                    "issue": "Trait record missing names key",
                    "available_keys": sorted(
                        trait_record.keys()
                    ),
                })

        else:
            trait_names = trait_record["names"]

        if "identifiers" not in trait_record:
            trait_records_missing_identifiers_key += 1
            trait_ids = []

            if len(issue_examples) < 25:
                issue_examples.append({
                    "rcv_accession": rcv,
                    "trait_position": trait_position,
                    "issue": "Trait record missing identifiers key",
                    "available_keys": sorted(
                        trait_record.keys()
                    ),
                })

        else:
            trait_ids = trait_record["identifiers"]

        if not isinstance(trait_names, list):
            trait_names_not_lists += 1
            trait_names = []

            if len(issue_examples) < 25:
                issue_examples.append({
                    "rcv_accession": rcv,
                    "trait_position": trait_position,
                    "issue": "Trait names value is not a list",
                })

        if not isinstance(trait_ids, list):
            trait_identifiers_not_lists += 1
            trait_ids = []

            if len(issue_examples) < 25:
                issue_examples.append({
                    "rcv_accession": rcv,
                    "trait_position": trait_position,
                    "issue": "Trait identifiers value is not a list",
                })

        blank_trait_name_entries += sum(
            normalize_text(value) is None
            for value in trait_names
        )

        blank_trait_id_entries += sum(
            normalize_text(value) is None
            for value in trait_ids
        )

        nested_names_raw.extend(trait_names)
        nested_ids_raw.extend(trait_ids)

    nested_names = normalized_unique(
        nested_names_raw
    )

    nested_ids = normalized_unique(
        nested_ids_raw
    )

    if condition_names != nested_names:
        condition_name_mismatches += 1

        if len(issue_examples) < 25:
            issue_examples.append({
                "rcv_accession": rcv,
                "issue": (
                    "condition_names_json does not match "
                    "trait_records_json"
                ),
                "condition_names": condition_names,
                "nested_names": nested_names,
            })

    if condition_ids != nested_ids:
        condition_id_mismatches += 1

        if len(issue_examples) < 25:
            issue_examples.append({
                "rcv_accession": rcv,
                "issue": (
                    "condition_ids_json does not match "
                    "trait_records_json"
                ),
                "condition_ids": condition_ids,
                "nested_ids": nested_ids,
            })

    top_level_ids_empty = len(condition_ids) == 0
    nested_ids_empty = len(nested_ids) == 0

    if top_level_ids_empty:
        empty_condition_id_rows += 1

    if nested_ids_empty:
        empty_nested_identifier_rows += 1

    if top_level_ids_empty != nested_ids_empty:
        empty_id_source_inconsistencies += 1

    if (
        top_level_ids_empty
        and condition_names == ["See cases"]
    ):
        empty_id_rows_named_see_cases += 1

    for condition_id in condition_ids:
        prefix = (
            condition_id.split(":", 1)[0]
            if ":" in condition_id
            else "NO_PREFIX"
        )

        condition_id_prefix_frequency[
            prefix
        ] += 1


# ------------------------------------------------------------------
# Build summary tables
# ------------------------------------------------------------------

validation_summary = pd.DataFrame([{
    "rows_processed":
        rows_processed,

    "json_structure_errors":
        json_structure_errors,

    "non_dictionary_trait_records":
        non_dictionary_trait_records,

    "trait_records_missing_names_key":
        trait_records_missing_names_key,

    "trait_records_missing_identifiers_key":
        trait_records_missing_identifiers_key,

    "trait_names_not_lists":
        trait_names_not_lists,

    "trait_identifiers_not_lists":
        trait_identifiers_not_lists,

    "rows_without_condition_names":
        rows_without_condition_names,

    "rows_without_trait_records":
        rows_without_trait_records,

    "condition_name_mismatches":
        condition_name_mismatches,

    "condition_id_mismatches":
        condition_id_mismatches,

    "empty_condition_id_rows":
        empty_condition_id_rows,

    "empty_nested_identifier_rows":
        empty_nested_identifier_rows,

    "empty_id_source_inconsistencies":
        empty_id_source_inconsistencies,
}])

empty_id_summary = pd.DataFrame([{
    "total_rcvs":
        rows_processed,

    "rcvs_with_nonempty_condition_ids":
        rows_processed - empty_condition_id_rows,

    "rcvs_with_empty_condition_ids":
        empty_condition_id_rows,

    "expected_empty_condition_id_rows":
        EXPECTED_EMPTY_ID_ROWS,

    "empty_id_rows_with_empty_nested_identifiers":
        empty_nested_identifier_rows,

    "empty_id_rows_named_exactly_see_cases":
        empty_id_rows_named_see_cases,
}])

blank_duplicate_summary = pd.DataFrame([{
    "blank_condition_name_entries":
        blank_condition_name_entries,

    "blank_condition_id_entries":
        blank_condition_id_entries,

    "blank_nested_trait_name_entries":
        blank_trait_name_entries,

    "blank_nested_trait_id_entries":
        blank_trait_id_entries,

    "rows_with_duplicate_condition_names":
        rows_with_duplicate_condition_names,

    "rows_with_duplicate_condition_ids":
        rows_with_duplicate_condition_ids,
}])

trait_count_summary = pd.DataFrame([
    {
        "trait_records_per_rcv": trait_count,
        "rcv_count": frequency,
    }
    for trait_count, frequency
    in sorted(trait_count_frequency.items())
])

id_prefix_summary = pd.DataFrame([
    {
        "identifier_prefix": prefix,
        "count": count,
    }
    for prefix, count
    in condition_id_prefix_frequency.most_common(25)
])


# ------------------------------------------------------------------
# Critical acceptance checks
# ------------------------------------------------------------------

critical_checks = {
    "All 71,659 RCV rows were processed":
        rows_processed == EXPECTED_ROWS,

    "All condition fields contain valid JSON lists":
        json_structure_errors == 0,

    "Every nested trait record is a dictionary":
        non_dictionary_trait_records == 0,

    "Every trait record contains a names list":
        (
            trait_records_missing_names_key == 0
            and trait_names_not_lists == 0
        ),

    "Every trait record contains an identifiers list":
        (
            trait_records_missing_identifiers_key == 0
            and trait_identifiers_not_lists == 0
        ),

    "Every RCV has at least one condition name":
        rows_without_condition_names == 0,

    "Every RCV has at least one trait record":
        rows_without_trait_records == 0,

    "Top-level condition names match nested trait names":
        condition_name_mismatches == 0,

    "Top-level condition IDs match nested trait identifiers":
        condition_id_mismatches == 0,

    "Exactly 144 RCVs lack structured condition IDs":
        empty_condition_id_rows == EXPECTED_EMPTY_ID_ROWS,

    "Empty condition-ID records are source-consistent":
        empty_id_source_inconsistencies == 0,

    "No blank condition or trait entries exist":
        (
            blank_condition_name_entries == 0
            and blank_condition_id_entries == 0
            and blank_trait_name_entries == 0
            and blank_trait_id_entries == 0
        ),
}


# ------------------------------------------------------------------
# Output
# ------------------------------------------------------------------

print("=" * 116)
print("STEP 5B — CONDITION AND TRAIT FIELD VALIDATION")
print("=" * 116)

print("\nCONDITION VALIDATION SUMMARY")
print("-" * 116)
display(validation_summary)

print("\nEMPTY CONDITION-ID ACCEPTANCE SUMMARY")
print("-" * 116)
display(empty_id_summary)

print("\nBLANK AND DUPLICATE VALUE SUMMARY")
print("-" * 116)
display(blank_duplicate_summary)

print("\nTRAIT-RECORD COUNT DISTRIBUTION")
print("-" * 116)
display(trait_count_summary)

print("\nTOP CONDITION IDENTIFIER PREFIXES")
print("-" * 116)
display(id_prefix_summary)

print("\nCRITICAL VALIDATION RESULTS")
print("=" * 116)

for check_name, passed in critical_checks.items():
    print(
        f"{'✅ PASS' if passed else '❌ FAIL'} — "
        f"{check_name}"
    )

print("\nCONDITION ISSUE EXAMPLES")
print("-" * 116)

if issue_examples:
    display(pd.DataFrame(issue_examples))
else:
    print(
        "✅ No condition-name, identifier, trait-structure, "
        "or source-consistency issues were found."
    )

failed_checks = [
    name
    for name, passed in critical_checks.items()
    if not passed
]

if failed_checks:
    raise AssertionError(
        "STEP 5B REQUIRES INVESTIGATION. "
        + "; ".join(failed_checks)
    )

print()
print("✅ STEP 5B COMPLETE")
print(
    "Condition names and identifiers exactly match the nested "
    "trait records."
)
print(
    "The 144 records without structured condition identifiers "
    "are accepted as documented source missingness."
)
print("No dataframe or saved artifact was modified.")

STEP 5B — CONDITION AND TRAIT FIELD VALIDATION

CONDITION VALIDATION SUMMARY
--------------------------------------------------------------------------------------------------------------------


,rows_processed,json_structure_errors,non_dictionary_trait_records,trait_records_missing_names_key,trait_records_missing_identifiers_key,trait_names_not_lists,trait_identifiers_not_lists,rows_without_condition_names,rows_without_trait_records,condition_name_mismatches,condition_id_mismatches,empty_condition_id_rows,empty_nested_identifier_rows,empty_id_source_inconsistencies
0,71659,0,0,0,0,0,0,0,0,0,0,144,144,0



EMPTY CONDITION-ID ACCEPTANCE SUMMARY
--------------------------------------------------------------------------------------------------------------------


,total_rcvs,rcvs_with_nonempty_condition_ids,rcvs_with_empty_condition_ids,expected_empty_condition_id_rows,empty_id_rows_with_empty_nested_identifiers,empty_id_rows_named_exactly_see_cases
0,71659,71515,144,144,144,80



BLANK AND DUPLICATE VALUE SUMMARY
--------------------------------------------------------------------------------------------------------------------


...


TRAIT-RECORD COUNT DISTRIBUTION
--------------------------------------------------------------------------------------------------------------------


,trait_records_per_rcv,rcv_count
0,1,71207
1,2,59
2,3,73
3,4,136
4,8,177
5,9,7



TOP CONDITION IDENTIFIER PREFIXES
--------------------------------------------------------------------------------------------------------------------


,identifier_prefix,count
0,Genetic Testing Registry (GTR),1939723
1,MedGen,73423
2,MONDO,56153
3,OMIM,45571
4,MeSH,37794
5,Genetic Alliance,37613
6,GeneReviews,34780
7,Orphanet,33409
8,SNOMED CT,21807
9,"CSER _CC_NCGL, University of Washington",16917



CRITICAL VALIDATION RESULTS
✅ PASS — All 71,659 RCV rows were processed
✅ PASS — All condition fields contain valid JSON lists
✅ PASS — Every nested trait record is a dictionary
✅ PASS — Every trait record contains a names list
✅ PASS — Every trait record contains an identifiers list
✅ PASS — Every RCV has at least one condition name
✅ PASS — Every RCV has at least one trait record
✅ PASS — Top-level condition names match nested trait names
✅ PASS — Top-level condition IDs match nested trait identifiers
✅ PASS — Exactly 144 RCVs lack structured condition IDs
✅ PASS — Empty condition-ID records are source-consistent
✅ PASS — No blank condition or trait entries exist

CONDITION ISSUE EXAMPLES
--------------------------------------------------------------------------------------------------------------------
✅ No condition-name, identifier, trait-structure, or source-consistency issues were found.

✅ STEP 5B COMPLETE
Condition names and identifiers exactly match the nested trait records.

In [7]:
# ==================================================================================================
# STAGE 2A — FREEZE CONSOLIDATED T0 EXTRACTION-VALIDATION MANIFEST
# GES-RAG Temporal Validation Study
#
# Purpose:
#   1. Mount Google Drive.
#   2. Locate all critical T0 artifacts.
#   3. Recalculate and verify published SHA-256 checksums.
#   4. Directly validate the corrected T0 Parquet v1.2.
#   5. Create an immutable consolidated T0 validation manifest.
#   6. Create a separate SHA-256 sidecar for the consolidated manifest.
#
# Important:
#   - This cell does NOT delete the raw T0 XML.
#   - This cell does NOT use T1 data.
#   - This cell does NOT construct temporal outcomes.
#   - This cell does NOT fit or modify the GES model.
# ==================================================================================================

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import sys
import json
import hashlib
import subprocess
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter

# --------------------------------------------------------------------------------------------------
# 1. Ensure required Python packages are available
# --------------------------------------------------------------------------------------------------

try:
    import pandas as pd
    import pyarrow.parquet as pq
except ImportError:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "pandas", "pyarrow"],
        check=True
    )
    import pandas as pd
    import pyarrow.parquet as pq


# --------------------------------------------------------------------------------------------------
# 2. Study paths
# --------------------------------------------------------------------------------------------------

DRIVE_ROOT = Path("/content/drive/MyDrive")
STUDY_ROOT = DRIVE_ROOT / "GES_RAG_Temporal_Study"

CONFIG_DIR = STUDY_ROOT / "configs"
DATA_INTERIM_DIR = STUDY_ROOT / "data_interim"
QUALITY_DIR = STUDY_ROOT / "outputs" / "quality_checks"
LOG_DIR = STUDY_ROOT / "outputs" / "logs"

REPO_DIR = Path("/content/genomic-evidence-reliability")
NOTEBOOK_DIR = DRIVE_ROOT / "Colab Notebooks"

for directory in [
    CONFIG_DIR,
    DATA_INTERIM_DIR,
    QUALITY_DIR,
    LOG_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

print("=" * 110)
print("STAGE 2A — CONSOLIDATED T0 VALIDATION AND ADMINISTRATIVE FREEZE")
print("=" * 110)
print(f"Study root: {STUDY_ROOT}")
print()


# --------------------------------------------------------------------------------------------------
# 3. Obtain a read-only working clone when the repository is not already present
# --------------------------------------------------------------------------------------------------

if not REPO_DIR.exists():
    print("Repository clone not found in the current runtime.")
    print("Cloning the public repository to recover versioned protocol files...")

    subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            "https://github.com/SANGHATI23/genomic-evidence-reliability.git",
            str(REPO_DIR),
        ],
        check=True
    )

print(f"Repository available: {REPO_DIR.exists()}")
print()


# --------------------------------------------------------------------------------------------------
# 4. Helper functions
# --------------------------------------------------------------------------------------------------

def sha256_file(path: Path, chunk_size: int = 8 * 1024 * 1024) -> str:
    """Calculate SHA-256 without loading the complete file into memory."""
    digest = hashlib.sha256()

    with path.open("rb") as file_handle:
        while True:
            chunk = file_handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)

    return digest.hexdigest()


def first_existing(candidates):
    """Return the first existing path from a list of candidate locations."""
    for candidate in candidates:
        candidate = Path(candidate)
        if candidate.exists():
            return candidate
    return None


def parse_json_list(value, field_name: str, row_number=None):
    """Parse a serialized JSON list while accepting an already materialized Python list."""
    if isinstance(value, list):
        parsed = value

    elif value is None:
        parsed = []

    elif isinstance(value, float) and pd.isna(value):
        parsed = []

    elif isinstance(value, str):
        stripped = value.strip()

        if not stripped:
            parsed = []
        else:
            try:
                parsed = json.loads(stripped)
            except json.JSONDecodeError as exc:
                location = f" at row {row_number}" if row_number is not None else ""
                raise ValueError(
                    f"Malformed JSON in {field_name}{location}: {exc}"
                ) from exc
    else:
        location = f" at row {row_number}" if row_number is not None else ""
        raise TypeError(
            f"Unexpected value type in {field_name}{location}: {type(value).__name__}"
        )

    if not isinstance(parsed, list):
        location = f" at row {row_number}" if row_number is not None else ""
        raise TypeError(
            f"{field_name}{location} must contain a JSON list, "
            f"but found {type(parsed).__name__}"
        )

    return parsed


def normalize_boolean(value):
    """Normalize common Boolean representations."""
    if isinstance(value, bool):
        return value

    if value is None or (isinstance(value, float) and pd.isna(value)):
        return False

    normalized = str(value).strip().lower()

    if normalized in {"true", "1", "yes", "y", "t"}:
        return True

    if normalized in {"false", "0", "no", "n", "f", ""}:
        return False

    raise ValueError(f"Unrecognized Boolean value: {value!r}")


def validate_json_document(path: Path):
    """Confirm that a JSON or notebook file is syntactically readable."""
    with path.open("r", encoding="utf-8") as file_handle:
        return json.load(file_handle)


def atomic_write_json(payload, destination: Path):
    """Write JSON through a temporary file and then atomically replace the destination."""
    temporary_path = destination.with_suffix(destination.suffix + ".tmp")

    with temporary_path.open("w", encoding="utf-8") as file_handle:
        json.dump(
            payload,
            file_handle,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            allow_nan=False,
        )
        file_handle.write("\n")

    temporary_path.replace(destination)


# --------------------------------------------------------------------------------------------------
# 5. Expected immutable artifacts and published checksums
# --------------------------------------------------------------------------------------------------

artifact_specs = {
    "protocol_v1_0": {
        "candidates": [
            REPO_DIR / "configs" / "temporal_validation_protocol_v1.json",
            CONFIG_DIR / "temporal_validation_protocol_v1.json",
        ],
        "expected_sha256":
            "60ea29236abddec303b624f85b9a47ad7accf5b0eff47295c772147b9c5f1c43",
        "required": True,
    },

    "protocol_v1_1": {
        "candidates": [
            REPO_DIR / "configs" / "temporal_validation_protocol_v1_1.json",
            CONFIG_DIR / "temporal_validation_protocol_v1_1.json",
        ],
        "expected_sha256":
            "7db783abbcf920127675ba9965bc3c387b5e5cb81f5c80eec276f4bfd48b1482",
        "required": True,
    },

    "original_t0_parquet": {
        "candidates": [
            DATA_INTERIM_DIR / "t0_rcv_target_genes_raw.parquet",
        ],
        "expected_sha256":
            "663ad4c20ce114a763dc6512ff95951f95c2e8448c20a6c4ccb5818486c25f8d",
        "required": True,
    },

    "t0_extraction_manifest": {
        "candidates": [
            CONFIG_DIR / "clinvar_t0_target_gene_rcv_extraction_manifest_v1.json",
        ],
        "expected_sha256":
            "5352cc25e7232444828fc8a5a9906e7a75781463272222b107c20d81c961b094",
        "required": True,
    },

    "corrected_t0_parquet_v1_1": {
        "candidates": [
            DATA_INTERIM_DIR / "t0_rcv_target_genes_corrected_v1_1.parquet",
        ],
        "expected_sha256": None,
        "required": True,
    },

    "conflict_correction_manifest_v1_1": {
        "candidates": [
            CONFIG_DIR / "clinvar_t0_conflict_flag_correction_manifest_v1_1.json",
        ],
        "expected_sha256": None,
        "required": True,
    },

    "parser_conflict_patch_manifest_v1_1": {
        "candidates": [
            CONFIG_DIR / "t0_parser_conflict_rule_patch_manifest_v1_1.json",
        ],
        "expected_sha256": None,
        "required": True,
    },

    "temporal_notebook_v1_1": {
        "candidates": [
            NOTEBOOK_DIR / "02_GES_temporal_validation_v1_1.ipynb",
        ],
        "expected_sha256":
            "856ca46cedfc10dc17fd1c80b651a742a77827430808c4a862abe1145ba27531",
        "required": True,
    },

    "t0_orgid_source_mapping_v1_2": {
        "candidates": [
            DATA_INTERIM_DIR / "t0_rcv_submitter_orgid_source_map_v1_2.parquet",
        ],
        "expected_sha256":
            "0726fbe0542903cd6a5892f8092eb97a7d916d085003e21ac4f5a5a0bc32a197",
        "required": True,
    },

    "orgid_source_audit_manifest_v1_2": {
        "candidates": [
            CONFIG_DIR / "clinvar_t0_submitter_orgid_source_audit_manifest_v1_2.json",
        ],
        "expected_sha256":
            "d239fd7abfee46456c1e6f9537aea06cf4b1c615e5e4843924337be7e7bd6999",
        "required": True,
    },

    "corrected_t0_parquet_v1_2": {
        "candidates": [
            DATA_INTERIM_DIR / "t0_rcv_target_genes_corrected_v1_2.parquet",
        ],
        "expected_sha256":
            "f6b6760b2ad6e4352e3bdecdeaf89827e8a514b031abf2373bb17568d999466d",
        "required": True,
    },

    "submitter_id_repair_manifest_v1_2": {
        "candidates": [
            CONFIG_DIR / "clinvar_t0_submitter_id_repair_manifest_v1_2.json",
        ],
        "expected_sha256":
            "965f2353445b20df3e42c3b9f91398861e76030acd7e16e6f463c105035bdf54",
        "required": True,
    },

    "temporal_notebook_v1_2": {
        "candidates": [
            NOTEBOOK_DIR / "02_GES_temporal_validation_v1_2.ipynb",
        ],
        "expected_sha256":
            "a6c283113cbb8720428fd4a05934381c79a7a17c0b0666d3d05841dd1c07a966",
        "required": True,
    },

    "parser_orgid_patch_manifest_v1_2": {
        "candidates": [
            CONFIG_DIR / "t0_parser_submitter_orgid_patch_manifest_v1_2.json",
        ],
        "expected_sha256":
            "b03545d638c0fd8b2d4a245455d464c3c3a82c647667632e5de2006cd36b5e7a",
        "required": True,
    },

    "parser_runtime_validation_manifest_v1_2": {
        "candidates": [
            CONFIG_DIR / "t0_parser_submitter_orgid_runtime_validation_manifest_v1_2.json",
        ],
        "expected_sha256":
            "095c9edd3e6947af22548ed73fe0b3c4c8272fff33fc98382df8b3ba5da86c86",
        "required": True,
    },
}


# --------------------------------------------------------------------------------------------------
# 6. Locate and verify artifacts
# --------------------------------------------------------------------------------------------------

print("-" * 110)
print("ARTIFACT LOCATION AND CHECKSUM VERIFICATION")
print("-" * 110)

artifact_records = {}
missing_required = []
checksum_failures = []
document_read_failures = []

for artifact_name, specification in artifact_specs.items():
    located_path = first_existing(specification["candidates"])
    expected_hash = specification["expected_sha256"]
    required = specification["required"]

    if located_path is None:
        status = "MISSING"

        artifact_records[artifact_name] = {
            "status": status,
            "required": required,
            "candidate_paths": [str(path) for path in specification["candidates"]],
            "expected_sha256": expected_hash,
        }

        if required:
            missing_required.append(artifact_name)

        print(f"[MISSING] {artifact_name}")
        continue

    observed_hash = sha256_file(located_path)
    size_bytes = located_path.stat().st_size

    checksum_matches = (
        expected_hash is None or observed_hash.lower() == expected_hash.lower()
    )

    readable = True
    read_error = None

    if located_path.suffix.lower() in {".json", ".ipynb"}:
        try:
            validate_json_document(located_path)
        except Exception as exc:
            readable = False
            read_error = f"{type(exc).__name__}: {exc}"
            document_read_failures.append(artifact_name)

    if not checksum_matches:
        checksum_failures.append(artifact_name)

    status = "PASS" if checksum_matches and readable else "FAIL"

    artifact_records[artifact_name] = {
        "status": status,
        "required": required,
        "path": str(located_path),
        "filename": located_path.name,
        "size_bytes": int(size_bytes),
        "observed_sha256": observed_hash,
        "expected_sha256": expected_hash,
        "checksum_matches_expected": bool(checksum_matches),
        "json_or_notebook_readable": bool(readable),
        "read_error": read_error,
    }

    checksum_message = (
        "checksum verified"
        if expected_hash is not None and checksum_matches
        else "checksum recorded"
        if expected_hash is None
        else "CHECKSUM MISMATCH"
    )

    print(
        f"[{status}] {artifact_name}\n"
        f"       path: {located_path}\n"
        f"       {checksum_message}: {observed_hash}"
    )

print()


# --------------------------------------------------------------------------------------------------
# 7. Stop before creating a freeze manifest if a required artifact failed
# --------------------------------------------------------------------------------------------------

if missing_required or checksum_failures or document_read_failures:
    print("=" * 110)
    print("T0 ADMINISTRATIVE FREEZE NOT CREATED")
    print("=" * 110)

    if missing_required:
        print("\nMissing required artifacts:")
        for name in missing_required:
            print(f"  - {name}")

    if checksum_failures:
        print("\nArtifacts with checksum mismatches:")
        for name in checksum_failures:
            print(f"  - {name}")

    if document_read_failures:
        print("\nUnreadable JSON/notebook artifacts:")
        for name in document_read_failures:
            print(f"  - {name}")

    raise RuntimeError(
        "The consolidated manifest was not written because one or more "
        "required T0 artifacts were missing, unreadable, or checksum-inconsistent."
    )


# --------------------------------------------------------------------------------------------------
# 8. Load and directly validate corrected T0 Parquet v1.2
# --------------------------------------------------------------------------------------------------

print("-" * 110)
print("DIRECT VALIDATION OF CORRECTED T0 PARQUET v1.2")
print("-" * 110)

T0_V1_2_PATH = Path(
    artifact_records["corrected_t0_parquet_v1_2"]["path"]
)

parquet_metadata = pq.ParquetFile(T0_V1_2_PATH)
dataframe = pd.read_parquet(T0_V1_2_PATH)

expected_columns = {
    "timepoint",
    "release_label",
    "archive_publication_date",
    "embedded_data_cutoff_date",
    "xml_record_index",
    "source_filename",
    "source_sha256",
    "rcv_accession",
    "rcv_version",
    "variation_id",
    "vcv_accession",
    "vcv_version",
    "target_genes_json",
    "study_scope",
    "measure_set_type",
    "measure_types_json",
    "condition_names_json",
    "condition_ids_json",
    "trait_records_json",
    "aggregate_classification",
    "aggregate_classification_group",
    "aggregate_review_status",
    "aggregate_review_stars",
    "aggregate_last_evaluated",
    "aggregate_explanation",
    "aggregate_conflict_flag",
    "scv_group_disagreement_flag",
    "scv_count_xml",
    "unique_submitter_count_xml",
    "submitters_json",
    "submitter_ids_json",
    "scv_classification_counts_json",
    "scv_group_counts_json",
    "scv_records_json",
}

missing_columns = sorted(expected_columns - set(dataframe.columns))
unexpected_columns = sorted(set(dataframe.columns) - expected_columns)

if missing_columns:
    raise AssertionError(f"Required T0 columns are missing: {missing_columns}")

if unexpected_columns:
    raise AssertionError(f"Unexpected T0 columns were found: {unexpected_columns}")


# Core row and identifier checks
row_count = int(len(dataframe))
column_count = int(len(dataframe.columns))
row_group_count = int(parquet_metadata.num_row_groups)

missing_rcv = int(dataframe["rcv_accession"].isna().sum())
blank_rcv = int(
    dataframe["rcv_accession"].fillna("").astype(str).str.strip().eq("").sum()
)

missing_variation_id = int(dataframe["variation_id"].isna().sum())

unique_rcvs = int(dataframe["rcv_accession"].nunique(dropna=True))
unique_rcv_version_pairs = int(
    dataframe[["rcv_accession", "rcv_version"]]
    .drop_duplicates()
    .shape[0]
)

unique_xml_indexes = int(dataframe["xml_record_index"].nunique(dropna=True))
unique_variation_ids = int(dataframe["variation_id"].nunique(dropna=True))
unique_vcv_accessions = int(dataframe["vcv_accession"].nunique(dropna=True))


# Gene validation
gene_counter = Counter()
invalid_gene_rows = 0

for row_number, value in enumerate(dataframe["target_genes_json"], start=1):
    genes = parse_json_list(value, "target_genes_json", row_number)

    if len(genes) != 1:
        invalid_gene_rows += 1
        continue

    normalized_gene = str(genes[0]).strip().upper()
    gene_counter[normalized_gene] += 1

expected_gene_counts = {
    "BRCA1": 25408,
    "BRCA2": 34915,
    "MLH1": 8936,
    "EGFR": 2400,
}


# Conflict validation
conflict_values = dataframe["aggregate_conflict_flag"].map(normalize_boolean)
aggregate_conflict_positive = int(conflict_values.sum())


# Review-star distribution
review_star_series = pd.to_numeric(
    dataframe["aggregate_review_stars"],
    errors="raise"
).astype(int)

review_star_counts = {
    str(int(star)): int(count)
    for star, count in review_star_series.value_counts().sort_index().items()
}

expected_review_star_counts = {
    "0": 3878,
    "1": 50399,
    "2": 9224,
    "3": 8158,
}


# Submitter-ID and condition-ID validation
rows_with_nonempty_submitter_ids = 0
rows_with_empty_condition_ids = 0
rows_named_exactly_see_cases = 0
total_rcv_level_orgid_entries = 0

for row_number, row in enumerate(
    dataframe[
        [
            "submitter_ids_json",
            "condition_ids_json",
            "condition_names_json",
        ]
    ].itertuples(index=False),
    start=1
):
    submitter_ids = parse_json_list(
        row.submitter_ids_json,
        "submitter_ids_json",
        row_number
    )

    condition_ids = parse_json_list(
        row.condition_ids_json,
        "condition_ids_json",
        row_number
    )

    condition_names = parse_json_list(
        row.condition_names_json,
        "condition_names_json",
        row_number
    )

    normalized_submitter_ids = {
        str(identifier).strip()
        for identifier in submitter_ids
        if str(identifier).strip()
    }

    total_rcv_level_orgid_entries += len(normalized_submitter_ids)

    if normalized_submitter_ids:
        rows_with_nonempty_submitter_ids += 1

    if len(condition_ids) == 0:
        rows_with_empty_condition_ids += 1

        normalized_condition_names = [
            str(name).strip()
            for name in condition_names
            if str(name).strip()
        ]

        if normalized_condition_names == ["See cases"]:
            rows_named_exactly_see_cases += 1


# SCV totals
scv_count_series = pd.to_numeric(
    dataframe["scv_count_xml"],
    errors="raise"
).astype(int)

total_scv_records = int(scv_count_series.sum())


# Provenance constants
timepoints = sorted(
    dataframe["timepoint"].dropna().astype(str).str.strip().unique().tolist()
)

release_labels = sorted(
    dataframe["release_label"].dropna().astype(str).str.strip().unique().tolist()
)

publication_dates = sorted(
    dataframe["archive_publication_date"]
    .dropna()
    .astype(str)
    .str.strip()
    .unique()
    .tolist()
)

embedded_cutoffs = sorted(
    dataframe["embedded_data_cutoff_date"]
    .dropna()
    .astype(str)
    .str.strip()
    .unique()
    .tolist()
)

source_filenames = sorted(
    dataframe["source_filename"]
    .dropna()
    .astype(str)
    .str.strip()
    .unique()
    .tolist()
)

source_sha256_values = sorted(
    dataframe["source_sha256"]
    .dropna()
    .astype(str)
    .str.strip()
    .unique()
    .tolist()
)


# Aggregate-date validation
aggregate_date_text = (
    dataframe["aggregate_last_evaluated"]
    .fillna("")
    .astype(str)
    .str.strip()
)

aggregate_date_present = aggregate_date_text.ne("")
aggregate_dates = pd.to_datetime(
    aggregate_date_text.where(aggregate_date_present),
    errors="coerce"
)

aggregate_date_malformed = int(
    (aggregate_date_present & aggregate_dates.isna()).sum()
)

aggregate_dates_after_cutoff = int(
    (
        aggregate_dates.notna()
        & (aggregate_dates > pd.Timestamp("2022-12-31"))
    ).sum()
)

aggregate_date_available = int(aggregate_dates.notna().sum())
aggregate_date_missing = int(row_count - aggregate_date_available)

aggregate_date_min = (
    aggregate_dates.min().date().isoformat()
    if aggregate_date_available
    else None
)

aggregate_date_max = (
    aggregate_dates.max().date().isoformat()
    if aggregate_date_available
    else None
)


# --------------------------------------------------------------------------------------------------
# 9. Expected-result assertions
# --------------------------------------------------------------------------------------------------

assert row_count == 71659, f"Expected 71,659 rows, found {row_count:,}"
assert column_count == 34, f"Expected 34 columns, found {column_count}"
# v1.2 was rewritten after the submitter-ID repair, so its physical
# Parquet row-group layout may differ from the original extraction.
# Row groups are a storage-layout property, not a scientific-content requirement.

assert row_group_count >= 1, (
    f"Parquet must contain at least one row group, found {row_group_count}"
)

print(
    f"Parquet row-group validation: PASS "
    f"(observed {row_group_count}; physical layout accepted)"
)

assert missing_rcv == 0
assert blank_rcv == 0
assert missing_variation_id == 0
assert unique_rcvs == 71659
assert unique_rcv_version_pairs == 71659
assert unique_xml_indexes == 71659
assert unique_variation_ids == 36807
assert unique_vcv_accessions == 36807

assert invalid_gene_rows == 0
assert dict(gene_counter) == expected_gene_counts, (
    f"Unexpected gene counts: {dict(gene_counter)}"
)

assert aggregate_conflict_positive == 1484, (
    f"Expected 1,484 conflict-positive RCVs, "
    f"found {aggregate_conflict_positive:,}"
)

assert review_star_counts == expected_review_star_counts, (
    f"Unexpected review-star distribution: {review_star_counts}"
)

assert rows_with_nonempty_submitter_ids == 71659
assert total_rcv_level_orgid_entries == 100614
assert total_scv_records == 100633

assert rows_with_empty_condition_ids == 144
assert rows_named_exactly_see_cases == 80

assert timepoints == ["T0"]
assert release_labels == ["2023-01"]
assert publication_dates == ["2023-01-05"]
assert embedded_cutoffs == ["2022-12-31"]
assert source_filenames == ["ClinVarFullRelease_2023-01.xml.gz"]

assert source_sha256_values == [
    "911c8a58872ea89cc7bb4f1ee3362596d965f5103422b30f456abaf99c41e5e7"
]

assert aggregate_date_available == 65770
assert aggregate_date_missing == 5889
assert aggregate_date_malformed == 0
assert aggregate_dates_after_cutoff == 0
assert aggregate_date_min == "1994-03-17"
assert aggregate_date_max == "2022-12-23"


validation_summary = {
    "parquet_rows": row_count,
    "parquet_columns": column_count,
    "parquet_row_groups": row_group_count,
    "unique_rcv_accessions": unique_rcvs,
    "unique_rcv_version_pairs": unique_rcv_version_pairs,
    "unique_xml_record_indexes": unique_xml_indexes,
    "unique_variation_ids": unique_variation_ids,
    "unique_vcv_accessions": unique_vcv_accessions,
    "missing_rcv_accessions": missing_rcv,
    "blank_rcv_accessions": blank_rcv,
    "missing_variation_ids": missing_variation_id,
    "gene_counts": dict(gene_counter),
    "aggregate_conflict_positive_records": aggregate_conflict_positive,
    "review_star_counts": review_star_counts,
    "total_nested_scv_records_from_saved_counts": total_scv_records,
    "rows_with_nonempty_submitter_ids": rows_with_nonempty_submitter_ids,
    "rcv_level_orgid_entries_after_within_rcv_deduplication":
        total_rcv_level_orgid_entries,
    "rows_with_empty_condition_ids": rows_with_empty_condition_ids,
    "empty_condition_id_rows_named_exactly_see_cases":
        rows_named_exactly_see_cases,
    "aggregate_date_available": aggregate_date_available,
    "aggregate_date_missing": aggregate_date_missing,
    "aggregate_date_range": {
        "minimum": aggregate_date_min,
        "maximum": aggregate_date_max,
    },
    "aggregate_date_malformed": aggregate_date_malformed,
    "aggregate_dates_after_t0_cutoff": aggregate_dates_after_cutoff,
    "provenance_constants": {
        "timepoint": timepoints,
        "release_label": release_labels,
        "archive_publication_date": publication_dates,
        "embedded_data_cutoff_date": embedded_cutoffs,
        "source_filename": source_filenames,
        "source_sha256": source_sha256_values,
    },
}

for key, value in validation_summary.items():
    print(f"{key}: {value}")

print()


# --------------------------------------------------------------------------------------------------
# 10. Discover supplementary validation artifacts without modifying them
# --------------------------------------------------------------------------------------------------

supplementary_patterns = [
    "*json*valid*",
    "*nested*valid*",
    "*date*valid*",
    "*condition*valid*",
    "*trait*valid*",
    "*field*valid*",
    "*quality*check*",
]

supplementary_paths = set()

for search_root in [CONFIG_DIR, QUALITY_DIR, LOG_DIR]:
    if search_root.exists():
        for pattern in supplementary_patterns:
            supplementary_paths.update(
                path
                for path in search_root.rglob(pattern)
                if path.is_file()
            )

supplementary_artifacts = []

for path in sorted(supplementary_paths):
    supplementary_artifacts.append(
        {
            "path": str(path),
            "filename": path.name,
            "size_bytes": int(path.stat().st_size),
            "sha256": sha256_file(path),
        }
    )


# --------------------------------------------------------------------------------------------------
# 11. Detect whether the temporary raw T0 XML remains in the active Colab runtime
# --------------------------------------------------------------------------------------------------

raw_xml_candidates = [
    Path("/content/clinvar_rcv_raw/ClinVarFullRelease_2023-01.xml.gz"),
    Path("/content/ClinVarFullRelease_2023-01.xml.gz"),
]

raw_xml_path = first_existing(raw_xml_candidates)

raw_xml_record = {
    "expected_filename": "ClinVarFullRelease_2023-01.xml.gz",
    "expected_size_bytes": 2510598470,
    "expected_md5": "d00be8862bbbd1d5a8b991c30246d224",
    "expected_sha256":
        "911c8a58872ea89cc7bb4f1ee3362596d965f5103422b30f456abaf99c41e5e7",
    "paths_checked": [str(path) for path in raw_xml_candidates],
    "present_in_current_runtime": bool(raw_xml_path is not None),
    "observed_path": str(raw_xml_path) if raw_xml_path else None,
    "deletion_performed_by_this_cell": False,
    "deletion_policy":
        "Deletion is permitted only after this consolidated manifest and its "
        "SHA-256 sidecar are successfully written and read back.",
}


# --------------------------------------------------------------------------------------------------
# 12. Create consolidated T0 validation manifest
# --------------------------------------------------------------------------------------------------

creation_time = datetime.now(timezone.utc).replace(microsecond=0).isoformat()

manifest = {
    "manifest_name":
        "clinvar_t0_consolidated_extraction_validation_manifest_v1_2",
    "manifest_version": "1.2.0",
    "created_utc": creation_time,

    "study": {
        "title":
            "GES-RAG: Temporal Validation and Stability-Aware Context Assembly "
            "for Reliable Genomic Question Answering",
        "experiment": "Experiment 1 — Temporal Validation of GES",
        "timepoint": "T0",
        "release_label": "2023-01",
        "archive_publication_date": "2023-01-05",
        "embedded_data_cutoff_date": "2022-12-31",
        "primary_unit": "RCV-level variant-condition aggregate",
        "primary_genes": ["BRCA1", "BRCA2", "MLH1"],
        "exploratory_genes": ["EGFR"],
    },

    "administrative_freeze_decision": {
        "decision": "ACCEPTED_AND_FROZEN",
        "accepted_artifact":
            "t0_rcv_target_genes_corrected_v1_2.parquet",
        "accepted_for": [
            "T0-only GES feature reconstruction",
            "T1 schema harmonization planning",
            "T0-T1 RCV crosswalk construction after T1 extraction",
            "future-instability outcome construction after linkage",
        ],
        "not_yet_authorized_for": [
            "Claiming temporal predictive performance",
            "Claiming superiority over ClinVar review stars",
            "Claiming downstream RAG improvement",
        ],
        "rationale": [
            "All required immutable T0 artifacts were located.",
            "All available published SHA-256 checksums matched.",
            "The corrected T0 v1.2 Parquet passed direct readback validation.",
            "The final cohort contains 71,659 unique RCV records.",
            "The final cohort contains 100,633 submitted SCVs.",
            "Aggregate conflict-positive records equal the corrected count of 1,484.",
            "Primary submitter OrgIDs are nonempty for all 71,659 RCVs.",
            "The 144 empty condition-ID rows match documented source missingness.",
            "No T1 data or temporal outcomes were used in the T0 corrections.",
        ],
    },

    "accepted_t0_artifact": {
        "path": str(T0_V1_2_PATH),
        "filename": T0_V1_2_PATH.name,
        "sha256": sha256_file(T0_V1_2_PATH),
        "size_bytes": int(T0_V1_2_PATH.stat().st_size),
        "rows": row_count,
        "columns": column_count,
        "row_groups": row_group_count,
    },

    "validation_summary": validation_summary,

    "accepted_source_missingness": {
        "structured_condition_identifier_missing_rows": 144,
        "source_consistency_confirmed": True,
        "imputation_performed": False,
        "rows_named_exactly_see_cases": 80,
        "interpretation":
            "These records retain condition names and trait structures but do "
            "not contain structured condition identifiers in the historical source.",
    },

    "corrected_parser_defects": {
        "aggregate_conflict_semantics": {
            "status": "CORRECTED_BEFORE_T1_OUTCOME_CONSTRUCTION",
            "original_conflict_positive_records": 10708,
            "corrected_conflict_positive_records": 1484,
            "true_to_false_corrections": 9224,
            "false_to_true_corrections": 0,
        },
        "submitter_orgid_extraction": {
            "status": "CORRECTED_AND_RUNTIME_VALIDATED",
            "submitted_scvs_with_primary_orgid": 100633,
            "rcvs_with_primary_orgid": 71659,
            "unique_primary_organization_ids": 262,
            "runtime_exceptions": 0,
            "runtime_parquet_mismatches": 0,
        },
    },

    "leakage_controls": {
        "t1_information_used_during_t0_extraction": False,
        "t1_information_used_during_t0_validation": False,
        "temporal_outcome_created": False,
        "ges_model_fitted": False,
        "ges_threshold_selected": False,
        "t0_model_or_features_tuned_using_t1": False,
    },

    "immutable_artifacts": artifact_records,
    "supplementary_validation_artifacts": supplementary_artifacts,
    "raw_t0_xml_disposition": raw_xml_record,

    "next_required_stage_2_actions": [
        "Download the January 2026 condition-specific RCV XML release.",
        "Verify official file size and MD5.",
        "Calculate SHA-256.",
        "Inspect root metadata and determine the embedded T1 cutoff.",
        "Document the current ClinVar RCV XML schema.",
        "Stream-extract BRCA1, BRCA2, MLH1, and EGFR RCV records.",
        "Validate and freeze the harmonized T1 extraction.",
    ],

    "stage_3_entry_condition": {
        "stage": "Build the T0-T1 variant-condition crosswalk",
        "may_begin_now": False,
        "reason":
            "Stage 3 must wait until the T1 RCV extraction and schema "
            "harmonization are complete.",
    },
}


# --------------------------------------------------------------------------------------------------
# 13. Write, hash, and read back the consolidated manifest
# --------------------------------------------------------------------------------------------------

MANIFEST_PATH = (
    CONFIG_DIR /
    "clinvar_t0_consolidated_extraction_validation_manifest_v1_2.json"
)

SHA256_PATH = MANIFEST_PATH.with_suffix(MANIFEST_PATH.suffix + ".sha256")

atomic_write_json(manifest, MANIFEST_PATH)

manifest_sha256 = sha256_file(MANIFEST_PATH)

SHA256_PATH.write_text(
    f"{manifest_sha256}  {MANIFEST_PATH.name}\n",
    encoding="utf-8"
)

# Complete readback verification
with MANIFEST_PATH.open("r", encoding="utf-8") as file_handle:
    readback_manifest = json.load(file_handle)

assert readback_manifest == manifest, (
    "Manifest readback did not exactly match the object written."
)

sidecar_text = SHA256_PATH.read_text(encoding="utf-8").strip()
assert sidecar_text == f"{manifest_sha256}  {MANIFEST_PATH.name}"

assert sha256_file(MANIFEST_PATH) == manifest_sha256


# --------------------------------------------------------------------------------------------------
# 14. Final output
# --------------------------------------------------------------------------------------------------

print("=" * 110)
print("STAGE 2A RESULT — PASS")
print("=" * 110)
print("T0 administrative freeze decision: ACCEPTED_AND_FROZEN")
print(f"Accepted T0 artifact: {T0_V1_2_PATH}")
print(f"Accepted T0 artifact SHA-256: {sha256_file(T0_V1_2_PATH)}")
print()
print(f"Consolidated manifest: {MANIFEST_PATH}")
print(f"Consolidated manifest SHA-256: {manifest_sha256}")
print(f"SHA-256 sidecar: {SHA256_PATH}")
print()
print(f"Raw T0 XML present in this runtime: {raw_xml_record['present_in_current_runtime']}")
print("Raw T0 XML deleted by this cell: False")
print()
print("NEXT REQUIRED ACTION:")
print("Download, verify, inspect, and stream-extract the January 2026 T1 RCV XML.")
print("Stage 3 crosswalk must not begin until the T1 extraction is validated.")
print("=" * 110)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
STAGE 2A — CONSOLIDATED T0 VALIDATION AND ADMINISTRATIVE FREEZE
Study root: /content/drive/MyDrive/GES_RAG_Temporal_Study

Repository available: True

--------------------------------------------------------------------------------------------------------------
ARTIFACT LOCATION AND CHECKSUM VERIFICATION
--------------------------------------------------------------------------------------------------------------
[PASS] protocol_v1_0
       path: /content/drive/MyDrive/GES_RAG_Temporal_Study/configs/temporal_validation_protocol_v1.json
       checksum verified: 60ea29236abddec303b624f85b9a47ad7accf5b0eff47295c772147b9c5f1c43
[PASS] protocol_v1_1
       path: /content/drive/MyDrive/GES_RAG_Temporal_Study/configs/temporal_validation_protocol_v1_1.json
       checksum verified: 7db783abbcf920127675ba9965bc3c387b5e5cb81f5c80eec276f4bfd48b1482
[PASS] original_t0_p

In [8]:
# ==================================================================================================
# STAGE 2B — DOWNLOAD AND VERIFY THE JANUARY 2026 T1 RCV XML
# GES-RAG Temporal Validation Study
#
# This cell:
#   1. Mounts Google Drive.
#   2. Confirms the frozen T0 manifest exists.
#   3. Retrieves the official NCBI MD5 sidecar.
#   4. Checks temporary Colab storage.
#   5. Downloads the 5.43 GB T1 RCV XML with resume support.
#   6. Verifies exact byte size.
#   7. Calculates MD5 and SHA-256.
#   8. Performs a complete gzip-stream integrity test.
#   9. Saves a persistent download-verification receipt to Drive.
#
# This cell does NOT:
#   - extract T1 target-gene records;
#   - inspect or assume the embedded T1 data cutoff;
#   - build the T0–T1 crosswalk;
#   - use T1 information to alter the T0 cohort or GES model.
# ==================================================================================================

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os
import re
import json
import shutil
import hashlib
import subprocess
from pathlib import Path
from datetime import datetime, timezone

import requests


# --------------------------------------------------------------------------------------------------
# 1. Paths and frozen source definitions
# --------------------------------------------------------------------------------------------------

DRIVE_ROOT = Path("/content/drive/MyDrive")
STUDY_ROOT = DRIVE_ROOT / "GES_RAG_Temporal_Study"

CONFIG_DIR = STUDY_ROOT / "configs"
LOG_DIR = STUDY_ROOT / "outputs" / "logs"

TEMP_RAW_DIR = Path("/content/clinvar_rcv_raw")
TEMP_RAW_DIR.mkdir(parents=True, exist_ok=True)

CONFIG_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

T0_FREEZE_MANIFEST = (
    CONFIG_DIR /
    "clinvar_t0_consolidated_extraction_validation_manifest_v1_2.json"
)

T1_FILENAME = "ClinVarRCVRelease_2026-01.xml.gz"
T1_MD5_FILENAME = f"{T1_FILENAME}.md5"

T1_URL = (
    "https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/"
    "RCV_release/ClinVarRCVRelease_2026-01.xml.gz"
)

T1_MD5_URL = (
    "https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/"
    "RCV_release/ClinVarRCVRelease_2026-01.xml.gz.md5"
)

T1_LOCAL_PATH = TEMP_RAW_DIR / T1_FILENAME
T1_MD5_LOCAL_PATH = TEMP_RAW_DIR / T1_MD5_FILENAME

# Official NCBI archive listing value.
EXPECTED_SIZE_BYTES = 5_434_707_247

# Keep free temporary space after downloading.
RESERVE_BYTES = int(1.5 * 1024**3)

print("=" * 112)
print("STAGE 2B — T1 RCV XML DOWNLOAD AND SOURCE VERIFICATION")
print("=" * 112)
print(f"T1 release label: January 2026")
print(f"Temporary destination: {T1_LOCAL_PATH}")
print(f"Expected compressed size: {EXPECTED_SIZE_BYTES:,} bytes")
print()


# --------------------------------------------------------------------------------------------------
# 2. Confirm T0 was formally frozen before T1 work begins
# --------------------------------------------------------------------------------------------------

if not T0_FREEZE_MANIFEST.exists():
    raise FileNotFoundError(
        "The consolidated T0 freeze manifest was not found:\n"
        f"{T0_FREEZE_MANIFEST}\n\n"
        "Do not proceed with T1 processing until Stage 2A has passed."
    )

with T0_FREEZE_MANIFEST.open("r", encoding="utf-8") as handle:
    t0_manifest = json.load(handle)

t0_decision = (
    t0_manifest
    .get("administrative_freeze_decision", {})
    .get("decision")
)

if t0_decision != "ACCEPTED_AND_FROZEN":
    raise AssertionError(
        "The T0 manifest exists, but its administrative decision is not "
        f"ACCEPTED_AND_FROZEN. Observed: {t0_decision!r}"
    )

print("T0 prerequisite: PASS — ACCEPTED_AND_FROZEN")
print()


# --------------------------------------------------------------------------------------------------
# 3. Helper functions
# --------------------------------------------------------------------------------------------------

def utc_now():
    return datetime.now(timezone.utc).replace(microsecond=0).isoformat()


def human_gib(number_of_bytes):
    return number_of_bytes / (1024**3)


def atomic_write_json(payload, destination):
    temporary_path = destination.with_suffix(destination.suffix + ".tmp")

    with temporary_path.open("w", encoding="utf-8") as handle:
        json.dump(
            payload,
            handle,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            allow_nan=False,
        )
        handle.write("\n")

    temporary_path.replace(destination)


def fetch_http_metadata(url):
    """
    Attempt HEAD first. If Content-Length is unavailable, request one byte
    and infer full size from Content-Range.
    """
    headers = {
        "User-Agent": "GES-RAG-temporal-validation/1.0"
    }

    metadata = {
        "requested_url": url,
        "final_url": None,
        "status_code": None,
        "content_length": None,
        "last_modified": None,
        "etag": None,
        "accept_ranges": None,
        "method_used": None,
    }

    try:
        response = requests.head(
            url,
            headers=headers,
            allow_redirects=True,
            timeout=60,
        )

        metadata.update(
            {
                "final_url": response.url,
                "status_code": response.status_code,
                "content_length": response.headers.get("Content-Length"),
                "last_modified": response.headers.get("Last-Modified"),
                "etag": response.headers.get("ETag"),
                "accept_ranges": response.headers.get("Accept-Ranges"),
                "method_used": "HEAD",
            }
        )

        if response.ok and metadata["content_length"]:
            metadata["content_length"] = int(metadata["content_length"])
            return metadata

    except requests.RequestException:
        pass

    range_headers = {
        **headers,
        "Range": "bytes=0-0",
    }

    response = requests.get(
        url,
        headers=range_headers,
        allow_redirects=True,
        stream=True,
        timeout=60,
    )

    response.raise_for_status()

    content_range = response.headers.get("Content-Range")
    inferred_size = None

    if content_range:
        match = re.search(r"/(\d+)$", content_range)
        if match:
            inferred_size = int(match.group(1))

    if inferred_size is None:
        length_header = response.headers.get("Content-Length")
        if length_header and response.status_code == 200:
            inferred_size = int(length_header)

    metadata.update(
        {
            "final_url": response.url,
            "status_code": response.status_code,
            "content_length": inferred_size,
            "last_modified": response.headers.get("Last-Modified"),
            "etag": response.headers.get("ETag"),
            "accept_ranges": response.headers.get("Accept-Ranges"),
            "method_used": "RANGE_GET",
        }
    )

    response.close()
    return metadata


def download_small_text_file(url, destination):
    headers = {
        "User-Agent": "GES-RAG-temporal-validation/1.0"
    }

    response = requests.get(
        url,
        headers=headers,
        allow_redirects=True,
        timeout=120,
    )

    response.raise_for_status()

    destination.write_bytes(response.content)

    return {
        "final_url": response.url,
        "status_code": response.status_code,
        "content_length": len(response.content),
        "last_modified": response.headers.get("Last-Modified"),
        "etag": response.headers.get("ETag"),
    }


def parse_md5_sidecar(path):
    text = path.read_text(encoding="utf-8").strip()

    match = re.search(r"\b([a-fA-F0-9]{32})\b", text)

    if not match:
        raise ValueError(
            f"Could not locate a valid 32-character MD5 in {path}.\n"
            f"Observed content: {text!r}"
        )

    return match.group(1).lower(), text


def calculate_md5_and_sha256(path, chunk_size=16 * 1024 * 1024):
    md5_digest = hashlib.md5()
    sha256_digest = hashlib.sha256()
    bytes_read = 0

    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)

            if not chunk:
                break

            bytes_read += len(chunk)
            md5_digest.update(chunk)
            sha256_digest.update(chunk)

            if bytes_read % (1024**3) < chunk_size:
                print(
                    f"  Hashed {human_gib(bytes_read):.2f} GiB "
                    f"of {human_gib(path.stat().st_size):.2f} GiB"
                )

    return {
        "bytes_read": bytes_read,
        "md5": md5_digest.hexdigest(),
        "sha256": sha256_digest.hexdigest(),
    }


# --------------------------------------------------------------------------------------------------
# 4. Verify official HTTP metadata
# --------------------------------------------------------------------------------------------------

print("-" * 112)
print("OFFICIAL SOURCE METADATA")
print("-" * 112)

http_metadata = fetch_http_metadata(T1_URL)

print(f"HTTP status: {http_metadata['status_code']}")
print(f"Final URL: {http_metadata['final_url']}")
print(f"HTTP Content-Length: {http_metadata['content_length']}")
print(f"Last-Modified: {http_metadata['last_modified']}")
print(f"ETag: {http_metadata['etag']}")
print(f"Accept-Ranges: {http_metadata['accept_ranges']}")
print(f"Metadata method: {http_metadata['method_used']}")

if (
    http_metadata["content_length"] is not None
    and http_metadata["content_length"] != EXPECTED_SIZE_BYTES
):
    raise AssertionError(
        "Official HTTP size does not match the frozen January 2026 archive size.\n"
        f"Expected: {EXPECTED_SIZE_BYTES:,}\n"
        f"Observed: {http_metadata['content_length']:,}"
    )

print("Official size check: PASS")
print()


# --------------------------------------------------------------------------------------------------
# 5. Download and parse official MD5 sidecar
# --------------------------------------------------------------------------------------------------

print("-" * 112)
print("OFFICIAL MD5 SIDECAR")
print("-" * 112)

md5_http_metadata = download_small_text_file(
    T1_MD5_URL,
    T1_MD5_LOCAL_PATH,
)

official_md5, md5_sidecar_text = parse_md5_sidecar(
    T1_MD5_LOCAL_PATH
)

print(f"MD5 sidecar saved: {T1_MD5_LOCAL_PATH}")
print(f"Official MD5: {official_md5}")
print(f"Raw sidecar content: {md5_sidecar_text}")
print()


# --------------------------------------------------------------------------------------------------
# 6. Check current file state and temporary storage
# --------------------------------------------------------------------------------------------------

existing_size = (
    T1_LOCAL_PATH.stat().st_size
    if T1_LOCAL_PATH.exists()
    else 0
)

if existing_size > EXPECTED_SIZE_BYTES:
    raise RuntimeError(
        "The existing local file is larger than the official archive.\n"
        f"Path: {T1_LOCAL_PATH}\n"
        f"Existing size: {existing_size:,}\n"
        f"Expected size: {EXPECTED_SIZE_BYTES:,}\n\n"
        "The file was not modified or deleted automatically."
    )

disk_usage_before = shutil.disk_usage(TEMP_RAW_DIR)
remaining_download_bytes = max(
    EXPECTED_SIZE_BYTES - existing_size,
    0,
)

required_free_bytes = remaining_download_bytes + RESERVE_BYTES

print("-" * 112)
print("TEMPORARY STORAGE CHECK")
print("-" * 112)
print(f"Existing local bytes: {existing_size:,}")
print(f"Remaining download bytes: {remaining_download_bytes:,}")
print(
    f"Available temporary space: "
    f"{disk_usage_before.free:,} bytes "
    f"({human_gib(disk_usage_before.free):.2f} GiB)"
)
print(
    f"Required free space including reserve: "
    f"{required_free_bytes:,} bytes "
    f"({human_gib(required_free_bytes):.2f} GiB)"
)

if disk_usage_before.free < required_free_bytes:
    raise RuntimeError(
        "Insufficient temporary Colab storage.\n"
        f"Available: {human_gib(disk_usage_before.free):.2f} GiB\n"
        f"Required: {human_gib(required_free_bytes):.2f} GiB\n\n"
        "The T1 XML was not downloaded and no existing file was deleted."
    )

print("Temporary storage check: PASS")
print()


# --------------------------------------------------------------------------------------------------
# 7. Download or resume the T1 XML
# --------------------------------------------------------------------------------------------------

print("-" * 112)
print("T1 RCV XML DOWNLOAD")
print("-" * 112)

download_started_utc = utc_now()

if existing_size == EXPECTED_SIZE_BYTES:
    print("A complete-size local file already exists.")
    print("Skipping network download and proceeding to cryptographic verification.")

else:
    print(
        "Starting resumable download..."
        if existing_size == 0
        else f"Resuming download from byte {existing_size:,}..."
    )

    curl_command = [
        "curl",
        "--fail",
        "--location",
        "--continue-at", "-",
        "--retry", "20",
        "--retry-delay", "10",
        "--retry-max-time", "7200",
        "--connect-timeout", "60",
        "--speed-time", "300",
        "--speed-limit", "1024",
        "--progress-bar",
        "--output", str(T1_LOCAL_PATH),
        T1_URL,
    ]

    result = subprocess.run(curl_command)

    if result.returncode != 0:
        partial_size = (
            T1_LOCAL_PATH.stat().st_size
            if T1_LOCAL_PATH.exists()
            else 0
        )

        raise RuntimeError(
            "The download command did not complete successfully.\n"
            f"curl return code: {result.returncode}\n"
            f"Retained partial file size: {partial_size:,} bytes\n\n"
            "The partial file has been preserved so the same cell can resume it."
        )

download_completed_utc = utc_now()

observed_size = T1_LOCAL_PATH.stat().st_size

print()
print(f"Downloaded file: {T1_LOCAL_PATH}")
print(f"Observed size: {observed_size:,} bytes")

if observed_size != EXPECTED_SIZE_BYTES:
    raise AssertionError(
        "Downloaded file size does not match the official archive size.\n"
        f"Expected: {EXPECTED_SIZE_BYTES:,}\n"
        f"Observed: {observed_size:,}\n\n"
        "The file was retained for investigation and was not deleted automatically."
    )

print("Exact byte-size verification: PASS")
print()


# --------------------------------------------------------------------------------------------------
# 8. Calculate MD5 and SHA-256
# --------------------------------------------------------------------------------------------------

print("-" * 112)
print("CRYPTOGRAPHIC HASH VERIFICATION")
print("-" * 112)

hash_started_utc = utc_now()

hash_results = calculate_md5_and_sha256(T1_LOCAL_PATH)

hash_completed_utc = utc_now()

print()
print(f"Observed MD5: {hash_results['md5']}")
print(f"Official MD5: {official_md5}")
print(f"Observed SHA-256: {hash_results['sha256']}")

if hash_results["bytes_read"] != EXPECTED_SIZE_BYTES:
    raise AssertionError(
        "The hashing pass did not read the expected number of bytes."
    )

if hash_results["md5"].lower() != official_md5.lower():
    raise AssertionError(
        "MD5 VERIFICATION FAILED.\n"
        f"Official MD5: {official_md5}\n"
        f"Observed MD5: {hash_results['md5']}\n\n"
        "The file was retained and was not deleted automatically."
    )

print("Official MD5 verification: PASS")
print("SHA-256 calculation: PASS")
print()


# --------------------------------------------------------------------------------------------------
# 9. Complete gzip-stream integrity test
# --------------------------------------------------------------------------------------------------

print("-" * 112)
print("COMPLETE GZIP INTEGRITY TEST")
print("-" * 112)
print(
    "This reads and decompresses the complete stream without writing the "
    "decompressed XML to disk. It may take several minutes."
)

gzip_test_started_utc = utc_now()

gzip_result = subprocess.run(
    ["gzip", "-t", str(T1_LOCAL_PATH)],
    capture_output=True,
    text=True,
)

gzip_test_completed_utc = utc_now()

if gzip_result.returncode != 0:
    raise RuntimeError(
        "GZIP INTEGRITY TEST FAILED.\n"
        f"Return code: {gzip_result.returncode}\n"
        f"stderr: {gzip_result.stderr.strip()}\n\n"
        "The file was retained and was not deleted automatically."
    )

print("Complete gzip-stream integrity: PASS")
print()


# --------------------------------------------------------------------------------------------------
# 10. Create persistent download-verification receipt
# --------------------------------------------------------------------------------------------------

disk_usage_after = shutil.disk_usage(TEMP_RAW_DIR)

receipt = {
    "receipt_name": "clinvar_t1_rcv_xml_download_receipt_v1",
    "receipt_version": "1.0.0",
    "created_utc": utc_now(),

    "study": {
        "experiment": "Experiment 1 — Temporal Validation of GES",
        "timepoint": "T1",
        "release_label": "2026-01",
        "primary_unit": "RCV-level variant-condition aggregate",
        "target_genes": ["BRCA1", "BRCA2", "MLH1", "EGFR"],
    },

    "t0_prerequisite": {
        "manifest_path": str(T0_FREEZE_MANIFEST),
        "administrative_decision": t0_decision,
        "passed": True,
    },

    "official_source": {
        "filename": T1_FILENAME,
        "file_url": T1_URL,
        "md5_url": T1_MD5_URL,
        "expected_size_bytes": EXPECTED_SIZE_BYTES,
        "official_md5": official_md5,
        "md5_sidecar_content": md5_sidecar_text,
        "http_metadata": http_metadata,
        "md5_http_metadata": md5_http_metadata,
    },

    "local_artifact": {
        "path": str(T1_LOCAL_PATH),
        "storage_type": "temporary_colab_storage",
        "persistent_copy_created": False,
        "observed_size_bytes": observed_size,
        "observed_md5": hash_results["md5"],
        "observed_sha256": hash_results["sha256"],
    },

    "verification_results": {
        "exact_size_passed": observed_size == EXPECTED_SIZE_BYTES,
        "official_md5_passed":
            hash_results["md5"].lower() == official_md5.lower(),
        "sha256_calculated": True,
        "gzip_integrity_passed": gzip_result.returncode == 0,
        "overall_decision": "PASS",
    },

    "runtime": {
        "download_started_utc": download_started_utc,
        "download_completed_utc": download_completed_utc,
        "hash_started_utc": hash_started_utc,
        "hash_completed_utc": hash_completed_utc,
        "gzip_test_started_utc": gzip_test_started_utc,
        "gzip_test_completed_utc": gzip_test_completed_utc,
        "temporary_free_bytes_before": disk_usage_before.free,
        "temporary_free_bytes_after": disk_usage_after.free,
    },

    "temporal_boundary": {
        "embedded_t1_data_cutoff": None,
        "status": "PENDING_XML_ROOT_INSPECTION",
        "warning":
            "The release label, HTTP metadata, and file-server timestamps "
            "must not be substituted for the XML embedded data cutoff.",
    },

    "leakage_controls": {
        "t1_used_to_modify_t0_artifact": False,
        "t1_used_to_fit_ges": False,
        "t1_temporal_outcomes_constructed": False,
        "t0_t1_crosswalk_constructed": False,
    },

    "next_action": {
        "stage": "Stage 2C",
        "description":
            "Inspect the T1 XML root and current schema, determine the embedded "
            "data cutoff, probe representative target-gene records, and create "
            "a schema/provenance manifest before full target-gene extraction.",
    },
}

BASE_RECEIPT_PATH = (
    CONFIG_DIR /
    "clinvar_t1_rcv_xml_download_receipt_v1.json"
)

if not BASE_RECEIPT_PATH.exists():
    RECEIPT_PATH = BASE_RECEIPT_PATH
else:
    timestamp_for_filename = (
        datetime.now(timezone.utc)
        .strftime("%Y%m%dT%H%M%SZ")
    )

    RECEIPT_PATH = (
        CONFIG_DIR /
        f"clinvar_t1_rcv_xml_download_receipt_v1_rerun_"
        f"{timestamp_for_filename}.json"
    )

atomic_write_json(receipt, RECEIPT_PATH)

receipt_sha256 = hashlib.sha256(
    RECEIPT_PATH.read_bytes()
).hexdigest()

RECEIPT_SHA256_PATH = RECEIPT_PATH.with_suffix(
    RECEIPT_PATH.suffix + ".sha256"
)

RECEIPT_SHA256_PATH.write_text(
    f"{receipt_sha256}  {RECEIPT_PATH.name}\n",
    encoding="utf-8",
)

with RECEIPT_PATH.open("r", encoding="utf-8") as handle:
    receipt_readback = json.load(handle)

assert receipt_readback == receipt
assert hashlib.sha256(RECEIPT_PATH.read_bytes()).hexdigest() == receipt_sha256


# --------------------------------------------------------------------------------------------------
# 11. Final result
# --------------------------------------------------------------------------------------------------

print("=" * 112)
print("STAGE 2B RESULT — PASS")
print("=" * 112)
print(f"T1 file: {T1_LOCAL_PATH}")
print(f"Verified size: {observed_size:,} bytes")
print(f"Verified MD5: {hash_results['md5']}")
print(f"Calculated SHA-256: {hash_results['sha256']}")
print("Gzip integrity: PASS")
print()
print(f"Persistent receipt: {RECEIPT_PATH}")
print(f"Receipt SHA-256: {receipt_sha256}")
print(f"Receipt sidecar: {RECEIPT_SHA256_PATH}")
print()
print("T1 embedded cutoff: PENDING XML ROOT INSPECTION")
print("T1 target-gene extraction: NOT YET STARTED")
print("Stage 3 T0–T1 crosswalk: NOT YET AUTHORIZED")
print()
print("NEXT REQUIRED ACTION:")
print(
    "Run the T1 root/schema probe and inspect representative "
    "BRCA1, BRCA2, MLH1, and EGFR records."
)
print("=" * 112)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
STAGE 2B — T1 RCV XML DOWNLOAD AND SOURCE VERIFICATION
T1 release label: January 2026
Temporary destination: /content/clinvar_rcv_raw/ClinVarRCVRelease_2026-01.xml.gz
Expected compressed size: 5,434,707,247 bytes

T0 prerequisite: PASS — ACCEPTED_AND_FROZEN

----------------------------------------------------------------------------------------------------------------
OFFICIAL SOURCE METADATA
----------------------------------------------------------------------------------------------------------------
HTTP status: 200
Final URL: https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/RCV_release/ClinVarRCVRelease_2026-01.xml.gz
HTTP Content-Length: 5434707247
Last-Modified: Thu, 26 Mar 2026 19:43:29 GMT
ETag: None
Accept-Ranges: bytes
Metadata method: HEAD
Official size check: PASS

-------------------------------------------------------------------------------------

In [9]:
# ==================================================================================================
# STAGE 2C — T1 ROOT, SCHEMA, AND REPRESENTATIVE TARGET-GENE RECORD PROBE
# GES-RAG Temporal Validation Study
#
# Purpose:
#   1. Confirm the verified January 2026 T1 XML and receipt are present.
#   2. Inspect the XML root, namespaces, attributes, schema declaration, and embedded date.
#   3. Stream through top-level RCV records without loading the full XML into memory.
#   4. Locate one representative RCV for each target gene:
#         BRCA1, BRCA2, MLH1, and EGFR
#   5. Profile identifier, condition, aggregate-classification, SCV, submitter,
#      and schema-path availability.
#   6. Save a versioned T1 schema/provenance probe manifest and compressed sample records.
#
# This cell does NOT:
#   - extract the complete T1 target-gene cohort;
#   - construct the T0–T1 crosswalk;
#   - create future-instability outcomes;
#   - fit or modify GES;
#   - delete the T1 XML.
# ==================================================================================================

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import sys
import json
import gzip
import hashlib
import subprocess
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter, defaultdict


# --------------------------------------------------------------------------------------------------
# 1. Ensure lxml is available
# --------------------------------------------------------------------------------------------------

try:
    from lxml import etree
except ImportError:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "lxml"],
        check=True
    )
    from lxml import etree


# --------------------------------------------------------------------------------------------------
# 2. Paths
# --------------------------------------------------------------------------------------------------

DRIVE_ROOT = Path("/content/drive/MyDrive")
STUDY_ROOT = DRIVE_ROOT / "GES_RAG_Temporal_Study"

CONFIG_DIR = STUDY_ROOT / "configs"
QUALITY_DIR = STUDY_ROOT / "outputs" / "quality_checks"

T1_XML_PATH = Path(
    "/content/clinvar_rcv_raw/ClinVarRCVRelease_2026-01.xml.gz"
)

T1_RECEIPT_PATH = (
    CONFIG_DIR /
    "clinvar_t1_rcv_xml_download_receipt_v1.json"
)

T1_RECEIPT_SHA_PATH = T1_RECEIPT_PATH.with_suffix(
    T1_RECEIPT_PATH.suffix + ".sha256"
)

EXPECTED_T1_SIZE = 5_434_707_247
EXPECTED_T1_MD5 = "5740de7f8f74a49ba8c58e3ec1b8cc26"
EXPECTED_T1_SHA256 = (
    "3fba206f1e3086306472ab7b0ae324d4"
    "ae516da84857038d03f2937fd20a6e55"
)

TARGET_GENES = ("BRCA1", "BRCA2", "MLH1", "EGFR")

CONFIG_DIR.mkdir(parents=True, exist_ok=True)
QUALITY_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 112)
print("STAGE 2C — T1 ROOT, SCHEMA, AND REPRESENTATIVE RECORD PROBE")
print("=" * 112)
print(f"T1 source: {T1_XML_PATH}")
print(f"Target genes: {', '.join(TARGET_GENES)}")
print()


# --------------------------------------------------------------------------------------------------
# 3. General helper functions
# --------------------------------------------------------------------------------------------------

def utc_now():
    return datetime.now(timezone.utc).replace(microsecond=0).isoformat()


def sha256_file(path: Path, chunk_size: int = 8 * 1024 * 1024):
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def atomic_write_json(payload, destination: Path):
    temporary_path = destination.with_suffix(
        destination.suffix + ".tmp"
    )

    with temporary_path.open("w", encoding="utf-8") as handle:
        json.dump(
            payload,
            handle,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            allow_nan=False,
        )
        handle.write("\n")

    temporary_path.replace(destination)


def local_name(tag):
    """Return the local XML element/attribute name without its namespace."""
    if not isinstance(tag, str):
        return None

    try:
        return etree.QName(tag).localname
    except Exception:
        return tag


def namespace_uri(tag):
    if not isinstance(tag, str) or not tag.startswith("{"):
        return None

    return tag[1:].split("}", 1)[0]


def clean_text(value):
    if value is None:
        return None

    normalized = " ".join(str(value).split())
    return normalized if normalized else None


def unique_preserve_order(values):
    seen = set()
    result = []

    for value in values:
        normalized = clean_text(value)

        if normalized is None or normalized in seen:
            continue

        seen.add(normalized)
        result.append(normalized)

    return result


def descendants_by_local(element, wanted_name):
    for node in element.iter():
        if local_name(node.tag) == wanted_name:
            yield node


def first_descendant_by_local(element, wanted_name):
    return next(descendants_by_local(element, wanted_name), None)


def direct_children_by_local(element, wanted_name):
    for child in element:
        if local_name(child.tag) == wanted_name:
            yield child


def first_direct_child_by_local(element, wanted_name):
    return next(direct_children_by_local(element, wanted_name), None)


def first_text_by_local(element, wanted_name):
    node = first_descendant_by_local(element, wanted_name)

    if node is None:
        return None

    return clean_text(node.text)


def normalized_attributes(element):
    """Represent namespaced attributes in readable local-name form."""
    result = {}

    for key, value in element.attrib.items():
        local = local_name(key)
        namespace = namespace_uri(key)

        display_key = (
            f"{{{namespace}}}{local}"
            if namespace
            else local
        )

        result[display_key] = value

    return result


def first_attr(element, candidate_names):
    if element is None:
        return None

    lowered = {
        str(name).lower()
        for name in candidate_names
    }

    for key, value in element.attrib.items():
        if local_name(key).lower() in lowered:
            return clean_text(value)

    return None


def deterministic_gzip_write(payload: bytes, destination: Path):
    """
    Save compressed sample XML deterministically by setting gzip mtime to zero.
    """
    temporary_path = destination.with_suffix(
        destination.suffix + ".tmp"
    )

    with temporary_path.open("wb") as raw_handle:
        with gzip.GzipFile(
            filename="",
            mode="wb",
            fileobj=raw_handle,
            compresslevel=9,
            mtime=0,
        ) as gzip_handle:
            gzip_handle.write(payload)

    temporary_path.replace(destination)


def parse_date_candidate(value):
    """
    Parse the first YYYY-MM-DD portion of an XML date-like value.
    The exact observed value is retained separately.
    """
    if not value:
        return None

    candidate = str(value).strip()[:10]

    try:
        return datetime.strptime(candidate, "%Y-%m-%d").date().isoformat()
    except ValueError:
        return None


# --------------------------------------------------------------------------------------------------
# 4. Verify receipt and temporary raw source
# --------------------------------------------------------------------------------------------------

print("-" * 112)
print("T1 SOURCE AND RECEIPT PREREQUISITES")
print("-" * 112)

if not T1_XML_PATH.exists():
    raise FileNotFoundError(
        "The verified T1 XML is not present in temporary Colab storage:\n"
        f"{T1_XML_PATH}\n\n"
        "The Colab runtime may have restarted. Rerun Stage 2B to restore it."
    )

if not T1_RECEIPT_PATH.exists():
    raise FileNotFoundError(
        f"T1 verification receipt not found: {T1_RECEIPT_PATH}"
    )

if not T1_RECEIPT_SHA_PATH.exists():
    raise FileNotFoundError(
        f"T1 receipt SHA-256 sidecar not found: {T1_RECEIPT_SHA_PATH}"
    )

observed_receipt_sha256 = sha256_file(T1_RECEIPT_PATH)
sidecar_content = T1_RECEIPT_SHA_PATH.read_text(
    encoding="utf-8"
).strip()

expected_sidecar_content = (
    f"{observed_receipt_sha256}  {T1_RECEIPT_PATH.name}"
)

if sidecar_content != expected_sidecar_content:
    raise AssertionError(
        "The T1 receipt SHA-256 sidecar does not match the receipt."
    )

with T1_RECEIPT_PATH.open("r", encoding="utf-8") as handle:
    t1_receipt = json.load(handle)

receipt_decision = (
    t1_receipt
    .get("verification_results", {})
    .get("overall_decision")
)

receipt_size = (
    t1_receipt
    .get("local_artifact", {})
    .get("observed_size_bytes")
)

receipt_md5 = (
    t1_receipt
    .get("local_artifact", {})
    .get("observed_md5")
)

receipt_sha256 = (
    t1_receipt
    .get("local_artifact", {})
    .get("observed_sha256")
)

observed_file_size = T1_XML_PATH.stat().st_size

assert receipt_decision == "PASS"
assert receipt_size == EXPECTED_T1_SIZE
assert receipt_md5.lower() == EXPECTED_T1_MD5
assert receipt_sha256.lower() == EXPECTED_T1_SHA256
assert observed_file_size == EXPECTED_T1_SIZE

with T1_XML_PATH.open("rb") as handle:
    gzip_magic = handle.read(2)

assert gzip_magic == b"\x1f\x8b", (
    "The T1 file does not begin with the expected gzip signature."
)

print("T1 download receipt: PASS")
print("T1 receipt SHA-256 sidecar: PASS")
print(f"T1 compressed size: {observed_file_size:,} bytes")
print(f"Previously verified MD5: {receipt_md5}")
print(f"Previously calculated SHA-256: {receipt_sha256}")
print("Gzip signature: PASS")
print()


# --------------------------------------------------------------------------------------------------
# 5. Create version-preserving output locations
# --------------------------------------------------------------------------------------------------

BASE_MANIFEST_PATH = (
    CONFIG_DIR /
    "clinvar_t1_rcv_xml_schema_probe_manifest_v1.json"
)

timestamp_token = datetime.now(timezone.utc).strftime(
    "%Y%m%dT%H%M%SZ"
)

if BASE_MANIFEST_PATH.exists():
    MANIFEST_PATH = (
        CONFIG_DIR /
        f"clinvar_t1_rcv_xml_schema_probe_manifest_v1_"
        f"rerun_{timestamp_token}.json"
    )

    SAMPLE_DIR = (
        QUALITY_DIR /
        f"t1_rcv_schema_probe_samples_v1_rerun_{timestamp_token}"
    )
else:
    MANIFEST_PATH = BASE_MANIFEST_PATH
    SAMPLE_DIR = (
        QUALITY_DIR /
        "t1_rcv_schema_probe_samples_v1"
    )

SAMPLE_DIR.mkdir(parents=True, exist_ok=False)

print(f"Probe manifest destination: {MANIFEST_PATH}")
print(f"Compressed sample directory: {SAMPLE_DIR}")
print()


# --------------------------------------------------------------------------------------------------
# 6. Gene-symbol extraction
# --------------------------------------------------------------------------------------------------

def extract_structured_gene_symbols(record):
    """
    Extract gene symbols from structured Symbol/GeneSymbol elements.

    The current and historical ClinVar XML formats may represent the gene
    symbol through different element arrangements, so the probe records all
    structured symbol values before selecting exact target-gene matches.
    """
    symbols = []

    for node in record.iter():
        node_name = local_name(node.tag)

        if node_name == "GeneSymbol":
            value = clean_text(node.text)

            if value:
                symbols.append(value.upper())

        elif node_name == "Symbol":
            direct_value = clean_text(node.text)

            if direct_value:
                symbols.append(direct_value.upper())

            for descendant in node.iter():
                if local_name(descendant.tag) == "ElementValue":
                    value = clean_text(descendant.text)

                    if value:
                        symbols.append(value.upper())

    return unique_preserve_order(symbols)


# --------------------------------------------------------------------------------------------------
# 7. RCV, VCV, condition, classification, SCV, and submitter probes
# --------------------------------------------------------------------------------------------------

def extract_reference_assertion(record):
    return first_descendant_by_local(
        record,
        "ReferenceClinVarAssertion"
    )


def extract_rcv_identifier(reference_assertion):
    candidates = list(
        descendants_by_local(
            reference_assertion,
            "ClinVarAccession"
        )
    )

    for candidate in candidates:
        accession_type = first_attr(candidate, ["Type"])
        accession = first_attr(
            candidate,
            ["Acc", "Accession"]
        )

        if (
            accession
            and (
                accession.upper().startswith("RCV")
                or (
                    accession_type
                    and accession_type.upper() == "RCV"
                )
            )
        ):
            return {
                "accession": accession.upper(),
                "version": first_attr(candidate, ["Version"]),
                "attributes": normalized_attributes(candidate),
            }

    return {
        "accession": None,
        "version": None,
        "attributes": {},
    }


def extract_measure_set(reference_assertion):
    measure_set = first_descendant_by_local(
        reference_assertion,
        "MeasureSet"
    )

    if measure_set is None:
        return {
            "variation_id": None,
            "vcv_accession": None,
            "vcv_version": None,
            "measure_set_type": None,
            "attributes": {},
        }

    accession = first_attr(
        measure_set,
        ["Acc", "Accession"]
    )

    return {
        "variation_id": first_attr(
            measure_set,
            ["ID", "VariationID"]
        ),
        "vcv_accession": (
            accession.upper()
            if accession and accession.upper().startswith("VCV")
            else accession
        ),
        "vcv_version": first_attr(
            measure_set,
            ["Version"]
        ),
        "measure_set_type": first_attr(
            measure_set,
            ["Type"]
        ),
        "attributes": normalized_attributes(measure_set),
    }


def extract_condition_information(reference_assertion):
    condition_names = []
    condition_identifiers = []
    trait_count = 0

    for trait in descendants_by_local(
        reference_assertion,
        "Trait"
    ):
        trait_count += 1

        for name_node in descendants_by_local(trait, "Name"):
            element_values = list(
                descendants_by_local(
                    name_node,
                    "ElementValue"
                )
            )

            preferred_values = [
                clean_text(node.text)
                for node in element_values
                if (
                    first_attr(node, ["Type"])
                    and first_attr(node, ["Type"]).lower()
                    == "preferred"
                )
            ]

            all_values = [
                clean_text(node.text)
                for node in element_values
            ]

            condition_names.extend(
                preferred_values or all_values
            )

        for xref in descendants_by_local(trait, "XRef"):
            database = first_attr(
                xref,
                ["DB", "Database"]
            )

            identifier = first_attr(
                xref,
                ["ID", "Identifier"]
            )

            if database and identifier:
                condition_identifiers.append(
                    f"{database}:{identifier}"
                )

    return {
        "trait_count": trait_count,
        "condition_names": unique_preserve_order(
            condition_names
        ),
        "condition_identifiers": unique_preserve_order(
            condition_identifiers
        ),
    }


def summarize_classification_node(node):
    date_value = (
        first_attr(
            node,
            [
                "DateLastEvaluated",
                "LastEvaluated",
                "DateEvaluated",
            ]
        )
        or first_text_by_local(node, "DateLastEvaluated")
        or first_text_by_local(node, "LastEvaluated")
    )

    return {
        "classification_type": local_name(node.tag),
        "description": (
            first_text_by_local(node, "Description")
            or first_text_by_local(
                node,
                "ClinicalSignificance"
            )
        ),
        "review_status": first_text_by_local(
            node,
            "ReviewStatus"
        ),
        "date_last_evaluated": date_value,
        "explanation": first_text_by_local(
            node,
            "Explanation"
        ),
        "attributes": normalized_attributes(node),
    }


def extract_aggregate_classifications(reference_assertion):
    results = []

    classifications_container = (
        first_direct_child_by_local(
            reference_assertion,
            "Classifications"
        )
        or first_descendant_by_local(
            reference_assertion,
            "Classifications"
        )
    )

    if classifications_container is not None:
        for child in classifications_container:
            if not isinstance(child.tag, str):
                continue

            results.append(
                summarize_classification_node(child)
            )

    if not results:
        historical_node = (
            first_direct_child_by_local(
                reference_assertion,
                "ClinicalSignificance"
            )
            or first_descendant_by_local(
                reference_assertion,
                "ClinicalSignificance"
            )
        )

        if historical_node is not None:
            results.append(
                summarize_classification_node(
                    historical_node
                )
            )

    return results


def extract_scv_information(record):
    scv_assertions = list(
        descendants_by_local(
            record,
            "ClinVarAssertion"
        )
    )

    scv_samples = []
    submitter_names = []
    organization_ids = []

    for scv_assertion in scv_assertions:
        accession_node = None

        for candidate in descendants_by_local(
            scv_assertion,
            "ClinVarAccession"
        ):
            accession = first_attr(
                candidate,
                ["Acc", "Accession"]
            )

            accession_type = first_attr(
                candidate,
                ["Type"]
            )

            if (
                accession
                and (
                    accession.upper().startswith("SCV")
                    or (
                        accession_type
                        and accession_type.upper() == "SCV"
                    )
                )
            ):
                accession_node = candidate
                break

        submission_node = first_descendant_by_local(
            scv_assertion,
            "ClinVarSubmissionID"
        )

        scv_accession = (
            first_attr(
                accession_node,
                ["Acc", "Accession"]
            )
            if accession_node is not None
            else None
        )

        scv_version = (
            first_attr(accession_node, ["Version"])
            if accession_node is not None
            else None
        )

        org_id = (
            first_attr(
                accession_node,
                ["OrgID", "OrganizationID"]
            )
            if accession_node is not None
            else None
        )

        submitter_name = (
            first_attr(
                submission_node,
                ["submitter", "Submitter"]
            )
            if submission_node is not None
            else None
        )

        if submitter_name:
            submitter_names.append(submitter_name)

        if org_id:
            organization_ids.append(org_id)

        if len(scv_samples) < 5:
            classification_node = (
                first_descendant_by_local(
                    scv_assertion,
                    "Classification"
                )
                or first_descendant_by_local(
                    scv_assertion,
                    "ClinicalSignificance"
                )
            )

            scv_samples.append(
                {
                    "scv_accession": (
                        scv_accession.upper()
                        if scv_accession
                        else None
                    ),
                    "scv_version": scv_version,
                    "submitter_name": submitter_name,
                    "primary_orgid": org_id,
                    "classification_container": (
                        local_name(classification_node.tag)
                        if classification_node is not None
                        else None
                    ),
                    "classification_description": (
                        first_text_by_local(
                            classification_node,
                            "Description"
                        )
                        if classification_node is not None
                        else None
                    ),
                    "review_status": (
                        first_text_by_local(
                            classification_node,
                            "ReviewStatus"
                        )
                        if classification_node is not None
                        else None
                    ),
                }
            )

    return {
        "scv_assertion_count": len(scv_assertions),
        "unique_submitter_names": unique_preserve_order(
            submitter_names
        ),
        "unique_primary_orgids": unique_preserve_order(
            organization_ids
        ),
        "first_five_scv_records": scv_samples,
    }


# --------------------------------------------------------------------------------------------------
# 8. Schema fingerprint for representative records
# --------------------------------------------------------------------------------------------------

def local_element_path(node, record_root):
    components = []
    current = node

    while current is not None:
        name = local_name(current.tag)

        if name:
            components.append(name)

        if current is record_root:
            break

        current = current.getparent()

    return "/" + "/".join(reversed(components))


def create_schema_fingerprint(record):
    tag_counts = Counter()
    unique_paths = set()
    attributes_by_path = defaultdict(set)

    for node in record.iter():
        node_name = local_name(node.tag)

        if not node_name:
            continue

        tag_counts[node_name] += 1

        path = local_element_path(
            node,
            record
        )

        unique_paths.add(path)

        for attribute_name in node.attrib:
            attributes_by_path[path].add(
                local_name(attribute_name)
            )

    return {
        "unique_element_paths": sorted(unique_paths),
        "tag_counts": dict(
            sorted(tag_counts.items())
        ),
        "attributes_by_path": {
            path: sorted(attributes)
            for path, attributes
            in sorted(attributes_by_path.items())
        },
    }


# --------------------------------------------------------------------------------------------------
# 9. Summarize and save a representative record
# --------------------------------------------------------------------------------------------------

def summarize_record(record, source_record_index):
    reference_assertion = extract_reference_assertion(record)

    if reference_assertion is None:
        raise ValueError(
            "Top-level record did not contain ReferenceClinVarAssertion."
        )

    gene_symbols = extract_structured_gene_symbols(record)
    rcv = extract_rcv_identifier(reference_assertion)
    measure_set = extract_measure_set(reference_assertion)

    condition_information = extract_condition_information(
        reference_assertion
    )

    aggregate_classifications = (
        extract_aggregate_classifications(
            reference_assertion
        )
    )

    scv_information = extract_scv_information(record)

    return {
        "source_record_index": source_record_index,
        "record_element_name": local_name(record.tag),
        "record_namespace": namespace_uri(record.tag),
        "record_attributes": normalized_attributes(record),
        "structured_gene_symbols": gene_symbols,
        "target_genes_in_record": [
            gene
            for gene in TARGET_GENES
            if gene in gene_symbols
        ],
        "rcv": rcv,
        "measure_set": measure_set,
        "condition_information": condition_information,
        "aggregate_classifications": aggregate_classifications,
        "scv_information": scv_information,
        "field_availability": {
            "rcv_accession": bool(rcv["accession"]),
            "rcv_version": bool(rcv["version"]),
            "variation_id": bool(
                measure_set["variation_id"]
            ),
            "vcv_accession": bool(
                measure_set["vcv_accession"]
            ),
            "vcv_version": bool(
                measure_set["vcv_version"]
            ),
            "structured_gene_symbol": bool(
                gene_symbols
            ),
            "condition_name": bool(
                condition_information[
                    "condition_names"
                ]
            ),
            "condition_identifier": bool(
                condition_information[
                    "condition_identifiers"
                ]
            ),
            "aggregate_classification": bool(
                aggregate_classifications
            ),
            "aggregate_review_status": any(
                item.get("review_status")
                for item in aggregate_classifications
            ),
            "aggregate_last_evaluated": any(
                item.get("date_last_evaluated")
                for item in aggregate_classifications
            ),
            "scv_assertion": (
                scv_information[
                    "scv_assertion_count"
                ] > 0
            ),
            "scv_accession": any(
                item.get("scv_accession")
                for item in scv_information[
                    "first_five_scv_records"
                ]
            ),
            "primary_submitter_name": bool(
                scv_information[
                    "unique_submitter_names"
                ]
            ),
            "primary_submitter_orgid": bool(
                scv_information[
                    "unique_primary_orgids"
                ]
            ),
        },
        "schema_fingerprint": create_schema_fingerprint(
            record
        ),
    }


def save_compressed_record_sample(
    record,
    gene,
    source_record_index,
    rcv_accession,
):
    xml_bytes = etree.tostring(
        record,
        encoding="UTF-8",
        xml_declaration=True,
        pretty_print=False,
    )

    safe_rcv = rcv_accession or "RCV_UNKNOWN"

    destination = (
        SAMPLE_DIR /
        f"{gene}_record_{source_record_index}_"
        f"{safe_rcv}.xml.gz"
    )

    deterministic_gzip_write(
        xml_bytes,
        destination
    )

    return {
        "path": str(destination),
        "filename": destination.name,
        "serialized_uncompressed_bytes": len(xml_bytes),
        "compressed_bytes": destination.stat().st_size,
        "compressed_sha256": sha256_file(destination),
        "note":
            "This is a reserialized representative XML element, "
            "not a byte-range copy of the compressed source archive.",
    }


# --------------------------------------------------------------------------------------------------
# 10. Stream the T1 XML and locate one record for each target gene
# --------------------------------------------------------------------------------------------------

print("-" * 112)
print("STREAMING T1 ROOT AND TARGET-GENE SCHEMA PROBE")
print("-" * 112)

probe_started_utc = utc_now()

root = None
root_metadata = None

top_level_child_tags = Counter()
nonrecord_top_level_tags = Counter()

top_level_children_seen = 0
rcv_records_scanned = 0

representative_records = {}
representative_sample_files = {}

with gzip.open(T1_XML_PATH, "rb") as compressed_stream:
    context = etree.iterparse(
        compressed_stream,
        events=("start", "end"),
        huge_tree=True,
        recover=False,
        remove_blank_text=False,
    )

    for event, element in context:

        # Capture root immediately.
        if event == "start" and root is None:
            root = element

            root_attributes = normalized_attributes(root)

            date_candidates = {
                local_name(key): value
                for key, value in root.attrib.items()
                if local_name(key).lower()
                in {
                    "dated",
                    "date",
                    "releasedate",
                    "released",
                    "generated",
                }
            }

            schema_candidates = {
                local_name(key): value
                for key, value in root.attrib.items()
                if "schema" in local_name(key).lower()
            }

            root_metadata = {
                "element_name": local_name(root.tag),
                "namespace": namespace_uri(root.tag),
                "attributes": root_attributes,
                "namespace_map": {
                    str(prefix) if prefix is not None else "default":
                        uri
                    for prefix, uri in (root.nsmap or {}).items()
                },
                "date_attribute_candidates": date_candidates,
                "schema_attribute_candidates": schema_candidates,
            }

            print(f"Root element: {root_metadata['element_name']}")
            print(f"Root namespace: {root_metadata['namespace']}")
            print(f"Root attributes: {root_metadata['attributes']}")
            print(f"Namespace map: {root_metadata['namespace_map']}")
            print()

            continue

        # Process only completed direct children of the XML root.
        if (
            event == "end"
            and root is not None
            and element.getparent() is root
        ):
            top_level_children_seen += 1

            element_name = (
                local_name(element.tag)
                or "NON_ELEMENT_NODE"
            )

            top_level_child_tags[element_name] += 1

            reference_assertion = (
                first_descendant_by_local(
                    element,
                    "ReferenceClinVarAssertion"
                )
            )

            if reference_assertion is None:
                nonrecord_top_level_tags[element_name] += 1

            else:
                rcv_records_scanned += 1

                structured_genes = (
                    extract_structured_gene_symbols(
                        element
                    )
                )

                target_matches = [
                    gene
                    for gene in TARGET_GENES
                    if gene in structured_genes
                ]

                new_matches = [
                    gene
                    for gene in target_matches
                    if gene not in representative_records
                ]

                if new_matches:
                    record_summary = summarize_record(
                        element,
                        rcv_records_scanned
                    )

                    rcv_accession = (
                        record_summary
                        .get("rcv", {})
                        .get("accession")
                    )

                    for gene in new_matches:
                        representative_records[gene] = (
                            record_summary
                        )

                        representative_sample_files[gene] = (
                            save_compressed_record_sample(
                                element,
                                gene,
                                rcv_records_scanned,
                                rcv_accession,
                            )
                        )

                        print(
                            f"FOUND {gene}: "
                            f"record {rcv_records_scanned:,}, "
                            f"RCV={rcv_accession}, "
                            f"VCV="
                            f"{record_summary['measure_set']['vcv_accession']}, "
                            f"VariationID="
                            f"{record_summary['measure_set']['variation_id']}, "
                            f"SCVs="
                            f"{record_summary['scv_information']['scv_assertion_count']:,}"
                        )

                if rcv_records_scanned % 100_000 == 0:
                    print(
                        f"Scanned {rcv_records_scanned:,} RCV records; "
                        f"found {len(representative_records)}/"
                        f"{len(TARGET_GENES)} target genes: "
                        f"{sorted(representative_records)}"
                    )

            # Release processed XML memory.
            element.clear()

            parent = element.getparent()

            if parent is not None:
                while element.getprevious() is not None:
                    del parent[0]

            if len(representative_records) == len(TARGET_GENES):
                break

    del context

probe_completed_utc = utc_now()

print()


# --------------------------------------------------------------------------------------------------
# 11. Validate root date and probe completeness
# --------------------------------------------------------------------------------------------------

if root_metadata is None:
    raise RuntimeError("No XML root element was detected.")

date_candidates = root_metadata[
    "date_attribute_candidates"
]

parsed_date_candidates = {
    key: parse_date_candidate(value)
    for key, value in date_candidates.items()
}

valid_parsed_dates = {
    key: value
    for key, value in parsed_date_candidates.items()
    if value is not None
}

if not valid_parsed_dates:
    raise AssertionError(
        "No parseable embedded date was found among the XML root attributes.\n"
        f"Observed date candidates: {date_candidates}"
    )

missing_target_genes = sorted(
    set(TARGET_GENES) -
    set(representative_records)
)

if missing_target_genes:
    raise AssertionError(
        "The probe reached the available stream without locating all "
        f"target genes. Missing: {missing_target_genes}"
    )

print("-" * 112)
print("ROOT AND REPRESENTATIVE RECORD RESULTS")
print("-" * 112)
print(f"Root element: {root_metadata['element_name']}")
print(f"Root attributes: {root_metadata['attributes']}")
print(f"Embedded date candidates: {date_candidates}")
print(f"Parsed embedded dates: {valid_parsed_dates}")
print(f"Schema candidates: {root_metadata['schema_attribute_candidates']}")
print(f"Top-level RCV records scanned: {rcv_records_scanned:,}")
print(f"Representative genes found: {sorted(representative_records)}")
print()


# --------------------------------------------------------------------------------------------------
# 12. Build a cross-gene field-availability matrix
# --------------------------------------------------------------------------------------------------

availability_field_names = sorted({
    field_name
    for summary in representative_records.values()
    for field_name in summary[
        "field_availability"
    ]
})

field_availability_matrix = {
    field_name: {
        gene: bool(
            representative_records[gene][
                "field_availability"
            ].get(field_name)
        )
        for gene in TARGET_GENES
    }
    for field_name in availability_field_names
}

print("-" * 112)
print("REPRESENTATIVE FIELD-AVAILABILITY MATRIX")
print("-" * 112)

header = (
    f"{'Field':38}"
    + "".join(
        f"{gene:>10}"
        for gene in TARGET_GENES
    )
)

print(header)
print("-" * len(header))

for field_name, values in field_availability_matrix.items():
    row = f"{field_name:38}"

    for gene in TARGET_GENES:
        row += f"{('YES' if values[gene] else 'NO'):>10}"

    print(row)

print()


# --------------------------------------------------------------------------------------------------
# 13. Determine the provisional embedded T1 cutoff
# --------------------------------------------------------------------------------------------------

# Prefer the historically used root attribute name "Dated" when present.
embedded_cutoff_attribute = None
embedded_cutoff_raw_value = None
embedded_cutoff_parsed_value = None

for preferred_name in (
    "Dated",
    "Date",
    "ReleaseDate",
    "Released",
    "Generated",
):
    for observed_name, observed_value in date_candidates.items():
        if observed_name.lower() == preferred_name.lower():
            parsed_value = parse_date_candidate(
                observed_value
            )

            if parsed_value:
                embedded_cutoff_attribute = observed_name
                embedded_cutoff_raw_value = observed_value
                embedded_cutoff_parsed_value = parsed_value
                break

    if embedded_cutoff_parsed_value:
        break

assert embedded_cutoff_parsed_value is not None


# --------------------------------------------------------------------------------------------------
# 14. Create schema/provenance probe manifest
# --------------------------------------------------------------------------------------------------

manifest = {
    "manifest_name":
        "clinvar_t1_rcv_xml_schema_probe_manifest_v1",
    "manifest_version": "1.0.0",
    "created_utc": utc_now(),

    "study": {
        "experiment":
            "Experiment 1 — Temporal Validation of GES",
        "timepoint": "T1",
        "release_label": "2026-01",
        "primary_unit":
            "RCV-level variant-condition aggregate",
        "target_genes": list(TARGET_GENES),
    },

    "source_artifact": {
        "path": str(T1_XML_PATH),
        "filename": T1_XML_PATH.name,
        "compressed_size_bytes": observed_file_size,
        "verified_md5": receipt_md5,
        "verified_sha256": receipt_sha256,
        "download_receipt_path": str(
            T1_RECEIPT_PATH
        ),
        "download_receipt_sha256":
            observed_receipt_sha256,
        "complete_source_rehashed_in_this_cell":
            False,
        "reason_not_rehashed":
            "The complete source was MD5-, SHA-256-, size-, and "
            "gzip-verified immediately before this schema probe. "
            "This cell reconfirmed receipt integrity, exact size, "
            "and the gzip signature.",
    },

    "xml_root": root_metadata,

    "temporal_provenance": {
        "release_label": "2026-01",
        "http_last_modified_from_download_receipt": (
            t1_receipt
            .get("official_source", {})
            .get("http_metadata", {})
            .get("last_modified")
        ),
        "embedded_cutoff_attribute":
            embedded_cutoff_attribute,
        "embedded_cutoff_raw_value":
            embedded_cutoff_raw_value,
        "embedded_cutoff_parsed_date":
            embedded_cutoff_parsed_value,
        "decision":
            "USE_EMBEDDED_XML_DATE_AS_T1_TEMPORAL_BOUNDARY",
        "warning":
            "HTTP Last-Modified is file-server metadata and is "
            "not used as the biological data cutoff.",
    },

    "streaming_probe": {
        "probe_started_utc": probe_started_utc,
        "probe_completed_utc": probe_completed_utc,
        "top_level_children_seen":
            top_level_children_seen,
        "rcv_records_scanned":
            rcv_records_scanned,
        "scan_stopped_after_all_target_genes_found":
            True,
        "complete_file_scanned": False,
        "top_level_child_tag_counts":
            dict(top_level_child_tags),
        "nonrecord_top_level_tag_counts":
            dict(nonrecord_top_level_tags),
    },

    "representative_records":
        representative_records,

    "representative_sample_files":
        representative_sample_files,

    "field_availability_matrix":
        field_availability_matrix,

    "schema_probe_decision": {
        "decision": "PASS",
        "root_metadata_captured": True,
        "embedded_date_captured": True,
        "all_target_genes_located": True,
        "representative_rcv_records_saved": True,
        "full_t1_extraction_authorized": True,
        "stage_3_crosswalk_authorized": False,
        "reason_stage_3_remains_blocked":
            "The complete T1 target-gene extraction, full-field "
            "validation, and T0/T1 harmonization must be completed first.",
    },

    "leakage_controls": {
        "t1_used_to_modify_t0": False,
        "t1_used_to_fit_or_tune_ges": False,
        "future_instability_outcome_created": False,
        "t0_t1_crosswalk_created": False,
    },

    "next_action": {
        "stage": "Stage 2D",
        "description":
            "Build and run the current-format streaming T1 parser "
            "for BRCA1, BRCA2, MLH1, and EGFR, write a compact "
            "Parquet artifact, and validate identifiers, JSON, SCV "
            "counts, dates, conditions, submitter names, and OrgIDs.",
    },
}


# --------------------------------------------------------------------------------------------------
# 15. Write and verify manifest and SHA-256 sidecar
# --------------------------------------------------------------------------------------------------

atomic_write_json(manifest, MANIFEST_PATH)

manifest_sha256 = sha256_file(MANIFEST_PATH)

MANIFEST_SHA_PATH = MANIFEST_PATH.with_suffix(
    MANIFEST_PATH.suffix + ".sha256"
)

MANIFEST_SHA_PATH.write_text(
    f"{manifest_sha256}  {MANIFEST_PATH.name}\n",
    encoding="utf-8",
)

with MANIFEST_PATH.open("r", encoding="utf-8") as handle:
    manifest_readback = json.load(handle)

assert manifest_readback == manifest
assert sha256_file(MANIFEST_PATH) == manifest_sha256

assert MANIFEST_SHA_PATH.read_text(
    encoding="utf-8"
).strip() == (
    f"{manifest_sha256}  {MANIFEST_PATH.name}"
)


# --------------------------------------------------------------------------------------------------
# 16. Final result
# --------------------------------------------------------------------------------------------------

print("=" * 112)
print("STAGE 2C RESULT — PASS")
print("=" * 112)
print(f"XML root element: {root_metadata['element_name']}")
print(f"XML root attributes: {root_metadata['attributes']}")
print(
    f"Provisional embedded T1 cutoff: "
    f"{embedded_cutoff_parsed_value} "
    f"(from root attribute {embedded_cutoff_attribute})"
)
print(f"RCV records scanned during probe: {rcv_records_scanned:,}")
print()

for gene in TARGET_GENES:
    summary = representative_records[gene]

    print(
        f"{gene}: "
        f"RCV={summary['rcv']['accession']}, "
        f"VCV={summary['measure_set']['vcv_accession']}, "
        f"VariationID={summary['measure_set']['variation_id']}, "
        f"SCVs={summary['scv_information']['scv_assertion_count']:,}"
    )

print()
print(f"Schema probe manifest: {MANIFEST_PATH}")
print(f"Manifest SHA-256: {manifest_sha256}")
print(f"Manifest sidecar: {MANIFEST_SHA_PATH}")
print(f"Representative XML samples: {SAMPLE_DIR}")
print()
print("Full T1 target-gene extraction: AUTHORIZED")
print("Stage 3 T0–T1 crosswalk: NOT YET AUTHORIZED")
print()
print("NEXT REQUIRED ACTION:")
print(
    "Run the complete current-format T1 streaming extraction "
    "and write the harmonized T1 Parquet artifact."
)
print("=" * 112)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
STAGE 2C — T1 ROOT, SCHEMA, AND REPRESENTATIVE RECORD PROBE
T1 source: /content/clinvar_rcv_raw/ClinVarRCVRelease_2026-01.xml.gz
Target genes: BRCA1, BRCA2, MLH1, EGFR

----------------------------------------------------------------------------------------------------------------
T1 SOURCE AND RECEIPT PREREQUISITES
----------------------------------------------------------------------------------------------------------------
T1 download receipt: PASS
T1 receipt SHA-256 sidecar: PASS
T1 compressed size: 5,434,707,247 bytes
Previously verified MD5: 5740de7f8f74a49ba8c58e3ec1b8cc26
Previously calculated SHA-256: 3fba206f1e3086306472ab7b0ae324d4ae516da84857038d03f2937fd20a6e55
Gzip signature: PASS

Probe manifest destination: /content/drive/MyDrive/GES_RAG_Temporal_Study/configs/clinvar_t1_rcv_xml_schema_probe_manifest_v1.json
Compressed sample directory: /cont

/tmp/ipykernel_569/1831682680.py:641: FutureWarning: Truth-testing of elements was a source of confusion and will always return True in future versions. Use specific 'len(elem)' or 'elem is not None' test instead.
  first_direct_child_by_local(
/tmp/ipykernel_569/1831682680.py:770: FutureWarning: Truth-testing of elements was a source of confusion and will always return True in future versions. Use specific 'len(elem)' or 'elem is not None' test instead.
  first_descendant_by_local(


FOUND EGFR: record 1,617, RCV=RCV000119352, VCV=VCV000132951, VariationID=132951, SCVs=1
FOUND MLH1: record 16,911, RCV=RCV000114851, VCV=VCV000126987, VariationID=126987, SCVs=1
FOUND BRCA1: record 22,583, RCV=RCV000159851, VCV=VCV000182080, VariationID=182080, SCVs=1
Scanned 100,000 RCV records; found 3/4 target genes: ['BRCA1', 'EGFR', 'MLH1']
FOUND BRCA2: record 121,894, RCV=RCV000051374, VCV=VCV000057639, VariationID=57639, SCVs=1

----------------------------------------------------------------------------------------------------------------
ROOT AND REPRESENTATIVE RECORD RESULTS
----------------------------------------------------------------------------------------------------------------
Root element: ReleaseSet
Root attributes: {'{http://www.w3.org/2001/XMLSchema-instance}noNamespaceSchemaLocation': 'http://ftp.ncbi.nlm.nih.gov/pub/clinvar/xsd_public/RCV/ClinVar_RCV_2.2.xsd', 'Dated': '2025-12-27', 'Type': 'full'}
Embedded date candidates: {'Dated': '2025-12-27'}
Parsed embed

In [10]:
# ==================================================================================================
# STAGE 2D — COMPLETE T1 TARGET-GENE RCV EXTRACTION, VALIDATION, AND FREEZE
#
# This cell:
#   1. Verifies the T0 freeze, T1 download receipt, and T1 schema probe.
#   2. Freezes the T1 embedded cutoff through Protocol Amendment A002.
#   3. Stream-parses the complete January 2026 RCV XML.
#   4. Extracts BRCA1, BRCA2, MLH1, and EGFR RCV records.
#   5. Preserves current classification axes.
#   6. Writes a harmonized, Zstandard-compressed Parquet artifact.
#   7. Independently validates all serialized fields, nested SCVs, identifiers,
#      conditions, submitters, conflict semantics, counts, and dates.
#   8. Copies the accepted artifact to Drive and verifies its SHA-256.
#   9. Creates and freezes the consolidated T1 extraction-validation manifest.
#
# This cell does NOT:
#   - create the T0–T1 crosswalk;
#   - create future-instability outcomes;
#   - fit or tune GES;
#   - delete the T1 XML.
# ==================================================================================================

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import sys
import json
import gzip
import hashlib
import shutil
import subprocess
import re

from pathlib import Path
from datetime import datetime, timezone
from collections import Counter


# --------------------------------------------------------------------------------------------------
# 1. Install/import required packages
# --------------------------------------------------------------------------------------------------

try:
    import pandas as pd
    import pyarrow as pa
    import pyarrow.parquet as pq
    from lxml import etree

except ImportError:
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "pandas",
            "pyarrow",
            "lxml",
        ],
        check=True,
    )

    import pandas as pd
    import pyarrow as pa
    import pyarrow.parquet as pq
    from lxml import etree


# --------------------------------------------------------------------------------------------------
# 2. Frozen paths and study constants
# --------------------------------------------------------------------------------------------------

DRIVE_ROOT = Path("/content/drive/MyDrive")
STUDY_ROOT = DRIVE_ROOT / "GES_RAG_Temporal_Study"

CONFIG_DIR = STUDY_ROOT / "configs"
DATA_INTERIM_DIR = STUDY_ROOT / "data_interim"
QUALITY_DIR = STUDY_ROOT / "outputs" / "quality_checks"

for directory in (
    CONFIG_DIR,
    DATA_INTERIM_DIR,
    QUALITY_DIR,
):
    directory.mkdir(parents=True, exist_ok=True)


T1_XML_PATH = Path(
    "/content/clinvar_rcv_raw/ClinVarRCVRelease_2026-01.xml.gz"
)

T1_RECEIPT_PATH = (
    CONFIG_DIR
    / "clinvar_t1_rcv_xml_download_receipt_v1.json"
)

T1_SCHEMA_PROBE_PATH = (
    CONFIG_DIR
    / "clinvar_t1_rcv_xml_schema_probe_manifest_v1.json"
)

T0_FREEZE_PATH = (
    CONFIG_DIR
    / "clinvar_t0_consolidated_extraction_validation_manifest_v1_2.json"
)

T1_OUTPUT_PATH = (
    DATA_INTERIM_DIR
    / "t1_rcv_target_genes_harmonized_v1.parquet"
)

T1_MANIFEST_PATH = (
    CONFIG_DIR
    / "clinvar_t1_target_gene_rcv_extraction_validation_manifest_v1.json"
)

T1_VALIDATION_PATH = (
    QUALITY_DIR
    / "clinvar_t1_target_gene_rcv_field_validation_v1.json"
)

A002_PATH = (
    CONFIG_DIR
    / "protocol_amendment_A002_t1_embedded_cutoff.json"
)

LOCAL_OUTPUT_PATH = Path(
    "/content/t1_rcv_target_genes_harmonized_v1.parquet.tmp"
)


EXPECTED_SOURCE_SIZE = 5_434_707_247

EXPECTED_SOURCE_MD5 = (
    "5740de7f8f74a49ba8c58e3ec1b8cc26"
)

EXPECTED_SOURCE_SHA256 = (
    "3fba206f1e3086306472ab7b0ae324d4"
    "ae516da84857038d03f2937fd20a6e55"
)

SOURCE_FILENAME = "ClinVarRCVRelease_2026-01.xml.gz"

RELEASE_LABEL = "2026-01"
ARCHIVE_PUBLICATION_DATE = "2026-01-01"
EMBEDDED_CUTOFF = "2025-12-27"

SCHEMA_URL = (
    "http://ftp.ncbi.nlm.nih.gov/pub/clinvar/"
    "xsd_public/RCV/ClinVar_RCV_2.2.xsd"
)

TARGET_GENES = (
    "BRCA1",
    "BRCA2",
    "MLH1",
    "EGFR",
)

PRIMARY_GENES = {
    "BRCA1",
    "BRCA2",
    "MLH1",
}

BATCH_SIZE = 5000


print("=" * 116)
print(
    "STAGE 2D — COMPLETE T1 TARGET-GENE RCV "
    "EXTRACTION, VALIDATION, AND FREEZE"
)
print("=" * 116)
print(f"Source: {T1_XML_PATH}")
print(f"Output: {T1_OUTPUT_PATH}")
print(f"T1 embedded cutoff: {EMBEDDED_CUTOFF}")
print()


# --------------------------------------------------------------------------------------------------
# 3. General helpers
# --------------------------------------------------------------------------------------------------

def utc_now():
    return (
        datetime.now(timezone.utc)
        .replace(microsecond=0)
        .isoformat()
    )


def sha256_file(
    path: Path,
    chunk_size: int = 16 * 1024 * 1024,
) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def atomic_write_json(
    payload,
    destination: Path,
):
    temporary_path = destination.with_suffix(
        destination.suffix + ".tmp"
    )

    with temporary_path.open(
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            payload,
            handle,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            allow_nan=False,
        )

        handle.write("\n")

    temporary_path.replace(destination)


def write_sha_sidecar(
    path: Path,
):
    digest = sha256_file(path)

    sidecar = path.with_suffix(
        path.suffix + ".sha256"
    )

    sidecar.write_text(
        f"{digest}  {path.name}\n",
        encoding="utf-8",
    )

    return sidecar, digest


def clean_text(value):
    if value is None:
        return None

    text = " ".join(str(value).split())

    return text or None


def unique_ordered(values):
    seen = set()
    result = []

    for value in values:
        value = clean_text(value)

        if value is None or value in seen:
            continue

        seen.add(value)
        result.append(value)

    return result


def json_compact(
    value,
    sort_keys=False,
):
    return json.dumps(
        value,
        ensure_ascii=False,
        separators=(",", ":"),
        sort_keys=sort_keys,
    )


def safe_int(value):
    value = clean_text(value)

    if value is None:
        return None

    try:
        return int(value)

    except (TypeError, ValueError):
        return None


def first_element(
    parent,
    paths,
):
    for path in paths:
        element = parent.find(path)

        if element is not None:
            return element

    return None


def first_text(
    parent,
    tags,
):
    for tag in tags:
        for element in parent.iter(tag):
            value = clean_text(element.text)

            if value is not None:
                return value

    return None


def first_attribute(
    element,
    names,
):
    if element is None:
        return None

    wanted = {
        name.lower()
        for name in names
    }

    for key, value in element.attrib.items():
        local_name = (
            key.split("}", 1)[-1]
            .lower()
        )

        if local_name in wanted:
            return clean_text(value)

    return None


def parse_iso_date(value):
    value = clean_text(value)

    if value is None:
        return None

    match = re.search(
        r"\d{4}-\d{2}-\d{2}",
        value,
    )

    if not match:
        return None

    try:
        return (
            datetime.strptime(
                match.group(0),
                "%Y-%m-%d",
            )
            .date()
            .isoformat()
        )

    except ValueError:
        return None


def extract_date_from_node(node):
    raw_candidates = []

    date_names = {
        "datelastevaluated",
        "lastevaluated",
        "dateevaluated",
    }

    for element in node.iter():

        for key, value in element.attrib.items():
            local_name = (
                key.split("}", 1)[-1]
                .lower()
            )

            if local_name in date_names:
                cleaned = clean_text(value)

                if cleaned:
                    raw_candidates.append(cleaned)

        if element.tag in {
            "DateLastEvaluated",
            "LastEvaluated",
            "DateEvaluated",
        }:
            cleaned = clean_text(element.text)

            if cleaned:
                raw_candidates.append(cleaned)

    raw_candidates = unique_ordered(
        raw_candidates
    )

    for raw_value in raw_candidates:
        parsed_value = parse_iso_date(
            raw_value
        )

        if parsed_value:
            return parsed_value, raw_value

    return (
        None,
        raw_candidates[0]
        if raw_candidates
        else None,
    )


# --------------------------------------------------------------------------------------------------
# 4. Classification normalization
# --------------------------------------------------------------------------------------------------

def normalize_axis(
    tag,
    node,
):
    if tag == "ClinicalSignificance":
        return "GermlineClassification"

    if tag == "Classification":
        return (
            first_attribute(
                node,
                [
                    "Type",
                    "ClassificationType",
                ],
            )
            or first_text(
                node,
                ["ClassificationType"],
            )
            or "Classification"
        )

    return tag


def normalize_classification_group(value):
    text = clean_text(value)

    if text is None:
        return "Missing"

    lowered = (
        text.lower()
        .replace("_", " ")
    )

    if any(
        term in lowered
        for term in (
            "conflicting classification",
            "conflicting interpretation",
            "conflicting data",
        )
    ):
        return "Conflicting"

    if (
        "uncertain significance" in lowered
        or re.search(r"\bvus\b", lowered)
    ):
        return "VUS"

    if any(
        term in lowered
        for term in (
            "likely pathogenic",
            "pathogenic",
            "likely oncogenic",
            "oncogenic",
        )
    ):
        return "Pathogenic/Likely pathogenic"

    if (
        "likely benign" in lowered
        or re.search(r"\bbenign\b", lowered)
    ):
        return "Benign/Likely benign"

    if lowered in {
        "not provided",
        "not classified",
        "no classification",
        "missing",
        "none",
    }:
        return "Missing"

    return "Other"


def review_stars(review_status):
    status = (
        clean_text(review_status)
        or ""
    ).lower()

    if "practice guideline" in status:
        return 4

    if "reviewed by expert panel" in status:
        return 3

    if (
        "multiple submitters" in status
        and "no conflict" in status
    ):
        return 2

    if (
        "single submitter" in status
        or "conflicting" in status
    ):
        return 1

    return 0


def semantic_conflict_flag(
    classification,
    group,
    review_status,
):
    if group == "Conflicting":
        return True

    status = (
        clean_text(review_status)
        or ""
    ).lower()

    if "no conflict" in status:
        return False

    return (
        "conflicting classification" in status
        or "conflicting interpretation" in status
    )


def classification_priority(
    target_genes,
):
    if target_genes == ["EGFR"]:
        return [
            "OncogenicityClassification",
            "SomaticClinicalImpact",
            "GermlineClassification",
            "Classification",
        ]

    return [
        "GermlineClassification",
        "OncogenicityClassification",
        "SomaticClinicalImpact",
        "Classification",
    ]


def choose_classification(
    classifications,
    target_genes,
    preferred_axis=None,
):
    if not classifications:
        return None

    if preferred_axis:
        for item in classifications:
            if item["axis"] == preferred_axis:
                return item

    for axis in classification_priority(
        target_genes
    ):
        for item in classifications:
            if item["axis"] == axis:
                return item

    return classifications[0]


# --------------------------------------------------------------------------------------------------
# 5. Verify prerequisites
# --------------------------------------------------------------------------------------------------

for required_path in (
    T1_XML_PATH,
    T1_RECEIPT_PATH,
    T1_SCHEMA_PROBE_PATH,
    T0_FREEZE_PATH,
):
    if not required_path.exists():
        raise FileNotFoundError(
            "Required prerequisite is missing:\n"
            f"{required_path}"
        )


if (
    T1_XML_PATH.stat().st_size
    != EXPECTED_SOURCE_SIZE
):
    raise AssertionError(
        "T1 source byte size no longer "
        "matches the verified receipt."
    )


with T1_RECEIPT_PATH.open(
    "r",
    encoding="utf-8",
) as handle:
    receipt = json.load(handle)


with T1_SCHEMA_PROBE_PATH.open(
    "r",
    encoding="utf-8",
) as handle:
    schema_probe = json.load(handle)


with T0_FREEZE_PATH.open(
    "r",
    encoding="utf-8",
) as handle:
    t0_freeze = json.load(handle)


assert (
    receipt["verification_results"]
    ["overall_decision"]
    == "PASS"
)

assert (
    receipt["local_artifact"]
    ["observed_md5"]
    .lower()
    == EXPECTED_SOURCE_MD5
)

assert (
    receipt["local_artifact"]
    ["observed_sha256"]
    .lower()
    == EXPECTED_SOURCE_SHA256
)

assert (
    schema_probe["schema_probe_decision"]
    ["decision"]
    == "PASS"
)

assert (
    schema_probe["temporal_provenance"]
    ["embedded_cutoff_parsed_date"]
    == EMBEDDED_CUTOFF
)

assert (
    t0_freeze["administrative_freeze_decision"]
    ["decision"]
    == "ACCEPTED_AND_FROZEN"
)


if (
    T1_OUTPUT_PATH.exists()
    or T1_MANIFEST_PATH.exists()
):
    raise FileExistsError(
        "An accepted Stage 2D artifact or manifest "
        "already exists and was not overwritten.\n\n"
        f"Artifact: {T1_OUTPUT_PATH}\n"
        f"Manifest: {T1_MANIFEST_PATH}"
    )


if LOCAL_OUTPUT_PATH.exists():
    LOCAL_OUTPUT_PATH.unlink()


# --------------------------------------------------------------------------------------------------
# 6. Freeze Protocol Amendment A002
# --------------------------------------------------------------------------------------------------

amendment = {
    "amendment_id": "A002",
    "created_utc": utc_now(),

    "protocol_context":
        "Temporal validation protocol v1.1",

    "change": {
        "field":
            "T1 embedded data cutoff",

        "previous_value":
            "Pending T1 XML inspection",

        "new_value":
            EMBEDDED_CUTOFF,

        "release_label":
            RELEASE_LABEL,

        "archive_publication_date":
            ARCHIVE_PUBLICATION_DATE,

        "xml_schema":
            SCHEMA_URL,
    },

    "basis": {
        "schema_probe_manifest":
            str(T1_SCHEMA_PROBE_PATH),

        "schema_probe_sha256":
            sha256_file(
                T1_SCHEMA_PROBE_PATH
            ),

        "xml_root_element":
            "ReleaseSet",

        "xml_root_Dated":
            EMBEDDED_CUTOFF,

        "source_sha256":
            EXPECTED_SOURCE_SHA256,
    },

    "timing_and_leakage": {
        "created_before_full_t1_extraction_completed":
            True,

        "t1_temporal_outcomes_examined":
            False,

        "ges_model_fitted_or_tuned":
            False,

        "t0_artifact_modified":
            False,
    },

    "decision":
        "ACCEPTED",
}


if A002_PATH.exists():
    with A002_PATH.open(
        "r",
        encoding="utf-8",
    ) as handle:
        existing_amendment = json.load(
            handle
        )

    assert (
        existing_amendment["change"]
        ["new_value"]
        == EMBEDDED_CUTOFF
    )

    assert (
        existing_amendment["basis"]
        ["source_sha256"]
        == EXPECTED_SOURCE_SHA256
    )

else:
    atomic_write_json(
        amendment,
        A002_PATH,
    )


write_sha_sidecar(A002_PATH)

print("Prerequisite verification: PASS")
print(f"Protocol amendment A002: {A002_PATH}")
print()


# --------------------------------------------------------------------------------------------------
# 7. Current-format XML extraction helpers
# --------------------------------------------------------------------------------------------------

def extract_measure_set(reference):
    return first_element(
        reference,
        [
            "MeasureSet",
            ".//MeasureSet",
        ],
    )


def extract_target_genes(reference):
    measure_set = extract_measure_set(
        reference
    )

    if measure_set is None:
        return []

    symbols = []

    for symbol_node in measure_set.iter(
        "Symbol"
    ):
        for value_node in symbol_node.iter(
            "ElementValue"
        ):
            value = clean_text(
                value_node.text
            )

            if value:
                symbols.append(
                    value.upper()
                )

    for node in measure_set.iter(
        "GeneSymbol"
    ):
        value = clean_text(node.text)

        if value:
            symbols.append(
                value.upper()
            )

    symbols = unique_ordered(symbols)

    return [
        gene
        for gene in TARGET_GENES
        if gene in symbols
    ]


def extract_rcv(reference):
    for node in reference.iter(
        "ClinVarAccession"
    ):
        accession = first_attribute(
            node,
            [
                "Acc",
                "Accession",
            ],
        )

        accession_type = first_attribute(
            node,
            ["Type"],
        )

        if (
            accession
            and (
                accession.upper().startswith(
                    "RCV"
                )
                or (
                    accession_type
                    or ""
                ).upper() == "RCV"
            )
        ):
            return (
                accession.upper(),
                safe_int(
                    first_attribute(
                        node,
                        ["Version"],
                    )
                ),
            )

    return None, None


def extract_measure_information(
    reference,
):
    measure_set = extract_measure_set(
        reference
    )

    if measure_set is None:
        return (
            None,
            None,
            None,
            None,
            [],
        )

    variation_id = safe_int(
        first_attribute(
            measure_set,
            [
                "ID",
                "VariationID",
            ],
        )
    )

    accession = first_attribute(
        measure_set,
        [
            "Acc",
            "Accession",
        ],
    )

    vcv_accession = (
        accession.upper()
        if (
            accession
            and accession.upper().startswith(
                "VCV"
            )
        )
        else accession
    )

    vcv_version = safe_int(
        first_attribute(
            measure_set,
            ["Version"],
        )
    )

    measure_set_type = first_attribute(
        measure_set,
        ["Type"],
    )

    measure_types = unique_ordered(
        first_attribute(
            node,
            ["Type"],
        )
        for node in measure_set.iter(
            "Measure"
        )
    )

    return (
        variation_id,
        vcv_accession,
        vcv_version,
        measure_set_type,
        measure_types,
    )


# --------------------------------------------------------------------------------------------------
# 8. Condition and trait parsing
# --------------------------------------------------------------------------------------------------

def names_from_name_node(name_node):
    preferred = []
    all_values = []

    for value_node in name_node.iter(
        "ElementValue"
    ):
        value = clean_text(
            value_node.text
        )

        if not value:
            continue

        all_values.append(value)

        value_type = (
            first_attribute(
                value_node,
                ["Type"],
            )
            or ""
        ).lower()

        if value_type == "preferred":
            preferred.append(value)

    return (
        unique_ordered(preferred),
        unique_ordered(all_values),
    )


def extract_conditions(reference):
    trait_records = []

    trait_nodes = reference.findall(
        ".//TraitSet/Trait"
    )

    for trait in trait_nodes:
        preferred_names = []
        all_names = []
        identifiers = []

        for name_node in trait.findall(
            ".//Name"
        ):
            preferred, all_values = (
                names_from_name_node(
                    name_node
                )
            )

            preferred_names.extend(
                preferred
            )

            all_names.extend(
                all_values
            )

        for xref in trait.findall(
            ".//XRef"
        ):
            database = first_attribute(
                xref,
                [
                    "DB",
                    "Database",
                ],
            )

            identifier = first_attribute(
                xref,
                [
                    "ID",
                    "Identifier",
                ],
            )

            if database and identifier:
                identifiers.append(
                    f"{database}:{identifier}"
                )

        trait_records.append(
            {
                "trait_type":
                    first_attribute(
                        trait,
                        ["Type"],
                    ),

                "preferred_names":
                    unique_ordered(
                        preferred_names
                    ),

                "names":
                    unique_ordered(
                        all_names
                    ),

                "identifiers":
                    unique_ordered(
                        identifiers
                    ),
            }
        )

    # Defensive fallback for a TraitSet without explicit Trait children.
    if not trait_records:
        trait_set = first_element(
            reference,
            [
                "TraitSet",
                ".//TraitSet",
            ],
        )

        if trait_set is not None:
            preferred_names = []
            all_names = []
            identifiers = []

            for name_node in trait_set.findall(
                ".//Name"
            ):
                preferred, all_values = (
                    names_from_name_node(
                        name_node
                    )
                )

                preferred_names.extend(
                    preferred
                )

                all_names.extend(
                    all_values
                )

            for xref in trait_set.findall(
                ".//XRef"
            ):
                database = first_attribute(
                    xref,
                    [
                        "DB",
                        "Database",
                    ],
                )

                identifier = first_attribute(
                    xref,
                    [
                        "ID",
                        "Identifier",
                    ],
                )

                if database and identifier:
                    identifiers.append(
                        f"{database}:{identifier}"
                    )

            trait_records.append(
                {
                    "trait_type":
                        first_attribute(
                            trait_set,
                            ["Type"],
                        ),

                    "preferred_names":
                        unique_ordered(
                            preferred_names
                        ),

                    "names":
                        unique_ordered(
                            all_names
                        ),

                    "identifiers":
                        unique_ordered(
                            identifiers
                        ),
                }
            )

    condition_names = []
    condition_ids = []

    for trait in trait_records:
        condition_names.extend(
            trait["preferred_names"]
            or trait["names"]
        )

        condition_ids.extend(
            trait["identifiers"]
        )

    return (
        unique_ordered(
            condition_names
        ),
        unique_ordered(
            condition_ids
        ),
        trait_records,
    )


# --------------------------------------------------------------------------------------------------
# 9. Aggregate and submitted classification parsing
# --------------------------------------------------------------------------------------------------

def summarize_classification_node(node):
    axis = normalize_axis(
        node.tag,
        node,
    )

    description = (
        first_text(
            node,
            [
                "Description",
                "ClinicalSignificance",
                "Value",
            ],
        )
        or clean_text(node.text)
    )

    review_status = first_text(
        node,
        ["ReviewStatus"],
    )

    date_iso, date_raw = (
        extract_date_from_node(node)
    )

    explanation = first_text(
        node,
        [
            "Explanation",
            "Comment",
        ],
    )

    return {
        "axis":
            axis,

        "classification":
            description,

        "classification_group":
            normalize_classification_group(
                description
            ),

        "review_status":
            review_status,

        "last_evaluated":
            date_iso,

        "last_evaluated_raw":
            date_raw,

        "explanation":
            explanation,
    }


def extract_classifications(assertion):
    results = []

    classifications_container = (
        first_element(
            assertion,
            [
                "Classifications",
                ".//Classifications",
            ],
        )
    )

    if classifications_container is not None:
        for child in classifications_container:

            if not isinstance(
                child.tag,
                str,
            ):
                continue

            if child.tag in {
                "Comment",
                "Citation",
                "DescriptionHistory",
            }:
                continue

            has_description = (
                first_text(
                    child,
                    [
                        "Description",
                        "ClinicalSignificance",
                        "Value",
                    ],
                )
                is not None
            )

            has_review = (
                first_text(
                    child,
                    ["ReviewStatus"],
                )
                is not None
            )

            if (
                child.tag in {
                    "GermlineClassification",
                    "SomaticClinicalImpact",
                    "OncogenicityClassification",
                    "Classification",
                }
                or has_description
                or has_review
            ):
                results.append(
                    summarize_classification_node(
                        child
                    )
                )

    if not results:
        historical_node = first_element(
            assertion,
            [
                "ClinicalSignificance",
                ".//ClinicalSignificance",
            ],
        )

        if historical_node is not None:
            results.append(
                summarize_classification_node(
                    historical_node
                )
            )

    return results


# --------------------------------------------------------------------------------------------------
# 10. SCV parsing
# --------------------------------------------------------------------------------------------------

def extract_scv_accession(scv):
    for node in scv.iter(
        "ClinVarAccession"
    ):
        accession = first_attribute(
            node,
            [
                "Acc",
                "Accession",
            ],
        )

        accession_type = first_attribute(
            node,
            ["Type"],
        )

        if (
            accession
            and (
                accession.upper().startswith(
                    "SCV"
                )
                or (
                    accession_type
                    or ""
                ).upper() == "SCV"
            )
        ):
            return (
                node,
                accession.upper(),
                safe_int(
                    first_attribute(
                        node,
                        ["Version"],
                    )
                ),
            )

    return None, None, None


def extract_assertion_method(scv):
    values = []

    for tag in (
        "MethodType",
        "MethodName",
    ):
        for node in scv.iter(tag):
            value = clean_text(
                node.text
            )

            if value:
                values.append(value)

    values = unique_ordered(values)

    return (
        " | ".join(values)
        if values
        else None
    )


def extract_scv_records(
    record,
    target_genes,
    preferred_axis,
):
    submitted_assertions = (
        record.findall(
            ".//ClinVarAssertion"
        )
    )

    scv_records = []
    submitter_names = []
    submitter_ids = []

    raw_counts = Counter()
    group_counts = Counter()

    for scv in submitted_assertions:

        (
            accession_node,
            scv_accession,
            scv_version,
        ) = extract_scv_accession(scv)

        submission_node = first_element(
            scv,
            [
                "ClinVarSubmissionID",
                ".//ClinVarSubmissionID",
            ],
        )

        submitter_name = first_attribute(
            submission_node,
            [
                "submitter",
                "Submitter",
            ],
        )

        organization_id = first_attribute(
            accession_node,
            [
                "OrgID",
                "OrganizationID",
            ],
        )

        classifications = (
            extract_classifications(scv)
        )

        selected = choose_classification(
            classifications,
            target_genes,
            preferred_axis=preferred_axis,
        )

        if selected is None:
            selected = {
                "axis":
                    preferred_axis
                    or "Missing",

                "classification":
                    None,

                "classification_group":
                    "Missing",

                "review_status":
                    None,

                "last_evaluated":
                    None,

                "last_evaluated_raw":
                    None,

                "explanation":
                    None,
            }

        raw_key = (
            selected["classification"]
            or "Missing"
        )

        raw_counts[raw_key] += 1

        group_counts[
            selected[
                "classification_group"
            ]
        ] += 1

        if submitter_name:
            submitter_names.append(
                submitter_name
            )

        if organization_id:
            submitter_ids.append(
                str(organization_id)
            )

        origin = first_text(
            scv,
            ["Origin"],
        )

        scv_records.append(
            {
                "scv_accession":
                    scv_accession,

                "scv_version":
                    scv_version,

                "classification_axis":
                    selected["axis"],

                "classification":
                    selected[
                        "classification"
                    ],

                "classification_group":
                    selected[
                        "classification_group"
                    ],

                "review_status":
                    selected[
                        "review_status"
                    ],

                "last_evaluated":
                    selected[
                        "last_evaluated"
                    ],

                "last_evaluated_raw":
                    selected[
                        "last_evaluated_raw"
                    ],

                "submitter":
                    submitter_name,

                "submitter_orgid":
                    (
                        str(organization_id)
                        if organization_id
                        is not None
                        else None
                    ),

                "origin":
                    origin,

                "assertion_method":
                    extract_assertion_method(
                        scv
                    ),

                "all_classifications":
                    classifications,
            }
        )

    submitter_names = unique_ordered(
        submitter_names
    )

    submitter_ids = unique_ordered(
        submitter_ids
    )

    unique_submitter_keys = set(
        submitter_names
    )

    if not unique_submitter_keys:
        unique_submitter_keys = {
            f"ORGID:{value}"
            for value in submitter_ids
        }

    primary_groups = {
        item["classification_group"]
        for item in scv_records
        if item["classification_group"]
        in {
            "Pathogenic/Likely pathogenic",
            "Benign/Likely benign",
            "VUS",
        }
    }

    return {
        "records":
            scv_records,

        "submitter_names":
            submitter_names,

        "submitter_ids":
            submitter_ids,

        "unique_submitter_count":
            len(unique_submitter_keys),

        "raw_counts":
            dict(
                sorted(
                    raw_counts.items()
                )
            ),

        "group_counts":
            dict(
                sorted(
                    group_counts.items()
                )
            ),

        "group_disagreement":
            len(primary_groups) > 1,
    }


def scope_for(target_genes):
    genes = set(target_genes)

    if genes == {"EGFR"}:
        return "exploratory"

    if genes.issubset(
        PRIMARY_GENES
    ):
        return "primary"

    return "mixed"


# --------------------------------------------------------------------------------------------------
# 11. Parse one retained RCV record
# --------------------------------------------------------------------------------------------------

def parse_record(
    record,
    xml_record_index,
):
    reference = first_element(
        record,
        [
            "ReferenceClinVarAssertion",
            ".//ReferenceClinVarAssertion",
        ],
    )

    if reference is None:
        return None

    target_genes = extract_target_genes(
        reference
    )

    if not target_genes:
        return None

    (
        rcv_accession,
        rcv_version,
    ) = extract_rcv(reference)

    (
        variation_id,
        vcv_accession,
        vcv_version,
        measure_set_type,
        measure_types,
    ) = extract_measure_information(
        reference
    )

    (
        condition_names,
        condition_ids,
        trait_records,
    ) = extract_conditions(reference)

    aggregate_classifications = (
        extract_classifications(
            reference
        )
    )

    selected_aggregate = (
        choose_classification(
            aggregate_classifications,
            target_genes,
        )
    )

    if selected_aggregate is None:
        selected_aggregate = {
            "axis":
                "Missing",

            "classification":
                None,

            "classification_group":
                "Missing",

            "review_status":
                None,

            "last_evaluated":
                None,

            "last_evaluated_raw":
                None,

            "explanation":
                None,
        }

    scv = extract_scv_records(
        record,
        target_genes,
        selected_aggregate["axis"],
    )

    conflict_flag = (
        semantic_conflict_flag(
            selected_aggregate[
                "classification"
            ],
            selected_aggregate[
                "classification_group"
            ],
            selected_aggregate[
                "review_status"
            ],
        )
    )

    return {
        "timepoint":
            "T1",

        "release_label":
            RELEASE_LABEL,

        "archive_publication_date":
            ARCHIVE_PUBLICATION_DATE,

        "embedded_data_cutoff_date":
            EMBEDDED_CUTOFF,

        "xml_record_index":
            xml_record_index,

        "source_filename":
            SOURCE_FILENAME,

        "source_sha256":
            EXPECTED_SOURCE_SHA256,

        "rcv_accession":
            rcv_accession,

        "rcv_version":
            rcv_version,

        "variation_id":
            variation_id,

        "vcv_accession":
            vcv_accession,

        "vcv_version":
            vcv_version,

        "target_genes_json":
            json_compact(
                target_genes
            ),

        "study_scope":
            scope_for(
                target_genes
            ),

        "measure_set_type":
            measure_set_type,

        "measure_types_json":
            json_compact(
                measure_types
            ),

        "condition_names_json":
            json_compact(
                condition_names
            ),

        "condition_ids_json":
            json_compact(
                condition_ids
            ),

        "trait_records_json":
            json_compact(
                trait_records
            ),

        "aggregate_classification":
            selected_aggregate[
                "classification"
            ],

        "aggregate_classification_group":
            selected_aggregate[
                "classification_group"
            ],

        "aggregate_review_status":
            selected_aggregate[
                "review_status"
            ],

        "aggregate_review_stars":
            review_stars(
                selected_aggregate[
                    "review_status"
                ]
            ),

        "aggregate_last_evaluated":
            selected_aggregate[
                "last_evaluated"
            ],

        "aggregate_explanation":
            selected_aggregate[
                "explanation"
            ],

        "aggregate_conflict_flag":
            conflict_flag,

        "scv_group_disagreement_flag":
            scv[
                "group_disagreement"
            ],

        "scv_count_xml":
            len(
                scv["records"]
            ),

        "unique_submitter_count_xml":
            scv[
                "unique_submitter_count"
            ],

        "submitters_json":
            json_compact(
                scv[
                    "submitter_names"
                ]
            ),

        "submitter_ids_json":
            json_compact(
                scv[
                    "submitter_ids"
                ]
            ),

        "scv_classification_counts_json":
            json_compact(
                scv["raw_counts"],
                sort_keys=True,
            ),

        "scv_group_counts_json":
            json_compact(
                scv["group_counts"],
                sort_keys=True,
            ),

        "scv_records_json":
            json_compact(
                scv["records"]
            ),

        "aggregate_classification_axis":
            selected_aggregate[
                "axis"
            ],

        "aggregate_classifications_json":
            json_compact(
                aggregate_classifications
            ),
    }


# --------------------------------------------------------------------------------------------------
# 12. Explicit harmonized Parquet schema
# --------------------------------------------------------------------------------------------------

PARQUET_SCHEMA = pa.schema(
    [
        ("timepoint", pa.string()),
        ("release_label", pa.string()),
        ("archive_publication_date", pa.string()),
        ("embedded_data_cutoff_date", pa.string()),
        ("xml_record_index", pa.int64()),
        ("source_filename", pa.string()),
        ("source_sha256", pa.string()),

        ("rcv_accession", pa.string()),
        ("rcv_version", pa.int32()),
        ("variation_id", pa.int64()),
        ("vcv_accession", pa.string()),
        ("vcv_version", pa.int32()),

        ("target_genes_json", pa.string()),
        ("study_scope", pa.string()),

        ("measure_set_type", pa.string()),
        ("measure_types_json", pa.string()),

        ("condition_names_json", pa.string()),
        ("condition_ids_json", pa.string()),
        ("trait_records_json", pa.string()),

        ("aggregate_classification", pa.string()),
        ("aggregate_classification_group", pa.string()),
        ("aggregate_review_status", pa.string()),
        ("aggregate_review_stars", pa.int8()),
        ("aggregate_last_evaluated", pa.string()),
        ("aggregate_explanation", pa.string()),

        ("aggregate_conflict_flag", pa.bool_()),
        ("scv_group_disagreement_flag", pa.bool_()),

        ("scv_count_xml", pa.int32()),
        ("unique_submitter_count_xml", pa.int32()),

        ("submitters_json", pa.string()),
        ("submitter_ids_json", pa.string()),

        ("scv_classification_counts_json", pa.string()),
        ("scv_group_counts_json", pa.string()),
        ("scv_records_json", pa.string()),

        # Current-schema extensions.
        ("aggregate_classification_axis", pa.string()),
        ("aggregate_classifications_json", pa.string()),
    ]
)


# --------------------------------------------------------------------------------------------------
# 13. Complete streaming extraction
# --------------------------------------------------------------------------------------------------

print("-" * 116)
print("COMPLETE T1 STREAMING EXTRACTION")
print("-" * 116)
print(
    "The 5.06 GiB compressed XML will now be scanned completely. "
    "Keep the runtime active."
)
print()


started_utc = utc_now()

scanned = 0
retained = 0

batch = []

writer = pq.ParquetWriter(
    str(LOCAL_OUTPUT_PATH),
    PARQUET_SCHEMA,
    compression="zstd",
    use_dictionary=True,
    write_statistics=True,
)


runtime_counts = {
    "parser_exceptions": 0,
    "missing_rcv": 0,
    "missing_variation_id": 0,
    "missing_vcv": 0,
    "multi_target_gene_rows": 0,
}

gene_counts = Counter()
axis_counts = Counter()
scope_counts = Counter()


try:
    with gzip.open(
        T1_XML_PATH,
        "rb",
    ) as stream:

        context = etree.iterparse(
            stream,
            events=("end",),
            tag="ClinVarSet",
            huge_tree=True,
            recover=False,
        )

        for _, record in context:
            scanned += 1

            try:
                row = parse_record(
                    record,
                    scanned,
                )

            except Exception as exc:
                runtime_counts[
                    "parser_exceptions"
                ] += 1

                raise RuntimeError(
                    "Parser exception at ClinVarSet "
                    f"record {scanned:,}: "
                    f"{type(exc).__name__}: {exc}"
                ) from exc

            if row is not None:
                retained += 1

                genes = json.loads(
                    row[
                        "target_genes_json"
                    ]
                )

                for gene in genes:
                    gene_counts[gene] += 1

                axis_counts[
                    row[
                        "aggregate_classification_axis"
                    ]
                ] += 1

                scope_counts[
                    row["study_scope"]
                ] += 1

                runtime_counts[
                    "missing_rcv"
                ] += int(
                    row[
                        "rcv_accession"
                    ]
                    is None
                )

                runtime_counts[
                    "missing_variation_id"
                ] += int(
                    row[
                        "variation_id"
                    ]
                    is None
                )

                runtime_counts[
                    "missing_vcv"
                ] += int(
                    row[
                        "vcv_accession"
                    ]
                    is None
                )

                runtime_counts[
                    "multi_target_gene_rows"
                ] += int(
                    len(genes) > 1
                )

                batch.append(row)

                if len(batch) >= BATCH_SIZE:
                    writer.write_table(
                        pa.Table.from_pylist(
                            batch,
                            schema=PARQUET_SCHEMA,
                        )
                    )

                    batch.clear()

            if scanned % 100_000 == 0:
                print(
                    f"Scanned {scanned:,} ClinVarSet records | "
                    f"retained {retained:,} target RCVs | "
                    f"genes {dict(gene_counts)}"
                )

            parent = record.getparent()

            record.clear()

            if parent is not None:
                while (
                    record.getprevious()
                    is not None
                ):
                    del parent[0]

        del context

    if batch:
        writer.write_table(
            pa.Table.from_pylist(
                batch,
                schema=PARQUET_SCHEMA,
            )
        )

        batch.clear()

finally:
    writer.close()


completed_utc = utc_now()


if retained == 0:
    raise AssertionError(
        "The full parser retained zero "
        "target-gene RCV records."
    )


if (
    runtime_counts["parser_exceptions"]
    != 0
):
    raise AssertionError(
        "At least one parser exception occurred."
    )


if (
    runtime_counts["missing_rcv"] != 0
    or runtime_counts[
        "missing_variation_id"
    ] != 0
):
    raise AssertionError(
        "Mandatory identifier missingness detected:\n"
        f"{runtime_counts}"
    )


print()
print(
    "Complete ClinVarSet records scanned: "
    f"{scanned:,}"
)
print(
    "Target RCV records retained: "
    f"{retained:,}"
)
print(f"Gene counts: {dict(gene_counts)}")
print(
    "Classification-axis counts: "
    f"{dict(axis_counts)}"
)
print(
    "Study-scope counts: "
    f"{dict(scope_counts)}"
)
print(f"Local Parquet: {LOCAL_OUTPUT_PATH}")
print()


# --------------------------------------------------------------------------------------------------
# 14. Independent complete-field validation
# --------------------------------------------------------------------------------------------------

print("-" * 116)
print("INDEPENDENT T1 PARQUET FIELD VALIDATION")
print("-" * 116)


parquet_file = pq.ParquetFile(
    LOCAL_OUTPUT_PATH
)

dataframe = pd.read_parquet(
    LOCAL_OUTPUT_PATH
)


assert len(dataframe) == retained
assert len(dataframe.columns) == len(
    PARQUET_SCHEMA
)

assert dataframe[
    "rcv_accession"
].notna().all()

assert dataframe[
    "variation_id"
].notna().all()

assert (
    dataframe[
        "rcv_accession"
    ].nunique()
    == len(dataframe)
), "Duplicate RCV accessions detected in T1."

assert (
    dataframe[
        "xml_record_index"
    ].nunique()
    == len(dataframe)
), "Duplicate retained XML indexes detected."

assert set(
    dataframe[
        "timepoint"
    ].unique()
) == {"T1"}

assert set(
    dataframe[
        "release_label"
    ].unique()
) == {RELEASE_LABEL}

assert set(
    dataframe[
        "embedded_data_cutoff_date"
    ].unique()
) == {EMBEDDED_CUTOFF}

assert set(
    dataframe[
        "source_sha256"
    ].unique()
) == {EXPECTED_SOURCE_SHA256}


json_list_fields = {
    "target_genes_json",
    "measure_types_json",
    "condition_names_json",
    "condition_ids_json",
    "trait_records_json",
    "submitters_json",
    "submitter_ids_json",
    "scv_records_json",
    "aggregate_classifications_json",
}

json_dict_fields = {
    "scv_classification_counts_json",
    "scv_group_counts_json",
}


validation = Counter()
validation_gene_counts = Counter()
validation_axis_counts = Counter()

aggregate_dates = []
scv_dates = []

cutoff_date = datetime.strptime(
    EMBEDDED_CUTOFF,
    "%Y-%m-%d",
).date()


for row_number, row in enumerate(
    dataframe.itertuples(
        index=False
    ),
    start=1,
):
    parsed = {}

    for field in json_list_fields:
        try:
            value = json.loads(
                getattr(row, field)
            )

        except Exception as exc:
            validation[
                "json_parse_errors"
            ] += 1

            raise ValueError(
                f"Malformed JSON in {field}, "
                f"row {row_number}: {exc}"
            ) from exc

        if not isinstance(
            value,
            list,
        ):
            validation[
                "json_type_errors"
            ] += 1

            raise TypeError(
                f"{field} must contain a list "
                f"at row {row_number}."
            )

        parsed[field] = value

    for field in json_dict_fields:
        try:
            value = json.loads(
                getattr(row, field)
            )

        except Exception as exc:
            validation[
                "json_parse_errors"
            ] += 1

            raise ValueError(
                f"Malformed JSON in {field}, "
                f"row {row_number}: {exc}"
            ) from exc

        if not isinstance(
            value,
            dict,
        ):
            validation[
                "json_type_errors"
            ] += 1

            raise TypeError(
                f"{field} must contain a dictionary "
                f"at row {row_number}."
            )

        parsed[field] = value

    genes = parsed[
        "target_genes_json"
    ]

    if (
        not genes
        or any(
            gene not in TARGET_GENES
            for gene in genes
        )
    ):
        validation[
            "invalid_target_gene_rows"
        ] += 1

    for gene in genes:
        validation_gene_counts[
            gene
        ] += 1

    validation[
        "multi_target_gene_rows"
    ] += int(
        len(genes) > 1
    )

    validation_axis_counts[
        row.aggregate_classification_axis
    ] += 1

    scv_records = parsed[
        "scv_records_json"
    ]

    if (
        len(scv_records)
        != int(row.scv_count_xml)
    ):
        validation[
            "scv_count_mismatches"
        ] += 1

    reconstructed_raw = Counter(
        (
            item.get(
                "classification"
            )
            or "Missing"
        )
        for item in scv_records
    )

    reconstructed_groups = Counter(
        (
            item.get(
                "classification_group"
            )
            or "Missing"
        )
        for item in scv_records
    )

    if (
        dict(
            sorted(
                reconstructed_raw.items()
            )
        )
        != parsed[
            "scv_classification_counts_json"
        ]
    ):
        validation[
            "scv_classification_count_mismatches"
        ] += 1

    if (
        dict(
            sorted(
                reconstructed_groups.items()
            )
        )
        != parsed[
            "scv_group_counts_json"
        ]
    ):
        validation[
            "scv_group_count_mismatches"
        ] += 1

    names = unique_ordered(
        item.get("submitter")
        for item in scv_records
    )

    organization_ids = unique_ordered(
        item.get(
            "submitter_orgid"
        )
        for item in scv_records
    )

    submitter_key_count = (
        len(names)
        if names
        else len(organization_ids)
    )

    if (
        submitter_key_count
        != int(
            row.unique_submitter_count_xml
        )
    ):
        validation[
            "unique_submitter_count_mismatches"
        ] += 1

    trait_records = parsed[
        "trait_records_json"
    ]

    flattened_names = []
    flattened_ids = []

    for trait in trait_records:

        if not isinstance(
            trait,
            dict,
        ):
            validation[
                "non_dictionary_trait_records"
            ] += 1

            continue

        flattened_names.extend(
            trait.get(
                "preferred_names"
            )
            or trait.get("names")
            or []
        )

        flattened_ids.extend(
            trait.get(
                "identifiers"
            )
            or []
        )

    if (
        unique_ordered(
            flattened_names
        )
        != parsed[
            "condition_names_json"
        ]
    ):
        validation[
            "condition_name_mismatches"
        ] += 1

    if (
        unique_ordered(
            flattened_ids
        )
        != parsed[
            "condition_ids_json"
        ]
    ):
        validation[
            "condition_id_mismatches"
        ] += 1

    expected_disagreement = (
        len(
            {
                item.get(
                    "classification_group"
                )
                for item in scv_records
                if item.get(
                    "classification_group"
                )
                in {
                    "Pathogenic/Likely pathogenic",
                    "Benign/Likely benign",
                    "VUS",
                }
            }
        )
        > 1
    )

    if (
        bool(
            row.scv_group_disagreement_flag
        )
        != expected_disagreement
    ):
        validation[
            "scv_disagreement_flag_mismatches"
        ] += 1

    expected_conflict = (
        semantic_conflict_flag(
            row.aggregate_classification,
            row.aggregate_classification_group,
            row.aggregate_review_status,
        )
    )

    if (
        bool(
            row.aggregate_conflict_flag
        )
        != expected_conflict
    ):
        validation[
            "aggregate_conflict_flag_mismatches"
        ] += 1

    if (
        bool(
            row.aggregate_conflict_flag
        )
        and "no conflict"
        in (
            clean_text(
                row.aggregate_review_status
            )
            or ""
        ).lower()
    ):
        validation[
            "no_conflicts_false_positive_rows"
        ] += 1

    aggregate_date_value = clean_text(
        row.aggregate_last_evaluated
    )

    if aggregate_date_value:
        parsed_date = parse_iso_date(
            aggregate_date_value
        )

        if parsed_date is None:
            validation[
                "malformed_aggregate_dates"
            ] += 1

        else:
            date_object = datetime.strptime(
                parsed_date,
                "%Y-%m-%d",
            ).date()

            aggregate_dates.append(
                parsed_date
            )

            if date_object > cutoff_date:
                validation[
                    "aggregate_dates_after_cutoff"
                ] += 1

    else:
        validation[
            "missing_aggregate_dates"
        ] += 1

    for item in scv_records:

        if (
            item.get(
                "scv_accession"
            )
            is None
        ):
            validation[
                "scvs_missing_accession"
            ] += 1

        if (
            item.get("submitter")
            is None
        ):
            validation[
                "scvs_missing_submitter_name"
            ] += 1

        if (
            item.get(
                "submitter_orgid"
            )
            is None
        ):
            validation[
                "scvs_missing_submitter_orgid"
            ] += 1

        date_value = clean_text(
            item.get(
                "last_evaluated"
            )
        )

        if date_value:
            parsed_date = parse_iso_date(
                date_value
            )

            if parsed_date is None:
                validation[
                    "malformed_scv_dates"
                ] += 1

            else:
                date_object = (
                    datetime.strptime(
                        parsed_date,
                        "%Y-%m-%d",
                    )
                    .date()
                )

                scv_dates.append(
                    parsed_date
                )

                if date_object > cutoff_date:
                    validation[
                        "scv_dates_after_cutoff"
                    ] += 1

        else:
            validation[
                "missing_scv_dates"
            ] += 1

    validation[
        "rows_with_empty_condition_ids"
    ] += int(
        len(
            parsed[
                "condition_ids_json"
            ]
        )
        == 0
    )

    validation[
        "rows_with_empty_condition_names"
    ] += int(
        len(
            parsed[
                "condition_names_json"
            ]
        )
        == 0
    )

    validation[
        "rows_with_nonempty_submitter_ids"
    ] += int(
        len(
            parsed[
                "submitter_ids_json"
            ]
        )
        > 0
    )

    validation[
        "nested_scv_records"
    ] += len(scv_records)


critical_validation_keys = [
    "json_parse_errors",
    "json_type_errors",
    "invalid_target_gene_rows",
    "scv_count_mismatches",
    "scv_classification_count_mismatches",
    "scv_group_count_mismatches",
    "unique_submitter_count_mismatches",
    "non_dictionary_trait_records",
    "condition_name_mismatches",
    "condition_id_mismatches",
    "scv_disagreement_flag_mismatches",
    "aggregate_conflict_flag_mismatches",
    "no_conflicts_false_positive_rows",
    "malformed_aggregate_dates",
    "aggregate_dates_after_cutoff",
    "malformed_scv_dates",
    "scv_dates_after_cutoff",
    "scvs_missing_accession",
]


critical_failures = {
    key: int(validation[key])
    for key in critical_validation_keys
    if validation[key] != 0
}


if critical_failures:
    raise AssertionError(
        "Critical T1 validation failures:\n"
        f"{critical_failures}"
    )


validation_summary = {
    "decision":
        "PASS",

    "rows":
        int(len(dataframe)),

    "columns":
        int(len(dataframe.columns)),

    "row_groups":
        int(
            parquet_file.num_row_groups
        ),

    "unique_rcv_accessions":
        int(
            dataframe[
                "rcv_accession"
            ].nunique()
        ),

    "unique_variation_ids":
        int(
            dataframe[
                "variation_id"
            ].nunique()
        ),

    "unique_vcv_accessions":
        int(
            dataframe[
                "vcv_accession"
            ].nunique(
                dropna=True
            )
        ),

    "gene_counts":
        dict(
            validation_gene_counts
        ),

    "classification_axis_counts":
        dict(
            validation_axis_counts
        ),

    "aggregate_conflict_positive_records":
        int(
            dataframe[
                "aggregate_conflict_flag"
            ].sum()
        ),

    "scv_group_disagreement_positive_records":
        int(
            dataframe[
                "scv_group_disagreement_flag"
            ].sum()
        ),

    "nested_scv_records":
        int(
            validation[
                "nested_scv_records"
            ]
        ),

    "rows_with_nonempty_submitter_ids":
        int(
            validation[
                "rows_with_nonempty_submitter_ids"
            ]
        ),

    "scvs_missing_submitter_name":
        int(
            validation[
                "scvs_missing_submitter_name"
            ]
        ),

    "scvs_missing_submitter_orgid":
        int(
            validation[
                "scvs_missing_submitter_orgid"
            ]
        ),

    "rows_with_empty_condition_ids":
        int(
            validation[
                "rows_with_empty_condition_ids"
            ]
        ),

    "rows_with_empty_condition_names":
        int(
            validation[
                "rows_with_empty_condition_names"
            ]
        ),

    "aggregate_dates_available":
        len(aggregate_dates),

    "aggregate_dates_missing":
        int(
            validation[
                "missing_aggregate_dates"
            ]
        ),

    "aggregate_date_range": {
        "minimum":
            min(aggregate_dates)
            if aggregate_dates
            else None,

        "maximum":
            max(aggregate_dates)
            if aggregate_dates
            else None,
    },

    "scv_dates_available":
        len(scv_dates),

    "scv_dates_missing":
        int(
            validation[
                "missing_scv_dates"
            ]
        ),

    "scv_date_range": {
        "minimum":
            min(scv_dates)
            if scv_dates
            else None,

        "maximum":
            max(scv_dates)
            if scv_dates
            else None,
    },

    "critical_failure_counts": {
        key: int(validation[key])
        for key in critical_validation_keys
    },

    "accepted_source_missingness_to_review_before_crosswalk": {
        "condition_id_empty_rows":
            int(
                validation[
                    "rows_with_empty_condition_ids"
                ]
            ),

        "condition_name_empty_rows":
            int(
                validation[
                    "rows_with_empty_condition_names"
                ]
            ),

        "scv_submitter_name_missing":
            int(
                validation[
                    "scvs_missing_submitter_name"
                ]
            ),

        "scv_submitter_orgid_missing":
            int(
                validation[
                    "scvs_missing_submitter_orgid"
                ]
            ),

        "aggregate_date_missing":
            int(
                validation[
                    "missing_aggregate_dates"
                ]
            ),

        "scv_date_missing":
            int(
                validation[
                    "missing_scv_dates"
                ]
            ),
    },
}


atomic_write_json(
    validation_summary,
    T1_VALIDATION_PATH,
)

(
    validation_sidecar,
    validation_sha256,
) = write_sha_sidecar(
    T1_VALIDATION_PATH
)


print(
    f"Rows validated: "
    f"{len(dataframe):,}"
)

print(
    "Nested SCVs validated: "
    f"{validation_summary['nested_scv_records']:,}"
)

print(
    "Gene counts: "
    f"{validation_summary['gene_counts']}"
)

print(
    "Classification axes: "
    f"{validation_summary['classification_axis_counts']}"
)

print(
    "Conflict-positive RCVs: "
    f"{validation_summary['aggregate_conflict_positive_records']:,}"
)

print(
    "Rows with empty condition IDs: "
    f"{validation_summary['rows_with_empty_condition_ids']:,}"
)

print(
    "SCVs missing primary OrgID: "
    f"{validation_summary['scvs_missing_submitter_orgid']:,}"
)

print("Critical validation failures: 0")
print()


# --------------------------------------------------------------------------------------------------
# 15. Copy accepted Parquet to Drive and verify exact checksum
# --------------------------------------------------------------------------------------------------

local_sha256 = sha256_file(
    LOCAL_OUTPUT_PATH
)

shutil.copy2(
    LOCAL_OUTPUT_PATH,
    T1_OUTPUT_PATH,
)

drive_sha256 = sha256_file(
    T1_OUTPUT_PATH
)


if local_sha256 != drive_sha256:
    raise AssertionError(
        "Local-to-Drive Parquet "
        "SHA-256 mismatch."
    )


drive_readback = pd.read_parquet(
    T1_OUTPUT_PATH
)


if not drive_readback.equals(
    dataframe
):
    raise AssertionError(
        "Drive Parquet readback did not "
        "exactly match the validated "
        "local dataframe."
    )


(
    output_sidecar,
    output_sha256,
) = write_sha_sidecar(
    T1_OUTPUT_PATH
)


# --------------------------------------------------------------------------------------------------
# 16. Freeze consolidated T1 extraction-validation manifest
# --------------------------------------------------------------------------------------------------

manifest = {
    "manifest_name":
        "clinvar_t1_target_gene_rcv_"
        "extraction_validation_manifest_v1",

    "manifest_version":
        "1.0.0",

    "created_utc":
        utc_now(),

    "study": {
        "experiment":
            "Experiment 1 — Temporal Validation of GES",

        "timepoint":
            "T1",

        "release_label":
            RELEASE_LABEL,

        "archive_publication_date":
            ARCHIVE_PUBLICATION_DATE,

        "embedded_data_cutoff_date":
            EMBEDDED_CUTOFF,

        "xml_schema":
            SCHEMA_URL,

        "primary_unit":
            "RCV-level variant-condition aggregate",

        "primary_genes": [
            "BRCA1",
            "BRCA2",
            "MLH1",
        ],

        "exploratory_gene":
            "EGFR",
    },

    "source": {
        "path":
            str(T1_XML_PATH),

        "filename":
            SOURCE_FILENAME,

        "compressed_size_bytes":
            EXPECTED_SOURCE_SIZE,

        "verified_md5":
            EXPECTED_SOURCE_MD5,

        "verified_sha256":
            EXPECTED_SOURCE_SHA256,

        "download_receipt_path":
            str(T1_RECEIPT_PATH),

        "download_receipt_sha256":
            sha256_file(
                T1_RECEIPT_PATH
            ),

        "schema_probe_path":
            str(T1_SCHEMA_PROBE_PATH),

        "schema_probe_sha256":
            sha256_file(
                T1_SCHEMA_PROBE_PATH
            ),

        "protocol_amendment_A002_path":
            str(A002_PATH),

        "protocol_amendment_A002_sha256":
            sha256_file(
                A002_PATH
            ),
    },

    "extraction": {
        "started_utc":
            started_utc,

        "completed_utc":
            completed_utc,

        "complete_clinvarset_records_scanned":
            scanned,

        "target_rcv_records_retained":
            retained,

        "batch_size":
            BATCH_SIZE,

        "compression":
            "Zstandard",

        "parser_version":
            "current_rcv_schema_2_2_harmonized_v1",

        "classification_selection_policy": {
            "all_current_classification_axes_preserved_in":
                "aggregate_classifications_json "
                "and scv_records_json",

            "primary_genes_precedence": [
                "GermlineClassification",
                "OncogenicityClassification",
                "SomaticClinicalImpact",
                "Classification",
            ],

            "egfr_precedence": [
                "OncogenicityClassification",
                "SomaticClinicalImpact",
                "GermlineClassification",
                "Classification",
            ],

            "warning":
                "Axis-specific comparisons must be "
                "enforced during the T0-T1 crosswalk "
                "and outcome construction.",
        },

        "runtime_counts":
            runtime_counts,

        "gene_counts":
            dict(gene_counts),

        "classification_axis_counts":
            dict(axis_counts),

        "scope_counts":
            dict(scope_counts),
    },

    "accepted_artifact": {
        "decision":
            "ACCEPTED_AND_FROZEN",

        "path":
            str(T1_OUTPUT_PATH),

        "filename":
            T1_OUTPUT_PATH.name,

        "size_bytes":
            T1_OUTPUT_PATH.stat().st_size,

        "sha256":
            output_sha256,

        "sha256_sidecar":
            str(output_sidecar),

        "rows":
            int(len(dataframe)),

        "columns":
            int(len(dataframe.columns)),

        "row_groups":
            int(
                parquet_file.num_row_groups
            ),

        "core_t0_harmonized_columns":
            34,

        "current_schema_extension_columns": [
            "aggregate_classification_axis",
            "aggregate_classifications_json",
        ],
    },

    "field_validation": {
        "decision":
            "PASS",

        "validation_path":
            str(T1_VALIDATION_PATH),

        "validation_sha256":
            validation_sha256,

        "validation_sidecar":
            str(validation_sidecar),

        "summary":
            validation_summary,
    },

    "leakage_controls": {
        "t1_used_to_modify_t0":
            False,

        "t1_used_to_fit_or_tune_ges":
            False,

        "future_instability_outcome_created":
            False,

        "t0_t1_crosswalk_created":
            False,

        "unmatched_records_called_stable":
            False,
    },

    "stage_decision": {
        "blueprint_stage_2":
            "COMPLETE",

        "blueprint_stage_3_crosswalk_authorized":
            True,

        "authorization_boundary":
            "Authorization is for linkage and linkage "
            "auditing only. It does not authorize "
            "outcome creation until crosswalk rules "
            "and classification-axis compatibility "
            "are frozen.",
    },

    "raw_t1_xml_disposition": {
        "present_in_current_runtime":
            T1_XML_PATH.exists(),

        "deleted_by_this_cell":
            False,

        "recommended_action":
            "Retain until the Stage 3 crosswalk "
            "and initial linkage audit have passed.",
    },
}


atomic_write_json(
    manifest,
    T1_MANIFEST_PATH,
)

(
    manifest_sidecar,
    manifest_sha256,
) = write_sha_sidecar(
    T1_MANIFEST_PATH
)


with T1_MANIFEST_PATH.open(
    "r",
    encoding="utf-8",
) as handle:
    manifest_readback = json.load(
        handle
    )


assert manifest_readback == manifest


# Delete only the unaccepted local temporary Parquet.
# The Drive artifact and raw T1 XML remain untouched.
LOCAL_OUTPUT_PATH.unlink(
    missing_ok=True
)


# --------------------------------------------------------------------------------------------------
# 17. Final result
# --------------------------------------------------------------------------------------------------

print("=" * 116)
print("STAGE 2D RESULT — PASS")
print("=" * 116)

print(
    "Complete ClinVarSet records scanned: "
    f"{scanned:,}"
)

print(
    "T1 target RCV records extracted: "
    f"{retained:,}"
)

print(
    "T1 nested SCVs: "
    f"{validation_summary['nested_scv_records']:,}"
)

print(
    "Gene counts: "
    f"{validation_summary['gene_counts']}"
)

print(
    "Classification-axis counts: "
    f"{validation_summary['classification_axis_counts']}"
)

print(
    f"Accepted T1 artifact: "
    f"{T1_OUTPUT_PATH}"
)

print(
    f"Accepted T1 SHA-256: "
    f"{output_sha256}"
)

print(
    f"T1 validation report: "
    f"{T1_VALIDATION_PATH}"
)

print(
    f"T1 freeze manifest: "
    f"{T1_MANIFEST_PATH}"
)

print(
    f"T1 freeze manifest SHA-256: "
    f"{manifest_sha256}"
)

print(
    f"T1 freeze manifest sidecar: "
    f"{manifest_sidecar}"
)

print()
print("BLUEPRINT STAGE 2: COMPLETE")
print(
    "BLUEPRINT STAGE 3 T0-T1 CROSSWALK: "
    "AUTHORIZED"
)
print(
    "Future-instability outcome construction: "
    "NOT YET AUTHORIZED"
)
print("Raw T1 XML deleted by this cell: False")
print("=" * 116)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
STAGE 2D — COMPLETE T1 TARGET-GENE RCV EXTRACTION, VALIDATION, AND FREEZE
Source: /content/clinvar_rcv_raw/ClinVarRCVRelease_2026-01.xml.gz
Output: /content/drive/MyDrive/GES_RAG_Temporal_Study/data_interim/t1_rcv_target_genes_harmonized_v1.parquet
T1 embedded cutoff: 2025-12-27

Prerequisite verification: PASS
Protocol amendment A002: /content/drive/MyDrive/GES_RAG_Temporal_Study/configs/protocol_amendment_A002_t1_embedded_cutoff.json

--------------------------------------------------------------------------------------------------------------------
COMPLETE T1 STREAMING EXTRACTION
--------------------------------------------------------------------------------------------------------------------
The 5.06 GiB compressed XML will now be scanned completely. Keep the runtime active.

Scanned 100,000 ClinVarSet records | retained 534 target RCVs | genes {'EGFR'

In [11]:
# ==================================================================================================
# PUBLISH GES-RAG TEMPORAL VALIDATION THROUGH BLUEPRINT STAGE 2
#
# Target repository:
#   SANGHATI23/mci-cardiomyopathy-concordance
#
# This cell:
#   1. Authenticates to GitHub using a securely entered Personal Access Token.
#   2. Clones the current main branch.
#   3. Creates an isolated ges_rag_temporal_validation/ project directory.
#   4. Copies compact, reproducible Stage 1/Stage 2 artifacts from Google Drive.
#   5. Copies available versioned Colab notebooks.
#   6. Generates a detailed Stage 2 completion report and artifact inventory.
#   7. Preserves and repairs the repository README if conflict markers are present.
#   8. Excludes raw multi-gigabyte ClinVar archives and secrets.
#   9. Commits and pushes to GitHub.
#
# Required GitHub token permission:
#   Fine-grained PAT: Repository access to this repository + Contents: Read and write
#   OR classic PAT: repo
#
# Security:
#   - The token is entered with getpass and is not printed.
#   - The token is not written into the Git remote URL.
#   - The temporary authentication script is deleted.
#   - Large raw XML and variant_summary source archives are not pushed.
# ==================================================================================================

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os
import re
import json
import stat
import shutil
import hashlib
import subprocess
from pathlib import Path
from datetime import datetime, timezone
from getpass import getpass


# --------------------------------------------------------------------------------------------------
# 1. GitHub and local paths
# --------------------------------------------------------------------------------------------------

GITHUB_USERNAME = "SANGHATI23"
REPOSITORY_NAME = "mci-cardiomyopathy-concordance"
REPOSITORY_URL = (
    f"https://github.com/{GITHUB_USERNAME}/{REPOSITORY_NAME}.git"
)
PUBLIC_REPOSITORY_URL = (
    f"https://github.com/{GITHUB_USERNAME}/{REPOSITORY_NAME}"
)
TARGET_BRANCH = "main"

timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
REPO_DIR = Path(
    f"/content/{REPOSITORY_NAME}_stage2_publish_{timestamp}"
)

DRIVE_ROOT = Path("/content/drive/MyDrive")
STUDY_ROOT = DRIVE_ROOT / "GES_RAG_Temporal_Study"
COLAB_NOTEBOOK_DIR = DRIVE_ROOT / "Colab Notebooks"

PROJECT_DIR_NAME = "ges_rag_temporal_validation"
PROJECT_DIR = REPO_DIR / PROJECT_DIR_NAME

PROJECT_CONFIG_DIR = PROJECT_DIR / "configs"
PROJECT_DATA_DIR = PROJECT_DIR / "data_interim"
PROJECT_QC_DIR = PROJECT_DIR / "outputs" / "quality_checks"
PROJECT_LOG_DIR = PROJECT_DIR / "outputs" / "logs"
PROJECT_NOTEBOOK_DIR = PROJECT_DIR / "notebooks"
PROJECT_DOCS_DIR = PROJECT_DIR / "docs"
PROJECT_ARCHIVE_DOCS_DIR = PROJECT_DOCS_DIR / "archive"

MAX_GITHUB_FILE_BYTES = 95 * 1024 * 1024
MAX_OPTIONAL_LOG_BYTES = 10 * 1024 * 1024

print("=" * 116)
print("PUBLISH GES-RAG TEMPORAL VALIDATION THROUGH BLUEPRINT STAGE 2")
print("=" * 116)
print(f"Target repository: {PUBLIC_REPOSITORY_URL}")
print(f"Target branch: {TARGET_BRANCH}")
print(f"Study source: {STUDY_ROOT}")
print()


# --------------------------------------------------------------------------------------------------
# 2. Verify the completed Stage 2 artifacts before touching GitHub
# --------------------------------------------------------------------------------------------------

required_artifacts = {
    "T0 accepted Parquet": (
        STUDY_ROOT
        / "data_interim"
        / "t0_rcv_target_genes_corrected_v1_2.parquet"
    ),
    "T0 consolidated freeze manifest": (
        STUDY_ROOT
        / "configs"
        / "clinvar_t0_consolidated_extraction_validation_manifest_v1_2.json"
    ),
    "T1 accepted Parquet": (
        STUDY_ROOT
        / "data_interim"
        / "t1_rcv_target_genes_harmonized_v1.parquet"
    ),
    "T1 field-validation report": (
        STUDY_ROOT
        / "outputs"
        / "quality_checks"
        / "clinvar_t1_target_gene_rcv_field_validation_v1.json"
    ),
    "T1 freeze manifest": (
        STUDY_ROOT
        / "configs"
        / "clinvar_t1_target_gene_rcv_extraction_validation_manifest_v1.json"
    ),
    "T1 protocol amendment A002": (
        STUDY_ROOT
        / "configs"
        / "protocol_amendment_A002_t1_embedded_cutoff.json"
    ),
}

missing_required = [
    f"{name}: {path}"
    for name, path in required_artifacts.items()
    if not path.exists()
]

if missing_required:
    raise FileNotFoundError(
        "Required completed Stage 2 artifacts are missing:\n\n"
        + "\n".join(missing_required)
    )

with required_artifacts["T0 consolidated freeze manifest"].open(
    "r", encoding="utf-8"
) as handle:
    t0_freeze = json.load(handle)

with required_artifacts["T1 freeze manifest"].open(
    "r", encoding="utf-8"
) as handle:
    t1_freeze = json.load(handle)

assert (
    t0_freeze["administrative_freeze_decision"]["decision"]
    == "ACCEPTED_AND_FROZEN"
), "T0 is not marked ACCEPTED_AND_FROZEN."

assert (
    t1_freeze["accepted_artifact"]["decision"]
    == "ACCEPTED_AND_FROZEN"
), "T1 is not marked ACCEPTED_AND_FROZEN."

assert (
    t1_freeze["stage_decision"]["blueprint_stage_2"]
    == "COMPLETE"
), "T1 manifest does not mark Blueprint Stage 2 complete."

assert (
    t1_freeze["stage_decision"]
    ["blueprint_stage_3_crosswalk_authorized"]
    is True
), "T1 manifest has not authorized Stage 3."

print("Stage 2 prerequisite verification: PASS")
print("T0 status: ACCEPTED_AND_FROZEN")
print("T1 status: ACCEPTED_AND_FROZEN")
print("Blueprint Stage 2: COMPLETE")
print("Blueprint Stage 3 crosswalk: AUTHORIZED")
print()


# --------------------------------------------------------------------------------------------------
# 3. Secure token entry and Git authentication helper
# --------------------------------------------------------------------------------------------------

github_token = getpass(
    "Enter your GitHub Personal Access Token "
    "(input is hidden and will not be printed): "
).strip()

if not github_token:
    raise ValueError("A GitHub Personal Access Token is required.")

git_author_name = (
    input("Git commit author name [Sanghati Basu]: ").strip()
    or "Sanghati Basu"
)

git_author_email = (
    input(
        "Git commit email "
        "[SANGHATI23@users.noreply.github.com]: "
    ).strip()
    or "SANGHATI23@users.noreply.github.com"
)

ASKPASS_PATH = Path(f"/tmp/github_askpass_{timestamp}.py")

ASKPASS_PATH.write_text(
    """#!/usr/bin/env python3
import os
import sys

prompt = sys.argv[1].lower() if len(sys.argv) > 1 else ""

if "username" in prompt:
    print(os.environ["GITHUB_AUTH_USERNAME"])
else:
    print(os.environ["GITHUB_AUTH_TOKEN"])
""",
    encoding="utf-8",
)

ASKPASS_PATH.chmod(
    ASKPASS_PATH.stat().st_mode
    | stat.S_IXUSR
)

git_env = os.environ.copy()
git_env.update(
    {
        "GIT_ASKPASS": str(ASKPASS_PATH),
        "GIT_TERMINAL_PROMPT": "0",
        "GITHUB_AUTH_USERNAME": GITHUB_USERNAME,
        "GITHUB_AUTH_TOKEN": github_token,
    }
)


def run_command(
    command,
    cwd=None,
    env=None,
    check=True,
    capture_output=False,
):
    """Run a command and raise a readable error when it fails."""
    result = subprocess.run(
        command,
        cwd=str(cwd) if cwd else None,
        env=env,
        check=False,
        text=True,
        capture_output=capture_output,
    )

    if check and result.returncode != 0:
        stdout = result.stdout or ""
        stderr = result.stderr or ""

        raise RuntimeError(
            "Command failed:\n"
            f"{' '.join(command)}\n\n"
            f"STDOUT:\n{stdout}\n\n"
            f"STDERR:\n{stderr}"
        )

    return result


# --------------------------------------------------------------------------------------------------
# 4. Clone the repository securely
# --------------------------------------------------------------------------------------------------

try:
    print("-" * 116)
    print("CLONING GITHUB REPOSITORY")
    print("-" * 116)

    run_command(
        [
            "git",
            "clone",
            "--branch",
            TARGET_BRANCH,
            "--single-branch",
            REPOSITORY_URL,
            str(REPO_DIR),
        ],
        env=git_env,
    )

    run_command(
        ["git", "config", "user.name", git_author_name],
        cwd=REPO_DIR,
    )

    run_command(
        ["git", "config", "user.email", git_author_email],
        cwd=REPO_DIR,
    )

    # Ensure the stored origin remains token-free.
    run_command(
        ["git", "remote", "set-url", "origin", REPOSITORY_URL],
        cwd=REPO_DIR,
    )

    print(f"Repository cloned: {REPO_DIR}")
    print()

    # ----------------------------------------------------------------------------------------------
    # 5. Build the isolated GES-RAG directory
    # ----------------------------------------------------------------------------------------------

    for directory in (
        PROJECT_CONFIG_DIR,
        PROJECT_DATA_DIR,
        PROJECT_QC_DIR,
        PROJECT_LOG_DIR,
        PROJECT_NOTEBOOK_DIR,
        PROJECT_DOCS_DIR,
        PROJECT_ARCHIVE_DOCS_DIR,
    ):
        directory.mkdir(parents=True, exist_ok=True)

    copied_files = []
    skipped_files = []
    warnings = []

    def sha256_file(path: Path, chunk_size=8 * 1024 * 1024):
        digest = hashlib.sha256()

        with path.open("rb") as handle:
            while True:
                chunk = handle.read(chunk_size)

                if not chunk:
                    break

                digest.update(chunk)

        return digest.hexdigest()

    def copy_file_with_policy(
        source: Path,
        destination: Path,
        category: str,
        maximum_bytes=MAX_GITHUB_FILE_BYTES,
    ):
        if not source.exists() or not source.is_file():
            return False

        size_bytes = source.stat().st_size

        if size_bytes > maximum_bytes:
            skipped_files.append(
                {
                    "source": str(source),
                    "reason": "file_exceeds_repository_size_policy",
                    "size_bytes": int(size_bytes),
                }
            )
            return False

        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, destination)

        copied_files.append(
            {
                "category": category,
                "source": str(source),
                "repository_path": str(
                    destination.relative_to(REPO_DIR)
                ),
                "size_bytes": int(destination.stat().st_size),
                "sha256": sha256_file(destination),
            }
        )

        return True

    # ----------------------------------------------------------------------------------------------
    # 6. Copy all configuration, manifest, checksum, and protocol records
    # ----------------------------------------------------------------------------------------------

    source_config_dir = STUDY_ROOT / "configs"

    for source in sorted(source_config_dir.rglob("*")):
        if not source.is_file():
            continue

        if source.suffix.lower() not in {
            ".json",
            ".sha256",
            ".txt",
            ".md",
            ".yaml",
            ".yml",
        }:
            continue

        relative_path = source.relative_to(source_config_dir)

        copy_file_with_policy(
            source,
            PROJECT_CONFIG_DIR / relative_path,
            "configuration_or_manifest",
        )

    # ----------------------------------------------------------------------------------------------
    # 7. Copy compact interim artifacts but never raw ClinVar archives
    # ----------------------------------------------------------------------------------------------

    source_data_dir = STUDY_ROOT / "data_interim"

    allowed_data_suffixes = {
        ".parquet",
        ".csv",
        ".tsv",
        ".json",
        ".sha256",
    }

    forbidden_raw_patterns = (
        "clinvarfullrelease",
        "clinvarrcvrelease",
        "variant_summary",
        ".xml.gz",
    )

    for source in sorted(source_data_dir.rglob("*")):
        if not source.is_file():
            continue

        lower_name = source.name.lower()

        if source.suffix.lower() not in allowed_data_suffixes:
            continue

        if any(
            pattern in lower_name
            for pattern in forbidden_raw_patterns
        ):
            skipped_files.append(
                {
                    "source": str(source),
                    "reason": "raw_source_archive_excluded",
                    "size_bytes": int(source.stat().st_size),
                }
            )
            continue

        relative_path = source.relative_to(source_data_dir)

        copy_file_with_policy(
            source,
            PROJECT_DATA_DIR / relative_path,
            "compact_interim_data",
        )

    # ----------------------------------------------------------------------------------------------
    # 8. Copy validation reports and lightweight logs
    # ----------------------------------------------------------------------------------------------

    source_qc_dir = (
        STUDY_ROOT
        / "outputs"
        / "quality_checks"
    )

    if source_qc_dir.exists():
        for source in sorted(source_qc_dir.rglob("*")):
            if not source.is_file():
                continue

            if source.suffix.lower() not in {
                ".json",
                ".sha256",
                ".txt",
                ".csv",
                ".tsv",
                ".md",
            }:
                # Representative XML samples remain excluded from Git.
                continue

            relative_path = source.relative_to(source_qc_dir)

            copy_file_with_policy(
                source,
                PROJECT_QC_DIR / relative_path,
                "quality_control",
            )

    source_log_dir = STUDY_ROOT / "outputs" / "logs"

    if source_log_dir.exists():
        for source in sorted(source_log_dir.rglob("*")):
            if not source.is_file():
                continue

            if source.suffix.lower() not in {
                ".json",
                ".txt",
                ".log",
                ".csv",
                ".tsv",
                ".md",
            }:
                continue

            relative_path = source.relative_to(source_log_dir)

            copy_file_with_policy(
                source,
                PROJECT_LOG_DIR / relative_path,
                "execution_log",
                maximum_bytes=MAX_OPTIONAL_LOG_BYTES,
            )

    # ----------------------------------------------------------------------------------------------
    # 9. Copy all available versioned temporal-validation notebooks
    # ----------------------------------------------------------------------------------------------

    notebook_candidates = []

    if COLAB_NOTEBOOK_DIR.exists():
        notebook_candidates.extend(
            COLAB_NOTEBOOK_DIR.glob(
                "02_GES_temporal_validation*.ipynb"
            )
        )

    notebook_candidates.extend(
        STUDY_ROOT.rglob(
            "02_GES_temporal_validation*.ipynb"
        )
    )

    unique_notebook_candidates = sorted(
        {
            candidate.resolve()
            for candidate in notebook_candidates
            if candidate.is_file()
        },
        key=lambda path: path.name,
    )

    for source in unique_notebook_candidates:
        copy_file_with_policy(
            source,
            PROJECT_NOTEBOOK_DIR / source.name,
            "notebook",
        )

    if not unique_notebook_candidates:
        warnings.append(
            "No saved 02_GES_temporal_validation*.ipynb notebook "
            "was found. Save the active Colab notebook to Drive and "
            "rerun this publication cell to include it."
        )

    # ----------------------------------------------------------------------------------------------
    # 10. Preserve the available T0 technical report as a historical document
    # ----------------------------------------------------------------------------------------------

    report_candidates = []

    likely_report_locations = [
        STUDY_ROOT,
        DRIVE_ROOT,
        DRIVE_ROOT / "Colab Notebooks",
    ]

    for search_root in likely_report_locations:
        if not search_root.exists():
            continue

        try:
            report_candidates.extend(
                search_root.glob(
                    "GES_T0_Temporal_Validation_Workflow_Report*.docx"
                )
            )
            report_candidates.extend(
                search_root.glob(
                    "**/GES_T0_Temporal_Validation_Workflow_Report*.docx"
                )
            )
        except OSError:
            pass

    report_candidates = [
        path
        for path in set(report_candidates)
        if path.is_file()
    ]

    if report_candidates:
        latest_report = max(
            report_candidates,
            key=lambda path: path.stat().st_mtime,
        )

        archived_report_name = (
            "GES_T0_Temporal_Validation_Workflow_Report_"
            "historical_pre_Stage2_completion.docx"
        )

        copy_file_with_policy(
            latest_report,
            PROJECT_ARCHIVE_DOCS_DIR / archived_report_name,
            "historical_technical_report",
            maximum_bytes=50 * 1024 * 1024,
        )
    else:
        warnings.append(
            "The historical T0 DOCX report was not found in Google Drive. "
            "The complete current status is still documented in Markdown."
        )

    # ----------------------------------------------------------------------------------------------
    # 11. Create a structured Stage 2 completion summary
    # ----------------------------------------------------------------------------------------------

    stage2_summary = {
        "project": (
            "GES-RAG: Temporal Validation and Stability-Aware "
            "Context Assembly for Reliable Genomic Question Answering"
        ),
        "created_utc": (
            datetime.now(timezone.utc)
            .replace(microsecond=0)
            .isoformat()
        ),
        "repository_location": (
            f"{REPOSITORY_NAME}/{PROJECT_DIR_NAME}"
        ),
        "blueprint_status": {
            "stage_1": "COMPLETE",
            "stage_2": "COMPLETE",
            "stage_3_crosswalk": "AUTHORIZED_NOT_STARTED",
            "future_instability_outcomes": "NOT_AUTHORIZED",
            "ges_model_fitting": "NOT_STARTED",
            "rag_experiment": "NOT_STARTED",
        },
        "t0": {
            "release_label": "2023-01",
            "embedded_cutoff": "2022-12-31",
            "xml_records_scanned": 2_302_323,
            "rcv_records": 71_659,
            "nested_scv_records": 100_633,
            "gene_counts": {
                "BRCA1": 25_408,
                "BRCA2": 34_915,
                "MLH1": 8_936,
                "EGFR": 2_400,
            },
            "corrected_conflict_positive_records": 1_484,
            "false_conflict_positives_corrected": 9_224,
            "rcvs_with_primary_orgid": 71_659,
            "scvs_with_primary_orgid": 100_633,
            "source_consistent_empty_condition_id_rows": 144,
            "accepted_parquet_sha256": (
                "f6b6760b2ad6e4352e3bdecdeaf89827"
                "e8a514b031abf2373bb17568d999466d"
            ),
            "consolidated_manifest_sha256": (
                "66d4d84b5dea9617055d44cf0c2c91e4"
                "b008d31c85dde3d591c5c7beaf010a10"
            ),
        },
        "t1": {
            "release_label": "2026-01",
            "embedded_cutoff": "2025-12-27",
            "schema": "ClinVar_RCV_2.2.xsd",
            "compressed_source_bytes": 5_434_707_247,
            "source_md5": "5740de7f8f74a49ba8c58e3ec1b8cc26",
            "source_sha256": (
                "3fba206f1e3086306472ab7b0ae324d4"
                "ae516da84857038d03f2937fd20a6e55"
            ),
            "xml_records_scanned": 5_584_799,
            "rcv_records": 100_920,
            "nested_scv_records": 145_400,
            "gene_counts": {
                "BRCA1": 32_603,
                "BRCA2": 49_221,
                "MLH1": 13_684,
                "EGFR": 5_412,
            },
            "classification_axis_counts": {
                "GermlineClassification": 97_526,
                "OncogenicityClassification": 52,
                "SomaticClinicalImpact": 25,
                "NoClassification": 3_317,
            },
            "aggregate_conflict_positive_records": 6_602,
            "empty_condition_id_rows": 730,
            "scvs_missing_primary_orgid": 0,
            "critical_validation_failures": 0,
            "accepted_parquet_sha256": (
                "5713a11bdbf4804758cc011f9b2f302a"
                "fc91fa1f88c1b178d675c28bb277d37c"
            ),
            "freeze_manifest_sha256": (
                "7eaeff0fee3df96973130f721d6c1f2a0"
                "2fd9108e85af7b2743cdd7750a4372e"
            ),
        },
        "data_leakage_controls": {
            "t1_used_to_modify_t0": False,
            "t1_used_to_fit_or_tune_ges": False,
            "future_outcomes_created": False,
            "unmatched_records_called_stable": False,
            "stage_3_linkage_started": False,
        },
        "excluded_from_github": [
            "ClinVarFullRelease_2023-01.xml.gz",
            "ClinVarRCVRelease_2026-01.xml.gz",
            "large variant_summary archives",
            "GitHub Personal Access Tokens",
            "temporary Colab files",
        ],
    }

    stage2_summary_path = (
        PROJECT_CONFIG_DIR
        / "stage2_completion_summary_v1.json"
    )

    stage2_summary_path.write_text(
        json.dumps(
            stage2_summary,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
        )
        + "\n",
        encoding="utf-8",
    )

    # ----------------------------------------------------------------------------------------------
    # 12. Generate the detailed current technical report
    # ----------------------------------------------------------------------------------------------

    detailed_report = """
# GES-RAG Temporal Validation — Blueprint Stage 2 Completion Report

## Current scientific position

The release-acquisition, provenance, condition-specific extraction, schema
harmonization, and field-validation phase is complete for both T0 and T1.

The project is now authorized to begin **Blueprint Stage 3: the T0–T1
RCV-level variant-condition crosswalk**.

Future-instability outcomes have not been created, and the GES model has not
been fitted or tuned. Stage 3 authorization is limited to record linkage,
cross-release compatibility assessment, and unmatched/merge/split auditing.

## Research objective

The broader study asks:

> Can a genomic evidence-stability score calculated from an earlier ClinVar
> release predict later interpretation instability, and can that validated
> signal improve genomic RAG evidence selection, uncertainty communication,
> conflict recognition, and safe abstention?

The primary unit remains one **RCV-level variant-condition aggregate**.
Variant-level and condition-level units must not be mixed.

Primary genes:

- BRCA1
- BRCA2
- MLH1

Separate exploratory gene:

- EGFR

## Why the RCV XML source was required

The original `variant_summary` audit showed that it was not one row per
variant-condition relationship. In the January 2023 target-gene subset,
45.23% of rows contained multiple RCV accessions; in January 2026, the
corresponding proportion was 52.52%.

Exploding those rows while copying one aggregate classification would have
incorrectly assigned one variant-level interpretation to several conditions.
The workflow therefore selected ClinVar's condition-specific RCV XML releases.

## T0 baseline cohort

### Provenance

- Archive label: January 2023
- Archive publication date: 2023-01-05
- Embedded XML cutoff: 2022-12-31
- Source: `ClinVarFullRelease_2023-01.xml.gz`
- Historical XML schema: `clinvar_public_1.69.xsd`
- Source SHA-256:
  `911c8a58872ea89cc7bb4f1ee3362596d965f5103422b30f456abaf99c41e5e7`

The embedded XML date, rather than the later HTTP `Last-Modified` value, was
used as the temporal boundary.

### Extraction and cohort size

The historical archive was parsed with streaming `lxml.iterparse` without
writing a second decompressed XML file.

- Complete ClinVarSet records scanned: **2,302,323**
- Target-gene RCV records retained: **71,659**
- Nested SCV assertions: **100,633**
- Missing mandatory RCV accessions: **0**
- Missing VariationIDs: **0**

Gene counts:

| Gene | T0 RCV records |
|---|---:|
| BRCA1 | 25,408 |
| BRCA2 | 34,915 |
| MLH1 | 8,936 |
| EGFR | 2,400 |
| **Total** | **71,659** |

### T0 semantic corrections

Two defects were found through prespecified field validation before outcome
construction or model fitting.

#### Aggregate-conflict semantic correction

The original parser treated any review status containing the text `conflict`
as conflict-positive. This also matched `multiple submitters, no conflicts`.

- Original conflict-positive count: **10,708**
- False positives corrected: **9,224**
- Final conflict-positive count: **1,484**
- False-to-true changes: **0**

The correction used exact semantic categories, and SCV group disagreement
remained a separate variable.

#### Submitter organization-ID correction

The historical parser searched `ClinVarSubmissionID` for a submitter
identifier. In the historical schema, the primary organization identifier is
stored in `ClinVarAccession[@Type="SCV"]/@OrgID`.

A complete source audit recovered:

- Primary OrgIDs for **100,633 of 100,633 SCVs**
- Primary OrgIDs for **71,659 of 71,659 RCVs**
- Unique primary organization IDs: **262**

The corrected parser was executed across all 2,302,323 source records with
zero RCV, SCV, submitter-name, or submitter-ID mismatches.

### T0 accepted artifact

- Artifact:
  `data_interim/t0_rcv_target_genes_corrected_v1_2.parquet`
- Rows: **71,659**
- Columns: **34**
- SHA-256:
  `f6b6760b2ad6e4352e3bdecdeaf89827e8a514b031abf2373bb17568d999466d`
- Administrative decision: **ACCEPTED_AND_FROZEN**

The 144 rows without structured condition identifiers were confirmed as
source-consistent missingness and retained without imputation.

## T1 follow-up cohort

### Download and source verification

- Archive label: January 2026
- Filename: `ClinVarRCVRelease_2026-01.xml.gz`
- Compressed size: **5,434,707,247 bytes**
- Official and observed MD5:
  `5740de7f8f74a49ba8c58e3ec1b8cc26`
- SHA-256:
  `3fba206f1e3086306472ab7b0ae324d4ae516da84857038d03f2937fd20a6e55`
- Complete gzip-stream integrity: **PASS**

The raw archive was retained only in temporary Colab storage and was not
committed to GitHub.

### T1 root and schema provenance

The XML root was:

- Element: `ReleaseSet`
- `Dated`: `2025-12-27`
- `Type`: `full`
- Schema: `ClinVar_RCV_2.2.xsd`

Protocol Amendment A002 froze **2025-12-27** as the T1 embedded data cutoff
before complete extraction and before outcome construction.

### Complete T1 extraction

- Complete ClinVarSet records scanned: **5,584,799**
- Target RCV records retained: **100,920**
- Nested SCV records validated: **145,400**
- Critical validation failures: **0**
- SCVs missing primary OrgID: **0**

Gene counts:

| Gene | T1 RCV records |
|---|---:|
| BRCA1 | 32,603 |
| BRCA2 | 49,221 |
| MLH1 | 13,684 |
| EGFR | 5,412 |
| **Total** | **100,920** |

Current-schema classification axes:

| Classification axis | RCV records |
|---|---:|
| GermlineClassification | 97,526 |
| OncogenicityClassification | 52 |
| SomaticClinicalImpact | 25 |
| NoClassification | 3,317 |

Additional findings:

- Aggregate conflict-positive RCVs: **6,602**
- Rows with empty structured condition IDs: **730**
- T1 field-validation decision: **PASS**

The parser preserved all current classification axes in the nested JSON
fields. Axis compatibility must be enforced explicitly during cross-release
linkage and outcome construction, particularly for EGFR.

### T1 accepted artifact

- Artifact:
  `data_interim/t1_rcv_target_genes_harmonized_v1.parquet`
- Rows: **100,920**
- Harmonized T0-compatible fields: **34**
- T1 schema-extension fields: **2**
- SHA-256:
  `5713a11bdbf4804758cc011f9b2f302afc91fa1f88c1b178d675c28bb277d37c`
- Administrative decision: **ACCEPTED_AND_FROZEN**

## Validation coverage

The completed T0 and T1 workflow includes:

- Source URL and archive-size recording
- Official MD5 and SHA-256 verification
- Gzip integrity testing
- Embedded cutoff inspection
- XML schema inspection
- Streaming source parsing
- RCV key and mandatory-identifier validation
- Gene and study-scope validation
- Review-star and classification-group normalization
- Exact conflict-semantic validation
- Nested SCV count reconciliation
- Submitter-name and organization-ID reconciliation
- Classification distribution reconciliation
- Aggregate and SCV date-boundary validation
- Condition-name and condition-ID reconstruction
- Deterministic JSON serialization
- Local-to-Drive checksum verification
- Versioned protocol amendments and manifests
- Administrative freeze decisions

## Leakage protections maintained

At the completion of Stage 2:

- T1 data have not been used to alter the accepted T0 cohort.
- T1 outcomes have not been used to fit, tune, or threshold GES.
- Future-instability outcomes have not been created.
- Unmatched T0 records have not been labeled stable.
- The T0–T1 crosswalk has not yet been created.
- No RAG results have been used to modify the temporal design.

## Current blueprint status

| Blueprint stage | Status |
|---|---|
| 1. Freeze questions, outcomes, genes, and exclusions | Complete |
| 2. Acquire, verify, extract, validate, and freeze T0/T1 releases | **Complete** |
| 3. Build the T0–T1 RCV variant-condition crosswalk | Authorized; not started |
| 4. Reconstruct and freeze T0 GES | Not started |
| 5. Generate future-instability outcomes | Not started |
| 6. Run locked temporal validation | Not started |
| 7–13. Build and evaluate GES-aware genomic RAG | Not started |

## Next step

Stage 3 must:

1. Match exact RCV accessions first.
2. Audit RCV version changes.
3. Preserve the variant-condition unit.
4. Evaluate condition-name and structured-identifier compatibility.
5. Inspect VCV and VariationID continuity.
6. Separate current classification axes.
7. Identify unmatched, withdrawn, merged, split, and changed-association cases.
8. Keep unmatched records separate rather than labeling them stable.
9. Freeze the crosswalk and its linkage rules before creating outcomes.

## Data distribution policy

The following are intentionally excluded from GitHub:

- Multi-gigabyte raw ClinVar XML archives
- Large `variant_summary` source archives
- Temporary Colab files
- Authentication tokens
- Any file exceeding the repository's safe size policy

Compact Parquet artifacts, manifests, validation reports, protocol amendments,
checksums, notebooks, and documentation are included for reproducibility.
""".strip() + "\n"

    detailed_report_path = (
        PROJECT_DOCS_DIR
        / "STAGE_2_COMPLETION_REPORT.md"
    )

    detailed_report_path.write_text(
        detailed_report,
        encoding="utf-8",
    )

    # ----------------------------------------------------------------------------------------------
    # 13. Create the project README
    # ----------------------------------------------------------------------------------------------

    project_readme = """
# GES-RAG Temporal Validation

## Study title

**GES-RAG: Temporal Validation and Stability-Aware Context Assembly for
Reliable Genomic Question Answering**

This directory contains the reproducible artifacts produced through
**Blueprint Stage 2** of the GES-RAG study.

## Current milestone

- T0 January 2023 RCV cohort: validated and frozen
- T1 January 2026 RCV cohort: validated and frozen
- Blueprint Stage 2: complete
- Blueprint Stage 3 crosswalk: authorized but not started
- Future-instability outcomes: not created
- GES model: not fitted
- RAG intervention: not started

## Cohort summary

| Timepoint | Embedded cutoff | RCVs | SCVs |
|---|---:|---:|---:|
| T0 | 2022-12-31 | 71,659 | 100,633 |
| T1 | 2025-12-27 | 100,920 | 145,400 |

## Principal completed work

1. Rejected invalid direct condition-level use of `variant_summary`.
2. Selected condition-specific RCV XML releases.
3. Verified source sizes, checksums, gzip integrity, dates, and schemas.
4. Stream-extracted BRCA1, BRCA2, MLH1, and EGFR RCV records.
5. Corrected 9,224 false T0 conflict-positive records.
6. Recovered source OrgIDs for all T0 submitted SCVs.
7. Validated all T0 and T1 nested SCV records and count distributions.
8. Preserved current T1 classification axes.
9. Froze consolidated T0 and T1 acceptance manifests.
10. Maintained the prespecified data-leakage boundary.

## Directory structure

```text
ges_rag_temporal_validation/
├── configs/
│   ├── protocols
│   ├── protocol amendments
│   ├── download receipts
│   ├── schema probes
│   ├── correction manifests
│   ├── runtime-validation manifests
│   └── consolidated freeze manifests
├── data_interim/
│   ├── compact T0 Parquet artifacts
│   ├── compact T1 Parquet artifact
│   └── source-derived mapping artifacts
├── notebooks/
│   └── versioned temporal-validation notebooks
├── outputs/
│   ├── quality_checks/
│   └── logs/
├── docs/
│   ├── STAGE_2_COMPLETION_REPORT.md
│   └── archive/
└── ARTIFACT_INDEX.tsv

SyntaxError: incomplete input (1436222129.py, line 1040)

In [ ]:
# =================================================================================================
# STAGE 3A — VERIFY FROZEN T0/T1 INPUTS BEFORE CROSS-RELEASE LINKAGE
# =================================================================================================
# Purpose:
#   1. Mount Google Drive.
#   2. Locate the administratively frozen T0 and T1 Parquet artifacts.
#   3. Recalculate and verify SHA-256 checksums.
#   4. Verify row counts, column counts, required fields, and unique RCV keys.
#   5. Create a checksum-controlled Stage 3 input-verification receipt.
#
# This cell DOES NOT:
#   - construct the T0–T1 crosswalk,
#   - create future-instability outcomes,
#   - label unmatched records as stable,
#   - fit or tune GES.
# =================================================================================================

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import os

import pandas as pd
import pyarrow.parquet as pq


print("=" * 112)
print("STAGE 3A — FROZEN T0/T1 INPUT VERIFICATION")
print("=" * 112)


# -------------------------------------------------------------------------------------------------
# 1. FROZEN STUDY PATHS
# -------------------------------------------------------------------------------------------------

STUDY_ROOT = Path("/content/drive/MyDrive/GES_RAG_Temporal_Study")

T0_PATH = (
    STUDY_ROOT
    / "data_interim"
    / "t0_rcv_target_genes_corrected_v1_2.parquet"
)

T1_PATH = (
    STUDY_ROOT
    / "data_interim"
    / "t1_rcv_target_genes_harmonized_v1.parquet"
)

STAGE3_CONFIG_DIR = STUDY_ROOT / "configs" / "stage3_crosswalk"
STAGE3_OUTPUT_DIR = STUDY_ROOT / "data_processed" / "stage3_crosswalk"

STAGE3_CONFIG_DIR.mkdir(parents=True, exist_ok=True)
STAGE3_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RECEIPT_PATH = (
    STAGE3_CONFIG_DIR
    / "stage3_t0_t1_input_verification_receipt_v1.json"
)


# -------------------------------------------------------------------------------------------------
# 2. EXPECTED ACCEPTED-ARTIFACT VALUES FROM THE STAGE 2 FREEZE
# -------------------------------------------------------------------------------------------------

EXPECTED = {
    "T0": {
        "path": T0_PATH,
        "sha256": (
            "f6b6760b2ad6e4352e3bdecdeaf89827"
            "e8a514b031abf2373bb17568d999466d"
        ),
        "rows": 71659,
        "columns": 34,
        "expected_timepoint": "T0",
        "expected_cutoff": "2022-12-31",
    },
    "T1": {
        "path": T1_PATH,
        "sha256": (
            "5713a11bdbf4804758cc011f9b2f302a"
            "fc91fa1f88c1b178d675c28bb277d37c"
        ),
        "rows": 100920,
        "columns": 36,
        "expected_timepoint": "T1",
        "expected_cutoff": "2025-12-27",
    },
}


SHARED_REQUIRED_COLUMNS = {
    "timepoint",
    "release_label",
    "embedded_data_cutoff_date",
    "source_filename",
    "source_sha256",
    "rcv_accession",
    "rcv_version",
    "variation_id",
    "vcv_accession",
    "vcv_version",
    "target_genes_json",
    "study_scope",
    "condition_names_json",
    "condition_ids_json",
    "trait_records_json",
    "aggregate_classification",
    "aggregate_classification_group",
    "aggregate_review_status",
    "aggregate_review_stars",
    "aggregate_conflict_flag",
    "scv_group_disagreement_flag",
    "scv_count_xml",
    "unique_submitter_count_xml",
    "submitters_json",
    "submitter_ids_json",
    "scv_records_json",
}

T1_EXTENSION_COLUMNS = {
    "aggregate_classification_axis",
    "aggregate_classifications_json",
}


# -------------------------------------------------------------------------------------------------
# 3. UTILITY FUNCTIONS
# -------------------------------------------------------------------------------------------------

def sha256_file(path: Path, chunk_size: int = 8 * 1024 * 1024) -> str:
    """Calculate SHA-256 without loading the entire file into memory."""
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)

    return digest.hexdigest()


def normalize_scalar(value):
    """Normalize a value for stable comparison and JSON serialization."""
    if pd.isna(value):
        return None
    return str(value).strip()


def verify_artifact(label: str, specification: dict) -> dict:
    """Verify one frozen Parquet artifact and return a structured result."""

    path = specification["path"]
    failures = []

    result = {
        "label": label,
        "path": str(path),
        "exists": path.exists(),
        "failures": failures,
    }

    if not path.exists():
        failures.append(f"Required artifact does not exist: {path}")
        result["decision"] = "FAIL"
        return result

    # File-level verification
    observed_size = path.stat().st_size
    observed_sha256 = sha256_file(path)

    parquet_file = pq.ParquetFile(path)
    observed_rows = int(parquet_file.metadata.num_rows)
    observed_columns = int(parquet_file.metadata.num_columns)
    observed_row_groups = int(parquet_file.metadata.num_row_groups)
    schema_columns = list(parquet_file.schema_arrow.names)

    result.update(
        {
            "file_size_bytes": int(observed_size),
            "expected_sha256": specification["sha256"],
            "observed_sha256": observed_sha256,
            "sha256_match": observed_sha256 == specification["sha256"],
            "expected_rows": int(specification["rows"]),
            "observed_rows": observed_rows,
            "row_count_match": observed_rows == specification["rows"],
            "expected_columns": int(specification["columns"]),
            "observed_columns": observed_columns,
            "column_count_match": observed_columns == specification["columns"],
            "row_groups": observed_row_groups,
            "schema_columns": schema_columns,
        }
    )

    if not result["sha256_match"]:
        failures.append("SHA-256 does not match the accepted Stage 2 checksum.")

    if not result["row_count_match"]:
        failures.append(
            f"Row count mismatch: expected {specification['rows']:,}, "
            f"observed {observed_rows:,}."
        )

    if not result["column_count_match"]:
        failures.append(
            f"Column count mismatch: expected {specification['columns']}, "
            f"observed {observed_columns}."
        )

    missing_shared_columns = sorted(
        SHARED_REQUIRED_COLUMNS - set(schema_columns)
    )

    result["missing_shared_required_columns"] = missing_shared_columns

    if missing_shared_columns:
        failures.append(
            "Missing shared required columns: "
            + ", ".join(missing_shared_columns)
        )

    if label == "T1":
        missing_t1_extensions = sorted(
            T1_EXTENSION_COLUMNS - set(schema_columns)
        )
        result["missing_t1_extension_columns"] = missing_t1_extensions

        if missing_t1_extensions:
            failures.append(
                "Missing T1 classification-axis extension columns: "
                + ", ".join(missing_t1_extensions)
            )

    # Read only the columns needed for Stage 3 prerequisite checks.
    probe_columns = [
        "timepoint",
        "embedded_data_cutoff_date",
        "rcv_accession",
        "rcv_version",
        "variation_id",
        "vcv_accession",
        "vcv_version",
        "target_genes_json",
        "condition_names_json",
        "condition_ids_json",
    ]

    if label == "T1":
        probe_columns.append("aggregate_classification_axis")

    probe = pd.read_parquet(path, columns=probe_columns)

    rcv_text = probe["rcv_accession"].astype("string").str.strip().str.upper()

    missing_rcv = int(rcv_text.isna().sum())
    blank_rcv = int(rcv_text.fillna("").eq("").sum())
    duplicate_rcv = int(rcv_text.duplicated(keep=False).sum())
    unique_rcv = int(rcv_text.nunique(dropna=True))
    malformed_rcv = int(
        (~rcv_text.fillna("").str.fullmatch(r"RCV\d+")).sum()
    )

    timepoints = sorted(
        {
            normalize_scalar(value)
            for value in probe["timepoint"].dropna().unique().tolist()
        }
    )

    cutoffs = sorted(
        {
            normalize_scalar(value)
            for value in probe[
                "embedded_data_cutoff_date"
            ].dropna().unique().tolist()
        }
    )

    result.update(
        {
            "unique_rcv_accessions": unique_rcv,
            "missing_rcv_accessions": missing_rcv,
            "blank_rcv_accessions": blank_rcv,
            "rows_in_duplicate_rcv_groups": duplicate_rcv,
            "malformed_rcv_accessions": malformed_rcv,
            "observed_timepoints": timepoints,
            "observed_embedded_cutoffs": cutoffs,
            "expected_timepoint": specification["expected_timepoint"],
            "expected_embedded_cutoff": specification["expected_cutoff"],
        }
    )

    if missing_rcv != 0:
        failures.append(f"{missing_rcv:,} RCV accessions are missing.")

    if blank_rcv != 0:
        failures.append(f"{blank_rcv:,} RCV accessions are blank.")

    if duplicate_rcv != 0:
        failures.append(
            f"{duplicate_rcv:,} rows belong to duplicate RCV-accession groups."
        )

    if malformed_rcv != 0:
        failures.append(
            f"{malformed_rcv:,} RCV accessions do not match the expected format."
        )

    if unique_rcv != observed_rows:
        failures.append(
            f"Unique RCV count ({unique_rcv:,}) does not equal "
            f"the Parquet row count ({observed_rows:,})."
        )

    if timepoints != [specification["expected_timepoint"]]:
        failures.append(
            f"Unexpected timepoint values: {timepoints}"
        )

    if cutoffs != [specification["expected_cutoff"]]:
        failures.append(
            f"Unexpected embedded cutoff values: {cutoffs}"
        )

    result["decision"] = "PASS" if not failures else "FAIL"

    return result


# -------------------------------------------------------------------------------------------------
# 4. EXECUTE T0 AND T1 VERIFICATION
# -------------------------------------------------------------------------------------------------

verification_results = {}

for label, specification in EXPECTED.items():
    print()
    print("-" * 112)
    print(f"{label} ACCEPTED ARTIFACT")
    print("-" * 112)
    print(f"Path: {specification['path']}")

    artifact_result = verify_artifact(label, specification)
    verification_results[label] = artifact_result

    print(f"Exists:                    {artifact_result['exists']}")

    if artifact_result["exists"]:
        print(
            f"Rows:                      "
            f"{artifact_result['observed_rows']:,} "
            f"(expected {artifact_result['expected_rows']:,})"
        )
        print(
            f"Columns:                   "
            f"{artifact_result['observed_columns']} "
            f"(expected {artifact_result['expected_columns']})"
        )
        print(
            f"Row groups:                "
            f"{artifact_result['row_groups']}"
        )
        print(
            f"Unique RCV accessions:     "
            f"{artifact_result['unique_rcv_accessions']:,}"
        )
        print(
            f"Duplicate-RCV rows:        "
            f"{artifact_result['rows_in_duplicate_rcv_groups']:,}"
        )
        print(
            f"Malformed RCV accessions:  "
            f"{artifact_result['malformed_rcv_accessions']:,}"
        )
        print(
            f"Observed timepoint:        "
            f"{artifact_result['observed_timepoints']}"
        )
        print(
            f"Observed cutoff:           "
            f"{artifact_result['observed_embedded_cutoffs']}"
        )
        print(
            f"SHA-256 match:             "
            f"{artifact_result['sha256_match']}"
        )
        print(
            f"Observed SHA-256:          "
            f"{artifact_result['observed_sha256']}"
        )

    print(f"Decision:                  {artifact_result['decision']}")

    if artifact_result["failures"]:
        print("Failures:")
        for failure in artifact_result["failures"]:
            print(f"  - {failure}")


# -------------------------------------------------------------------------------------------------
# 5. WRITE THE STAGE 3 INPUT-VERIFICATION RECEIPT
# -------------------------------------------------------------------------------------------------

overall_pass = all(
    result["decision"] == "PASS"
    for result in verification_results.values()
)

receipt = {
    "receipt_name": "Stage 3 T0-T1 Input Verification Receipt",
    "receipt_version": "1.0.0",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "study_stage": "Stage 3A",
    "purpose": (
        "Verify the frozen T0 and T1 RCV-level artifacts before constructing "
        "the cross-release variant-condition crosswalk."
    ),
    "scientific_unit": "RCV-level variant-condition aggregate",
    "linkage_started": False,
    "future_instability_outcomes_created": False,
    "ges_model_fitted_or_tuned": False,
    "unmatched_records_labeled_stable": False,
    "verification_results": verification_results,
    "overall_decision": (
        "PASS_STAGE3_INPUTS_VERIFIED"
        if overall_pass
        else "FAIL_STAGE3_INPUTS_NOT_VERIFIED"
    ),
}

with RECEIPT_PATH.open("w", encoding="utf-8") as handle:
    json.dump(receipt, handle, indent=2, sort_keys=True)

receipt_sha256 = sha256_file(RECEIPT_PATH)

print()
print("=" * 112)
print("STAGE 3A SUMMARY")
print("=" * 112)
print(f"Receipt:        {RECEIPT_PATH}")
print(f"Receipt SHA-256:{receipt_sha256}")
print(f"Overall result: {receipt['overall_decision']}")
print("=" * 112)


# -------------------------------------------------------------------------------------------------
# 6. STOP THE WORKFLOW IF ANY FROZEN INPUT FAILED VERIFICATION
# -------------------------------------------------------------------------------------------------

if not overall_pass:
    raise RuntimeError(
        "Stage 3 is blocked because one or more accepted inputs failed verification. "
        "Review the printed failures before constructing the crosswalk."
    )

print()
print("PASS — The frozen T0 and T1 artifacts are verified.")
print("AUTHORIZED NEXT ACTION — Stage 3B exact RCV-accession crosswalk construction.")

In [12]:
# =================================================================================================
# STAGE 3A — VERIFY FROZEN T0/T1 INPUTS BEFORE CROSS-RELEASE LINKAGE
# =================================================================================================
# Purpose:
#   1. Mount Google Drive.
#   2. Locate the administratively frozen T0 and T1 Parquet artifacts.
#   3. Recalculate and verify SHA-256 checksums.
#   4. Verify row counts, column counts, required fields, and unique RCV keys.
#   5. Create a checksum-controlled Stage 3 input-verification receipt.
#
# This cell DOES NOT:
#   - construct the T0–T1 crosswalk,
#   - create future-instability outcomes,
#   - label unmatched records as stable,
#   - fit or tune GES.
# =================================================================================================

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import os

import pandas as pd
import pyarrow.parquet as pq


print("=" * 112)
print("STAGE 3A — FROZEN T0/T1 INPUT VERIFICATION")
print("=" * 112)


# -------------------------------------------------------------------------------------------------
# 1. FROZEN STUDY PATHS
# -------------------------------------------------------------------------------------------------

STUDY_ROOT = Path("/content/drive/MyDrive/GES_RAG_Temporal_Study")

T0_PATH = (
    STUDY_ROOT
    / "data_interim"
    / "t0_rcv_target_genes_corrected_v1_2.parquet"
)

T1_PATH = (
    STUDY_ROOT
    / "data_interim"
    / "t1_rcv_target_genes_harmonized_v1.parquet"
)

STAGE3_CONFIG_DIR = STUDY_ROOT / "configs" / "stage3_crosswalk"
STAGE3_OUTPUT_DIR = STUDY_ROOT / "data_processed" / "stage3_crosswalk"

STAGE3_CONFIG_DIR.mkdir(parents=True, exist_ok=True)
STAGE3_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RECEIPT_PATH = (
    STAGE3_CONFIG_DIR
    / "stage3_t0_t1_input_verification_receipt_v1.json"
)


# -------------------------------------------------------------------------------------------------
# 2. EXPECTED ACCEPTED-ARTIFACT VALUES FROM THE STAGE 2 FREEZE
# -------------------------------------------------------------------------------------------------

EXPECTED = {
    "T0": {
        "path": T0_PATH,
        "sha256": (
            "f6b6760b2ad6e4352e3bdecdeaf89827"
            "e8a514b031abf2373bb17568d999466d"
        ),
        "rows": 71659,
        "columns": 34,
        "expected_timepoint": "T0",
        "expected_cutoff": "2022-12-31",
    },
    "T1": {
        "path": T1_PATH,
        "sha256": (
            "5713a11bdbf4804758cc011f9b2f302a"
            "fc91fa1f88c1b178d675c28bb277d37c"
        ),
        "rows": 100920,
        "columns": 36,
        "expected_timepoint": "T1",
        "expected_cutoff": "2025-12-27",
    },
}


SHARED_REQUIRED_COLUMNS = {
    "timepoint",
    "release_label",
    "embedded_data_cutoff_date",
    "source_filename",
    "source_sha256",
    "rcv_accession",
    "rcv_version",
    "variation_id",
    "vcv_accession",
    "vcv_version",
    "target_genes_json",
    "study_scope",
    "condition_names_json",
    "condition_ids_json",
    "trait_records_json",
    "aggregate_classification",
    "aggregate_classification_group",
    "aggregate_review_status",
    "aggregate_review_stars",
    "aggregate_conflict_flag",
    "scv_group_disagreement_flag",
    "scv_count_xml",
    "unique_submitter_count_xml",
    "submitters_json",
    "submitter_ids_json",
    "scv_records_json",
}

T1_EXTENSION_COLUMNS = {
    "aggregate_classification_axis",
    "aggregate_classifications_json",
}


# -------------------------------------------------------------------------------------------------
# 3. UTILITY FUNCTIONS
# -------------------------------------------------------------------------------------------------

def sha256_file(path: Path, chunk_size: int = 8 * 1024 * 1024) -> str:
    """Calculate SHA-256 without loading the entire file into memory."""
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)

    return digest.hexdigest()


def normalize_scalar(value):
    """Normalize a value for stable comparison and JSON serialization."""
    if pd.isna(value):
        return None
    return str(value).strip()


def verify_artifact(label: str, specification: dict) -> dict:
    """Verify one frozen Parquet artifact and return a structured result."""

    path = specification["path"]
    failures = []

    result = {
        "label": label,
        "path": str(path),
        "exists": path.exists(),
        "failures": failures,
    }

    if not path.exists():
        failures.append(f"Required artifact does not exist: {path}")
        result["decision"] = "FAIL"
        return result

    # File-level verification
    observed_size = path.stat().st_size
    observed_sha256 = sha256_file(path)

    parquet_file = pq.ParquetFile(path)
    observed_rows = int(parquet_file.metadata.num_rows)
    observed_columns = int(parquet_file.metadata.num_columns)
    observed_row_groups = int(parquet_file.metadata.num_row_groups)
    schema_columns = list(parquet_file.schema_arrow.names)

    result.update(
        {
            "file_size_bytes": int(observed_size),
            "expected_sha256": specification["sha256"],
            "observed_sha256": observed_sha256,
            "sha256_match": observed_sha256 == specification["sha256"],
            "expected_rows": int(specification["rows"]),
            "observed_rows": observed_rows,
            "row_count_match": observed_rows == specification["rows"],
            "expected_columns": int(specification["columns"]),
            "observed_columns": observed_columns,
            "column_count_match": observed_columns == specification["columns"],
            "row_groups": observed_row_groups,
            "schema_columns": schema_columns,
        }
    )

    if not result["sha256_match"]:
        failures.append("SHA-256 does not match the accepted Stage 2 checksum.")

    if not result["row_count_match"]:
        failures.append(
            f"Row count mismatch: expected {specification['rows']:,}, "
            f"observed {observed_rows:,}."
        )

    if not result["column_count_match"]:
        failures.append(
            f"Column count mismatch: expected {specification['columns']}, "
            f"observed {observed_columns}."
        )

    missing_shared_columns = sorted(
        SHARED_REQUIRED_COLUMNS - set(schema_columns)
    )

    result["missing_shared_required_columns"] = missing_shared_columns

    if missing_shared_columns:
        failures.append(
            "Missing shared required columns: "
            + ", ".join(missing_shared_columns)
        )

    if label == "T1":
        missing_t1_extensions = sorted(
            T1_EXTENSION_COLUMNS - set(schema_columns)
        )
        result["missing_t1_extension_columns"] = missing_t1_extensions

        if missing_t1_extensions:
            failures.append(
                "Missing T1 classification-axis extension columns: "
                + ", ".join(missing_t1_extensions)
            )

    # Read only the columns needed for Stage 3 prerequisite checks.
    probe_columns = [
        "timepoint",
        "embedded_data_cutoff_date",
        "rcv_accession",
        "rcv_version",
        "variation_id",
        "vcv_accession",
        "vcv_version",
        "target_genes_json",
        "condition_names_json",
        "condition_ids_json",
    ]

    if label == "T1":
        probe_columns.append("aggregate_classification_axis")

    probe = pd.read_parquet(path, columns=probe_columns)

    rcv_text = probe["rcv_accession"].astype("string").str.strip().str.upper()

    missing_rcv = int(rcv_text.isna().sum())
    blank_rcv = int(rcv_text.fillna("").eq("").sum())
    duplicate_rcv = int(rcv_text.duplicated(keep=False).sum())
    unique_rcv = int(rcv_text.nunique(dropna=True))
    malformed_rcv = int(
        (~rcv_text.fillna("").str.fullmatch(r"RCV\d+")).sum()
    )

    timepoints = sorted(
        {
            normalize_scalar(value)
            for value in probe["timepoint"].dropna().unique().tolist()
        }
    )

    cutoffs = sorted(
        {
            normalize_scalar(value)
            for value in probe[
                "embedded_data_cutoff_date"
            ].dropna().unique().tolist()
        }
    )

    result.update(
        {
            "unique_rcv_accessions": unique_rcv,
            "missing_rcv_accessions": missing_rcv,
            "blank_rcv_accessions": blank_rcv,
            "rows_in_duplicate_rcv_groups": duplicate_rcv,
            "malformed_rcv_accessions": malformed_rcv,
            "observed_timepoints": timepoints,
            "observed_embedded_cutoffs": cutoffs,
            "expected_timepoint": specification["expected_timepoint"],
            "expected_embedded_cutoff": specification["expected_cutoff"],
        }
    )

    if missing_rcv != 0:
        failures.append(f"{missing_rcv:,} RCV accessions are missing.")

    if blank_rcv != 0:
        failures.append(f"{blank_rcv:,} RCV accessions are blank.")

    if duplicate_rcv != 0:
        failures.append(
            f"{duplicate_rcv:,} rows belong to duplicate RCV-accession groups."
        )

    if malformed_rcv != 0:
        failures.append(
            f"{malformed_rcv:,} RCV accessions do not match the expected format."
        )

    if unique_rcv != observed_rows:
        failures.append(
            f"Unique RCV count ({unique_rcv:,}) does not equal "
            f"the Parquet row count ({observed_rows:,})."
        )

    if timepoints != [specification["expected_timepoint"]]:
        failures.append(
            f"Unexpected timepoint values: {timepoints}"
        )

    if cutoffs != [specification["expected_cutoff"]]:
        failures.append(
            f"Unexpected embedded cutoff values: {cutoffs}"
        )

    result["decision"] = "PASS" if not failures else "FAIL"

    return result


# -------------------------------------------------------------------------------------------------
# 4. EXECUTE T0 AND T1 VERIFICATION
# -------------------------------------------------------------------------------------------------

verification_results = {}

for label, specification in EXPECTED.items():
    print()
    print("-" * 112)
    print(f"{label} ACCEPTED ARTIFACT")
    print("-" * 112)
    print(f"Path: {specification['path']}")

    artifact_result = verify_artifact(label, specification)
    verification_results[label] = artifact_result

    print(f"Exists:                    {artifact_result['exists']}")

    if artifact_result["exists"]:
        print(
            f"Rows:                      "
            f"{artifact_result['observed_rows']:,} "
            f"(expected {artifact_result['expected_rows']:,})"
        )
        print(
            f"Columns:                   "
            f"{artifact_result['observed_columns']} "
            f"(expected {artifact_result['expected_columns']})"
        )
        print(
            f"Row groups:                "
            f"{artifact_result['row_groups']}"
        )
        print(
            f"Unique RCV accessions:     "
            f"{artifact_result['unique_rcv_accessions']:,}"
        )
        print(
            f"Duplicate-RCV rows:        "
            f"{artifact_result['rows_in_duplicate_rcv_groups']:,}"
        )
        print(
            f"Malformed RCV accessions:  "
            f"{artifact_result['malformed_rcv_accessions']:,}"
        )
        print(
            f"Observed timepoint:        "
            f"{artifact_result['observed_timepoints']}"
        )
        print(
            f"Observed cutoff:           "
            f"{artifact_result['observed_embedded_cutoffs']}"
        )
        print(
            f"SHA-256 match:             "
            f"{artifact_result['sha256_match']}"
        )
        print(
            f"Observed SHA-256:          "
            f"{artifact_result['observed_sha256']}"
        )

    print(f"Decision:                  {artifact_result['decision']}")

    if artifact_result["failures"]:
        print("Failures:")
        for failure in artifact_result["failures"]:
            print(f"  - {failure}")


# -------------------------------------------------------------------------------------------------
# 5. WRITE THE STAGE 3 INPUT-VERIFICATION RECEIPT
# -------------------------------------------------------------------------------------------------

overall_pass = all(
    result["decision"] == "PASS"
    for result in verification_results.values()
)

receipt = {
    "receipt_name": "Stage 3 T0-T1 Input Verification Receipt",
    "receipt_version": "1.0.0",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "study_stage": "Stage 3A",
    "purpose": (
        "Verify the frozen T0 and T1 RCV-level artifacts before constructing "
        "the cross-release variant-condition crosswalk."
    ),
    "scientific_unit": "RCV-level variant-condition aggregate",
    "linkage_started": False,
    "future_instability_outcomes_created": False,
    "ges_model_fitted_or_tuned": False,
    "unmatched_records_labeled_stable": False,
    "verification_results": verification_results,
    "overall_decision": (
        "PASS_STAGE3_INPUTS_VERIFIED"
        if overall_pass
        else "FAIL_STAGE3_INPUTS_NOT_VERIFIED"
    ),
}

with RECEIPT_PATH.open("w", encoding="utf-8") as handle:
    json.dump(receipt, handle, indent=2, sort_keys=True)

receipt_sha256 = sha256_file(RECEIPT_PATH)

print()
print("=" * 112)
print("STAGE 3A SUMMARY")
print("=" * 112)
print(f"Receipt:        {RECEIPT_PATH}")
print(f"Receipt SHA-256:{receipt_sha256}")
print(f"Overall result: {receipt['overall_decision']}")
print("=" * 112)


# -------------------------------------------------------------------------------------------------
# 6. STOP THE WORKFLOW IF ANY FROZEN INPUT FAILED VERIFICATION
# -------------------------------------------------------------------------------------------------

if not overall_pass:
    raise RuntimeError(
        "Stage 3 is blocked because one or more accepted inputs failed verification. "
        "Review the printed failures before constructing the crosswalk."
    )

print()
print("PASS — The frozen T0 and T1 artifacts are verified.")
print("AUTHORIZED NEXT ACTION — Stage 3B exact RCV-accession crosswalk construction.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
STAGE 3A — FROZEN T0/T1 INPUT VERIFICATION

----------------------------------------------------------------------------------------------------------------
T0 ACCEPTED ARTIFACT
----------------------------------------------------------------------------------------------------------------
Path: /content/drive/MyDrive/GES_RAG_Temporal_Study/data_interim/t0_rcv_target_genes_corrected_v1_2.parquet
Exists:                    True
Rows:                      71,659 (expected 71,659)
Columns:                   34 (expected 34)
Row groups:                1
Unique RCV accessions:     71,659
Duplicate-RCV rows:        0
Malformed RCV accessions:  0
Observed timepoint:        ['T0']
Observed cutoff:           ['2022-12-31']
SHA-256 match:             True
Observed SHA-256:          f6b6760b2ad6e4352e3bdecdeaf89827e8a514b031abf2373bb17568d999466d
Decision:              

In [13]:
# =================================================================================================
# STAGE 3B — CONSTRUCT THE EXACT T0–T1 RCV-ACCESSION CROSSWALK
# =================================================================================================
# Primary linkage rule:
#   Match T0 and T1 records using the normalized RCV accession.
#
# This cell:
#   1. Revalidates that the Stage 3A verification receipt passed.
#   2. Loads linkage and condition-audit fields from the frozen T0 and T1 Parquets.
#   3. Performs a one-to-one outer join on normalized RCV accession.
#   4. Separates records into:
#        - EXACT_RCV_MATCH
#        - T0_ONLY_UNMATCHED_BY_EXACT_RCV
#        - T1_ONLY_UNLINKED_BY_EXACT_RCV
#   5. Writes a deterministic draft crosswalk and a checksum-controlled manifest.
#
# This cell DOES NOT:
#   - infer merges, splits, withdrawals, or accession replacements,
#   - calculate classification changes,
#   - create future-instability outcomes,
#   - label unmatched T0 records as stable,
#   - fit or tune GES.
# =================================================================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

import pandas as pd
import pyarrow.parquet as pq


print("=" * 116)
print("STAGE 3B — EXACT T0–T1 RCV-ACCESSION CROSSWALK")
print("=" * 116)


# -------------------------------------------------------------------------------------------------
# 1. PATHS
# -------------------------------------------------------------------------------------------------

STUDY_ROOT = Path("/content/drive/MyDrive/GES_RAG_Temporal_Study")

T0_PATH = (
    STUDY_ROOT
    / "data_interim"
    / "t0_rcv_target_genes_corrected_v1_2.parquet"
)

T1_PATH = (
    STUDY_ROOT
    / "data_interim"
    / "t1_rcv_target_genes_harmonized_v1.parquet"
)

STAGE3_CONFIG_DIR = STUDY_ROOT / "configs" / "stage3_crosswalk"
STAGE3_OUTPUT_DIR = STUDY_ROOT / "data_processed" / "stage3_crosswalk"

STAGE3_CONFIG_DIR.mkdir(parents=True, exist_ok=True)
STAGE3_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

INPUT_RECEIPT_PATH = (
    STAGE3_CONFIG_DIR
    / "stage3_t0_t1_input_verification_receipt_v1.json"
)

CROSSWALK_PATH = (
    STAGE3_OUTPUT_DIR
    / "stage3_t0_t1_exact_rcv_crosswalk_draft_v1.parquet"
)

MANIFEST_PATH = (
    STAGE3_CONFIG_DIR
    / "stage3_t0_t1_exact_rcv_crosswalk_manifest_v1.json"
)


# -------------------------------------------------------------------------------------------------
# 2. UTILITY FUNCTIONS
# -------------------------------------------------------------------------------------------------

def sha256_file(path: Path, chunk_size: int = 8 * 1024 * 1024) -> str:
    """Calculate SHA-256 without loading the complete file into memory."""
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)

    return digest.hexdigest()


def normalize_rcv(series: pd.Series) -> pd.Series:
    """Normalize RCV accessions without changing their scientific identity."""
    return (
        series.astype("string")
        .str.strip()
        .str.upper()
    )


def identifier_as_string(series: pd.Series) -> pd.Series:
    """
    Preserve identifiers as nullable strings.

    This avoids conversion of numeric identifiers to floating-point values
    after an outer merge introduces missing records.
    """
    return series.astype("string").str.strip()


def prepare_linkage_table(
    path: Path,
    prefix: str,
    include_classification_axis: bool
) -> pd.DataFrame:
    """Load and prefix only the fields authorized for linkage auditing."""

    columns = [
        "release_label",
        "embedded_data_cutoff_date",
        "xml_record_index",
        "source_filename",
        "rcv_accession",
        "rcv_version",
        "variation_id",
        "vcv_accession",
        "vcv_version",
        "target_genes_json",
        "study_scope",
        "condition_names_json",
        "condition_ids_json",
    ]

    if include_classification_axis:
        columns.append("aggregate_classification_axis")

    frame = pd.read_parquet(path, columns=columns)

    frame["rcv_accession_normalized"] = normalize_rcv(
        frame["rcv_accession"]
    )

    # Preserve identifier-like values as strings.
    for column in [
        "rcv_accession",
        "rcv_version",
        "variation_id",
        "vcv_accession",
        "vcv_version",
        "xml_record_index",
    ]:
        frame[column] = identifier_as_string(frame[column])

    missing_rcv = int(frame["rcv_accession_normalized"].isna().sum())
    blank_rcv = int(
        frame["rcv_accession_normalized"].fillna("").eq("").sum()
    )
    duplicate_rcv = int(
        frame["rcv_accession_normalized"].duplicated(keep=False).sum()
    )
    malformed_rcv = int(
        (
            ~frame["rcv_accession_normalized"]
            .fillna("")
            .str.fullmatch(r"RCV\d+")
        ).sum()
    )

    if missing_rcv != 0:
        raise ValueError(
            f"{prefix}: {missing_rcv:,} missing RCV accessions."
        )

    if blank_rcv != 0:
        raise ValueError(
            f"{prefix}: {blank_rcv:,} blank RCV accessions."
        )

    if duplicate_rcv != 0:
        raise ValueError(
            f"{prefix}: {duplicate_rcv:,} rows belong to duplicate RCV groups."
        )

    if malformed_rcv != 0:
        raise ValueError(
            f"{prefix}: {malformed_rcv:,} malformed RCV accessions."
        )

    rename_map = {
        column: f"{prefix}_{column}"
        for column in frame.columns
        if column != "rcv_accession_normalized"
    }

    return frame.rename(columns=rename_map)


# -------------------------------------------------------------------------------------------------
# 3. VERIFY THAT STAGE 3A PASSED
# -------------------------------------------------------------------------------------------------

if not INPUT_RECEIPT_PATH.exists():
    raise FileNotFoundError(
        "The Stage 3A verification receipt was not found:\n"
        f"{INPUT_RECEIPT_PATH}"
    )

with INPUT_RECEIPT_PATH.open("r", encoding="utf-8") as handle:
    input_receipt = json.load(handle)

stage3a_decision = input_receipt.get("overall_decision")

if stage3a_decision != "PASS_STAGE3_INPUTS_VERIFIED":
    raise RuntimeError(
        "Stage 3B is blocked because the Stage 3A receipt does not contain "
        "PASS_STAGE3_INPUTS_VERIFIED."
    )

input_receipt_sha256 = sha256_file(INPUT_RECEIPT_PATH)

print(f"Stage 3A decision:       {stage3a_decision}")
print(f"Stage 3A receipt SHA-256:{input_receipt_sha256}")


# -------------------------------------------------------------------------------------------------
# 4. LOAD THE FROZEN INPUTS
# -------------------------------------------------------------------------------------------------

print()
print("Loading frozen linkage fields...")

t0 = prepare_linkage_table(
    path=T0_PATH,
    prefix="t0",
    include_classification_axis=False,
)

t1 = prepare_linkage_table(
    path=T1_PATH,
    prefix="t1",
    include_classification_axis=True,
)

t0_rows = len(t0)
t1_rows = len(t1)

print(f"T0 rows loaded: {t0_rows:,}")
print(f"T1 rows loaded: {t1_rows:,}")


# -------------------------------------------------------------------------------------------------
# 5. EXACT ONE-TO-ONE OUTER JOIN ON NORMALIZED RCV ACCESSION
# -------------------------------------------------------------------------------------------------

crosswalk = t0.merge(
    t1,
    on="rcv_accession_normalized",
    how="outer",
    validate="one_to_one",
    indicator=True,
)

status_map = {
    "both": "EXACT_RCV_MATCH",
    "left_only": "T0_ONLY_UNMATCHED_BY_EXACT_RCV",
    "right_only": "T1_ONLY_UNLINKED_BY_EXACT_RCV",
}

crosswalk["linkage_status"] = (
    crosswalk["_merge"]
    .astype("string")
    .map(status_map)
)

crosswalk["t0_present"] = crosswalk["_merge"].isin(
    ["both", "left_only"]
)

crosswalk["t1_present"] = crosswalk["_merge"].isin(
    ["both", "right_only"]
)

crosswalk["exact_rcv_accession_match"] = (
    crosswalk["_merge"] == "both"
)

crosswalk["crosswalk_id"] = crosswalk[
    "rcv_accession_normalized"
]

crosswalk = crosswalk.drop(columns=["_merge"])


# -------------------------------------------------------------------------------------------------
# 6. VALIDATE THE CROSSWALK STRUCTURE
# -------------------------------------------------------------------------------------------------

crosswalk_rows = len(crosswalk)

exact_matches = int(
    crosswalk["linkage_status"].eq("EXACT_RCV_MATCH").sum()
)

t0_only = int(
    crosswalk["linkage_status"]
    .eq("T0_ONLY_UNMATCHED_BY_EXACT_RCV")
    .sum()
)

t1_only = int(
    crosswalk["linkage_status"]
    .eq("T1_ONLY_UNLINKED_BY_EXACT_RCV")
    .sum()
)

duplicate_crosswalk_keys = int(
    crosswalk["crosswalk_id"].duplicated(keep=False).sum()
)

missing_crosswalk_keys = int(
    crosswalk["crosswalk_id"].isna().sum()
)

if duplicate_crosswalk_keys != 0:
    raise AssertionError(
        f"{duplicate_crosswalk_keys:,} rows have duplicate crosswalk IDs."
    )

if missing_crosswalk_keys != 0:
    raise AssertionError(
        f"{missing_crosswalk_keys:,} rows have missing crosswalk IDs."
    )

if exact_matches + t0_only != t0_rows:
    raise AssertionError(
        "T0 accounting failed: exact matches + T0-only records "
        "does not equal the T0 cohort size."
    )

if exact_matches + t1_only != t1_rows:
    raise AssertionError(
        "T1 accounting failed: exact matches + T1-only records "
        "does not equal the T1 cohort size."
    )

if crosswalk_rows != exact_matches + t0_only + t1_only:
    raise AssertionError(
        "Crosswalk status counts do not equal the crosswalk row count."
    )

# Confirm matched records truly contain the same normalized RCV accession.
matched = crosswalk[
    crosswalk["exact_rcv_accession_match"]
].copy()

matched_t0_rcv = normalize_rcv(matched["t0_rcv_accession"])
matched_t1_rcv = normalize_rcv(matched["t1_rcv_accession"])

matched_accession_mismatches = int(
    (
        (matched_t0_rcv != matched["rcv_accession_normalized"])
        |
        (matched_t1_rcv != matched["rcv_accession_normalized"])
    ).sum()
)

if matched_accession_mismatches != 0:
    raise AssertionError(
        f"{matched_accession_mismatches:,} exact-match rows failed "
        "normalized accession verification."
    )


# -------------------------------------------------------------------------------------------------
# 7. CREATE A DETERMINISTIC OUTPUT ORDER
# -------------------------------------------------------------------------------------------------

status_sort_order = {
    "EXACT_RCV_MATCH": 0,
    "T0_ONLY_UNMATCHED_BY_EXACT_RCV": 1,
    "T1_ONLY_UNLINKED_BY_EXACT_RCV": 2,
}

crosswalk["_status_sort"] = crosswalk[
    "linkage_status"
].map(status_sort_order)

crosswalk = (
    crosswalk
    .sort_values(
        by=["_status_sort", "rcv_accession_normalized"],
        kind="mergesort",
    )
    .drop(columns=["_status_sort"])
    .reset_index(drop=True)
)


# -------------------------------------------------------------------------------------------------
# 8. ORDER OUTPUT COLUMNS
# -------------------------------------------------------------------------------------------------

ordered_columns = [
    "crosswalk_id",
    "rcv_accession_normalized",
    "linkage_status",
    "t0_present",
    "t1_present",
    "exact_rcv_accession_match",

    "t0_release_label",
    "t0_embedded_data_cutoff_date",
    "t0_xml_record_index",
    "t0_source_filename",
    "t0_rcv_accession",
    "t0_rcv_version",
    "t0_variation_id",
    "t0_vcv_accession",
    "t0_vcv_version",
    "t0_target_genes_json",
    "t0_study_scope",
    "t0_condition_names_json",
    "t0_condition_ids_json",

    "t1_release_label",
    "t1_embedded_data_cutoff_date",
    "t1_xml_record_index",
    "t1_source_filename",
    "t1_rcv_accession",
    "t1_rcv_version",
    "t1_variation_id",
    "t1_vcv_accession",
    "t1_vcv_version",
    "t1_target_genes_json",
    "t1_study_scope",
    "t1_condition_names_json",
    "t1_condition_ids_json",
    "t1_aggregate_classification_axis",
]

missing_output_columns = [
    column
    for column in ordered_columns
    if column not in crosswalk.columns
]

if missing_output_columns:
    raise KeyError(
        "The following expected crosswalk columns are missing: "
        + ", ".join(missing_output_columns)
    )

crosswalk = crosswalk[ordered_columns]


# -------------------------------------------------------------------------------------------------
# 9. WRITE AND READ BACK THE DRAFT CROSSWALK
# -------------------------------------------------------------------------------------------------

crosswalk.to_parquet(
    CROSSWALK_PATH,
    engine="pyarrow",
    compression="zstd",
    index=False,
)

crosswalk_sha256 = sha256_file(CROSSWALK_PATH)

parquet_metadata = pq.ParquetFile(CROSSWALK_PATH).metadata

readback = pd.read_parquet(CROSSWALK_PATH)

if len(readback) != crosswalk_rows:
    raise AssertionError(
        "Crosswalk readback row count does not match the written dataframe."
    )

if list(readback.columns) != list(crosswalk.columns):
    raise AssertionError(
        "Crosswalk readback columns do not match the written dataframe."
    )

readback_status_counts = (
    readback["linkage_status"]
    .value_counts(dropna=False)
    .to_dict()
)

expected_status_counts = (
    crosswalk["linkage_status"]
    .value_counts(dropna=False)
    .to_dict()
)

if readback_status_counts != expected_status_counts:
    raise AssertionError(
        "Crosswalk readback linkage-status counts do not match."
    )


# -------------------------------------------------------------------------------------------------
# 10. CREATE THE STAGE 3B MANIFEST
# -------------------------------------------------------------------------------------------------

t0_sha256 = sha256_file(T0_PATH)
t1_sha256 = sha256_file(T1_PATH)

t0_match_rate = (
    exact_matches / t0_rows
    if t0_rows
    else None
)

t1_exact_link_share = (
    exact_matches / t1_rows
    if t1_rows
    else None
)

manifest = {
    "manifest_name": "Stage 3B Exact RCV Crosswalk Manifest",
    "manifest_version": "1.0.0",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "study_stage": "Stage 3B",
    "crosswalk_status": "DRAFT_EXACT_RCV_LINKAGE_COMPLETE",
    "scientific_unit": "RCV-level variant-condition aggregate",
    "primary_linkage_rule": (
        "One-to-one exact match on stripped, uppercase-normalized "
        "RCV accession."
    ),
    "input_verification": {
        "stage3a_receipt": str(INPUT_RECEIPT_PATH),
        "stage3a_receipt_sha256": input_receipt_sha256,
        "stage3a_decision": stage3a_decision,
    },
    "inputs": {
        "T0": {
            "path": str(T0_PATH),
            "sha256": t0_sha256,
            "rows": int(t0_rows),
        },
        "T1": {
            "path": str(T1_PATH),
            "sha256": t1_sha256,
            "rows": int(t1_rows),
        },
    },
    "output": {
        "path": str(CROSSWALK_PATH),
        "sha256": crosswalk_sha256,
        "rows": int(crosswalk_rows),
        "columns": int(len(crosswalk.columns)),
        "row_groups": int(parquet_metadata.num_row_groups),
        "compression": "Zstandard",
    },
    "linkage_counts": {
        "exact_rcv_matches": int(exact_matches),
        "t0_only_unmatched_by_exact_rcv": int(t0_only),
        "t1_only_unlinked_by_exact_rcv": int(t1_only),
        "union_rcv_count": int(crosswalk_rows),
        "t0_exact_match_rate": (
            round(float(t0_match_rate), 8)
            if t0_match_rate is not None
            else None
        ),
        "t1_exact_link_share": (
            round(float(t1_exact_link_share), 8)
            if t1_exact_link_share is not None
            else None
        ),
    },
    "validation": {
        "duplicate_crosswalk_ids": int(duplicate_crosswalk_keys),
        "missing_crosswalk_ids": int(missing_crosswalk_keys),
        "matched_accession_mismatches": int(
            matched_accession_mismatches
        ),
        "t0_accounting_passed": (
            exact_matches + t0_only == t0_rows
        ),
        "t1_accounting_passed": (
            exact_matches + t1_only == t1_rows
        ),
        "readback_passed": True,
    },
    "scientific_boundaries": {
        "non_exact_linkage_inferred": False,
        "merges_or_splits_inferred": False,
        "withdrawals_inferred": False,
        "condition_equivalence_inferred": False,
        "classification_changes_calculated": False,
        "future_instability_outcomes_created": False,
        "unmatched_t0_records_labeled_stable": False,
        "ges_model_fitted_or_tuned": False,
    },
    "next_authorized_step": (
        "Audit exact-match identifier continuity, RCV/VCV version changes, "
        "VariationID continuity, gene consistency, condition consistency, "
        "and classification-axis compatibility."
    ),
}

with MANIFEST_PATH.open("w", encoding="utf-8") as handle:
    json.dump(
        manifest,
        handle,
        indent=2,
        sort_keys=True,
    )

manifest_sha256 = sha256_file(MANIFEST_PATH)


# -------------------------------------------------------------------------------------------------
# 11. PRINT THE STAGE 3B RESULT
# -------------------------------------------------------------------------------------------------

print()
print("=" * 116)
print("STAGE 3B CROSSWALK SUMMARY")
print("=" * 116)

print(f"T0 records:                         {t0_rows:,}")
print(f"T1 records:                         {t1_rows:,}")
print(f"Union of unique RCV accessions:     {crosswalk_rows:,}")
print(f"Exact RCV matches:                  {exact_matches:,}")
print(f"T0-only exact-unmatched records:    {t0_only:,}")
print(f"T1-only exact-unlinked records:     {t1_only:,}")

print(
    f"T0 exact-match rate:                "
    f"{t0_match_rate:.2%}"
)

print(
    f"T1 records linked to T0 exactly:    "
    f"{t1_exact_link_share:.2%}"
)

print()
print(f"Crosswalk:        {CROSSWALK_PATH}")
print(f"Crosswalk SHA-256:{crosswalk_sha256}")
print(f"Manifest:         {MANIFEST_PATH}")
print(f"Manifest SHA-256: {manifest_sha256}")

print()
print("PASS — Stage 3B exact RCV-accession crosswalk created.")
print(
    "IMPORTANT — T0-only records remain unmatched and have not "
    "been classified as stable."
)
print(
    "AUTHORIZED NEXT ACTION — Stage 3C matched-record identifier, "
    "version, condition, gene, and classification-axis continuity audit."
)
print("=" * 116)

STAGE 3B — EXACT T0–T1 RCV-ACCESSION CROSSWALK
Stage 3A decision:       PASS_STAGE3_INPUTS_VERIFIED
Stage 3A receipt SHA-256:4eaae01e17b93b3eb5384dc471b3d7f34d37bce3c3ff18262211cb60ba1f27c5

Loading frozen linkage fields...
T0 rows loaded: 71,659
T1 rows loaded: 100,920

STAGE 3B CROSSWALK SUMMARY
T0 records:                         71,659
T1 records:                         100,920
Union of unique RCV accessions:     102,166
Exact RCV matches:                  70,413
T0-only exact-unmatched records:    1,246
T1-only exact-unlinked records:     30,507
T0 exact-match rate:                98.26%
T1 records linked to T0 exactly:    69.77%

Crosswalk:        /content/drive/MyDrive/GES_RAG_Temporal_Study/data_processed/stage3_crosswalk/stage3_t0_t1_exact_rcv_crosswalk_draft_v1.parquet
Crosswalk SHA-256:b462304a4bb31db301e2dac3aefbf685e12f1fea179a38ca04b8500b64bb4acc
Manifest:         /content/drive/MyDrive/GES_RAG_Temporal_Study/configs/stage3_crosswalk/stage3_t0_t1_exact_rcv_crosswalk_mani

In [14]:
# =================================================================================================
# STAGE 3C — EXACT-MATCH IDENTIFIER, VERSION, GENE, CONDITION, AND AXIS CONTINUITY AUDIT
# =================================================================================================
# Purpose:
#   Audit the 70,413 exact RCV-accession matches created during Stage 3B.
#
# This cell evaluates:
#   1. VariationID continuity.
#   2. VCV accession continuity.
#   3. RCV and VCV version changes.
#   4. Target-gene and study-scope continuity.
#   5. Condition-name and structured condition-ID continuity.
#   6. T1 classification-axis compatibility categories.
#   7. Records requiring later review before outcome construction.
#
# Important scientific boundary:
#   T0 does not contain the explicit current-schema classification-axis field.
#   Therefore, this cell does not claim direct T0–T1 axis equality.
#   It performs a conservative gene-aware audit of the T1 selected axis.
#
# This cell DOES NOT:
#   - infer temporal instability,
#   - compare aggregate classifications,
#   - construct future-instability outcomes,
#   - classify unmatched T0 records as stable,
#   - infer merges, splits, withdrawals, or replacement accessions,
#   - fit or tune GES.
# =================================================================================================

from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
import hashlib
import json
import re
import unicodedata

import pandas as pd
import pyarrow.parquet as pq


print("=" * 120)
print("STAGE 3C — EXACT-MATCH CONTINUITY AUDIT")
print("=" * 120)


# -------------------------------------------------------------------------------------------------
# 1. PATHS AND ACCEPTED STAGE 3B CHECKSUMS
# -------------------------------------------------------------------------------------------------

STUDY_ROOT = Path("/content/drive/MyDrive/GES_RAG_Temporal_Study")

STAGE3_CONFIG_DIR = STUDY_ROOT / "configs" / "stage3_crosswalk"
STAGE3_OUTPUT_DIR = STUDY_ROOT / "data_processed" / "stage3_crosswalk"

CROSSWALK_PATH = (
    STAGE3_OUTPUT_DIR
    / "stage3_t0_t1_exact_rcv_crosswalk_draft_v1.parquet"
)

STAGE3B_MANIFEST_PATH = (
    STAGE3_CONFIG_DIR
    / "stage3_t0_t1_exact_rcv_crosswalk_manifest_v1.json"
)

AUDIT_PATH = (
    STAGE3_OUTPUT_DIR
    / "stage3_t0_t1_exact_match_continuity_audit_v1.parquet"
)

REPORT_PATH = (
    STAGE3_CONFIG_DIR
    / "stage3_t0_t1_exact_match_continuity_report_v1.json"
)

MANIFEST_PATH = (
    STAGE3_CONFIG_DIR
    / "stage3_t0_t1_exact_match_continuity_manifest_v1.json"
)

EXPECTED_STAGE3B_CROSSWALK_SHA256 = (
    "b462304a4bb31db301e2dac3aefbf685"
    "e12f1fea179a38ca04b8500b64bb4acc"
)

EXPECTED_STAGE3B_MANIFEST_SHA256 = (
    "bb5274383a83ce70673f43dd97b09bf7"
    "c1303c18292b74da577bae38430e1f93"
)


# -------------------------------------------------------------------------------------------------
# 2. UTILITY FUNCTIONS
# -------------------------------------------------------------------------------------------------

def sha256_file(path: Path, chunk_size: int = 8 * 1024 * 1024) -> str:
    """Calculate SHA-256 without reading the complete file into memory."""
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)

    return digest.hexdigest()


def is_missing(value) -> bool:
    """Return True for None, pandas missing values, or blank strings."""
    if value is None:
        return True

    try:
        if pd.isna(value):
            return True
    except (TypeError, ValueError):
        pass

    return str(value).strip() == ""


def clean_identifier(value, uppercase: bool = True):
    """
    Normalize an identifier conservatively.

    Numeric identifiers accidentally represented as '123.0' are returned as '123'.
    """
    if is_missing(value):
        return None

    text = str(value).strip()

    if re.fullmatch(r"\d+\.0", text):
        text = text[:-2]

    if uppercase:
        text = text.upper()

    return text


def parse_json_list(value, field_name: str):
    """Parse a serialized JSON list and enforce its expected top-level type."""
    if value is None:
        return []

    if isinstance(value, list):
        parsed = value
    elif isinstance(value, tuple):
        parsed = list(value)
    else:
        try:
            if pd.isna(value):
                return []
        except (TypeError, ValueError):
            pass

        text = str(value).strip()

        if text == "":
            return []

        try:
            parsed = json.loads(text)
        except json.JSONDecodeError as exc:
            raise ValueError(
                f"Malformed JSON in {field_name}: {text[:200]}"
            ) from exc

    if not isinstance(parsed, list):
        raise TypeError(
            f"{field_name} must contain a JSON list, not {type(parsed).__name__}."
        )

    return parsed


def normalize_text(value):
    """Conservative normalization for condition names."""
    if value is None:
        return None

    text = unicodedata.normalize("NFKC", str(value))
    text = re.sub(r"\s+", " ", text).strip().casefold()

    return text if text else None


def normalize_condition_id(value):
    """Normalize condition identifiers while preserving punctuation and namespaces."""
    if value is None:
        return None

    text = unicodedata.normalize("NFKC", str(value))
    text = re.sub(r"\s+", "", text).strip().upper()

    return text if text else None


def normalize_gene(value):
    """Normalize target-gene symbols."""
    if value is None:
        return None

    text = str(value).strip().upper()

    return text if text else None


def normalized_json_set(value, field_name: str, normalizer):
    """
    Parse a JSON list, normalize its entries, remove blanks and duplicates,
    and return a sorted tuple.
    """
    parsed = parse_json_list(value, field_name)

    normalized_values = []

    for item in parsed:
        normalized = normalizer(item)

        if normalized is not None:
            normalized_values.append(normalized)

    return tuple(sorted(set(normalized_values)))


def tuple_to_json(values) -> str:
    """Serialize a tuple deterministically for Parquet storage."""
    return json.dumps(
        list(values),
        ensure_ascii=False,
        separators=(",", ":"),
    )


def set_relation(t0_values, t1_values) -> str:
    """Describe the relationship between two normalized sets."""
    t0_set = set(t0_values)
    t1_set = set(t1_values)

    if not t0_set and not t1_set:
        return "BOTH_EMPTY"

    if not t0_set:
        return "T0_EMPTY"

    if not t1_set:
        return "T1_EMPTY"

    if t0_set == t1_set:
        return "EXACT_SET_MATCH"

    if t0_set.intersection(t1_set):
        return "PARTIAL_SET_OVERLAP"

    return "DISJOINT_SETS"


def scalar_identifier_relation(t0_value, t1_value) -> str:
    """Compare two scalar identifiers."""
    t0_clean = clean_identifier(t0_value)
    t1_clean = clean_identifier(t1_value)

    if t0_clean is None and t1_clean is None:
        return "BOTH_MISSING"

    if t0_clean is None:
        return "T0_MISSING"

    if t1_clean is None:
        return "T1_MISSING"

    if t0_clean == t1_clean:
        return "SAME"

    return "CHANGED"


def parse_integer_version(value):
    """Parse a version as an integer where possible."""
    cleaned = clean_identifier(value, uppercase=False)

    if cleaned is None:
        return None

    if re.fullmatch(r"\d+", cleaned):
        return int(cleaned)

    return None


def version_relation(t0_value, t1_value) -> str:
    """Describe version continuity and direction."""
    t0_clean = clean_identifier(t0_value, uppercase=False)
    t1_clean = clean_identifier(t1_value, uppercase=False)

    if t0_clean is None and t1_clean is None:
        return "BOTH_MISSING"

    if t0_clean is None:
        return "T0_MISSING"

    if t1_clean is None:
        return "T1_MISSING"

    if t0_clean == t1_clean:
        return "SAME_VERSION"

    t0_number = parse_integer_version(t0_value)
    t1_number = parse_integer_version(t1_value)

    if t0_number is not None and t1_number is not None:
        if t1_number > t0_number:
            return "VERSION_INCREASED"

        if t1_number < t0_number:
            return "VERSION_DECREASED"

    return "VERSION_CHANGED_NONNUMERIC"


def combined_identifier_status(variation_relation, vcv_relation) -> str:
    """Create a conservative core-identifier continuity category."""
    if variation_relation == "SAME" and vcv_relation == "SAME":
        return "STABLE_CORE_IDENTIFIERS"

    missing_relations = {
        "BOTH_MISSING",
        "T0_MISSING",
        "T1_MISSING",
    }

    if (
        variation_relation in missing_relations
        or vcv_relation in missing_relations
    ):
        return "CORE_IDENTIFIER_MISSING_REVIEW"

    if variation_relation == "SAME" and vcv_relation == "CHANGED":
        return "VARIATION_STABLE_VCV_CHANGED_REVIEW"

    if variation_relation == "CHANGED" and vcv_relation == "SAME":
        return "VCV_STABLE_VARIATION_CHANGED_REVIEW"

    return "CORE_IDENTIFIERS_CHANGED_REVIEW"


def combined_condition_status(id_relation, name_relation) -> str:
    """
    Combine structured condition-ID and normalized condition-name relationships.

    These categories are audit categories only. They do not alter the exact RCV link.
    """
    if (
        id_relation == "EXACT_SET_MATCH"
        and name_relation == "EXACT_SET_MATCH"
    ):
        return "CONDITION_IDS_AND_NAMES_CONSISTENT"

    if id_relation == "EXACT_SET_MATCH":
        return "CONDITION_ID_CONTINUITY_WITH_NAME_DRIFT"

    if id_relation == "PARTIAL_SET_OVERLAP":
        return "PARTIAL_CONDITION_ID_CONTINUITY"

    if (
        id_relation == "BOTH_EMPTY"
        and name_relation == "EXACT_SET_MATCH"
    ):
        return "NAME_CONTINUITY_IDS_MISSING_BOTH_RELEASES"

    if (
        id_relation in {"T0_EMPTY", "T1_EMPTY"}
        and name_relation in {
            "EXACT_SET_MATCH",
            "PARTIAL_SET_OVERLAP",
        }
    ):
        return "NAME_CONTINUITY_ONE_RELEASE_MISSING_IDS"

    if (
        id_relation == "DISJOINT_SETS"
        and name_relation in {
            "EXACT_SET_MATCH",
            "PARTIAL_SET_OVERLAP",
        }
    ):
        return "NAME_CONTINUITY_WITH_ID_CHANGE_REVIEW"

    if (
        name_relation == "EXACT_SET_MATCH"
        and id_relation != "EXACT_SET_MATCH"
    ):
        return "NAME_CONTINUITY_REQUIRES_ID_REVIEW"

    return "CONDITION_DISCORDANCE_REVIEW"


def canonical_axis(value) -> str:
    """Map T1 axis wording to one canonical category."""
    if is_missing(value):
        return "MissingAxis"

    raw = str(value).strip()
    key = re.sub(r"[^a-z]", "", raw.casefold())

    mapping = {
        "germlineclassification": "GermlineClassification",
        "oncogenicityclassification": "OncogenicityClassification",
        "somaticclinicalimpact": "SomaticClinicalImpact",
        "noclassification": "NoClassification",
    }

    return mapping.get(key, raw)


def axis_audit_category(gene_values, t1_axis) -> str:
    """
    Create a gene-aware T1 classification-axis audit category.

    This does not infer an explicit T0 axis.
    """
    genes = set(gene_values)
    axis = canonical_axis(t1_axis)

    primary_genes = {"BRCA1", "BRCA2", "MLH1"}

    if len(genes) != 1:
        return "MULTIPLE_OR_MISSING_GENE_AXIS_REVIEW"

    gene = next(iter(genes))

    if gene in primary_genes:
        if axis == "GermlineClassification":
            return "PRIMARY_GENE_GERMLINE_AXIS"

        if axis in {"NoClassification", "MissingAxis"}:
            return "PRIMARY_GENE_NO_T1_CLASSIFICATION"

        return "PRIMARY_GENE_NON_GERMLINE_AXIS_REVIEW"

    if gene == "EGFR":
        if axis == "OncogenicityClassification":
            return "EGFR_ONCOGENICITY_AXIS"

        if axis == "SomaticClinicalImpact":
            return "EGFR_SOMATIC_CLINICAL_IMPACT_AXIS"

        if axis == "GermlineClassification":
            return "EGFR_GERMLINE_AXIS"

        if axis in {"NoClassification", "MissingAxis"}:
            return "EGFR_NO_T1_CLASSIFICATION"

        return "EGFR_OTHER_AXIS_REVIEW"

    return "UNEXPECTED_GENE_AXIS_REVIEW"


def build_review_reasons(row) -> list:
    """Create transparent reasons for records requiring later review."""
    reasons = []

    if row["core_identifier_continuity_status"] != "STABLE_CORE_IDENTIFIERS":
        reasons.append("CORE_IDENTIFIER_CHANGE_OR_MISSING")

    if row["gene_set_relation"] != "EXACT_SET_MATCH":
        reasons.append("GENE_SET_DISCORDANCE")

    if row["study_scope_relation"] != "SAME":
        reasons.append("STUDY_SCOPE_CHANGE")

    if row["condition_continuity_status"] in {
        "NAME_CONTINUITY_WITH_ID_CHANGE_REVIEW",
        "NAME_CONTINUITY_REQUIRES_ID_REVIEW",
        "CONDITION_DISCORDANCE_REVIEW",
    }:
        reasons.append("CONDITION_RELATIONSHIP_REVIEW")

    if row["rcv_version_relation"] == "VERSION_DECREASED":
        reasons.append("RCV_VERSION_DECREASED")

    if row["vcv_version_relation"] == "VERSION_DECREASED":
        reasons.append("VCV_VERSION_DECREASED")

    if row["classification_axis_audit_category"] in {
        "PRIMARY_GENE_NON_GERMLINE_AXIS_REVIEW",
        "PRIMARY_GENE_NO_T1_CLASSIFICATION",
        "EGFR_NO_T1_CLASSIFICATION",
        "EGFR_OTHER_AXIS_REVIEW",
        "MULTIPLE_OR_MISSING_GENE_AXIS_REVIEW",
        "UNEXPECTED_GENE_AXIS_REVIEW",
    }:
        reasons.append("CLASSIFICATION_AXIS_REVIEW")

    return sorted(set(reasons))


def value_counts_dict(series: pd.Series) -> dict:
    """Convert pandas value counts into a JSON-safe dictionary."""
    counts = series.value_counts(dropna=False)

    result = {}

    for key, value in counts.items():
        safe_key = "<MISSING>" if pd.isna(key) else str(key)
        result[safe_key] = int(value)

    return result


def nested_counts_dict(frame: pd.DataFrame, rows, columns) -> dict:
    """Create a JSON-safe grouped-count dictionary."""
    grouped = (
        frame.groupby(rows + columns, dropna=False)
        .size()
        .reset_index(name="record_count")
    )

    records = []

    for _, grouped_row in grouped.iterrows():
        record = {}

        for column in rows + columns:
            value = grouped_row[column]
            record[column] = None if pd.isna(value) else str(value)

        record["record_count"] = int(grouped_row["record_count"])
        records.append(record)

    return {"records": records}


# -------------------------------------------------------------------------------------------------
# 3. VERIFY STAGE 3B ARTIFACTS
# -------------------------------------------------------------------------------------------------

for required_path in [CROSSWALK_PATH, STAGE3B_MANIFEST_PATH]:
    if not required_path.exists():
        raise FileNotFoundError(
            f"Required Stage 3B artifact does not exist: {required_path}"
        )

observed_crosswalk_sha256 = sha256_file(CROSSWALK_PATH)
observed_stage3b_manifest_sha256 = sha256_file(STAGE3B_MANIFEST_PATH)

if observed_crosswalk_sha256 != EXPECTED_STAGE3B_CROSSWALK_SHA256:
    raise RuntimeError(
        "Stage 3B crosswalk checksum mismatch.\n"
        f"Expected: {EXPECTED_STAGE3B_CROSSWALK_SHA256}\n"
        f"Observed: {observed_crosswalk_sha256}"
    )

if observed_stage3b_manifest_sha256 != EXPECTED_STAGE3B_MANIFEST_SHA256:
    raise RuntimeError(
        "Stage 3B manifest checksum mismatch.\n"
        f"Expected: {EXPECTED_STAGE3B_MANIFEST_SHA256}\n"
        f"Observed: {observed_stage3b_manifest_sha256}"
    )

with STAGE3B_MANIFEST_PATH.open("r", encoding="utf-8") as handle:
    stage3b_manifest = json.load(handle)

if (
    stage3b_manifest.get("crosswalk_status")
    != "DRAFT_EXACT_RCV_LINKAGE_COMPLETE"
):
    raise RuntimeError(
        "Stage 3B manifest does not authorize the exact-match continuity audit."
    )

expected_exact_matches = int(
    stage3b_manifest["linkage_counts"]["exact_rcv_matches"]
)

print(f"Stage 3B crosswalk SHA-256: {observed_crosswalk_sha256}")
print(f"Stage 3B manifest SHA-256:  {observed_stage3b_manifest_sha256}")
print(f"Expected exact matches:     {expected_exact_matches:,}")


# -------------------------------------------------------------------------------------------------
# 4. LOAD EXACT RCV MATCHES
# -------------------------------------------------------------------------------------------------

required_columns = [
    "crosswalk_id",
    "rcv_accession_normalized",
    "linkage_status",
    "exact_rcv_accession_match",

    "t0_release_label",
    "t0_embedded_data_cutoff_date",
    "t0_rcv_accession",
    "t0_rcv_version",
    "t0_variation_id",
    "t0_vcv_accession",
    "t0_vcv_version",
    "t0_target_genes_json",
    "t0_study_scope",
    "t0_condition_names_json",
    "t0_condition_ids_json",

    "t1_release_label",
    "t1_embedded_data_cutoff_date",
    "t1_rcv_accession",
    "t1_rcv_version",
    "t1_variation_id",
    "t1_vcv_accession",
    "t1_vcv_version",
    "t1_target_genes_json",
    "t1_study_scope",
    "t1_condition_names_json",
    "t1_condition_ids_json",
    "t1_aggregate_classification_axis",
]

crosswalk_schema = set(
    pq.ParquetFile(CROSSWALK_PATH).schema_arrow.names
)

missing_columns = sorted(
    set(required_columns) - crosswalk_schema
)

if missing_columns:
    raise KeyError(
        "Stage 3B crosswalk is missing required columns: "
        + ", ".join(missing_columns)
    )

crosswalk = pd.read_parquet(
    CROSSWALK_PATH,
    columns=required_columns,
)

exact = crosswalk.loc[
    crosswalk["linkage_status"].eq("EXACT_RCV_MATCH")
    & crosswalk["exact_rcv_accession_match"].eq(True)
].copy()

if len(exact) != expected_exact_matches:
    raise AssertionError(
        f"Expected {expected_exact_matches:,} exact matches, "
        f"but loaded {len(exact):,}."
    )

if exact["crosswalk_id"].duplicated().any():
    raise AssertionError(
        "Duplicate crosswalk IDs were found among exact matches."
    )

print()
print(f"Exact-match rows loaded: {len(exact):,}")


# -------------------------------------------------------------------------------------------------
# 5. NORMALIZE IDENTIFIERS AND JSON FIELDS
# -------------------------------------------------------------------------------------------------

print("Normalizing identifiers, genes, conditions, and classification axes...")

# Scalar identifier relationships
exact["variation_id_relation"] = [
    scalar_identifier_relation(t0_value, t1_value)
    for t0_value, t1_value in zip(
        exact["t0_variation_id"],
        exact["t1_variation_id"],
    )
]

exact["vcv_accession_relation"] = [
    scalar_identifier_relation(t0_value, t1_value)
    for t0_value, t1_value in zip(
        exact["t0_vcv_accession"],
        exact["t1_vcv_accession"],
    )
]

exact["rcv_version_relation"] = [
    version_relation(t0_value, t1_value)
    for t0_value, t1_value in zip(
        exact["t0_rcv_version"],
        exact["t1_rcv_version"],
    )
]

exact["vcv_version_relation"] = [
    version_relation(t0_value, t1_value)
    for t0_value, t1_value in zip(
        exact["t0_vcv_version"],
        exact["t1_vcv_version"],
    )
]

exact["core_identifier_continuity_status"] = [
    combined_identifier_status(variation_relation, vcv_relation)
    for variation_relation, vcv_relation in zip(
        exact["variation_id_relation"],
        exact["vcv_accession_relation"],
    )
]

# Parse normalized genes
t0_gene_sets = [
    normalized_json_set(
        value,
        "t0_target_genes_json",
        normalize_gene,
    )
    for value in exact["t0_target_genes_json"]
]

t1_gene_sets = [
    normalized_json_set(
        value,
        "t1_target_genes_json",
        normalize_gene,
    )
    for value in exact["t1_target_genes_json"]
]

exact["t0_target_genes_normalized_json"] = [
    tuple_to_json(values)
    for values in t0_gene_sets
]

exact["t1_target_genes_normalized_json"] = [
    tuple_to_json(values)
    for values in t1_gene_sets
]

exact["gene_set_relation"] = [
    set_relation(t0_values, t1_values)
    for t0_values, t1_values in zip(
        t0_gene_sets,
        t1_gene_sets,
    )
]

# Study-scope relationship
exact["study_scope_relation"] = [
    scalar_identifier_relation(t0_value, t1_value)
    for t0_value, t1_value in zip(
        exact["t0_study_scope"],
        exact["t1_study_scope"],
    )
]

# Parse normalized condition names
t0_condition_name_sets = [
    normalized_json_set(
        value,
        "t0_condition_names_json",
        normalize_text,
    )
    for value in exact["t0_condition_names_json"]
]

t1_condition_name_sets = [
    normalized_json_set(
        value,
        "t1_condition_names_json",
        normalize_text,
    )
    for value in exact["t1_condition_names_json"]
]

# Parse normalized condition IDs
t0_condition_id_sets = [
    normalized_json_set(
        value,
        "t0_condition_ids_json",
        normalize_condition_id,
    )
    for value in exact["t0_condition_ids_json"]
]

t1_condition_id_sets = [
    normalized_json_set(
        value,
        "t1_condition_ids_json",
        normalize_condition_id,
    )
    for value in exact["t1_condition_ids_json"]
]

exact["t0_condition_names_normalized_json"] = [
    tuple_to_json(values)
    for values in t0_condition_name_sets
]

exact["t1_condition_names_normalized_json"] = [
    tuple_to_json(values)
    for values in t1_condition_name_sets
]

exact["t0_condition_ids_normalized_json"] = [
    tuple_to_json(values)
    for values in t0_condition_id_sets
]

exact["t1_condition_ids_normalized_json"] = [
    tuple_to_json(values)
    for values in t1_condition_id_sets
]

exact["condition_name_relation"] = [
    set_relation(t0_values, t1_values)
    for t0_values, t1_values in zip(
        t0_condition_name_sets,
        t1_condition_name_sets,
    )
]

exact["condition_id_relation"] = [
    set_relation(t0_values, t1_values)
    for t0_values, t1_values in zip(
        t0_condition_id_sets,
        t1_condition_id_sets,
    )
]

exact["condition_continuity_status"] = [
    combined_condition_status(id_relation, name_relation)
    for id_relation, name_relation in zip(
        exact["condition_id_relation"],
        exact["condition_name_relation"],
    )
]

# T1 classification-axis audit
exact["t1_classification_axis_canonical"] = [
    canonical_axis(value)
    for value in exact["t1_aggregate_classification_axis"]
]

exact["classification_axis_audit_category"] = [
    axis_audit_category(
        t1_gene_values,
        t1_axis,
    )
    for t1_gene_values, t1_axis in zip(
        t1_gene_sets,
        exact["t1_classification_axis_canonical"],
    )
]


# -------------------------------------------------------------------------------------------------
# 6. CREATE REVIEW FLAGS
# -------------------------------------------------------------------------------------------------

review_reasons = [
    build_review_reasons(row)
    for _, row in exact.iterrows()
]

exact["review_reasons_json"] = [
    json.dumps(
        reasons,
        separators=(",", ":"),
    )
    for reasons in review_reasons
]

exact["requires_linkage_review"] = [
    len(reasons) > 0
    for reasons in review_reasons
]

exact["review_reason_count"] = [
    len(reasons)
    for reasons in review_reasons
]


# -------------------------------------------------------------------------------------------------
# 7. STRUCTURAL VALIDATION
# -------------------------------------------------------------------------------------------------

if exact["crosswalk_id"].isna().any():
    raise AssertionError(
        "Missing crosswalk IDs were found in the exact-match audit."
    )

if exact["rcv_accession_normalized"].isna().any():
    raise AssertionError(
        "Missing normalized RCV accessions were found."
    )

if not exact["linkage_status"].eq("EXACT_RCV_MATCH").all():
    raise AssertionError(
        "A non-exact linkage row entered the Stage 3C audit."
    )

if not exact["exact_rcv_accession_match"].eq(True).all():
    raise AssertionError(
        "An exact-match flag was False inside the Stage 3C cohort."
    )

matched_rcv_mismatches = int(
    (
        exact["t0_rcv_accession"]
        .astype("string")
        .str.strip()
        .str.upper()
        != exact["t1_rcv_accession"]
        .astype("string")
        .str.strip()
        .str.upper()
    ).sum()
)

if matched_rcv_mismatches != 0:
    raise AssertionError(
        f"{matched_rcv_mismatches:,} exact-match rows have unequal "
        "T0 and T1 RCV accession strings."
    )


# -------------------------------------------------------------------------------------------------
# 8. SELECT AND ORDER AUDIT OUTPUT COLUMNS
# -------------------------------------------------------------------------------------------------

audit_columns = [
    "crosswalk_id",
    "rcv_accession_normalized",
    "linkage_status",

    "t0_release_label",
    "t1_release_label",
    "t0_embedded_data_cutoff_date",
    "t1_embedded_data_cutoff_date",

    "t0_rcv_accession",
    "t1_rcv_accession",
    "t0_rcv_version",
    "t1_rcv_version",
    "rcv_version_relation",

    "t0_variation_id",
    "t1_variation_id",
    "variation_id_relation",

    "t0_vcv_accession",
    "t1_vcv_accession",
    "vcv_accession_relation",

    "t0_vcv_version",
    "t1_vcv_version",
    "vcv_version_relation",

    "core_identifier_continuity_status",

    "t0_target_genes_normalized_json",
    "t1_target_genes_normalized_json",
    "gene_set_relation",

    "t0_study_scope",
    "t1_study_scope",
    "study_scope_relation",

    "t0_condition_names_normalized_json",
    "t1_condition_names_normalized_json",
    "condition_name_relation",

    "t0_condition_ids_normalized_json",
    "t1_condition_ids_normalized_json",
    "condition_id_relation",

    "condition_continuity_status",

    "t1_aggregate_classification_axis",
    "t1_classification_axis_canonical",
    "classification_axis_audit_category",

    "requires_linkage_review",
    "review_reason_count",
    "review_reasons_json",
]

audit = exact[audit_columns].copy()

audit = (
    audit.sort_values(
        by="rcv_accession_normalized",
        kind="mergesort",
    )
    .reset_index(drop=True)
)


# -------------------------------------------------------------------------------------------------
# 9. WRITE AND VERIFY THE AUDIT PARQUET
# -------------------------------------------------------------------------------------------------

audit.to_parquet(
    AUDIT_PATH,
    engine="pyarrow",
    compression="zstd",
    index=False,
)

audit_sha256 = sha256_file(AUDIT_PATH)

audit_metadata = pq.ParquetFile(AUDIT_PATH).metadata
audit_readback = pd.read_parquet(AUDIT_PATH)

if len(audit_readback) != len(audit):
    raise AssertionError(
        "Audit readback row count does not match the written dataframe."
    )

if list(audit_readback.columns) != list(audit.columns):
    raise AssertionError(
        "Audit readback columns do not match the written dataframe."
    )

if audit_readback["crosswalk_id"].duplicated().any():
    raise AssertionError(
        "Duplicate crosswalk IDs were detected after audit readback."
    )


# -------------------------------------------------------------------------------------------------
# 10. CREATE SUMMARY REPORT
# -------------------------------------------------------------------------------------------------

review_reason_counter = Counter()

for reasons in review_reasons:
    review_reason_counter.update(reasons)

report = {
    "report_name": "Stage 3C Exact-Match Continuity Audit Report",
    "report_version": "1.0.0",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "study_stage": "Stage 3C",
    "scientific_unit": "RCV-level variant-condition aggregate",
    "input_exact_match_rows": int(len(exact)),
    "identifier_continuity": {
        "variation_id_relation": value_counts_dict(
            audit["variation_id_relation"]
        ),
        "vcv_accession_relation": value_counts_dict(
            audit["vcv_accession_relation"]
        ),
        "core_identifier_continuity_status": value_counts_dict(
            audit["core_identifier_continuity_status"]
        ),
    },
    "version_continuity": {
        "rcv_version_relation": value_counts_dict(
            audit["rcv_version_relation"]
        ),
        "vcv_version_relation": value_counts_dict(
            audit["vcv_version_relation"]
        ),
    },
    "gene_and_scope_continuity": {
        "gene_set_relation": value_counts_dict(
            audit["gene_set_relation"]
        ),
        "study_scope_relation": value_counts_dict(
            audit["study_scope_relation"]
        ),
    },
    "condition_continuity": {
        "condition_name_relation": value_counts_dict(
            audit["condition_name_relation"]
        ),
        "condition_id_relation": value_counts_dict(
            audit["condition_id_relation"]
        ),
        "condition_continuity_status": value_counts_dict(
            audit["condition_continuity_status"]
        ),
    },
    "classification_axis_audit": {
        "t1_axis_counts": value_counts_dict(
            audit["t1_classification_axis_canonical"]
        ),
        "gene_aware_axis_categories": value_counts_dict(
            audit["classification_axis_audit_category"]
        ),
        "gene_by_axis": nested_counts_dict(
            audit,
            rows=["t1_target_genes_normalized_json"],
            columns=["t1_classification_axis_canonical"],
        ),
        "t0_explicit_axis_available": False,
        "interpretation": (
            "T1 axis categories were audited using the matched target gene. "
            "No direct T0–T1 axis equality was inferred because the historical "
            "T0 schema does not contain the explicit current-schema axis field."
        ),
    },
    "review_queue": {
        "records_requiring_linkage_review": int(
            audit["requires_linkage_review"].sum()
        ),
        "records_without_review_flags": int(
            (~audit["requires_linkage_review"]).sum()
        ),
        "review_reason_counts": {
            key: int(value)
            for key, value in sorted(review_reason_counter.items())
        },
    },
    "validation": {
        "matched_rcv_accession_mismatches": int(
            matched_rcv_mismatches
        ),
        "duplicate_crosswalk_ids": int(
            audit["crosswalk_id"].duplicated().sum()
        ),
        "missing_crosswalk_ids": int(
            audit["crosswalk_id"].isna().sum()
        ),
        "readback_passed": True,
        "critical_structural_failures": 0,
    },
    "scientific_boundaries": {
        "classification_changes_calculated": False,
        "future_instability_outcomes_created": False,
        "exact_rcv_matches_removed_due_to_condition_drift": False,
        "classification_axis_equivalence_inferred_across_schemas": False,
        "unmatched_t0_records_labeled_stable": False,
        "merges_splits_or_withdrawals_inferred": False,
        "ges_model_fitted_or_tuned": False,
    },
    "next_authorized_step": (
        "Audit the 1,246 T0-only records and relevant T1-only records for "
        "prespecified non-exact linkage categories, including accession changes, "
        "possible merges, splits, withdrawals, and changed condition associations, "
        "without assigning unmatched records a stable outcome."
    ),
}

with REPORT_PATH.open("w", encoding="utf-8") as handle:
    json.dump(
        report,
        handle,
        indent=2,
        sort_keys=True,
        ensure_ascii=False,
    )

report_sha256 = sha256_file(REPORT_PATH)


# -------------------------------------------------------------------------------------------------
# 11. CREATE THE STAGE 3C MANIFEST
# -------------------------------------------------------------------------------------------------

manifest = {
    "manifest_name": "Stage 3C Exact-Match Continuity Audit Manifest",
    "manifest_version": "1.0.0",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "study_stage": "Stage 3C",
    "audit_status": "EXACT_MATCH_CONTINUITY_AUDIT_COMPLETE",
    "input": {
        "stage3b_crosswalk_path": str(CROSSWALK_PATH),
        "stage3b_crosswalk_sha256": observed_crosswalk_sha256,
        "stage3b_manifest_path": str(STAGE3B_MANIFEST_PATH),
        "stage3b_manifest_sha256": observed_stage3b_manifest_sha256,
        "exact_match_rows": int(len(exact)),
    },
    "outputs": {
        "audit_parquet": {
            "path": str(AUDIT_PATH),
            "sha256": audit_sha256,
            "rows": int(audit_metadata.num_rows),
            "columns": int(audit_metadata.num_columns),
            "row_groups": int(audit_metadata.num_row_groups),
            "compression": "Zstandard",
        },
        "audit_report": {
            "path": str(REPORT_PATH),
            "sha256": report_sha256,
        },
    },
    "audit_components": [
        "VariationID continuity",
        "VCV accession continuity",
        "RCV version continuity",
        "VCV version continuity",
        "target-gene continuity",
        "study-scope continuity",
        "condition-name continuity",
        "structured condition-ID continuity",
        "gene-aware T1 classification-axis audit",
        "transparent linkage-review queue",
    ],
    "validation_decision": "PASS",
    "scientific_boundaries": report["scientific_boundaries"],
    "next_authorized_step": report["next_authorized_step"],
}

with MANIFEST_PATH.open("w", encoding="utf-8") as handle:
    json.dump(
        manifest,
        handle,
        indent=2,
        sort_keys=True,
        ensure_ascii=False,
    )

manifest_sha256 = sha256_file(MANIFEST_PATH)


# -------------------------------------------------------------------------------------------------
# 12. PRINT RESULTS
# -------------------------------------------------------------------------------------------------

print()
print("=" * 120)
print("STAGE 3C CONTINUITY-AUDIT SUMMARY")
print("=" * 120)

print(f"Exact RCV matches audited:                 {len(audit):,}")
print(
    f"Stable VariationID + VCV continuity:       "
    f"{audit['core_identifier_continuity_status'].eq('STABLE_CORE_IDENTIFIERS').sum():,}"
)
print(
    f"Exact target-gene continuity:              "
    f"{audit['gene_set_relation'].eq('EXACT_SET_MATCH').sum():,}"
)
print(
    f"Condition IDs and names both consistent:   "
    f"{audit['condition_continuity_status'].eq('CONDITION_IDS_AND_NAMES_CONSISTENT').sum():,}"
)
print(
    f"Records requiring later linkage review:    "
    f"{audit['requires_linkage_review'].sum():,}"
)

print()
print("RCV version relationships:")
print(
    audit["rcv_version_relation"]
    .value_counts(dropna=False)
    .to_string()
)

print()
print("Core identifier continuity:")
print(
    audit["core_identifier_continuity_status"]
    .value_counts(dropna=False)
    .to_string()
)

print()
print("Condition continuity categories:")
print(
    audit["condition_continuity_status"]
    .value_counts(dropna=False)
    .to_string()
)

print()
print("Classification-axis audit categories:")
print(
    audit["classification_axis_audit_category"]
    .value_counts(dropna=False)
    .to_string()
)

print()
print(f"Audit Parquet:        {AUDIT_PATH}")
print(f"Audit SHA-256:        {audit_sha256}")
print(f"Audit report:         {REPORT_PATH}")
print(f"Report SHA-256:       {report_sha256}")
print(f"Manifest:             {MANIFEST_PATH}")
print(f"Manifest SHA-256:     {manifest_sha256}")

print()
print("PASS — Stage 3C exact-match continuity audit completed.")
print(
    "IMPORTANT — No classification-change or future-instability "
    "outcome was created."
)
print(
    "AUTHORIZED NEXT ACTION — Stage 3D audit of the 1,246 T0-only "
    "records and candidate non-exact linkage cases."
)
print("=" * 120)

STAGE 3C — EXACT-MATCH CONTINUITY AUDIT
Stage 3B crosswalk SHA-256: b462304a4bb31db301e2dac3aefbf685e12f1fea179a38ca04b8500b64bb4acc
Stage 3B manifest SHA-256:  bb5274383a83ce70673f43dd97b09bf7c1303c18292b74da577bae38430e1f93
Expected exact matches:     70,413

Exact-match rows loaded: 70,413
Normalizing identifiers, genes, conditions, and classification axes...

STAGE 3C CONTINUITY-AUDIT SUMMARY
Exact RCV matches audited:                 70,413
Stable VariationID + VCV continuity:       70,413
Exact target-gene continuity:              70,413
Condition IDs and names both consistent:   3,160
Records requiring later linkage review:    10,342

RCV version relationships:
rcv_version_relation
VERSION_INCREASED    59113
SAME_VERSION         11300

Core identifier continuity:
core_identifier_continuity_status
STABLE_CORE_IDENTIFIERS    70413

Condition continuity categories:
condition_continuity_status
PARTIAL_CONDITION_ID_CONTINUITY              53578
NAME_CONTINUITY_WITH_ID_CHANGE_REVIEW  

In [15]:
# =================================================================================================
# STAGE 3D — AUDIT T0-ONLY RECORDS AND GENERATE NON-EXACT LINKAGE CANDIDATES
# =================================================================================================
# Purpose:
#   Examine the 1,246 T0-only RCV records against the 30,507 T1-only RCV records.
#
# Candidate generation is limited to records sharing at least one stable variant identifier:
#   - exact VariationID, and/or
#   - exact VCV accession.
#
# Candidate prioritization also audits:
#   - target-gene continuity,
#   - structured condition-ID overlap,
#   - normalized condition-name overlap,
#   - study-scope continuity,
#   - T1 classification axis,
#   - one-to-one, one-to-many, many-to-one, and complex candidate topology.
#
# IMPORTANT:
#   Candidate tiers are review-prioritization categories only.
#   They are NOT accepted non-exact links.
#
# This cell DOES NOT:
#   - automatically replace one RCV accession with another,
#   - confirm merges, splits, withdrawals, or renamed accessions,
#   - compare clinical classifications,
#   - create future-instability outcomes,
#   - classify unmatched records as stable,
#   - alter the Stage 3B exact crosswalk,
#   - fit or tune GES.
# =================================================================================================

from pathlib import Path
from datetime import datetime, timezone
from collections import defaultdict
import hashlib
import json
import re
import unicodedata

import pandas as pd
import pyarrow.parquet as pq


print("=" * 122)
print("STAGE 3D — T0-ONLY AND NON-EXACT LINKAGE-CANDIDATE AUDIT")
print("=" * 122)


# -------------------------------------------------------------------------------------------------
# 1. PATHS AND ACCEPTED CHECKSUMS
# -------------------------------------------------------------------------------------------------

STUDY_ROOT = Path("/content/drive/MyDrive/GES_RAG_Temporal_Study")

STAGE3_CONFIG_DIR = STUDY_ROOT / "configs" / "stage3_crosswalk"
STAGE3_OUTPUT_DIR = STUDY_ROOT / "data_processed" / "stage3_crosswalk"

CROSSWALK_PATH = (
    STAGE3_OUTPUT_DIR
    / "stage3_t0_t1_exact_rcv_crosswalk_draft_v1.parquet"
)

STAGE3B_MANIFEST_PATH = (
    STAGE3_CONFIG_DIR
    / "stage3_t0_t1_exact_rcv_crosswalk_manifest_v1.json"
)

STAGE3C_MANIFEST_PATH = (
    STAGE3_CONFIG_DIR
    / "stage3_t0_t1_exact_match_continuity_manifest_v1.json"
)

CANDIDATE_PAIRS_PATH = (
    STAGE3_OUTPUT_DIR
    / "stage3_t0_unmatched_nonexact_candidate_pairs_v1.parquet"
)

T0_SUMMARY_PATH = (
    STAGE3_OUTPUT_DIR
    / "stage3_t0_unmatched_record_audit_summary_v1.parquet"
)

REPORT_PATH = (
    STAGE3_CONFIG_DIR
    / "stage3_t0_unmatched_nonexact_candidate_report_v1.json"
)

MANIFEST_PATH = (
    STAGE3_CONFIG_DIR
    / "stage3_t0_unmatched_nonexact_candidate_manifest_v1.json"
)

EXPECTED_CROSSWALK_SHA256 = (
    "b462304a4bb31db301e2dac3aefbf685"
    "e12f1fea179a38ca04b8500b64bb4acc"
)

EXPECTED_STAGE3B_MANIFEST_SHA256 = (
    "bb5274383a83ce70673f43dd97b09bf7"
    "c1303c18292b74da577bae38430e1f93"
)

EXPECTED_STAGE3C_MANIFEST_SHA256 = (
    "a71cec44134721d6c620306465ec841b7"
    "1d5050b891221aec8574ec60a17d9e6"
)

EXPECTED_T0_ONLY_RECORDS = 1246
EXPECTED_T1_ONLY_RECORDS = 30507


# -------------------------------------------------------------------------------------------------
# 2. UTILITY FUNCTIONS
# -------------------------------------------------------------------------------------------------

def sha256_file(path: Path, chunk_size: int = 8 * 1024 * 1024) -> str:
    """Calculate a file SHA-256 without reading the whole file into memory."""
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def is_missing(value) -> bool:
    """Return True for None, pandas missing values, and blank scalar strings."""
    if value is None:
        return True

    if isinstance(value, str):
        return value.strip() == ""

    try:
        missing = pd.isna(value)

        if isinstance(missing, bool):
            return missing
    except (TypeError, ValueError):
        pass

    return False


def clean_identifier(value, uppercase: bool = True):
    """
    Normalize identifiers conservatively.

    Numeric values represented as strings such as '123.0' become '123'.
    """
    if is_missing(value):
        return None

    text = str(value).strip()

    if re.fullmatch(r"\d+\.0", text):
        text = text[:-2]

    if uppercase:
        text = text.upper()

    return text or None


def parse_json_list(value, field_name: str):
    """Parse and type-check a serialized JSON list."""
    if is_missing(value):
        return []

    if isinstance(value, list):
        parsed = value
    elif isinstance(value, tuple):
        parsed = list(value)
    else:
        text = str(value).strip()

        try:
            parsed = json.loads(text)
        except json.JSONDecodeError as exc:
            raise ValueError(
                f"Malformed JSON in {field_name}: {text[:200]}"
            ) from exc

    if not isinstance(parsed, list):
        raise TypeError(
            f"{field_name} must contain a JSON list, "
            f"not {type(parsed).__name__}."
        )

    return parsed


def normalize_gene(value):
    """Normalize one gene symbol."""
    if value is None:
        return None

    text = str(value).strip().upper()

    return text or None


def normalize_condition_id(value):
    """Normalize a structured condition identifier."""
    if value is None:
        return None

    text = unicodedata.normalize("NFKC", str(value))
    text = re.sub(r"\s+", "", text).strip().upper()

    return text or None


def normalize_condition_name(value):
    """Normalize condition text conservatively without synonym inference."""
    if value is None:
        return None

    text = unicodedata.normalize("NFKC", str(value))
    text = re.sub(r"\s+", " ", text).strip().casefold()

    return text or None


def normalized_json_tuple(value, field_name: str, normalizer):
    """Return a sorted, deduplicated tuple from a serialized JSON list."""
    parsed = parse_json_list(value, field_name)

    normalized = []

    for item in parsed:
        item_normalized = normalizer(item)

        if item_normalized is not None:
            normalized.append(item_normalized)

    return tuple(sorted(set(normalized)))


def tuple_to_json(values) -> str:
    """Serialize normalized tuple values deterministically."""
    return json.dumps(
        list(values),
        ensure_ascii=False,
        separators=(",", ":"),
    )


def set_relation(left_values, right_values) -> str:
    """Describe the relationship between two normalized sets."""
    left_set = set(left_values)
    right_set = set(right_values)

    if not left_set and not right_set:
        return "BOTH_EMPTY"

    if not left_set:
        return "T0_EMPTY"

    if not right_set:
        return "T1_EMPTY"

    if left_set == right_set:
        return "EXACT_SET_MATCH"

    if left_set.intersection(right_set):
        return "PARTIAL_SET_OVERLAP"

    return "DISJOINT_SETS"


def scalar_relation(left_value, right_value) -> str:
    """Compare two normalized scalar values."""
    left_clean = clean_identifier(left_value)
    right_clean = clean_identifier(right_value)

    if left_clean is None and right_clean is None:
        return "BOTH_MISSING"

    if left_clean is None:
        return "T0_MISSING"

    if right_clean is None:
        return "T1_MISSING"

    if left_clean == right_clean:
        return "SAME"

    return "CHANGED"


def canonical_axis(value) -> str:
    """Map T1 axis wording to a canonical classification-axis category."""
    if is_missing(value):
        return "MissingAxis"

    raw = str(value).strip()
    key = re.sub(r"[^a-z]", "", raw.casefold())

    mapping = {
        "germlineclassification": "GermlineClassification",
        "oncogenicityclassification": "OncogenicityClassification",
        "somaticclinicalimpact": "SomaticClinicalImpact",
        "noclassification": "NoClassification",
    }

    return mapping.get(key, raw)


def condition_evidence_category(
    condition_id_overlap_count: int,
    condition_name_overlap_count: int,
    t0_condition_ids,
    t1_condition_ids,
    t0_condition_names,
    t1_condition_names,
) -> str:
    """Create a transparent condition-continuity evidence category."""

    if (
        condition_id_overlap_count > 0
        and condition_name_overlap_count > 0
    ):
        return "ID_AND_NAME_OVERLAP"

    if condition_id_overlap_count > 0:
        return "ID_OVERLAP_ONLY"

    if condition_name_overlap_count > 0:
        return "NAME_OVERLAP_ONLY"

    if (
        not t0_condition_ids
        and not t1_condition_ids
        and set(t0_condition_names) == set(t1_condition_names)
        and len(t0_condition_names) > 0
    ):
        return "EXACT_NAME_MATCH_IDS_EMPTY_BOTH"

    if (
        not t0_condition_ids
        or not t1_condition_ids
    ):
        return "NO_OVERLAP_WITH_MISSING_CONDITION_IDS"

    return "NO_CONDITION_OVERLAP"


def candidate_tier(
    variation_id_same: bool,
    vcv_accession_same: bool,
    gene_relation: str,
    any_condition_overlap: bool,
) -> str:
    """
    Assign a review-prioritization tier.

    These tiers do not constitute accepted linkage decisions.
    """
    gene_exact = gene_relation == "EXACT_SET_MATCH"
    both_variant_identifiers = variation_id_same and vcv_accession_same
    at_least_one_variant_identifier = (
        variation_id_same or vcv_accession_same
    )

    if (
        both_variant_identifiers
        and gene_exact
        and any_condition_overlap
    ):
        return "TIER_1_VARIANT_GENE_CONDITION_CONTINUITY"

    if (
        at_least_one_variant_identifier
        and gene_exact
        and any_condition_overlap
    ):
        return "TIER_2_PARTIAL_IDENTIFIER_GENE_CONDITION_CONTINUITY"

    if (
        both_variant_identifiers
        and gene_exact
        and not any_condition_overlap
    ):
        return "TIER_3_VARIANT_CONTINUITY_CONDITION_CHANGE_CANDIDATE"

    if (
        at_least_one_variant_identifier
        and gene_exact
        and not any_condition_overlap
    ):
        return "TIER_4_PARTIAL_IDENTIFIER_CONDITION_CHANGE_CANDIDATE"

    return "TIER_5_GENE_DISCORDANT_OR_INCOMPLETE_CANDIDATE"


TIER_RANK = {
    "TIER_1_VARIANT_GENE_CONDITION_CONTINUITY": 1,
    "TIER_2_PARTIAL_IDENTIFIER_GENE_CONDITION_CONTINUITY": 2,
    "TIER_3_VARIANT_CONTINUITY_CONDITION_CHANGE_CANDIDATE": 3,
    "TIER_4_PARTIAL_IDENTIFIER_CONDITION_CHANGE_CANDIDATE": 4,
    "TIER_5_GENE_DISCORDANT_OR_INCOMPLETE_CANDIDATE": 5,
}


def t0_record_audit_category(
    candidate_count: int,
    best_tier_rank,
    best_tier_count: int,
) -> str:
    """Create a T0-level candidate-audit category."""

    if candidate_count == 0:
        return "NO_VARIATIONID_OR_VCV_CANDIDATE"

    if best_tier_rank == 1:
        if best_tier_count == 1:
            return "UNIQUE_TIER_1_CANDIDATE_REQUIRES_ADJUDICATION"

        return "MULTIPLE_TIER_1_CANDIDATES_POSSIBLE_SPLIT_OR_COMPLEX"

    if best_tier_rank == 2:
        if best_tier_count == 1:
            return "UNIQUE_TIER_2_CANDIDATE_REQUIRES_ADJUDICATION"

        return "MULTIPLE_TIER_2_CANDIDATES_REQUIRE_REVIEW"

    if best_tier_rank in {3, 4}:
        if best_tier_count == 1:
            return "UNIQUE_VARIANT_CONTINUITY_WITH_CONDITION_CHANGE_CANDIDATE"

        return "MULTIPLE_VARIANT_CONTINUITY_CONDITION_CHANGE_CANDIDATES"

    return "IDENTIFIER_CANDIDATE_WITH_GENE_DISCORDANCE_OR_MISSINGNESS"


def topology_category(
    strong_t0_degree: int,
    strong_t1_degree: int,
    is_strong: bool,
) -> str:
    """Describe candidate topology without confirming merges or splits."""

    if not is_strong:
        return "NOT_STRONG_CANDIDATE"

    if strong_t0_degree == 1 and strong_t1_degree == 1:
        return "POSSIBLE_ONE_TO_ONE_ACCESSION_REPLACEMENT"

    if strong_t0_degree > 1 and strong_t1_degree == 1:
        return "POSSIBLE_ONE_TO_MANY_SPLIT"

    if strong_t0_degree == 1 and strong_t1_degree > 1:
        return "POSSIBLE_MANY_TO_ONE_MERGE"

    return "POSSIBLE_COMPLEX_MANY_TO_MANY_CHANGE"


def value_counts_dict(series: pd.Series) -> dict:
    """Convert pandas value counts to a JSON-safe dictionary."""
    counts = series.value_counts(dropna=False)

    result = {}

    for key, value in counts.items():
        safe_key = "<MISSING>" if pd.isna(key) else str(key)
        result[safe_key] = int(value)

    return result


# -------------------------------------------------------------------------------------------------
# 3. VERIFY STAGE 3B AND STAGE 3C ARTIFACTS
# -------------------------------------------------------------------------------------------------

required_files = [
    CROSSWALK_PATH,
    STAGE3B_MANIFEST_PATH,
    STAGE3C_MANIFEST_PATH,
]

for required_file in required_files:
    if not required_file.exists():
        raise FileNotFoundError(
            f"Required artifact does not exist: {required_file}"
        )

observed_crosswalk_sha256 = sha256_file(CROSSWALK_PATH)
observed_stage3b_manifest_sha256 = sha256_file(
    STAGE3B_MANIFEST_PATH
)
observed_stage3c_manifest_sha256 = sha256_file(
    STAGE3C_MANIFEST_PATH
)

if observed_crosswalk_sha256 != EXPECTED_CROSSWALK_SHA256:
    raise RuntimeError(
        "Stage 3B crosswalk checksum mismatch.\n"
        f"Expected: {EXPECTED_CROSSWALK_SHA256}\n"
        f"Observed: {observed_crosswalk_sha256}"
    )

if (
    observed_stage3b_manifest_sha256
    != EXPECTED_STAGE3B_MANIFEST_SHA256
):
    raise RuntimeError(
        "Stage 3B manifest checksum mismatch.\n"
        f"Expected: {EXPECTED_STAGE3B_MANIFEST_SHA256}\n"
        f"Observed: {observed_stage3b_manifest_sha256}"
    )

if (
    observed_stage3c_manifest_sha256
    != EXPECTED_STAGE3C_MANIFEST_SHA256
):
    raise RuntimeError(
        "Stage 3C manifest checksum mismatch.\n"
        f"Expected: {EXPECTED_STAGE3C_MANIFEST_SHA256}\n"
        f"Observed: {observed_stage3c_manifest_sha256}"
    )

with STAGE3B_MANIFEST_PATH.open("r", encoding="utf-8") as handle:
    stage3b_manifest = json.load(handle)

with STAGE3C_MANIFEST_PATH.open("r", encoding="utf-8") as handle:
    stage3c_manifest = json.load(handle)

if (
    stage3b_manifest.get("crosswalk_status")
    != "DRAFT_EXACT_RCV_LINKAGE_COMPLETE"
):
    raise RuntimeError(
        "The Stage 3B manifest does not contain the expected status."
    )

if (
    stage3c_manifest.get("audit_status")
    != "EXACT_MATCH_CONTINUITY_AUDIT_COMPLETE"
):
    raise RuntimeError(
        "Stage 3C has not authorized unmatched-record auditing."
    )

print(f"Stage 3B crosswalk SHA-256: {observed_crosswalk_sha256}")
print(f"Stage 3B manifest SHA-256:  {observed_stage3b_manifest_sha256}")
print(f"Stage 3C manifest SHA-256:  {observed_stage3c_manifest_sha256}")


# -------------------------------------------------------------------------------------------------
# 4. LOAD T0-ONLY AND T1-ONLY RECORDS
# -------------------------------------------------------------------------------------------------

required_columns = [
    "crosswalk_id",
    "rcv_accession_normalized",
    "linkage_status",

    "t0_rcv_accession",
    "t0_rcv_version",
    "t0_variation_id",
    "t0_vcv_accession",
    "t0_vcv_version",
    "t0_target_genes_json",
    "t0_study_scope",
    "t0_condition_names_json",
    "t0_condition_ids_json",

    "t1_rcv_accession",
    "t1_rcv_version",
    "t1_variation_id",
    "t1_vcv_accession",
    "t1_vcv_version",
    "t1_target_genes_json",
    "t1_study_scope",
    "t1_condition_names_json",
    "t1_condition_ids_json",
    "t1_aggregate_classification_axis",
]

crosswalk_schema = set(
    pq.ParquetFile(CROSSWALK_PATH).schema_arrow.names
)

missing_columns = sorted(
    set(required_columns) - crosswalk_schema
)

if missing_columns:
    raise KeyError(
        "Crosswalk is missing required columns: "
        + ", ".join(missing_columns)
    )

crosswalk = pd.read_parquet(
    CROSSWALK_PATH,
    columns=required_columns,
)

t0_only = (
    crosswalk.loc[
        crosswalk["linkage_status"].eq(
            "T0_ONLY_UNMATCHED_BY_EXACT_RCV"
        )
    ]
    .copy()
    .reset_index(drop=True)
)

t1_only = (
    crosswalk.loc[
        crosswalk["linkage_status"].eq(
            "T1_ONLY_UNLINKED_BY_EXACT_RCV"
        )
    ]
    .copy()
    .reset_index(drop=True)
)

if len(t0_only) != EXPECTED_T0_ONLY_RECORDS:
    raise AssertionError(
        f"Expected {EXPECTED_T0_ONLY_RECORDS:,} T0-only records, "
        f"observed {len(t0_only):,}."
    )

if len(t1_only) != EXPECTED_T1_ONLY_RECORDS:
    raise AssertionError(
        f"Expected {EXPECTED_T1_ONLY_RECORDS:,} T1-only records, "
        f"observed {len(t1_only):,}."
    )

print()
print(f"T0-only records loaded: {len(t0_only):,}")
print(f"T1-only records loaded: {len(t1_only):,}")


# -------------------------------------------------------------------------------------------------
# 5. NORMALIZE LINKAGE AND AUDIT FIELDS ONCE
# -------------------------------------------------------------------------------------------------

print("Normalizing unmatched identifiers, genes, and conditions...")


def add_normalized_fields(frame: pd.DataFrame, prefix: str) -> pd.DataFrame:
    """Add internal normalized columns for one release side."""

    frame = frame.copy()

    frame["_variation_id"] = [
        clean_identifier(value)
        for value in frame[f"{prefix}_variation_id"]
    ]

    frame["_vcv_accession"] = [
        clean_identifier(value)
        for value in frame[f"{prefix}_vcv_accession"]
    ]

    frame["_rcv_accession"] = [
        clean_identifier(value)
        for value in frame[f"{prefix}_rcv_accession"]
    ]

    frame["_genes"] = [
        normalized_json_tuple(
            value,
            f"{prefix}_target_genes_json",
            normalize_gene,
        )
        for value in frame[f"{prefix}_target_genes_json"]
    ]

    frame["_condition_ids"] = [
        normalized_json_tuple(
            value,
            f"{prefix}_condition_ids_json",
            normalize_condition_id,
        )
        for value in frame[f"{prefix}_condition_ids_json"]
    ]

    frame["_condition_names"] = [
        normalized_json_tuple(
            value,
            f"{prefix}_condition_names_json",
            normalize_condition_name,
        )
        for value in frame[f"{prefix}_condition_names_json"]
    ]

    frame["_study_scope"] = [
        clean_identifier(value)
        for value in frame[f"{prefix}_study_scope"]
    ]

    return frame


t0_only = add_normalized_fields(t0_only, "t0")
t1_only = add_normalized_fields(t1_only, "t1")


# -------------------------------------------------------------------------------------------------
# 6. BUILD T1-ONLY IDENTIFIER INDEXES
# -------------------------------------------------------------------------------------------------

variation_id_index = defaultdict(list)
vcv_accession_index = defaultdict(list)

for t1_index, row in t1_only.iterrows():
    variation_id = row["_variation_id"]
    vcv_accession = row["_vcv_accession"]

    if variation_id is not None:
        variation_id_index[variation_id].append(t1_index)

    if vcv_accession is not None:
        vcv_accession_index[vcv_accession].append(t1_index)


# -------------------------------------------------------------------------------------------------
# 7. GENERATE ALL IDENTIFIER-BASED CANDIDATE PAIRS
# -------------------------------------------------------------------------------------------------

candidate_records = []

for t0_index, t0_row in t0_only.iterrows():
    candidate_t1_indices = set()

    t0_variation_id = t0_row["_variation_id"]
    t0_vcv_accession = t0_row["_vcv_accession"]

    if t0_variation_id is not None:
        candidate_t1_indices.update(
            variation_id_index.get(t0_variation_id, [])
        )

    if t0_vcv_accession is not None:
        candidate_t1_indices.update(
            vcv_accession_index.get(t0_vcv_accession, [])
        )

    for t1_index in sorted(candidate_t1_indices):
        t1_row = t1_only.iloc[t1_index]

        variation_id_same = (
            t0_variation_id is not None
            and t0_variation_id == t1_row["_variation_id"]
        )

        vcv_accession_same = (
            t0_vcv_accession is not None
            and t0_vcv_accession == t1_row["_vcv_accession"]
        )

        if not variation_id_same and not vcv_accession_same:
            raise AssertionError(
                "A candidate pair was generated without exact VariationID "
                "or exact VCV continuity."
            )

        gene_relation = set_relation(
            t0_row["_genes"],
            t1_row["_genes"],
        )

        condition_id_relation = set_relation(
            t0_row["_condition_ids"],
            t1_row["_condition_ids"],
        )

        condition_name_relation = set_relation(
            t0_row["_condition_names"],
            t1_row["_condition_names"],
        )

        condition_id_overlap = sorted(
            set(t0_row["_condition_ids"]).intersection(
                t1_row["_condition_ids"]
            )
        )

        condition_name_overlap = sorted(
            set(t0_row["_condition_names"]).intersection(
                t1_row["_condition_names"]
            )
        )

        condition_id_overlap_count = len(condition_id_overlap)
        condition_name_overlap_count = len(
            condition_name_overlap
        )

        any_condition_overlap = (
            condition_id_overlap_count > 0
            or condition_name_overlap_count > 0
        )

        condition_evidence = condition_evidence_category(
            condition_id_overlap_count=condition_id_overlap_count,
            condition_name_overlap_count=condition_name_overlap_count,
            t0_condition_ids=t0_row["_condition_ids"],
            t1_condition_ids=t1_row["_condition_ids"],
            t0_condition_names=t0_row["_condition_names"],
            t1_condition_names=t1_row["_condition_names"],
        )

        tier = candidate_tier(
            variation_id_same=variation_id_same,
            vcv_accession_same=vcv_accession_same,
            gene_relation=gene_relation,
            any_condition_overlap=any_condition_overlap,
        )

        rule_sources = []

        if variation_id_same:
            rule_sources.append("EXACT_VARIATION_ID")

        if vcv_accession_same:
            rule_sources.append("EXACT_VCV_ACCESSION")

        candidate_records.append(
            {
                "candidate_pair_id": (
                    f"{t0_row['t0_rcv_accession']}"
                    f"__TO__{t1_row['t1_rcv_accession']}"
                ),

                "t0_crosswalk_id": t0_row["crosswalk_id"],
                "t1_crosswalk_id": t1_row["crosswalk_id"],

                "t0_rcv_accession": t0_row["t0_rcv_accession"],
                "t1_rcv_accession": t1_row["t1_rcv_accession"],

                "t0_rcv_version": t0_row["t0_rcv_version"],
                "t1_rcv_version": t1_row["t1_rcv_version"],

                "t0_variation_id": t0_row["t0_variation_id"],
                "t1_variation_id": t1_row["t1_variation_id"],
                "variation_id_same": bool(variation_id_same),

                "t0_vcv_accession": t0_row["t0_vcv_accession"],
                "t1_vcv_accession": t1_row["t1_vcv_accession"],
                "vcv_accession_same": bool(vcv_accession_same),

                "t0_vcv_version": t0_row["t0_vcv_version"],
                "t1_vcv_version": t1_row["t1_vcv_version"],

                "candidate_rule_sources_json": json.dumps(
                    rule_sources,
                    separators=(",", ":"),
                ),

                "t0_genes_normalized_json": tuple_to_json(
                    t0_row["_genes"]
                ),
                "t1_genes_normalized_json": tuple_to_json(
                    t1_row["_genes"]
                ),
                "gene_relation": gene_relation,

                "t0_study_scope": t0_row["t0_study_scope"],
                "t1_study_scope": t1_row["t1_study_scope"],
                "study_scope_relation": scalar_relation(
                    t0_row["t0_study_scope"],
                    t1_row["t1_study_scope"],
                ),

                "t0_condition_ids_normalized_json": tuple_to_json(
                    t0_row["_condition_ids"]
                ),
                "t1_condition_ids_normalized_json": tuple_to_json(
                    t1_row["_condition_ids"]
                ),
                "condition_id_relation": condition_id_relation,
                "condition_id_overlap_count": int(
                    condition_id_overlap_count
                ),
                "condition_id_overlap_json": json.dumps(
                    condition_id_overlap,
                    ensure_ascii=False,
                    separators=(",", ":"),
                ),

                "t0_condition_names_normalized_json": tuple_to_json(
                    t0_row["_condition_names"]
                ),
                "t1_condition_names_normalized_json": tuple_to_json(
                    t1_row["_condition_names"]
                ),
                "condition_name_relation": condition_name_relation,
                "condition_name_overlap_count": int(
                    condition_name_overlap_count
                ),
                "condition_name_overlap_json": json.dumps(
                    condition_name_overlap,
                    ensure_ascii=False,
                    separators=(",", ":"),
                ),

                "any_condition_overlap": bool(
                    any_condition_overlap
                ),
                "condition_evidence_category": condition_evidence,

                "t1_classification_axis": canonical_axis(
                    t1_row["t1_aggregate_classification_axis"]
                ),

                "candidate_tier": tier,
                "candidate_tier_rank": int(TIER_RANK[tier]),

                "candidate_status": (
                    "UNADJUDICATED_NONEXACT_CANDIDATE"
                ),
                "accepted_nonexact_link": False,
            }
        )


candidate_columns = [
    "candidate_pair_id",
    "t0_crosswalk_id",
    "t1_crosswalk_id",
    "t0_rcv_accession",
    "t1_rcv_accession",
    "t0_rcv_version",
    "t1_rcv_version",
    "t0_variation_id",
    "t1_variation_id",
    "variation_id_same",
    "t0_vcv_accession",
    "t1_vcv_accession",
    "vcv_accession_same",
    "t0_vcv_version",
    "t1_vcv_version",
    "candidate_rule_sources_json",
    "t0_genes_normalized_json",
    "t1_genes_normalized_json",
    "gene_relation",
    "t0_study_scope",
    "t1_study_scope",
    "study_scope_relation",
    "t0_condition_ids_normalized_json",
    "t1_condition_ids_normalized_json",
    "condition_id_relation",
    "condition_id_overlap_count",
    "condition_id_overlap_json",
    "t0_condition_names_normalized_json",
    "t1_condition_names_normalized_json",
    "condition_name_relation",
    "condition_name_overlap_count",
    "condition_name_overlap_json",
    "any_condition_overlap",
    "condition_evidence_category",
    "t1_classification_axis",
    "candidate_tier",
    "candidate_tier_rank",
    "candidate_status",
    "accepted_nonexact_link",
]

candidate_pairs = pd.DataFrame(
    candidate_records,
    columns=candidate_columns,
)

if not candidate_pairs.empty:
    candidate_pairs = (
        candidate_pairs.sort_values(
            by=[
                "t0_rcv_accession",
                "candidate_tier_rank",
                "t1_rcv_accession",
            ],
            kind="mergesort",
        )
        .reset_index(drop=True)
    )


# -------------------------------------------------------------------------------------------------
# 8. VALIDATE CANDIDATE-PAIR CONSTRUCTION
# -------------------------------------------------------------------------------------------------

if candidate_pairs["candidate_pair_id"].duplicated().any():
    raise AssertionError(
        "Duplicate non-exact candidate-pair identifiers were generated."
    )

if (
    candidate_pairs["t0_rcv_accession"]
    .astype("string")
    .str.strip()
    .str.upper()
    ==
    candidate_pairs["t1_rcv_accession"]
    .astype("string")
    .str.strip()
    .str.upper()
).any():
    raise AssertionError(
        "An exact RCV-accession match entered the non-exact candidate file."
    )

invalid_identifier_candidates = int(
    (
        ~(
            candidate_pairs["variation_id_same"]
            | candidate_pairs["vcv_accession_same"]
        )
    ).sum()
)

if invalid_identifier_candidates != 0:
    raise AssertionError(
        f"{invalid_identifier_candidates:,} candidate pairs lack "
        "VariationID and VCV continuity."
    )


# -------------------------------------------------------------------------------------------------
# 9. CALCULATE STRONG-CANDIDATE DEGREES AND POSSIBLE TOPOLOGY
# -------------------------------------------------------------------------------------------------

strong_tiers = {
    "TIER_1_VARIANT_GENE_CONDITION_CONTINUITY",
    "TIER_2_PARTIAL_IDENTIFIER_GENE_CONDITION_CONTINUITY",
}

candidate_pairs["is_strong_candidate"] = (
    candidate_pairs["candidate_tier"].isin(strong_tiers)
)

strong_pairs = candidate_pairs.loc[
    candidate_pairs["is_strong_candidate"]
].copy()

strong_t0_degree = (
    strong_pairs.groupby("t0_crosswalk_id")
    .size()
    .to_dict()
)

strong_t1_degree = (
    strong_pairs.groupby("t1_crosswalk_id")
    .size()
    .to_dict()
)

candidate_pairs["strong_candidates_for_t0"] = [
    int(strong_t0_degree.get(value, 0))
    for value in candidate_pairs["t0_crosswalk_id"]
]

candidate_pairs["strong_candidates_for_t1"] = [
    int(strong_t1_degree.get(value, 0))
    for value in candidate_pairs["t1_crosswalk_id"]
]

candidate_pairs["candidate_topology"] = [
    topology_category(
        strong_t0_degree=int(t0_degree),
        strong_t1_degree=int(t1_degree),
        is_strong=bool(is_strong),
    )
    for t0_degree, t1_degree, is_strong in zip(
        candidate_pairs["strong_candidates_for_t0"],
        candidate_pairs["strong_candidates_for_t1"],
        candidate_pairs["is_strong_candidate"],
    )
]


# -------------------------------------------------------------------------------------------------
# 10. CREATE ONE SUMMARY RECORD FOR EACH OF THE 1,246 T0-ONLY RECORDS
# -------------------------------------------------------------------------------------------------

pairs_by_t0 = {
    key: group.copy()
    for key, group in candidate_pairs.groupby(
        "t0_crosswalk_id",
        sort=False,
    )
}

t0_summary_records = []

for _, t0_row in t0_only.iterrows():
    t0_crosswalk_id = t0_row["crosswalk_id"]
    group = pairs_by_t0.get(t0_crosswalk_id)

    if group is None or group.empty:
        candidate_count = 0
        strong_candidate_count = 0
        best_tier_rank = None
        best_tier = None
        best_tier_count = 0
        unique_best_t1_rcv = None
        topology_values = []
        tier_counts = {}
    else:
        candidate_count = int(len(group))
        strong_candidate_count = int(
            group["is_strong_candidate"].sum()
        )

        best_tier_rank = int(
            group["candidate_tier_rank"].min()
        )

        best_group = group.loc[
            group["candidate_tier_rank"].eq(best_tier_rank)
        ]

        best_tier = str(
            best_group["candidate_tier"].iloc[0]
        )

        best_tier_count = int(len(best_group))

        unique_best_t1_rcv = (
            str(best_group["t1_rcv_accession"].iloc[0])
            if best_tier_count == 1
            else None
        )

        topology_values = sorted(
            set(
                group.loc[
                    group["is_strong_candidate"],
                    "candidate_topology",
                ].tolist()
            )
        )

        tier_counts = {
            str(key): int(value)
            for key, value in (
                group["candidate_tier"]
                .value_counts()
                .sort_index()
                .items()
            )
        }

    audit_category = t0_record_audit_category(
        candidate_count=candidate_count,
        best_tier_rank=best_tier_rank,
        best_tier_count=best_tier_count,
    )

    t0_summary_records.append(
        {
            "t0_crosswalk_id": t0_crosswalk_id,
            "t0_rcv_accession": t0_row["t0_rcv_accession"],
            "t0_rcv_version": t0_row["t0_rcv_version"],
            "t0_variation_id": t0_row["t0_variation_id"],
            "t0_vcv_accession": t0_row["t0_vcv_accession"],
            "t0_vcv_version": t0_row["t0_vcv_version"],

            "t0_genes_normalized_json": tuple_to_json(
                t0_row["_genes"]
            ),
            "t0_study_scope": t0_row["t0_study_scope"],
            "t0_condition_ids_normalized_json": tuple_to_json(
                t0_row["_condition_ids"]
            ),
            "t0_condition_names_normalized_json": tuple_to_json(
                t0_row["_condition_names"]
            ),

            "identifier_based_candidate_count": int(
                candidate_count
            ),
            "strong_candidate_count": int(
                strong_candidate_count
            ),

            "best_candidate_tier_rank": best_tier_rank,
            "best_candidate_tier": best_tier,
            "best_tier_candidate_count": int(
                best_tier_count
            ),
            "unique_best_candidate_t1_rcv": (
                unique_best_t1_rcv
            ),

            "candidate_tier_counts_json": json.dumps(
                tier_counts,
                sort_keys=True,
                separators=(",", ":"),
            ),
            "strong_candidate_topologies_json": json.dumps(
                topology_values,
                sort_keys=True,
                separators=(",", ":"),
            ),

            "t0_unmatched_audit_category": audit_category,

            "accepted_nonexact_link": False,
            "future_instability_outcome_created": False,
            "unmatched_labeled_stable": False,
        }
    )


t0_summary = pd.DataFrame(t0_summary_records)

t0_summary = (
    t0_summary.sort_values(
        by="t0_rcv_accession",
        kind="mergesort",
    )
    .reset_index(drop=True)
)


# -------------------------------------------------------------------------------------------------
# 11. VALIDATE T0 SUMMARY ACCOUNTING
# -------------------------------------------------------------------------------------------------

if len(t0_summary) != EXPECTED_T0_ONLY_RECORDS:
    raise AssertionError(
        "T0 unmatched-summary row count does not equal 1,246."
    )

if t0_summary["t0_crosswalk_id"].duplicated().any():
    raise AssertionError(
        "Duplicate T0 crosswalk IDs were found in the summary."
    )

t0_records_with_candidates = int(
    t0_summary[
        "identifier_based_candidate_count"
    ].gt(0).sum()
)

t0_records_without_candidates = int(
    t0_summary[
        "identifier_based_candidate_count"
    ].eq(0).sum()
)

if (
    t0_records_with_candidates
    + t0_records_without_candidates
    != EXPECTED_T0_ONLY_RECORDS
):
    raise AssertionError(
        "T0 candidate/no-candidate accounting failed."
    )

if (
    t0_summary["accepted_nonexact_link"].any()
    or t0_summary["future_instability_outcome_created"].any()
    or t0_summary["unmatched_labeled_stable"].any()
):
    raise AssertionError(
        "A prohibited linkage or outcome flag was set."
    )


# -------------------------------------------------------------------------------------------------
# 12. WRITE AND READ BACK OUTPUT PARQUETS
# -------------------------------------------------------------------------------------------------

candidate_pairs.to_parquet(
    CANDIDATE_PAIRS_PATH,
    engine="pyarrow",
    compression="zstd",
    index=False,
)

t0_summary.to_parquet(
    T0_SUMMARY_PATH,
    engine="pyarrow",
    compression="zstd",
    index=False,
)

candidate_pairs_sha256 = sha256_file(
    CANDIDATE_PAIRS_PATH
)
t0_summary_sha256 = sha256_file(
    T0_SUMMARY_PATH
)

candidate_pairs_metadata = pq.ParquetFile(
    CANDIDATE_PAIRS_PATH
).metadata

t0_summary_metadata = pq.ParquetFile(
    T0_SUMMARY_PATH
).metadata

candidate_pairs_readback = pd.read_parquet(
    CANDIDATE_PAIRS_PATH
)
t0_summary_readback = pd.read_parquet(
    T0_SUMMARY_PATH
)

if len(candidate_pairs_readback) != len(candidate_pairs):
    raise AssertionError(
        "Candidate-pair readback row count mismatch."
    )

if len(t0_summary_readback) != len(t0_summary):
    raise AssertionError(
        "T0-summary readback row count mismatch."
    )

if list(candidate_pairs_readback.columns) != list(
    candidate_pairs.columns
):
    raise AssertionError(
        "Candidate-pair readback schema mismatch."
    )

if list(t0_summary_readback.columns) != list(
    t0_summary.columns
):
    raise AssertionError(
        "T0-summary readback schema mismatch."
    )


# -------------------------------------------------------------------------------------------------
# 13. CREATE REPORT
# -------------------------------------------------------------------------------------------------

report = {
    "report_name": (
        "Stage 3D T0-Unmatched Non-Exact Linkage Candidate Report"
    ),
    "report_version": "1.0.0",
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "study_stage": "Stage 3D",
    "scientific_unit": (
        "RCV-level variant-condition aggregate"
    ),

    "input_counts": {
        "t0_only_records": int(len(t0_only)),
        "t1_only_records": int(len(t1_only)),
    },

    "candidate_generation_rule": {
        "required": (
            "Exact VariationID and/or exact VCV accession"
        ),
        "condition_synonym_inference_used": False,
        "gene_similarity_inference_used": False,
        "classification_information_used": False,
    },

    "candidate_pair_results": {
        "candidate_pairs": int(len(candidate_pairs)),
        "strong_candidate_pairs": int(
            candidate_pairs[
                "is_strong_candidate"
            ].sum()
        ),
        "candidate_tier_counts": value_counts_dict(
            candidate_pairs["candidate_tier"]
        ),
        "candidate_topology_counts": value_counts_dict(
            candidate_pairs["candidate_topology"]
        ),
        "condition_evidence_counts": value_counts_dict(
            candidate_pairs[
                "condition_evidence_category"
            ]
        ),
    },

    "t0_record_results": {
        "t0_records_with_identifier_candidate": int(
            t0_records_with_candidates
        ),
        "t0_records_without_identifier_candidate": int(
            t0_records_without_candidates
        ),
        "t0_records_with_strong_candidate": int(
            t0_summary["strong_candidate_count"].gt(0).sum()
        ),
        "t0_audit_category_counts": value_counts_dict(
            t0_summary[
                "t0_unmatched_audit_category"
            ]
        ),
    },

    "validation": {
        "duplicate_candidate_pair_ids": int(
            candidate_pairs[
                "candidate_pair_id"
            ].duplicated().sum()
        ),
        "invalid_identifier_candidates": int(
            invalid_identifier_candidates
        ),
        "exact_rcv_pairs_in_candidate_file": 0,
        "t0_summary_rows": int(len(t0_summary)),
        "t0_summary_unique_keys": int(
            t0_summary[
                "t0_crosswalk_id"
            ].nunique()
        ),
        "readback_passed": True,
        "critical_failures": 0,
    },

    "scientific_boundaries": {
        "candidate_pairs_are_confirmed_links": False,
        "nonexact_links_accepted": False,
        "merges_confirmed": False,
        "splits_confirmed": False,
        "withdrawals_confirmed": False,
        "replacement_accessions_confirmed": False,
        "classification_changes_calculated": False,
        "future_instability_outcomes_created": False,
        "unmatched_records_labeled_stable": False,
        "ges_model_fitted_or_tuned": False,
    },

    "next_authorized_step": (
        "Adjudicate Tier 1 and Tier 2 candidate pairs using "
        "prespecified one-to-one, merge, split, condition-association, "
        "and classification-axis rules. Preserve unresolved cases as "
        "unmatched or censored and do not construct temporal outcomes "
        "until the full linkage artifact is frozen."
    ),
}

with REPORT_PATH.open("w", encoding="utf-8") as handle:
    json.dump(
        report,
        handle,
        indent=2,
        sort_keys=True,
        ensure_ascii=False,
    )

report_sha256 = sha256_file(REPORT_PATH)


# -------------------------------------------------------------------------------------------------
# 14. CREATE MANIFEST
# -------------------------------------------------------------------------------------------------

manifest = {
    "manifest_name": (
        "Stage 3D T0-Unmatched Non-Exact Candidate Manifest"
    ),
    "manifest_version": "1.0.0",
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "study_stage": "Stage 3D",
    "audit_status": (
        "NONEXACT_CANDIDATE_GENERATION_COMPLETE"
    ),

    "inputs": {
        "stage3b_crosswalk": {
            "path": str(CROSSWALK_PATH),
            "sha256": observed_crosswalk_sha256,
        },
        "stage3b_manifest": {
            "path": str(STAGE3B_MANIFEST_PATH),
            "sha256": observed_stage3b_manifest_sha256,
        },
        "stage3c_manifest": {
            "path": str(STAGE3C_MANIFEST_PATH),
            "sha256": observed_stage3c_manifest_sha256,
        },
    },

    "outputs": {
        "candidate_pairs": {
            "path": str(CANDIDATE_PAIRS_PATH),
            "sha256": candidate_pairs_sha256,
            "rows": int(
                candidate_pairs_metadata.num_rows
            ),
            "columns": int(
                candidate_pairs_metadata.num_columns
            ),
            "row_groups": int(
                candidate_pairs_metadata.num_row_groups
            ),
        },
        "t0_unmatched_summary": {
            "path": str(T0_SUMMARY_PATH),
            "sha256": t0_summary_sha256,
            "rows": int(
                t0_summary_metadata.num_rows
            ),
            "columns": int(
                t0_summary_metadata.num_columns
            ),
            "row_groups": int(
                t0_summary_metadata.num_row_groups
            ),
        },
        "report": {
            "path": str(REPORT_PATH),
            "sha256": report_sha256,
        },
    },

    "candidate_tiers_are_review_only": True,
    "validation_decision": "PASS",
    "scientific_boundaries": (
        report["scientific_boundaries"]
    ),
    "next_authorized_step": (
        report["next_authorized_step"]
    ),
}

with MANIFEST_PATH.open("w", encoding="utf-8") as handle:
    json.dump(
        manifest,
        handle,
        indent=2,
        sort_keys=True,
        ensure_ascii=False,
    )

manifest_sha256 = sha256_file(MANIFEST_PATH)


# -------------------------------------------------------------------------------------------------
# 15. PRINT RESULTS
# -------------------------------------------------------------------------------------------------

print()
print("=" * 122)
print("STAGE 3D NON-EXACT CANDIDATE-AUDIT SUMMARY")
print("=" * 122)

print(f"T0-only records audited:                    {len(t0_summary):,}")
print(f"T1-only candidate pool:                     {len(t1_only):,}")
print(f"Identifier-based candidate pairs:           {len(candidate_pairs):,}")
print(
    f"Strong Tier 1/Tier 2 candidate pairs:       "
    f"{candidate_pairs['is_strong_candidate'].sum():,}"
)
print(
    f"T0 records with ≥1 identifier candidate:    "
    f"{t0_records_with_candidates:,}"
)
print(
    f"T0 records with no identifier candidate:    "
    f"{t0_records_without_candidates:,}"
)
print(
    f"T0 records with ≥1 strong candidate:        "
    f"{t0_summary['strong_candidate_count'].gt(0).sum():,}"
)

print()
print("Candidate-pair tiers:")
print(
    candidate_pairs["candidate_tier"]
    .value_counts(dropna=False)
    .to_string()
)

print()
print("T0 unmatched audit categories:")
print(
    t0_summary["t0_unmatched_audit_category"]
    .value_counts(dropna=False)
    .to_string()
)

print()
print("Strong-candidate topology:")
print(
    candidate_pairs.loc[
        candidate_pairs["is_strong_candidate"],
        "candidate_topology",
    ]
    .value_counts(dropna=False)
    .to_string()
)

print()
print(f"Candidate pairs:        {CANDIDATE_PAIRS_PATH}")
print(f"Candidate SHA-256:      {candidate_pairs_sha256}")
print(f"T0 audit summary:       {T0_SUMMARY_PATH}")
print(f"T0 summary SHA-256:     {t0_summary_sha256}")
print(f"Audit report:           {REPORT_PATH}")
print(f"Report SHA-256:         {report_sha256}")
print(f"Manifest:               {MANIFEST_PATH}")
print(f"Manifest SHA-256:       {manifest_sha256}")

print()
print("PASS — Stage 3D non-exact candidate audit completed.")
print(
    "IMPORTANT — Candidate pairs are unadjudicated and no "
    "non-exact link has been accepted."
)
print(
    "IMPORTANT — No unmatched T0 record was labeled stable and "
    "no temporal outcome was created."
)
print(
    "AUTHORIZED NEXT ACTION — Stage 3E adjudication of Tier 1 and "
    "Tier 2 candidate relationships."
)
print("=" * 122)

STAGE 3D — T0-ONLY AND NON-EXACT LINKAGE-CANDIDATE AUDIT
Stage 3B crosswalk SHA-256: b462304a4bb31db301e2dac3aefbf685e12f1fea179a38ca04b8500b64bb4acc
Stage 3B manifest SHA-256:  bb5274383a83ce70673f43dd97b09bf7c1303c18292b74da577bae38430e1f93
Stage 3C manifest SHA-256:  a71cec44134721d6c620306465ec841b71d5050b891221aec8574ec60a17d9e6

T0-only records loaded: 1,246
T1-only records loaded: 30,507
Normalizing unmatched identifiers, genes, and conditions...

STAGE 3D NON-EXACT CANDIDATE-AUDIT SUMMARY
T0-only records audited:                    1,246
T1-only candidate pool:                     30,507
Identifier-based candidate pairs:           1,078
Strong Tier 1/Tier 2 candidate pairs:       248
T0 records with ≥1 identifier candidate:    624
T0 records with no identifier candidate:    622
T0 records with ≥1 strong candidate:        208

Candidate-pair tiers:
candidate_tier
TIER_3_VARIANT_CONTINUITY_CONDITION_CHANGE_CANDIDATE    830
TIER_1_VARIANT_GENE_CONDITION_CONTINUITY                2

In [16]:
# =================================================================================================
# STAGE 3E — CONSERVATIVE ADJUDICATION OF TIER 1 AND TIER 2 NON-EXACT CANDIDATES
# =================================================================================================
# Purpose:
#   Apply a frozen, conservative adjudication rule to strong non-exact candidates.
#
# Automatic acceptance requires ALL of the following:
#   1. Tier 1 candidate.
#   2. Exact VariationID continuity.
#   3. Exact VCV accession continuity.
#   4. Exact target-gene continuity.
#   5. At least one structured condition-ID or normalized condition-name overlap.
#   6. Unique strong candidate from T0 to T1.
#   7. Unique strong candidate from T1 to T0.
#
# Possible splits, merges, complex relationships, Tier 2 candidates, Tier 3 candidates,
# and records without identifier candidates remain unresolved.
#
# This cell DOES NOT:
#   - confirm that an RCV was officially replaced,
#   - accept split, merge, or complex mappings,
#   - compare aggregate classifications,
#   - construct future-instability outcomes,
#   - label unresolved records as stable,
#   - fit or tune GES,
#   - overwrite the Stage 3B exact crosswalk.
# =================================================================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

import pandas as pd
import pyarrow.parquet as pq


print("=" * 124)
print("STAGE 3E — CONSERVATIVE STRONG-CANDIDATE ADJUDICATION")
print("=" * 124)


# -------------------------------------------------------------------------------------------------
# 1. PATHS AND EXPECTED STAGE 3D CHECKSUMS
# -------------------------------------------------------------------------------------------------

STUDY_ROOT = Path("/content/drive/MyDrive/GES_RAG_Temporal_Study")

STAGE3_CONFIG_DIR = STUDY_ROOT / "configs" / "stage3_crosswalk"
STAGE3_OUTPUT_DIR = STUDY_ROOT / "data_processed" / "stage3_crosswalk"

CANDIDATE_PAIRS_PATH = (
    STAGE3_OUTPUT_DIR
    / "stage3_t0_unmatched_nonexact_candidate_pairs_v1.parquet"
)

T0_STAGE3D_SUMMARY_PATH = (
    STAGE3_OUTPUT_DIR
    / "stage3_t0_unmatched_record_audit_summary_v1.parquet"
)

STAGE3D_MANIFEST_PATH = (
    STAGE3_CONFIG_DIR
    / "stage3_t0_unmatched_nonexact_candidate_manifest_v1.json"
)

RULES_PATH = (
    STAGE3_CONFIG_DIR
    / "stage3_nonexact_candidate_adjudication_rules_v1.json"
)

ADJUDICATION_PATH = (
    STAGE3_OUTPUT_DIR
    / "stage3_nonexact_candidate_adjudication_v1.parquet"
)

ACCEPTED_LINKS_PATH = (
    STAGE3_OUTPUT_DIR
    / "stage3_accepted_nonexact_one_to_one_links_v1.parquet"
)

T0_POST_ADJUDICATION_PATH = (
    STAGE3_OUTPUT_DIR
    / "stage3_t0_post_strong_candidate_adjudication_v1.parquet"
)

REPORT_PATH = (
    STAGE3_CONFIG_DIR
    / "stage3_strong_candidate_adjudication_report_v1.json"
)

MANIFEST_PATH = (
    STAGE3_CONFIG_DIR
    / "stage3_strong_candidate_adjudication_manifest_v1.json"
)


EXPECTED_CANDIDATE_PAIRS_SHA256 = (
    "b2b23a9da290d2cd472a8378ef1ab03c"
    "0388503928351def8ae37cc3ef615c68"
)

EXPECTED_T0_SUMMARY_SHA256 = (
    "d90fea3f98c88c3537f026be0ed05db3"
    "8b75c6a242e75e073ae7fb1379fc4a29"
)

EXPECTED_STAGE3D_MANIFEST_SHA256 = (
    "8d4a060a50aba9958b6918821323de557"
    "41c98576776a89059609b3e1041b90b"
)

EXPECTED_CANDIDATE_PAIRS = 1078
EXPECTED_STRONG_CANDIDATE_PAIRS = 248
EXPECTED_T0_ONLY_RECORDS = 1246
EXPECTED_ONE_TO_ONE_STRONG_PAIRS = 170


# -------------------------------------------------------------------------------------------------
# 2. UTILITY FUNCTIONS
# -------------------------------------------------------------------------------------------------

def sha256_file(path: Path, chunk_size: int = 8 * 1024 * 1024) -> str:
    """Calculate SHA-256 without loading the whole file into memory."""
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def value_counts_dict(series: pd.Series) -> dict:
    """Create a JSON-safe value-count dictionary."""
    counts = series.value_counts(dropna=False)

    result = {}

    for key, value in counts.items():
        safe_key = "<MISSING>" if pd.isna(key) else str(key)
        result[safe_key] = int(value)

    return result


def adjudicate_candidate(row) -> tuple[str, bool, str]:
    """
    Apply the frozen Stage 3E adjudication rules.

    Returns:
        decision, accepted flag, decision basis
    """

    is_strong = bool(row["is_strong_candidate"])

    if not is_strong:
        return (
            "OUTSIDE_STAGE3E_STRONG_CANDIDATE_SCOPE",
            False,
            "Tier 3–5 candidates are not adjudicated or accepted in Stage 3E.",
        )

    tier = str(row["candidate_tier"])
    topology = str(row["candidate_topology"])

    variation_same = bool(row["variation_id_same"])
    vcv_same = bool(row["vcv_accession_same"])
    gene_exact = str(row["gene_relation"]) == "EXACT_SET_MATCH"
    condition_overlap = bool(row["any_condition_overlap"])

    unique_t0_side = int(row["strong_candidates_for_t0"]) == 1
    unique_t1_side = int(row["strong_candidates_for_t1"]) == 1

    tier1_acceptance_criteria = (
        tier == "TIER_1_VARIANT_GENE_CONDITION_CONTINUITY"
        and topology == "POSSIBLE_ONE_TO_ONE_ACCESSION_REPLACEMENT"
        and variation_same
        and vcv_same
        and gene_exact
        and condition_overlap
        and unique_t0_side
        and unique_t1_side
    )

    if tier1_acceptance_criteria:
        return (
            "ACCEPTED_NONEXACT_ONE_TO_ONE_CONTINUITY",
            True,
            (
                "Accepted by frozen rule: exact VariationID, exact VCV accession, "
                "exact target gene, condition overlap, and unique strong candidate "
                "on both the T0 and T1 sides."
            ),
        )

    if tier.startswith("TIER_2"):
        return (
            "DEFERRED_TIER_2_PARTIAL_IDENTIFIER_CONTINUITY",
            False,
            (
                "Tier 2 evidence contains only partial stable-identifier continuity; "
                "automatic acceptance is prohibited."
            ),
        )

    if topology == "POSSIBLE_ONE_TO_MANY_SPLIT":
        return (
            "DEFERRED_POSSIBLE_ONE_TO_MANY_SPLIT",
            False,
            (
                "One T0 record has multiple strong T1 candidates; possible split "
                "requires separate source-history adjudication."
            ),
        )

    if topology == "POSSIBLE_MANY_TO_ONE_MERGE":
        return (
            "DEFERRED_POSSIBLE_MANY_TO_ONE_MERGE",
            False,
            (
                "Multiple T0 records share one strong T1 candidate; possible merge "
                "requires separate source-history adjudication."
            ),
        )

    if topology == "POSSIBLE_COMPLEX_MANY_TO_MANY_CHANGE":
        return (
            "DEFERRED_POSSIBLE_COMPLEX_MANY_TO_MANY_CHANGE",
            False,
            (
                "The strong-candidate graph is many-to-many and cannot be resolved "
                "through an automatic one-to-one rule."
            ),
        )

    return (
        "DEFERRED_STRONG_CANDIDATE_REQUIRES_REVIEW",
        False,
        (
            "The candidate did not satisfy every frozen automatic-acceptance "
            "criterion."
        ),
    )


def t0_post_adjudication_status(row) -> str:
    """Create one post-Stage-3E status for each T0-only record."""

    accepted_count = int(row["accepted_stage3e_link_count"])
    deferred_strong_count = int(row["deferred_strong_candidate_count"])
    identifier_candidate_count = int(
        row["identifier_based_candidate_count"]
    )

    if accepted_count == 1:
        return "ACCEPTED_NONEXACT_ONE_TO_ONE_CONTINUITY"

    if accepted_count > 1:
        return "ERROR_MULTIPLE_ACCEPTED_LINKS"

    if deferred_strong_count > 0:
        return "UNRESOLVED_COMPLEX_STRONG_CANDIDATE_RELATIONSHIP"

    if identifier_candidate_count > 0:
        return "UNRESOLVED_VARIANT_CONTINUITY_WITHOUT_CONDITION_OVERLAP"

    return "UNRESOLVED_NO_VARIATIONID_OR_VCV_CANDIDATE"


# -------------------------------------------------------------------------------------------------
# 3. VERIFY STAGE 3D INPUT ARTIFACTS
# -------------------------------------------------------------------------------------------------

required_files = [
    CANDIDATE_PAIRS_PATH,
    T0_STAGE3D_SUMMARY_PATH,
    STAGE3D_MANIFEST_PATH,
]

for required_file in required_files:
    if not required_file.exists():
        raise FileNotFoundError(
            f"Required Stage 3D artifact does not exist: {required_file}"
        )

observed_candidate_pairs_sha256 = sha256_file(
    CANDIDATE_PAIRS_PATH
)

observed_t0_summary_sha256 = sha256_file(
    T0_STAGE3D_SUMMARY_PATH
)

observed_stage3d_manifest_sha256 = sha256_file(
    STAGE3D_MANIFEST_PATH
)

if observed_candidate_pairs_sha256 != EXPECTED_CANDIDATE_PAIRS_SHA256:
    raise RuntimeError(
        "Stage 3D candidate-pair checksum mismatch.\n"
        f"Expected: {EXPECTED_CANDIDATE_PAIRS_SHA256}\n"
        f"Observed: {observed_candidate_pairs_sha256}"
    )

if observed_t0_summary_sha256 != EXPECTED_T0_SUMMARY_SHA256:
    raise RuntimeError(
        "Stage 3D T0-summary checksum mismatch.\n"
        f"Expected: {EXPECTED_T0_SUMMARY_SHA256}\n"
        f"Observed: {observed_t0_summary_sha256}"
    )

if observed_stage3d_manifest_sha256 != EXPECTED_STAGE3D_MANIFEST_SHA256:
    raise RuntimeError(
        "Stage 3D manifest checksum mismatch.\n"
        f"Expected: {EXPECTED_STAGE3D_MANIFEST_SHA256}\n"
        f"Observed: {observed_stage3d_manifest_sha256}"
    )

with STAGE3D_MANIFEST_PATH.open("r", encoding="utf-8") as handle:
    stage3d_manifest = json.load(handle)

if (
    stage3d_manifest.get("audit_status")
    != "NONEXACT_CANDIDATE_GENERATION_COMPLETE"
):
    raise RuntimeError(
        "The Stage 3D manifest does not authorize candidate adjudication."
    )

print(f"Stage 3D candidate SHA-256: {observed_candidate_pairs_sha256}")
print(f"Stage 3D T0 summary SHA-256:{observed_t0_summary_sha256}")
print(f"Stage 3D manifest SHA-256:  {observed_stage3d_manifest_sha256}")


# -------------------------------------------------------------------------------------------------
# 4. FREEZE THE ADJUDICATION RULES BEFORE APPLYING THEM
# -------------------------------------------------------------------------------------------------

rules = {
    "rules_name": "Stage 3E Non-Exact Candidate Adjudication Rules",
    "rules_version": "1.0.0",
    "scientific_unit": "RCV-level variant-condition aggregate",
    "scope": [
        "Tier 1 candidates",
        "Tier 2 candidates",
        "strong-candidate topology",
    ],
    "automatic_acceptance_rule": {
        "candidate_tier": "TIER_1_VARIANT_GENE_CONDITION_CONTINUITY",
        "variation_id_same": True,
        "vcv_accession_same": True,
        "gene_relation": "EXACT_SET_MATCH",
        "any_condition_overlap": True,
        "strong_candidates_for_t0": 1,
        "strong_candidates_for_t1": 1,
        "candidate_topology": "POSSIBLE_ONE_TO_ONE_ACCESSION_REPLACEMENT",
        "decision": "ACCEPTED_NONEXACT_ONE_TO_ONE_CONTINUITY",
    },
    "automatic_rejection_or_deferral_rules": {
        "tier_2": (
            "Deferred because only partial stable-identifier continuity is present."
        ),
        "one_to_many": (
            "Deferred as a possible split; no automatic acceptance."
        ),
        "many_to_one": (
            "Deferred as a possible merge; no automatic acceptance."
        ),
        "many_to_many": (
            "Deferred as a complex relationship; no automatic acceptance."
        ),
        "tier_3_to_5": (
            "Outside Stage 3E acceptance scope."
        ),
    },
    "prohibited_operations": [
        "No classification comparison",
        "No temporal outcome construction",
        "No stable label for unresolved records",
        "No confirmation of official replacement, merge, split, or withdrawal",
        "No GES fitting or tuning",
    ],
}

rules_json = json.dumps(
    rules,
    indent=2,
    sort_keys=True,
    ensure_ascii=False,
)

if RULES_PATH.exists():
    with RULES_PATH.open("r", encoding="utf-8") as handle:
        existing_rules = json.load(handle)

    if existing_rules != rules:
        raise RuntimeError(
            "An existing Stage 3E rule file differs from the current frozen rules. "
            "Do not overwrite it silently."
        )
else:
    with RULES_PATH.open("w", encoding="utf-8") as handle:
        handle.write(rules_json)

rules_sha256 = sha256_file(RULES_PATH)

print()
print(f"Frozen adjudication rules: {RULES_PATH}")
print(f"Rules SHA-256:             {rules_sha256}")


# -------------------------------------------------------------------------------------------------
# 5. LOAD AND VALIDATE STAGE 3D DATA
# -------------------------------------------------------------------------------------------------

candidate_pairs = pd.read_parquet(CANDIDATE_PAIRS_PATH)
t0_summary = pd.read_parquet(T0_STAGE3D_SUMMARY_PATH)

if len(candidate_pairs) != EXPECTED_CANDIDATE_PAIRS:
    raise AssertionError(
        f"Expected {EXPECTED_CANDIDATE_PAIRS:,} candidate pairs, "
        f"observed {len(candidate_pairs):,}."
    )

if len(t0_summary) != EXPECTED_T0_ONLY_RECORDS:
    raise AssertionError(
        f"Expected {EXPECTED_T0_ONLY_RECORDS:,} T0 records, "
        f"observed {len(t0_summary):,}."
    )

strong_pair_count = int(
    candidate_pairs["is_strong_candidate"].fillna(False).sum()
)

if strong_pair_count != EXPECTED_STRONG_CANDIDATE_PAIRS:
    raise AssertionError(
        f"Expected {EXPECTED_STRONG_CANDIDATE_PAIRS:,} strong candidate pairs, "
        f"observed {strong_pair_count:,}."
    )

observed_one_to_one_strong = int(
    (
        candidate_pairs["is_strong_candidate"].fillna(False)
        & candidate_pairs["candidate_topology"].eq(
            "POSSIBLE_ONE_TO_ONE_ACCESSION_REPLACEMENT"
        )
    ).sum()
)

if observed_one_to_one_strong != EXPECTED_ONE_TO_ONE_STRONG_PAIRS:
    raise AssertionError(
        f"Expected {EXPECTED_ONE_TO_ONE_STRONG_PAIRS:,} strong one-to-one pairs, "
        f"observed {observed_one_to_one_strong:,}."
    )

print()
print(f"Candidate pairs loaded:       {len(candidate_pairs):,}")
print(f"Strong candidate pairs:       {strong_pair_count:,}")
print(f"Strong one-to-one candidates: {observed_one_to_one_strong:,}")


# -------------------------------------------------------------------------------------------------
# 6. APPLY THE FROZEN ADJUDICATION RULES
# -------------------------------------------------------------------------------------------------

adjudication_results = [
    adjudicate_candidate(row)
    for _, row in candidate_pairs.iterrows()
]

candidate_pairs["stage3e_decision"] = [
    result[0]
    for result in adjudication_results
]

candidate_pairs["stage3e_accepted_nonexact_link"] = [
    result[1]
    for result in adjudication_results
]

candidate_pairs["stage3e_decision_basis"] = [
    result[2]
    for result in adjudication_results
]

candidate_pairs["stage3e_rules_version"] = rules["rules_version"]
candidate_pairs["future_instability_outcome_created"] = False
candidate_pairs["unmatched_labeled_stable"] = False


# -------------------------------------------------------------------------------------------------
# 7. VALIDATE ACCEPTED NON-EXACT LINKS
# -------------------------------------------------------------------------------------------------

accepted_links = candidate_pairs.loc[
    candidate_pairs["stage3e_accepted_nonexact_link"]
].copy()

if len(accepted_links) != EXPECTED_ONE_TO_ONE_STRONG_PAIRS:
    raise AssertionError(
        f"Expected {EXPECTED_ONE_TO_ONE_STRONG_PAIRS:,} accepted one-to-one links, "
        f"observed {len(accepted_links):,}."
    )

if accepted_links["t0_crosswalk_id"].duplicated().any():
    raise AssertionError(
        "More than one accepted link was assigned to at least one T0 record."
    )

if accepted_links["t1_crosswalk_id"].duplicated().any():
    raise AssertionError(
        "More than one accepted link was assigned to at least one T1 record."
    )

required_acceptance_checks = {
    "all_tier_1": accepted_links["candidate_tier"].eq(
        "TIER_1_VARIANT_GENE_CONDITION_CONTINUITY"
    ).all(),
    "all_variation_ids_same": accepted_links["variation_id_same"].eq(True).all(),
    "all_vcv_accessions_same": accepted_links["vcv_accession_same"].eq(True).all(),
    "all_genes_exact": accepted_links["gene_relation"].eq(
        "EXACT_SET_MATCH"
    ).all(),
    "all_have_condition_overlap": accepted_links["any_condition_overlap"].eq(
        True
    ).all(),
    "all_unique_from_t0": accepted_links["strong_candidates_for_t0"].eq(1).all(),
    "all_unique_from_t1": accepted_links["strong_candidates_for_t1"].eq(1).all(),
    "all_one_to_one_topology": accepted_links["candidate_topology"].eq(
        "POSSIBLE_ONE_TO_ONE_ACCESSION_REPLACEMENT"
    ).all(),
}

failed_acceptance_checks = [
    check_name
    for check_name, passed in required_acceptance_checks.items()
    if not passed
]

if failed_acceptance_checks:
    raise AssertionError(
        "Accepted-link validation failed: "
        + ", ".join(failed_acceptance_checks)
    )

accepted_link_columns = [
    "candidate_pair_id",
    "t0_crosswalk_id",
    "t1_crosswalk_id",
    "t0_rcv_accession",
    "t1_rcv_accession",
    "t0_rcv_version",
    "t1_rcv_version",
    "t0_variation_id",
    "t1_variation_id",
    "t0_vcv_accession",
    "t1_vcv_accession",
    "t0_vcv_version",
    "t1_vcv_version",
    "t0_genes_normalized_json",
    "t1_genes_normalized_json",
    "condition_evidence_category",
    "condition_id_overlap_count",
    "condition_id_overlap_json",
    "condition_name_overlap_count",
    "condition_name_overlap_json",
    "t1_classification_axis",
    "candidate_tier",
    "candidate_topology",
    "stage3e_decision",
    "stage3e_decision_basis",
    "stage3e_rules_version",
    "stage3e_accepted_nonexact_link",
]

accepted_links = (
    accepted_links[accepted_link_columns]
    .sort_values(
        by=["t0_rcv_accession", "t1_rcv_accession"],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


# -------------------------------------------------------------------------------------------------
# 8. CREATE T0-LEVEL POST-ADJUDICATION STATUS
# -------------------------------------------------------------------------------------------------

accepted_by_t0 = (
    candidate_pairs.loc[
        candidate_pairs["stage3e_accepted_nonexact_link"]
    ]
    .groupby("t0_crosswalk_id")
    .agg(
        accepted_stage3e_link_count=(
            "candidate_pair_id",
            "size",
        ),
        accepted_stage3e_candidate_pair_id=(
            "candidate_pair_id",
            "first",
        ),
        accepted_stage3e_t1_rcv_accession=(
            "t1_rcv_accession",
            "first",
        ),
        accepted_stage3e_t1_crosswalk_id=(
            "t1_crosswalk_id",
            "first",
        ),
    )
    .reset_index()
)

deferred_strong_by_t0 = (
    candidate_pairs.loc[
        candidate_pairs["is_strong_candidate"].fillna(False)
        & ~candidate_pairs["stage3e_accepted_nonexact_link"]
    ]
    .groupby("t0_crosswalk_id")
    .size()
    .rename("deferred_strong_candidate_count")
    .reset_index()
)

t0_post = (
    t0_summary
    .merge(
        accepted_by_t0,
        on="t0_crosswalk_id",
        how="left",
        validate="one_to_one",
    )
    .merge(
        deferred_strong_by_t0,
        on="t0_crosswalk_id",
        how="left",
        validate="one_to_one",
    )
)

integer_fill_columns = [
    "accepted_stage3e_link_count",
    "deferred_strong_candidate_count",
]

for column in integer_fill_columns:
    t0_post[column] = (
        t0_post[column]
        .fillna(0)
        .astype(int)
    )

t0_post["stage3e_post_adjudication_status"] = [
    t0_post_adjudication_status(row)
    for _, row in t0_post.iterrows()
]

t0_post["stage3e_accepted_nonexact_link"] = (
    t0_post["accepted_stage3e_link_count"].eq(1)
)

t0_post["stage3e_rules_version"] = rules["rules_version"]
t0_post["future_instability_outcome_created"] = False
t0_post["unmatched_labeled_stable"] = False

if t0_post["stage3e_post_adjudication_status"].eq(
    "ERROR_MULTIPLE_ACCEPTED_LINKS"
).any():
    raise AssertionError(
        "At least one T0 record received multiple accepted links."
    )

accepted_t0_count = int(
    t0_post["stage3e_accepted_nonexact_link"].sum()
)

deferred_complex_t0_count = int(
    t0_post["stage3e_post_adjudication_status"].eq(
        "UNRESOLVED_COMPLEX_STRONG_CANDIDATE_RELATIONSHIP"
    ).sum()
)

tier3_unresolved_t0_count = int(
    t0_post["stage3e_post_adjudication_status"].eq(
        "UNRESOLVED_VARIANT_CONTINUITY_WITHOUT_CONDITION_OVERLAP"
    ).sum()
)

no_identifier_candidate_t0_count = int(
    t0_post["stage3e_post_adjudication_status"].eq(
        "UNRESOLVED_NO_VARIATIONID_OR_VCV_CANDIDATE"
    ).sum()
)

if accepted_t0_count != len(accepted_links):
    raise AssertionError(
        "Accepted pair count does not equal accepted T0-record count."
    )

if (
    accepted_t0_count
    + deferred_complex_t0_count
    + tier3_unresolved_t0_count
    + no_identifier_candidate_t0_count
    != EXPECTED_T0_ONLY_RECORDS
):
    raise AssertionError(
        "Post-adjudication T0 accounting failed."
    )

t0_post = (
    t0_post.sort_values(
        by="t0_rcv_accession",
        kind="mergesort",
    )
    .reset_index(drop=True)
)


# -------------------------------------------------------------------------------------------------
# 9. WRITE AND READ BACK OUTPUT ARTIFACTS
# -------------------------------------------------------------------------------------------------

candidate_pairs = (
    candidate_pairs.sort_values(
        by=[
            "t0_rcv_accession",
            "candidate_tier_rank",
            "t1_rcv_accession",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)

candidate_pairs.to_parquet(
    ADJUDICATION_PATH,
    engine="pyarrow",
    compression="zstd",
    index=False,
)

accepted_links.to_parquet(
    ACCEPTED_LINKS_PATH,
    engine="pyarrow",
    compression="zstd",
    index=False,
)

t0_post.to_parquet(
    T0_POST_ADJUDICATION_PATH,
    engine="pyarrow",
    compression="zstd",
    index=False,
)

adjudication_sha256 = sha256_file(ADJUDICATION_PATH)
accepted_links_sha256 = sha256_file(ACCEPTED_LINKS_PATH)
t0_post_sha256 = sha256_file(T0_POST_ADJUDICATION_PATH)

adjudication_metadata = pq.ParquetFile(
    ADJUDICATION_PATH
).metadata

accepted_links_metadata = pq.ParquetFile(
    ACCEPTED_LINKS_PATH
).metadata

t0_post_metadata = pq.ParquetFile(
    T0_POST_ADJUDICATION_PATH
).metadata

adjudication_readback = pd.read_parquet(ADJUDICATION_PATH)
accepted_links_readback = pd.read_parquet(ACCEPTED_LINKS_PATH)
t0_post_readback = pd.read_parquet(T0_POST_ADJUDICATION_PATH)

if len(adjudication_readback) != len(candidate_pairs):
    raise AssertionError(
        "Adjudication readback row-count mismatch."
    )

if len(accepted_links_readback) != len(accepted_links):
    raise AssertionError(
        "Accepted-links readback row-count mismatch."
    )

if len(t0_post_readback) != len(t0_post):
    raise AssertionError(
        "T0 post-adjudication readback row-count mismatch."
    )

if accepted_links_readback["t0_crosswalk_id"].duplicated().any():
    raise AssertionError(
        "Accepted-links readback contains duplicate T0 identifiers."
    )

if accepted_links_readback["t1_crosswalk_id"].duplicated().any():
    raise AssertionError(
        "Accepted-links readback contains duplicate T1 identifiers."
    )


# -------------------------------------------------------------------------------------------------
# 10. CREATE STAGE 3E REPORT
# -------------------------------------------------------------------------------------------------

deferred_strong_pairs = int(
    (
        candidate_pairs["is_strong_candidate"].fillna(False)
        & ~candidate_pairs["stage3e_accepted_nonexact_link"]
    ).sum()
)

unresolved_t0_count = (
    EXPECTED_T0_ONLY_RECORDS - accepted_t0_count
)

report = {
    "report_name": "Stage 3E Strong-Candidate Adjudication Report",
    "report_version": "1.0.0",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "study_stage": "Stage 3E",
    "scientific_unit": "RCV-level variant-condition aggregate",

    "frozen_rules": {
        "path": str(RULES_PATH),
        "sha256": rules_sha256,
        "version": rules["rules_version"],
    },

    "input_counts": {
        "candidate_pairs": int(len(candidate_pairs)),
        "strong_candidate_pairs": int(strong_pair_count),
        "t0_only_records": int(len(t0_post)),
    },

    "pair_decisions": {
        "decision_counts": value_counts_dict(
            candidate_pairs["stage3e_decision"]
        ),
        "accepted_nonexact_one_to_one_pairs": int(
            len(accepted_links)
        ),
        "deferred_strong_candidate_pairs": int(
            deferred_strong_pairs
        ),
    },

    "t0_record_decisions": {
        "status_counts": value_counts_dict(
            t0_post["stage3e_post_adjudication_status"]
        ),
        "accepted_t0_records": int(accepted_t0_count),
        "unresolved_t0_records": int(unresolved_t0_count),
        "complex_strong_candidate_t0_records": int(
            deferred_complex_t0_count
        ),
        "tier3_condition_change_candidate_t0_records": int(
            tier3_unresolved_t0_count
        ),
        "no_identifier_candidate_t0_records": int(
            no_identifier_candidate_t0_count
        ),
    },

    "validation": {
        "accepted_t0_duplicate_count": int(
            accepted_links["t0_crosswalk_id"].duplicated().sum()
        ),
        "accepted_t1_duplicate_count": int(
            accepted_links["t1_crosswalk_id"].duplicated().sum()
        ),
        "accepted_pair_count": int(len(accepted_links)),
        "all_acceptance_checks_passed": True,
        "post_adjudication_t0_accounting_passed": True,
        "readback_passed": True,
        "critical_failures": 0,
    },

    "scientific_boundaries": {
        "one_to_one_nonexact_continuity_links_accepted": True,
        "accepted_links_called_official_rcv_replacements": False,
        "tier2_links_accepted": False,
        "splits_accepted": False,
        "merges_accepted": False,
        "complex_links_accepted": False,
        "tier3_links_accepted": False,
        "classification_changes_calculated": False,
        "future_instability_outcomes_created": False,
        "unresolved_records_labeled_stable": False,
        "ges_model_fitted_or_tuned": False,
    },

    "next_authorized_step": (
        "Audit the deferred complex strong-candidate relationships and the "
        "Tier 3 variant-continuity/condition-change candidates. Preserve all "
        "unresolved records as unmatched or censored until the final crosswalk "
        "and linkage-exclusion policy are frozen."
    ),
}

with REPORT_PATH.open("w", encoding="utf-8") as handle:
    json.dump(
        report,
        handle,
        indent=2,
        sort_keys=True,
        ensure_ascii=False,
    )

report_sha256 = sha256_file(REPORT_PATH)


# -------------------------------------------------------------------------------------------------
# 11. CREATE STAGE 3E MANIFEST
# -------------------------------------------------------------------------------------------------

manifest = {
    "manifest_name": "Stage 3E Strong-Candidate Adjudication Manifest",
    "manifest_version": "1.0.0",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "study_stage": "Stage 3E",
    "adjudication_status": "CONSERVATIVE_STRONG_CANDIDATE_ADJUDICATION_COMPLETE",

    "inputs": {
        "candidate_pairs": {
            "path": str(CANDIDATE_PAIRS_PATH),
            "sha256": observed_candidate_pairs_sha256,
        },
        "t0_stage3d_summary": {
            "path": str(T0_STAGE3D_SUMMARY_PATH),
            "sha256": observed_t0_summary_sha256,
        },
        "stage3d_manifest": {
            "path": str(STAGE3D_MANIFEST_PATH),
            "sha256": observed_stage3d_manifest_sha256,
        },
        "frozen_rules": {
            "path": str(RULES_PATH),
            "sha256": rules_sha256,
        },
    },

    "outputs": {
        "candidate_adjudication": {
            "path": str(ADJUDICATION_PATH),
            "sha256": adjudication_sha256,
            "rows": int(adjudication_metadata.num_rows),
            "columns": int(adjudication_metadata.num_columns),
            "row_groups": int(adjudication_metadata.num_row_groups),
        },
        "accepted_nonexact_links": {
            "path": str(ACCEPTED_LINKS_PATH),
            "sha256": accepted_links_sha256,
            "rows": int(accepted_links_metadata.num_rows),
            "columns": int(accepted_links_metadata.num_columns),
            "row_groups": int(accepted_links_metadata.num_row_groups),
        },
        "t0_post_adjudication": {
            "path": str(T0_POST_ADJUDICATION_PATH),
            "sha256": t0_post_sha256,
            "rows": int(t0_post_metadata.num_rows),
            "columns": int(t0_post_metadata.num_columns),
            "row_groups": int(t0_post_metadata.num_row_groups),
        },
        "report": {
            "path": str(REPORT_PATH),
            "sha256": report_sha256,
        },
    },

    "accepted_nonexact_pair_count": int(len(accepted_links)),
    "unresolved_t0_record_count": int(unresolved_t0_count),
    "validation_decision": "PASS",
    "scientific_boundaries": report["scientific_boundaries"],
    "next_authorized_step": report["next_authorized_step"],
}

with MANIFEST_PATH.open("w", encoding="utf-8") as handle:
    json.dump(
        manifest,
        handle,
        indent=2,
        sort_keys=True,
        ensure_ascii=False,
    )

manifest_sha256 = sha256_file(MANIFEST_PATH)


# -------------------------------------------------------------------------------------------------
# 12. PRINT RESULTS
# -------------------------------------------------------------------------------------------------

print()
print("=" * 124)
print("STAGE 3E ADJUDICATION SUMMARY")
print("=" * 124)

print(f"Candidate pairs evaluated:                     {len(candidate_pairs):,}")
print(f"Strong Tier 1/Tier 2 pairs evaluated:          {strong_pair_count:,}")
print(f"Accepted one-to-one non-exact links:           {len(accepted_links):,}")
print(f"Deferred strong candidate pairs:               {deferred_strong_pairs:,}")

print()
print(f"T0 records accepted through non-exact linkage: {accepted_t0_count:,}")
print(f"T0 records still unresolved:                   {unresolved_t0_count:,}")
print(f"  Complex strong-candidate relationships:      {deferred_complex_t0_count:,}")
print(f"  Tier 3 condition-change candidates:          {tier3_unresolved_t0_count:,}")
print(f"  No VariationID/VCV candidate:                {no_identifier_candidate_t0_count:,}")

print()
print("Pair-level adjudication decisions:")
print(
    candidate_pairs["stage3e_decision"]
    .value_counts(dropna=False)
    .to_string()
)

print()
print("T0 post-adjudication statuses:")
print(
    t0_post["stage3e_post_adjudication_status"]
    .value_counts(dropna=False)
    .to_string()
)

print()
print(f"Rules:                 {RULES_PATH}")
print(f"Rules SHA-256:         {rules_sha256}")
print(f"Adjudication:          {ADJUDICATION_PATH}")
print(f"Adjudication SHA-256:  {adjudication_sha256}")
print(f"Accepted links:        {ACCEPTED_LINKS_PATH}")
print(f"Accepted SHA-256:      {accepted_links_sha256}")
print(f"T0 post-adjudication:  {T0_POST_ADJUDICATION_PATH}")
print(f"T0 status SHA-256:     {t0_post_sha256}")
print(f"Report:                {REPORT_PATH}")
print(f"Report SHA-256:        {report_sha256}")
print(f"Manifest:              {MANIFEST_PATH}")
print(f"Manifest SHA-256:      {manifest_sha256}")

print()
print("PASS — Stage 3E conservative strong-candidate adjudication completed.")
print(
    "IMPORTANT — Only unique Tier 1 one-to-one continuity links were accepted."
)
print(
    "IMPORTANT — Possible splits, merges, complex mappings, Tier 3 candidates, "
    "and no-candidate records remain unresolved and were not labeled stable."
)
print(
    "IMPORTANT — No classification-change or future-instability outcome was created."
)
print(
    "AUTHORIZED NEXT ACTION — Stage 3F targeted audit of deferred complex "
    "strong candidates and Tier 3 condition-change candidates."
)
print("=" * 124)

STAGE 3E — CONSERVATIVE STRONG-CANDIDATE ADJUDICATION
Stage 3D candidate SHA-256: b2b23a9da290d2cd472a8378ef1ab03c0388503928351def8ae37cc3ef615c68
Stage 3D T0 summary SHA-256:d90fea3f98c88c3537f026be0ed05db38b75c6a242e75e073ae7fb1379fc4a29
Stage 3D manifest SHA-256:  8d4a060a50aba9958b6918821323de55741c98576776a89059609b3e1041b90b

Frozen adjudication rules: /content/drive/MyDrive/GES_RAG_Temporal_Study/configs/stage3_crosswalk/stage3_nonexact_candidate_adjudication_rules_v1.json
Rules SHA-256:             438437e5ddc1e9eab28dbda6e2d707731e0e6cc41ac89c593a9939d36e50c7c2

Candidate pairs loaded:       1,078
Strong candidate pairs:       248
Strong one-to-one candidates: 170

STAGE 3E ADJUDICATION SUMMARY
Candidate pairs evaluated:                     1,078
Strong Tier 1/Tier 2 pairs evaluated:          248
Accepted one-to-one non-exact links:           170
Deferred strong candidate pairs:               78

T0 records accepted through non-exact linkage: 170
T0 records still unresolved:  

In [17]:
# =================================================================================================
# STAGE 3F — TARGETED AUDIT OF DEFERRED STRONG AND TIER 3 CANDIDATE RELATIONSHIPS
# =================================================================================================
# Purpose:
#   Audit, without automatically accepting, the following unresolved candidate relationships:
#
#   1. Deferred strong candidate pairs:
#        - possible one-to-many splits,
#        - possible many-to-one merges,
#        - possible complex many-to-many relationships.
#
#   2. Tier 3 candidate pairs:
#        - exact VariationID continuity,
#        - exact VCV accession continuity,
#        - exact target-gene continuity,
#        - but no exact condition-ID or normalized condition-name overlap.
#
# This cell:
#   - creates a bipartite graph of targeted T0–T1 candidate relationships;
#   - assigns deterministic graph-component identifiers;
#   - characterizes one-to-one, one-to-many, many-to-one, and many-to-many components;
#   - audits condition metadata missingness;
#   - calculates descriptive condition-name lexical similarity;
#   - identifies condition-namespace continuity;
#   - identifies targeted candidates touching a Stage 3E accepted link;
#   - creates pair-, component-, and T0-level review artifacts.
#
# IMPORTANT:
#   Lexical similarity is an audit-prioritization signal only.
#   It is not evidence of condition equivalence and cannot accept a link.
#
# This cell DOES NOT:
#   - accept additional non-exact links,
#   - revoke or silently alter Stage 3E accepted links,
#   - confirm official RCV replacements, merges, splits, or withdrawals,
#   - compare aggregate clinical classifications,
#   - construct temporal-instability outcomes,
#   - label unresolved records as stable,
#   - fit or tune GES.
# =================================================================================================

from pathlib import Path
from datetime import datetime, timezone
from collections import Counter, defaultdict
import hashlib
import json
import re

import pandas as pd
import pyarrow.parquet as pq


print("=" * 126)
print("STAGE 3F — DEFERRED-COMPLEX AND TIER 3 CONDITION-CHANGE AUDIT")
print("=" * 126)


# -------------------------------------------------------------------------------------------------
# 1. PATHS AND EXPECTED STAGE 3E CHECKSUMS
# -------------------------------------------------------------------------------------------------

STUDY_ROOT = Path("/content/drive/MyDrive/GES_RAG_Temporal_Study")

STAGE3_CONFIG_DIR = STUDY_ROOT / "configs" / "stage3_crosswalk"
STAGE3_OUTPUT_DIR = STUDY_ROOT / "data_processed" / "stage3_crosswalk"

STAGE3E_ADJUDICATION_PATH = (
    STAGE3_OUTPUT_DIR
    / "stage3_nonexact_candidate_adjudication_v1.parquet"
)

STAGE3E_T0_STATUS_PATH = (
    STAGE3_OUTPUT_DIR
    / "stage3_t0_post_strong_candidate_adjudication_v1.parquet"
)

STAGE3E_MANIFEST_PATH = (
    STAGE3_CONFIG_DIR
    / "stage3_strong_candidate_adjudication_manifest_v1.json"
)

PAIR_AUDIT_PATH = (
    STAGE3_OUTPUT_DIR
    / "stage3_targeted_complex_and_tier3_pair_audit_v1.parquet"
)

COMPONENT_AUDIT_PATH = (
    STAGE3_OUTPUT_DIR
    / "stage3_targeted_candidate_graph_components_v1.parquet"
)

T0_REVIEW_SUMMARY_PATH = (
    STAGE3_OUTPUT_DIR
    / "stage3_targeted_candidate_t0_review_summary_v1.parquet"
)

REPORT_PATH = (
    STAGE3_CONFIG_DIR
    / "stage3_targeted_complex_and_tier3_audit_report_v1.json"
)

MANIFEST_PATH = (
    STAGE3_CONFIG_DIR
    / "stage3_targeted_complex_and_tier3_audit_manifest_v1.json"
)


EXPECTED_ADJUDICATION_SHA256 = (
    "f647a8fe66db8a695da2136f3ec59ad0"
    "03425f11a2767a3198eefa612dbcf4ef"
)

EXPECTED_T0_STATUS_SHA256 = (
    "c1e40c92389e2dc200cd1508c49fb7a"
    "a12c3a9665fc2e6aab56ed2127accb65e"
)

EXPECTED_STAGE3E_MANIFEST_SHA256 = (
    "b5ea53ba3fe6cafdb2abaa25ac582d683"
    "cfb35dd04d88dcfdaf03802edec4304"
)

EXPECTED_ALL_CANDIDATE_PAIRS = 1078
EXPECTED_ACCEPTED_STAGE3E_PAIRS = 170
EXPECTED_DEFERRED_STRONG_PAIRS = 78
EXPECTED_TIER3_PAIRS = 830
EXPECTED_TARGETED_PAIRS = 908
EXPECTED_T0_STATUS_ROWS = 1246


DEFERRED_STRONG_DECISIONS = {
    "DEFERRED_POSSIBLE_ONE_TO_MANY_SPLIT",
    "DEFERRED_POSSIBLE_MANY_TO_ONE_MERGE",
    "DEFERRED_POSSIBLE_COMPLEX_MANY_TO_MANY_CHANGE",
    "DEFERRED_STRONG_CANDIDATE_REQUIRES_REVIEW",
}

TIER3_NAME = (
    "TIER_3_VARIANT_CONTINUITY_CONDITION_CHANGE_CANDIDATE"
)


# -------------------------------------------------------------------------------------------------
# 2. GENERAL UTILITY FUNCTIONS
# -------------------------------------------------------------------------------------------------

def sha256_file(path: Path, chunk_size: int = 8 * 1024 * 1024) -> str:
    """Calculate SHA-256 without loading the entire file into memory."""
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def value_counts_dict(series: pd.Series) -> dict:
    """Convert pandas value counts into a JSON-safe dictionary."""
    counts = series.value_counts(dropna=False)

    result = {}

    for key, value in counts.items():
        safe_key = "<MISSING>" if pd.isna(key) else str(key)
        result[safe_key] = int(value)

    return result


def safe_json_list(value, field_name: str) -> list:
    """Parse a JSON-list field safely and enforce its top-level type."""
    if value is None:
        return []

    if isinstance(value, list):
        parsed = value

    elif isinstance(value, tuple):
        parsed = list(value)

    else:
        try:
            if pd.isna(value):
                return []
        except (TypeError, ValueError):
            pass

        text = str(value).strip()

        if text == "":
            return []

        try:
            parsed = json.loads(text)
        except json.JSONDecodeError as exc:
            raise ValueError(
                f"Malformed JSON in {field_name}: {text[:200]}"
            ) from exc

    if not isinstance(parsed, list):
        raise TypeError(
            f"{field_name} must contain a JSON list, "
            f"not {type(parsed).__name__}."
        )

    cleaned = []

    for item in parsed:
        if item is None:
            continue

        text = str(item).strip()

        if text:
            cleaned.append(text)

    return cleaned


def json_compact(values) -> str:
    """Serialize a collection deterministically as compact JSON."""
    return json.dumps(
        list(values),
        ensure_ascii=False,
        separators=(",", ":"),
    )


def json_count_dict(values) -> str:
    """Serialize value counts deterministically."""
    counts = Counter(values)

    return json.dumps(
        {
            str(key): int(value)
            for key, value in sorted(counts.items())
        },
        ensure_ascii=False,
        separators=(",", ":"),
    )


def set_relation(left_values, right_values) -> str:
    """Describe the relationship between two sets."""
    left = set(left_values)
    right = set(right_values)

    if not left and not right:
        return "BOTH_EMPTY"

    if not left:
        return "T0_EMPTY"

    if not right:
        return "T1_EMPTY"

    if left == right:
        return "EXACT_SET_MATCH"

    if left.intersection(right):
        return "PARTIAL_SET_OVERLAP"

    return "DISJOINT_SETS"


# -------------------------------------------------------------------------------------------------
# 3. CONDITION-METADATA AUDIT FUNCTIONS
# -------------------------------------------------------------------------------------------------

def condition_id_namespace(condition_id: str):
    """
    Extract a descriptive condition-identifier namespace.

    Examples:
        MEDGEN:C123      -> MEDGEN
        OMIM:123456      -> OMIM
        MONDO:0000001    -> MONDO
    """
    text = str(condition_id).strip().upper()

    if ":" in text:
        return text.split(":", 1)[0]

    return None


def condition_names_to_tokens(names) -> set:
    """
    Convert normalized condition names into descriptive lexical tokens.

    No synonym inference, ontology mapping, stemming, or clinical equivalence
    inference is performed.
    """
    tokens = set()

    for name in names:
        for token in re.findall(r"[a-z0-9]+", str(name).casefold()):
            if len(token) >= 2:
                tokens.add(token)

    return tokens


def relaxed_condition_name_keys(names) -> set:
    """
    Remove punctuation and spaces from normalized names.

    This may detect formatting-only differences, but the result remains
    an audit signal and cannot establish condition equivalence.
    """
    keys = set()

    for name in names:
        key = re.sub(
            r"[^a-z0-9]+",
            "",
            str(name).casefold(),
        )

        if key:
            keys.add(key)

    return keys


def jaccard_similarity(left_values, right_values):
    """Calculate Jaccard similarity, preserving empty-set meaning."""
    left = set(left_values)
    right = set(right_values)

    if not left and not right:
        return None

    union = left.union(right)

    if not union:
        return None

    return len(left.intersection(right)) / len(union)


def pair_metadata_audit(row) -> dict:
    """Calculate descriptive condition-metadata audit fields for one pair."""

    t0_ids = safe_json_list(
        row["t0_condition_ids_normalized_json"],
        "t0_condition_ids_normalized_json",
    )

    t1_ids = safe_json_list(
        row["t1_condition_ids_normalized_json"],
        "t1_condition_ids_normalized_json",
    )

    t0_names = safe_json_list(
        row["t0_condition_names_normalized_json"],
        "t0_condition_names_normalized_json",
    )

    t1_names = safe_json_list(
        row["t1_condition_names_normalized_json"],
        "t1_condition_names_normalized_json",
    )

    t0_namespaces = sorted(
        {
            namespace
            for namespace in (
                condition_id_namespace(value)
                for value in t0_ids
            )
            if namespace
        }
    )

    t1_namespaces = sorted(
        {
            namespace
            for namespace in (
                condition_id_namespace(value)
                for value in t1_ids
            )
            if namespace
        }
    )

    namespace_overlap = sorted(
        set(t0_namespaces).intersection(t1_namespaces)
    )

    t0_tokens = condition_names_to_tokens(t0_names)
    t1_tokens = condition_names_to_tokens(t1_names)

    token_overlap = sorted(
        t0_tokens.intersection(t1_tokens)
    )

    token_jaccard = jaccard_similarity(
        t0_tokens,
        t1_tokens,
    )

    t0_relaxed_keys = relaxed_condition_name_keys(t0_names)
    t1_relaxed_keys = relaxed_condition_name_keys(t1_names)

    relaxed_name_overlap = sorted(
        t0_relaxed_keys.intersection(t1_relaxed_keys)
    )

    if not t0_ids and not t1_ids:
        missingness_category = "CONDITION_IDS_EMPTY_BOTH_RELEASES"

    elif not t0_ids:
        missingness_category = "CONDITION_IDS_EMPTY_AT_T0"

    elif not t1_ids:
        missingness_category = "CONDITION_IDS_EMPTY_AT_T1"

    elif not t0_names and not t1_names:
        missingness_category = "CONDITION_NAMES_EMPTY_BOTH_RELEASES"

    elif not t0_names:
        missingness_category = "CONDITION_NAMES_EMPTY_AT_T0"

    elif not t1_names:
        missingness_category = "CONDITION_NAMES_EMPTY_AT_T1"

    else:
        missingness_category = "CONDITION_METADATA_PRESENT_BOTH_RELEASES"

    return {
        "t0_condition_id_count": int(len(t0_ids)),
        "t1_condition_id_count": int(len(t1_ids)),
        "t0_condition_name_count": int(len(t0_names)),
        "t1_condition_name_count": int(len(t1_names)),

        "condition_metadata_missingness_category": (
            missingness_category
        ),

        "t0_condition_namespaces_json": json_compact(
            t0_namespaces
        ),
        "t1_condition_namespaces_json": json_compact(
            t1_namespaces
        ),
        "condition_namespace_relation": set_relation(
            t0_namespaces,
            t1_namespaces,
        ),
        "condition_namespace_overlap_count": int(
            len(namespace_overlap)
        ),
        "condition_namespace_overlap_json": json_compact(
            namespace_overlap
        ),

        "condition_name_token_overlap_count": int(
            len(token_overlap)
        ),
        "condition_name_token_overlap_json": json_compact(
            token_overlap
        ),
        "condition_name_token_jaccard": (
            round(float(token_jaccard), 6)
            if token_jaccard is not None
            else None
        ),

        "relaxed_condition_name_overlap_count": int(
            len(relaxed_name_overlap)
        ),
        "relaxed_condition_name_overlap_json": json_compact(
            relaxed_name_overlap
        ),
        "relaxed_condition_name_match_signal": bool(
            len(relaxed_name_overlap) > 0
        ),
    }


# -------------------------------------------------------------------------------------------------
# 4. UNION-FIND FOR BIPARTITE GRAPH COMPONENTS
# -------------------------------------------------------------------------------------------------

class UnionFind:
    """Minimal deterministic union-find implementation."""

    def __init__(self):
        self.parent = {}
        self.rank = {}

    def add(self, item):
        if item not in self.parent:
            self.parent[item] = item
            self.rank[item] = 0

    def find(self, item):
        self.add(item)

        if self.parent[item] != item:
            self.parent[item] = self.find(self.parent[item])

        return self.parent[item]

    def union(self, left, right):
        left_root = self.find(left)
        right_root = self.find(right)

        if left_root == right_root:
            return

        left_rank = self.rank[left_root]
        right_rank = self.rank[right_root]

        if left_rank < right_rank:
            self.parent[left_root] = right_root

        elif left_rank > right_rank:
            self.parent[right_root] = left_root

        else:
            self.parent[right_root] = left_root
            self.rank[left_root] += 1


def component_topology(t0_count: int, t1_count: int) -> str:
    """Describe component topology without confirming biological history."""

    if t0_count == 1 and t1_count == 1:
        return "ONE_TO_ONE_TARGETED_COMPONENT"

    if t0_count == 1 and t1_count > 1:
        return "ONE_TO_MANY_POSSIBLE_SPLIT_COMPONENT"

    if t0_count > 1 and t1_count == 1:
        return "MANY_TO_ONE_POSSIBLE_MERGE_COMPONENT"

    return "MANY_TO_MANY_COMPLEX_COMPONENT"


def component_scope(scope_values) -> str:
    """Describe whether a graph component contains strong, Tier 3, or mixed edges."""
    scopes = set(scope_values)

    if scopes == {"DEFERRED_STRONG_CANDIDATE"}:
        return "DEFERRED_STRONG_ONLY"

    if scopes == {"TIER3_CONDITION_CHANGE_CANDIDATE"}:
        return "TIER3_ONLY"

    return "MIXED_DEFERRED_STRONG_AND_TIER3"


# -------------------------------------------------------------------------------------------------
# 5. PAIR-LEVEL REVIEW PRIORITY
# -------------------------------------------------------------------------------------------------

def targeted_pair_review_priority(row) -> str:
    """Assign an audit priority without accepting a link."""

    scope = str(row["stage3f_target_scope"])
    topology = str(row["stage3f_component_topology"])

    if scope == "DEFERRED_STRONG_CANDIDATE":

        if topology == "ONE_TO_MANY_POSSIBLE_SPLIT_COMPONENT":
            return "PRIORITY_1_DEFERRED_STRONG_POSSIBLE_SPLIT"

        if topology == "MANY_TO_ONE_POSSIBLE_MERGE_COMPONENT":
            return "PRIORITY_1_DEFERRED_STRONG_POSSIBLE_MERGE"

        if topology == "MANY_TO_MANY_COMPLEX_COMPONENT":
            return "PRIORITY_1_DEFERRED_STRONG_COMPLEX"

        return "PRIORITY_1_DEFERRED_STRONG_ONE_TO_ONE_REVIEW"

    missingness = str(
        row["condition_metadata_missingness_category"]
    )

    relaxed_match = bool(
        row["relaxed_condition_name_match_signal"]
    )

    token_jaccard = row["condition_name_token_jaccard"]

    if missingness != "CONDITION_METADATA_PRESENT_BOTH_RELEASES":
        return "PRIORITY_2_TIER3_MISSINGNESS_INFLUENCED"

    if relaxed_match:
        return "PRIORITY_2_TIER3_FORMATTING_OR_RENAMING_SIGNAL"

    if (
        token_jaccard is not None
        and not pd.isna(token_jaccard)
        and float(token_jaccard) >= 0.80
    ):
        return "PRIORITY_2_TIER3_HIGH_LEXICAL_SIMILARITY"

    if topology == "ONE_TO_ONE_TARGETED_COMPONENT":
        return "PRIORITY_3_TIER3_ONE_TO_ONE_CONDITION_REASSOCIATION"

    if topology == "ONE_TO_MANY_POSSIBLE_SPLIT_COMPONENT":
        return "PRIORITY_3_TIER3_ONE_TO_MANY_CONDITION_CHANGE"

    if topology == "MANY_TO_ONE_POSSIBLE_MERGE_COMPONENT":
        return "PRIORITY_3_TIER3_MANY_TO_ONE_CONDITION_CHANGE"

    return "PRIORITY_3_TIER3_COMPLEX_CONDITION_CHANGE"


# -------------------------------------------------------------------------------------------------
# 6. VERIFY STAGE 3E INPUTS
# -------------------------------------------------------------------------------------------------

required_files = [
    STAGE3E_ADJUDICATION_PATH,
    STAGE3E_T0_STATUS_PATH,
    STAGE3E_MANIFEST_PATH,
]

for required_file in required_files:
    if not required_file.exists():
        raise FileNotFoundError(
            f"Required Stage 3E artifact does not exist: {required_file}"
        )

observed_adjudication_sha256 = sha256_file(
    STAGE3E_ADJUDICATION_PATH
)

observed_t0_status_sha256 = sha256_file(
    STAGE3E_T0_STATUS_PATH
)

observed_stage3e_manifest_sha256 = sha256_file(
    STAGE3E_MANIFEST_PATH
)

if observed_adjudication_sha256 != EXPECTED_ADJUDICATION_SHA256:
    raise RuntimeError(
        "Stage 3E adjudication checksum mismatch.\n"
        f"Expected: {EXPECTED_ADJUDICATION_SHA256}\n"
        f"Observed: {observed_adjudication_sha256}"
    )

if observed_t0_status_sha256 != EXPECTED_T0_STATUS_SHA256:
    raise RuntimeError(
        "Stage 3E T0-status checksum mismatch.\n"
        f"Expected: {EXPECTED_T0_STATUS_SHA256}\n"
        f"Observed: {observed_t0_status_sha256}"
    )

if (
    observed_stage3e_manifest_sha256
    != EXPECTED_STAGE3E_MANIFEST_SHA256
):
    raise RuntimeError(
        "Stage 3E manifest checksum mismatch.\n"
        f"Expected: {EXPECTED_STAGE3E_MANIFEST_SHA256}\n"
        f"Observed: {observed_stage3e_manifest_sha256}"
    )

with STAGE3E_MANIFEST_PATH.open("r", encoding="utf-8") as handle:
    stage3e_manifest = json.load(handle)

if (
    stage3e_manifest.get("adjudication_status")
    != "CONSERVATIVE_STRONG_CANDIDATE_ADJUDICATION_COMPLETE"
):
    raise RuntimeError(
        "Stage 3E does not authorize the targeted Stage 3F audit."
    )

print(f"Stage 3E adjudication SHA-256: {observed_adjudication_sha256}")
print(f"Stage 3E T0 status SHA-256:    {observed_t0_status_sha256}")
print(f"Stage 3E manifest SHA-256:     {observed_stage3e_manifest_sha256}")


# -------------------------------------------------------------------------------------------------
# 7. LOAD AND VALIDATE STAGE 3E DATA
# -------------------------------------------------------------------------------------------------

candidate_pairs = pd.read_parquet(
    STAGE3E_ADJUDICATION_PATH
)

t0_status = pd.read_parquet(
    STAGE3E_T0_STATUS_PATH
)

if len(candidate_pairs) != EXPECTED_ALL_CANDIDATE_PAIRS:
    raise AssertionError(
        f"Expected {EXPECTED_ALL_CANDIDATE_PAIRS:,} candidate pairs, "
        f"observed {len(candidate_pairs):,}."
    )

if len(t0_status) != EXPECTED_T0_STATUS_ROWS:
    raise AssertionError(
        f"Expected {EXPECTED_T0_STATUS_ROWS:,} T0 status rows, "
        f"observed {len(t0_status):,}."
    )

accepted_mask = (
    candidate_pairs[
        "stage3e_accepted_nonexact_link"
    ]
    .fillna(False)
    .astype(bool)
)

if int(accepted_mask.sum()) != EXPECTED_ACCEPTED_STAGE3E_PAIRS:
    raise AssertionError(
        f"Expected {EXPECTED_ACCEPTED_STAGE3E_PAIRS:,} accepted Stage 3E pairs, "
        f"observed {int(accepted_mask.sum()):,}."
    )

deferred_strong_mask = (
    candidate_pairs[
        "stage3e_decision"
    ].isin(DEFERRED_STRONG_DECISIONS)
)

tier3_mask = (
    candidate_pairs["candidate_tier"].eq(TIER3_NAME)
)

if int(deferred_strong_mask.sum()) != EXPECTED_DEFERRED_STRONG_PAIRS:
    raise AssertionError(
        f"Expected {EXPECTED_DEFERRED_STRONG_PAIRS:,} deferred strong pairs, "
        f"observed {int(deferred_strong_mask.sum()):,}."
    )

if int(tier3_mask.sum()) != EXPECTED_TIER3_PAIRS:
    raise AssertionError(
        f"Expected {EXPECTED_TIER3_PAIRS:,} Tier 3 pairs, "
        f"observed {int(tier3_mask.sum()):,}."
    )

if (deferred_strong_mask & tier3_mask).any():
    raise AssertionError(
        "A pair was classified as both deferred strong and Tier 3."
    )

targeted = (
    candidate_pairs.loc[
        deferred_strong_mask | tier3_mask
    ]
    .copy()
    .reset_index(drop=True)
)

if len(targeted) != EXPECTED_TARGETED_PAIRS:
    raise AssertionError(
        f"Expected {EXPECTED_TARGETED_PAIRS:,} targeted pairs, "
        f"observed {len(targeted):,}."
    )

if targeted[
    "stage3e_accepted_nonexact_link"
].fillna(False).any():
    raise AssertionError(
        "An accepted Stage 3E pair entered the unresolved Stage 3F audit."
    )

targeted["stage3f_target_scope"] = [
    (
        "DEFERRED_STRONG_CANDIDATE"
        if decision in DEFERRED_STRONG_DECISIONS
        else "TIER3_CONDITION_CHANGE_CANDIDATE"
    )
    for decision in targeted["stage3e_decision"]
]

print()
print(f"All candidate pairs loaded:       {len(candidate_pairs):,}")
print(f"Deferred strong pairs targeted:   {int(deferred_strong_mask.sum()):,}")
print(f"Tier 3 pairs targeted:            {int(tier3_mask.sum()):,}")
print(f"Total Stage 3F targeted pairs:    {len(targeted):,}")


# -------------------------------------------------------------------------------------------------
# 8. ADD STAGE 3E ACCEPTED-LINK CONTEXT
# -------------------------------------------------------------------------------------------------

accepted_context = candidate_pairs.loc[
    accepted_mask,
    [
        "t0_crosswalk_id",
        "t1_crosswalk_id",
        "t0_rcv_accession",
        "t1_rcv_accession",
        "candidate_pair_id",
    ],
].copy()

accepted_t0_to_t1_rcv = dict(
    zip(
        accepted_context["t0_crosswalk_id"],
        accepted_context["t1_rcv_accession"],
    )
)

accepted_t0_to_pair = dict(
    zip(
        accepted_context["t0_crosswalk_id"],
        accepted_context["candidate_pair_id"],
    )
)

accepted_t1_ids = set(
    accepted_context["t1_crosswalk_id"]
)

targeted["t0_has_stage3e_accepted_link"] = [
    value in accepted_t0_to_t1_rcv
    for value in targeted["t0_crosswalk_id"]
]

targeted["t0_stage3e_accepted_t1_rcv"] = [
    accepted_t0_to_t1_rcv.get(value)
    for value in targeted["t0_crosswalk_id"]
]

targeted["t0_stage3e_accepted_pair_id"] = [
    accepted_t0_to_pair.get(value)
    for value in targeted["t0_crosswalk_id"]
]

targeted["t1_used_by_stage3e_accepted_link"] = [
    value in accepted_t1_ids
    for value in targeted["t1_crosswalk_id"]
]

targeted["accepted_link_context_present"] = (
    targeted["t0_has_stage3e_accepted_link"]
    | targeted["t1_used_by_stage3e_accepted_link"]
)


# -------------------------------------------------------------------------------------------------
# 9. ADD CONDITION-METADATA AUDIT FIELDS
# -------------------------------------------------------------------------------------------------

print("Auditing condition metadata and descriptive lexical similarity...")

metadata_results = [
    pair_metadata_audit(row)
    for _, row in targeted.iterrows()
]

metadata_frame = pd.DataFrame(metadata_results)

targeted = pd.concat(
    [
        targeted.reset_index(drop=True),
        metadata_frame.reset_index(drop=True),
    ],
    axis=1,
)


# -------------------------------------------------------------------------------------------------
# 10. CONSTRUCT TARGETED BIPARTITE GRAPH COMPONENTS
# -------------------------------------------------------------------------------------------------

print("Constructing deterministic T0–T1 candidate graph components...")

union_find = UnionFind()

for _, row in targeted.iterrows():
    t0_node = f"T0::{row['t0_crosswalk_id']}"
    t1_node = f"T1::{row['t1_crosswalk_id']}"

    union_find.union(t0_node, t1_node)

nodes_by_root = defaultdict(list)

for node in sorted(union_find.parent):
    root = union_find.find(node)
    nodes_by_root[root].append(node)

component_id_by_node = {}

for nodes in nodes_by_root.values():
    sorted_nodes = sorted(nodes)

    component_hash = hashlib.sha256(
        "||".join(sorted_nodes).encode("utf-8")
    ).hexdigest()[:16]

    component_id = f"S3F_COMP_{component_hash}"

    for node in sorted_nodes:
        component_id_by_node[node] = component_id

targeted["stage3f_component_id"] = [
    component_id_by_node[
        f"T0::{t0_crosswalk_id}"
    ]
    for t0_crosswalk_id in targeted["t0_crosswalk_id"]
]

if targeted["stage3f_component_id"].isna().any():
    raise AssertionError(
        "At least one targeted pair is missing a graph-component ID."
    )


# -------------------------------------------------------------------------------------------------
# 11. CREATE COMPONENT-LEVEL AUDIT
# -------------------------------------------------------------------------------------------------

component_records = []

for component_id, group in targeted.groupby(
    "stage3f_component_id",
    sort=True,
):
    t0_ids = sorted(
        set(group["t0_crosswalk_id"].astype(str))
    )

    t1_ids = sorted(
        set(group["t1_crosswalk_id"].astype(str))
    )

    t0_rcvs = sorted(
        set(group["t0_rcv_accession"].astype(str))
    )

    t1_rcvs = sorted(
        set(group["t1_rcv_accession"].astype(str))
    )

    topology = component_topology(
        t0_count=len(t0_ids),
        t1_count=len(t1_ids),
    )

    scope = component_scope(
        group["stage3f_target_scope"].tolist()
    )

    lexical_values = (
        group["condition_name_token_jaccard"]
        .dropna()
        .astype(float)
        .tolist()
    )

    component_records.append(
        {
            "stage3f_component_id": component_id,
            "stage3f_component_scope": scope,
            "stage3f_component_topology": topology,

            "component_t0_count": int(len(t0_ids)),
            "component_t1_count": int(len(t1_ids)),
            "component_edge_count": int(len(group)),

            "component_t0_crosswalk_ids_json": json_compact(
                t0_ids
            ),
            "component_t1_crosswalk_ids_json": json_compact(
                t1_ids
            ),
            "component_t0_rcv_accessions_json": json_compact(
                t0_rcvs
            ),
            "component_t1_rcv_accessions_json": json_compact(
                t1_rcvs
            ),

            "component_pair_scope_counts_json": json_count_dict(
                group["stage3f_target_scope"].tolist()
            ),
            "component_candidate_tier_counts_json": json_count_dict(
                group["candidate_tier"].tolist()
            ),
            "component_stage3e_decision_counts_json": json_count_dict(
                group["stage3e_decision"].tolist()
            ),
            "component_condition_evidence_counts_json": json_count_dict(
                group["condition_evidence_category"].tolist()
            ),
            "component_t1_axis_counts_json": json_count_dict(
                group["t1_classification_axis"].tolist()
            ),

            "component_contains_stage3e_accepted_link_context": bool(
                group["accepted_link_context_present"].any()
            ),
            "component_t0_nodes_with_accepted_link_context": int(
                group.loc[
                    group["t0_has_stage3e_accepted_link"],
                    "t0_crosswalk_id",
                ].nunique()
            ),
            "component_t1_nodes_used_by_accepted_link": int(
                group.loc[
                    group["t1_used_by_stage3e_accepted_link"],
                    "t1_crosswalk_id",
                ].nunique()
            ),

            "component_pairs_with_missing_condition_metadata": int(
                group[
                    "condition_metadata_missingness_category"
                ]
                .ne("CONDITION_METADATA_PRESENT_BOTH_RELEASES")
                .sum()
            ),
            "component_relaxed_name_match_pairs": int(
                group["relaxed_condition_name_match_signal"].sum()
            ),
            "component_max_condition_name_token_jaccard": (
                round(max(lexical_values), 6)
                if lexical_values
                else None
            ),

            "stage3f_component_resolution_status": (
                "REQUIRES_SOURCE_HISTORY_OR_PRESPECIFIED_CENSORING_REVIEW"
            ),
            "stage3f_component_accepted_link_count": 0,
            "future_instability_outcome_created": False,
            "unresolved_component_labeled_stable": False,
        }
    )

component_audit = pd.DataFrame(component_records)

if component_audit["stage3f_component_id"].duplicated().any():
    raise AssertionError(
        "Duplicate Stage 3F component identifiers were generated."
    )

component_metadata_columns = [
    "stage3f_component_id",
    "stage3f_component_scope",
    "stage3f_component_topology",
    "component_t0_count",
    "component_t1_count",
    "component_edge_count",
]

targeted = targeted.merge(
    component_audit[component_metadata_columns],
    on="stage3f_component_id",
    how="left",
    validate="many_to_one",
)


# -------------------------------------------------------------------------------------------------
# 12. ASSIGN PAIR-LEVEL REVIEW PRIORITIES
# -------------------------------------------------------------------------------------------------

targeted["stage3f_review_priority"] = [
    targeted_pair_review_priority(row)
    for _, row in targeted.iterrows()
]

targeted["stage3f_resolution_status"] = (
    "UNRESOLVED_REQUIRES_SOURCE_HISTORY_OR_CENSORING_DECISION"
)

targeted["stage3f_accepted_nonexact_link"] = False
targeted["future_instability_outcome_created"] = False
targeted["unresolved_pair_labeled_stable"] = False


# -------------------------------------------------------------------------------------------------
# 13. CREATE T0-LEVEL TARGETED REVIEW SUMMARY
# -------------------------------------------------------------------------------------------------

target_t0_ids = sorted(
    set(targeted["t0_crosswalk_id"].astype(str))
)

t0_target_base = t0_status.loc[
    t0_status[
        "t0_crosswalk_id"
    ].astype(str).isin(target_t0_ids)
].copy()

if len(t0_target_base) != len(target_t0_ids):
    raise AssertionError(
        "The T0 status artifact does not contain every targeted T0 identifier."
    )

pair_groups_by_t0 = {
    str(key): group.copy()
    for key, group in targeted.groupby(
        "t0_crosswalk_id",
        sort=False,
    )
}

t0_review_records = []

for _, t0_row in t0_target_base.iterrows():
    t0_id = str(t0_row["t0_crosswalk_id"])
    group = pair_groups_by_t0[t0_id]

    deferred_count = int(
        group["stage3f_target_scope"]
        .eq("DEFERRED_STRONG_CANDIDATE")
        .sum()
    )

    tier3_count = int(
        group["stage3f_target_scope"]
        .eq("TIER3_CONDITION_CHANGE_CANDIDATE")
        .sum()
    )

    original_status = str(
        t0_row["stage3e_post_adjudication_status"]
    )

    if original_status == "ACCEPTED_NONEXACT_ONE_TO_ONE_CONTINUITY":
        stage3f_t0_category = (
            "ACCEPTED_STAGE3E_LINK_WITH_ADDITIONAL_TIER3_OR_DEFERRED_CONTEXT"
        )

    elif deferred_count > 0:
        stage3f_t0_category = (
            "DEFERRED_COMPLEX_STRONG_RELATIONSHIP_REVIEW"
        )

    elif tier3_count > 0:
        stage3f_t0_category = (
            "TIER3_CONDITION_ASSOCIATION_CHANGE_REVIEW"
        )

    else:
        stage3f_t0_category = (
            "UNEXPECTED_TARGETED_RECORD_CATEGORY"
        )

    lexical_values = (
        group["condition_name_token_jaccard"]
        .dropna()
        .astype(float)
        .tolist()
    )

    t0_review_records.append(
        {
            "t0_crosswalk_id": t0_row["t0_crosswalk_id"],
            "t0_rcv_accession": t0_row["t0_rcv_accession"],
            "t0_variation_id": t0_row["t0_variation_id"],
            "t0_vcv_accession": t0_row["t0_vcv_accession"],
            "t0_genes_normalized_json": (
                t0_row["t0_genes_normalized_json"]
            ),
            "t0_study_scope": t0_row["t0_study_scope"],

            "stage3e_post_adjudication_status": original_status,
            "stage3e_accepted_nonexact_link": bool(
                t0_row["stage3e_accepted_nonexact_link"]
            ),
            "accepted_stage3e_t1_rcv_accession": (
                t0_row.get(
                    "accepted_stage3e_t1_rcv_accession"
                )
            ),

            "stage3f_targeted_candidate_pair_count": int(
                len(group)
            ),
            "stage3f_deferred_strong_pair_count": deferred_count,
            "stage3f_tier3_pair_count": tier3_count,

            "stage3f_component_count": int(
                group["stage3f_component_id"].nunique()
            ),
            "stage3f_component_ids_json": json_compact(
                sorted(
                    set(group["stage3f_component_id"])
                )
            ),
            "stage3f_component_topology_counts_json": json_count_dict(
                group["stage3f_component_topology"].tolist()
            ),

            "stage3f_candidate_t1_rcvs_json": json_compact(
                sorted(
                    set(group["t1_rcv_accession"].astype(str))
                )
            ),
            "stage3f_review_priority_counts_json": json_count_dict(
                group["stage3f_review_priority"].tolist()
            ),

            "stage3f_pairs_with_missing_condition_metadata": int(
                group[
                    "condition_metadata_missingness_category"
                ]
                .ne("CONDITION_METADATA_PRESENT_BOTH_RELEASES")
                .sum()
            ),
            "stage3f_pairs_with_relaxed_name_signal": int(
                group["relaxed_condition_name_match_signal"].sum()
            ),
            "stage3f_max_condition_name_token_jaccard": (
                round(max(lexical_values), 6)
                if lexical_values
                else None
            ),

            "stage3f_t0_review_category": stage3f_t0_category,
            "stage3f_resolution_status": (
                "REQUIRES_SOURCE_HISTORY_OR_FINAL_CENSORING_DECISION"
            ),

            "stage3f_accepted_additional_link": False,
            "future_instability_outcome_created": False,
            "unresolved_record_labeled_stable": False,
        }
    )

t0_review_summary = pd.DataFrame(t0_review_records)

if t0_review_summary["t0_crosswalk_id"].duplicated().any():
    raise AssertionError(
        "Duplicate T0 identifiers were created in the Stage 3F summary."
    )

if t0_review_summary["stage3f_accepted_additional_link"].any():
    raise AssertionError(
        "Stage 3F unexpectedly accepted an additional linkage."
    )


# -------------------------------------------------------------------------------------------------
# 14. SELECT AND ORDER PAIR-AUDIT COLUMNS
# -------------------------------------------------------------------------------------------------

pair_audit_columns = [
    "candidate_pair_id",
    "t0_crosswalk_id",
    "t1_crosswalk_id",

    "t0_rcv_accession",
    "t1_rcv_accession",
    "t0_rcv_version",
    "t1_rcv_version",

    "t0_variation_id",
    "t1_variation_id",
    "variation_id_same",

    "t0_vcv_accession",
    "t1_vcv_accession",
    "vcv_accession_same",

    "t0_genes_normalized_json",
    "t1_genes_normalized_json",
    "gene_relation",

    "t0_study_scope",
    "t1_study_scope",
    "study_scope_relation",

    "t0_condition_ids_normalized_json",
    "t1_condition_ids_normalized_json",
    "condition_id_relation",
    "condition_id_overlap_count",
    "condition_id_overlap_json",

    "t0_condition_names_normalized_json",
    "t1_condition_names_normalized_json",
    "condition_name_relation",
    "condition_name_overlap_count",
    "condition_name_overlap_json",

    "condition_evidence_category",
    "condition_metadata_missingness_category",

    "t0_condition_id_count",
    "t1_condition_id_count",
    "t0_condition_name_count",
    "t1_condition_name_count",

    "t0_condition_namespaces_json",
    "t1_condition_namespaces_json",
    "condition_namespace_relation",
    "condition_namespace_overlap_count",
    "condition_namespace_overlap_json",

    "condition_name_token_overlap_count",
    "condition_name_token_overlap_json",
    "condition_name_token_jaccard",

    "relaxed_condition_name_overlap_count",
    "relaxed_condition_name_overlap_json",
    "relaxed_condition_name_match_signal",

    "t1_classification_axis",

    "candidate_tier",
    "candidate_topology",
    "stage3e_decision",

    "t0_has_stage3e_accepted_link",
    "t0_stage3e_accepted_t1_rcv",
    "t0_stage3e_accepted_pair_id",
    "t1_used_by_stage3e_accepted_link",
    "accepted_link_context_present",

    "stage3f_target_scope",
    "stage3f_component_id",
    "stage3f_component_scope",
    "stage3f_component_topology",
    "component_t0_count",
    "component_t1_count",
    "component_edge_count",

    "stage3f_review_priority",
    "stage3f_resolution_status",
    "stage3f_accepted_nonexact_link",

    "future_instability_outcome_created",
    "unresolved_pair_labeled_stable",
]

missing_pair_columns = [
    column
    for column in pair_audit_columns
    if column not in targeted.columns
]

if missing_pair_columns:
    raise KeyError(
        "Missing Stage 3F pair-audit columns: "
        + ", ".join(missing_pair_columns)
    )

pair_audit = targeted[pair_audit_columns].copy()

pair_audit = (
    pair_audit.sort_values(
        by=[
            "stage3f_component_id",
            "stage3f_target_scope",
            "t0_rcv_accession",
            "t1_rcv_accession",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)

component_audit = (
    component_audit.sort_values(
        by=[
            "stage3f_component_scope",
            "stage3f_component_topology",
            "stage3f_component_id",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)

t0_review_summary = (
    t0_review_summary.sort_values(
        by=[
            "stage3f_t0_review_category",
            "t0_rcv_accession",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


# -------------------------------------------------------------------------------------------------
# 15. FINAL STRUCTURAL VALIDATION
# -------------------------------------------------------------------------------------------------

if len(pair_audit) != EXPECTED_TARGETED_PAIRS:
    raise AssertionError(
        "Stage 3F pair-audit row count changed unexpectedly."
    )

if pair_audit["candidate_pair_id"].duplicated().any():
    raise AssertionError(
        "Duplicate pair identifiers were found in the Stage 3F pair audit."
    )

if pair_audit["stage3f_component_id"].isna().any():
    raise AssertionError(
        "Missing graph-component IDs were found in the pair audit."
    )

component_edge_total = int(
    component_audit["component_edge_count"].sum()
)

if component_edge_total != EXPECTED_TARGETED_PAIRS:
    raise AssertionError(
        "Component edge accounting does not equal the targeted pair count."
    )

if pair_audit["stage3f_accepted_nonexact_link"].any():
    raise AssertionError(
        "A Stage 3F targeted pair was incorrectly accepted."
    )

if pair_audit["future_instability_outcome_created"].any():
    raise AssertionError(
        "A temporal outcome was unexpectedly created."
    )

if pair_audit["unresolved_pair_labeled_stable"].any():
    raise AssertionError(
        "An unresolved pair was unexpectedly labeled stable."
    )


# -------------------------------------------------------------------------------------------------
# 16. WRITE AND READ BACK OUTPUT ARTIFACTS
# -------------------------------------------------------------------------------------------------

pair_audit.to_parquet(
    PAIR_AUDIT_PATH,
    engine="pyarrow",
    compression="zstd",
    index=False,
)

component_audit.to_parquet(
    COMPONENT_AUDIT_PATH,
    engine="pyarrow",
    compression="zstd",
    index=False,
)

t0_review_summary.to_parquet(
    T0_REVIEW_SUMMARY_PATH,
    engine="pyarrow",
    compression="zstd",
    index=False,
)

pair_audit_sha256 = sha256_file(PAIR_AUDIT_PATH)
component_audit_sha256 = sha256_file(COMPONENT_AUDIT_PATH)
t0_review_summary_sha256 = sha256_file(
    T0_REVIEW_SUMMARY_PATH
)

pair_metadata = pq.ParquetFile(
    PAIR_AUDIT_PATH
).metadata

component_metadata = pq.ParquetFile(
    COMPONENT_AUDIT_PATH
).metadata

t0_review_metadata = pq.ParquetFile(
    T0_REVIEW_SUMMARY_PATH
).metadata

pair_readback = pd.read_parquet(PAIR_AUDIT_PATH)
component_readback = pd.read_parquet(
    COMPONENT_AUDIT_PATH
)
t0_review_readback = pd.read_parquet(
    T0_REVIEW_SUMMARY_PATH
)

if len(pair_readback) != len(pair_audit):
    raise AssertionError(
        "Pair-audit readback row-count mismatch."
    )

if len(component_readback) != len(component_audit):
    raise AssertionError(
        "Component-audit readback row-count mismatch."
    )

if len(t0_review_readback) != len(t0_review_summary):
    raise AssertionError(
        "T0-review readback row-count mismatch."
    )

if list(pair_readback.columns) != list(pair_audit.columns):
    raise AssertionError(
        "Pair-audit readback schema mismatch."
    )

if list(component_readback.columns) != list(
    component_audit.columns
):
    raise AssertionError(
        "Component-audit readback schema mismatch."
    )

if list(t0_review_readback.columns) != list(
    t0_review_summary.columns
):
    raise AssertionError(
        "T0-review readback schema mismatch."
    )


# -------------------------------------------------------------------------------------------------
# 17. CREATE STAGE 3F REPORT
# -------------------------------------------------------------------------------------------------

t0_records_with_stage3e_accepted_context = int(
    t0_review_summary[
        "stage3e_accepted_nonexact_link"
    ].sum()
)

pair_missingness_count = int(
    pair_audit[
        "condition_metadata_missingness_category"
    ]
    .ne("CONDITION_METADATA_PRESENT_BOTH_RELEASES")
    .sum()
)

pair_relaxed_name_signal_count = int(
    pair_audit[
        "relaxed_condition_name_match_signal"
    ].sum()
)

high_lexical_similarity_count = int(
    pair_audit[
        "condition_name_token_jaccard"
    ]
    .fillna(-1)
    .ge(0.80)
    .sum()
)

report = {
    "report_name": (
        "Stage 3F Deferred-Complex and Tier 3 Targeted Audit Report"
    ),
    "report_version": "1.0.0",
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "study_stage": "Stage 3F",
    "scientific_unit": (
        "RCV-level variant-condition aggregate"
    ),

    "inputs": {
        "all_stage3e_candidate_pairs": int(
            len(candidate_pairs)
        ),
        "accepted_stage3e_pairs_excluded_from_targeted_audit": int(
            accepted_mask.sum()
        ),
        "t0_status_records": int(len(t0_status)),
    },

    "targeted_pair_counts": {
        "targeted_pairs": int(len(pair_audit)),
        "deferred_strong_pairs": int(
            pair_audit[
                "stage3f_target_scope"
            ]
            .eq("DEFERRED_STRONG_CANDIDATE")
            .sum()
        ),
        "tier3_condition_change_pairs": int(
            pair_audit[
                "stage3f_target_scope"
            ]
            .eq("TIER3_CONDITION_CHANGE_CANDIDATE")
            .sum()
        ),
        "target_scope_counts": value_counts_dict(
            pair_audit["stage3f_target_scope"]
        ),
        "review_priority_counts": value_counts_dict(
            pair_audit["stage3f_review_priority"]
        ),
    },

    "graph_component_results": {
        "component_count": int(len(component_audit)),
        "component_scope_counts": value_counts_dict(
            component_audit["stage3f_component_scope"]
        ),
        "component_topology_counts": value_counts_dict(
            component_audit["stage3f_component_topology"]
        ),
        "components_with_stage3e_accepted_link_context": int(
            component_audit[
                "component_contains_stage3e_accepted_link_context"
            ].sum()
        ),
        "component_edge_accounting": int(
            component_audit["component_edge_count"].sum()
        ),
    },

    "condition_metadata_results": {
        "missingness_category_counts": value_counts_dict(
            pair_audit[
                "condition_metadata_missingness_category"
            ]
        ),
        "pairs_with_condition_metadata_missingness": int(
            pair_missingness_count
        ),
        "namespace_relation_counts": value_counts_dict(
            pair_audit["condition_namespace_relation"]
        ),
        "pairs_with_relaxed_condition_name_signal": int(
            pair_relaxed_name_signal_count
        ),
        "pairs_with_token_jaccard_at_least_0_80": int(
            high_lexical_similarity_count
        ),
        "lexical_similarity_interpretation": (
            "Lexical results are descriptive review-prioritization "
            "signals only and do not establish condition equivalence."
        ),
    },

    "t0_review_results": {
        "unique_t0_records_in_targeted_pair_graph": int(
            len(t0_review_summary)
        ),
        "t0_review_category_counts": value_counts_dict(
            t0_review_summary[
                "stage3f_t0_review_category"
            ]
        ),
        "targeted_t0_records_with_existing_stage3e_accepted_link": int(
            t0_records_with_stage3e_accepted_context
        ),
    },

    "validation": {
        "expected_targeted_pairs": int(
            EXPECTED_TARGETED_PAIRS
        ),
        "observed_targeted_pairs": int(
            len(pair_audit)
        ),
        "component_edge_total": int(
            component_edge_total
        ),
        "duplicate_pair_ids": int(
            pair_audit[
                "candidate_pair_id"
            ].duplicated().sum()
        ),
        "duplicate_component_ids": int(
            component_audit[
                "stage3f_component_id"
            ].duplicated().sum()
        ),
        "additional_links_accepted": int(
            pair_audit[
                "stage3f_accepted_nonexact_link"
            ].sum()
        ),
        "readback_passed": True,
        "critical_failures": 0,
    },

    "scientific_boundaries": {
        "additional_nonexact_links_accepted": False,
        "stage3e_links_revoked_or_modified": False,
        "official_rcv_replacements_confirmed": False,
        "merges_confirmed": False,
        "splits_confirmed": False,
        "condition_equivalence_inferred_from_lexical_similarity": False,
        "classification_changes_calculated": False,
        "future_instability_outcomes_created": False,
        "unresolved_records_labeled_stable": False,
        "ges_model_fitted_or_tuned": False,
    },

    "next_authorized_step": (
        "Audit the remaining T0 records with no VariationID- or "
        "VCV-based T1 candidate and assign prespecified unresolved, "
        "unmatched, or censored categories. No temporal outcome may "
        "be created until the complete linkage artifact and exclusion "
        "policy are frozen."
    ),
}

with REPORT_PATH.open("w", encoding="utf-8") as handle:
    json.dump(
        report,
        handle,
        indent=2,
        sort_keys=True,
        ensure_ascii=False,
    )

report_sha256 = sha256_file(REPORT_PATH)


# -------------------------------------------------------------------------------------------------
# 18. CREATE STAGE 3F MANIFEST
# -------------------------------------------------------------------------------------------------

manifest = {
    "manifest_name": (
        "Stage 3F Deferred-Complex and Tier 3 Targeted Audit Manifest"
    ),
    "manifest_version": "1.0.0",
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "study_stage": "Stage 3F",
    "audit_status": (
        "TARGETED_COMPLEX_AND_TIER3_AUDIT_COMPLETE"
    ),

    "inputs": {
        "stage3e_adjudication": {
            "path": str(STAGE3E_ADJUDICATION_PATH),
            "sha256": observed_adjudication_sha256,
        },
        "stage3e_t0_status": {
            "path": str(STAGE3E_T0_STATUS_PATH),
            "sha256": observed_t0_status_sha256,
        },
        "stage3e_manifest": {
            "path": str(STAGE3E_MANIFEST_PATH),
            "sha256": observed_stage3e_manifest_sha256,
        },
    },

    "outputs": {
        "targeted_pair_audit": {
            "path": str(PAIR_AUDIT_PATH),
            "sha256": pair_audit_sha256,
            "rows": int(pair_metadata.num_rows),
            "columns": int(pair_metadata.num_columns),
            "row_groups": int(pair_metadata.num_row_groups),
        },
        "targeted_component_audit": {
            "path": str(COMPONENT_AUDIT_PATH),
            "sha256": component_audit_sha256,
            "rows": int(component_metadata.num_rows),
            "columns": int(component_metadata.num_columns),
            "row_groups": int(component_metadata.num_row_groups),
        },
        "targeted_t0_review_summary": {
            "path": str(T0_REVIEW_SUMMARY_PATH),
            "sha256": t0_review_summary_sha256,
            "rows": int(t0_review_metadata.num_rows),
            "columns": int(t0_review_metadata.num_columns),
            "row_groups": int(t0_review_metadata.num_row_groups),
        },
        "report": {
            "path": str(REPORT_PATH),
            "sha256": report_sha256,
        },
    },

    "targeted_pair_count": int(len(pair_audit)),
    "graph_component_count": int(len(component_audit)),
    "unique_targeted_t0_record_count": int(
        len(t0_review_summary)
    ),

    "validation_decision": "PASS",
    "scientific_boundaries": (
        report["scientific_boundaries"]
    ),
    "next_authorized_step": (
        report["next_authorized_step"]
    ),
}

with MANIFEST_PATH.open("w", encoding="utf-8") as handle:
    json.dump(
        manifest,
        handle,
        indent=2,
        sort_keys=True,
        ensure_ascii=False,
    )

manifest_sha256 = sha256_file(MANIFEST_PATH)


# -------------------------------------------------------------------------------------------------
# 19. PRINT RESULTS
# -------------------------------------------------------------------------------------------------

print()
print("=" * 126)
print("STAGE 3F TARGETED-AUDIT SUMMARY")
print("=" * 126)

print(f"Targeted candidate pairs audited:             {len(pair_audit):,}")
print(
    f"  Deferred strong pairs:                      "
    f"{pair_audit['stage3f_target_scope'].eq('DEFERRED_STRONG_CANDIDATE').sum():,}"
)
print(
    f"  Tier 3 condition-change pairs:              "
    f"{pair_audit['stage3f_target_scope'].eq('TIER3_CONDITION_CHANGE_CANDIDATE').sum():,}"
)

print()
print(f"Targeted graph components:                    {len(component_audit):,}")
print(f"Unique T0 records in targeted graph:          {len(t0_review_summary):,}")
print(
    f"T0 records with an existing Stage 3E link:    "
    f"{t0_records_with_stage3e_accepted_context:,}"
)

print()
print(f"Pairs influenced by condition missingness:    {pair_missingness_count:,}")
print(f"Pairs with relaxed-name match signal:         {pair_relaxed_name_signal_count:,}")
print(f"Pairs with token Jaccard ≥ 0.80:              {high_lexical_similarity_count:,}")

print()
print("Graph-component topology:")
print(
    component_audit[
        "stage3f_component_topology"
    ]
    .value_counts(dropna=False)
    .to_string()
)

print()
print("Graph-component scope:")
print(
    component_audit[
        "stage3f_component_scope"
    ]
    .value_counts(dropna=False)
    .to_string()
)

print()
print("Pair-level review priorities:")
print(
    pair_audit[
        "stage3f_review_priority"
    ]
    .value_counts(dropna=False)
    .to_string()
)

print()
print("T0-level targeted review categories:")
print(
    t0_review_summary[
        "stage3f_t0_review_category"
    ]
    .value_counts(dropna=False)
    .to_string()
)

print()
print(f"Pair audit:             {PAIR_AUDIT_PATH}")
print(f"Pair audit SHA-256:     {pair_audit_sha256}")
print(f"Component audit:        {COMPONENT_AUDIT_PATH}")
print(f"Component SHA-256:      {component_audit_sha256}")
print(f"T0 review summary:      {T0_REVIEW_SUMMARY_PATH}")
print(f"T0 review SHA-256:      {t0_review_summary_sha256}")
print(f"Report:                 {REPORT_PATH}")
print(f"Report SHA-256:         {report_sha256}")
print(f"Manifest:               {MANIFEST_PATH}")
print(f"Manifest SHA-256:       {manifest_sha256}")

print()
print("PASS — Stage 3F targeted complex and Tier 3 audit completed.")
print(
    "IMPORTANT — No additional non-exact link was accepted or rejected automatically."
)
print(
    "IMPORTANT — Lexical similarity was used only for review prioritization, "
    "not condition-equivalence inference."
)
print(
    "IMPORTANT — No unresolved record was labeled stable and no temporal "
    "outcome was created."
)
print(
    "AUTHORIZED NEXT ACTION — Stage 3G audit and categorization of records "
    "with no VariationID- or VCV-based T1 candidate."
)
print("=" * 126)

STAGE 3F — DEFERRED-COMPLEX AND TIER 3 CONDITION-CHANGE AUDIT
Stage 3E adjudication SHA-256: f647a8fe66db8a695da2136f3ec59ad003425f11a2767a3198eefa612dbcf4ef
Stage 3E T0 status SHA-256:    c1e40c92389e2dc200cd1508c49fb7aa12c3a9665fc2e6aab56ed2127accb65e
Stage 3E manifest SHA-256:     b5ea53ba3fe6cafdb2abaa25ac582d683cfb35dd04d88dcfdaf03802edec4304

All candidate pairs loaded:       1,078
Deferred strong pairs targeted:   78
Tier 3 pairs targeted:            830
Total Stage 3F targeted pairs:    908
Auditing condition metadata and descriptive lexical similarity...
Constructing deterministic T0–T1 candidate graph components...

STAGE 3F TARGETED-AUDIT SUMMARY
Targeted candidate pairs audited:             908
  Deferred strong pairs:                      78
  Tier 3 condition-change pairs:              830

Targeted graph components:                    534
Unique T0 records in targeted graph:          561
T0 records with an existing Stage 3E link:    107

Pairs influenced by condition mis

In [18]:
# =================================================================================================
# STAGE 3G — AUDIT T0 RECORDS WITH NO VARIATIONID- OR VCV-BASED T1 CANDIDATE
# =================================================================================================
# Purpose:
#   Audit the 622 T0 records for which Stage 3D found no T1-only record sharing:
#       - VariationID, or
#       - VCV accession.
#
# This cell searches only for descriptive, same-gene condition-based candidates:
#       - exact structured condition-ID overlap, and/or
#       - exact normalized condition-name overlap.
#
# These are CONDITION-ONLY candidates. They do not establish variant continuity.
#
# This cell:
#   1. Verifies the Stage 3B, Stage 3E, and Stage 3F inputs.
#   2. Selects exactly the 622 no-stable-identifier T0 records.
#   3. Searches the 30,507 T1-only records using same-gene condition evidence.
#   4. Creates pair-level and T0-level audit artifacts.
#   5. Creates a source-history/final-censoring review queue.
#
# This cell DOES NOT:
#   - accept any condition-only candidate as a non-exact link,
#   - infer that condition similarity means variant equivalence,
#   - use current/live ClinVar data beyond the frozen T1 release,
#   - compare aggregate classifications,
#   - construct future-instability outcomes,
#   - label unresolved records as stable,
#   - fit or tune GES.
# =================================================================================================

from pathlib import Path
from datetime import datetime, timezone
from collections import Counter, defaultdict
import hashlib
import json
import re
import unicodedata

import pandas as pd
import pyarrow.parquet as pq


print("=" * 126)
print("STAGE 3G — NO-STABLE-IDENTIFIER CANDIDATE AUDIT")
print("=" * 126)


# -------------------------------------------------------------------------------------------------
# 1. PATHS AND EXPECTED CHECKSUMS
# -------------------------------------------------------------------------------------------------

STUDY_ROOT = Path("/content/drive/MyDrive/GES_RAG_Temporal_Study")

STAGE3_CONFIG_DIR = STUDY_ROOT / "configs" / "stage3_crosswalk"
STAGE3_OUTPUT_DIR = STUDY_ROOT / "data_processed" / "stage3_crosswalk"

CROSSWALK_PATH = (
    STAGE3_OUTPUT_DIR
    / "stage3_t0_t1_exact_rcv_crosswalk_draft_v1.parquet"
)

T0_STAGE3E_STATUS_PATH = (
    STAGE3_OUTPUT_DIR
    / "stage3_t0_post_strong_candidate_adjudication_v1.parquet"
)

STAGE3F_T0_REVIEW_PATH = (
    STAGE3_OUTPUT_DIR
    / "stage3_targeted_candidate_t0_review_summary_v1.parquet"
)

STAGE3F_MANIFEST_PATH = (
    STAGE3_CONFIG_DIR
    / "stage3_targeted_complex_and_tier3_audit_manifest_v1.json"
)

PAIR_AUDIT_PATH = (
    STAGE3_OUTPUT_DIR
    / "stage3_no_identifier_condition_only_candidate_pairs_v1.parquet"
)

T0_AUDIT_PATH = (
    STAGE3_OUTPUT_DIR
    / "stage3_no_identifier_t0_audit_summary_v1.parquet"
)

REVIEW_QUEUE_PATH = (
    STAGE3_OUTPUT_DIR
    / "stage3_no_identifier_source_history_review_queue_v1.parquet"
)

REPORT_PATH = (
    STAGE3_CONFIG_DIR
    / "stage3_no_identifier_condition_only_audit_report_v1.json"
)

MANIFEST_PATH = (
    STAGE3_CONFIG_DIR
    / "stage3_no_identifier_condition_only_audit_manifest_v1.json"
)


EXPECTED_CROSSWALK_SHA256 = (
    "b462304a4bb31db301e2dac3aefbf685"
    "e12f1fea179a38ca04b8500b64bb4acc"
)

EXPECTED_T0_STAGE3E_STATUS_SHA256 = (
    "c1e40c92389e2dc200cd1508c49fb7a"
    "a12c3a9665fc2e6aab56ed2127accb65e"
)

EXPECTED_STAGE3F_T0_REVIEW_SHA256 = (
    "09f6e4d24744781127e59d7ee03ddf3d"
    "5d02b0c4ea5517e52a184ad46a623d2a"
)

EXPECTED_STAGE3F_MANIFEST_SHA256 = (
    "004292519a8a33d261c9a57acdbd6573"
    "fc074338a2bb311bbd464b8fd6fa47b1"
)

EXPECTED_NO_IDENTIFIER_T0_RECORDS = 622
EXPECTED_T1_ONLY_RECORDS = 30507
EXPECTED_STAGE3F_TARGETED_T0_RECORDS = 561

MAX_CONDITION_ONLY_PAIR_ROWS = 2_000_000


PLACEHOLDER_CONDITION_NAMES = {
    "not provided",
    "not specified",
    "unspecified",
    "unknown",
    "see cases",
    "see case",
    "not applicable",
    "no condition provided",
}


# -------------------------------------------------------------------------------------------------
# 2. GENERAL UTILITIES
# -------------------------------------------------------------------------------------------------

def sha256_file(path: Path, chunk_size: int = 8 * 1024 * 1024) -> str:
    """Calculate SHA-256 without loading the entire file into memory."""
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def is_missing(value) -> bool:
    """Return True for scalar missing or blank values."""
    if value is None:
        return True

    if isinstance(value, str):
        return value.strip() == ""

    try:
        result = pd.isna(value)

        if isinstance(result, bool):
            return result
    except (TypeError, ValueError):
        pass

    return False


def clean_identifier(value, uppercase: bool = True):
    """Normalize an identifier conservatively."""
    if is_missing(value):
        return None

    text = str(value).strip()

    if re.fullmatch(r"\d+\.0", text):
        text = text[:-2]

    if uppercase:
        text = text.upper()

    return text or None


def parse_json_list(value, field_name: str) -> list:
    """Parse and validate a serialized JSON list."""
    if value is None:
        return []

    if isinstance(value, list):
        parsed = value

    elif isinstance(value, tuple):
        parsed = list(value)

    else:
        try:
            if pd.isna(value):
                return []
        except (TypeError, ValueError):
            pass

        text = str(value).strip()

        if text == "":
            return []

        try:
            parsed = json.loads(text)
        except json.JSONDecodeError as exc:
            raise ValueError(
                f"Malformed JSON in {field_name}: {text[:200]}"
            ) from exc

    if not isinstance(parsed, list):
        raise TypeError(
            f"{field_name} must contain a JSON list, "
            f"not {type(parsed).__name__}."
        )

    return parsed


def normalize_gene(value):
    """Normalize a gene symbol."""
    if value is None:
        return None

    text = str(value).strip().upper()

    return text or None


def normalize_condition_id(value):
    """Normalize a structured condition identifier."""
    if value is None:
        return None

    text = unicodedata.normalize("NFKC", str(value))
    text = re.sub(r"\s+", "", text).strip().upper()

    return text or None


def normalize_condition_name(value):
    """Normalize condition text without synonym inference."""
    if value is None:
        return None

    text = unicodedata.normalize("NFKC", str(value))
    text = re.sub(r"\s+", " ", text).strip().casefold()

    return text or None


def normalized_json_tuple(value, field_name: str, normalizer) -> tuple:
    """Parse a JSON list and return sorted, normalized unique values."""
    parsed = parse_json_list(value, field_name)

    values = []

    for item in parsed:
        normalized = normalizer(item)

        if normalized is not None:
            values.append(normalized)

    return tuple(sorted(set(values)))


def compact_json(values) -> str:
    """Serialize a collection deterministically."""
    return json.dumps(
        list(values),
        ensure_ascii=False,
        separators=(",", ":"),
    )


def count_json(values) -> str:
    """Serialize value counts deterministically."""
    counts = Counter(values)

    return json.dumps(
        {
            str(key): int(value)
            for key, value in sorted(counts.items())
        },
        ensure_ascii=False,
        separators=(",", ":"),
    )


def set_relation(left_values, right_values) -> str:
    """Describe the relationship between two sets."""
    left = set(left_values)
    right = set(right_values)

    if not left and not right:
        return "BOTH_EMPTY"

    if not left:
        return "T0_EMPTY"

    if not right:
        return "T1_EMPTY"

    if left == right:
        return "EXACT_SET_MATCH"

    if left.intersection(right):
        return "PARTIAL_SET_OVERLAP"

    return "DISJOINT_SETS"


def canonical_axis(value) -> str:
    """Normalize T1 classification-axis wording."""
    if is_missing(value):
        return "MissingAxis"

    raw = str(value).strip()
    key = re.sub(r"[^a-z]", "", raw.casefold())

    mapping = {
        "germlineclassification": "GermlineClassification",
        "oncogenicityclassification": "OncogenicityClassification",
        "somaticclinicalimpact": "SomaticClinicalImpact",
        "noclassification": "NoClassification",
    }

    return mapping.get(key, raw)


def value_counts_dict(series: pd.Series) -> dict:
    """Create a JSON-safe count dictionary."""
    counts = series.value_counts(dropna=False)

    result = {}

    for key, value in counts.items():
        safe_key = "<MISSING>" if pd.isna(key) else str(key)
        result[safe_key] = int(value)

    return result


# -------------------------------------------------------------------------------------------------
# 3. CANDIDATE AND T0-LEVEL CATEGORIZATION
# -------------------------------------------------------------------------------------------------

def pair_evidence_category(
    condition_id_overlap_count: int,
    condition_name_overlap_count: int,
) -> str:
    """Classify exact condition evidence for one same-gene candidate pair."""

    if (
        condition_id_overlap_count > 0
        and condition_name_overlap_count > 0
    ):
        return "CONDITION_ID_AND_NAME_OVERLAP"

    if condition_id_overlap_count > 0:
        return "CONDITION_ID_OVERLAP_ONLY"

    if condition_name_overlap_count > 0:
        return "CONDITION_NAME_OVERLAP_ONLY"

    return "ERROR_NO_CONDITION_EVIDENCE"


def pair_review_priority(
    id_relation: str,
    name_relation: str,
    evidence_category: str,
) -> str:
    """Prioritize review without accepting a condition-only candidate."""

    if (
        id_relation == "EXACT_SET_MATCH"
        and name_relation == "EXACT_SET_MATCH"
    ):
        return "PRIORITY_1_EXACT_CONDITION_ID_AND_NAME_SETS"

    if evidence_category == "CONDITION_ID_AND_NAME_OVERLAP":
        return "PRIORITY_2_CONDITION_ID_AND_NAME_OVERLAP"

    if evidence_category == "CONDITION_ID_OVERLAP_ONLY":
        return "PRIORITY_3_CONDITION_ID_OVERLAP_ONLY"

    return "PRIORITY_4_CONDITION_NAME_OVERLAP_ONLY"


def t0_audit_category(
    candidate_count: int,
    exact_condition_set_candidate_count: int,
    condition_ids,
    condition_names,
) -> str:
    """Create a conservative T0-level no-identifier disposition."""

    names = set(condition_names)
    placeholder_only = bool(names) and names.issubset(
        PLACEHOLDER_CONDITION_NAMES
    )

    if candidate_count == 1:
        if exact_condition_set_candidate_count == 1:
            return (
                "UNIQUE_EXACT_CONDITION_SET_CANDIDATE_"
                "WITHOUT_STABLE_VARIANT_IDENTIFIER"
            )

        return (
            "UNIQUE_CONDITION_OVERLAP_CANDIDATE_"
            "WITHOUT_STABLE_VARIANT_IDENTIFIER"
        )

    if candidate_count > 1:
        return "MULTIPLE_CONDITION_ONLY_CANDIDATES_AMBIGUOUS"

    if placeholder_only:
        return "NO_CONDITION_CANDIDATE_PLACEHOLDER_CONDITION_NAME"

    if not condition_ids:
        return "NO_CONDITION_CANDIDATE_STRUCTURED_IDS_MISSING"

    return "NO_CONDITION_CANDIDATE_DESPITE_STRUCTURED_CONDITION_IDS"


def source_history_priority(audit_category: str) -> str:
    """Assign a final linkage/censoring review priority."""

    if audit_category.startswith("UNIQUE_"):
        return "PRIORITY_1_UNIQUE_CONDITION_ONLY_CANDIDATE_REVIEW"

    if audit_category == (
        "NO_CONDITION_CANDIDATE_DESPITE_STRUCTURED_CONDITION_IDS"
    ):
        return "PRIORITY_1_NO_T1_CONDITION_CANDIDATE_DESPITE_IDS"

    if audit_category == (
        "MULTIPLE_CONDITION_ONLY_CANDIDATES_AMBIGUOUS"
    ):
        return "PRIORITY_2_MULTIPLE_AMBIGUOUS_CONDITION_CANDIDATES"

    return "PRIORITY_3_MISSING_OR_NONSPECIFIC_CONDITION_METADATA"


# -------------------------------------------------------------------------------------------------
# 4. VERIFY INPUT ARTIFACTS
# -------------------------------------------------------------------------------------------------

required_files = [
    CROSSWALK_PATH,
    T0_STAGE3E_STATUS_PATH,
    STAGE3F_T0_REVIEW_PATH,
    STAGE3F_MANIFEST_PATH,
]

for required_file in required_files:
    if not required_file.exists():
        raise FileNotFoundError(
            f"Required artifact does not exist: {required_file}"
        )

observed_crosswalk_sha256 = sha256_file(CROSSWALK_PATH)
observed_t0_status_sha256 = sha256_file(T0_STAGE3E_STATUS_PATH)
observed_stage3f_t0_review_sha256 = sha256_file(
    STAGE3F_T0_REVIEW_PATH
)
observed_stage3f_manifest_sha256 = sha256_file(
    STAGE3F_MANIFEST_PATH
)

if observed_crosswalk_sha256 != EXPECTED_CROSSWALK_SHA256:
    raise RuntimeError(
        "Stage 3B crosswalk checksum mismatch.\n"
        f"Expected: {EXPECTED_CROSSWALK_SHA256}\n"
        f"Observed: {observed_crosswalk_sha256}"
    )

if (
    observed_t0_status_sha256
    != EXPECTED_T0_STAGE3E_STATUS_SHA256
):
    raise RuntimeError(
        "Stage 3E T0-status checksum mismatch.\n"
        f"Expected: {EXPECTED_T0_STAGE3E_STATUS_SHA256}\n"
        f"Observed: {observed_t0_status_sha256}"
    )

if (
    observed_stage3f_t0_review_sha256
    != EXPECTED_STAGE3F_T0_REVIEW_SHA256
):
    raise RuntimeError(
        "Stage 3F T0-review checksum mismatch.\n"
        f"Expected: {EXPECTED_STAGE3F_T0_REVIEW_SHA256}\n"
        f"Observed: {observed_stage3f_t0_review_sha256}"
    )

if (
    observed_stage3f_manifest_sha256
    != EXPECTED_STAGE3F_MANIFEST_SHA256
):
    raise RuntimeError(
        "Stage 3F manifest checksum mismatch.\n"
        f"Expected: {EXPECTED_STAGE3F_MANIFEST_SHA256}\n"
        f"Observed: {observed_stage3f_manifest_sha256}"
    )

with STAGE3F_MANIFEST_PATH.open("r", encoding="utf-8") as handle:
    stage3f_manifest = json.load(handle)

if (
    stage3f_manifest.get("audit_status")
    != "TARGETED_COMPLEX_AND_TIER3_AUDIT_COMPLETE"
):
    raise RuntimeError(
        "Stage 3F does not authorize the no-identifier audit."
    )

print(f"Stage 3B crosswalk SHA-256: {observed_crosswalk_sha256}")
print(f"Stage 3E T0 status SHA-256:{observed_t0_status_sha256}")
print(f"Stage 3F T0 review SHA-256:{observed_stage3f_t0_review_sha256}")
print(f"Stage 3F manifest SHA-256: {observed_stage3f_manifest_sha256}")


# -------------------------------------------------------------------------------------------------
# 5. LOAD THE 622 T0 RECORDS AND THE 30,507 T1-ONLY RECORDS
# -------------------------------------------------------------------------------------------------

t0_required_columns = [
    "t0_crosswalk_id",
    "t0_rcv_accession",
    "t0_rcv_version",
    "t0_variation_id",
    "t0_vcv_accession",
    "t0_vcv_version",
    "t0_genes_normalized_json",
    "t0_study_scope",
    "t0_condition_ids_normalized_json",
    "t0_condition_names_normalized_json",
    "stage3e_post_adjudication_status",
]

t0_status_schema = set(
    pq.ParquetFile(T0_STAGE3E_STATUS_PATH).schema_arrow.names
)

missing_t0_columns = sorted(
    set(t0_required_columns) - t0_status_schema
)

if missing_t0_columns:
    raise KeyError(
        "Stage 3E T0 status is missing columns: "
        + ", ".join(missing_t0_columns)
    )

t0_status = pd.read_parquet(
    T0_STAGE3E_STATUS_PATH,
    columns=t0_required_columns,
)

t0_no_identifier = (
    t0_status.loc[
        t0_status["stage3e_post_adjudication_status"].eq(
            "UNRESOLVED_NO_VARIATIONID_OR_VCV_CANDIDATE"
        )
    ]
    .copy()
    .reset_index(drop=True)
)

if len(t0_no_identifier) != EXPECTED_NO_IDENTIFIER_T0_RECORDS:
    raise AssertionError(
        f"Expected {EXPECTED_NO_IDENTIFIER_T0_RECORDS:,} no-identifier "
        f"T0 records, observed {len(t0_no_identifier):,}."
    )


stage3f_t0_review = pd.read_parquet(
    STAGE3F_T0_REVIEW_PATH,
    columns=["t0_crosswalk_id"],
)

if len(stage3f_t0_review) != EXPECTED_STAGE3F_TARGETED_T0_RECORDS:
    raise AssertionError(
        f"Expected {EXPECTED_STAGE3F_TARGETED_T0_RECORDS:,} Stage 3F "
        f"targeted T0 records, observed {len(stage3f_t0_review):,}."
    )

overlap_with_stage3f = set(
    t0_no_identifier["t0_crosswalk_id"].astype(str)
).intersection(
    stage3f_t0_review["t0_crosswalk_id"].astype(str)
)

if overlap_with_stage3f:
    raise AssertionError(
        f"{len(overlap_with_stage3f):,} Stage 3G T0 records unexpectedly "
        "overlap the Stage 3F targeted T0 cohort."
    )


crosswalk_required_columns = [
    "crosswalk_id",
    "linkage_status",
    "t1_rcv_accession",
    "t1_rcv_version",
    "t1_variation_id",
    "t1_vcv_accession",
    "t1_vcv_version",
    "t1_target_genes_json",
    "t1_study_scope",
    "t1_condition_ids_json",
    "t1_condition_names_json",
    "t1_aggregate_classification_axis",
]

crosswalk_schema = set(
    pq.ParquetFile(CROSSWALK_PATH).schema_arrow.names
)

missing_crosswalk_columns = sorted(
    set(crosswalk_required_columns) - crosswalk_schema
)

if missing_crosswalk_columns:
    raise KeyError(
        "Stage 3B crosswalk is missing columns: "
        + ", ".join(missing_crosswalk_columns)
    )

crosswalk = pd.read_parquet(
    CROSSWALK_PATH,
    columns=crosswalk_required_columns,
)

t1_only = (
    crosswalk.loc[
        crosswalk["linkage_status"].eq(
            "T1_ONLY_UNLINKED_BY_EXACT_RCV"
        )
    ]
    .copy()
    .reset_index(drop=True)
)

if len(t1_only) != EXPECTED_T1_ONLY_RECORDS:
    raise AssertionError(
        f"Expected {EXPECTED_T1_ONLY_RECORDS:,} T1-only records, "
        f"observed {len(t1_only):,}."
    )

print()
print(f"No-identifier T0 records loaded: {len(t0_no_identifier):,}")
print(f"T1-only records loaded:          {len(t1_only):,}")


# -------------------------------------------------------------------------------------------------
# 6. NORMALIZE T0 AND T1 FIELDS
# -------------------------------------------------------------------------------------------------

print("Normalizing genes, conditions, and identifiers...")


def normalize_t0_row_fields(frame: pd.DataFrame) -> pd.DataFrame:
    """Add internal normalized fields to T0 records."""
    frame = frame.copy()

    frame["_genes"] = [
        normalized_json_tuple(
            value,
            "t0_genes_normalized_json",
            normalize_gene,
        )
        for value in frame["t0_genes_normalized_json"]
    ]

    frame["_condition_ids"] = [
        normalized_json_tuple(
            value,
            "t0_condition_ids_normalized_json",
            normalize_condition_id,
        )
        for value in frame["t0_condition_ids_normalized_json"]
    ]

    frame["_condition_names"] = [
        normalized_json_tuple(
            value,
            "t0_condition_names_normalized_json",
            normalize_condition_name,
        )
        for value in frame["t0_condition_names_normalized_json"]
    ]

    frame["_variation_id"] = [
        clean_identifier(value)
        for value in frame["t0_variation_id"]
    ]

    frame["_vcv_accession"] = [
        clean_identifier(value)
        for value in frame["t0_vcv_accession"]
    ]

    return frame


def normalize_t1_row_fields(frame: pd.DataFrame) -> pd.DataFrame:
    """Add internal normalized fields to T1-only records."""
    frame = frame.copy()

    frame["_genes"] = [
        normalized_json_tuple(
            value,
            "t1_target_genes_json",
            normalize_gene,
        )
        for value in frame["t1_target_genes_json"]
    ]

    frame["_condition_ids"] = [
        normalized_json_tuple(
            value,
            "t1_condition_ids_json",
            normalize_condition_id,
        )
        for value in frame["t1_condition_ids_json"]
    ]

    frame["_condition_names"] = [
        normalized_json_tuple(
            value,
            "t1_condition_names_json",
            normalize_condition_name,
        )
        for value in frame["t1_condition_names_json"]
    ]

    frame["_variation_id"] = [
        clean_identifier(value)
        for value in frame["t1_variation_id"]
    ]

    frame["_vcv_accession"] = [
        clean_identifier(value)
        for value in frame["t1_vcv_accession"]
    ]

    frame["_classification_axis"] = [
        canonical_axis(value)
        for value in frame["t1_aggregate_classification_axis"]
    ]

    return frame


t0_no_identifier = normalize_t0_row_fields(t0_no_identifier)
t1_only = normalize_t1_row_fields(t1_only)


if not t0_no_identifier["_genes"].map(len).eq(1).all():
    raise AssertionError(
        "A Stage 3G T0 record does not contain exactly one target gene."
    )

if not t1_only["_genes"].map(len).eq(1).all():
    raise AssertionError(
        "A T1-only record does not contain exactly one target gene."
    )


# -------------------------------------------------------------------------------------------------
# 7. BUILD SAME-GENE CONDITION INDEXES FOR T1-ONLY RECORDS
# -------------------------------------------------------------------------------------------------

gene_index = defaultdict(set)
condition_id_index = defaultdict(set)
condition_name_index = defaultdict(set)

for t1_index, row in t1_only.iterrows():
    gene = row["_genes"][0]

    gene_index[gene].add(t1_index)

    for condition_id in row["_condition_ids"]:
        condition_id_index[(gene, condition_id)].add(t1_index)

    for condition_name in row["_condition_names"]:
        condition_name_index[(gene, condition_name)].add(t1_index)


# -------------------------------------------------------------------------------------------------
# 8. PRECOMPUTE CANDIDATE INDEX SETS AND APPLY A MEMORY GUARD
# -------------------------------------------------------------------------------------------------

candidate_sets = {}
candidate_sources = {}

estimated_pair_rows = 0

for _, t0_row in t0_no_identifier.iterrows():
    t0_id = str(t0_row["t0_crosswalk_id"])
    gene = t0_row["_genes"][0]

    id_candidates = set()
    name_candidates = set()

    for condition_id in t0_row["_condition_ids"]:
        id_candidates.update(
            condition_id_index.get(
                (gene, condition_id),
                set(),
            )
        )

    for condition_name in t0_row["_condition_names"]:
        name_candidates.update(
            condition_name_index.get(
                (gene, condition_name),
                set(),
            )
        )

    all_candidates = id_candidates.union(name_candidates)

    candidate_sets[t0_id] = all_candidates
    candidate_sources[t0_id] = {
        "id_candidates": id_candidates,
        "name_candidates": name_candidates,
    }

    estimated_pair_rows += len(all_candidates)

print()
print(
    f"Estimated same-gene condition-only candidate pairs: "
    f"{estimated_pair_rows:,}"
)

if estimated_pair_rows > MAX_CONDITION_ONLY_PAIR_ROWS:
    raise RuntimeError(
        f"Stage 3G would generate {estimated_pair_rows:,} condition-only "
        f"candidate pairs, exceeding the frozen safety limit of "
        f"{MAX_CONDITION_ONLY_PAIR_ROWS:,}. A more restrictive, versioned "
        "candidate-generation rule is required before continuing."
    )


# -------------------------------------------------------------------------------------------------
# 9. MATERIALIZE CONDITION-ONLY CANDIDATE PAIRS
# -------------------------------------------------------------------------------------------------

print("Constructing condition-only candidate audit pairs...")

pair_records = []

for _, t0_row in t0_no_identifier.iterrows():
    t0_id = str(t0_row["t0_crosswalk_id"])
    candidate_indices = candidate_sets[t0_id]

    for t1_index in sorted(candidate_indices):
        t1_row = t1_only.iloc[t1_index]

        if t0_row["_genes"] != t1_row["_genes"]:
            raise AssertionError(
                "A same-gene condition candidate has gene discordance."
            )

        variation_id_same = (
            t0_row["_variation_id"] is not None
            and t0_row["_variation_id"] == t1_row["_variation_id"]
        )

        vcv_accession_same = (
            t0_row["_vcv_accession"] is not None
            and t0_row["_vcv_accession"] == t1_row["_vcv_accession"]
        )

        if variation_id_same or vcv_accession_same:
            raise AssertionError(
                "A Stage 3G pair unexpectedly has VariationID or VCV "
                "continuity and should have been detected during Stage 3D."
            )

        condition_id_overlap = sorted(
            set(t0_row["_condition_ids"]).intersection(
                t1_row["_condition_ids"]
            )
        )

        condition_name_overlap = sorted(
            set(t0_row["_condition_names"]).intersection(
                t1_row["_condition_names"]
            )
        )

        id_overlap_count = len(condition_id_overlap)
        name_overlap_count = len(condition_name_overlap)

        evidence_category = pair_evidence_category(
            condition_id_overlap_count=id_overlap_count,
            condition_name_overlap_count=name_overlap_count,
        )

        if evidence_category == "ERROR_NO_CONDITION_EVIDENCE":
            raise AssertionError(
                "A condition-only candidate was created without exact "
                "condition-ID or condition-name overlap."
            )

        id_relation = set_relation(
            t0_row["_condition_ids"],
            t1_row["_condition_ids"],
        )

        name_relation = set_relation(
            t0_row["_condition_names"],
            t1_row["_condition_names"],
        )

        pair_records.append(
            {
                "condition_only_pair_id": (
                    f"{t0_row['t0_rcv_accession']}"
                    f"__TO__{t1_row['t1_rcv_accession']}"
                ),

                "t0_crosswalk_id": t0_row["t0_crosswalk_id"],
                "t1_crosswalk_id": t1_row["crosswalk_id"],

                "t0_rcv_accession": t0_row["t0_rcv_accession"],
                "t1_rcv_accession": t1_row["t1_rcv_accession"],

                "t0_rcv_version": t0_row["t0_rcv_version"],
                "t1_rcv_version": t1_row["t1_rcv_version"],

                "t0_variation_id": t0_row["t0_variation_id"],
                "t1_variation_id": t1_row["t1_variation_id"],
                "variation_id_same": False,

                "t0_vcv_accession": t0_row["t0_vcv_accession"],
                "t1_vcv_accession": t1_row["t1_vcv_accession"],
                "vcv_accession_same": False,

                "target_gene": t0_row["_genes"][0],
                "same_gene_exact": True,

                "t0_condition_ids_normalized_json": compact_json(
                    t0_row["_condition_ids"]
                ),
                "t1_condition_ids_normalized_json": compact_json(
                    t1_row["_condition_ids"]
                ),
                "condition_id_relation": id_relation,
                "condition_id_overlap_count": int(id_overlap_count),
                "condition_id_overlap_json": compact_json(
                    condition_id_overlap
                ),

                "t0_condition_names_normalized_json": compact_json(
                    t0_row["_condition_names"]
                ),
                "t1_condition_names_normalized_json": compact_json(
                    t1_row["_condition_names"]
                ),
                "condition_name_relation": name_relation,
                "condition_name_overlap_count": int(name_overlap_count),
                "condition_name_overlap_json": compact_json(
                    condition_name_overlap
                ),

                "condition_only_evidence_category": evidence_category,
                "condition_only_review_priority": pair_review_priority(
                    id_relation=id_relation,
                    name_relation=name_relation,
                    evidence_category=evidence_category,
                ),

                "t0_study_scope": t0_row["t0_study_scope"],
                "t1_study_scope": t1_row["t1_study_scope"],
                "t1_classification_axis": t1_row[
                    "_classification_axis"
                ],

                "linkage_basis": (
                    "SAME_GENE_AND_EXACT_CONDITION_METADATA_ONLY"
                ),
                "condition_equivalence_inferred": False,
                "stage3g_accepted_nonexact_link": False,
                "future_instability_outcome_created": False,
                "unresolved_pair_labeled_stable": False,
            }
        )


pair_columns = [
    "condition_only_pair_id",
    "t0_crosswalk_id",
    "t1_crosswalk_id",
    "t0_rcv_accession",
    "t1_rcv_accession",
    "t0_rcv_version",
    "t1_rcv_version",
    "t0_variation_id",
    "t1_variation_id",
    "variation_id_same",
    "t0_vcv_accession",
    "t1_vcv_accession",
    "vcv_accession_same",
    "target_gene",
    "same_gene_exact",
    "t0_condition_ids_normalized_json",
    "t1_condition_ids_normalized_json",
    "condition_id_relation",
    "condition_id_overlap_count",
    "condition_id_overlap_json",
    "t0_condition_names_normalized_json",
    "t1_condition_names_normalized_json",
    "condition_name_relation",
    "condition_name_overlap_count",
    "condition_name_overlap_json",
    "condition_only_evidence_category",
    "condition_only_review_priority",
    "t0_study_scope",
    "t1_study_scope",
    "t1_classification_axis",
    "linkage_basis",
    "condition_equivalence_inferred",
    "stage3g_accepted_nonexact_link",
    "future_instability_outcome_created",
    "unresolved_pair_labeled_stable",
]

pair_audit = pd.DataFrame(
    pair_records,
    columns=pair_columns,
)

if len(pair_audit) != estimated_pair_rows:
    raise AssertionError(
        "Materialized condition-only pair count does not match the estimate."
    )

if not pair_audit.empty:
    pair_audit = (
        pair_audit.sort_values(
            by=[
                "t0_rcv_accession",
                "condition_only_review_priority",
                "t1_rcv_accession",
            ],
            kind="mergesort",
        )
        .reset_index(drop=True)
    )

    if pair_audit["condition_only_pair_id"].duplicated().any():
        raise AssertionError(
            "Duplicate Stage 3G condition-only pair identifiers were generated."
        )

    if pair_audit["stage3g_accepted_nonexact_link"].any():
        raise AssertionError(
            "A Stage 3G condition-only candidate was incorrectly accepted."
        )


# -------------------------------------------------------------------------------------------------
# 10. CREATE ONE AUDIT SUMMARY ROW FOR EACH OF THE 622 T0 RECORDS
# -------------------------------------------------------------------------------------------------

pairs_by_t0 = {
    str(key): group.copy()
    for key, group in pair_audit.groupby(
        "t0_crosswalk_id",
        sort=False,
    )
}

t0_summary_records = []

for _, t0_row in t0_no_identifier.iterrows():
    t0_id = str(t0_row["t0_crosswalk_id"])
    group = pairs_by_t0.get(t0_id)

    if group is None or group.empty:
        candidate_count = 0
        id_candidate_count = 0
        name_candidate_count = 0
        both_evidence_count = 0
        exact_condition_set_count = 0
        unique_candidate_t1_rcv = None
        priority_counts = {}
        evidence_counts = {}
        axis_counts = {}

    else:
        candidate_count = int(len(group))

        id_candidate_count = int(
            group["condition_id_overlap_count"].gt(0).sum()
        )

        name_candidate_count = int(
            group["condition_name_overlap_count"].gt(0).sum()
        )

        both_evidence_count = int(
            group["condition_only_evidence_category"].eq(
                "CONDITION_ID_AND_NAME_OVERLAP"
            ).sum()
        )

        exact_condition_set_count = int(
            (
                group["condition_id_relation"].eq(
                    "EXACT_SET_MATCH"
                )
                & group["condition_name_relation"].eq(
                    "EXACT_SET_MATCH"
                )
            ).sum()
        )

        unique_candidate_t1_rcv = (
            str(group["t1_rcv_accession"].iloc[0])
            if candidate_count == 1
            else None
        )

        priority_counts = {
            str(key): int(value)
            for key, value in (
                group["condition_only_review_priority"]
                .value_counts()
                .sort_index()
                .items()
            )
        }

        evidence_counts = {
            str(key): int(value)
            for key, value in (
                group["condition_only_evidence_category"]
                .value_counts()
                .sort_index()
                .items()
            )
        }

        axis_counts = {
            str(key): int(value)
            for key, value in (
                group["t1_classification_axis"]
                .value_counts(dropna=False)
                .sort_index()
                .items()
            )
        }

    audit_category = t0_audit_category(
        candidate_count=candidate_count,
        exact_condition_set_candidate_count=exact_condition_set_count,
        condition_ids=t0_row["_condition_ids"],
        condition_names=t0_row["_condition_names"],
    )

    t0_summary_records.append(
        {
            "t0_crosswalk_id": t0_row["t0_crosswalk_id"],
            "t0_rcv_accession": t0_row["t0_rcv_accession"],
            "t0_rcv_version": t0_row["t0_rcv_version"],
            "t0_variation_id": t0_row["t0_variation_id"],
            "t0_vcv_accession": t0_row["t0_vcv_accession"],
            "t0_vcv_version": t0_row["t0_vcv_version"],

            "target_gene": t0_row["_genes"][0],
            "t0_study_scope": t0_row["t0_study_scope"],

            "t0_condition_ids_normalized_json": compact_json(
                t0_row["_condition_ids"]
            ),
            "t0_condition_names_normalized_json": compact_json(
                t0_row["_condition_names"]
            ),

            "same_gene_t1_only_pool_size": int(
                len(gene_index.get(t0_row["_genes"][0], set()))
            ),

            "condition_only_candidate_count": int(
                candidate_count
            ),
            "condition_id_candidate_count": int(
                id_candidate_count
            ),
            "condition_name_candidate_count": int(
                name_candidate_count
            ),
            "id_and_name_evidence_candidate_count": int(
                both_evidence_count
            ),
            "exact_condition_set_candidate_count": int(
                exact_condition_set_count
            ),

            "unique_condition_only_candidate_t1_rcv": (
                unique_candidate_t1_rcv
            ),

            "candidate_priority_counts_json": json.dumps(
                priority_counts,
                sort_keys=True,
                separators=(",", ":"),
            ),
            "candidate_evidence_counts_json": json.dumps(
                evidence_counts,
                sort_keys=True,
                separators=(",", ":"),
            ),
            "candidate_t1_axis_counts_json": json.dumps(
                axis_counts,
                sort_keys=True,
                separators=(",", ":"),
            ),

            "stage3g_t0_audit_category": audit_category,
            "stage3g_source_history_review_priority": (
                source_history_priority(audit_category)
            ),

            "stage3g_resolution_status": (
                "UNRESOLVED_REQUIRES_FINAL_LINKAGE_OR_CENSORING_DECISION"
            ),

            "condition_only_candidate_is_accepted_link": False,
            "future_instability_outcome_created": False,
            "unresolved_record_labeled_stable": False,
        }
    )


t0_audit = pd.DataFrame(t0_summary_records)

t0_audit = (
    t0_audit.sort_values(
        by=[
            "stage3g_source_history_review_priority",
            "target_gene",
            "t0_rcv_accession",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


# -------------------------------------------------------------------------------------------------
# 11. CREATE THE FINAL REVIEW/CENSORING QUEUE
# -------------------------------------------------------------------------------------------------

review_queue_columns = [
    "t0_crosswalk_id",
    "t0_rcv_accession",
    "t0_rcv_version",
    "t0_variation_id",
    "t0_vcv_accession",
    "t0_vcv_version",
    "target_gene",
    "t0_study_scope",
    "t0_condition_ids_normalized_json",
    "t0_condition_names_normalized_json",
    "condition_only_candidate_count",
    "condition_id_candidate_count",
    "condition_name_candidate_count",
    "exact_condition_set_candidate_count",
    "unique_condition_only_candidate_t1_rcv",
    "stage3g_t0_audit_category",
    "stage3g_source_history_review_priority",
    "stage3g_resolution_status",
]

review_queue = t0_audit[review_queue_columns].copy()

review_queue["recommended_next_action"] = (
    "FINAL_FROZEN_T0_T1_LINKAGE_DISPOSITION_REVIEW"
)

review_queue["permitted_final_dispositions_json"] = json.dumps(
    [
        "UNRESOLVED_UNMATCHED",
        "CENSORED_NO_DEFENSIBLE_T1_LINK",
        "REQUIRES_MANUAL_SOURCE_HISTORY_REVIEW",
    ],
    separators=(",", ":"),
)

review_queue["prohibited_final_disposition"] = (
    "DO_NOT_LABEL_STABLE_FROM_ABSENCE_OF_A_MATCH"
)

review_queue["stage3g_accepted_additional_link"] = False
review_queue["future_instability_outcome_created"] = False


# -------------------------------------------------------------------------------------------------
# 12. STRUCTURAL AND SCIENTIFIC VALIDATION
# -------------------------------------------------------------------------------------------------

if len(t0_audit) != EXPECTED_NO_IDENTIFIER_T0_RECORDS:
    raise AssertionError(
        "Stage 3G T0 audit does not contain exactly 622 records."
    )

if t0_audit["t0_crosswalk_id"].duplicated().any():
    raise AssertionError(
        "Duplicate T0 identifiers were found in the Stage 3G audit."
    )

if len(review_queue) != EXPECTED_NO_IDENTIFIER_T0_RECORDS:
    raise AssertionError(
        "Stage 3G review queue does not contain exactly 622 records."
    )

if t0_audit["condition_only_candidate_is_accepted_link"].any():
    raise AssertionError(
        "A condition-only candidate was incorrectly accepted."
    )

if t0_audit["future_instability_outcome_created"].any():
    raise AssertionError(
        "A future-instability outcome was unexpectedly created."
    )

if t0_audit["unresolved_record_labeled_stable"].any():
    raise AssertionError(
        "An unresolved T0 record was unexpectedly labeled stable."
    )

t0_with_candidates = int(
    t0_audit["condition_only_candidate_count"].gt(0).sum()
)

t0_with_unique_candidate = int(
    t0_audit["condition_only_candidate_count"].eq(1).sum()
)

t0_with_multiple_candidates = int(
    t0_audit["condition_only_candidate_count"].gt(1).sum()
)

t0_without_candidates = int(
    t0_audit["condition_only_candidate_count"].eq(0).sum()
)

if (
    t0_with_unique_candidate
    + t0_with_multiple_candidates
    + t0_without_candidates
    != EXPECTED_NO_IDENTIFIER_T0_RECORDS
):
    raise AssertionError(
        "Stage 3G T0 candidate accounting failed."
    )


# -------------------------------------------------------------------------------------------------
# 13. WRITE AND READ BACK OUTPUT ARTIFACTS
# -------------------------------------------------------------------------------------------------

pair_audit.to_parquet(
    PAIR_AUDIT_PATH,
    engine="pyarrow",
    compression="zstd",
    index=False,
)

t0_audit.to_parquet(
    T0_AUDIT_PATH,
    engine="pyarrow",
    compression="zstd",
    index=False,
)

review_queue.to_parquet(
    REVIEW_QUEUE_PATH,
    engine="pyarrow",
    compression="zstd",
    index=False,
)

pair_audit_sha256 = sha256_file(PAIR_AUDIT_PATH)
t0_audit_sha256 = sha256_file(T0_AUDIT_PATH)
review_queue_sha256 = sha256_file(REVIEW_QUEUE_PATH)

pair_metadata = pq.ParquetFile(PAIR_AUDIT_PATH).metadata
t0_metadata = pq.ParquetFile(T0_AUDIT_PATH).metadata
queue_metadata = pq.ParquetFile(REVIEW_QUEUE_PATH).metadata

pair_readback = pd.read_parquet(PAIR_AUDIT_PATH)
t0_readback = pd.read_parquet(T0_AUDIT_PATH)
queue_readback = pd.read_parquet(REVIEW_QUEUE_PATH)

if len(pair_readback) != len(pair_audit):
    raise AssertionError(
        "Stage 3G pair-audit readback row-count mismatch."
    )

if len(t0_readback) != len(t0_audit):
    raise AssertionError(
        "Stage 3G T0-audit readback row-count mismatch."
    )

if len(queue_readback) != len(review_queue):
    raise AssertionError(
        "Stage 3G review-queue readback row-count mismatch."
    )

if list(pair_readback.columns) != list(pair_audit.columns):
    raise AssertionError(
        "Stage 3G pair-audit readback schema mismatch."
    )

if list(t0_readback.columns) != list(t0_audit.columns):
    raise AssertionError(
        "Stage 3G T0-audit readback schema mismatch."
    )

if list(queue_readback.columns) != list(review_queue.columns):
    raise AssertionError(
        "Stage 3G review-queue readback schema mismatch."
    )


# -------------------------------------------------------------------------------------------------
# 14. CREATE REPORT
# -------------------------------------------------------------------------------------------------

report = {
    "report_name": (
        "Stage 3G No-Stable-Identifier Condition-Only Candidate Audit Report"
    ),
    "report_version": "1.0.0",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "study_stage": "Stage 3G",
    "scientific_unit": "RCV-level variant-condition aggregate",

    "input_counts": {
        "no_identifier_t0_records": int(len(t0_no_identifier)),
        "t1_only_records": int(len(t1_only)),
        "stage3f_targeted_t0_records_excluded": int(
            len(stage3f_t0_review)
        ),
    },

    "candidate_generation_rule": {
        "required_gene_rule": "Exact target-gene equality",
        "condition_rule": (
            "At least one exact normalized condition ID or exact "
            "normalized condition name"
        ),
        "variation_id_continuity_required": False,
        "vcv_continuity_required": False,
        "condition_synonym_inference_used": False,
        "ontology_equivalence_inference_used": False,
        "current_live_clinvar_used": False,
    },

    "pair_results": {
        "condition_only_candidate_pairs": int(len(pair_audit)),
        "evidence_category_counts": value_counts_dict(
            pair_audit["condition_only_evidence_category"]
        ),
        "review_priority_counts": value_counts_dict(
            pair_audit["condition_only_review_priority"]
        ),
        "t1_axis_counts": value_counts_dict(
            pair_audit["t1_classification_axis"]
        ),
    },

    "t0_results": {
        "t0_records_with_condition_only_candidate": int(
            t0_with_candidates
        ),
        "t0_records_with_unique_condition_only_candidate": int(
            t0_with_unique_candidate
        ),
        "t0_records_with_multiple_condition_only_candidates": int(
            t0_with_multiple_candidates
        ),
        "t0_records_without_condition_only_candidate": int(
            t0_without_candidates
        ),
        "t0_audit_category_counts": value_counts_dict(
            t0_audit["stage3g_t0_audit_category"]
        ),
        "source_history_priority_counts": value_counts_dict(
            t0_audit[
                "stage3g_source_history_review_priority"
            ]
        ),
    },

    "validation": {
        "expected_t0_records": int(
            EXPECTED_NO_IDENTIFIER_T0_RECORDS
        ),
        "observed_t0_records": int(len(t0_audit)),
        "duplicate_t0_ids": int(
            t0_audit["t0_crosswalk_id"].duplicated().sum()
        ),
        "accepted_condition_only_links": int(
            t0_audit[
                "condition_only_candidate_is_accepted_link"
            ].sum()
        ),
        "stage3f_overlap_count": int(
            len(overlap_with_stage3f)
        ),
        "readback_passed": True,
        "critical_failures": 0,
    },

    "scientific_boundaries": {
        "condition_only_candidates_are_accepted_links": False,
        "additional_nonexact_links_accepted": False,
        "condition_equivalence_inferred": False,
        "live_post_t1_clinvar_information_used": False,
        "official_replacements_confirmed": False,
        "withdrawals_confirmed": False,
        "merges_confirmed": False,
        "splits_confirmed": False,
        "classification_changes_calculated": False,
        "future_instability_outcomes_created": False,
        "unresolved_records_labeled_stable": False,
        "ges_model_fitted_or_tuned": False,
    },

    "next_authorized_step": (
        "Assemble the complete final T0–T1 linkage artifact using exact "
        "RCV matches, the 170 accepted conservative non-exact one-to-one "
        "links, and explicit unresolved or censored dispositions for all "
        "remaining T0 records. Freeze the linkage and exclusion policy "
        "before constructing future-instability outcomes."
    ),
}

with REPORT_PATH.open("w", encoding="utf-8") as handle:
    json.dump(
        report,
        handle,
        indent=2,
        sort_keys=True,
        ensure_ascii=False,
    )

report_sha256 = sha256_file(REPORT_PATH)


# -------------------------------------------------------------------------------------------------
# 15. CREATE MANIFEST
# -------------------------------------------------------------------------------------------------

manifest = {
    "manifest_name": (
        "Stage 3G No-Stable-Identifier Condition-Only Audit Manifest"
    ),
    "manifest_version": "1.0.0",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "study_stage": "Stage 3G",
    "audit_status": (
        "NO_STABLE_IDENTIFIER_CONDITION_ONLY_AUDIT_COMPLETE"
    ),

    "inputs": {
        "stage3b_crosswalk": {
            "path": str(CROSSWALK_PATH),
            "sha256": observed_crosswalk_sha256,
        },
        "stage3e_t0_status": {
            "path": str(T0_STAGE3E_STATUS_PATH),
            "sha256": observed_t0_status_sha256,
        },
        "stage3f_t0_review": {
            "path": str(STAGE3F_T0_REVIEW_PATH),
            "sha256": observed_stage3f_t0_review_sha256,
        },
        "stage3f_manifest": {
            "path": str(STAGE3F_MANIFEST_PATH),
            "sha256": observed_stage3f_manifest_sha256,
        },
    },

    "outputs": {
        "condition_only_pair_audit": {
            "path": str(PAIR_AUDIT_PATH),
            "sha256": pair_audit_sha256,
            "rows": int(pair_metadata.num_rows),
            "columns": int(pair_metadata.num_columns),
            "row_groups": int(pair_metadata.num_row_groups),
        },
        "t0_audit_summary": {
            "path": str(T0_AUDIT_PATH),
            "sha256": t0_audit_sha256,
            "rows": int(t0_metadata.num_rows),
            "columns": int(t0_metadata.num_columns),
            "row_groups": int(t0_metadata.num_row_groups),
        },
        "final_review_queue": {
            "path": str(REVIEW_QUEUE_PATH),
            "sha256": review_queue_sha256,
            "rows": int(queue_metadata.num_rows),
            "columns": int(queue_metadata.num_columns),
            "row_groups": int(queue_metadata.num_row_groups),
        },
        "report": {
            "path": str(REPORT_PATH),
            "sha256": report_sha256,
        },
    },

    "no_identifier_t0_record_count": int(len(t0_audit)),
    "condition_only_candidate_pair_count": int(
        len(pair_audit)
    ),
    "additional_links_accepted": 0,

    "validation_decision": "PASS",
    "scientific_boundaries": report["scientific_boundaries"],
    "next_authorized_step": report["next_authorized_step"],
}

with MANIFEST_PATH.open("w", encoding="utf-8") as handle:
    json.dump(
        manifest,
        handle,
        indent=2,
        sort_keys=True,
        ensure_ascii=False,
    )

manifest_sha256 = sha256_file(MANIFEST_PATH)


# -------------------------------------------------------------------------------------------------
# 16. PRINT RESULTS
# -------------------------------------------------------------------------------------------------

print()
print("=" * 126)
print("STAGE 3G NO-STABLE-IDENTIFIER AUDIT SUMMARY")
print("=" * 126)

print(f"T0 records audited:                           {len(t0_audit):,}")
print(f"T1-only comparison pool:                      {len(t1_only):,}")
print(f"Same-gene condition-only candidate pairs:     {len(pair_audit):,}")

print()
print(f"T0 records with ≥1 condition-only candidate:  {t0_with_candidates:,}")
print(f"  With exactly one candidate:                 {t0_with_unique_candidate:,}")
print(f"  With multiple candidates:                   {t0_with_multiple_candidates:,}")
print(f"T0 records with no condition-only candidate:  {t0_without_candidates:,}")

print()
print("Condition-only pair evidence:")
print(
    pair_audit[
        "condition_only_evidence_category"
    ]
    .value_counts(dropna=False)
    .to_string()
)

print()
print("T0 audit categories:")
print(
    t0_audit[
        "stage3g_t0_audit_category"
    ]
    .value_counts(dropna=False)
    .to_string()
)

print()
print("Final review priorities:")
print(
    t0_audit[
        "stage3g_source_history_review_priority"
    ]
    .value_counts(dropna=False)
    .to_string()
)

print()
print(f"Pair audit:          {PAIR_AUDIT_PATH}")
print(f"Pair SHA-256:        {pair_audit_sha256}")
print(f"T0 audit:            {T0_AUDIT_PATH}")
print(f"T0 audit SHA-256:    {t0_audit_sha256}")
print(f"Review queue:        {REVIEW_QUEUE_PATH}")
print(f"Queue SHA-256:       {review_queue_sha256}")
print(f"Report:              {REPORT_PATH}")
print(f"Report SHA-256:      {report_sha256}")
print(f"Manifest:            {MANIFEST_PATH}")
print(f"Manifest SHA-256:    {manifest_sha256}")

print()
print("PASS — Stage 3G no-stable-identifier audit completed.")
print(
    "IMPORTANT — Condition-only candidates were not accepted as variant links."
)
print(
    "IMPORTANT — No unresolved record was labeled stable and no temporal "
    "outcome was created."
)
print(
    "AUTHORIZED NEXT ACTION — Stage 3H assembly, validation, and freezing "
    "of the final T0–T1 linkage artifact and unresolved/censoring policy."
)
print("=" * 126)

STAGE 3G — NO-STABLE-IDENTIFIER CANDIDATE AUDIT
Stage 3B crosswalk SHA-256: b462304a4bb31db301e2dac3aefbf685e12f1fea179a38ca04b8500b64bb4acc
Stage 3E T0 status SHA-256:c1e40c92389e2dc200cd1508c49fb7aa12c3a9665fc2e6aab56ed2127accb65e
Stage 3F T0 review SHA-256:09f6e4d24744781127e59d7ee03ddf3d5d02b0c4ea5517e52a184ad46a623d2a
Stage 3F manifest SHA-256: 004292519a8a33d261c9a57acdbd6573fc074338a2bb311bbd464b8fd6fa47b1

No-identifier T0 records loaded: 622
T1-only records loaded:          30,507
Normalizing genes, conditions, and identifiers...

Estimated same-gene condition-only candidate pairs: 1,042,053
Constructing condition-only candidate audit pairs...

STAGE 3G NO-STABLE-IDENTIFIER AUDIT SUMMARY
T0 records audited:                           622
T1-only comparison pool:                      30,507
Same-gene condition-only candidate pairs:     1,042,053

T0 records with ≥1 condition-only candidate:  576
  With exactly one candidate:                 1
  With multiple candidates:         

In [19]:
# =================================================================================================
# STAGE 3F — TARGETED AUDIT OF DEFERRED STRONG AND TIER 3 CANDIDATE RELATIONSHIPS
# =================================================================================================
# Purpose:
#   Audit, without automatically accepting, the following unresolved candidate relationships:
#
#   1. Deferred strong candidate pairs:
#        - possible one-to-many splits,
#        - possible many-to-one merges,
#        - possible complex many-to-many relationships.
#
#   2. Tier 3 candidate pairs:
#        - exact VariationID continuity,
#        - exact VCV accession continuity,
#        - exact target-gene continuity,
#        - but no exact condition-ID or normalized condition-name overlap.
#
# This cell:
#   - creates a bipartite graph of targeted T0–T1 candidate relationships;
#   - assigns deterministic graph-component identifiers;
#   - characterizes one-to-one, one-to-many, many-to-one, and many-to-many components;
#   - audits condition metadata missingness;
#   - calculates descriptive condition-name lexical similarity;
#   - identifies condition-namespace continuity;
#   - identifies targeted candidates touching a Stage 3E accepted link;
#   - creates pair-, component-, and T0-level review artifacts.
#
# IMPORTANT:
#   Lexical similarity is an audit-prioritization signal only.
#   It is not evidence of condition equivalence and cannot accept a link.
#
# This cell DOES NOT:
#   - accept additional non-exact links,
#   - revoke or silently alter Stage 3E accepted links,
#   - confirm official RCV replacements, merges, splits, or withdrawals,
#   - compare aggregate clinical classifications,
#   - construct temporal-instability outcomes,
#   - label unresolved records as stable,
#   - fit or tune GES.
# =================================================================================================

from pathlib import Path
from datetime import datetime, timezone
from collections import Counter, defaultdict
import hashlib
import json
import re

import pandas as pd
import pyarrow.parquet as pq


print("=" * 126)
print("STAGE 3F — DEFERRED-COMPLEX AND TIER 3 CONDITION-CHANGE AUDIT")
print("=" * 126)


# -------------------------------------------------------------------------------------------------
# 1. PATHS AND EXPECTED STAGE 3E CHECKSUMS
# -------------------------------------------------------------------------------------------------

STUDY_ROOT = Path("/content/drive/MyDrive/GES_RAG_Temporal_Study")

STAGE3_CONFIG_DIR = STUDY_ROOT / "configs" / "stage3_crosswalk"
STAGE3_OUTPUT_DIR = STUDY_ROOT / "data_processed" / "stage3_crosswalk"

STAGE3E_ADJUDICATION_PATH = (
    STAGE3_OUTPUT_DIR
    / "stage3_nonexact_candidate_adjudication_v1.parquet"
)

STAGE3E_T0_STATUS_PATH = (
    STAGE3_OUTPUT_DIR
    / "stage3_t0_post_strong_candidate_adjudication_v1.parquet"
)

STAGE3E_MANIFEST_PATH = (
    STAGE3_CONFIG_DIR
    / "stage3_strong_candidate_adjudication_manifest_v1.json"
)

PAIR_AUDIT_PATH = (
    STAGE3_OUTPUT_DIR
    / "stage3_targeted_complex_and_tier3_pair_audit_v1.parquet"
)

COMPONENT_AUDIT_PATH = (
    STAGE3_OUTPUT_DIR
    / "stage3_targeted_candidate_graph_components_v1.parquet"
)

T0_REVIEW_SUMMARY_PATH = (
    STAGE3_OUTPUT_DIR
    / "stage3_targeted_candidate_t0_review_summary_v1.parquet"
)

REPORT_PATH = (
    STAGE3_CONFIG_DIR
    / "stage3_targeted_complex_and_tier3_audit_report_v1.json"
)

MANIFEST_PATH = (
    STAGE3_CONFIG_DIR
    / "stage3_targeted_complex_and_tier3_audit_manifest_v1.json"
)


EXPECTED_ADJUDICATION_SHA256 = (
    "f647a8fe66db8a695da2136f3ec59ad0"
    "03425f11a2767a3198eefa612dbcf4ef"
)

EXPECTED_T0_STATUS_SHA256 = (
    "c1e40c92389e2dc200cd1508c49fb7a"
    "a12c3a9665fc2e6aab56ed2127accb65e"
)

EXPECTED_STAGE3E_MANIFEST_SHA256 = (
    "b5ea53ba3fe6cafdb2abaa25ac582d683"
    "cfb35dd04d88dcfdaf03802edec4304"
)

EXPECTED_ALL_CANDIDATE_PAIRS = 1078
EXPECTED_ACCEPTED_STAGE3E_PAIRS = 170
EXPECTED_DEFERRED_STRONG_PAIRS = 78
EXPECTED_TIER3_PAIRS = 830
EXPECTED_TARGETED_PAIRS = 908
EXPECTED_T0_STATUS_ROWS = 1246


DEFERRED_STRONG_DECISIONS = {
    "DEFERRED_POSSIBLE_ONE_TO_MANY_SPLIT",
    "DEFERRED_POSSIBLE_MANY_TO_ONE_MERGE",
    "DEFERRED_POSSIBLE_COMPLEX_MANY_TO_MANY_CHANGE",
    "DEFERRED_STRONG_CANDIDATE_REQUIRES_REVIEW",
}

TIER3_NAME = (
    "TIER_3_VARIANT_CONTINUITY_CONDITION_CHANGE_CANDIDATE"
)


# -------------------------------------------------------------------------------------------------
# 2. GENERAL UTILITY FUNCTIONS
# -------------------------------------------------------------------------------------------------

def sha256_file(path: Path, chunk_size: int = 8 * 1024 * 1024) -> str:
    """Calculate SHA-256 without loading the entire file into memory."""
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def value_counts_dict(series: pd.Series) -> dict:
    """Convert pandas value counts into a JSON-safe dictionary."""
    counts = series.value_counts(dropna=False)

    result = {}

    for key, value in counts.items():
        safe_key = "<MISSING>" if pd.isna(key) else str(key)
        result[safe_key] = int(value)

    return result


def safe_json_list(value, field_name: str) -> list:
    """Parse a JSON-list field safely and enforce its top-level type."""
    if value is None:
        return []

    if isinstance(value, list):
        parsed = value

    elif isinstance(value, tuple):
        parsed = list(value)

    else:
        try:
            if pd.isna(value):
                return []
        except (TypeError, ValueError):
            pass

        text = str(value).strip()

        if text == "":
            return []

        try:
            parsed = json.loads(text)
        except json.JSONDecodeError as exc:
            raise ValueError(
                f"Malformed JSON in {field_name}: {text[:200]}"
            ) from exc

    if not isinstance(parsed, list):
        raise TypeError(
            f"{field_name} must contain a JSON list, "
            f"not {type(parsed).__name__}."
        )

    cleaned = []

    for item in parsed:
        if item is None:
            continue

        text = str(item).strip()

        if text:
            cleaned.append(text)

    return cleaned


def json_compact(values) -> str:
    """Serialize a collection deterministically as compact JSON."""
    return json.dumps(
        list(values),
        ensure_ascii=False,
        separators=(",", ":"),
    )


def json_count_dict(values) -> str:
    """Serialize value counts deterministically."""
    counts = Counter(values)

    return json.dumps(
        {
            str(key): int(value)
            for key, value in sorted(counts.items())
        },
        ensure_ascii=False,
        separators=(",", ":"),
    )


def set_relation(left_values, right_values) -> str:
    """Describe the relationship between two sets."""
    left = set(left_values)
    right = set(right_values)

    if not left and not right:
        return "BOTH_EMPTY"

    if not left:
        return "T0_EMPTY"

    if not right:
        return "T1_EMPTY"

    if left == right:
        return "EXACT_SET_MATCH"

    if left.intersection(right):
        return "PARTIAL_SET_OVERLAP"

    return "DISJOINT_SETS"


# -------------------------------------------------------------------------------------------------
# 3. CONDITION-METADATA AUDIT FUNCTIONS
# -------------------------------------------------------------------------------------------------

def condition_id_namespace(condition_id: str):
    """
    Extract a descriptive condition-identifier namespace.

    Examples:
        MEDGEN:C123      -> MEDGEN
        OMIM:123456      -> OMIM
        MONDO:0000001    -> MONDO
    """
    text = str(condition_id).strip().upper()

    if ":" in text:
        return text.split(":", 1)[0]

    return None


def condition_names_to_tokens(names) -> set:
    """
    Convert normalized condition names into descriptive lexical tokens.

    No synonym inference, ontology mapping, stemming, or clinical equivalence
    inference is performed.
    """
    tokens = set()

    for name in names:
        for token in re.findall(r"[a-z0-9]+", str(name).casefold()):
            if len(token) >= 2:
                tokens.add(token)

    return tokens


def relaxed_condition_name_keys(names) -> set:
    """
    Remove punctuation and spaces from normalized names.

    This may detect formatting-only differences, but the result remains
    an audit signal and cannot establish condition equivalence.
    """
    keys = set()

    for name in names:
        key = re.sub(
            r"[^a-z0-9]+",
            "",
            str(name).casefold(),
        )

        if key:
            keys.add(key)

    return keys


def jaccard_similarity(left_values, right_values):
    """Calculate Jaccard similarity, preserving empty-set meaning."""
    left = set(left_values)
    right = set(right_values)

    if not left and not right:
        return None

    union = left.union(right)

    if not union:
        return None

    return len(left.intersection(right)) / len(union)


def pair_metadata_audit(row) -> dict:
    """Calculate descriptive condition-metadata audit fields for one pair."""

    t0_ids = safe_json_list(
        row["t0_condition_ids_normalized_json"],
        "t0_condition_ids_normalized_json",
    )

    t1_ids = safe_json_list(
        row["t1_condition_ids_normalized_json"],
        "t1_condition_ids_normalized_json",
    )

    t0_names = safe_json_list(
        row["t0_condition_names_normalized_json"],
        "t0_condition_names_normalized_json",
    )

    t1_names = safe_json_list(
        row["t1_condition_names_normalized_json"],
        "t1_condition_names_normalized_json",
    )

    t0_namespaces = sorted(
        {
            namespace
            for namespace in (
                condition_id_namespace(value)
                for value in t0_ids
            )
            if namespace
        }
    )

    t1_namespaces = sorted(
        {
            namespace
            for namespace in (
                condition_id_namespace(value)
                for value in t1_ids
            )
            if namespace
        }
    )

    namespace_overlap = sorted(
        set(t0_namespaces).intersection(t1_namespaces)
    )

    t0_tokens = condition_names_to_tokens(t0_names)
    t1_tokens = condition_names_to_tokens(t1_names)

    token_overlap = sorted(
        t0_tokens.intersection(t1_tokens)
    )

    token_jaccard = jaccard_similarity(
        t0_tokens,
        t1_tokens,
    )

    t0_relaxed_keys = relaxed_condition_name_keys(t0_names)
    t1_relaxed_keys = relaxed_condition_name_keys(t1_names)

    relaxed_name_overlap = sorted(
        t0_relaxed_keys.intersection(t1_relaxed_keys)
    )

    if not t0_ids and not t1_ids:
        missingness_category = "CONDITION_IDS_EMPTY_BOTH_RELEASES"

    elif not t0_ids:
        missingness_category = "CONDITION_IDS_EMPTY_AT_T0"

    elif not t1_ids:
        missingness_category = "CONDITION_IDS_EMPTY_AT_T1"

    elif not t0_names and not t1_names:
        missingness_category = "CONDITION_NAMES_EMPTY_BOTH_RELEASES"

    elif not t0_names:
        missingness_category = "CONDITION_NAMES_EMPTY_AT_T0"

    elif not t1_names:
        missingness_category = "CONDITION_NAMES_EMPTY_AT_T1"

    else:
        missingness_category = "CONDITION_METADATA_PRESENT_BOTH_RELEASES"

    return {
        "t0_condition_id_count": int(len(t0_ids)),
        "t1_condition_id_count": int(len(t1_ids)),
        "t0_condition_name_count": int(len(t0_names)),
        "t1_condition_name_count": int(len(t1_names)),

        "condition_metadata_missingness_category": (
            missingness_category
        ),

        "t0_condition_namespaces_json": json_compact(
            t0_namespaces
        ),
        "t1_condition_namespaces_json": json_compact(
            t1_namespaces
        ),
        "condition_namespace_relation": set_relation(
            t0_namespaces,
            t1_namespaces,
        ),
        "condition_namespace_overlap_count": int(
            len(namespace_overlap)
        ),
        "condition_namespace_overlap_json": json_compact(
            namespace_overlap
        ),

        "condition_name_token_overlap_count": int(
            len(token_overlap)
        ),
        "condition_name_token_overlap_json": json_compact(
            token_overlap
        ),
        "condition_name_token_jaccard": (
            round(float(token_jaccard), 6)
            if token_jaccard is not None
            else None
        ),

        "relaxed_condition_name_overlap_count": int(
            len(relaxed_name_overlap)
        ),
        "relaxed_condition_name_overlap_json": json_compact(
            relaxed_name_overlap
        ),
        "relaxed_condition_name_match_signal": bool(
            len(relaxed_name_overlap) > 0
        ),
    }


# -------------------------------------------------------------------------------------------------
# 4. UNION-FIND FOR BIPARTITE GRAPH COMPONENTS
# -------------------------------------------------------------------------------------------------

class UnionFind:
    """Minimal deterministic union-find implementation."""

    def __init__(self):
        self.parent = {}
        self.rank = {}

    def add(self, item):
        if item not in self.parent:
            self.parent[item] = item
            self.rank[item] = 0

    def find(self, item):
        self.add(item)

        if self.parent[item] != item:
            self.parent[item] = self.find(self.parent[item])

        return self.parent[item]

    def union(self, left, right):
        left_root = self.find(left)
        right_root = self.find(right)

        if left_root == right_root:
            return

        left_rank = self.rank[left_root]
        right_rank = self.rank[right_root]

        if left_rank < right_rank:
            self.parent[left_root] = right_root

        elif left_rank > right_rank:
            self.parent[right_root] = left_root

        else:
            self.parent[right_root] = left_root
            self.rank[left_root] += 1


def component_topology(t0_count: int, t1_count: int) -> str:
    """Describe component topology without confirming biological history."""

    if t0_count == 1 and t1_count == 1:
        return "ONE_TO_ONE_TARGETED_COMPONENT"

    if t0_count == 1 and t1_count > 1:
        return "ONE_TO_MANY_POSSIBLE_SPLIT_COMPONENT"

    if t0_count > 1 and t1_count == 1:
        return "MANY_TO_ONE_POSSIBLE_MERGE_COMPONENT"

    return "MANY_TO_MANY_COMPLEX_COMPONENT"


def component_scope(scope_values) -> str:
    """Describe whether a graph component contains strong, Tier 3, or mixed edges."""
    scopes = set(scope_values)

    if scopes == {"DEFERRED_STRONG_CANDIDATE"}:
        return "DEFERRED_STRONG_ONLY"

    if scopes == {"TIER3_CONDITION_CHANGE_CANDIDATE"}:
        return "TIER3_ONLY"

    return "MIXED_DEFERRED_STRONG_AND_TIER3"


# -------------------------------------------------------------------------------------------------
# 5. PAIR-LEVEL REVIEW PRIORITY
# -------------------------------------------------------------------------------------------------

def targeted_pair_review_priority(row) -> str:
    """Assign an audit priority without accepting a link."""

    scope = str(row["stage3f_target_scope"])
    topology = str(row["stage3f_component_topology"])

    if scope == "DEFERRED_STRONG_CANDIDATE":

        if topology == "ONE_TO_MANY_POSSIBLE_SPLIT_COMPONENT":
            return "PRIORITY_1_DEFERRED_STRONG_POSSIBLE_SPLIT"

        if topology == "MANY_TO_ONE_POSSIBLE_MERGE_COMPONENT":
            return "PRIORITY_1_DEFERRED_STRONG_POSSIBLE_MERGE"

        if topology == "MANY_TO_MANY_COMPLEX_COMPONENT":
            return "PRIORITY_1_DEFERRED_STRONG_COMPLEX"

        return "PRIORITY_1_DEFERRED_STRONG_ONE_TO_ONE_REVIEW"

    missingness = str(
        row["condition_metadata_missingness_category"]
    )

    relaxed_match = bool(
        row["relaxed_condition_name_match_signal"]
    )

    token_jaccard = row["condition_name_token_jaccard"]

    if missingness != "CONDITION_METADATA_PRESENT_BOTH_RELEASES":
        return "PRIORITY_2_TIER3_MISSINGNESS_INFLUENCED"

    if relaxed_match:
        return "PRIORITY_2_TIER3_FORMATTING_OR_RENAMING_SIGNAL"

    if (
        token_jaccard is not None
        and not pd.isna(token_jaccard)
        and float(token_jaccard) >= 0.80
    ):
        return "PRIORITY_2_TIER3_HIGH_LEXICAL_SIMILARITY"

    if topology == "ONE_TO_ONE_TARGETED_COMPONENT":
        return "PRIORITY_3_TIER3_ONE_TO_ONE_CONDITION_REASSOCIATION"

    if topology == "ONE_TO_MANY_POSSIBLE_SPLIT_COMPONENT":
        return "PRIORITY_3_TIER3_ONE_TO_MANY_CONDITION_CHANGE"

    if topology == "MANY_TO_ONE_POSSIBLE_MERGE_COMPONENT":
        return "PRIORITY_3_TIER3_MANY_TO_ONE_CONDITION_CHANGE"

    return "PRIORITY_3_TIER3_COMPLEX_CONDITION_CHANGE"


# -------------------------------------------------------------------------------------------------
# 6. VERIFY STAGE 3E INPUTS
# -------------------------------------------------------------------------------------------------

required_files = [
    STAGE3E_ADJUDICATION_PATH,
    STAGE3E_T0_STATUS_PATH,
    STAGE3E_MANIFEST_PATH,
]

for required_file in required_files:
    if not required_file.exists():
        raise FileNotFoundError(
            f"Required Stage 3E artifact does not exist: {required_file}"
        )

observed_adjudication_sha256 = sha256_file(
    STAGE3E_ADJUDICATION_PATH
)

observed_t0_status_sha256 = sha256_file(
    STAGE3E_T0_STATUS_PATH
)

observed_stage3e_manifest_sha256 = sha256_file(
    STAGE3E_MANIFEST_PATH
)

if observed_adjudication_sha256 != EXPECTED_ADJUDICATION_SHA256:
    raise RuntimeError(
        "Stage 3E adjudication checksum mismatch.\n"
        f"Expected: {EXPECTED_ADJUDICATION_SHA256}\n"
        f"Observed: {observed_adjudication_sha256}"
    )

if observed_t0_status_sha256 != EXPECTED_T0_STATUS_SHA256:
    raise RuntimeError(
        "Stage 3E T0-status checksum mismatch.\n"
        f"Expected: {EXPECTED_T0_STATUS_SHA256}\n"
        f"Observed: {observed_t0_status_sha256}"
    )

if (
    observed_stage3e_manifest_sha256
    != EXPECTED_STAGE3E_MANIFEST_SHA256
):
    raise RuntimeError(
        "Stage 3E manifest checksum mismatch.\n"
        f"Expected: {EXPECTED_STAGE3E_MANIFEST_SHA256}\n"
        f"Observed: {observed_stage3e_manifest_sha256}"
    )

with STAGE3E_MANIFEST_PATH.open("r", encoding="utf-8") as handle:
    stage3e_manifest = json.load(handle)

if (
    stage3e_manifest.get("adjudication_status")
    != "CONSERVATIVE_STRONG_CANDIDATE_ADJUDICATION_COMPLETE"
):
    raise RuntimeError(
        "Stage 3E does not authorize the targeted Stage 3F audit."
    )

print(f"Stage 3E adjudication SHA-256: {observed_adjudication_sha256}")
print(f"Stage 3E T0 status SHA-256:    {observed_t0_status_sha256}")
print(f"Stage 3E manifest SHA-256:     {observed_stage3e_manifest_sha256}")


# -------------------------------------------------------------------------------------------------
# 7. LOAD AND VALIDATE STAGE 3E DATA
# -------------------------------------------------------------------------------------------------

candidate_pairs = pd.read_parquet(
    STAGE3E_ADJUDICATION_PATH
)

t0_status = pd.read_parquet(
    STAGE3E_T0_STATUS_PATH
)

if len(candidate_pairs) != EXPECTED_ALL_CANDIDATE_PAIRS:
    raise AssertionError(
        f"Expected {EXPECTED_ALL_CANDIDATE_PAIRS:,} candidate pairs, "
        f"observed {len(candidate_pairs):,}."
    )

if len(t0_status) != EXPECTED_T0_STATUS_ROWS:
    raise AssertionError(
        f"Expected {EXPECTED_T0_STATUS_ROWS:,} T0 status rows, "
        f"observed {len(t0_status):,}."
    )

accepted_mask = (
    candidate_pairs[
        "stage3e_accepted_nonexact_link"
    ]
    .fillna(False)
    .astype(bool)
)

if int(accepted_mask.sum()) != EXPECTED_ACCEPTED_STAGE3E_PAIRS:
    raise AssertionError(
        f"Expected {EXPECTED_ACCEPTED_STAGE3E_PAIRS:,} accepted Stage 3E pairs, "
        f"observed {int(accepted_mask.sum()):,}."
    )

deferred_strong_mask = (
    candidate_pairs[
        "stage3e_decision"
    ].isin(DEFERRED_STRONG_DECISIONS)
)

tier3_mask = (
    candidate_pairs["candidate_tier"].eq(TIER3_NAME)
)

if int(deferred_strong_mask.sum()) != EXPECTED_DEFERRED_STRONG_PAIRS:
    raise AssertionError(
        f"Expected {EXPECTED_DEFERRED_STRONG_PAIRS:,} deferred strong pairs, "
        f"observed {int(deferred_strong_mask.sum()):,}."
    )

if int(tier3_mask.sum()) != EXPECTED_TIER3_PAIRS:
    raise AssertionError(
        f"Expected {EXPECTED_TIER3_PAIRS:,} Tier 3 pairs, "
        f"observed {int(tier3_mask.sum()):,}."
    )

if (deferred_strong_mask & tier3_mask).any():
    raise AssertionError(
        "A pair was classified as both deferred strong and Tier 3."
    )

targeted = (
    candidate_pairs.loc[
        deferred_strong_mask | tier3_mask
    ]
    .copy()
    .reset_index(drop=True)
)

if len(targeted) != EXPECTED_TARGETED_PAIRS:
    raise AssertionError(
        f"Expected {EXPECTED_TARGETED_PAIRS:,} targeted pairs, "
        f"observed {len(targeted):,}."
    )

if targeted[
    "stage3e_accepted_nonexact_link"
].fillna(False).any():
    raise AssertionError(
        "An accepted Stage 3E pair entered the unresolved Stage 3F audit."
    )

targeted["stage3f_target_scope"] = [
    (
        "DEFERRED_STRONG_CANDIDATE"
        if decision in DEFERRED_STRONG_DECISIONS
        else "TIER3_CONDITION_CHANGE_CANDIDATE"
    )
    for decision in targeted["stage3e_decision"]
]

print()
print(f"All candidate pairs loaded:       {len(candidate_pairs):,}")
print(f"Deferred strong pairs targeted:   {int(deferred_strong_mask.sum()):,}")
print(f"Tier 3 pairs targeted:            {int(tier3_mask.sum()):,}")
print(f"Total Stage 3F targeted pairs:    {len(targeted):,}")


# -------------------------------------------------------------------------------------------------
# 8. ADD STAGE 3E ACCEPTED-LINK CONTEXT
# -------------------------------------------------------------------------------------------------

accepted_context = candidate_pairs.loc[
    accepted_mask,
    [
        "t0_crosswalk_id",
        "t1_crosswalk_id",
        "t0_rcv_accession",
        "t1_rcv_accession",
        "candidate_pair_id",
    ],
].copy()

accepted_t0_to_t1_rcv = dict(
    zip(
        accepted_context["t0_crosswalk_id"],
        accepted_context["t1_rcv_accession"],
    )
)

accepted_t0_to_pair = dict(
    zip(
        accepted_context["t0_crosswalk_id"],
        accepted_context["candidate_pair_id"],
    )
)

accepted_t1_ids = set(
    accepted_context["t1_crosswalk_id"]
)

targeted["t0_has_stage3e_accepted_link"] = [
    value in accepted_t0_to_t1_rcv
    for value in targeted["t0_crosswalk_id"]
]

targeted["t0_stage3e_accepted_t1_rcv"] = [
    accepted_t0_to_t1_rcv.get(value)
    for value in targeted["t0_crosswalk_id"]
]

targeted["t0_stage3e_accepted_pair_id"] = [
    accepted_t0_to_pair.get(value)
    for value in targeted["t0_crosswalk_id"]
]

targeted["t1_used_by_stage3e_accepted_link"] = [
    value in accepted_t1_ids
    for value in targeted["t1_crosswalk_id"]
]

targeted["accepted_link_context_present"] = (
    targeted["t0_has_stage3e_accepted_link"]
    | targeted["t1_used_by_stage3e_accepted_link"]
)


# -------------------------------------------------------------------------------------------------
# 9. ADD CONDITION-METADATA AUDIT FIELDS
# -------------------------------------------------------------------------------------------------

print("Auditing condition metadata and descriptive lexical similarity...")

metadata_results = [
    pair_metadata_audit(row)
    for _, row in targeted.iterrows()
]

metadata_frame = pd.DataFrame(metadata_results)

targeted = pd.concat(
    [
        targeted.reset_index(drop=True),
        metadata_frame.reset_index(drop=True),
    ],
    axis=1,
)


# -------------------------------------------------------------------------------------------------
# 10. CONSTRUCT TARGETED BIPARTITE GRAPH COMPONENTS
# -------------------------------------------------------------------------------------------------

print("Constructing deterministic T0–T1 candidate graph components...")

union_find = UnionFind()

for _, row in targeted.iterrows():
    t0_node = f"T0::{row['t0_crosswalk_id']}"
    t1_node = f"T1::{row['t1_crosswalk_id']}"

    union_find.union(t0_node, t1_node)

nodes_by_root = defaultdict(list)

for node in sorted(union_find.parent):
    root = union_find.find(node)
    nodes_by_root[root].append(node)

component_id_by_node = {}

for nodes in nodes_by_root.values():
    sorted_nodes = sorted(nodes)

    component_hash = hashlib.sha256(
        "||".join(sorted_nodes).encode("utf-8")
    ).hexdigest()[:16]

    component_id = f"S3F_COMP_{component_hash}"

    for node in sorted_nodes:
        component_id_by_node[node] = component_id

targeted["stage3f_component_id"] = [
    component_id_by_node[
        f"T0::{t0_crosswalk_id}"
    ]
    for t0_crosswalk_id in targeted["t0_crosswalk_id"]
]

if targeted["stage3f_component_id"].isna().any():
    raise AssertionError(
        "At least one targeted pair is missing a graph-component ID."
    )


# -------------------------------------------------------------------------------------------------
# 11. CREATE COMPONENT-LEVEL AUDIT
# -------------------------------------------------------------------------------------------------

component_records = []

for component_id, group in targeted.groupby(
    "stage3f_component_id",
    sort=True,
):
    t0_ids = sorted(
        set(group["t0_crosswalk_id"].astype(str))
    )

    t1_ids = sorted(
        set(group["t1_crosswalk_id"].astype(str))
    )

    t0_rcvs = sorted(
        set(group["t0_rcv_accession"].astype(str))
    )

    t1_rcvs = sorted(
        set(group["t1_rcv_accession"].astype(str))
    )

    topology = component_topology(
        t0_count=len(t0_ids),
        t1_count=len(t1_ids),
    )

    scope = component_scope(
        group["stage3f_target_scope"].tolist()
    )

    lexical_values = (
        group["condition_name_token_jaccard"]
        .dropna()
        .astype(float)
        .tolist()
    )

    component_records.append(
        {
            "stage3f_component_id": component_id,
            "stage3f_component_scope": scope,
            "stage3f_component_topology": topology,

            "component_t0_count": int(len(t0_ids)),
            "component_t1_count": int(len(t1_ids)),
            "component_edge_count": int(len(group)),

            "component_t0_crosswalk_ids_json": json_compact(
                t0_ids
            ),
            "component_t1_crosswalk_ids_json": json_compact(
                t1_ids
            ),
            "component_t0_rcv_accessions_json": json_compact(
                t0_rcvs
            ),
            "component_t1_rcv_accessions_json": json_compact(
                t1_rcvs
            ),

            "component_pair_scope_counts_json": json_count_dict(
                group["stage3f_target_scope"].tolist()
            ),
            "component_candidate_tier_counts_json": json_count_dict(
                group["candidate_tier"].tolist()
            ),
            "component_stage3e_decision_counts_json": json_count_dict(
                group["stage3e_decision"].tolist()
            ),
            "component_condition_evidence_counts_json": json_count_dict(
                group["condition_evidence_category"].tolist()
            ),
            "component_t1_axis_counts_json": json_count_dict(
                group["t1_classification_axis"].tolist()
            ),

            "component_contains_stage3e_accepted_link_context": bool(
                group["accepted_link_context_present"].any()
            ),
            "component_t0_nodes_with_accepted_link_context": int(
                group.loc[
                    group["t0_has_stage3e_accepted_link"],
                    "t0_crosswalk_id",
                ].nunique()
            ),
            "component_t1_nodes_used_by_accepted_link": int(
                group.loc[
                    group["t1_used_by_stage3e_accepted_link"],
                    "t1_crosswalk_id",
                ].nunique()
            ),

            "component_pairs_with_missing_condition_metadata": int(
                group[
                    "condition_metadata_missingness_category"
                ]
                .ne("CONDITION_METADATA_PRESENT_BOTH_RELEASES")
                .sum()
            ),
            "component_relaxed_name_match_pairs": int(
                group["relaxed_condition_name_match_signal"].sum()
            ),
            "component_max_condition_name_token_jaccard": (
                round(max(lexical_values), 6)
                if lexical_values
                else None
            ),

            "stage3f_component_resolution_status": (
                "REQUIRES_SOURCE_HISTORY_OR_PRESPECIFIED_CENSORING_REVIEW"
            ),
            "stage3f_component_accepted_link_count": 0,
            "future_instability_outcome_created": False,
            "unresolved_component_labeled_stable": False,
        }
    )

component_audit = pd.DataFrame(component_records)

if component_audit["stage3f_component_id"].duplicated().any():
    raise AssertionError(
        "Duplicate Stage 3F component identifiers were generated."
    )

component_metadata_columns = [
    "stage3f_component_id",
    "stage3f_component_scope",
    "stage3f_component_topology",
    "component_t0_count",
    "component_t1_count",
    "component_edge_count",
]

targeted = targeted.merge(
    component_audit[component_metadata_columns],
    on="stage3f_component_id",
    how="left",
    validate="many_to_one",
)


# -------------------------------------------------------------------------------------------------
# 12. ASSIGN PAIR-LEVEL REVIEW PRIORITIES
# -------------------------------------------------------------------------------------------------

targeted["stage3f_review_priority"] = [
    targeted_pair_review_priority(row)
    for _, row in targeted.iterrows()
]

targeted["stage3f_resolution_status"] = (
    "UNRESOLVED_REQUIRES_SOURCE_HISTORY_OR_CENSORING_DECISION"
)

targeted["stage3f_accepted_nonexact_link"] = False
targeted["future_instability_outcome_created"] = False
targeted["unresolved_pair_labeled_stable"] = False


# -------------------------------------------------------------------------------------------------
# 13. CREATE T0-LEVEL TARGETED REVIEW SUMMARY
# -------------------------------------------------------------------------------------------------

target_t0_ids = sorted(
    set(targeted["t0_crosswalk_id"].astype(str))
)

t0_target_base = t0_status.loc[
    t0_status[
        "t0_crosswalk_id"
    ].astype(str).isin(target_t0_ids)
].copy()

if len(t0_target_base) != len(target_t0_ids):
    raise AssertionError(
        "The T0 status artifact does not contain every targeted T0 identifier."
    )

pair_groups_by_t0 = {
    str(key): group.copy()
    for key, group in targeted.groupby(
        "t0_crosswalk_id",
        sort=False,
    )
}

t0_review_records = []

for _, t0_row in t0_target_base.iterrows():
    t0_id = str(t0_row["t0_crosswalk_id"])
    group = pair_groups_by_t0[t0_id]

    deferred_count = int(
        group["stage3f_target_scope"]
        .eq("DEFERRED_STRONG_CANDIDATE")
        .sum()
    )

    tier3_count = int(
        group["stage3f_target_scope"]
        .eq("TIER3_CONDITION_CHANGE_CANDIDATE")
        .sum()
    )

    original_status = str(
        t0_row["stage3e_post_adjudication_status"]
    )

    if original_status == "ACCEPTED_NONEXACT_ONE_TO_ONE_CONTINUITY":
        stage3f_t0_category = (
            "ACCEPTED_STAGE3E_LINK_WITH_ADDITIONAL_TIER3_OR_DEFERRED_CONTEXT"
        )

    elif deferred_count > 0:
        stage3f_t0_category = (
            "DEFERRED_COMPLEX_STRONG_RELATIONSHIP_REVIEW"
        )

    elif tier3_count > 0:
        stage3f_t0_category = (
            "TIER3_CONDITION_ASSOCIATION_CHANGE_REVIEW"
        )

    else:
        stage3f_t0_category = (
            "UNEXPECTED_TARGETED_RECORD_CATEGORY"
        )

    lexical_values = (
        group["condition_name_token_jaccard"]
        .dropna()
        .astype(float)
        .tolist()
    )

    t0_review_records.append(
        {
            "t0_crosswalk_id": t0_row["t0_crosswalk_id"],
            "t0_rcv_accession": t0_row["t0_rcv_accession"],
            "t0_variation_id": t0_row["t0_variation_id"],
            "t0_vcv_accession": t0_row["t0_vcv_accession"],
            "t0_genes_normalized_json": (
                t0_row["t0_genes_normalized_json"]
            ),
            "t0_study_scope": t0_row["t0_study_scope"],

            "stage3e_post_adjudication_status": original_status,
            "stage3e_accepted_nonexact_link": bool(
                t0_row["stage3e_accepted_nonexact_link"]
            ),
            "accepted_stage3e_t1_rcv_accession": (
                t0_row.get(
                    "accepted_stage3e_t1_rcv_accession"
                )
            ),

            "stage3f_targeted_candidate_pair_count": int(
                len(group)
            ),
            "stage3f_deferred_strong_pair_count": deferred_count,
            "stage3f_tier3_pair_count": tier3_count,

            "stage3f_component_count": int(
                group["stage3f_component_id"].nunique()
            ),
            "stage3f_component_ids_json": json_compact(
                sorted(
                    set(group["stage3f_component_id"])
                )
            ),
            "stage3f_component_topology_counts_json": json_count_dict(
                group["stage3f_component_topology"].tolist()
            ),

            "stage3f_candidate_t1_rcvs_json": json_compact(
                sorted(
                    set(group["t1_rcv_accession"].astype(str))
                )
            ),
            "stage3f_review_priority_counts_json": json_count_dict(
                group["stage3f_review_priority"].tolist()
            ),

            "stage3f_pairs_with_missing_condition_metadata": int(
                group[
                    "condition_metadata_missingness_category"
                ]
                .ne("CONDITION_METADATA_PRESENT_BOTH_RELEASES")
                .sum()
            ),
            "stage3f_pairs_with_relaxed_name_signal": int(
                group["relaxed_condition_name_match_signal"].sum()
            ),
            "stage3f_max_condition_name_token_jaccard": (
                round(max(lexical_values), 6)
                if lexical_values
                else None
            ),

            "stage3f_t0_review_category": stage3f_t0_category,
            "stage3f_resolution_status": (
                "REQUIRES_SOURCE_HISTORY_OR_FINAL_CENSORING_DECISION"
            ),

            "stage3f_accepted_additional_link": False,
            "future_instability_outcome_created": False,
            "unresolved_record_labeled_stable": False,
        }
    )

t0_review_summary = pd.DataFrame(t0_review_records)

if t0_review_summary["t0_crosswalk_id"].duplicated().any():
    raise AssertionError(
        "Duplicate T0 identifiers were created in the Stage 3F summary."
    )

if t0_review_summary["stage3f_accepted_additional_link"].any():
    raise AssertionError(
        "Stage 3F unexpectedly accepted an additional linkage."
    )


# -------------------------------------------------------------------------------------------------
# 14. SELECT AND ORDER PAIR-AUDIT COLUMNS
# -------------------------------------------------------------------------------------------------

pair_audit_columns = [
    "candidate_pair_id",
    "t0_crosswalk_id",
    "t1_crosswalk_id",

    "t0_rcv_accession",
    "t1_rcv_accession",
    "t0_rcv_version",
    "t1_rcv_version",

    "t0_variation_id",
    "t1_variation_id",
    "variation_id_same",

    "t0_vcv_accession",
    "t1_vcv_accession",
    "vcv_accession_same",

    "t0_genes_normalized_json",
    "t1_genes_normalized_json",
    "gene_relation",

    "t0_study_scope",
    "t1_study_scope",
    "study_scope_relation",

    "t0_condition_ids_normalized_json",
    "t1_condition_ids_normalized_json",
    "condition_id_relation",
    "condition_id_overlap_count",
    "condition_id_overlap_json",

    "t0_condition_names_normalized_json",
    "t1_condition_names_normalized_json",
    "condition_name_relation",
    "condition_name_overlap_count",
    "condition_name_overlap_json",

    "condition_evidence_category",
    "condition_metadata_missingness_category",

    "t0_condition_id_count",
    "t1_condition_id_count",
    "t0_condition_name_count",
    "t1_condition_name_count",

    "t0_condition_namespaces_json",
    "t1_condition_namespaces_json",
    "condition_namespace_relation",
    "condition_namespace_overlap_count",
    "condition_namespace_overlap_json",

    "condition_name_token_overlap_count",
    "condition_name_token_overlap_json",
    "condition_name_token_jaccard",

    "relaxed_condition_name_overlap_count",
    "relaxed_condition_name_overlap_json",
    "relaxed_condition_name_match_signal",

    "t1_classification_axis",

    "candidate_tier",
    "candidate_topology",
    "stage3e_decision",

    "t0_has_stage3e_accepted_link",
    "t0_stage3e_accepted_t1_rcv",
    "t0_stage3e_accepted_pair_id",
    "t1_used_by_stage3e_accepted_link",
    "accepted_link_context_present",

    "stage3f_target_scope",
    "stage3f_component_id",
    "stage3f_component_scope",
    "stage3f_component_topology",
    "component_t0_count",
    "component_t1_count",
    "component_edge_count",

    "stage3f_review_priority",
    "stage3f_resolution_status",
    "stage3f_accepted_nonexact_link",

    "future_instability_outcome_created",
    "unresolved_pair_labeled_stable",
]

missing_pair_columns = [
    column
    for column in pair_audit_columns
    if column not in targeted.columns
]

if missing_pair_columns:
    raise KeyError(
        "Missing Stage 3F pair-audit columns: "
        + ", ".join(missing_pair_columns)
    )

pair_audit = targeted[pair_audit_columns].copy()

pair_audit = (
    pair_audit.sort_values(
        by=[
            "stage3f_component_id",
            "stage3f_target_scope",
            "t0_rcv_accession",
            "t1_rcv_accession",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)

component_audit = (
    component_audit.sort_values(
        by=[
            "stage3f_component_scope",
            "stage3f_component_topology",
            "stage3f_component_id",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)

t0_review_summary = (
    t0_review_summary.sort_values(
        by=[
            "stage3f_t0_review_category",
            "t0_rcv_accession",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


# -------------------------------------------------------------------------------------------------
# 15. FINAL STRUCTURAL VALIDATION
# -------------------------------------------------------------------------------------------------

if len(pair_audit) != EXPECTED_TARGETED_PAIRS:
    raise AssertionError(
        "Stage 3F pair-audit row count changed unexpectedly."
    )

if pair_audit["candidate_pair_id"].duplicated().any():
    raise AssertionError(
        "Duplicate pair identifiers were found in the Stage 3F pair audit."
    )

if pair_audit["stage3f_component_id"].isna().any():
    raise AssertionError(
        "Missing graph-component IDs were found in the pair audit."
    )

component_edge_total = int(
    component_audit["component_edge_count"].sum()
)

if component_edge_total != EXPECTED_TARGETED_PAIRS:
    raise AssertionError(
        "Component edge accounting does not equal the targeted pair count."
    )

if pair_audit["stage3f_accepted_nonexact_link"].any():
    raise AssertionError(
        "A Stage 3F targeted pair was incorrectly accepted."
    )

if pair_audit["future_instability_outcome_created"].any():
    raise AssertionError(
        "A temporal outcome was unexpectedly created."
    )

if pair_audit["unresolved_pair_labeled_stable"].any():
    raise AssertionError(
        "An unresolved pair was unexpectedly labeled stable."
    )


# -------------------------------------------------------------------------------------------------
# 16. WRITE AND READ BACK OUTPUT ARTIFACTS
# -------------------------------------------------------------------------------------------------

pair_audit.to_parquet(
    PAIR_AUDIT_PATH,
    engine="pyarrow",
    compression="zstd",
    index=False,
)

component_audit.to_parquet(
    COMPONENT_AUDIT_PATH,
    engine="pyarrow",
    compression="zstd",
    index=False,
)

t0_review_summary.to_parquet(
    T0_REVIEW_SUMMARY_PATH,
    engine="pyarrow",
    compression="zstd",
    index=False,
)

pair_audit_sha256 = sha256_file(PAIR_AUDIT_PATH)
component_audit_sha256 = sha256_file(COMPONENT_AUDIT_PATH)
t0_review_summary_sha256 = sha256_file(
    T0_REVIEW_SUMMARY_PATH
)

pair_metadata = pq.ParquetFile(
    PAIR_AUDIT_PATH
).metadata

component_metadata = pq.ParquetFile(
    COMPONENT_AUDIT_PATH
).metadata

t0_review_metadata = pq.ParquetFile(
    T0_REVIEW_SUMMARY_PATH
).metadata

pair_readback = pd.read_parquet(PAIR_AUDIT_PATH)
component_readback = pd.read_parquet(
    COMPONENT_AUDIT_PATH
)
t0_review_readback = pd.read_parquet(
    T0_REVIEW_SUMMARY_PATH
)

if len(pair_readback) != len(pair_audit):
    raise AssertionError(
        "Pair-audit readback row-count mismatch."
    )

if len(component_readback) != len(component_audit):
    raise AssertionError(
        "Component-audit readback row-count mismatch."
    )

if len(t0_review_readback) != len(t0_review_summary):
    raise AssertionError(
        "T0-review readback row-count mismatch."
    )

if list(pair_readback.columns) != list(pair_audit.columns):
    raise AssertionError(
        "Pair-audit readback schema mismatch."
    )

if list(component_readback.columns) != list(
    component_audit.columns
):
    raise AssertionError(
        "Component-audit readback schema mismatch."
    )

if list(t0_review_readback.columns) != list(
    t0_review_summary.columns
):
    raise AssertionError(
        "T0-review readback schema mismatch."
    )


# -------------------------------------------------------------------------------------------------
# 17. CREATE STAGE 3F REPORT
# -------------------------------------------------------------------------------------------------

t0_records_with_stage3e_accepted_context = int(
    t0_review_summary[
        "stage3e_accepted_nonexact_link"
    ].sum()
)

pair_missingness_count = int(
    pair_audit[
        "condition_metadata_missingness_category"
    ]
    .ne("CONDITION_METADATA_PRESENT_BOTH_RELEASES")
    .sum()
)

pair_relaxed_name_signal_count = int(
    pair_audit[
        "relaxed_condition_name_match_signal"
    ].sum()
)

high_lexical_similarity_count = int(
    pair_audit[
        "condition_name_token_jaccard"
    ]
    .fillna(-1)
    .ge(0.80)
    .sum()
)

report = {
    "report_name": (
        "Stage 3F Deferred-Complex and Tier 3 Targeted Audit Report"
    ),
    "report_version": "1.0.0",
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "study_stage": "Stage 3F",
    "scientific_unit": (
        "RCV-level variant-condition aggregate"
    ),

    "inputs": {
        "all_stage3e_candidate_pairs": int(
            len(candidate_pairs)
        ),
        "accepted_stage3e_pairs_excluded_from_targeted_audit": int(
            accepted_mask.sum()
        ),
        "t0_status_records": int(len(t0_status)),
    },

    "targeted_pair_counts": {
        "targeted_pairs": int(len(pair_audit)),
        "deferred_strong_pairs": int(
            pair_audit[
                "stage3f_target_scope"
            ]
            .eq("DEFERRED_STRONG_CANDIDATE")
            .sum()
        ),
        "tier3_condition_change_pairs": int(
            pair_audit[
                "stage3f_target_scope"
            ]
            .eq("TIER3_CONDITION_CHANGE_CANDIDATE")
            .sum()
        ),
        "target_scope_counts": value_counts_dict(
            pair_audit["stage3f_target_scope"]
        ),
        "review_priority_counts": value_counts_dict(
            pair_audit["stage3f_review_priority"]
        ),
    },

    "graph_component_results": {
        "component_count": int(len(component_audit)),
        "component_scope_counts": value_counts_dict(
            component_audit["stage3f_component_scope"]
        ),
        "component_topology_counts": value_counts_dict(
            component_audit["stage3f_component_topology"]
        ),
        "components_with_stage3e_accepted_link_context": int(
            component_audit[
                "component_contains_stage3e_accepted_link_context"
            ].sum()
        ),
        "component_edge_accounting": int(
            component_audit["component_edge_count"].sum()
        ),
    },

    "condition_metadata_results": {
        "missingness_category_counts": value_counts_dict(
            pair_audit[
                "condition_metadata_missingness_category"
            ]
        ),
        "pairs_with_condition_metadata_missingness": int(
            pair_missingness_count
        ),
        "namespace_relation_counts": value_counts_dict(
            pair_audit["condition_namespace_relation"]
        ),
        "pairs_with_relaxed_condition_name_signal": int(
            pair_relaxed_name_signal_count
        ),
        "pairs_with_token_jaccard_at_least_0_80": int(
            high_lexical_similarity_count
        ),
        "lexical_similarity_interpretation": (
            "Lexical results are descriptive review-prioritization "
            "signals only and do not establish condition equivalence."
        ),
    },

    "t0_review_results": {
        "unique_t0_records_in_targeted_pair_graph": int(
            len(t0_review_summary)
        ),
        "t0_review_category_counts": value_counts_dict(
            t0_review_summary[
                "stage3f_t0_review_category"
            ]
        ),
        "targeted_t0_records_with_existing_stage3e_accepted_link": int(
            t0_records_with_stage3e_accepted_context
        ),
    },

    "validation": {
        "expected_targeted_pairs": int(
            EXPECTED_TARGETED_PAIRS
        ),
        "observed_targeted_pairs": int(
            len(pair_audit)
        ),
        "component_edge_total": int(
            component_edge_total
        ),
        "duplicate_pair_ids": int(
            pair_audit[
                "candidate_pair_id"
            ].duplicated().sum()
        ),
        "duplicate_component_ids": int(
            component_audit[
                "stage3f_component_id"
            ].duplicated().sum()
        ),
        "additional_links_accepted": int(
            pair_audit[
                "stage3f_accepted_nonexact_link"
            ].sum()
        ),
        "readback_passed": True,
        "critical_failures": 0,
    },

    "scientific_boundaries": {
        "additional_nonexact_links_accepted": False,
        "stage3e_links_revoked_or_modified": False,
        "official_rcv_replacements_confirmed": False,
        "merges_confirmed": False,
        "splits_confirmed": False,
        "condition_equivalence_inferred_from_lexical_similarity": False,
        "classification_changes_calculated": False,
        "future_instability_outcomes_created": False,
        "unresolved_records_labeled_stable": False,
        "ges_model_fitted_or_tuned": False,
    },

    "next_authorized_step": (
        "Audit the remaining T0 records with no VariationID- or "
        "VCV-based T1 candidate and assign prespecified unresolved, "
        "unmatched, or censored categories. No temporal outcome may "
        "be created until the complete linkage artifact and exclusion "
        "policy are frozen."
    ),
}

with REPORT_PATH.open("w", encoding="utf-8") as handle:
    json.dump(
        report,
        handle,
        indent=2,
        sort_keys=True,
        ensure_ascii=False,
    )

report_sha256 = sha256_file(REPORT_PATH)


# -------------------------------------------------------------------------------------------------
# 18. CREATE STAGE 3F MANIFEST
# -------------------------------------------------------------------------------------------------

manifest = {
    "manifest_name": (
        "Stage 3F Deferred-Complex and Tier 3 Targeted Audit Manifest"
    ),
    "manifest_version": "1.0.0",
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "study_stage": "Stage 3F",
    "audit_status": (
        "TARGETED_COMPLEX_AND_TIER3_AUDIT_COMPLETE"
    ),

    "inputs": {
        "stage3e_adjudication": {
            "path": str(STAGE3E_ADJUDICATION_PATH),
            "sha256": observed_adjudication_sha256,
        },
        "stage3e_t0_status": {
            "path": str(STAGE3E_T0_STATUS_PATH),
            "sha256": observed_t0_status_sha256,
        },
        "stage3e_manifest": {
            "path": str(STAGE3E_MANIFEST_PATH),
            "sha256": observed_stage3e_manifest_sha256,
        },
    },

    "outputs": {
        "targeted_pair_audit": {
            "path": str(PAIR_AUDIT_PATH),
            "sha256": pair_audit_sha256,
            "rows": int(pair_metadata.num_rows),
            "columns": int(pair_metadata.num_columns),
            "row_groups": int(pair_metadata.num_row_groups),
        },
        "targeted_component_audit": {
            "path": str(COMPONENT_AUDIT_PATH),
            "sha256": component_audit_sha256,
            "rows": int(component_metadata.num_rows),
            "columns": int(component_metadata.num_columns),
            "row_groups": int(component_metadata.num_row_groups),
        },
        "targeted_t0_review_summary": {
            "path": str(T0_REVIEW_SUMMARY_PATH),
            "sha256": t0_review_summary_sha256,
            "rows": int(t0_review_metadata.num_rows),
            "columns": int(t0_review_metadata.num_columns),
            "row_groups": int(t0_review_metadata.num_row_groups),
        },
        "report": {
            "path": str(REPORT_PATH),
            "sha256": report_sha256,
        },
    },

    "targeted_pair_count": int(len(pair_audit)),
    "graph_component_count": int(len(component_audit)),
    "unique_targeted_t0_record_count": int(
        len(t0_review_summary)
    ),

    "validation_decision": "PASS",
    "scientific_boundaries": (
        report["scientific_boundaries"]
    ),
    "next_authorized_step": (
        report["next_authorized_step"]
    ),
}

with MANIFEST_PATH.open("w", encoding="utf-8") as handle:
    json.dump(
        manifest,
        handle,
        indent=2,
        sort_keys=True,
        ensure_ascii=False,
    )

manifest_sha256 = sha256_file(MANIFEST_PATH)


# -------------------------------------------------------------------------------------------------
# 19. PRINT RESULTS
# -------------------------------------------------------------------------------------------------

print()
print("=" * 126)
print("STAGE 3F TARGETED-AUDIT SUMMARY")
print("=" * 126)

print(f"Targeted candidate pairs audited:             {len(pair_audit):,}")
print(
    f"  Deferred strong pairs:                      "
    f"{pair_audit['stage3f_target_scope'].eq('DEFERRED_STRONG_CANDIDATE').sum():,}"
)
print(
    f"  Tier 3 condition-change pairs:              "
    f"{pair_audit['stage3f_target_scope'].eq('TIER3_CONDITION_CHANGE_CANDIDATE').sum():,}"
)

print()
print(f"Targeted graph components:                    {len(component_audit):,}")
print(f"Unique T0 records in targeted graph:          {len(t0_review_summary):,}")
print(
    f"T0 records with an existing Stage 3E link:    "
    f"{t0_records_with_stage3e_accepted_context:,}"
)

print()
print(f"Pairs influenced by condition missingness:    {pair_missingness_count:,}")
print(f"Pairs with relaxed-name match signal:         {pair_relaxed_name_signal_count:,}")
print(f"Pairs with token Jaccard ≥ 0.80:              {high_lexical_similarity_count:,}")

print()
print("Graph-component topology:")
print(
    component_audit[
        "stage3f_component_topology"
    ]
    .value_counts(dropna=False)
    .to_string()
)

print()
print("Graph-component scope:")
print(
    component_audit[
        "stage3f_component_scope"
    ]
    .value_counts(dropna=False)
    .to_string()
)

print()
print("Pair-level review priorities:")
print(
    pair_audit[
        "stage3f_review_priority"
    ]
    .value_counts(dropna=False)
    .to_string()
)

print()
print("T0-level targeted review categories:")
print(
    t0_review_summary[
        "stage3f_t0_review_category"
    ]
    .value_counts(dropna=False)
    .to_string()
)

print()
print(f"Pair audit:             {PAIR_AUDIT_PATH}")
print(f"Pair audit SHA-256:     {pair_audit_sha256}")
print(f"Component audit:        {COMPONENT_AUDIT_PATH}")
print(f"Component SHA-256:      {component_audit_sha256}")
print(f"T0 review summary:      {T0_REVIEW_SUMMARY_PATH}")
print(f"T0 review SHA-256:      {t0_review_summary_sha256}")
print(f"Report:                 {REPORT_PATH}")
print(f"Report SHA-256:         {report_sha256}")
print(f"Manifest:               {MANIFEST_PATH}")
print(f"Manifest SHA-256:       {manifest_sha256}")

print()
print("PASS — Stage 3F targeted complex and Tier 3 audit completed.")
print(
    "IMPORTANT — No additional non-exact link was accepted or rejected automatically."
)
print(
    "IMPORTANT — Lexical similarity was used only for review prioritization, "
    "not condition-equivalence inference."
)
print(
    "IMPORTANT — No unresolved record was labeled stable and no temporal "
    "outcome was created."
)
print(
    "AUTHORIZED NEXT ACTION — Stage 3G audit and categorization of records "
    "with no VariationID- or VCV-based T1 candidate."
)
print("=" * 126)

STAGE 3F — DEFERRED-COMPLEX AND TIER 3 CONDITION-CHANGE AUDIT
Stage 3E adjudication SHA-256: f647a8fe66db8a695da2136f3ec59ad003425f11a2767a3198eefa612dbcf4ef
Stage 3E T0 status SHA-256:    c1e40c92389e2dc200cd1508c49fb7aa12c3a9665fc2e6aab56ed2127accb65e
Stage 3E manifest SHA-256:     b5ea53ba3fe6cafdb2abaa25ac582d683cfb35dd04d88dcfdaf03802edec4304

All candidate pairs loaded:       1,078
Deferred strong pairs targeted:   78
Tier 3 pairs targeted:            830
Total Stage 3F targeted pairs:    908
Auditing condition metadata and descriptive lexical similarity...
Constructing deterministic T0–T1 candidate graph components...

STAGE 3F TARGETED-AUDIT SUMMARY
Targeted candidate pairs audited:             908
  Deferred strong pairs:                      78
  Tier 3 condition-change pairs:              830

Targeted graph components:                    534
Unique T0 records in targeted graph:          561
T0 records with an existing Stage 3E link:    107

Pairs influenced by condition mis

In [20]:
# =================================================================================================
# STAGE 3H — ASSEMBLE, VALIDATE, AND FREEZE THE FINAL T0–T1 LINKAGE ARTIFACT
# =================================================================================================
# Purpose:
#   Assemble one final linkage row for every frozen T0 RCV-level variant-condition record.
#
# Final accepted linkage:
#   1. Exact RCV accession linkage from Stage 3B.
#   2. Conservative one-to-one non-exact continuity links accepted in Stage 3E.
#
# Final unresolved/censored categories:
#   1. Non-unique or complex candidate relationships.
#   2. Variant continuity with unresolved condition-association continuity.
#   3. No stable VariationID/VCV-based T1 linkage.
#
# Scientific boundary:
#   Unresolved or censored records are NOT stable records.
#
# This cell:
#   - freezes the final linkage/censoring policy;
#   - assembles one row for each of the 71,659 T0 records;
#   - attaches the accepted T1 record where a defensible link exists;
#   - retains Stage 3C–3G audit context;
#   - audits T1 usage and one-to-one linkage integrity;
#   - writes checksum-controlled final artifacts and a freeze manifest.
#
# This cell DOES NOT:
#   - construct future-instability outcomes;
#   - compare T0 and T1 classifications;
#   - label unresolved records stable;
#   - confirm official ClinVar replacement, withdrawal, merge, or split history;
#   - fit, tune, or evaluate GES.
# =================================================================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import re

import pandas as pd
import pyarrow.parquet as pq


print("=" * 128)
print("STAGE 3H — FINAL T0–T1 LINKAGE ASSEMBLY AND FREEZE")
print("=" * 128)


# -------------------------------------------------------------------------------------------------
# 1. PATHS
# -------------------------------------------------------------------------------------------------

STUDY_ROOT = Path("/content/drive/MyDrive/GES_RAG_Temporal_Study")

DATA_INTERIM_DIR = STUDY_ROOT / "data_interim"
STAGE3_CONFIG_DIR = STUDY_ROOT / "configs" / "stage3_crosswalk"
STAGE3_OUTPUT_DIR = STUDY_ROOT / "data_processed" / "stage3_crosswalk"

T0_PATH = (
    DATA_INTERIM_DIR
    / "t0_rcv_target_genes_corrected_v1_2.parquet"
)

T1_PATH = (
    DATA_INTERIM_DIR
    / "t1_rcv_target_genes_harmonized_v1.parquet"
)

STAGE3B_CROSSWALK_PATH = (
    STAGE3_OUTPUT_DIR
    / "stage3_t0_t1_exact_rcv_crosswalk_draft_v1.parquet"
)

STAGE3C_AUDIT_PATH = (
    STAGE3_OUTPUT_DIR
    / "stage3_t0_t1_exact_match_continuity_audit_v1.parquet"
)

STAGE3E_ACCEPTED_LINKS_PATH = (
    STAGE3_OUTPUT_DIR
    / "stage3_accepted_nonexact_one_to_one_links_v1.parquet"
)

STAGE3E_T0_STATUS_PATH = (
    STAGE3_OUTPUT_DIR
    / "stage3_t0_post_strong_candidate_adjudication_v1.parquet"
)

STAGE3E_MANIFEST_PATH = (
    STAGE3_CONFIG_DIR
    / "stage3_strong_candidate_adjudication_manifest_v1.json"
)

STAGE3F_T0_REVIEW_PATH = (
    STAGE3_OUTPUT_DIR
    / "stage3_targeted_candidate_t0_review_summary_v1.parquet"
)

STAGE3F_MANIFEST_PATH = (
    STAGE3_CONFIG_DIR
    / "stage3_targeted_complex_and_tier3_audit_manifest_v1.json"
)

STAGE3G_T0_AUDIT_PATH = (
    STAGE3_OUTPUT_DIR
    / "stage3_no_identifier_t0_audit_summary_v1.parquet"
)

STAGE3G_MANIFEST_PATH = (
    STAGE3_CONFIG_DIR
    / "stage3_no_identifier_condition_only_audit_manifest_v1.json"
)


POLICY_PATH = (
    STAGE3_CONFIG_DIR
    / "stage3_final_linkage_and_censoring_policy_v1.json"
)

FINAL_LINKAGE_PATH = (
    STAGE3_OUTPUT_DIR
    / "stage3_t0_t1_final_linkage_frozen_v1.parquet"
)

T1_USAGE_PATH = (
    STAGE3_OUTPUT_DIR
    / "stage3_t1_linkage_usage_audit_v1.parquet"
)

REPORT_PATH = (
    STAGE3_CONFIG_DIR
    / "stage3_final_linkage_freeze_report_v1.json"
)

MANIFEST_PATH = (
    STAGE3_CONFIG_DIR
    / "stage3_final_linkage_freeze_manifest_v1.json"
)


# -------------------------------------------------------------------------------------------------
# 2. EXPECTED IMMUTABLE INPUT CHECKSUMS
# -------------------------------------------------------------------------------------------------

EXPECTED_CHECKSUMS = {
    T0_PATH: (
        "f6b6760b2ad6e4352e3bdecdeaf89827"
        "e8a514b031abf2373bb17568d999466d"
    ),
    T1_PATH: (
        "5713a11bdbf4804758cc011f9b2f302a"
        "fc91fa1f88c1b178d675c28bb277d37c"
    ),
    STAGE3B_CROSSWALK_PATH: (
        "b462304a4bb31db301e2dac3aefbf685"
        "e12f1fea179a38ca04b8500b64bb4acc"
    ),
    STAGE3C_AUDIT_PATH: (
        "7671792ebccae68ff77745355498d174e"
        "0a614f21703e518fb230ebfc54666ad"
    ),
    STAGE3E_ACCEPTED_LINKS_PATH: (
        "19e9eb00810f8b3faf2147c3a73895d0"
        "a939068beb20b3aaaf9a8bd8a8d64268"
    ),
    STAGE3E_T0_STATUS_PATH: (
        "c1e40c92389e2dc200cd1508c49fb7a"
        "a12c3a9665fc2e6aab56ed2127accb65e"
    ),
    STAGE3E_MANIFEST_PATH: (
        "b5ea53ba3fe6cafdb2abaa25ac582d683"
        "cfb35dd04d88dcfdaf03802edec4304"
    ),
    STAGE3F_T0_REVIEW_PATH: (
        "09f6e4d24744781127e59d7ee03ddf3d"
        "5d02b0c4ea5517e52a184ad46a623d2a"
    ),
    STAGE3F_MANIFEST_PATH: (
        "004292519a8a33d261c9a57acdbd6573"
        "fc074338a2bb311bbd464b8fd6fa47b1"
    ),
    STAGE3G_T0_AUDIT_PATH: (
        "7d3bfcd3428e72f7c48a3ed9aa954126"
        "fc7dd0eb4a1daec1a24ff1173670d776"
    ),
    STAGE3G_MANIFEST_PATH: (
        "99bef658295e208c650a7e693df0c70b"
        "54221557f3caa1b0d8a266666c5a72d2"
    ),
}


EXPECTED_T0_ROWS = 71659
EXPECTED_T1_ROWS = 100920

EXPECTED_EXACT_LINKS = 70413
EXPECTED_ACCEPTED_NONEXACT_LINKS = 170
EXPECTED_TOTAL_LINKED_T0 = 70583

EXPECTED_UNRESOLVED_COMPLEX = 38
EXPECTED_UNRESOLVED_CONDITION = 416
EXPECTED_UNRESOLVED_NO_IDENTIFIER = 622
EXPECTED_TOTAL_UNRESOLVED = 1076

EXPECTED_LINKED_T1 = 70583
EXPECTED_UNLINKED_T1 = 30337


# -------------------------------------------------------------------------------------------------
# 3. UTILITY FUNCTIONS
# -------------------------------------------------------------------------------------------------

def sha256_file(path: Path, chunk_size: int = 8 * 1024 * 1024) -> str:
    """Calculate a file SHA-256 without loading the complete file into memory."""
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def normalize_rcv(value):
    """Normalize one RCV accession."""
    if value is None:
        return None

    try:
        if pd.isna(value):
            return None
    except (TypeError, ValueError):
        pass

    text = str(value).strip().upper()

    return text or None


def normalize_identifier(value):
    """Normalize identifiers and remove accidental numeric '.0' suffixes."""
    if value is None:
        return None

    try:
        if pd.isna(value):
            return None
    except (TypeError, ValueError):
        pass

    text = str(value).strip().upper()

    if re.fullmatch(r"\d+\.0", text):
        text = text[:-2]

    return text or None


def value_counts_dict(series: pd.Series) -> dict:
    """Convert pandas value counts to a JSON-safe dictionary."""
    counts = series.value_counts(dropna=False)

    result = {}

    for key, value in counts.items():
        safe_key = "<MISSING>" if pd.isna(key) else str(key)
        result[safe_key] = int(value)

    return result


def prefix_columns(frame: pd.DataFrame, prefix: str, key: str) -> pd.DataFrame:
    """Prefix all columns except one normalized join key."""
    rename_map = {
        column: f"{prefix}_{column}"
        for column in frame.columns
        if column != key
    }

    return frame.rename(columns=rename_map)


def stage3c_axis_category(gene_json, axis_value) -> str:
    """
    Create a descriptive gene-aware axis category for accepted non-exact links.

    This does not infer a historical T0 classification axis.
    """
    try:
        genes = json.loads(gene_json)
    except (TypeError, json.JSONDecodeError):
        genes = []

    genes = {
        str(gene).strip().upper()
        for gene in genes
        if str(gene).strip()
    }

    axis = str(axis_value).strip() if pd.notna(axis_value) else "MissingAxis"

    if len(genes) != 1:
        return "MULTIPLE_OR_MISSING_GENE_AXIS_REVIEW"

    gene = next(iter(genes))

    if gene in {"BRCA1", "BRCA2", "MLH1"}:
        if axis == "GermlineClassification":
            return "PRIMARY_GENE_GERMLINE_AXIS"

        if axis in {"NoClassification", "MissingAxis", ""}:
            return "PRIMARY_GENE_NO_T1_CLASSIFICATION"

        return "PRIMARY_GENE_NON_GERMLINE_AXIS_REVIEW"

    if gene == "EGFR":
        if axis == "OncogenicityClassification":
            return "EGFR_ONCOGENICITY_AXIS"

        if axis == "SomaticClinicalImpact":
            return "EGFR_SOMATIC_CLINICAL_IMPACT_AXIS"

        if axis == "GermlineClassification":
            return "EGFR_GERMLINE_AXIS"

        if axis in {"NoClassification", "MissingAxis", ""}:
            return "EGFR_NO_T1_CLASSIFICATION"

        return "EGFR_OTHER_AXIS_REVIEW"

    return "UNEXPECTED_GENE_AXIS_REVIEW"


# -------------------------------------------------------------------------------------------------
# 4. VERIFY ALL IMMUTABLE INPUT ARTIFACTS
# -------------------------------------------------------------------------------------------------

print("Verifying immutable Stage 2 and Stage 3 input artifacts...")

observed_checksums = {}

for path, expected_sha256 in EXPECTED_CHECKSUMS.items():

    if not path.exists():
        raise FileNotFoundError(
            f"Required input artifact does not exist: {path}"
        )

    observed_sha256 = sha256_file(path)
    observed_checksums[str(path)] = observed_sha256

    if observed_sha256 != expected_sha256:
        raise RuntimeError(
            f"Checksum mismatch for:\n{path}\n"
            f"Expected: {expected_sha256}\n"
            f"Observed: {observed_sha256}"
        )

    print(f"PASS  {path.name}")
    print(f"      {observed_sha256}")


# -------------------------------------------------------------------------------------------------
# 5. VERIFY AUTHORIZING MANIFEST STATUSES
# -------------------------------------------------------------------------------------------------

with STAGE3E_MANIFEST_PATH.open("r", encoding="utf-8") as handle:
    stage3e_manifest = json.load(handle)

with STAGE3F_MANIFEST_PATH.open("r", encoding="utf-8") as handle:
    stage3f_manifest = json.load(handle)

with STAGE3G_MANIFEST_PATH.open("r", encoding="utf-8") as handle:
    stage3g_manifest = json.load(handle)

if (
    stage3e_manifest.get("adjudication_status")
    != "CONSERVATIVE_STRONG_CANDIDATE_ADJUDICATION_COMPLETE"
):
    raise RuntimeError(
        "Stage 3E does not contain the expected completed status."
    )

if (
    stage3f_manifest.get("audit_status")
    != "TARGETED_COMPLEX_AND_TIER3_AUDIT_COMPLETE"
):
    raise RuntimeError(
        "Stage 3F does not contain the expected completed status."
    )

if (
    stage3g_manifest.get("audit_status")
    != "NO_STABLE_IDENTIFIER_CONDITION_ONLY_AUDIT_COMPLETE"
):
    raise RuntimeError(
        "Stage 3G does not contain the expected completed status."
    )


# -------------------------------------------------------------------------------------------------
# 6. FREEZE THE FINAL LINKAGE AND CENSORING POLICY
# -------------------------------------------------------------------------------------------------

policy = {
    "policy_name": "Stage 3 Final T0-T1 Linkage and Censoring Policy",
    "policy_version": "1.0.0",
    "scientific_unit": "RCV-level variant-condition aggregate",

    "accepted_linkage_rules": {
        "exact_link": {
            "rule": "Exact normalized RCV accession match",
            "one_to_one_required": True,
            "accepted_status": "EXACT_RCV_MATCH",
        },
        "conservative_nonexact_link": {
            "rule": (
                "Stage 3E unique one-to-one Tier 1 continuity link with "
                "exact VariationID, exact VCV accession, exact target gene, "
                "and condition overlap"
            ),
            "one_to_one_required": True,
            "accepted_status": (
                "ACCEPTED_NONEXACT_ONE_TO_ONE_CONTINUITY"
            ),
            "official_rcv_replacement_claimed": False,
        },
    },

    "unresolved_censoring_rules": {
        "UNRESOLVED_COMPLEX_STRONG_CANDIDATE_RELATIONSHIP": (
            "CENSORED_NONUNIQUE_OR_COMPLEX_LINKAGE"
        ),
        "UNRESOLVED_VARIANT_CONTINUITY_WITHOUT_CONDITION_OVERLAP": (
            "CENSORED_VARIANT_CONTINUITY_WITH_UNRESOLVED_CONDITION_ASSOCIATION"
        ),
        "UNRESOLVED_NO_VARIATIONID_OR_VCV_CANDIDATE": (
            "CENSORED_NO_STABLE_VARIANT_IDENTIFIER_LINK"
        ),
    },

    "primary_analysis_rule": (
        "Only accepted one-to-one linked records are linkage-eligible for "
        "subsequent temporal-outcome construction. Additional classification-axis, "
        "classification-availability, and outcome-specific eligibility rules remain "
        "to be applied later."
    ),

    "unmatched_record_rule": (
        "Unresolved, unmatched, or censored T0 records must not be assigned "
        "a stable outcome because no defensible one-to-one T1 comparison exists."
    ),

    "condition_only_rule": (
        "Condition-only candidates are descriptive audit evidence and cannot "
        "establish variant continuity."
    ),

    "prohibited_operations_at_stage3": [
        "No future-instability outcome construction",
        "No stable or unstable label assignment",
        "No classification-change calculation",
        "No GES fitting or tuning",
        "No confirmation of official ClinVar replacement history",
        "No automatic acceptance of merge, split, or many-to-many relationships",
    ],
}


if POLICY_PATH.exists():

    with POLICY_PATH.open("r", encoding="utf-8") as handle:
        existing_policy = json.load(handle)

    if existing_policy != policy:
        raise RuntimeError(
            "An existing Stage 3 linkage policy differs from the current policy. "
            "Do not overwrite it silently."
        )

else:

    with POLICY_PATH.open("w", encoding="utf-8") as handle:
        json.dump(
            policy,
            handle,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
        )


policy_sha256 = sha256_file(POLICY_PATH)

print()
print(f"Frozen policy:       {POLICY_PATH}")
print(f"Policy SHA-256:      {policy_sha256}")


# -------------------------------------------------------------------------------------------------
# 7. LOAD THE FROZEN T0 AND T1 COHORTS
# -------------------------------------------------------------------------------------------------

source_columns = [
    "release_label",
    "embedded_data_cutoff_date",
    "rcv_accession",
    "rcv_version",
    "variation_id",
    "vcv_accession",
    "vcv_version",
    "target_genes_json",
    "study_scope",
    "condition_names_json",
    "condition_ids_json",
]

t1_source_columns = source_columns + [
    "aggregate_classification_axis",
]

t0 = pd.read_parquet(
    T0_PATH,
    columns=source_columns,
)

t1 = pd.read_parquet(
    T1_PATH,
    columns=t1_source_columns,
)

if len(t0) != EXPECTED_T0_ROWS:
    raise AssertionError(
        f"Expected {EXPECTED_T0_ROWS:,} T0 rows, observed {len(t0):,}."
    )

if len(t1) != EXPECTED_T1_ROWS:
    raise AssertionError(
        f"Expected {EXPECTED_T1_ROWS:,} T1 rows, observed {len(t1):,}."
    )

t0["t0_rcv_key"] = [
    normalize_rcv(value)
    for value in t0["rcv_accession"]
]

t1["t1_rcv_key"] = [
    normalize_rcv(value)
    for value in t1["rcv_accession"]
]

if t0["t0_rcv_key"].isna().any():
    raise AssertionError("T0 contains a missing normalized RCV key.")

if t1["t1_rcv_key"].isna().any():
    raise AssertionError("T1 contains a missing normalized RCV key.")

if t0["t0_rcv_key"].duplicated().any():
    raise AssertionError("T0 contains duplicate normalized RCV keys.")

if t1["t1_rcv_key"].duplicated().any():
    raise AssertionError("T1 contains duplicate normalized RCV keys.")

t0 = prefix_columns(
    t0,
    prefix="t0",
    key="t0_rcv_key",
)

t1 = prefix_columns(
    t1,
    prefix="t1",
    key="t1_rcv_key",
)


# -------------------------------------------------------------------------------------------------
# 8. LOAD EXACT RCV LINKS
# -------------------------------------------------------------------------------------------------

exact_crosswalk = pd.read_parquet(
    STAGE3B_CROSSWALK_PATH,
    columns=[
        "linkage_status",
        "t0_rcv_accession",
        "t1_rcv_accession",
    ],
)

exact_links = (
    exact_crosswalk.loc[
        exact_crosswalk["linkage_status"].eq("EXACT_RCV_MATCH")
    ]
    .copy()
    .reset_index(drop=True)
)

exact_links["t0_rcv_key"] = [
    normalize_rcv(value)
    for value in exact_links["t0_rcv_accession"]
]

exact_links["linked_t1_rcv_key"] = [
    normalize_rcv(value)
    for value in exact_links["t1_rcv_accession"]
]

exact_links = exact_links[
    [
        "t0_rcv_key",
        "linked_t1_rcv_key",
    ]
].copy()

exact_links["final_linkage_status"] = "EXACT_RCV_MATCH"
exact_links["linkage_method"] = "EXACT_RCV_ACCESSION"
exact_links["accepted_nonexact_candidate_pair_id"] = None
exact_links["accepted_nonexact_condition_evidence"] = None

if len(exact_links) != EXPECTED_EXACT_LINKS:
    raise AssertionError(
        f"Expected {EXPECTED_EXACT_LINKS:,} exact links, "
        f"observed {len(exact_links):,}."
    )

if exact_links["t0_rcv_key"].duplicated().any():
    raise AssertionError("Duplicate T0 keys exist among exact links.")

if exact_links["linked_t1_rcv_key"].duplicated().any():
    raise AssertionError("Duplicate T1 keys exist among exact links.")


# -------------------------------------------------------------------------------------------------
# 9. LOAD ACCEPTED CONSERVATIVE NON-EXACT LINKS
# -------------------------------------------------------------------------------------------------

accepted_nonexact = pd.read_parquet(
    STAGE3E_ACCEPTED_LINKS_PATH,
)

if len(accepted_nonexact) != EXPECTED_ACCEPTED_NONEXACT_LINKS:
    raise AssertionError(
        f"Expected {EXPECTED_ACCEPTED_NONEXACT_LINKS:,} accepted non-exact links, "
        f"observed {len(accepted_nonexact):,}."
    )

accepted_nonexact["t0_rcv_key"] = [
    normalize_rcv(value)
    for value in accepted_nonexact["t0_rcv_accession"]
]

accepted_nonexact["linked_t1_rcv_key"] = [
    normalize_rcv(value)
    for value in accepted_nonexact["t1_rcv_accession"]
]

if accepted_nonexact["t0_rcv_key"].duplicated().any():
    raise AssertionError(
        "Duplicate T0 keys exist among accepted non-exact links."
    )

if accepted_nonexact["linked_t1_rcv_key"].duplicated().any():
    raise AssertionError(
        "Duplicate T1 keys exist among accepted non-exact links."
    )

accepted_links = accepted_nonexact[
    [
        "t0_rcv_key",
        "linked_t1_rcv_key",
        "candidate_pair_id",
        "condition_evidence_category",
    ]
].copy()

accepted_links = accepted_links.rename(
    columns={
        "candidate_pair_id": (
            "accepted_nonexact_candidate_pair_id"
        ),
        "condition_evidence_category": (
            "accepted_nonexact_condition_evidence"
        ),
    }
)

accepted_links["final_linkage_status"] = (
    "ACCEPTED_NONEXACT_ONE_TO_ONE_CONTINUITY"
)

accepted_links["linkage_method"] = (
    "CONSERVATIVE_VARIATIONID_VCV_GENE_CONDITION_CONTINUITY"
)


# -------------------------------------------------------------------------------------------------
# 10. COMBINE ACCEPTED LINKS AND VALIDATE ONE-TO-ONE INTEGRITY
# -------------------------------------------------------------------------------------------------

link_map = pd.concat(
    [
        exact_links,
        accepted_links,
    ],
    ignore_index=True,
)

if len(link_map) != EXPECTED_TOTAL_LINKED_T0:
    raise AssertionError(
        f"Expected {EXPECTED_TOTAL_LINKED_T0:,} accepted links, "
        f"observed {len(link_map):,}."
    )

if link_map["t0_rcv_key"].duplicated().any():
    raise AssertionError(
        "A T0 record received more than one accepted final link."
    )

if link_map["linked_t1_rcv_key"].duplicated().any():
    raise AssertionError(
        "A T1 record was assigned to more than one T0 record."
    )

t0_source_keys = set(t0["t0_rcv_key"])
t1_source_keys = set(t1["t1_rcv_key"])

missing_t0_link_keys = (
    set(link_map["t0_rcv_key"]) - t0_source_keys
)

missing_t1_link_keys = (
    set(link_map["linked_t1_rcv_key"]) - t1_source_keys
)

if missing_t0_link_keys:
    raise AssertionError(
        f"{len(missing_t0_link_keys):,} accepted T0 keys are absent "
        "from the frozen T0 cohort."
    )

if missing_t1_link_keys:
    raise AssertionError(
        f"{len(missing_t1_link_keys):,} accepted T1 keys are absent "
        "from the frozen T1 cohort."
    )


# -------------------------------------------------------------------------------------------------
# 11. LOAD STAGE 3C–3G AUDIT CONTEXT
# -------------------------------------------------------------------------------------------------

stage3c = pd.read_parquet(
    STAGE3C_AUDIT_PATH,
    columns=[
        "t0_rcv_accession",
        "core_identifier_continuity_status",
        "gene_set_relation",
        "condition_continuity_status",
        "classification_axis_audit_category",
        "requires_linkage_review",
        "review_reason_count",
        "review_reasons_json",
    ],
)

stage3c["t0_rcv_key"] = [
    normalize_rcv(value)
    for value in stage3c["t0_rcv_accession"]
]

stage3c = stage3c.drop(
    columns=["t0_rcv_accession"]
)

if len(stage3c) != EXPECTED_EXACT_LINKS:
    raise AssertionError(
        "Stage 3C exact-match audit row count is unexpected."
    )

if stage3c["t0_rcv_key"].duplicated().any():
    raise AssertionError(
        "Stage 3C contains duplicate T0 linkage keys."
    )


stage3e_status = pd.read_parquet(
    STAGE3E_T0_STATUS_PATH,
    columns=[
        "t0_rcv_accession",
        "stage3e_post_adjudication_status",
        "identifier_based_candidate_count",
        "strong_candidate_count",
        "accepted_stage3e_t1_rcv_accession",
    ],
)

stage3e_status["t0_rcv_key"] = [
    normalize_rcv(value)
    for value in stage3e_status["t0_rcv_accession"]
]

stage3e_status = stage3e_status.drop(
    columns=["t0_rcv_accession"]
)

if stage3e_status["t0_rcv_key"].duplicated().any():
    raise AssertionError(
        "Stage 3E T0 status contains duplicate T0 keys."
    )


stage3f_review = pd.read_parquet(
    STAGE3F_T0_REVIEW_PATH,
    columns=[
        "t0_rcv_accession",
        "stage3f_targeted_candidate_pair_count",
        "stage3f_component_count",
        "stage3f_t0_review_category",
        "stage3f_resolution_status",
    ],
)

stage3f_review["t0_rcv_key"] = [
    normalize_rcv(value)
    for value in stage3f_review["t0_rcv_accession"]
]

stage3f_review = stage3f_review.drop(
    columns=["t0_rcv_accession"]
)

if stage3f_review["t0_rcv_key"].duplicated().any():
    raise AssertionError(
        "Stage 3F T0 review contains duplicate T0 keys."
    )


stage3g_audit = pd.read_parquet(
    STAGE3G_T0_AUDIT_PATH,
    columns=[
        "t0_rcv_accession",
        "condition_only_candidate_count",
        "exact_condition_set_candidate_count",
        "unique_condition_only_candidate_t1_rcv",
        "stage3g_t0_audit_category",
        "stage3g_source_history_review_priority",
        "stage3g_resolution_status",
    ],
)

stage3g_audit["t0_rcv_key"] = [
    normalize_rcv(value)
    for value in stage3g_audit["t0_rcv_accession"]
]

stage3g_audit = stage3g_audit.drop(
    columns=["t0_rcv_accession"]
)

if stage3g_audit["t0_rcv_key"].duplicated().any():
    raise AssertionError(
        "Stage 3G T0 audit contains duplicate T0 keys."
    )


# -------------------------------------------------------------------------------------------------
# 12. ASSEMBLE ONE FINAL ROW FOR EVERY T0 RECORD
# -------------------------------------------------------------------------------------------------

final_linkage = (
    t0
    .merge(
        link_map,
        on="t0_rcv_key",
        how="left",
        validate="one_to_one",
    )
    .merge(
        t1,
        left_on="linked_t1_rcv_key",
        right_on="t1_rcv_key",
        how="left",
        validate="many_to_one",
    )
    .merge(
        stage3c,
        on="t0_rcv_key",
        how="left",
        validate="one_to_one",
    )
    .merge(
        stage3e_status,
        on="t0_rcv_key",
        how="left",
        validate="one_to_one",
    )
    .merge(
        stage3f_review,
        on="t0_rcv_key",
        how="left",
        validate="one_to_one",
    )
    .merge(
        stage3g_audit,
        on="t0_rcv_key",
        how="left",
        validate="one_to_one",
    )
)


if len(final_linkage) != EXPECTED_T0_ROWS:
    raise AssertionError(
        "Final linkage does not contain exactly one row per T0 record."
    )

if final_linkage["t0_rcv_key"].duplicated().any():
    raise AssertionError(
        "Final linkage contains duplicate T0 keys."
    )


# -------------------------------------------------------------------------------------------------
# 13. ASSIGN FINAL LINKAGE AND CENSORING DISPOSITIONS
# -------------------------------------------------------------------------------------------------

final_linkage["final_linkage_status"] = (
    final_linkage["final_linkage_status"]
    .fillna("UNRESOLVED_CENSORED")
)

final_linkage["linkage_method"] = (
    final_linkage["linkage_method"]
    .fillna("NO_ACCEPTED_ONE_TO_ONE_T1_LINK")
)

final_linkage["linked_t1_record_present"] = (
    final_linkage["linked_t1_rcv_key"].notna()
)

final_linkage["one_to_one_linkage_accepted"] = (
    final_linkage["final_linkage_status"].isin(
        [
            "EXACT_RCV_MATCH",
            "ACCEPTED_NONEXACT_ONE_TO_ONE_CONTINUITY",
        ]
    )
)

final_linkage[
    "linkage_eligible_for_future_outcome_construction"
] = final_linkage["one_to_one_linkage_accepted"]


unresolved_disposition_map = {
    "UNRESOLVED_COMPLEX_STRONG_CANDIDATE_RELATIONSHIP": (
        "CENSORED_NONUNIQUE_OR_COMPLEX_LINKAGE"
    ),
    "UNRESOLVED_VARIANT_CONTINUITY_WITHOUT_CONDITION_OVERLAP": (
        "CENSORED_VARIANT_CONTINUITY_WITH_UNRESOLVED_CONDITION_ASSOCIATION"
    ),
    "UNRESOLVED_NO_VARIATIONID_OR_VCV_CANDIDATE": (
        "CENSORED_NO_STABLE_VARIANT_IDENTIFIER_LINK"
    ),
}

final_linkage["final_unresolved_disposition"] = (
    final_linkage["stage3e_post_adjudication_status"]
    .map(unresolved_disposition_map)
)

linked_mask = final_linkage["one_to_one_linkage_accepted"]
unresolved_mask = ~linked_mask

final_linkage.loc[
    linked_mask,
    "final_unresolved_disposition",
] = None

if final_linkage.loc[
    unresolved_mask,
    "final_unresolved_disposition",
].isna().any():

    missing_statuses = (
        final_linkage.loc[
            unresolved_mask
            & final_linkage[
                "final_unresolved_disposition"
            ].isna(),
            "stage3e_post_adjudication_status",
        ]
        .value_counts(dropna=False)
        .to_dict()
    )

    raise AssertionError(
        "At least one unresolved T0 record lacks a frozen censoring "
        f"disposition: {missing_statuses}"
    )


# -------------------------------------------------------------------------------------------------
# 14. STANDARDIZE AUDIT CONTEXT FOR EXACT AND NON-EXACT LINKS
# -------------------------------------------------------------------------------------------------

exact_mask = final_linkage[
    "final_linkage_status"
].eq("EXACT_RCV_MATCH")

nonexact_mask = final_linkage[
    "final_linkage_status"
].eq(
    "ACCEPTED_NONEXACT_ONE_TO_ONE_CONTINUITY"
)


final_linkage.loc[
    exact_mask,
    "stage3e_post_adjudication_status",
] = "NOT_APPLICABLE_EXACT_RCV_MATCH"


final_linkage.loc[
    nonexact_mask,
    "core_identifier_continuity_status",
] = "STABLE_CORE_IDENTIFIERS"

final_linkage.loc[
    nonexact_mask,
    "gene_set_relation",
] = "EXACT_SET_MATCH"

final_linkage.loc[
    nonexact_mask,
    "condition_continuity_status",
] = (
    "ACCEPTED_NONEXACT_CONDITION_OVERLAP"
)

final_linkage.loc[
    nonexact_mask,
    "classification_axis_audit_category",
] = [
    stage3c_axis_category(gene_json, axis_value)
    for gene_json, axis_value in zip(
        final_linkage.loc[
            nonexact_mask,
            "t1_target_genes_json",
        ],
        final_linkage.loc[
            nonexact_mask,
            "t1_aggregate_classification_axis",
        ],
    )
]

final_linkage.loc[
    nonexact_mask,
    "requires_linkage_review",
] = False

final_linkage.loc[
    nonexact_mask,
    "review_reason_count",
] = 0

final_linkage.loc[
    nonexact_mask,
    "review_reasons_json",
] = "[]"


# -------------------------------------------------------------------------------------------------
# 15. ADD EXPLICIT SCIENTIFIC-BOUNDARY FIELDS
# -------------------------------------------------------------------------------------------------

final_linkage["linkage_policy_version"] = policy["policy_version"]
final_linkage["linkage_policy_sha256"] = policy_sha256

final_linkage["stage3_linkage_frozen"] = True
final_linkage["future_instability_outcome_created"] = False
final_linkage["future_instability_label"] = None

final_linkage["unresolved_record_labeled_stable"] = False
final_linkage["condition_only_candidate_used_for_linkage"] = False

final_linkage["classification_change_calculated"] = False
final_linkage["official_rcv_replacement_confirmed"] = False

final_linkage["outcome_construction_status"] = (
    "NOT_STARTED_STAGE3_LINKAGE_ONLY"
)

final_linkage["final_crosswalk_record_id"] = (
    final_linkage["t0_rcv_key"]
)


# -------------------------------------------------------------------------------------------------
# 16. VALIDATE LINKED IDENTIFIER CONTINUITY
# -------------------------------------------------------------------------------------------------

linked = final_linkage.loc[
    linked_mask
].copy()

linked_variation_mismatches = int(
    (
        linked["t0_variation_id"]
        .map(normalize_identifier)
        != linked["t1_variation_id"]
        .map(normalize_identifier)
    ).sum()
)

linked_vcv_mismatches = int(
    (
        linked["t0_vcv_accession"]
        .map(normalize_identifier)
        != linked["t1_vcv_accession"]
        .map(normalize_identifier)
    ).sum()
)

if linked_variation_mismatches != 0:
    raise AssertionError(
        f"{linked_variation_mismatches:,} accepted links have "
        "VariationID discontinuity."
    )

if linked_vcv_mismatches != 0:
    raise AssertionError(
        f"{linked_vcv_mismatches:,} accepted links have "
        "VCV-accession discontinuity."
    )

if linked["linked_t1_rcv_key"].isna().any():
    raise AssertionError(
        "At least one accepted link has no linked T1 RCV."
    )

if linked["t1_rcv_accession"].isna().any():
    raise AssertionError(
        "At least one accepted link failed to retrieve its frozen T1 record."
    )

if linked["linked_t1_rcv_key"].duplicated().any():
    raise AssertionError(
        "The final linked cohort contains duplicate T1 assignments."
    )

if final_linkage.loc[
    unresolved_mask,
    "linked_t1_rcv_key",
].notna().any():
    raise AssertionError(
        "An unresolved record contains an accepted T1 linkage key."
    )


# -------------------------------------------------------------------------------------------------
# 17. VALIDATE FINAL STATUS COUNTS
# -------------------------------------------------------------------------------------------------

final_status_counts = final_linkage[
    "final_linkage_status"
].value_counts()

observed_exact = int(
    final_status_counts.get("EXACT_RCV_MATCH", 0)
)

observed_nonexact = int(
    final_status_counts.get(
        "ACCEPTED_NONEXACT_ONE_TO_ONE_CONTINUITY",
        0,
    )
)

observed_unresolved = int(
    final_status_counts.get("UNRESOLVED_CENSORED", 0)
)

if observed_exact != EXPECTED_EXACT_LINKS:
    raise AssertionError(
        f"Expected {EXPECTED_EXACT_LINKS:,} exact links, "
        f"observed {observed_exact:,}."
    )

if observed_nonexact != EXPECTED_ACCEPTED_NONEXACT_LINKS:
    raise AssertionError(
        f"Expected {EXPECTED_ACCEPTED_NONEXACT_LINKS:,} accepted non-exact links, "
        f"observed {observed_nonexact:,}."
    )

if observed_unresolved != EXPECTED_TOTAL_UNRESOLVED:
    raise AssertionError(
        f"Expected {EXPECTED_TOTAL_UNRESOLVED:,} unresolved records, "
        f"observed {observed_unresolved:,}."
    )


unresolved_counts = final_linkage.loc[
    unresolved_mask,
    "final_unresolved_disposition",
].value_counts()

if int(
    unresolved_counts.get(
        "CENSORED_NONUNIQUE_OR_COMPLEX_LINKAGE",
        0,
    )
) != EXPECTED_UNRESOLVED_COMPLEX:
    raise AssertionError(
        "Complex/non-unique censoring count does not equal 38."
    )

if int(
    unresolved_counts.get(
        "CENSORED_VARIANT_CONTINUITY_WITH_UNRESOLVED_CONDITION_ASSOCIATION",
        0,
    )
) != EXPECTED_UNRESOLVED_CONDITION:
    raise AssertionError(
        "Unresolved condition-association censoring count does not equal 416."
    )

if int(
    unresolved_counts.get(
        "CENSORED_NO_STABLE_VARIANT_IDENTIFIER_LINK",
        0,
    )
) != EXPECTED_UNRESOLVED_NO_IDENTIFIER:
    raise AssertionError(
        "No-stable-identifier censoring count does not equal 622."
    )


# -------------------------------------------------------------------------------------------------
# 18. CREATE T1-SIDE USAGE AUDIT
# -------------------------------------------------------------------------------------------------

exact_t1_keys = set(
    exact_links["linked_t1_rcv_key"]
)

accepted_nonexact_t1_keys = set(
    accepted_links["linked_t1_rcv_key"]
)

if exact_t1_keys.intersection(accepted_nonexact_t1_keys):
    raise AssertionError(
        "A T1 record appears in both exact and accepted non-exact linkage sets."
    )

t1_usage = t1[
    [
        "t1_rcv_key",
        "t1_release_label",
        "t1_embedded_data_cutoff_date",
        "t1_rcv_accession",
        "t1_rcv_version",
        "t1_variation_id",
        "t1_vcv_accession",
        "t1_vcv_version",
        "t1_target_genes_json",
        "t1_study_scope",
        "t1_condition_names_json",
        "t1_condition_ids_json",
        "t1_aggregate_classification_axis",
    ]
].copy()


def t1_usage_status(rcv_key: str) -> str:
    if rcv_key in exact_t1_keys:
        return "LINKED_TO_T0_BY_EXACT_RCV"

    if rcv_key in accepted_nonexact_t1_keys:
        return "LINKED_TO_T0_BY_ACCEPTED_NONEXACT_CONTINUITY"

    return "T1_NOT_LINKED_TO_T0"


t1_usage["t1_linkage_usage_status"] = [
    t1_usage_status(value)
    for value in t1_usage["t1_rcv_key"]
]

t1_usage["linked_to_one_t0_record"] = (
    t1_usage["t1_linkage_usage_status"]
    .ne("T1_NOT_LINKED_TO_T0")
)

t1_to_t0_map = dict(
    zip(
        link_map["linked_t1_rcv_key"],
        link_map["t0_rcv_key"],
    )
)

t1_usage["linked_t0_rcv_accession"] = [
    t1_to_t0_map.get(value)
    for value in t1_usage["t1_rcv_key"]
]

t1_usage["future_instability_outcome_created"] = False

t1_usage_counts = t1_usage[
    "t1_linkage_usage_status"
].value_counts()

observed_linked_t1 = int(
    t1_usage["linked_to_one_t0_record"].sum()
)

observed_unlinked_t1 = int(
    (~t1_usage["linked_to_one_t0_record"]).sum()
)

if observed_linked_t1 != EXPECTED_LINKED_T1:
    raise AssertionError(
        f"Expected {EXPECTED_LINKED_T1:,} linked T1 records, "
        f"observed {observed_linked_t1:,}."
    )

if observed_unlinked_t1 != EXPECTED_UNLINKED_T1:
    raise AssertionError(
        f"Expected {EXPECTED_UNLINKED_T1:,} unlinked T1 records, "
        f"observed {observed_unlinked_t1:,}."
    )


# -------------------------------------------------------------------------------------------------
# 19. ORDER FINAL OUTPUT COLUMNS
# -------------------------------------------------------------------------------------------------

final_columns = [
    "final_crosswalk_record_id",

    "t0_rcv_key",
    "t0_release_label",
    "t0_embedded_data_cutoff_date",
    "t0_rcv_accession",
    "t0_rcv_version",
    "t0_variation_id",
    "t0_vcv_accession",
    "t0_vcv_version",
    "t0_target_genes_json",
    "t0_study_scope",
    "t0_condition_names_json",
    "t0_condition_ids_json",

    "final_linkage_status",
    "linkage_method",
    "one_to_one_linkage_accepted",
    "linkage_eligible_for_future_outcome_construction",

    "linked_t1_rcv_key",
    "linked_t1_record_present",

    "t1_release_label",
    "t1_embedded_data_cutoff_date",
    "t1_rcv_accession",
    "t1_rcv_version",
    "t1_variation_id",
    "t1_vcv_accession",
    "t1_vcv_version",
    "t1_target_genes_json",
    "t1_study_scope",
    "t1_condition_names_json",
    "t1_condition_ids_json",
    "t1_aggregate_classification_axis",

    "accepted_nonexact_candidate_pair_id",
    "accepted_nonexact_condition_evidence",

    "core_identifier_continuity_status",
    "gene_set_relation",
    "condition_continuity_status",
    "classification_axis_audit_category",
    "requires_linkage_review",
    "review_reason_count",
    "review_reasons_json",

    "stage3e_post_adjudication_status",
    "identifier_based_candidate_count",
    "strong_candidate_count",

    "stage3f_targeted_candidate_pair_count",
    "stage3f_component_count",
    "stage3f_t0_review_category",
    "stage3f_resolution_status",

    "condition_only_candidate_count",
    "exact_condition_set_candidate_count",
    "unique_condition_only_candidate_t1_rcv",
    "stage3g_t0_audit_category",
    "stage3g_source_history_review_priority",
    "stage3g_resolution_status",

    "final_unresolved_disposition",

    "linkage_policy_version",
    "linkage_policy_sha256",
    "stage3_linkage_frozen",

    "condition_only_candidate_used_for_linkage",
    "official_rcv_replacement_confirmed",
    "classification_change_calculated",

    "future_instability_outcome_created",
    "future_instability_label",
    "unresolved_record_labeled_stable",
    "outcome_construction_status",
]

missing_final_columns = [
    column
    for column in final_columns
    if column not in final_linkage.columns
]

if missing_final_columns:
    raise KeyError(
        "Final linkage is missing expected columns: "
        + ", ".join(missing_final_columns)
    )

final_linkage = final_linkage[final_columns]

final_linkage = (
    final_linkage.sort_values(
        by="t0_rcv_key",
        kind="mergesort",
    )
    .reset_index(drop=True)
)

t1_usage = (
    t1_usage.sort_values(
        by="t1_rcv_key",
        kind="mergesort",
    )
    .reset_index(drop=True)
)


# -------------------------------------------------------------------------------------------------
# 20. WRITE AND READ BACK THE FINAL FROZEN ARTIFACTS
# -------------------------------------------------------------------------------------------------

final_linkage.to_parquet(
    FINAL_LINKAGE_PATH,
    engine="pyarrow",
    compression="zstd",
    index=False,
)

t1_usage.to_parquet(
    T1_USAGE_PATH,
    engine="pyarrow",
    compression="zstd",
    index=False,
)

final_linkage_sha256 = sha256_file(FINAL_LINKAGE_PATH)
t1_usage_sha256 = sha256_file(T1_USAGE_PATH)

final_metadata = pq.ParquetFile(
    FINAL_LINKAGE_PATH
).metadata

t1_usage_metadata = pq.ParquetFile(
    T1_USAGE_PATH
).metadata

final_readback = pd.read_parquet(FINAL_LINKAGE_PATH)
t1_usage_readback = pd.read_parquet(T1_USAGE_PATH)

if len(final_readback) != EXPECTED_T0_ROWS:
    raise AssertionError(
        "Final linkage readback does not contain 71,659 rows."
    )

if final_readback["final_crosswalk_record_id"].duplicated().any():
    raise AssertionError(
        "Final linkage readback contains duplicate T0 record identifiers."
    )

if len(t1_usage_readback) != EXPECTED_T1_ROWS:
    raise AssertionError(
        "T1 usage readback does not contain 100,920 rows."
    )

if t1_usage_readback["t1_rcv_key"].duplicated().any():
    raise AssertionError(
        "T1 usage readback contains duplicate T1 RCV keys."
    )

if list(final_readback.columns) != list(final_linkage.columns):
    raise AssertionError(
        "Final linkage readback schema differs from the written dataframe."
    )

if list(t1_usage_readback.columns) != list(t1_usage.columns):
    raise AssertionError(
        "T1 usage readback schema differs from the written dataframe."
    )


# -------------------------------------------------------------------------------------------------
# 21. CREATE THE FINAL STAGE 3 FREEZE REPORT
# -------------------------------------------------------------------------------------------------

linked_rate = EXPECTED_TOTAL_LINKED_T0 / EXPECTED_T0_ROWS
unresolved_rate = EXPECTED_TOTAL_UNRESOLVED / EXPECTED_T0_ROWS

report = {
    "report_name": "Stage 3 Final T0-T1 Linkage Freeze Report",
    "report_version": "1.0.0",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "study_stage": "Stage 3H",
    "scientific_unit": "RCV-level variant-condition aggregate",

    "final_stage3_decision": (
        "ACCEPTED_AND_FROZEN_STAGE3_LINKAGE_COMPLETE"
    ),

    "policy": {
        "path": str(POLICY_PATH),
        "sha256": policy_sha256,
        "version": policy["policy_version"],
    },

    "t0_linkage_results": {
        "total_t0_records": int(EXPECTED_T0_ROWS),
        "exact_rcv_links": int(observed_exact),
        "accepted_nonexact_one_to_one_links": int(
            observed_nonexact
        ),
        "total_linked_t0_records": int(
            EXPECTED_TOTAL_LINKED_T0
        ),
        "linked_t0_share": round(float(linked_rate), 8),
        "total_unresolved_censored_t0_records": int(
            observed_unresolved
        ),
        "unresolved_t0_share": round(
            float(unresolved_rate),
            8,
        ),
        "final_linkage_status_counts": value_counts_dict(
            final_linkage["final_linkage_status"]
        ),
        "unresolved_disposition_counts": value_counts_dict(
            final_linkage.loc[
                unresolved_mask,
                "final_unresolved_disposition",
            ]
        ),
    },

    "t1_usage_results": {
        "total_t1_records": int(EXPECTED_T1_ROWS),
        "linked_t1_records": int(observed_linked_t1),
        "unlinked_t1_records": int(observed_unlinked_t1),
        "usage_status_counts": value_counts_dict(
            t1_usage["t1_linkage_usage_status"]
        ),
    },

    "audit_context": {
        "exact_match_records_flagged_for_stage3c_review": int(
            final_linkage.loc[
                exact_mask,
                "requires_linkage_review",
            ]
            .fillna(False)
            .sum()
        ),
        "stage3f_review_category_counts": value_counts_dict(
            final_linkage[
                "stage3f_t0_review_category"
            ]
        ),
        "stage3g_audit_category_counts": value_counts_dict(
            final_linkage[
                "stage3g_t0_audit_category"
            ]
        ),
    },

    "validation": {
        "one_row_per_t0_record": True,
        "unique_t0_keys": int(
            final_linkage["t0_rcv_key"].nunique()
        ),
        "accepted_linked_t1_keys": int(
            linked["linked_t1_rcv_key"].nunique()
        ),
        "duplicate_accepted_t1_assignments": int(
            linked["linked_t1_rcv_key"].duplicated().sum()
        ),
        "linked_variation_id_mismatches": int(
            linked_variation_mismatches
        ),
        "linked_vcv_accession_mismatches": int(
            linked_vcv_mismatches
        ),
        "unresolved_records_with_t1_link": int(
            final_linkage.loc[
                unresolved_mask,
                "linked_t1_rcv_key",
            ]
            .notna()
            .sum()
        ),
        "unresolved_records_labeled_stable": int(
            final_linkage[
                "unresolved_record_labeled_stable"
            ].sum()
        ),
        "future_instability_outcomes_created": int(
            final_linkage[
                "future_instability_outcome_created"
            ].sum()
        ),
        "readback_passed": True,
        "critical_failures": 0,
    },

    "scientific_boundaries": {
        "stage3_linkage_complete": True,
        "future_instability_outcomes_created": False,
        "classifications_compared": False,
        "unresolved_records_labeled_stable": False,
        "condition_only_candidates_accepted": False,
        "official_rcv_replacements_confirmed": False,
        "merges_confirmed": False,
        "splits_confirmed": False,
        "ges_model_fitted_or_tuned": False,
    },

    "next_authorized_step": (
        "Begin Blueprint Stage 4 using T0 information only: freeze the "
        "feature-processing specification, weak-supervision labeling rules, "
        "full GES model, and mandatory no-star GES ablation before inspecting "
        "future-instability outcomes."
    ),
}

with REPORT_PATH.open("w", encoding="utf-8") as handle:
    json.dump(
        report,
        handle,
        indent=2,
        sort_keys=True,
        ensure_ascii=False,
    )

report_sha256 = sha256_file(REPORT_PATH)


# -------------------------------------------------------------------------------------------------
# 22. CREATE THE FINAL STAGE 3 FREEZE MANIFEST
# -------------------------------------------------------------------------------------------------

manifest = {
    "manifest_name": "Stage 3 Final T0-T1 Linkage Freeze Manifest",
    "manifest_version": "1.0.0",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "study_stage": "Stage 3H",

    "freeze_status": (
        "ACCEPTED_AND_FROZEN_STAGE3_LINKAGE_COMPLETE"
    ),

    "inputs": {
        str(path): {
            "sha256": observed_checksums[str(path)],
        }
        for path in EXPECTED_CHECKSUMS
    },

    "policy": {
        "path": str(POLICY_PATH),
        "sha256": policy_sha256,
    },

    "outputs": {
        "final_t0_t1_linkage": {
            "path": str(FINAL_LINKAGE_PATH),
            "sha256": final_linkage_sha256,
            "rows": int(final_metadata.num_rows),
            "columns": int(final_metadata.num_columns),
            "row_groups": int(final_metadata.num_row_groups),
            "compression": "Zstandard",
        },
        "t1_linkage_usage_audit": {
            "path": str(T1_USAGE_PATH),
            "sha256": t1_usage_sha256,
            "rows": int(t1_usage_metadata.num_rows),
            "columns": int(t1_usage_metadata.num_columns),
            "row_groups": int(t1_usage_metadata.num_row_groups),
            "compression": "Zstandard",
        },
        "freeze_report": {
            "path": str(REPORT_PATH),
            "sha256": report_sha256,
        },
    },

    "frozen_counts": {
        "t0_records": int(EXPECTED_T0_ROWS),
        "exact_rcv_links": int(observed_exact),
        "accepted_nonexact_links": int(observed_nonexact),
        "total_linked_t0_records": int(
            EXPECTED_TOTAL_LINKED_T0
        ),
        "total_unresolved_censored_t0_records": int(
            observed_unresolved
        ),
        "t1_records": int(EXPECTED_T1_ROWS),
        "linked_t1_records": int(observed_linked_t1),
        "unlinked_t1_records": int(observed_unlinked_t1),
    },

    "validation_decision": "PASS",

    "scientific_boundaries": report[
        "scientific_boundaries"
    ],

    "next_authorized_step": report[
        "next_authorized_step"
    ],
}

with MANIFEST_PATH.open("w", encoding="utf-8") as handle:
    json.dump(
        manifest,
        handle,
        indent=2,
        sort_keys=True,
        ensure_ascii=False,
    )

manifest_sha256 = sha256_file(MANIFEST_PATH)


# -------------------------------------------------------------------------------------------------
# 23. PRINT FINAL RESULTS
# -------------------------------------------------------------------------------------------------

print()
print("=" * 128)
print("STAGE 3H FINAL LINKAGE-FREEZE SUMMARY")
print("=" * 128)

print(f"Frozen T0 records:                              {len(final_linkage):,}")
print(f"Exact RCV links:                                {observed_exact:,}")
print(f"Accepted conservative non-exact links:          {observed_nonexact:,}")
print(f"Total linked T0 records:                        {EXPECTED_TOTAL_LINKED_T0:,}")
print(f"Unresolved/censored T0 records:                 {observed_unresolved:,}")

print()
print("Unresolved/censored dispositions:")
print(
    final_linkage.loc[
        unresolved_mask,
        "final_unresolved_disposition",
    ]
    .value_counts(dropna=False)
    .to_string()
)

print()
print(f"Frozen T1 records:                              {len(t1_usage):,}")
print(f"T1 records linked to one T0 record:             {observed_linked_t1:,}")
print(f"T1 records not linked to the T0 cohort:         {observed_unlinked_t1:,}")

print()
print("Final linkage status:")
print(
    final_linkage[
        "final_linkage_status"
    ]
    .value_counts(dropna=False)
    .to_string()
)

print()
print("T1 usage status:")
print(
    t1_usage[
        "t1_linkage_usage_status"
    ]
    .value_counts(dropna=False)
    .to_string()
)

print()
print(f"Frozen policy:         {POLICY_PATH}")
print(f"Policy SHA-256:        {policy_sha256}")
print(f"Final linkage:         {FINAL_LINKAGE_PATH}")
print(f"Linkage SHA-256:       {final_linkage_sha256}")
print(f"T1 usage audit:        {T1_USAGE_PATH}")
print(f"T1 usage SHA-256:      {t1_usage_sha256}")
print(f"Freeze report:         {REPORT_PATH}")
print(f"Report SHA-256:        {report_sha256}")
print(f"Freeze manifest:       {MANIFEST_PATH}")
print(f"Manifest SHA-256:      {manifest_sha256}")

print()
print("PASS — Stage 3 final T0–T1 linkage artifact assembled and frozen.")
print(
    "DECISION — ACCEPTED_AND_FROZEN_STAGE3_LINKAGE_COMPLETE"
)
print(
    "IMPORTANT — The 1,076 unresolved records are censored/unresolved, "
    "not stable."
)
print(
    "IMPORTANT — No classification comparison or future-instability "
    "outcome was created."
)
print(
    "BLUEPRINT STAGE 3 — COMPLETE."
)
print(
    "AUTHORIZED NEXT ACTION — Blueprint Stage 4A: freeze the T0-only "
    "GES feature, weak-label, model, and no-star ablation specification."
)
print("=" * 128)

STAGE 3H — FINAL T0–T1 LINKAGE ASSEMBLY AND FREEZE
Verifying immutable Stage 2 and Stage 3 input artifacts...
PASS  t0_rcv_target_genes_corrected_v1_2.parquet
      f6b6760b2ad6e4352e3bdecdeaf89827e8a514b031abf2373bb17568d999466d
PASS  t1_rcv_target_genes_harmonized_v1.parquet
      5713a11bdbf4804758cc011f9b2f302afc91fa1f88c1b178d675c28bb277d37c
PASS  stage3_t0_t1_exact_rcv_crosswalk_draft_v1.parquet
      b462304a4bb31db301e2dac3aefbf685e12f1fea179a38ca04b8500b64bb4acc
PASS  stage3_t0_t1_exact_match_continuity_audit_v1.parquet
      7671792ebccae68ff77745355498d174e0a614f21703e518fb230ebfc54666ad
PASS  stage3_accepted_nonexact_one_to_one_links_v1.parquet
      19e9eb00810f8b3faf2147c3a73895d0a939068beb20b3aaaf9a8bd8a8d64268
PASS  stage3_t0_post_strong_candidate_adjudication_v1.parquet
      c1e40c92389e2dc200cd1508c49fb7aa12c3a9665fc2e6aab56ed2127accb65e
PASS  stage3_strong_candidate_adjudication_manifest_v1.json
      b5ea53ba3fe6cafdb2abaa25ac582d683cfb35dd04d88dcfdaf03802edec4304


RuntimeError: Checksum mismatch for:
/content/drive/MyDrive/GES_RAG_Temporal_Study/configs/stage3_crosswalk/stage3_targeted_complex_and_tier3_audit_manifest_v1.json
Expected: 004292519a8a33d261c9a57acdbd6573fc074338a2bb311bbd464b8fd6fa47b1
Observed: 806db054a88c14ea3e64474f00a3ad6241297d2efbdfaf3e068601eee25d0788

In [21]:
STAGE 3H — FINAL T0–T1 LINKAGE ASSEMBLY AND FREEZE
================================================================================================================================
Verifying immutable Stage 2 and Stage 3 input artifacts...
PASS  t0_rcv_target_genes_corrected_v1_2.parquet
      f6b6760b2ad6e4352e3bdecdeaf89827e8a514b031abf2373bb17568d999466d
PASS  t1_rcv_target_genes_harmonized_v1.parquet
      5713a11bdbf4804758cc011f9b2f302afc91fa1f88c1b178d675c28bb277d37c
PASS  stage3_t0_t1_exact_rcv_crosswalk_draft_v1.parquet
      b462304a4bb31db301e2dac3aefbf685e12f1fea179a38ca04b8500b64bb4acc
PASS  stage3_t0_t1_exact_match_continuity_audit_v1.parquet
      7671792ebccae68ff77745355498d174e0a614f21703e518fb230ebfc54666ad
PASS  stage3_accepted_nonexact_one_to_one_links_v1.parquet
      19e9eb00810f8b3faf2147c3a73895d0a939068beb20b3aaaf9a8bd8a8d64268
PASS  stage3_t0_post_strong_candidate_adjudication_v1.parquet
      c1e40c92389e2dc200cd1508c49fb7aa12c3a9665fc2e6aab56ed2127accb65e
PASS  stage3_strong_candidate_adjudication_manifest_v1.json
      b5ea53ba3fe6cafdb2abaa25ac582d683cfb35dd04d88dcfdaf03802edec4304
PASS  stage3_targeted_candidate_t0_review_summary_v1.parquet
      09f6e4d24744781127e59d7ee03ddf3d5d02b0c4ea5517e52a184ad46a623d2a
---------------------------------------------------------------------------
RuntimeError                              Traceback (most recent call last)
/tmp/ipykernel_569/2722362482.py in <cell line: 0>()
    357
    358     if observed_sha256 != expected_sha256:
--> 359         raise RuntimeError(
    360             f"Checksum mismatch for:\n{path}\n"
    361             f"Expected: {expected_sha256}\n"

RuntimeError: Checksum mismatch for:
/content/drive/MyDrive/GES_RAG_Temporal_Study/configs/stage3_crosswalk/stage3_targeted_complex_and_tier3_audit_manifest_v1.json
Expected: 004292519a8a33d261c9a57acdbd6573fc074338a2bb311bbd464b8fd6fa47b1
Observed: 806db054a88c14ea3e64474f00a3ad6241297d2efbdfaf3e068601eee25d0788


SyntaxError: invalid decimal literal (1729195565.py, line 1)

In [22]:
# =================================================================================================
# STAGE 3H — ASSEMBLE, VALIDATE, AND FREEZE THE FINAL T0–T1 LINKAGE ARTIFACT
# =================================================================================================
# Purpose:
#   Assemble one final linkage row for every frozen T0 RCV-level variant-condition record.
#
# Final accepted linkage:
#   1. Exact RCV accession linkage from Stage 3B.
#   2. Conservative one-to-one non-exact continuity links accepted in Stage 3E.
#
# Final unresolved/censored categories:
#   1. Non-unique or complex candidate relationships.
#   2. Variant continuity with unresolved condition-association continuity.
#   3. No stable VariationID/VCV-based T1 linkage.
#
# Scientific boundary:
#   Unresolved or censored records are NOT stable records.
#
# This cell:
#   - freezes the final linkage/censoring policy;
#   - assembles one row for each of the 71,659 T0 records;
#   - attaches the accepted T1 record where a defensible link exists;
#   - retains Stage 3C–3G audit context;
#   - audits T1 usage and one-to-one linkage integrity;
#   - writes checksum-controlled final artifacts and a freeze manifest.
#
# This cell DOES NOT:
#   - construct future-instability outcomes;
#   - compare T0 and T1 classifications;
#   - label unresolved records stable;
#   - confirm official ClinVar replacement, withdrawal, merge, or split history;
#   - fit, tune, or evaluate GES.
# =================================================================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import re

import pandas as pd
import pyarrow.parquet as pq


print("=" * 128)
print("STAGE 3H — FINAL T0–T1 LINKAGE ASSEMBLY AND FREEZE")
print("=" * 128)


# -------------------------------------------------------------------------------------------------
# 1. PATHS
# -------------------------------------------------------------------------------------------------

STUDY_ROOT = Path("/content/drive/MyDrive/GES_RAG_Temporal_Study")

DATA_INTERIM_DIR = STUDY_ROOT / "data_interim"
STAGE3_CONFIG_DIR = STUDY_ROOT / "configs" / "stage3_crosswalk"
STAGE3_OUTPUT_DIR = STUDY_ROOT / "data_processed" / "stage3_crosswalk"

T0_PATH = (
    DATA_INTERIM_DIR
    / "t0_rcv_target_genes_corrected_v1_2.parquet"
)

T1_PATH = (
    DATA_INTERIM_DIR
    / "t1_rcv_target_genes_harmonized_v1.parquet"
)

STAGE3B_CROSSWALK_PATH = (
    STAGE3_OUTPUT_DIR
    / "stage3_t0_t1_exact_rcv_crosswalk_draft_v1.parquet"
)

STAGE3C_AUDIT_PATH = (
    STAGE3_OUTPUT_DIR
    / "stage3_t0_t1_exact_match_continuity_audit_v1.parquet"
)

STAGE3E_ACCEPTED_LINKS_PATH = (
    STAGE3_OUTPUT_DIR
    / "stage3_accepted_nonexact_one_to_one_links_v1.parquet"
)

STAGE3E_T0_STATUS_PATH = (
    STAGE3_OUTPUT_DIR
    / "stage3_t0_post_strong_candidate_adjudication_v1.parquet"
)

STAGE3E_MANIFEST_PATH = (
    STAGE3_CONFIG_DIR
    / "stage3_strong_candidate_adjudication_manifest_v1.json"
)

STAGE3F_T0_REVIEW_PATH = (
    STAGE3_OUTPUT_DIR
    / "stage3_targeted_candidate_t0_review_summary_v1.parquet"
)

STAGE3F_MANIFEST_PATH = (
    STAGE3_CONFIG_DIR
    / "stage3_targeted_complex_and_tier3_audit_manifest_v1.json"
)

STAGE3G_T0_AUDIT_PATH = (
    STAGE3_OUTPUT_DIR
    / "stage3_no_identifier_t0_audit_summary_v1.parquet"
)

STAGE3G_MANIFEST_PATH = (
    STAGE3_CONFIG_DIR
    / "stage3_no_identifier_condition_only_audit_manifest_v1.json"
)


POLICY_PATH = (
    STAGE3_CONFIG_DIR
    / "stage3_final_linkage_and_censoring_policy_v1.json"
)

FINAL_LINKAGE_PATH = (
    STAGE3_OUTPUT_DIR
    / "stage3_t0_t1_final_linkage_frozen_v1.parquet"
)

T1_USAGE_PATH = (
    STAGE3_OUTPUT_DIR
    / "stage3_t1_linkage_usage_audit_v1.parquet"
)

REPORT_PATH = (
    STAGE3_CONFIG_DIR
    / "stage3_final_linkage_freeze_report_v1.json"
)

MANIFEST_PATH = (
    STAGE3_CONFIG_DIR
    / "stage3_final_linkage_freeze_manifest_v1.json"
)


# -------------------------------------------------------------------------------------------------
# 2. EXPECTED IMMUTABLE INPUT CHECKSUMS
# -------------------------------------------------------------------------------------------------

EXPECTED_CHECKSUMS = {
    T0_PATH: (
        "f6b6760b2ad6e4352e3bdecdeaf89827"
        "e8a514b031abf2373bb17568d999466d"
    ),
    T1_PATH: (
        "5713a11bdbf4804758cc011f9b2f302a"
        "fc91fa1f88c1b178d675c28bb277d37c"
    ),
    STAGE3B_CROSSWALK_PATH: (
        "b462304a4bb31db301e2dac3aefbf685"
        "e12f1fea179a38ca04b8500b64bb4acc"
    ),
    STAGE3C_AUDIT_PATH: (
        "7671792ebccae68ff77745355498d174e"
        "0a614f21703e518fb230ebfc54666ad"
    ),
    STAGE3E_ACCEPTED_LINKS_PATH: (
        "19e9eb00810f8b3faf2147c3a73895d0"
        "a939068beb20b3aaaf9a8bd8a8d64268"
    ),
    STAGE3E_T0_STATUS_PATH: (
        "c1e40c92389e2dc200cd1508c49fb7a"
        "a12c3a9665fc2e6aab56ed2127accb65e"
    ),
    STAGE3E_MANIFEST_PATH: (
        "b5ea53ba3fe6cafdb2abaa25ac582d683"
        "cfb35dd04d88dcfdaf03802edec4304"
    ),
    STAGE3F_T0_REVIEW_PATH: (
        "09f6e4d24744781127e59d7ee03ddf3d"
        "5d02b0c4ea5517e52a184ad46a623d2a"
    ),
    STAGE3F_MANIFEST_PATH: (
        "004292519a8a33d261c9a57acdbd6573"
        "fc074338a2bb311bbd464b8fd6fa47b1"
    ),
    STAGE3G_T0_AUDIT_PATH: (
        "7d3bfcd3428e72f7c48a3ed9aa954126"
        "fc7dd0eb4a1daec1a24ff1173670d776"
    ),
    STAGE3G_MANIFEST_PATH: (
        "99bef658295e208c650a7e693df0c70b"
        "54221557f3caa1b0d8a266666c5a72d2"
    ),
}


EXPECTED_T0_ROWS = 71659
EXPECTED_T1_ROWS = 100920

EXPECTED_EXACT_LINKS = 70413
EXPECTED_ACCEPTED_NONEXACT_LINKS = 170
EXPECTED_TOTAL_LINKED_T0 = 70583

EXPECTED_UNRESOLVED_COMPLEX = 38
EXPECTED_UNRESOLVED_CONDITION = 416
EXPECTED_UNRESOLVED_NO_IDENTIFIER = 622
EXPECTED_TOTAL_UNRESOLVED = 1076

EXPECTED_LINKED_T1 = 70583
EXPECTED_UNLINKED_T1 = 30337


# -------------------------------------------------------------------------------------------------
# 3. UTILITY FUNCTIONS
# -------------------------------------------------------------------------------------------------

def sha256_file(path: Path, chunk_size: int = 8 * 1024 * 1024) -> str:
    """Calculate a file SHA-256 without loading the complete file into memory."""
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def normalize_rcv(value):
    """Normalize one RCV accession."""
    if value is None:
        return None

    try:
        if pd.isna(value):
            return None
    except (TypeError, ValueError):
        pass

    text = str(value).strip().upper()

    return text or None


def normalize_identifier(value):
    """Normalize identifiers and remove accidental numeric '.0' suffixes."""
    if value is None:
        return None

    try:
        if pd.isna(value):
            return None
    except (TypeError, ValueError):
        pass

    text = str(value).strip().upper()

    if re.fullmatch(r"\d+\.0", text):
        text = text[:-2]

    return text or None


def value_counts_dict(series: pd.Series) -> dict:
    """Convert pandas value counts to a JSON-safe dictionary."""
    counts = series.value_counts(dropna=False)

    result = {}

    for key, value in counts.items():
        safe_key = "<MISSING>" if pd.isna(key) else str(key)
        result[safe_key] = int(value)

    return result


def prefix_columns(frame: pd.DataFrame, prefix: str, key: str) -> pd.DataFrame:
    """Prefix all columns except one normalized join key."""
    rename_map = {
        column: f"{prefix}_{column}"
        for column in frame.columns
        if column != key
    }

    return frame.rename(columns=rename_map)


def stage3c_axis_category(gene_json, axis_value) -> str:
    """
    Create a descriptive gene-aware axis category for accepted non-exact links.

    This does not infer a historical T0 classification axis.
    """
    try:
        genes = json.loads(gene_json)
    except (TypeError, json.JSONDecodeError):
        genes = []

    genes = {
        str(gene).strip().upper()
        for gene in genes
        if str(gene).strip()
    }

    axis = str(axis_value).strip() if pd.notna(axis_value) else "MissingAxis"

    if len(genes) != 1:
        return "MULTIPLE_OR_MISSING_GENE_AXIS_REVIEW"

    gene = next(iter(genes))

    if gene in {"BRCA1", "BRCA2", "MLH1"}:
        if axis == "GermlineClassification":
            return "PRIMARY_GENE_GERMLINE_AXIS"

        if axis in {"NoClassification", "MissingAxis", ""}:
            return "PRIMARY_GENE_NO_T1_CLASSIFICATION"

        return "PRIMARY_GENE_NON_GERMLINE_AXIS_REVIEW"

    if gene == "EGFR":
        if axis == "OncogenicityClassification":
            return "EGFR_ONCOGENICITY_AXIS"

        if axis == "SomaticClinicalImpact":
            return "EGFR_SOMATIC_CLINICAL_IMPACT_AXIS"

        if axis == "GermlineClassification":
            return "EGFR_GERMLINE_AXIS"

        if axis in {"NoClassification", "MissingAxis", ""}:
            return "EGFR_NO_T1_CLASSIFICATION"

        return "EGFR_OTHER_AXIS_REVIEW"

    return "UNEXPECTED_GENE_AXIS_REVIEW"


# -------------------------------------------------------------------------------------------------
# 4. VERIFY ALL IMMUTABLE INPUT ARTIFACTS
# -------------------------------------------------------------------------------------------------

print("Verifying immutable Stage 2 and Stage 3 input artifacts...")

observed_checksums = {}

for path, expected_sha256 in EXPECTED_CHECKSUMS.items():

    if not path.exists():
        raise FileNotFoundError(
            f"Required input artifact does not exist: {path}"
        )

    observed_sha256 = sha256_file(path)
    observed_checksums[str(path)] = observed_sha256

    if observed_sha256 != expected_sha256:
        raise RuntimeError(
            f"Checksum mismatch for:\n{path}\n"
            f"Expected: {expected_sha256}\n"
            f"Observed: {observed_sha256}"
        )

    print(f"PASS  {path.name}")
    print(f"      {observed_sha256}")


# -------------------------------------------------------------------------------------------------
# 5. VERIFY AUTHORIZING MANIFEST STATUSES
# -------------------------------------------------------------------------------------------------

with STAGE3E_MANIFEST_PATH.open("r", encoding="utf-8") as handle:
    stage3e_manifest = json.load(handle)

with STAGE3F_MANIFEST_PATH.open("r", encoding="utf-8") as handle:
    stage3f_manifest = json.load(handle)

with STAGE3G_MANIFEST_PATH.open("r", encoding="utf-8") as handle:
    stage3g_manifest = json.load(handle)

if (
    stage3e_manifest.get("adjudication_status")
    != "CONSERVATIVE_STRONG_CANDIDATE_ADJUDICATION_COMPLETE"
):
    raise RuntimeError(
        "Stage 3E does not contain the expected completed status."
    )

if (
    stage3f_manifest.get("audit_status")
    != "TARGETED_COMPLEX_AND_TIER3_AUDIT_COMPLETE"
):
    raise RuntimeError(
        "Stage 3F does not contain the expected completed status."
    )

if (
    stage3g_manifest.get("audit_status")
    != "NO_STABLE_IDENTIFIER_CONDITION_ONLY_AUDIT_COMPLETE"
):
    raise RuntimeError(
        "Stage 3G does not contain the expected completed status."
    )


# -------------------------------------------------------------------------------------------------
# 6. FREEZE THE FINAL LINKAGE AND CENSORING POLICY
# -------------------------------------------------------------------------------------------------

policy = {
    "policy_name": "Stage 3 Final T0-T1 Linkage and Censoring Policy",
    "policy_version": "1.0.0",
    "scientific_unit": "RCV-level variant-condition aggregate",

    "accepted_linkage_rules": {
        "exact_link": {
            "rule": "Exact normalized RCV accession match",
            "one_to_one_required": True,
            "accepted_status": "EXACT_RCV_MATCH",
        },
        "conservative_nonexact_link": {
            "rule": (
                "Stage 3E unique one-to-one Tier 1 continuity link with "
                "exact VariationID, exact VCV accession, exact target gene, "
                "and condition overlap"
            ),
            "one_to_one_required": True,
            "accepted_status": (
                "ACCEPTED_NONEXACT_ONE_TO_ONE_CONTINUITY"
            ),
            "official_rcv_replacement_claimed": False,
        },
    },

    "unresolved_censoring_rules": {
        "UNRESOLVED_COMPLEX_STRONG_CANDIDATE_RELATIONSHIP": (
            "CENSORED_NONUNIQUE_OR_COMPLEX_LINKAGE"
        ),
        "UNRESOLVED_VARIANT_CONTINUITY_WITHOUT_CONDITION_OVERLAP": (
            "CENSORED_VARIANT_CONTINUITY_WITH_UNRESOLVED_CONDITION_ASSOCIATION"
        ),
        "UNRESOLVED_NO_VARIATIONID_OR_VCV_CANDIDATE": (
            "CENSORED_NO_STABLE_VARIANT_IDENTIFIER_LINK"
        ),
    },

    "primary_analysis_rule": (
        "Only accepted one-to-one linked records are linkage-eligible for "
        "subsequent temporal-outcome construction. Additional classification-axis, "
        "classification-availability, and outcome-specific eligibility rules remain "
        "to be applied later."
    ),

    "unmatched_record_rule": (
        "Unresolved, unmatched, or censored T0 records must not be assigned "
        "a stable outcome because no defensible one-to-one T1 comparison exists."
    ),

    "condition_only_rule": (
        "Condition-only candidates are descriptive audit evidence and cannot "
        "establish variant continuity."
    ),

    "prohibited_operations_at_stage3": [
        "No future-instability outcome construction",
        "No stable or unstable label assignment",
        "No classification-change calculation",
        "No GES fitting or tuning",
        "No confirmation of official ClinVar replacement history",
        "No automatic acceptance of merge, split, or many-to-many relationships",
    ],
}


if POLICY_PATH.exists():

    with POLICY_PATH.open("r", encoding="utf-8") as handle:
        existing_policy = json.load(handle)

    if existing_policy != policy:
        raise RuntimeError(
            "An existing Stage 3 linkage policy differs from the current policy. "
            "Do not overwrite it silently."
        )

else:

    with POLICY_PATH.open("w", encoding="utf-8") as handle:
        json.dump(
            policy,
            handle,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
        )


policy_sha256 = sha256_file(POLICY_PATH)

print()
print(f"Frozen policy:       {POLICY_PATH}")
print(f"Policy SHA-256:      {policy_sha256}")


# -------------------------------------------------------------------------------------------------
# 7. LOAD THE FROZEN T0 AND T1 COHORTS
# -------------------------------------------------------------------------------------------------

source_columns = [
    "release_label",
    "embedded_data_cutoff_date",
    "rcv_accession",
    "rcv_version",
    "variation_id",
    "vcv_accession",
    "vcv_version",
    "target_genes_json",
    "study_scope",
    "condition_names_json",
    "condition_ids_json",
]

t1_source_columns = source_columns + [
    "aggregate_classification_axis",
]

t0 = pd.read_parquet(
    T0_PATH,
    columns=source_columns,
)

t1 = pd.read_parquet(
    T1_PATH,
    columns=t1_source_columns,
)

if len(t0) != EXPECTED_T0_ROWS:
    raise AssertionError(
        f"Expected {EXPECTED_T0_ROWS:,} T0 rows, observed {len(t0):,}."
    )

if len(t1) != EXPECTED_T1_ROWS:
    raise AssertionError(
        f"Expected {EXPECTED_T1_ROWS:,} T1 rows, observed {len(t1):,}."
    )

t0["t0_rcv_key"] = [
    normalize_rcv(value)
    for value in t0["rcv_accession"]
]

t1["t1_rcv_key"] = [
    normalize_rcv(value)
    for value in t1["rcv_accession"]
]

if t0["t0_rcv_key"].isna().any():
    raise AssertionError("T0 contains a missing normalized RCV key.")

if t1["t1_rcv_key"].isna().any():
    raise AssertionError("T1 contains a missing normalized RCV key.")

if t0["t0_rcv_key"].duplicated().any():
    raise AssertionError("T0 contains duplicate normalized RCV keys.")

if t1["t1_rcv_key"].duplicated().any():
    raise AssertionError("T1 contains duplicate normalized RCV keys.")

t0 = prefix_columns(
    t0,
    prefix="t0",
    key="t0_rcv_key",
)

t1 = prefix_columns(
    t1,
    prefix="t1",
    key="t1_rcv_key",
)


# -------------------------------------------------------------------------------------------------
# 8. LOAD EXACT RCV LINKS
# -------------------------------------------------------------------------------------------------

exact_crosswalk = pd.read_parquet(
    STAGE3B_CROSSWALK_PATH,
    columns=[
        "linkage_status",
        "t0_rcv_accession",
        "t1_rcv_accession",
    ],
)

exact_links = (
    exact_crosswalk.loc[
        exact_crosswalk["linkage_status"].eq("EXACT_RCV_MATCH")
    ]
    .copy()
    .reset_index(drop=True)
)

exact_links["t0_rcv_key"] = [
    normalize_rcv(value)
    for value in exact_links["t0_rcv_accession"]
]

exact_links["linked_t1_rcv_key"] = [
    normalize_rcv(value)
    for value in exact_links["t1_rcv_accession"]
]

exact_links = exact_links[
    [
        "t0_rcv_key",
        "linked_t1_rcv_key",
    ]
].copy()

exact_links["final_linkage_status"] = "EXACT_RCV_MATCH"
exact_links["linkage_method"] = "EXACT_RCV_ACCESSION"
exact_links["accepted_nonexact_candidate_pair_id"] = None
exact_links["accepted_nonexact_condition_evidence"] = None

if len(exact_links) != EXPECTED_EXACT_LINKS:
    raise AssertionError(
        f"Expected {EXPECTED_EXACT_LINKS:,} exact links, "
        f"observed {len(exact_links):,}."
    )

if exact_links["t0_rcv_key"].duplicated().any():
    raise AssertionError("Duplicate T0 keys exist among exact links.")

if exact_links["linked_t1_rcv_key"].duplicated().any():
    raise AssertionError("Duplicate T1 keys exist among exact links.")


# -------------------------------------------------------------------------------------------------
# 9. LOAD ACCEPTED CONSERVATIVE NON-EXACT LINKS
# -------------------------------------------------------------------------------------------------

accepted_nonexact = pd.read_parquet(
    STAGE3E_ACCEPTED_LINKS_PATH,
)

if len(accepted_nonexact) != EXPECTED_ACCEPTED_NONEXACT_LINKS:
    raise AssertionError(
        f"Expected {EXPECTED_ACCEPTED_NONEXACT_LINKS:,} accepted non-exact links, "
        f"observed {len(accepted_nonexact):,}."
    )

accepted_nonexact["t0_rcv_key"] = [
    normalize_rcv(value)
    for value in accepted_nonexact["t0_rcv_accession"]
]

accepted_nonexact["linked_t1_rcv_key"] = [
    normalize_rcv(value)
    for value in accepted_nonexact["t1_rcv_accession"]
]

if accepted_nonexact["t0_rcv_key"].duplicated().any():
    raise AssertionError(
        "Duplicate T0 keys exist among accepted non-exact links."
    )

if accepted_nonexact["linked_t1_rcv_key"].duplicated().any():
    raise AssertionError(
        "Duplicate T1 keys exist among accepted non-exact links."
    )

accepted_links = accepted_nonexact[
    [
        "t0_rcv_key",
        "linked_t1_rcv_key",
        "candidate_pair_id",
        "condition_evidence_category",
    ]
].copy()

accepted_links = accepted_links.rename(
    columns={
        "candidate_pair_id": (
            "accepted_nonexact_candidate_pair_id"
        ),
        "condition_evidence_category": (
            "accepted_nonexact_condition_evidence"
        ),
    }
)

accepted_links["final_linkage_status"] = (
    "ACCEPTED_NONEXACT_ONE_TO_ONE_CONTINUITY"
)

accepted_links["linkage_method"] = (
    "CONSERVATIVE_VARIATIONID_VCV_GENE_CONDITION_CONTINUITY"
)


# -------------------------------------------------------------------------------------------------
# 10. COMBINE ACCEPTED LINKS AND VALIDATE ONE-TO-ONE INTEGRITY
# -------------------------------------------------------------------------------------------------

link_map = pd.concat(
    [
        exact_links,
        accepted_links,
    ],
    ignore_index=True,
)

if len(link_map) != EXPECTED_TOTAL_LINKED_T0:
    raise AssertionError(
        f"Expected {EXPECTED_TOTAL_LINKED_T0:,} accepted links, "
        f"observed {len(link_map):,}."
    )

if link_map["t0_rcv_key"].duplicated().any():
    raise AssertionError(
        "A T0 record received more than one accepted final link."
    )

if link_map["linked_t1_rcv_key"].duplicated().any():
    raise AssertionError(
        "A T1 record was assigned to more than one T0 record."
    )

t0_source_keys = set(t0["t0_rcv_key"])
t1_source_keys = set(t1["t1_rcv_key"])

missing_t0_link_keys = (
    set(link_map["t0_rcv_key"]) - t0_source_keys
)

missing_t1_link_keys = (
    set(link_map["linked_t1_rcv_key"]) - t1_source_keys
)

if missing_t0_link_keys:
    raise AssertionError(
        f"{len(missing_t0_link_keys):,} accepted T0 keys are absent "
        "from the frozen T0 cohort."
    )

if missing_t1_link_keys:
    raise AssertionError(
        f"{len(missing_t1_link_keys):,} accepted T1 keys are absent "
        "from the frozen T1 cohort."
    )


# -------------------------------------------------------------------------------------------------
# 11. LOAD STAGE 3C–3G AUDIT CONTEXT
# -------------------------------------------------------------------------------------------------

stage3c = pd.read_parquet(
    STAGE3C_AUDIT_PATH,
    columns=[
        "t0_rcv_accession",
        "core_identifier_continuity_status",
        "gene_set_relation",
        "condition_continuity_status",
        "classification_axis_audit_category",
        "requires_linkage_review",
        "review_reason_count",
        "review_reasons_json",
    ],
)

stage3c["t0_rcv_key"] = [
    normalize_rcv(value)
    for value in stage3c["t0_rcv_accession"]
]

stage3c = stage3c.drop(
    columns=["t0_rcv_accession"]
)

if len(stage3c) != EXPECTED_EXACT_LINKS:
    raise AssertionError(
        "Stage 3C exact-match audit row count is unexpected."
    )

if stage3c["t0_rcv_key"].duplicated().any():
    raise AssertionError(
        "Stage 3C contains duplicate T0 linkage keys."
    )


stage3e_status = pd.read_parquet(
    STAGE3E_T0_STATUS_PATH,
    columns=[
        "t0_rcv_accession",
        "stage3e_post_adjudication_status",
        "identifier_based_candidate_count",
        "strong_candidate_count",
        "accepted_stage3e_t1_rcv_accession",
    ],
)

stage3e_status["t0_rcv_key"] = [
    normalize_rcv(value)
    for value in stage3e_status["t0_rcv_accession"]
]

stage3e_status = stage3e_status.drop(
    columns=["t0_rcv_accession"]
)

if stage3e_status["t0_rcv_key"].duplicated().any():
    raise AssertionError(
        "Stage 3E T0 status contains duplicate T0 keys."
    )


stage3f_review = pd.read_parquet(
    STAGE3F_T0_REVIEW_PATH,
    columns=[
        "t0_rcv_accession",
        "stage3f_targeted_candidate_pair_count",
        "stage3f_component_count",
        "stage3f_t0_review_category",
        "stage3f_resolution_status",
    ],
)

stage3f_review["t0_rcv_key"] = [
    normalize_rcv(value)
    for value in stage3f_review["t0_rcv_accession"]
]

stage3f_review = stage3f_review.drop(
    columns=["t0_rcv_accession"]
)

if stage3f_review["t0_rcv_key"].duplicated().any():
    raise AssertionError(
        "Stage 3F T0 review contains duplicate T0 keys."
    )


stage3g_audit = pd.read_parquet(
    STAGE3G_T0_AUDIT_PATH,
    columns=[
        "t0_rcv_accession",
        "condition_only_candidate_count",
        "exact_condition_set_candidate_count",
        "unique_condition_only_candidate_t1_rcv",
        "stage3g_t0_audit_category",
        "stage3g_source_history_review_priority",
        "stage3g_resolution_status",
    ],
)

stage3g_audit["t0_rcv_key"] = [
    normalize_rcv(value)
    for value in stage3g_audit["t0_rcv_accession"]
]

stage3g_audit = stage3g_audit.drop(
    columns=["t0_rcv_accession"]
)

if stage3g_audit["t0_rcv_key"].duplicated().any():
    raise AssertionError(
        "Stage 3G T0 audit contains duplicate T0 keys."
    )


# -------------------------------------------------------------------------------------------------
# 12. ASSEMBLE ONE FINAL ROW FOR EVERY T0 RECORD
# -------------------------------------------------------------------------------------------------

final_linkage = (
    t0
    .merge(
        link_map,
        on="t0_rcv_key",
        how="left",
        validate="one_to_one",
    )
    .merge(
        t1,
        left_on="linked_t1_rcv_key",
        right_on="t1_rcv_key",
        how="left",
        validate="many_to_one",
    )
    .merge(
        stage3c,
        on="t0_rcv_key",
        how="left",
        validate="one_to_one",
    )
    .merge(
        stage3e_status,
        on="t0_rcv_key",
        how="left",
        validate="one_to_one",
    )
    .merge(
        stage3f_review,
        on="t0_rcv_key",
        how="left",
        validate="one_to_one",
    )
    .merge(
        stage3g_audit,
        on="t0_rcv_key",
        how="left",
        validate="one_to_one",
    )
)


if len(final_linkage) != EXPECTED_T0_ROWS:
    raise AssertionError(
        "Final linkage does not contain exactly one row per T0 record."
    )

if final_linkage["t0_rcv_key"].duplicated().any():
    raise AssertionError(
        "Final linkage contains duplicate T0 keys."
    )


# -------------------------------------------------------------------------------------------------
# 13. ASSIGN FINAL LINKAGE AND CENSORING DISPOSITIONS
# -------------------------------------------------------------------------------------------------

final_linkage["final_linkage_status"] = (
    final_linkage["final_linkage_status"]
    .fillna("UNRESOLVED_CENSORED")
)

final_linkage["linkage_method"] = (
    final_linkage["linkage_method"]
    .fillna("NO_ACCEPTED_ONE_TO_ONE_T1_LINK")
)

final_linkage["linked_t1_record_present"] = (
    final_linkage["linked_t1_rcv_key"].notna()
)

final_linkage["one_to_one_linkage_accepted"] = (
    final_linkage["final_linkage_status"].isin(
        [
            "EXACT_RCV_MATCH",
            "ACCEPTED_NONEXACT_ONE_TO_ONE_CONTINUITY",
        ]
    )
)

final_linkage[
    "linkage_eligible_for_future_outcome_construction"
] = final_linkage["one_to_one_linkage_accepted"]


unresolved_disposition_map = {
    "UNRESOLVED_COMPLEX_STRONG_CANDIDATE_RELATIONSHIP": (
        "CENSORED_NONUNIQUE_OR_COMPLEX_LINKAGE"
    ),
    "UNRESOLVED_VARIANT_CONTINUITY_WITHOUT_CONDITION_OVERLAP": (
        "CENSORED_VARIANT_CONTINUITY_WITH_UNRESOLVED_CONDITION_ASSOCIATION"
    ),
    "UNRESOLVED_NO_VARIATIONID_OR_VCV_CANDIDATE": (
        "CENSORED_NO_STABLE_VARIANT_IDENTIFIER_LINK"
    ),
}

final_linkage["final_unresolved_disposition"] = (
    final_linkage["stage3e_post_adjudication_status"]
    .map(unresolved_disposition_map)
)

linked_mask = final_linkage["one_to_one_linkage_accepted"]
unresolved_mask = ~linked_mask

final_linkage.loc[
    linked_mask,
    "final_unresolved_disposition",
] = None

if final_linkage.loc[
    unresolved_mask,
    "final_unresolved_disposition",
].isna().any():

    missing_statuses = (
        final_linkage.loc[
            unresolved_mask
            & final_linkage[
                "final_unresolved_disposition"
            ].isna(),
            "stage3e_post_adjudication_status",
        ]
        .value_counts(dropna=False)
        .to_dict()
    )

    raise AssertionError(
        "At least one unresolved T0 record lacks a frozen censoring "
        f"disposition: {missing_statuses}"
    )


# -------------------------------------------------------------------------------------------------
# 14. STANDARDIZE AUDIT CONTEXT FOR EXACT AND NON-EXACT LINKS
# -------------------------------------------------------------------------------------------------

exact_mask = final_linkage[
    "final_linkage_status"
].eq("EXACT_RCV_MATCH")

nonexact_mask = final_linkage[
    "final_linkage_status"
].eq(
    "ACCEPTED_NONEXACT_ONE_TO_ONE_CONTINUITY"
)


final_linkage.loc[
    exact_mask,
    "stage3e_post_adjudication_status",
] = "NOT_APPLICABLE_EXACT_RCV_MATCH"


final_linkage.loc[
    nonexact_mask,
    "core_identifier_continuity_status",
] = "STABLE_CORE_IDENTIFIERS"

final_linkage.loc[
    nonexact_mask,
    "gene_set_relation",
] = "EXACT_SET_MATCH"

final_linkage.loc[
    nonexact_mask,
    "condition_continuity_status",
] = (
    "ACCEPTED_NONEXACT_CONDITION_OVERLAP"
)

final_linkage.loc[
    nonexact_mask,
    "classification_axis_audit_category",
] = [
    stage3c_axis_category(gene_json, axis_value)
    for gene_json, axis_value in zip(
        final_linkage.loc[
            nonexact_mask,
            "t1_target_genes_json",
        ],
        final_linkage.loc[
            nonexact_mask,
            "t1_aggregate_classification_axis",
        ],
    )
]

final_linkage.loc[
    nonexact_mask,
    "requires_linkage_review",
] = False

final_linkage.loc[
    nonexact_mask,
    "review_reason_count",
] = 0

final_linkage.loc[
    nonexact_mask,
    "review_reasons_json",
] = "[]"


# -------------------------------------------------------------------------------------------------
# 15. ADD EXPLICIT SCIENTIFIC-BOUNDARY FIELDS
# -------------------------------------------------------------------------------------------------

final_linkage["linkage_policy_version"] = policy["policy_version"]
final_linkage["linkage_policy_sha256"] = policy_sha256

final_linkage["stage3_linkage_frozen"] = True
final_linkage["future_instability_outcome_created"] = False
final_linkage["future_instability_label"] = None

final_linkage["unresolved_record_labeled_stable"] = False
final_linkage["condition_only_candidate_used_for_linkage"] = False

final_linkage["classification_change_calculated"] = False
final_linkage["official_rcv_replacement_confirmed"] = False

final_linkage["outcome_construction_status"] = (
    "NOT_STARTED_STAGE3_LINKAGE_ONLY"
)

final_linkage["final_crosswalk_record_id"] = (
    final_linkage["t0_rcv_key"]
)


# -------------------------------------------------------------------------------------------------
# 16. VALIDATE LINKED IDENTIFIER CONTINUITY
# -------------------------------------------------------------------------------------------------

linked = final_linkage.loc[
    linked_mask
].copy()

linked_variation_mismatches = int(
    (
        linked["t0_variation_id"]
        .map(normalize_identifier)
        != linked["t1_variation_id"]
        .map(normalize_identifier)
    ).sum()
)

linked_vcv_mismatches = int(
    (
        linked["t0_vcv_accession"]
        .map(normalize_identifier)
        != linked["t1_vcv_accession"]
        .map(normalize_identifier)
    ).sum()
)

if linked_variation_mismatches != 0:
    raise AssertionError(
        f"{linked_variation_mismatches:,} accepted links have "
        "VariationID discontinuity."
    )

if linked_vcv_mismatches != 0:
    raise AssertionError(
        f"{linked_vcv_mismatches:,} accepted links have "
        "VCV-accession discontinuity."
    )

if linked["linked_t1_rcv_key"].isna().any():
    raise AssertionError(
        "At least one accepted link has no linked T1 RCV."
    )

if linked["t1_rcv_accession"].isna().any():
    raise AssertionError(
        "At least one accepted link failed to retrieve its frozen T1 record."
    )

if linked["linked_t1_rcv_key"].duplicated().any():
    raise AssertionError(
        "The final linked cohort contains duplicate T1 assignments."
    )

if final_linkage.loc[
    unresolved_mask,
    "linked_t1_rcv_key",
].notna().any():
    raise AssertionError(
        "An unresolved record contains an accepted T1 linkage key."
    )


# -------------------------------------------------------------------------------------------------
# 17. VALIDATE FINAL STATUS COUNTS
# -------------------------------------------------------------------------------------------------

final_status_counts = final_linkage[
    "final_linkage_status"
].value_counts()

observed_exact = int(
    final_status_counts.get("EXACT_RCV_MATCH", 0)
)

observed_nonexact = int(
    final_status_counts.get(
        "ACCEPTED_NONEXACT_ONE_TO_ONE_CONTINUITY",
        0,
    )
)

observed_unresolved = int(
    final_status_counts.get("UNRESOLVED_CENSORED", 0)
)

if observed_exact != EXPECTED_EXACT_LINKS:
    raise AssertionError(
        f"Expected {EXPECTED_EXACT_LINKS:,} exact links, "
        f"observed {observed_exact:,}."
    )

if observed_nonexact != EXPECTED_ACCEPTED_NONEXACT_LINKS:
    raise AssertionError(
        f"Expected {EXPECTED_ACCEPTED_NONEXACT_LINKS:,} accepted non-exact links, "
        f"observed {observed_nonexact:,}."
    )

if observed_unresolved != EXPECTED_TOTAL_UNRESOLVED:
    raise AssertionError(
        f"Expected {EXPECTED_TOTAL_UNRESOLVED:,} unresolved records, "
        f"observed {observed_unresolved:,}."
    )


unresolved_counts = final_linkage.loc[
    unresolved_mask,
    "final_unresolved_disposition",
].value_counts()

if int(
    unresolved_counts.get(
        "CENSORED_NONUNIQUE_OR_COMPLEX_LINKAGE",
        0,
    )
) != EXPECTED_UNRESOLVED_COMPLEX:
    raise AssertionError(
        "Complex/non-unique censoring count does not equal 38."
    )

if int(
    unresolved_counts.get(
        "CENSORED_VARIANT_CONTINUITY_WITH_UNRESOLVED_CONDITION_ASSOCIATION",
        0,
    )
) != EXPECTED_UNRESOLVED_CONDITION:
    raise AssertionError(
        "Unresolved condition-association censoring count does not equal 416."
    )

if int(
    unresolved_counts.get(
        "CENSORED_NO_STABLE_VARIANT_IDENTIFIER_LINK",
        0,
    )
) != EXPECTED_UNRESOLVED_NO_IDENTIFIER:
    raise AssertionError(
        "No-stable-identifier censoring count does not equal 622."
    )


# -------------------------------------------------------------------------------------------------
# 18. CREATE T1-SIDE USAGE AUDIT
# -------------------------------------------------------------------------------------------------

exact_t1_keys = set(
    exact_links["linked_t1_rcv_key"]
)

accepted_nonexact_t1_keys = set(
    accepted_links["linked_t1_rcv_key"]
)

if exact_t1_keys.intersection(accepted_nonexact_t1_keys):
    raise AssertionError(
        "A T1 record appears in both exact and accepted non-exact linkage sets."
    )

t1_usage = t1[
    [
        "t1_rcv_key",
        "t1_release_label",
        "t1_embedded_data_cutoff_date",
        "t1_rcv_accession",
        "t1_rcv_version",
        "t1_variation_id",
        "t1_vcv_accession",
        "t1_vcv_version",
        "t1_target_genes_json",
        "t1_study_scope",
        "t1_condition_names_json",
        "t1_condition_ids_json",
        "t1_aggregate_classification_axis",
    ]
].copy()


def t1_usage_status(rcv_key: str) -> str:
    if rcv_key in exact_t1_keys:
        return "LINKED_TO_T0_BY_EXACT_RCV"

    if rcv_key in accepted_nonexact_t1_keys:
        return "LINKED_TO_T0_BY_ACCEPTED_NONEXACT_CONTINUITY"

    return "T1_NOT_LINKED_TO_T0"


t1_usage["t1_linkage_usage_status"] = [
    t1_usage_status(value)
    for value in t1_usage["t1_rcv_key"]
]

t1_usage["linked_to_one_t0_record"] = (
    t1_usage["t1_linkage_usage_status"]
    .ne("T1_NOT_LINKED_TO_T0")
)

t1_to_t0_map = dict(
    zip(
        link_map["linked_t1_rcv_key"],
        link_map["t0_rcv_key"],
    )
)

t1_usage["linked_t0_rcv_accession"] = [
    t1_to_t0_map.get(value)
    for value in t1_usage["t1_rcv_key"]
]

t1_usage["future_instability_outcome_created"] = False

t1_usage_counts = t1_usage[
    "t1_linkage_usage_status"
].value_counts()

observed_linked_t1 = int(
    t1_usage["linked_to_one_t0_record"].sum()
)

observed_unlinked_t1 = int(
    (~t1_usage["linked_to_one_t0_record"]).sum()
)

if observed_linked_t1 != EXPECTED_LINKED_T1:
    raise AssertionError(
        f"Expected {EXPECTED_LINKED_T1:,} linked T1 records, "
        f"observed {observed_linked_t1:,}."
    )

if observed_unlinked_t1 != EXPECTED_UNLINKED_T1:
    raise AssertionError(
        f"Expected {EXPECTED_UNLINKED_T1:,} unlinked T1 records, "
        f"observed {observed_unlinked_t1:,}."
    )


# -------------------------------------------------------------------------------------------------
# 19. ORDER FINAL OUTPUT COLUMNS
# -------------------------------------------------------------------------------------------------

final_columns = [
    "final_crosswalk_record_id",

    "t0_rcv_key",
    "t0_release_label",
    "t0_embedded_data_cutoff_date",
    "t0_rcv_accession",
    "t0_rcv_version",
    "t0_variation_id",
    "t0_vcv_accession",
    "t0_vcv_version",
    "t0_target_genes_json",
    "t0_study_scope",
    "t0_condition_names_json",
    "t0_condition_ids_json",

    "final_linkage_status",
    "linkage_method",
    "one_to_one_linkage_accepted",
    "linkage_eligible_for_future_outcome_construction",

    "linked_t1_rcv_key",
    "linked_t1_record_present",

    "t1_release_label",
    "t1_embedded_data_cutoff_date",
    "t1_rcv_accession",
    "t1_rcv_version",
    "t1_variation_id",
    "t1_vcv_accession",
    "t1_vcv_version",
    "t1_target_genes_json",
    "t1_study_scope",
    "t1_condition_names_json",
    "t1_condition_ids_json",
    "t1_aggregate_classification_axis",

    "accepted_nonexact_candidate_pair_id",
    "accepted_nonexact_condition_evidence",

    "core_identifier_continuity_status",
    "gene_set_relation",
    "condition_continuity_status",
    "classification_axis_audit_category",
    "requires_linkage_review",
    "review_reason_count",
    "review_reasons_json",

    "stage3e_post_adjudication_status",
    "identifier_based_candidate_count",
    "strong_candidate_count",

    "stage3f_targeted_candidate_pair_count",
    "stage3f_component_count",
    "stage3f_t0_review_category",
    "stage3f_resolution_status",

    "condition_only_candidate_count",
    "exact_condition_set_candidate_count",
    "unique_condition_only_candidate_t1_rcv",
    "stage3g_t0_audit_category",
    "stage3g_source_history_review_priority",
    "stage3g_resolution_status",

    "final_unresolved_disposition",

    "linkage_policy_version",
    "linkage_policy_sha256",
    "stage3_linkage_frozen",

    "condition_only_candidate_used_for_linkage",
    "official_rcv_replacement_confirmed",
    "classification_change_calculated",

    "future_instability_outcome_created",
    "future_instability_label",
    "unresolved_record_labeled_stable",
    "outcome_construction_status",
]

missing_final_columns = [
    column
    for column in final_columns
    if column not in final_linkage.columns
]

if missing_final_columns:
    raise KeyError(
        "Final linkage is missing expected columns: "
        + ", ".join(missing_final_columns)
    )

final_linkage = final_linkage[final_columns]

final_linkage = (
    final_linkage.sort_values(
        by="t0_rcv_key",
        kind="mergesort",
    )
    .reset_index(drop=True)
)

t1_usage = (
    t1_usage.sort_values(
        by="t1_rcv_key",
        kind="mergesort",
    )
    .reset_index(drop=True)
)


# -------------------------------------------------------------------------------------------------
# 20. WRITE AND READ BACK THE FINAL FROZEN ARTIFACTS
# -------------------------------------------------------------------------------------------------

final_linkage.to_parquet(
    FINAL_LINKAGE_PATH,
    engine="pyarrow",
    compression="zstd",
    index=False,
)

t1_usage.to_parquet(
    T1_USAGE_PATH,
    engine="pyarrow",
    compression="zstd",
    index=False,
)

final_linkage_sha256 = sha256_file(FINAL_LINKAGE_PATH)
t1_usage_sha256 = sha256_file(T1_USAGE_PATH)

final_metadata = pq.ParquetFile(
    FINAL_LINKAGE_PATH
).metadata

t1_usage_metadata = pq.ParquetFile(
    T1_USAGE_PATH
).metadata

final_readback = pd.read_parquet(FINAL_LINKAGE_PATH)
t1_usage_readback = pd.read_parquet(T1_USAGE_PATH)

if len(final_readback) != EXPECTED_T0_ROWS:
    raise AssertionError(
        "Final linkage readback does not contain 71,659 rows."
    )

if final_readback["final_crosswalk_record_id"].duplicated().any():
    raise AssertionError(
        "Final linkage readback contains duplicate T0 record identifiers."
    )

if len(t1_usage_readback) != EXPECTED_T1_ROWS:
    raise AssertionError(
        "T1 usage readback does not contain 100,920 rows."
    )

if t1_usage_readback["t1_rcv_key"].duplicated().any():
    raise AssertionError(
        "T1 usage readback contains duplicate T1 RCV keys."
    )

if list(final_readback.columns) != list(final_linkage.columns):
    raise AssertionError(
        "Final linkage readback schema differs from the written dataframe."
    )

if list(t1_usage_readback.columns) != list(t1_usage.columns):
    raise AssertionError(
        "T1 usage readback schema differs from the written dataframe."
    )


# -------------------------------------------------------------------------------------------------
# 21. CREATE THE FINAL STAGE 3 FREEZE REPORT
# -------------------------------------------------------------------------------------------------

linked_rate = EXPECTED_TOTAL_LINKED_T0 / EXPECTED_T0_ROWS
unresolved_rate = EXPECTED_TOTAL_UNRESOLVED / EXPECTED_T0_ROWS

report = {
    "report_name": "Stage 3 Final T0-T1 Linkage Freeze Report",
    "report_version": "1.0.0",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "study_stage": "Stage 3H",
    "scientific_unit": "RCV-level variant-condition aggregate",

    "final_stage3_decision": (
        "ACCEPTED_AND_FROZEN_STAGE3_LINKAGE_COMPLETE"
    ),

    "policy": {
        "path": str(POLICY_PATH),
        "sha256": policy_sha256,
        "version": policy["policy_version"],
    },

    "t0_linkage_results": {
        "total_t0_records": int(EXPECTED_T0_ROWS),
        "exact_rcv_links": int(observed_exact),
        "accepted_nonexact_one_to_one_links": int(
            observed_nonexact
        ),
        "total_linked_t0_records": int(
            EXPECTED_TOTAL_LINKED_T0
        ),
        "linked_t0_share": round(float(linked_rate), 8),
        "total_unresolved_censored_t0_records": int(
            observed_unresolved
        ),
        "unresolved_t0_share": round(
            float(unresolved_rate),
            8,
        ),
        "final_linkage_status_counts": value_counts_dict(
            final_linkage["final_linkage_status"]
        ),
        "unresolved_disposition_counts": value_counts_dict(
            final_linkage.loc[
                unresolved_mask,
                "final_unresolved_disposition",
            ]
        ),
    },

    "t1_usage_results": {
        "total_t1_records": int(EXPECTED_T1_ROWS),
        "linked_t1_records": int(observed_linked_t1),
        "unlinked_t1_records": int(observed_unlinked_t1),
        "usage_status_counts": value_counts_dict(
            t1_usage["t1_linkage_usage_status"]
        ),
    },

    "audit_context": {
        "exact_match_records_flagged_for_stage3c_review": int(
            final_linkage.loc[
                exact_mask,
                "requires_linkage_review",
            ]
            .fillna(False)
            .sum()
        ),
        "stage3f_review_category_counts": value_counts_dict(
            final_linkage[
                "stage3f_t0_review_category"
            ]
        ),
        "stage3g_audit_category_counts": value_counts_dict(
            final_linkage[
                "stage3g_t0_audit_category"
            ]
        ),
    },

    "validation": {
        "one_row_per_t0_record": True,
        "unique_t0_keys": int(
            final_linkage["t0_rcv_key"].nunique()
        ),
        "accepted_linked_t1_keys": int(
            linked["linked_t1_rcv_key"].nunique()
        ),
        "duplicate_accepted_t1_assignments": int(
            linked["linked_t1_rcv_key"].duplicated().sum()
        ),
        "linked_variation_id_mismatches": int(
            linked_variation_mismatches
        ),
        "linked_vcv_accession_mismatches": int(
            linked_vcv_mismatches
        ),
        "unresolved_records_with_t1_link": int(
            final_linkage.loc[
                unresolved_mask,
                "linked_t1_rcv_key",
            ]
            .notna()
            .sum()
        ),
        "unresolved_records_labeled_stable": int(
            final_linkage[
                "unresolved_record_labeled_stable"
            ].sum()
        ),
        "future_instability_outcomes_created": int(
            final_linkage[
                "future_instability_outcome_created"
            ].sum()
        ),
        "readback_passed": True,
        "critical_failures": 0,
    },

    "scientific_boundaries": {
        "stage3_linkage_complete": True,
        "future_instability_outcomes_created": False,
        "classifications_compared": False,
        "unresolved_records_labeled_stable": False,
        "condition_only_candidates_accepted": False,
        "official_rcv_replacements_confirmed": False,
        "merges_confirmed": False,
        "splits_confirmed": False,
        "ges_model_fitted_or_tuned": False,
    },

    "next_authorized_step": (
        "Begin Blueprint Stage 4 using T0 information only: freeze the "
        "feature-processing specification, weak-supervision labeling rules, "
        "full GES model, and mandatory no-star GES ablation before inspecting "
        "future-instability outcomes."
    ),
}

with REPORT_PATH.open("w", encoding="utf-8") as handle:
    json.dump(
        report,
        handle,
        indent=2,
        sort_keys=True,
        ensure_ascii=False,
    )

report_sha256 = sha256_file(REPORT_PATH)


# -------------------------------------------------------------------------------------------------
# 22. CREATE THE FINAL STAGE 3 FREEZE MANIFEST
# -------------------------------------------------------------------------------------------------

manifest = {
    "manifest_name": "Stage 3 Final T0-T1 Linkage Freeze Manifest",
    "manifest_version": "1.0.0",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "study_stage": "Stage 3H",

    "freeze_status": (
        "ACCEPTED_AND_FROZEN_STAGE3_LINKAGE_COMPLETE"
    ),

    "inputs": {
        str(path): {
            "sha256": observed_checksums[str(path)],
        }
        for path in EXPECTED_CHECKSUMS
    },

    "policy": {
        "path": str(POLICY_PATH),
        "sha256": policy_sha256,
    },

    "outputs": {
        "final_t0_t1_linkage": {
            "path": str(FINAL_LINKAGE_PATH),
            "sha256": final_linkage_sha256,
            "rows": int(final_metadata.num_rows),
            "columns": int(final_metadata.num_columns),
            "row_groups": int(final_metadata.num_row_groups),
            "compression": "Zstandard",
        },
        "t1_linkage_usage_audit": {
            "path": str(T1_USAGE_PATH),
            "sha256": t1_usage_sha256,
            "rows": int(t1_usage_metadata.num_rows),
            "columns": int(t1_usage_metadata.num_columns),
            "row_groups": int(t1_usage_metadata.num_row_groups),
            "compression": "Zstandard",
        },
        "freeze_report": {
            "path": str(REPORT_PATH),
            "sha256": report_sha256,
        },
    },

    "frozen_counts": {
        "t0_records": int(EXPECTED_T0_ROWS),
        "exact_rcv_links": int(observed_exact),
        "accepted_nonexact_links": int(observed_nonexact),
        "total_linked_t0_records": int(
            EXPECTED_TOTAL_LINKED_T0
        ),
        "total_unresolved_censored_t0_records": int(
            observed_unresolved
        ),
        "t1_records": int(EXPECTED_T1_ROWS),
        "linked_t1_records": int(observed_linked_t1),
        "unlinked_t1_records": int(observed_unlinked_t1),
    },

    "validation_decision": "PASS",

    "scientific_boundaries": report[
        "scientific_boundaries"
    ],

    "next_authorized_step": report[
        "next_authorized_step"
    ],
}

with MANIFEST_PATH.open("w", encoding="utf-8") as handle:
    json.dump(
        manifest,
        handle,
        indent=2,
        sort_keys=True,
        ensure_ascii=False,
    )

manifest_sha256 = sha256_file(MANIFEST_PATH)


# -------------------------------------------------------------------------------------------------
# 23. PRINT FINAL RESULTS
# -------------------------------------------------------------------------------------------------

print()
print("=" * 128)
print("STAGE 3H FINAL LINKAGE-FREEZE SUMMARY")
print("=" * 128)

print(f"Frozen T0 records:                              {len(final_linkage):,}")
print(f"Exact RCV links:                                {observed_exact:,}")
print(f"Accepted conservative non-exact links:          {observed_nonexact:,}")
print(f"Total linked T0 records:                        {EXPECTED_TOTAL_LINKED_T0:,}")
print(f"Unresolved/censored T0 records:                 {observed_unresolved:,}")

print()
print("Unresolved/censored dispositions:")
print(
    final_linkage.loc[
        unresolved_mask,
        "final_unresolved_disposition",
    ]
    .value_counts(dropna=False)
    .to_string()
)

print()
print(f"Frozen T1 records:                              {len(t1_usage):,}")
print(f"T1 records linked to one T0 record:             {observed_linked_t1:,}")
print(f"T1 records not linked to the T0 cohort:         {observed_unlinked_t1:,}")

print()
print("Final linkage status:")
print(
    final_linkage[
        "final_linkage_status"
    ]
    .value_counts(dropna=False)
    .to_string()
)

print()
print("T1 usage status:")
print(
    t1_usage[
        "t1_linkage_usage_status"
    ]
    .value_counts(dropna=False)
    .to_string()
)

print()
print(f"Frozen policy:         {POLICY_PATH}")
print(f"Policy SHA-256:        {policy_sha256}")
print(f"Final linkage:         {FINAL_LINKAGE_PATH}")
print(f"Linkage SHA-256:       {final_linkage_sha256}")
print(f"T1 usage audit:        {T1_USAGE_PATH}")
print(f"T1 usage SHA-256:      {t1_usage_sha256}")
print(f"Freeze report:         {REPORT_PATH}")
print(f"Report SHA-256:        {report_sha256}")
print(f"Freeze manifest:       {MANIFEST_PATH}")
print(f"Manifest SHA-256:      {manifest_sha256}")

print()
print("PASS — Stage 3 final T0–T1 linkage artifact assembled and frozen.")
print(
    "DECISION — ACCEPTED_AND_FROZEN_STAGE3_LINKAGE_COMPLETE"
)
print(
    "IMPORTANT — The 1,076 unresolved records are censored/unresolved, "
    "not stable."
)
print(
    "IMPORTANT — No classification comparison or future-instability "
    "outcome was created."
)
print(
    "BLUEPRINT STAGE 3 — COMPLETE."
)
print(
    "AUTHORIZED NEXT ACTION — Blueprint Stage 4A: freeze the T0-only "
    "GES feature, weak-label, model, and no-star ablation specification."
)
print("=" * 128)

STAGE 3H — FINAL T0–T1 LINKAGE ASSEMBLY AND FREEZE
Verifying immutable Stage 2 and Stage 3 input artifacts...
PASS  t0_rcv_target_genes_corrected_v1_2.parquet
      f6b6760b2ad6e4352e3bdecdeaf89827e8a514b031abf2373bb17568d999466d
PASS  t1_rcv_target_genes_harmonized_v1.parquet
      5713a11bdbf4804758cc011f9b2f302afc91fa1f88c1b178d675c28bb277d37c
PASS  stage3_t0_t1_exact_rcv_crosswalk_draft_v1.parquet
      b462304a4bb31db301e2dac3aefbf685e12f1fea179a38ca04b8500b64bb4acc
PASS  stage3_t0_t1_exact_match_continuity_audit_v1.parquet
      7671792ebccae68ff77745355498d174e0a614f21703e518fb230ebfc54666ad
PASS  stage3_accepted_nonexact_one_to_one_links_v1.parquet
      19e9eb00810f8b3faf2147c3a73895d0a939068beb20b3aaaf9a8bd8a8d64268
PASS  stage3_t0_post_strong_candidate_adjudication_v1.parquet
      c1e40c92389e2dc200cd1508c49fb7aa12c3a9665fc2e6aab56ed2127accb65e
PASS  stage3_strong_candidate_adjudication_manifest_v1.json
      b5ea53ba3fe6cafdb2abaa25ac582d683cfb35dd04d88dcfdaf03802edec4304


RuntimeError: Checksum mismatch for:
/content/drive/MyDrive/GES_RAG_Temporal_Study/configs/stage3_crosswalk/stage3_targeted_complex_and_tier3_audit_manifest_v1.json
Expected: 004292519a8a33d261c9a57acdbd6573fc074338a2bb311bbd464b8fd6fa47b1
Observed: 806db054a88c14ea3e64474f00a3ad6241297d2efbdfaf3e068601eee25d0788

In [23]:
# =================================================================================================
# STAGE 3H PRECHECK — VALIDATE THE CURRENT STAGE 3F MANIFEST SEMANTICALLY
# =================================================================================================
# Purpose:
#   Determine whether the Stage 3F manifest checksum changed only because the
#   manifest/report was regenerated, while the scientific data artifacts remain unchanged.
#
# This cell does not modify any file.
# =================================================================================================

from pathlib import Path
import hashlib
import json


print("=" * 120)
print("STAGE 3H PRECHECK — STAGE 3F MANIFEST VALIDATION")
print("=" * 120)


# -------------------------------------------------------------------------------------------------
# 1. PATHS
# -------------------------------------------------------------------------------------------------

STUDY_ROOT = Path("/content/drive/MyDrive/GES_RAG_Temporal_Study")

STAGE3_CONFIG_DIR = STUDY_ROOT / "configs" / "stage3_crosswalk"
STAGE3_OUTPUT_DIR = STUDY_ROOT / "data_processed" / "stage3_crosswalk"

MANIFEST_PATH = (
    STAGE3_CONFIG_DIR
    / "stage3_targeted_complex_and_tier3_audit_manifest_v1.json"
)

PAIR_AUDIT_PATH = (
    STAGE3_OUTPUT_DIR
    / "stage3_targeted_complex_and_tier3_pair_audit_v1.parquet"
)

COMPONENT_AUDIT_PATH = (
    STAGE3_OUTPUT_DIR
    / "stage3_targeted_candidate_graph_components_v1.parquet"
)

T0_REVIEW_PATH = (
    STAGE3_OUTPUT_DIR
    / "stage3_targeted_candidate_t0_review_summary_v1.parquet"
)

REPORT_PATH = (
    STAGE3_CONFIG_DIR
    / "stage3_targeted_complex_and_tier3_audit_report_v1.json"
)


# -------------------------------------------------------------------------------------------------
# 2. EXPECTED SCIENTIFIC OUTPUT HASHES FROM THE ACCEPTED STAGE 3F RUN
# -------------------------------------------------------------------------------------------------

EXPECTED_OUTPUT_HASHES = {
    "targeted_pair_audit": (
        "bb355e338aedc2e247aa8e874290c305"
        "acbbee8445d26c3fdd382ae0bbd1fceb"
    ),
    "targeted_component_audit": (
        "33b9a7f9374d3adc69ae64b0e53ec7f"
        "0fb98ba7f70211e7e6869f7c93f2e19ed"
    ),
    "targeted_t0_review_summary": (
        "09f6e4d24744781127e59d7ee03ddf3d"
        "5d02b0c4ea5517e52a184ad46a623d2a"
    ),
}

EXPECTED_COUNTS = {
    "targeted_pair_count": 908,
    "graph_component_count": 534,
    "unique_targeted_t0_record_count": 561,
}


# -------------------------------------------------------------------------------------------------
# 3. UTILITIES
# -------------------------------------------------------------------------------------------------

def sha256_file(path: Path, chunk_size: int = 8 * 1024 * 1024) -> str:
    """Calculate SHA-256 without loading the entire file into memory."""
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


# -------------------------------------------------------------------------------------------------
# 4. VERIFY FILE EXISTENCE
# -------------------------------------------------------------------------------------------------

required_files = [
    MANIFEST_PATH,
    PAIR_AUDIT_PATH,
    COMPONENT_AUDIT_PATH,
    T0_REVIEW_PATH,
    REPORT_PATH,
]

for path in required_files:
    if not path.exists():
        raise FileNotFoundError(
            f"Required Stage 3F artifact is missing: {path}"
        )


# -------------------------------------------------------------------------------------------------
# 5. LOAD THE CURRENT MANIFEST
# -------------------------------------------------------------------------------------------------

with MANIFEST_PATH.open("r", encoding="utf-8") as handle:
    manifest = json.load(handle)

current_manifest_sha256 = sha256_file(MANIFEST_PATH)

print(f"Current manifest:        {MANIFEST_PATH}")
print(f"Current manifest SHA-256:{current_manifest_sha256}")
print(f"Created at UTC:          {manifest.get('created_at_utc')}")
print(f"Audit status:            {manifest.get('audit_status')}")
print(f"Validation decision:     {manifest.get('validation_decision')}")


# -------------------------------------------------------------------------------------------------
# 6. VERIFY MANIFEST STATUS AND COUNTS
# -------------------------------------------------------------------------------------------------

errors = []

if (
    manifest.get("audit_status")
    != "TARGETED_COMPLEX_AND_TIER3_AUDIT_COMPLETE"
):
    errors.append(
        "Unexpected Stage 3F audit_status."
    )

if manifest.get("validation_decision") != "PASS":
    errors.append(
        "Stage 3F validation_decision is not PASS."
    )

for field, expected_value in EXPECTED_COUNTS.items():
    observed_value = manifest.get(field)

    if observed_value != expected_value:
        errors.append(
            f"{field}: expected {expected_value:,}, "
            f"observed {observed_value!r}."
        )


# -------------------------------------------------------------------------------------------------
# 7. CALCULATE CURRENT OUTPUT HASHES
# -------------------------------------------------------------------------------------------------

current_output_hashes = {
    "targeted_pair_audit": sha256_file(PAIR_AUDIT_PATH),
    "targeted_component_audit": sha256_file(
        COMPONENT_AUDIT_PATH
    ),
    "targeted_t0_review_summary": sha256_file(
        T0_REVIEW_PATH
    ),
    "report": sha256_file(REPORT_PATH),
}

output_paths = {
    "targeted_pair_audit": PAIR_AUDIT_PATH,
    "targeted_component_audit": COMPONENT_AUDIT_PATH,
    "targeted_t0_review_summary": T0_REVIEW_PATH,
    "report": REPORT_PATH,
}


# -------------------------------------------------------------------------------------------------
# 8. COMPARE CURRENT FILES WITH THE HASHES DECLARED INSIDE THE MANIFEST
# -------------------------------------------------------------------------------------------------

manifest_outputs = manifest.get("outputs", {})

for output_name, path in output_paths.items():
    declared_output = manifest_outputs.get(output_name)

    if not isinstance(declared_output, dict):
        errors.append(
            f"Manifest output section is missing: {output_name}"
        )
        continue

    declared_path = declared_output.get("path")
    declared_sha256 = declared_output.get("sha256")
    current_sha256 = current_output_hashes[output_name]

    print()
    print(f"{output_name}")
    print(f"  Current path:          {path}")
    print(f"  Declared path:         {declared_path}")
    print(f"  Current SHA-256:       {current_sha256}")
    print(f"  Manifest SHA-256:      {declared_sha256}")
    print(
        f"  Current/manifest match:{current_sha256 == declared_sha256}"
    )

    if declared_path != str(path):
        errors.append(
            f"{output_name}: manifest path does not match "
            "the expected artifact path."
        )

    if declared_sha256 != current_sha256:
        errors.append(
            f"{output_name}: current file hash does not match "
            "the hash declared by the current manifest."
        )


# -------------------------------------------------------------------------------------------------
# 9. CONFIRM THAT THE SCIENTIFIC PARQUET OUTPUTS MATCH THE ACCEPTED RUN
# -------------------------------------------------------------------------------------------------

print()
print("-" * 120)
print("ACCEPTED STAGE 3F SCIENTIFIC-OUTPUT CHECKS")
print("-" * 120)

for output_name, expected_sha256 in EXPECTED_OUTPUT_HASHES.items():
    observed_sha256 = current_output_hashes[output_name]
    passed = observed_sha256 == expected_sha256

    print(
        f"{output_name:<36} "
        f"{'PASS' if passed else 'FAIL'}"
    )
    print(f"  Expected: {expected_sha256}")
    print(f"  Observed: {observed_sha256}")

    if not passed:
        errors.append(
            f"{output_name}: scientific output differs from "
            "the accepted Stage 3F run."
        )


# -------------------------------------------------------------------------------------------------
# 10. VERIFY SCIENTIFIC BOUNDARIES
# -------------------------------------------------------------------------------------------------

boundaries = manifest.get("scientific_boundaries", {})

required_false_boundaries = [
    "additional_nonexact_links_accepted",
    "stage3e_links_revoked_or_modified",
    "official_rcv_replacements_confirmed",
    "merges_confirmed",
    "splits_confirmed",
    "condition_equivalence_inferred_from_lexical_similarity",
    "classification_changes_calculated",
    "future_instability_outcomes_created",
    "unresolved_records_labeled_stable",
    "ges_model_fitted_or_tuned",
]

for field in required_false_boundaries:
    observed_value = boundaries.get(field)

    if observed_value is not False:
        errors.append(
            f"Scientific boundary {field!r} is not explicitly False."
        )


# -------------------------------------------------------------------------------------------------
# 11. FINAL DECISION
# -------------------------------------------------------------------------------------------------

print()
print("=" * 120)
print("STAGE 3H PRECHECK RESULT")
print("=" * 120)

if errors:
    print("FAIL — The current Stage 3F manifest cannot yet be trusted.")
    print()
    print("Detected issues:")

    for number, error in enumerate(errors, start=1):
        print(f"{number}. {error}")

    raise RuntimeError(
        "Stage 3H remains blocked until the Stage 3F manifest "
        "or its underlying artifacts are reconciled."
    )

print("PASS — The current Stage 3F manifest is semantically valid.")
print(
    "PASS — All three scientific Parquet outputs match the accepted "
    "Stage 3F run."
)
print(
    "The manifest checksum changed because the JSON manifest/report "
    "was regenerated, not because the scientific Stage 3F outputs changed."
)
print()
print(
    "CURRENT AUTHORIZED STAGE 3F MANIFEST SHA-256:"
)
print(current_manifest_sha256)
print()
print(
    "AUTHORIZED NEXT ACTION — Run the corrected Stage 3H cell using "
    "semantic manifest validation rather than the old timestamp-sensitive "
    "manifest checksum."
)
print("=" * 120)

STAGE 3H PRECHECK — STAGE 3F MANIFEST VALIDATION
Current manifest:        /content/drive/MyDrive/GES_RAG_Temporal_Study/configs/stage3_crosswalk/stage3_targeted_complex_and_tier3_audit_manifest_v1.json
Current manifest SHA-256:806db054a88c14ea3e64474f00a3ad6241297d2efbdfaf3e068601eee25d0788
Created at UTC:          2026-07-20T20:21:06.475381+00:00
Audit status:            TARGETED_COMPLEX_AND_TIER3_AUDIT_COMPLETE
Validation decision:     PASS

targeted_pair_audit
  Current path:          /content/drive/MyDrive/GES_RAG_Temporal_Study/data_processed/stage3_crosswalk/stage3_targeted_complex_and_tier3_pair_audit_v1.parquet
  Declared path:         /content/drive/MyDrive/GES_RAG_Temporal_Study/data_processed/stage3_crosswalk/stage3_targeted_complex_and_tier3_pair_audit_v1.parquet
  Current SHA-256:       bb355e338aedc2e247aa8e874290c305acbbee8445d26c3fdd382ae0bbd1fceb
  Manifest SHA-256:      bb355e338aedc2e247aa8e874290c305acbbee8445d26c3fdd382ae0bbd1fceb
  Current/manifest match:True

tar

In [24]:
# ==================================================================================================
# STAGE 3H — FINAL T0–T1 LINKAGE ASSEMBLY, T1 USAGE AUDIT, AND CHECKSUM FREEZE
#
# Scientific boundary:
#   - One final row for every T0 RCV.
#   - Attach T1 evidence only for:
#         1. accepted exact RCV links;
#         2. accepted conservative Tier-1 one-to-one non-exact links.
#   - All remaining T0 records stay unresolved/censored.
#   - Do NOT create stable/unstable labels or temporal outcomes in this cell.
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import os
import re
import shutil

import pandas as pd


# --------------------------------------------------------------------------------------------------
# 1. Google Drive
# --------------------------------------------------------------------------------------------------

DRIVE_ROOT = Path("/content/drive/MyDrive")

if not DRIVE_ROOT.exists():
    from google.colab import drive
    drive.mount("/content/drive")

assert DRIVE_ROOT.exists(), "Google Drive is not mounted."


# --------------------------------------------------------------------------------------------------
# 2. Persistent directories
# --------------------------------------------------------------------------------------------------

BASE_DIR = DRIVE_ROOT / "GES_RAG_Temporal_Study"

T0_PATH = (
    BASE_DIR
    / "data_interim"
    / "t0_rcv_target_genes_corrected_v1_2.parquet"
)

T1_PATH = (
    BASE_DIR
    / "data_interim"
    / "t1_rcv_target_genes_harmonized_v1.parquet"
)

STAGE3_DATA_DIR = BASE_DIR / "data_processed" / "stage3_crosswalk"
STAGE3_CONFIG_DIR = BASE_DIR / "configs" / "stage3_crosswalk"

STAGE3_DATA_DIR.mkdir(parents=True, exist_ok=True)
STAGE3_CONFIG_DIR.mkdir(parents=True, exist_ok=True)


# --------------------------------------------------------------------------------------------------
# 3. Required Stage 3 inputs
# --------------------------------------------------------------------------------------------------

EXACT_CROSSWALK_PATH = (
    STAGE3_DATA_DIR
    / "stage3_t0_t1_exact_rcv_crosswalk_draft_v1.parquet"
)

NONEXACT_CANDIDATE_PATH = (
    STAGE3_DATA_DIR
    / "stage3_t0_unmatched_nonexact_candidate_pairs_v1.parquet"
)

ACCEPTED_NONEXACT_PATH = (
    STAGE3_DATA_DIR
    / "stage3_accepted_nonexact_one_to_one_links_v1.parquet"
)

STAGE3F_PAIR_AUDIT_PATH = (
    STAGE3_DATA_DIR
    / "stage3_targeted_complex_and_tier3_pair_audit_v1.parquet"
)

STAGE3F_COMPONENT_AUDIT_PATH = (
    STAGE3_DATA_DIR
    / "stage3_targeted_candidate_graph_components_v1.parquet"
)

STAGE3F_T0_REVIEW_PATH = (
    STAGE3_DATA_DIR
    / "stage3_targeted_candidate_t0_review_summary_v1.parquet"
)

STAGE3F_MANIFEST_PATH = (
    STAGE3_CONFIG_DIR
    / "stage3_targeted_complex_and_tier3_audit_manifest_v1.json"
)

STAGE3G_T0_AUDIT_PATH = (
    STAGE3_DATA_DIR
    / "stage3_no_identifier_t0_audit_summary_v1.parquet"
)


# --------------------------------------------------------------------------------------------------
# 4. Final Stage 3H outputs
# --------------------------------------------------------------------------------------------------

FINAL_LINKAGE_PATH = (
    STAGE3_DATA_DIR
    / "stage3h_final_t0_t1_linkage_v1.parquet"
)

T1_USAGE_AUDIT_PATH = (
    STAGE3_DATA_DIR
    / "stage3h_t1_usage_audit_v1.parquet"
)

LINKAGE_POLICY_PATH = (
    STAGE3_CONFIG_DIR
    / "stage3h_linkage_and_censoring_policy_v1.json"
)

FREEZE_REPORT_PATH = (
    STAGE3_CONFIG_DIR
    / "stage3h_final_linkage_freeze_report_v1.json"
)

FREEZE_MANIFEST_PATH = (
    STAGE3_CONFIG_DIR
    / "stage3h_final_linkage_freeze_manifest_v1.json"
)

FINAL_OUTPUTS = [
    FINAL_LINKAGE_PATH,
    T1_USAGE_AUDIT_PATH,
    LINKAGE_POLICY_PATH,
    FREEZE_REPORT_PATH,
    FREEZE_MANIFEST_PATH,
]

existing_outputs = [str(path) for path in FINAL_OUTPUTS if path.exists()]

if existing_outputs:
    raise FileExistsError(
        "Stage 3H frozen output files already exist. They were not overwritten:\n"
        + "\n".join(existing_outputs)
    )


# --------------------------------------------------------------------------------------------------
# 5. Accepted input hashes
# --------------------------------------------------------------------------------------------------

EXPECTED_HASHES = {
    str(T0_PATH): (
        "f6b6760b2ad6e4352e3bdecdeaf89827e8a514b031abf2373bb17568d999466d"
    ),
    str(T1_PATH): (
        "5713a11bdbf4804758cc011f9b2f302afc91fa1f88c1b178d675c28bb277d37c"
    ),
    str(EXACT_CROSSWALK_PATH): (
        "b462304a4bb31db301e2dac3aefbf685e12f1fea179a38ca04b8500b64bb4acc"
    ),
    str(NONEXACT_CANDIDATE_PATH): (
        "b2b23a9da290d2cd472a8378ef1ab03c0388503928351def8ae37cc3ef615c68"
    ),
    str(ACCEPTED_NONEXACT_PATH): (
        "19e9eb00810f8b3faf2147c3a73895d0a939068beb20b3aaaf9a8bd8a8d64268"
    ),
    str(STAGE3F_PAIR_AUDIT_PATH): (
        "bb355e338aedc2e247aa8e874290c305acbbee8445d26c3fdd382ae0bbd1fceb"
    ),
    str(STAGE3F_COMPONENT_AUDIT_PATH): (
        "33b9a7f9374d3adc69ae64b0e53ec7f0fb98ba7f70211e7e6869f7c93f2e19ed"
    ),
    str(STAGE3F_T0_REVIEW_PATH): (
        "09f6e4d24744781127e59d7ee03ddf3d5d02b0c4ea5517e52a184ad46a623d2a"
    ),
    str(STAGE3G_T0_AUDIT_PATH): (
        "7d3bfcd3428e72f7c48a3ed9aa954126fc7dd0eb4a1daec1a24ff1173670d776"
    ),
}


# --------------------------------------------------------------------------------------------------
# 6. Utility functions
# --------------------------------------------------------------------------------------------------

def sha256_file(path: Path, chunk_size: int = 8 * 1024 * 1024) -> str:
    """Calculate SHA-256 without loading the complete file into memory."""
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)

    return digest.hexdigest()


def write_json(path: Path, payload: dict) -> None:
    """Write deterministic, human-readable JSON."""
    text = json.dumps(
        payload,
        indent=2,
        sort_keys=True,
        ensure_ascii=False,
    )
    path.write_text(text + "\n", encoding="utf-8")


def normalize_rcv(series: pd.Series) -> pd.Series:
    """Normalize RCV accession formatting without creating replacements."""
    return (
        series.astype("string")
        .str.strip()
        .str.upper()
    )


def recursively_flatten_json(value, prefix=""):
    """Return flattened JSON key/value pairs for semantic manifest checks."""
    items = []

    if isinstance(value, dict):
        for key, child in value.items():
            child_prefix = f"{prefix}.{key}" if prefix else str(key)
            items.extend(recursively_flatten_json(child, child_prefix))

    elif isinstance(value, list):
        for index, child in enumerate(value):
            child_prefix = f"{prefix}[{index}]"
            items.extend(recursively_flatten_json(child, child_prefix))

    else:
        items.append((prefix, value))

    return items


def infer_rcv_column(
    dataframe: pd.DataFrame,
    valid_rcvs: set,
    side: str,
    exclude_columns=None,
) -> str:
    """
    Infer the appropriate RCV column using:
      1. expected column names;
      2. RCV-like column names;
      3. membership in the accepted T0 or T1 RCV set.
    """

    exclude_columns = set(exclude_columns or [])

    expected_names = {
        "t0": [
            "t0_rcv_accession",
            "rcv_accession_t0",
            "source_rcv_accession",
            "old_rcv_accession",
            "t0_rcv",
        ],
        "t1": [
            "t1_rcv_accession",
            "rcv_accession_t1",
            "candidate_t1_rcv_accession",
            "target_rcv_accession",
            "new_rcv_accession",
            "t1_rcv",
        ],
    }

    lower_to_original = {
        str(column).lower(): column
        for column in dataframe.columns
        if column not in exclude_columns
    }

    for expected in expected_names[side]:
        if expected in lower_to_original:
            return lower_to_original[expected]

    rcv_columns = [
        column
        for column in dataframe.columns
        if column not in exclude_columns
        and "rcv" in str(column).lower()
    ]

    if not rcv_columns:
        raise KeyError(
            f"No RCV-like column was found. Available columns:\n"
            f"{list(dataframe.columns)}"
        )

    scored_columns = []

    for column in rcv_columns:
        values = normalize_rcv(dataframe[column])
        available = values.notna()

        if available.sum() == 0:
            membership_score = 0.0
        else:
            membership_score = float(
                values.loc[available].isin(valid_rcvs).mean()
            )

        name = str(column).lower()
        side_bonus = 0.05 if side in name else 0.0

        scored_columns.append(
            (
                membership_score + side_bonus,
                membership_score,
                column,
            )
        )

    scored_columns.sort(reverse=True, key=lambda item: item[0])

    best_total_score, best_membership_score, best_column = scored_columns[0]

    if best_membership_score < 0.80:
        raise ValueError(
            f"Could not confidently identify the {side.upper()} RCV column.\n"
            f"Candidate scores: {scored_columns}\n"
            f"Available columns: {list(dataframe.columns)}"
        )

    return best_column


def find_strong_candidate_mask(
    dataframe: pd.DataFrame,
    t0_rcv_column: str,
) -> tuple[pd.Series, str]:
    """
    Locate the frozen Tier-1/strong-candidate rows.

    The accepted Stage 3D accounting requires:
      - 248 strong candidate pairs;
      - 208 distinct T0 RCVs having a strong candidate.
    """

    candidate_tests = []

    for column in dataframe.columns:
        column_name = str(column).lower()

        if "tier" not in column_name and "strong" not in column_name:
            continue

        series = dataframe[column]

        # Boolean strong-candidate column
        if pd.api.types.is_bool_dtype(series):
            candidate_tests.append(
                (
                    series.fillna(False).astype(bool),
                    f"{column}=True",
                )
            )

        text = (
            series.astype("string")
            .str.strip()
            .str.upper()
            .fillna("")
        )

        if "strong" in column_name:
            candidate_tests.append(
                (
                    text.isin(
                        {
                            "TRUE",
                            "1",
                            "YES",
                            "Y",
                            "STRONG",
                            "TIER_1",
                            "TIER 1",
                        }
                    ),
                    f"{column}=strong/true",
                )
            )

        if "tier" in column_name:
            tier1_mask = (
                text.eq("1")
                | text.eq("TIER1")
                | text.eq("TIER_1")
                | text.eq("TIER 1")
                | text.str.contains(
                    r"(^|[^A-Z0-9])TIER[_\-\s]*1([^0-9]|$)",
                    regex=True,
                    na=False,
                )
            )

            candidate_tests.append(
                (
                    tier1_mask,
                    f"{column}=Tier 1",
                )
            )

        # Also test individual categorical values.
        unique_values = text[text.ne("")].drop_duplicates().tolist()

        if len(unique_values) <= 25:
            for value in unique_values:
                candidate_tests.append(
                    (
                        text.eq(value),
                        f"{column}={value}",
                    )
                )

    t0_values = normalize_rcv(dataframe[t0_rcv_column])

    for mask, description in candidate_tests:
        pair_count = int(mask.sum())
        t0_count = int(t0_values.loc[mask].nunique())

        if pair_count == 248 and t0_count == 208:
            return mask, description

    diagnostics = {}

    for column in dataframe.columns:
        column_name = str(column).lower()

        if "tier" in column_name or "strong" in column_name:
            diagnostics[str(column)] = (
                dataframe[column]
                .astype("string")
                .value_counts(dropna=False)
                .head(20)
                .to_dict()
            )

    raise ValueError(
        "Could not identify the 248 frozen strong/Tier-1 candidate pairs.\n"
        f"Relevant column diagnostics:\n{json.dumps(diagnostics, indent=2)}"
    )


def copy_and_verify(source: Path, destination: Path) -> None:
    """Copy through a temporary Drive file and verify SHA-256 before renaming."""
    partial_destination = Path(str(destination) + ".partial")

    if partial_destination.exists():
        partial_destination.unlink()

    shutil.copy2(source, partial_destination)

    source_hash = sha256_file(source)
    copied_hash = sha256_file(partial_destination)

    assert source_hash == copied_hash, (
        f"Copy checksum mismatch for {destination.name}"
    )

    os.replace(partial_destination, destination)


# --------------------------------------------------------------------------------------------------
# 7. Verify all required files and immutable accepted hashes
# --------------------------------------------------------------------------------------------------

required_paths = [
    T0_PATH,
    T1_PATH,
    EXACT_CROSSWALK_PATH,
    NONEXACT_CANDIDATE_PATH,
    ACCEPTED_NONEXACT_PATH,
    STAGE3F_PAIR_AUDIT_PATH,
    STAGE3F_COMPONENT_AUDIT_PATH,
    STAGE3F_T0_REVIEW_PATH,
    STAGE3F_MANIFEST_PATH,
    STAGE3G_T0_AUDIT_PATH,
]

missing_paths = [str(path) for path in required_paths if not path.exists()]

if missing_paths:
    raise FileNotFoundError(
        "Required Stage 3 input files are missing:\n"
        + "\n".join(missing_paths)
    )

verified_input_hashes = {}

for path_string, expected_hash in EXPECTED_HASHES.items():
    path = Path(path_string)
    observed_hash = sha256_file(path)

    assert observed_hash == expected_hash, (
        f"Accepted-input checksum mismatch:\n"
        f"Path:     {path}\n"
        f"Expected: {expected_hash}\n"
        f"Observed: {observed_hash}"
    )

    verified_input_hashes[str(path)] = observed_hash


# --------------------------------------------------------------------------------------------------
# 8. Semantic validation of the regenerated Stage 3F manifest
#    Do not require an earlier timestamp-sensitive JSON manifest checksum.
# --------------------------------------------------------------------------------------------------

with STAGE3F_MANIFEST_PATH.open("r", encoding="utf-8") as handle:
    stage3f_manifest = json.load(handle)

stage3f_manifest_text = json.dumps(
    stage3f_manifest,
    sort_keys=True,
)

required_manifest_values = [
    "TARGETED_COMPLEX_AND_TIER3_AUDIT_COMPLETE",
    "PASS",
    STAGE3F_PAIR_AUDIT_PATH.name,
    EXPECTED_HASHES[str(STAGE3F_PAIR_AUDIT_PATH)],
    STAGE3F_COMPONENT_AUDIT_PATH.name,
    EXPECTED_HASHES[str(STAGE3F_COMPONENT_AUDIT_PATH)],
    STAGE3F_T0_REVIEW_PATH.name,
    EXPECTED_HASHES[str(STAGE3F_T0_REVIEW_PATH)],
]

missing_semantic_values = [
    value
    for value in required_manifest_values
    if value not in stage3f_manifest_text
]

assert not missing_semantic_values, (
    "Stage 3F semantic manifest validation failed. Missing values:\n"
    + "\n".join(missing_semantic_values)
)

flattened_manifest = recursively_flatten_json(stage3f_manifest)

forbidden_true_tokens = [
    "future_instability",
    "outcome_created",
    "ges_model_fitted",
    "ges_threshold",
    "labeled_stable",
    "classification_change_outcome",
]

for key, value in flattened_manifest:
    normalized_key = key.lower()

    if (
        isinstance(value, bool)
        and value is True
        and any(token in normalized_key for token in forbidden_true_tokens)
    ):
        raise AssertionError(
            f"Forbidden Stage 3F scientific-boundary flag is True: {key}"
        )

current_stage3f_manifest_hash = sha256_file(STAGE3F_MANIFEST_PATH)


# --------------------------------------------------------------------------------------------------
# 9. Read accepted T0 and T1 cohorts
# --------------------------------------------------------------------------------------------------

t0 = pd.read_parquet(T0_PATH)
t1 = pd.read_parquet(T1_PATH)

assert len(t0) == 71_659
assert len(t1) == 100_920
assert t0.shape[1] == 34
assert t1.shape[1] == 36

assert "rcv_accession" in t0.columns
assert "rcv_accession" in t1.columns

t0 = t0.copy()
t1 = t1.copy()

t0["rcv_accession"] = normalize_rcv(t0["rcv_accession"])
t1["rcv_accession"] = normalize_rcv(t1["rcv_accession"])

assert t0["rcv_accession"].notna().all()
assert t1["rcv_accession"].notna().all()
assert t0["rcv_accession"].is_unique
assert t1["rcv_accession"].is_unique

t0_rcv_set = set(t0["rcv_accession"])
t1_rcv_set = set(t1["rcv_accession"])


# --------------------------------------------------------------------------------------------------
# 10. Reconstruct and verify exact-RCV linkage
# --------------------------------------------------------------------------------------------------

exact_t0_set = t0_rcv_set.intersection(t1_rcv_set)
exact_t1_set = set(exact_t0_set)

assert len(exact_t0_set) == 70_413

t0_exact_unmatched_set = t0_rcv_set.difference(exact_t0_set)

assert len(t0_exact_unmatched_set) == 1_246


# --------------------------------------------------------------------------------------------------
# 11. Read Stage 3D identifier-based candidate pairs
# --------------------------------------------------------------------------------------------------

candidate_pairs = pd.read_parquet(NONEXACT_CANDIDATE_PATH)

assert len(candidate_pairs) == 1_078

candidate_t0_col = infer_rcv_column(
    candidate_pairs,
    valid_rcvs=t0_rcv_set,
    side="t0",
)

candidate_t1_col = infer_rcv_column(
    candidate_pairs,
    valid_rcvs=t1_rcv_set,
    side="t1",
    exclude_columns={candidate_t0_col},
)

candidate_pairs = candidate_pairs.copy()

candidate_pairs["_t0_rcv"] = normalize_rcv(
    candidate_pairs[candidate_t0_col]
)

candidate_pairs["_t1_rcv"] = normalize_rcv(
    candidate_pairs[candidate_t1_col]
)

assert candidate_pairs["_t0_rcv"].isin(t0_exact_unmatched_set).all()
assert candidate_pairs["_t1_rcv"].isin(t1_rcv_set).all()

candidate_t0_set = set(candidate_pairs["_t0_rcv"].dropna())

assert len(candidate_t0_set) == 624

strong_mask, strong_rule_detected = find_strong_candidate_mask(
    candidate_pairs,
    t0_rcv_column="_t0_rcv",
)

strong_candidate_t0_set = set(
    candidate_pairs.loc[strong_mask, "_t0_rcv"].dropna()
)

assert int(strong_mask.sum()) == 248
assert len(strong_candidate_t0_set) == 208


# --------------------------------------------------------------------------------------------------
# 12. Read accepted Stage 3E conservative one-to-one links
# --------------------------------------------------------------------------------------------------

accepted_nonexact = pd.read_parquet(ACCEPTED_NONEXACT_PATH)

assert len(accepted_nonexact) == 170

accepted_t0_col = infer_rcv_column(
    accepted_nonexact,
    valid_rcvs=t0_rcv_set,
    side="t0",
)

accepted_t1_col = infer_rcv_column(
    accepted_nonexact,
    valid_rcvs=t1_rcv_set,
    side="t1",
    exclude_columns={accepted_t0_col},
)

accepted_nonexact = accepted_nonexact.copy()

accepted_nonexact["_t0_rcv"] = normalize_rcv(
    accepted_nonexact[accepted_t0_col]
)

accepted_nonexact["_t1_rcv"] = normalize_rcv(
    accepted_nonexact[accepted_t1_col]
)

assert accepted_nonexact["_t0_rcv"].notna().all()
assert accepted_nonexact["_t1_rcv"].notna().all()
assert accepted_nonexact["_t0_rcv"].is_unique
assert accepted_nonexact["_t1_rcv"].is_unique

accepted_nonexact_t0_set = set(accepted_nonexact["_t0_rcv"])
accepted_nonexact_t1_set = set(accepted_nonexact["_t1_rcv"])

assert len(accepted_nonexact_t0_set) == 170
assert len(accepted_nonexact_t1_set) == 170

assert accepted_nonexact_t0_set.issubset(candidate_t0_set)
assert accepted_nonexact_t0_set.issubset(strong_candidate_t0_set)

assert accepted_nonexact_t0_set.isdisjoint(exact_t0_set)
assert accepted_nonexact_t1_set.isdisjoint(exact_t1_set)

assert accepted_nonexact_t0_set.issubset(t0_rcv_set)
assert accepted_nonexact_t1_set.issubset(t1_rcv_set)

nonexact_mapping = dict(
    zip(
        accepted_nonexact["_t0_rcv"],
        accepted_nonexact["_t1_rcv"],
    )
)


# --------------------------------------------------------------------------------------------------
# 13. Derive the three frozen unresolved/censoring categories
# --------------------------------------------------------------------------------------------------

complex_strong_unresolved_set = (
    strong_candidate_t0_set
    .difference(accepted_nonexact_t0_set)
)

variant_condition_unresolved_set = (
    candidate_t0_set
    .difference(accepted_nonexact_t0_set)
    .difference(complex_strong_unresolved_set)
)

no_identifier_candidate_set = (
    t0_exact_unmatched_set
    .difference(candidate_t0_set)
)

assert len(complex_strong_unresolved_set) == 38
assert len(variant_condition_unresolved_set) == 416
assert len(no_identifier_candidate_set) == 622

assert complex_strong_unresolved_set.isdisjoint(
    variant_condition_unresolved_set
)

assert complex_strong_unresolved_set.isdisjoint(
    no_identifier_candidate_set
)

assert variant_condition_unresolved_set.isdisjoint(
    no_identifier_candidate_set
)


# --------------------------------------------------------------------------------------------------
# 14. Independently verify Stage 3G no-identifier T0 audit
# --------------------------------------------------------------------------------------------------

stage3g_t0_audit = pd.read_parquet(STAGE3G_T0_AUDIT_PATH)

stage3g_t0_col = infer_rcv_column(
    stage3g_t0_audit,
    valid_rcvs=t0_rcv_set,
    side="t0",
)

stage3g_t0_set = set(
    normalize_rcv(stage3g_t0_audit[stage3g_t0_col]).dropna()
)

assert len(stage3g_t0_set) == 622
assert stage3g_t0_set == no_identifier_candidate_set


# --------------------------------------------------------------------------------------------------
# 15. Freeze the Stage 3H linkage and censoring policy in memory
#     This policy contains no observed temporal outcome.
# --------------------------------------------------------------------------------------------------

linkage_policy = {
    "policy_name": "Stage 3H final T0-T1 RCV linkage and censoring policy",
    "policy_version": "1.0.0",
    "policy_status": "FROZEN_BEFORE_TEMPORAL_OUTCOME_CONSTRUCTION",
    "unit_of_analysis": "RCV-level variant-condition aggregate",
    "accepted_linkage_rules": {
        "exact_rcv": {
            "decision_category": "ACCEPTED_EXACT_RCV",
            "rule": (
                "The normalized T0 RCV accession is exactly present as a unique "
                "RCV accession in the accepted T1 cohort."
            ),
        },
        "conservative_nonexact_one_to_one": {
            "decision_category": "ACCEPTED_NONEXACT_TIER1_ONE_TO_ONE",
            "rule": (
                "Use only the 170 Stage 3E accepted links satisfying exact "
                "VariationID, exact VCV accession, exact target gene, condition "
                "overlap, and unique strong-candidate topology on both sides."
            ),
            "interpretation": (
                "Accepted as a conservative continuity link, not asserted as an "
                "official ClinVar accession replacement."
            ),
        },
    },
    "censoring_dispositions": {
        "CENSORED_COMPLEX_STRONG_CANDIDATE": {
            "expected_t0_records": 38,
            "reason": (
                "Strong identifier and condition evidence exists, but candidate "
                "topology is nonunique, split, merge, or complex."
            ),
        },
        "CENSORED_VARIANT_CONTINUITY_CONDITION_ASSOCIATION_UNRESOLVED": {
            "expected_t0_records": 416,
            "reason": (
                "VariationID or VCV continuity exists, but condition-association "
                "continuity is not sufficiently resolved for the RCV-level unit."
            ),
        },
        "CENSORED_NO_VARIATIONID_OR_VCV_CANDIDATE": {
            "expected_t0_records": 622,
            "reason": (
                "No VariationID- or VCV-based T1 candidate exists. Condition-only "
                "similarity cannot establish variant continuity."
            ),
        },
    },
    "expected_complete_accounting": {
        "total_t0_records": 71_659,
        "accepted_exact_links": 70_413,
        "accepted_nonexact_links": 170,
        "total_linked": 70_583,
        "total_unresolved_censored": 1_076,
    },
    "classification_axis_policy": (
        "Germline, oncogenicity, somatic clinical-impact, and no-classification "
        "axes remain distinct. No axis is converted into another during linkage."
    ),
    "prohibited_actions": [
        "Do not infer a T1 link from condition similarity alone.",
        "Do not attach T1 evidence to an unresolved T0 record.",
        "Do not label an unresolved record stable.",
        "Do not calculate a classification-change outcome in Stage 3.",
        "Do not create a future-instability outcome in Stage 3.",
        "Do not fit or tune GES in Stage 3.",
    ],
}


# --------------------------------------------------------------------------------------------------
# 16. Build one final linkage-decision row per T0 RCV
# --------------------------------------------------------------------------------------------------

linkage_core = pd.DataFrame(
    {
        "t0_rcv_accession": t0["rcv_accession"].copy(),
    }
)

linkage_core["linked_t1_rcv_accession"] = pd.Series(
    pd.NA,
    index=linkage_core.index,
    dtype="string",
)

linkage_core["linkage_status"] = pd.Series(
    pd.NA,
    index=linkage_core.index,
    dtype="string",
)

linkage_core["linkage_method"] = pd.Series(
    pd.NA,
    index=linkage_core.index,
    dtype="string",
)

linkage_core["linkage_decision_category"] = pd.Series(
    pd.NA,
    index=linkage_core.index,
    dtype="string",
)

linkage_core["censoring_disposition"] = pd.Series(
    pd.NA,
    index=linkage_core.index,
    dtype="string",
)


# Exact RCV links
exact_mask = linkage_core["t0_rcv_accession"].isin(exact_t0_set)

linkage_core.loc[
    exact_mask,
    "linked_t1_rcv_accession",
] = linkage_core.loc[
    exact_mask,
    "t0_rcv_accession",
]

linkage_core.loc[exact_mask, "linkage_status"] = "ACCEPTED_LINK"
linkage_core.loc[exact_mask, "linkage_method"] = "EXACT_RCV_ACCESSION"
linkage_core.loc[
    exact_mask,
    "linkage_decision_category",
] = "ACCEPTED_EXACT_RCV"

linkage_core.loc[
    exact_mask,
    "censoring_disposition",
] = "NOT_CENSORED"


# Accepted conservative non-exact links
nonexact_mask = linkage_core["t0_rcv_accession"].isin(
    accepted_nonexact_t0_set
)

linkage_core.loc[
    nonexact_mask,
    "linked_t1_rcv_accession",
] = (
    linkage_core.loc[nonexact_mask, "t0_rcv_accession"]
    .map(nonexact_mapping)
    .astype("string")
)

linkage_core.loc[nonexact_mask, "linkage_status"] = "ACCEPTED_LINK"

linkage_core.loc[
    nonexact_mask,
    "linkage_method",
] = "CONSERVATIVE_TIER1_ONE_TO_ONE"

linkage_core.loc[
    nonexact_mask,
    "linkage_decision_category",
] = "ACCEPTED_NONEXACT_TIER1_ONE_TO_ONE"

linkage_core.loc[
    nonexact_mask,
    "censoring_disposition",
] = "NOT_CENSORED"


# Complex strong-candidate records
complex_mask = linkage_core["t0_rcv_accession"].isin(
    complex_strong_unresolved_set
)

linkage_core.loc[
    complex_mask,
    "linkage_status",
] = "CENSORED_UNRESOLVED"

linkage_core.loc[
    complex_mask,
    "linkage_method",
] = "NO_ACCEPTED_T1_LINK"

linkage_core.loc[
    complex_mask,
    "linkage_decision_category",
] = "CENSORED_COMPLEX_STRONG_CANDIDATE"

linkage_core.loc[
    complex_mask,
    "censoring_disposition",
] = "CENSORED_COMPLEX_STRONG_CANDIDATE"


# Variant continuity but unresolved condition association
condition_unresolved_mask = linkage_core["t0_rcv_accession"].isin(
    variant_condition_unresolved_set
)

linkage_core.loc[
    condition_unresolved_mask,
    "linkage_status",
] = "CENSORED_UNRESOLVED"

linkage_core.loc[
    condition_unresolved_mask,
    "linkage_method",
] = "NO_ACCEPTED_T1_LINK"

linkage_core.loc[
    condition_unresolved_mask,
    "linkage_decision_category",
] = (
    "CENSORED_VARIANT_CONTINUITY_CONDITION_ASSOCIATION_UNRESOLVED"
)

linkage_core.loc[
    condition_unresolved_mask,
    "censoring_disposition",
] = (
    "CENSORED_VARIANT_CONTINUITY_CONDITION_ASSOCIATION_UNRESOLVED"
)


# No stable variant-identifier candidate
no_identifier_mask = linkage_core["t0_rcv_accession"].isin(
    no_identifier_candidate_set
)

linkage_core.loc[
    no_identifier_mask,
    "linkage_status",
] = "CENSORED_UNRESOLVED"

linkage_core.loc[
    no_identifier_mask,
    "linkage_method",
] = "NO_ACCEPTED_T1_LINK"

linkage_core.loc[
    no_identifier_mask,
    "linkage_decision_category",
] = "CENSORED_NO_VARIATIONID_OR_VCV_CANDIDATE"

linkage_core.loc[
    no_identifier_mask,
    "censoring_disposition",
] = "CENSORED_NO_VARIATIONID_OR_VCV_CANDIDATE"


# Explicit scientific-boundary fields
linkage_core["temporal_outcome_eligible"] = (
    linkage_core["linkage_status"].eq("ACCEPTED_LINK")
)

linkage_core["future_instability_outcome_created"] = False

linkage_core["future_instability_label"] = pd.Series(
    pd.NA,
    index=linkage_core.index,
    dtype="string",
)

linkage_core["stage3_freeze_version"] = "1.0.0"


# --------------------------------------------------------------------------------------------------
# 17. Validate linkage-decision accounting before attaching T1 records
# --------------------------------------------------------------------------------------------------

assert len(linkage_core) == 71_659
assert linkage_core["t0_rcv_accession"].is_unique

mandatory_decision_columns = [
    "linkage_status",
    "linkage_method",
    "linkage_decision_category",
    "censoring_disposition",
]

assert linkage_core[mandatory_decision_columns].notna().all().all()

expected_decision_counts = {
    "ACCEPTED_EXACT_RCV": 70_413,
    "ACCEPTED_NONEXACT_TIER1_ONE_TO_ONE": 170,
    "CENSORED_COMPLEX_STRONG_CANDIDATE": 38,
    "CENSORED_VARIANT_CONTINUITY_CONDITION_ASSOCIATION_UNRESOLVED": 416,
    "CENSORED_NO_VARIATIONID_OR_VCV_CANDIDATE": 622,
}

observed_decision_counts = (
    linkage_core["linkage_decision_category"]
    .value_counts()
    .to_dict()
)

assert observed_decision_counts == expected_decision_counts, (
    f"Final disposition accounting mismatch:\n"
    f"Expected: {expected_decision_counts}\n"
    f"Observed: {observed_decision_counts}"
)

linked_mask = linkage_core["temporal_outcome_eligible"]
censored_mask = ~linked_mask

assert int(linked_mask.sum()) == 70_583
assert int(censored_mask.sum()) == 1_076

assert (
    linkage_core.loc[linked_mask, "linked_t1_rcv_accession"]
    .notna()
    .all()
)

assert (
    linkage_core.loc[censored_mask, "linked_t1_rcv_accession"]
    .isna()
    .all()
)

assert (
    linkage_core.loc[linked_mask, "linked_t1_rcv_accession"]
    .is_unique
)

assert (
    linkage_core["future_instability_outcome_created"]
    .eq(False)
    .all()
)

assert linkage_core["future_instability_label"].isna().all()


# --------------------------------------------------------------------------------------------------
# 18. Prefix all accepted T0 and T1 evidence fields and assemble the final artifact
# --------------------------------------------------------------------------------------------------

t0_prefixed = t0.copy()
t0_prefixed.insert(0, "row_order", range(len(t0_prefixed)))
t0_prefixed = t0_prefixed.rename(
    columns={column: f"t0_{column}" for column in t0_prefixed.columns}
)

t1_prefixed = t1.copy()
t1_prefixed.insert(0, "row_order", range(len(t1_prefixed)))
t1_prefixed = t1_prefixed.rename(
    columns={column: f"t1_{column}" for column in t1_prefixed.columns}
)

final_linkage = t0_prefixed.merge(
    linkage_core,
    on="t0_rcv_accession",
    how="left",
    validate="one_to_one",
)

final_linkage = final_linkage.merge(
    t1_prefixed,
    left_on="linked_t1_rcv_accession",
    right_on="t1_rcv_accession",
    how="left",
    validate="many_to_one",
)

final_linkage = final_linkage.sort_values(
    "t0_row_order",
    kind="stable",
).reset_index(drop=True)


# --------------------------------------------------------------------------------------------------
# 19. Final linkage integrity checks
# --------------------------------------------------------------------------------------------------

assert len(final_linkage) == 71_659
assert final_linkage["t0_rcv_accession"].is_unique

final_linked_mask = final_linkage["temporal_outcome_eligible"]
final_censored_mask = ~final_linked_mask

assert int(final_linked_mask.sum()) == 70_583
assert int(final_censored_mask.sum()) == 1_076

assert (
    final_linkage.loc[final_linked_mask, "t1_rcv_accession"]
    .notna()
    .all()
)

assert (
    final_linkage.loc[final_censored_mask, "t1_rcv_accession"]
    .isna()
    .all()
)

t1_attached_columns = [
    column
    for column in final_linkage.columns
    if column.startswith("t1_")
]

assert (
    final_linkage.loc[final_censored_mask, t1_attached_columns]
    .isna()
    .all()
    .all()
)

exact_final_mask = final_linkage[
    "linkage_decision_category"
].eq("ACCEPTED_EXACT_RCV")

assert (
    final_linkage.loc[
        exact_final_mask,
        "t0_rcv_accession",
    ].reset_index(drop=True)
    ==
    final_linkage.loc[
        exact_final_mask,
        "t1_rcv_accession",
    ].reset_index(drop=True)
).all()

nonexact_final_mask = final_linkage[
    "linkage_decision_category"
].eq("ACCEPTED_NONEXACT_TIER1_ONE_TO_ONE")

assert (
    final_linkage.loc[
        nonexact_final_mask,
        "t0_rcv_accession",
    ].reset_index(drop=True)
    !=
    final_linkage.loc[
        nonexact_final_mask,
        "t1_rcv_accession",
    ].reset_index(drop=True)
).all()

assert (
    final_linkage.loc[
        final_linked_mask,
        "t1_rcv_accession",
    ].nunique()
    == 70_583
)


# --------------------------------------------------------------------------------------------------
# 20. Construct the complete T1 usage audit
# --------------------------------------------------------------------------------------------------

exact_usage_map = {
    rcv: rcv
    for rcv in exact_t0_set
}

nonexact_usage_map = {
    t1_rcv: t0_rcv
    for t0_rcv, t1_rcv in nonexact_mapping.items()
}

combined_t1_to_t0_map = {
    **exact_usage_map,
    **nonexact_usage_map,
}

assert len(combined_t1_to_t0_map) == 70_583

usage_columns = [
    "t1_row_order",
    "t1_rcv_accession",
]

for optional_column in [
    "t1_rcv_version",
    "t1_variation_id",
    "t1_vcv_accession",
    "t1_vcv_version",
    "t1_target_genes_json",
    "t1_study_scope",
    "t1_aggregate_classification_axis",
]:
    if optional_column in t1_prefixed.columns:
        usage_columns.append(optional_column)

t1_usage_audit = t1_prefixed[usage_columns].copy()

t1_usage_audit["linked_t0_rcv_accession"] = (
    t1_usage_audit["t1_rcv_accession"]
    .map(combined_t1_to_t0_map)
    .astype("string")
)

t1_usage_audit["used_in_final_stage3_linkage"] = (
    t1_usage_audit["linked_t0_rcv_accession"].notna()
)

t1_usage_audit["usage_category"] = "NOT_USED_IN_STAGE3_FINAL_LINKAGE"

t1_usage_audit.loc[
    t1_usage_audit["t1_rcv_accession"].isin(exact_t1_set),
    "usage_category",
] = "LINKED_BY_EXACT_RCV"

t1_usage_audit.loc[
    t1_usage_audit["t1_rcv_accession"].isin(accepted_nonexact_t1_set),
    "usage_category",
] = "LINKED_BY_CONSERVATIVE_TIER1_ONE_TO_ONE"

t1_usage_audit["stage3_freeze_version"] = "1.0.0"

t1_usage_audit = t1_usage_audit.sort_values(
    "t1_row_order",
    kind="stable",
).reset_index(drop=True)

expected_t1_usage_counts = {
    "LINKED_BY_EXACT_RCV": 70_413,
    "LINKED_BY_CONSERVATIVE_TIER1_ONE_TO_ONE": 170,
    "NOT_USED_IN_STAGE3_FINAL_LINKAGE": 30_337,
}

observed_t1_usage_counts = (
    t1_usage_audit["usage_category"]
    .value_counts()
    .to_dict()
)

assert observed_t1_usage_counts == expected_t1_usage_counts

assert len(t1_usage_audit) == 100_920
assert t1_usage_audit["t1_rcv_accession"].is_unique
assert int(t1_usage_audit["used_in_final_stage3_linkage"].sum()) == 70_583

assert (
    t1_usage_audit.loc[
        t1_usage_audit["used_in_final_stage3_linkage"],
        "linked_t0_rcv_accession",
    ].is_unique
)


# --------------------------------------------------------------------------------------------------
# 21. Build artifacts in temporary local storage
# --------------------------------------------------------------------------------------------------

build_dir = Path("/content/stage3h_final_build_v1")

if build_dir.exists():
    shutil.rmtree(build_dir)

build_dir.mkdir(parents=True, exist_ok=False)

temporary_policy_path = build_dir / LINKAGE_POLICY_PATH.name
temporary_linkage_path = build_dir / FINAL_LINKAGE_PATH.name
temporary_usage_path = build_dir / T1_USAGE_AUDIT_PATH.name
temporary_report_path = build_dir / FREEZE_REPORT_PATH.name
temporary_manifest_path = build_dir / FREEZE_MANIFEST_PATH.name

write_json(temporary_policy_path, linkage_policy)

final_linkage.to_parquet(
    temporary_linkage_path,
    index=False,
    engine="pyarrow",
    compression="zstd",
)

t1_usage_audit.to_parquet(
    temporary_usage_path,
    index=False,
    engine="pyarrow",
    compression="zstd",
)


# --------------------------------------------------------------------------------------------------
# 22. Readback validation
# --------------------------------------------------------------------------------------------------

linkage_readback = pd.read_parquet(temporary_linkage_path)
usage_readback = pd.read_parquet(temporary_usage_path)

assert linkage_readback.shape == final_linkage.shape
assert usage_readback.shape == t1_usage_audit.shape

assert linkage_readback["t0_rcv_accession"].is_unique
assert usage_readback["t1_rcv_accession"].is_unique

readback_decision_counts = (
    linkage_readback["linkage_decision_category"]
    .value_counts()
    .to_dict()
)

assert readback_decision_counts == expected_decision_counts

readback_usage_counts = (
    usage_readback["usage_category"]
    .value_counts()
    .to_dict()
)

assert readback_usage_counts == expected_t1_usage_counts

assert (
    linkage_readback["future_instability_outcome_created"]
    .eq(False)
    .all()
)

assert linkage_readback["future_instability_label"].isna().all()


# --------------------------------------------------------------------------------------------------
# 23. Calculate output hashes
# --------------------------------------------------------------------------------------------------

policy_hash = sha256_file(temporary_policy_path)
final_linkage_hash = sha256_file(temporary_linkage_path)
t1_usage_hash = sha256_file(temporary_usage_path)

created_at_utc = datetime.now(timezone.utc).isoformat()


# --------------------------------------------------------------------------------------------------
# 24. Freeze report
# --------------------------------------------------------------------------------------------------

freeze_report = {
    "report_name": "Stage 3H final linkage freeze report",
    "report_version": "1.0.0",
    "created_at_utc": created_at_utc,
    "stage": "3H",
    "stage_status": "FINAL_LINKAGE_ASSEMBLY_COMPLETE",
    "validation_decision": "PASS",
    "input_validation": {
        "accepted_t0_sha256_verified": True,
        "accepted_t1_sha256_verified": True,
        "stage3b_exact_crosswalk_sha256_verified": True,
        "stage3d_candidate_pairs_sha256_verified": True,
        "stage3e_accepted_links_sha256_verified": True,
        "stage3f_scientific_parquet_hashes_verified": True,
        "stage3f_manifest_semantically_validated": True,
        "stage3f_current_manifest_sha256": current_stage3f_manifest_hash,
        "stage3g_no_identifier_t0_audit_sha256_verified": True,
        "strong_candidate_rule_detected": strong_rule_detected,
    },
    "final_linkage_accounting": {
        "total_t0_records": 71_659,
        "accepted_exact_rcv_links": 70_413,
        "accepted_nonexact_tier1_one_to_one_links": 170,
        "total_accepted_links": 70_583,
        "complex_strong_candidate_censored": 38,
        "variant_continuity_condition_association_censored": 416,
        "no_variationid_or_vcv_candidate_censored": 622,
        "total_unresolved_censored": 1_076,
    },
    "t1_usage_accounting": {
        "total_t1_records": 100_920,
        "linked_by_exact_rcv": 70_413,
        "linked_by_nonexact_one_to_one": 170,
        "unique_t1_records_used": 70_583,
        "t1_records_not_used": 30_337,
        "t1_records_used_more_than_once": 0,
    },
    "validation_checks": {
        "one_row_per_t0_rcv": True,
        "all_t0_rcvs_accounted_for": True,
        "all_accepted_links_have_t1_evidence": True,
        "all_censored_records_have_no_attached_t1_evidence": True,
        "accepted_t1_records_are_unique": True,
        "exact_links_retain_identical_rcv_accession": True,
        "nonexact_links_are_exactly_the_frozen_stage3e_set": True,
        "stage3g_no_identifier_set_matches_derived_set": True,
        "unresolved_records_labeled_stable": False,
        "future_instability_outcome_created": False,
        "classification_change_outcome_created": False,
        "ges_model_fitted": False,
        "ges_threshold_selected": False,
    },
    "output_artifacts": {
        "linkage_policy": {
            "path": str(LINKAGE_POLICY_PATH),
            "sha256": policy_hash,
        },
        "final_linkage_parquet": {
            "path": str(FINAL_LINKAGE_PATH),
            "sha256": final_linkage_hash,
            "rows": int(final_linkage.shape[0]),
            "columns": int(final_linkage.shape[1]),
        },
        "t1_usage_audit_parquet": {
            "path": str(T1_USAGE_AUDIT_PATH),
            "sha256": t1_usage_hash,
            "rows": int(t1_usage_audit.shape[0]),
            "columns": int(t1_usage_audit.shape[1]),
        },
    },
}

write_json(temporary_report_path, freeze_report)
freeze_report_hash = sha256_file(temporary_report_path)


# --------------------------------------------------------------------------------------------------
# 25. Final freeze manifest
# --------------------------------------------------------------------------------------------------

freeze_manifest = {
    "manifest_name": "Stage 3H final T0-T1 linkage freeze manifest",
    "manifest_version": "1.0.0",
    "created_at_utc": created_at_utc,
    "stage": "3H",
    "manifest_status": "STAGE3_FINAL_LINKAGE_FROZEN",
    "validation_decision": (
        "PASS_STAGE3_FINAL_LINKAGE_ACCEPTED_AND_FROZEN"
    ),
    "scientific_unit": "RCV-level variant-condition aggregate",
    "source_artifacts": {
        str(path): {
            "sha256": observed_hash,
        }
        for path, observed_hash in sorted(
            verified_input_hashes.items()
        )
    },
    "semantic_stage3f_manifest": {
        "path": str(STAGE3F_MANIFEST_PATH),
        "current_sha256": current_stage3f_manifest_hash,
        "validation_method": (
            "Semantic validation of manifest status, decision, declared "
            "scientific artifact paths, declared hashes, current hashes, "
            "and scientific-boundary flags."
        ),
        "validation_result": "PASS",
    },
    "final_accounting": {
        "total_t0_records": 71_659,
        "total_linked_records": 70_583,
        "total_unresolved_censored_records": 1_076,
        "accepted_exact_links": 70_413,
        "accepted_nonexact_links": 170,
        "complex_strong_candidate_censored": 38,
        "variant_condition_association_censored": 416,
        "no_identifier_candidate_censored": 622,
    },
    "frozen_outputs": {
        "linkage_policy": {
            "path": str(LINKAGE_POLICY_PATH),
            "sha256": policy_hash,
        },
        "final_linkage": {
            "path": str(FINAL_LINKAGE_PATH),
            "sha256": final_linkage_hash,
            "rows": int(final_linkage.shape[0]),
            "columns": int(final_linkage.shape[1]),
        },
        "t1_usage_audit": {
            "path": str(T1_USAGE_AUDIT_PATH),
            "sha256": t1_usage_hash,
            "rows": int(t1_usage_audit.shape[0]),
            "columns": int(t1_usage_audit.shape[1]),
        },
        "freeze_report": {
            "path": str(FREEZE_REPORT_PATH),
            "sha256": freeze_report_hash,
        },
    },
    "scientific_boundary": {
        "unresolved_records_labeled_stable": False,
        "future_instability_outcome_created": False,
        "classification_change_outcome_created": False,
        "ges_model_fitted": False,
        "ges_model_tuned": False,
        "ges_threshold_selected": False,
        "rag_experiment_started": False,
    },
    "stage3_decision": "COMPLETE",
    "next_authorized_stage": (
        "Blueprint Stage 4: construct and freeze T0-only baseline features, "
        "weak-supervision specification, full GES, and mandatory no-star GES. "
        "Temporal outcomes must use only this frozen linkage."
    ),
}

write_json(temporary_manifest_path, freeze_manifest)
freeze_manifest_hash = sha256_file(temporary_manifest_path)


# --------------------------------------------------------------------------------------------------
# 26. Persist outputs to Drive
#     The freeze manifest is copied last.
# --------------------------------------------------------------------------------------------------

copy_and_verify(temporary_policy_path, LINKAGE_POLICY_PATH)
copy_and_verify(temporary_linkage_path, FINAL_LINKAGE_PATH)
copy_and_verify(temporary_usage_path, T1_USAGE_AUDIT_PATH)
copy_and_verify(temporary_report_path, FREEZE_REPORT_PATH)
copy_and_verify(temporary_manifest_path, FREEZE_MANIFEST_PATH)


# --------------------------------------------------------------------------------------------------
# 27. Final persistent-file validation
# --------------------------------------------------------------------------------------------------

assert sha256_file(LINKAGE_POLICY_PATH) == policy_hash
assert sha256_file(FINAL_LINKAGE_PATH) == final_linkage_hash
assert sha256_file(T1_USAGE_AUDIT_PATH) == t1_usage_hash
assert sha256_file(FREEZE_REPORT_PATH) == freeze_report_hash
assert sha256_file(FREEZE_MANIFEST_PATH) == freeze_manifest_hash

persistent_linkage = pd.read_parquet(FINAL_LINKAGE_PATH)
persistent_usage = pd.read_parquet(T1_USAGE_AUDIT_PATH)

assert persistent_linkage.shape == final_linkage.shape
assert persistent_usage.shape == t1_usage_audit.shape

assert (
    persistent_linkage["linkage_decision_category"]
    .value_counts()
    .to_dict()
    == expected_decision_counts
)

assert (
    persistent_usage["usage_category"]
    .value_counts()
    .to_dict()
    == expected_t1_usage_counts
)


# --------------------------------------------------------------------------------------------------
# 28. Final Stage 3H completion output
# --------------------------------------------------------------------------------------------------

print("=" * 120)
print("STAGE 3H — FINAL T0–T1 LINKAGE ASSEMBLY AND FREEZE")
print("=" * 120)

print(f"Created at UTC:                 {created_at_utc}")
print("Stage 3 decision:              COMPLETE")
print("Validation decision:           PASS_STAGE3_FINAL_LINKAGE_ACCEPTED_AND_FROZEN")
print()

print("FINAL T0 ACCOUNTING")
print(f"  Total T0 records:            {len(persistent_linkage):,}")
print(f"  Exact RCV links:             {expected_decision_counts['ACCEPTED_EXACT_RCV']:,}")
print(
    "  Non-exact one-to-one links:  "
    f"{expected_decision_counts['ACCEPTED_NONEXACT_TIER1_ONE_TO_ONE']:,}"
)
print(
    "  Complex strong censored:     "
    f"{expected_decision_counts['CENSORED_COMPLEX_STRONG_CANDIDATE']:,}"
)
print(
    "  Condition unresolved:        "
    f"{expected_decision_counts['CENSORED_VARIANT_CONTINUITY_CONDITION_ASSOCIATION_UNRESOLVED']:,}"
)
print(
    "  No identifier candidate:     "
    f"{expected_decision_counts['CENSORED_NO_VARIATIONID_OR_VCV_CANDIDATE']:,}"
)
print(f"  Total linked:                {int(persistent_linkage['temporal_outcome_eligible'].sum()):,}")
print(f"  Total unresolved/censored:   {int((~persistent_linkage['temporal_outcome_eligible']).sum()):,}")
print()

print("T1 USAGE ACCOUNTING")
for category, count in expected_t1_usage_counts.items():
    print(f"  {category:<43} {count:,}")
print()

print("SCIENTIFIC BOUNDARY")
print("  Future-instability outcome created:     False")
print("  Classification-change outcome created:  False")
print("  Unresolved records labeled stable:      False")
print("  GES model fitted or tuned:               False")
print()

print("FROZEN OUTPUTS")
print(f"  Linkage policy:              {LINKAGE_POLICY_PATH}")
print(f"    SHA-256:                   {policy_hash}")
print(f"  Final linkage:              {FINAL_LINKAGE_PATH}")
print(f"    Rows / columns:            {persistent_linkage.shape[0]:,} / {persistent_linkage.shape[1]:,}")
print(f"    SHA-256:                   {final_linkage_hash}")
print(f"  T1 usage audit:             {T1_USAGE_AUDIT_PATH}")
print(f"    Rows / columns:            {persistent_usage.shape[0]:,} / {persistent_usage.shape[1]:,}")
print(f"    SHA-256:                   {t1_usage_hash}")
print(f"  Freeze report:              {FREEZE_REPORT_PATH}")
print(f"    SHA-256:                   {freeze_report_hash}")
print(f"  Freeze manifest:            {FREEZE_MANIFEST_PATH}")
print(f"    SHA-256:                   {freeze_manifest_hash}")
print()

print("NEXT AUTHORIZED WORK")
print("  Blueprint Stage 4 — T0-only feature construction and freezing of")
print("  the full GES and mandatory no-star GES specifications.")
print("=" * 120)

/tmp/ipykernel_569/3656337636.py:402: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  | text.str.contains(
/tmp/ipykernel_569/3656337636.py:402: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  | text.str.contains(


STAGE 3H — FINAL T0–T1 LINKAGE ASSEMBLY AND FREEZE
Created at UTC:                 2026-07-20T21:31:58.438904+00:00
Stage 3 decision:              COMPLETE
Validation decision:           PASS_STAGE3_FINAL_LINKAGE_ACCEPTED_AND_FROZEN

FINAL T0 ACCOUNTING
  Total T0 records:            71,659
  Exact RCV links:             70,413
  Non-exact one-to-one links:  170
  Complex strong censored:     38
  Condition unresolved:        416
  No identifier candidate:     622
  Total linked:                70,583
  Total unresolved/censored:   1,076

T1 USAGE ACCOUNTING
  LINKED_BY_EXACT_RCV                         70,413
  LINKED_BY_CONSERVATIVE_TIER1_ONE_TO_ONE     170
  NOT_USED_IN_STAGE3_FINAL_LINKAGE            30,337

SCIENTIFIC BOUNDARY
  Future-instability outcome created:     False
  Classification-change outcome created:  False
  Unresolved records labeled stable:      False
  GES model fitted or tuned:               False

FROZEN OUTPUTS
  Linkage policy:              /content/drive/MyD

In [25]:
# ==================================================================================================
# STAGE 4A — T0-ONLY BASELINE FEATURE CONSTRUCTION AND CHECKSUM FREEZE
#
# Builds:
#   1. Recency features
#   2. Submitter-structure features
#   3. Review-confidence feature
#   4. Aggregate conflict feature
#   5. SCV classification-entropy features
#
# Scientific boundary:
#   - Uses only the accepted and frozen T0 cohort.
#   - Stage 3H is used only to verify that every T0 RCV is represented.
#   - No T1 evidence is included in the feature table.
#   - No future-instability outcome is constructed.
#   - No weak labels or GES model are fitted in this cell.
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import math
import os
import shutil

import numpy as np
import pandas as pd


# --------------------------------------------------------------------------------------------------
# 1. Google Drive
# --------------------------------------------------------------------------------------------------

DRIVE_ROOT = Path("/content/drive/MyDrive")

if not DRIVE_ROOT.exists():
    from google.colab import drive
    drive.mount("/content/drive")

assert DRIVE_ROOT.exists(), "Google Drive is not mounted."


# --------------------------------------------------------------------------------------------------
# 2. Paths
# --------------------------------------------------------------------------------------------------

BASE_DIR = DRIVE_ROOT / "GES_RAG_Temporal_Study"

T0_PATH = (
    BASE_DIR
    / "data_interim"
    / "t0_rcv_target_genes_corrected_v1_2.parquet"
)

STAGE3H_LINKAGE_PATH = (
    BASE_DIR
    / "data_processed"
    / "stage3_crosswalk"
    / "stage3h_final_t0_t1_linkage_v1.parquet"
)

STAGE3H_MANIFEST_PATH = (
    BASE_DIR
    / "configs"
    / "stage3_crosswalk"
    / "stage3h_final_linkage_freeze_manifest_v1.json"
)

STAGE4_DATA_DIR = BASE_DIR / "data_processed" / "stage4_ges"
STAGE4_CONFIG_DIR = BASE_DIR / "configs" / "stage4_ges"

STAGE4_DATA_DIR.mkdir(parents=True, exist_ok=True)
STAGE4_CONFIG_DIR.mkdir(parents=True, exist_ok=True)


# --------------------------------------------------------------------------------------------------
# 3. Output files
# --------------------------------------------------------------------------------------------------

FEATURE_TABLE_PATH = (
    STAGE4_DATA_DIR
    / "stage4a_t0_ges_baseline_features_v1.parquet"
)

FEATURE_SPEC_PATH = (
    STAGE4_CONFIG_DIR
    / "stage4a_t0_feature_specification_v1.json"
)

QC_REPORT_PATH = (
    STAGE4_CONFIG_DIR
    / "stage4a_t0_feature_qc_report_v1.json"
)

FREEZE_MANIFEST_PATH = (
    STAGE4_CONFIG_DIR
    / "stage4a_t0_feature_freeze_manifest_v1.json"
)

OUTPUT_PATHS = [
    FEATURE_TABLE_PATH,
    FEATURE_SPEC_PATH,
    QC_REPORT_PATH,
    FREEZE_MANIFEST_PATH,
]

existing_outputs = [str(path) for path in OUTPUT_PATHS if path.exists()]

if existing_outputs:
    raise FileExistsError(
        "Stage 4A output files already exist and will not be overwritten:\n"
        + "\n".join(existing_outputs)
    )


# --------------------------------------------------------------------------------------------------
# 4. Accepted input hashes
# --------------------------------------------------------------------------------------------------

EXPECTED_HASHES = {
    str(T0_PATH): (
        "f6b6760b2ad6e4352e3bdecdeaf89827e8a514b031abf2373bb17568d999466d"
    ),
    str(STAGE3H_LINKAGE_PATH): (
        "77d0522af5ec3ac0938a03aecdb9ebd230d6cafae8e8ded7a8de1fd462cfbe75"
    ),
    str(STAGE3H_MANIFEST_PATH): (
        "5b8dd32aa03456eb5a340c4ae5ff7bc1583752151dd555f969accb91d673bca0"
    ),
}


# --------------------------------------------------------------------------------------------------
# 5. Utility functions
# --------------------------------------------------------------------------------------------------

def sha256_file(path: Path, chunk_size: int = 8 * 1024 * 1024) -> str:
    """Calculate SHA-256 without loading the entire file into memory."""
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def write_json(path: Path, payload: dict) -> None:
    """Write deterministic and human-readable JSON."""
    text = json.dumps(
        payload,
        indent=2,
        sort_keys=True,
        ensure_ascii=False,
    )

    path.write_text(text + "\n", encoding="utf-8")


def normalize_rcv(series: pd.Series) -> pd.Series:
    """Normalize RCV accessions without inferring replacements."""
    return (
        series.astype("string")
        .str.strip()
        .str.upper()
    )


def normalize_boolean(series: pd.Series, field_name: str) -> pd.Series:
    """Convert common Boolean representations to strict Boolean values."""

    if pd.api.types.is_bool_dtype(series):
        result = series.astype(bool)

    else:
        normalized = (
            series.astype("string")
            .str.strip()
            .str.lower()
        )

        mapping = {
            "true": True,
            "false": False,
            "1": True,
            "0": False,
            "yes": True,
            "no": False,
            "y": True,
            "n": False,
        }

        result = normalized.map(mapping)

    if result.isna().any():
        examples = series.loc[result.isna()].head(10).tolist()

        raise ValueError(
            f"Unable to convert {field_name} to Boolean. "
            f"Examples: {examples}"
        )

    return result.astype(bool)


def parse_count_dictionary(value, row_number: int) -> dict:
    """Parse an SCV classification-group count dictionary."""

    if isinstance(value, dict):
        parsed = value

    elif isinstance(value, str):
        try:
            parsed = json.loads(value)
        except json.JSONDecodeError as exc:
            raise ValueError(
                f"Malformed JSON at row {row_number}: {value[:200]}"
            ) from exc

    else:
        raise TypeError(
            f"Expected dictionary or JSON string at row {row_number}; "
            f"observed {type(value).__name__}"
        )

    if not isinstance(parsed, dict):
        raise TypeError(
            f"Expected dictionary at row {row_number}; "
            f"observed {type(parsed).__name__}"
        )

    cleaned = {}

    for key, raw_count in parsed.items():
        try:
            count = int(raw_count)
        except (TypeError, ValueError) as exc:
            raise ValueError(
                f"Invalid count at row {row_number}: "
                f"{key}={raw_count}"
            ) from exc

        if count < 0:
            raise ValueError(
                f"Negative count at row {row_number}: "
                f"{key}={count}"
            )

        cleaned[str(key)] = count

    return cleaned


def calculate_entropy_metrics(count_dictionary: dict) -> dict:
    """
    Calculate Shannon entropy from SCV broad-group counts.

    Entropy is based only on the distribution of submitted classifications
    available at T0.
    """

    positive_counts = np.array(
        [
            float(count)
            for count in count_dictionary.values()
            if count > 0
        ],
        dtype=float,
    )

    total = float(positive_counts.sum())

    if total <= 0:
        return {
            "scv_group_count_total": 0,
            "scv_nonzero_group_count": 0,
            "scv_group_entropy_nats": np.nan,
            "scv_group_entropy_bits": np.nan,
            "scv_group_entropy_normalized": np.nan,
            "scv_dominant_group_fraction": np.nan,
            "scv_effective_group_count": np.nan,
        }

    probabilities = positive_counts / total

    entropy_nats = float(
        -(probabilities * np.log(probabilities)).sum()
    )

    entropy_bits = float(
        -(probabilities * np.log2(probabilities)).sum()
    )

    nonzero_group_count = int(len(positive_counts))

    if nonzero_group_count <= 1:
        normalized_entropy = 0.0
    else:
        normalized_entropy = float(
            entropy_nats / math.log(nonzero_group_count)
        )

    return {
        "scv_group_count_total": int(total),
        "scv_nonzero_group_count": nonzero_group_count,
        "scv_group_entropy_nats": entropy_nats,
        "scv_group_entropy_bits": entropy_bits,
        "scv_group_entropy_normalized": normalized_entropy,
        "scv_dominant_group_fraction": float(probabilities.max()),
        "scv_effective_group_count": float(math.exp(entropy_nats)),
    }


def copy_and_verify(source: Path, destination: Path) -> None:
    """Copy an artifact to Drive and verify its checksum before renaming."""

    partial_destination = Path(str(destination) + ".partial")

    if partial_destination.exists():
        partial_destination.unlink()

    shutil.copy2(source, partial_destination)

    source_hash = sha256_file(source)
    copied_hash = sha256_file(partial_destination)

    assert source_hash == copied_hash, (
        f"Checksum mismatch while copying {destination.name}"
    )

    os.replace(partial_destination, destination)


# --------------------------------------------------------------------------------------------------
# 6. Verify required files and hashes
# --------------------------------------------------------------------------------------------------

required_paths = [
    T0_PATH,
    STAGE3H_LINKAGE_PATH,
    STAGE3H_MANIFEST_PATH,
]

missing_paths = [str(path) for path in required_paths if not path.exists()]

if missing_paths:
    raise FileNotFoundError(
        "Required input files are missing:\n"
        + "\n".join(missing_paths)
    )

verified_input_hashes = {}

for path_string, expected_hash in EXPECTED_HASHES.items():
    path = Path(path_string)
    observed_hash = sha256_file(path)

    assert observed_hash == expected_hash, (
        f"Input checksum mismatch:\n"
        f"Path:     {path}\n"
        f"Expected: {expected_hash}\n"
        f"Observed: {observed_hash}"
    )

    verified_input_hashes[str(path)] = observed_hash


# --------------------------------------------------------------------------------------------------
# 7. Read the accepted T0 cohort
# --------------------------------------------------------------------------------------------------

t0 = pd.read_parquet(T0_PATH)

assert t0.shape == (71_659, 34), (
    f"Unexpected T0 shape: {t0.shape}"
)

required_columns = [
    "rcv_accession",
    "variation_id",
    "vcv_accession",
    "timepoint",
    "release_label",
    "embedded_data_cutoff_date",
    "target_genes_json",
    "study_scope",
    "aggregate_classification_group",
    "aggregate_review_status",
    "aggregate_review_stars",
    "aggregate_last_evaluated",
    "aggregate_conflict_flag",
    "scv_group_disagreement_flag",
    "scv_count_xml",
    "unique_submitter_count_xml",
    "scv_group_counts_json",
]

missing_columns = [
    column
    for column in required_columns
    if column not in t0.columns
]

if missing_columns:
    raise KeyError(
        "Required T0 columns are missing:\n"
        + "\n".join(missing_columns)
    )

t0 = t0.copy()
t0["rcv_accession"] = normalize_rcv(t0["rcv_accession"])

assert t0["rcv_accession"].notna().all()
assert t0["rcv_accession"].is_unique
assert t0["timepoint"].astype(str).eq("T0").all()


# --------------------------------------------------------------------------------------------------
# 8. Confirm exact alignment with the frozen Stage 3H linkage
#    No T1 evidence is copied into the feature table.
# --------------------------------------------------------------------------------------------------

stage3h_rcvs = pd.read_parquet(
    STAGE3H_LINKAGE_PATH,
    columns=["t0_rcv_accession"],
)

stage3h_rcvs["t0_rcv_accession"] = normalize_rcv(
    stage3h_rcvs["t0_rcv_accession"]
)

assert len(stage3h_rcvs) == 71_659
assert stage3h_rcvs["t0_rcv_accession"].is_unique

assert set(t0["rcv_accession"]) == set(
    stage3h_rcvs["t0_rcv_accession"]
), "T0 and Stage 3H RCV sets do not match."


# --------------------------------------------------------------------------------------------------
# 9. Date and recency features
# --------------------------------------------------------------------------------------------------

cutoff_dates = pd.to_datetime(
    t0["embedded_data_cutoff_date"],
    errors="coerce",
)

last_evaluated_dates = pd.to_datetime(
    t0["aggregate_last_evaluated"],
    errors="coerce",
)

assert cutoff_dates.notna().all()
assert cutoff_dates.nunique() == 1
assert cutoff_dates.iloc[0].strftime("%Y-%m-%d") == "2022-12-31"

post_cutoff_mask = (
    last_evaluated_dates.notna()
    & (last_evaluated_dates > cutoff_dates)
)

assert int(post_cutoff_mask.sum()) == 0

recency_days = (
    cutoff_dates - last_evaluated_dates
).dt.days.astype("Float64")

recency_years = (
    recency_days / 365.25
).astype("Float64")

recency_missing = last_evaluated_dates.isna()

assert int(recency_missing.sum()) == 5_889

available_recency = recency_days.dropna()

assert (available_recency >= 0).all()


# --------------------------------------------------------------------------------------------------
# 10. Submitter-structure features
# --------------------------------------------------------------------------------------------------

scv_count = pd.to_numeric(
    t0["scv_count_xml"],
    errors="raise",
).astype("int64")

unique_submitter_count = pd.to_numeric(
    t0["unique_submitter_count_xml"],
    errors="raise",
).astype("int64")

assert (scv_count > 0).all()
assert (unique_submitter_count > 0).all()
assert (unique_submitter_count <= scv_count).all()

assert int(scv_count.sum()) == 100_633

log1p_scv_count = np.log1p(scv_count).astype(float)

log1p_unique_submitter_count = np.log1p(
    unique_submitter_count
).astype(float)

submitter_diversity_ratio = (
    unique_submitter_count / scv_count
).astype(float)

assert submitter_diversity_ratio.between(0, 1).all()


# --------------------------------------------------------------------------------------------------
# 11. Review-confidence and disagreement features
# --------------------------------------------------------------------------------------------------

review_stars = pd.to_numeric(
    t0["aggregate_review_stars"],
    errors="raise",
).astype("int64")

assert review_stars.between(0, 4).all()

aggregate_conflict = normalize_boolean(
    t0["aggregate_conflict_flag"],
    "aggregate_conflict_flag",
)

scv_group_disagreement = normalize_boolean(
    t0["scv_group_disagreement_flag"],
    "scv_group_disagreement_flag",
)

assert int(aggregate_conflict.sum()) == 1_484


# --------------------------------------------------------------------------------------------------
# 12. Classification entropy
# --------------------------------------------------------------------------------------------------

parsed_group_counts = [
    parse_count_dictionary(value, row_number=index)
    for index, value in enumerate(
        t0["scv_group_counts_json"].tolist()
    )
]

entropy_records = [
    calculate_entropy_metrics(count_dictionary)
    for count_dictionary in parsed_group_counts
]

entropy_frame = pd.DataFrame(entropy_records)

assert len(entropy_frame) == 71_659

assert (
    entropy_frame["scv_group_count_total"].astype("int64")
    .reset_index(drop=True)
    .equals(scv_count.reset_index(drop=True))
), "SCV group-count totals do not match scv_count_xml."

assert (
    entropy_frame["scv_group_entropy_normalized"]
    .dropna()
    .between(0, 1 + 1e-12)
    .all()
)


# --------------------------------------------------------------------------------------------------
# 13. Assemble the T0-only feature table
# --------------------------------------------------------------------------------------------------

feature_table = pd.DataFrame(
    {
        "t0_row_order": np.arange(len(t0), dtype=np.int64),

        # Identifiers and audit metadata
        "rcv_accession": t0["rcv_accession"],
        "variation_id": t0["variation_id"],
        "vcv_accession": t0["vcv_accession"],
        "timepoint": t0["timepoint"],
        "release_label": t0["release_label"],
        "embedded_data_cutoff_date": (
            cutoff_dates.dt.strftime("%Y-%m-%d")
        ),
        "target_genes_json": t0["target_genes_json"],
        "study_scope": t0["study_scope"],
        "aggregate_classification_group": (
            t0["aggregate_classification_group"]
        ),
        "aggregate_review_status": (
            t0["aggregate_review_status"]
        ),
        "aggregate_last_evaluated": (
            last_evaluated_dates.dt.strftime("%Y-%m-%d")
        ),

        # Recency
        "recency_days": recency_days,
        "recency_years": recency_years,
        "recency_missing_flag": recency_missing.astype(bool),

        # Submitter structure
        "scv_count": scv_count,
        "log1p_scv_count": log1p_scv_count,
        "unique_submitter_count": unique_submitter_count,
        "log1p_unique_submitter_count": (
            log1p_unique_submitter_count
        ),
        "submitter_diversity_ratio": (
            submitter_diversity_ratio
        ),

        # Review and conflict
        "aggregate_review_stars": review_stars,
        "aggregate_conflict_flag": aggregate_conflict,
        "scv_group_disagreement_flag": (
            scv_group_disagreement
        ),

        # Entropy
        "scv_group_count_total": (
            entropy_frame["scv_group_count_total"]
        ),
        "scv_nonzero_group_count": (
            entropy_frame["scv_nonzero_group_count"]
        ),
        "scv_group_entropy_nats": (
            entropy_frame["scv_group_entropy_nats"]
        ),
        "scv_group_entropy_bits": (
            entropy_frame["scv_group_entropy_bits"]
        ),
        "scv_group_entropy_normalized": (
            entropy_frame["scv_group_entropy_normalized"]
        ),
        "scv_dominant_group_fraction": (
            entropy_frame["scv_dominant_group_fraction"]
        ),
        "scv_effective_group_count": (
            entropy_frame["scv_effective_group_count"]
        ),
    }
)

assert len(feature_table) == 71_659
assert feature_table["rcv_accession"].is_unique

# No T1 or temporal-outcome fields are permitted.
forbidden_column_tokens = [
    "t1_",
    "future_instability",
    "temporal_outcome",
    "linked_t1",
    "outcome_label",
]

for column in feature_table.columns:
    normalized_column = column.lower()

    if any(
        token in normalized_column
        for token in forbidden_column_tokens
    ):
        raise AssertionError(
            f"Forbidden field found in T0 feature table: {column}"
        )


# --------------------------------------------------------------------------------------------------
# 14. Prespecified candidate feature sets
#
# These column lists are frozen for the next Stage 4B reconstruction.
# No coefficients, scaling, imputation values, labels, or thresholds are fitted here.
# --------------------------------------------------------------------------------------------------

FULL_GES_CANDIDATE_FEATURES = [
    "recency_years",
    "recency_missing_flag",
    "log1p_unique_submitter_count",
    "aggregate_review_stars",
    "aggregate_conflict_flag",
    "scv_group_entropy_normalized",
]

NO_STAR_GES_CANDIDATE_FEATURES = [
    "recency_years",
    "recency_missing_flag",
    "log1p_unique_submitter_count",
    "aggregate_conflict_flag",
    "scv_group_entropy_normalized",
]

AUDIT_ONLY_FEATURES = [
    "scv_count",
    "log1p_scv_count",
    "unique_submitter_count",
    "submitter_diversity_ratio",
    "scv_group_disagreement_flag",
    "scv_nonzero_group_count",
    "scv_group_entropy_nats",
    "scv_group_entropy_bits",
    "scv_dominant_group_fraction",
    "scv_effective_group_count",
]


# --------------------------------------------------------------------------------------------------
# 15. Feature specification
# --------------------------------------------------------------------------------------------------

created_at_utc = datetime.now(timezone.utc).isoformat()

feature_specification = {
    "specification_name": (
        "Stage 4A T0-only GES baseline feature specification"
    ),
    "specification_version": "1.0.0",
    "created_at_utc": created_at_utc,
    "status": "FROZEN_BEFORE_WEAK_LABEL_OR_MODEL_RECONSTRUCTION",
    "source_timepoint": "T0",
    "embedded_data_cutoff": "2022-12-31",
    "unit_of_analysis": "RCV-level variant-condition aggregate",

    "feature_definitions": {
        "recency_years": {
            "definition": (
                "Number of days between the frozen T0 embedded "
                "cutoff and aggregate last-evaluated date, divided "
                "by 365.25."
            ),
            "missingness_policy": (
                "Preserve missing values and include a separate "
                "recency_missing_flag. No imputation is applied "
                "during Stage 4A."
            ),
        },
        "log1p_unique_submitter_count": {
            "definition": (
                "Natural logarithm of one plus the number of unique "
                "primary submitters recorded at T0."
            ),
        },
        "aggregate_review_stars": {
            "definition": (
                "ClinVar aggregate review status mapped to its "
                "validated numeric star value at T0."
            ),
        },
        "aggregate_conflict_flag": {
            "definition": (
                "Semantically corrected T0 aggregate conflict flag. "
                "The historical 'multiple submitters, no conflicts' "
                "substring defect has already been removed."
            ),
        },
        "scv_group_entropy_normalized": {
            "definition": (
                "Normalized Shannon entropy of positive SCV broad "
                "classification-group counts available at T0."
            ),
            "normalization": (
                "Entropy in natural-log units divided by log of the "
                "number of nonzero classification groups. One-group "
                "records receive zero."
            ),
        },
    },

    "full_ges_candidate_features": (
        FULL_GES_CANDIDATE_FEATURES
    ),
    "no_star_ges_candidate_features": (
        NO_STAR_GES_CANDIDATE_FEATURES
    ),
    "audit_only_features": AUDIT_ONLY_FEATURES,

    "not_performed_in_stage4a": [
        "Weak-supervision labels were not reconstructed.",
        "Missing-value imputation parameters were not fitted.",
        "Feature standardization parameters were not fitted.",
        "Logistic-regression coefficients were not fitted.",
        "GES probabilities were not calculated.",
        "GES thresholds were not selected.",
        "T1 classifications were not inspected.",
        "Future-instability outcomes were not created.",
    ],

    "scientific_boundary": {
        "t1_information_used_as_model_feature": False,
        "future_outcome_created": False,
        "weak_labels_created": False,
        "model_fitted": False,
        "model_tuned": False,
        "threshold_selected": False,
    },
}


# --------------------------------------------------------------------------------------------------
# 16. QC report
# --------------------------------------------------------------------------------------------------

review_star_counts = {
    str(int(key)): int(value)
    for key, value in (
        feature_table["aggregate_review_stars"]
        .value_counts()
        .sort_index()
        .items()
    )
}

gene_counts = {
    str(key): int(value)
    for key, value in (
        feature_table["target_genes_json"]
        .value_counts()
        .sort_index()
        .items()
    )
}

qc_report = {
    "report_name": "Stage 4A T0 feature quality-control report",
    "report_version": "1.0.0",
    "created_at_utc": created_at_utc,
    "validation_decision": "PASS",

    "cohort": {
        "rows": int(len(feature_table)),
        "columns": int(feature_table.shape[1]),
        "unique_rcv_accessions": int(
            feature_table["rcv_accession"].nunique()
        ),
        "embedded_cutoff": "2022-12-31",
        "gene_counts_from_target_genes_json": gene_counts,
    },

    "recency": {
        "available": int(
            feature_table["recency_years"].notna().sum()
        ),
        "missing": int(
            feature_table["recency_years"].isna().sum()
        ),
        "minimum_years": float(
            feature_table["recency_years"].min()
        ),
        "maximum_years": float(
            feature_table["recency_years"].max()
        ),
        "post_cutoff_values": 0,
    },

    "submitter_structure": {
        "total_nested_scvs": int(
            feature_table["scv_count"].sum()
        ),
        "minimum_scv_count": int(
            feature_table["scv_count"].min()
        ),
        "maximum_scv_count": int(
            feature_table["scv_count"].max()
        ),
        "minimum_unique_submitters": int(
            feature_table["unique_submitter_count"].min()
        ),
        "maximum_unique_submitters": int(
            feature_table["unique_submitter_count"].max()
        ),
    },

    "review_and_conflict": {
        "review_star_counts": review_star_counts,
        "aggregate_conflict_positive": int(
            feature_table["aggregate_conflict_flag"].sum()
        ),
        "scv_group_disagreement_positive": int(
            feature_table[
                "scv_group_disagreement_flag"
            ].sum()
        ),
    },

    "entropy": {
        "group_count_total_mismatches": 0,
        "missing_normalized_entropy": int(
            feature_table[
                "scv_group_entropy_normalized"
            ].isna().sum()
        ),
        "minimum_normalized_entropy": float(
            feature_table[
                "scv_group_entropy_normalized"
            ].min()
        ),
        "maximum_normalized_entropy": float(
            feature_table[
                "scv_group_entropy_normalized"
            ].max()
        ),
    },

    "leakage_checks": {
        "stage3h_t0_rcv_alignment_passed": True,
        "t1_columns_in_feature_table": 0,
        "future_outcome_columns_in_feature_table": 0,
        "t1_values_used_for_feature_construction": False,
        "weak_labels_created": False,
        "model_fitted": False,
    },
}


# --------------------------------------------------------------------------------------------------
# 17. Build files locally
# --------------------------------------------------------------------------------------------------

BUILD_DIR = Path("/content/stage4a_t0_feature_build_v1")

if BUILD_DIR.exists():
    shutil.rmtree(BUILD_DIR)

BUILD_DIR.mkdir(parents=True, exist_ok=False)

temporary_feature_path = BUILD_DIR / FEATURE_TABLE_PATH.name
temporary_spec_path = BUILD_DIR / FEATURE_SPEC_PATH.name
temporary_qc_path = BUILD_DIR / QC_REPORT_PATH.name
temporary_manifest_path = BUILD_DIR / FREEZE_MANIFEST_PATH.name

feature_table.to_parquet(
    temporary_feature_path,
    index=False,
    engine="pyarrow",
    compression="zstd",
)

write_json(
    temporary_spec_path,
    feature_specification,
)

write_json(
    temporary_qc_path,
    qc_report,
)


# --------------------------------------------------------------------------------------------------
# 18. Readback validation
# --------------------------------------------------------------------------------------------------

feature_readback = pd.read_parquet(temporary_feature_path)

assert feature_readback.shape == feature_table.shape
assert len(feature_readback) == 71_659
assert feature_readback["rcv_accession"].is_unique
assert set(feature_readback.columns) == set(feature_table.columns)

assert int(
    feature_readback["aggregate_conflict_flag"].sum()
) == 1_484

assert int(
    feature_readback["recency_missing_flag"].sum()
) == 5_889

assert int(
    feature_readback["scv_count"].sum()
) == 100_633


# --------------------------------------------------------------------------------------------------
# 19. Calculate output hashes
# --------------------------------------------------------------------------------------------------

feature_table_hash = sha256_file(temporary_feature_path)
feature_spec_hash = sha256_file(temporary_spec_path)
qc_report_hash = sha256_file(temporary_qc_path)


# --------------------------------------------------------------------------------------------------
# 20. Freeze manifest
# --------------------------------------------------------------------------------------------------

freeze_manifest = {
    "manifest_name": "Stage 4A T0 feature freeze manifest",
    "manifest_version": "1.0.0",
    "created_at_utc": created_at_utc,
    "manifest_status": "STAGE4A_T0_BASELINE_FEATURES_FROZEN",
    "validation_decision": (
        "PASS_STAGE4A_T0_FEATURES_ACCEPTED_AND_FROZEN"
    ),

    "source_artifacts": {
        str(path): {
            "sha256": observed_hash,
        }
        for path, observed_hash in sorted(
            verified_input_hashes.items()
        )
    },

    "frozen_outputs": {
        "feature_table": {
            "path": str(FEATURE_TABLE_PATH),
            "sha256": feature_table_hash,
            "rows": int(feature_table.shape[0]),
            "columns": int(feature_table.shape[1]),
        },
        "feature_specification": {
            "path": str(FEATURE_SPEC_PATH),
            "sha256": feature_spec_hash,
        },
        "quality_control_report": {
            "path": str(QC_REPORT_PATH),
            "sha256": qc_report_hash,
        },
    },

    "feature_sets": {
        "full_ges_candidate_features": (
            FULL_GES_CANDIDATE_FEATURES
        ),
        "no_star_ges_candidate_features": (
            NO_STAR_GES_CANDIDATE_FEATURES
        ),
    },

    "validation_summary": {
        "total_t0_records": 71_659,
        "unique_t0_rcvs": 71_659,
        "recency_missing_records": 5_889,
        "aggregate_conflict_positive_records": 1_484,
        "nested_scv_total": 100_633,
        "t1_columns_in_feature_table": 0,
        "future_outcome_created": False,
        "weak_labels_created": False,
        "model_fitted": False,
    },

    "next_authorized_work": (
        "Stage 4B: reconstruct and freeze the original T0-only "
        "weak-supervision labeling functions and verify their exact "
        "relationship to the prior GES implementation before fitting "
        "the full and no-star logistic-regression models."
    ),
}

write_json(
    temporary_manifest_path,
    freeze_manifest,
)

freeze_manifest_hash = sha256_file(
    temporary_manifest_path
)


# --------------------------------------------------------------------------------------------------
# 21. Copy outputs to Drive and verify
# --------------------------------------------------------------------------------------------------

copy_and_verify(
    temporary_feature_path,
    FEATURE_TABLE_PATH,
)

copy_and_verify(
    temporary_spec_path,
    FEATURE_SPEC_PATH,
)

copy_and_verify(
    temporary_qc_path,
    QC_REPORT_PATH,
)

# Freeze manifest copied last
copy_and_verify(
    temporary_manifest_path,
    FREEZE_MANIFEST_PATH,
)

assert sha256_file(FEATURE_TABLE_PATH) == feature_table_hash
assert sha256_file(FEATURE_SPEC_PATH) == feature_spec_hash
assert sha256_file(QC_REPORT_PATH) == qc_report_hash
assert sha256_file(FREEZE_MANIFEST_PATH) == freeze_manifest_hash


# --------------------------------------------------------------------------------------------------
# 22. Persistent readback
# --------------------------------------------------------------------------------------------------

persistent_features = pd.read_parquet(FEATURE_TABLE_PATH)

assert persistent_features.shape == feature_table.shape
assert persistent_features["rcv_accession"].is_unique
assert int(
    persistent_features["aggregate_conflict_flag"].sum()
) == 1_484


# --------------------------------------------------------------------------------------------------
# 23. Final output
# --------------------------------------------------------------------------------------------------

print("=" * 120)
print("STAGE 4A — T0-ONLY BASELINE FEATURE CONSTRUCTION AND FREEZE")
print("=" * 120)

print(f"Created at UTC:                    {created_at_utc}")
print("Validation decision:              PASS_STAGE4A_T0_FEATURES_ACCEPTED_AND_FROZEN")
print()

print("T0 FEATURE COHORT")
print(f"  Rows:                           {len(persistent_features):,}")
print(f"  Columns:                        {persistent_features.shape[1]:,}")
print(f"  Unique RCVs:                    {persistent_features['rcv_accession'].nunique():,}")
print(f"  Embedded cutoff:                2022-12-31")
print()

print("RECENCY")
print(f"  Available dates:                {persistent_features['recency_years'].notna().sum():,}")
print(f"  Missing dates:                  {persistent_features['recency_years'].isna().sum():,}")
print(f"  Post-cutoff dates:              0")
print()

print("SUBMITTER STRUCTURE")
print(f"  Total nested SCVs:              {persistent_features['scv_count'].sum():,}")
print(f"  Minimum unique submitters:      {persistent_features['unique_submitter_count'].min():,}")
print(f"  Maximum unique submitters:      {persistent_features['unique_submitter_count'].max():,}")
print()

print("REVIEW AND CONFLICT")
print(
    "  Review-star distribution:      "
    f"{persistent_features['aggregate_review_stars'].value_counts().sort_index().to_dict()}"
)
print(f"  Aggregate conflict-positive:    {persistent_features['aggregate_conflict_flag'].sum():,}")
print(f"  SCV disagreement-positive:      {persistent_features['scv_group_disagreement_flag'].sum():,}")
print()

print("ENTROPY")
print(
    "  Normalized entropy range:       "
    f"{persistent_features['scv_group_entropy_normalized'].min():.6f} "
    f"to {persistent_features['scv_group_entropy_normalized'].max():.6f}"
)
print(
    "  Entropy missing:                "
    f"{persistent_features['scv_group_entropy_normalized'].isna().sum():,}"
)
print()

print("SCIENTIFIC BOUNDARY")
print("  T1 evidence used as a feature:  False")
print("  Future outcome created:         False")
print("  Weak labels created:            False")
print("  GES model fitted:               False")
print("  Threshold selected:             False")
print()

print("FROZEN OUTPUTS")
print(f"  Feature table:                  {FEATURE_TABLE_PATH}")
print(f"    SHA-256:                      {feature_table_hash}")
print(f"  Feature specification:          {FEATURE_SPEC_PATH}")
print(f"    SHA-256:                      {feature_spec_hash}")
print(f"  QC report:                      {QC_REPORT_PATH}")
print(f"    SHA-256:                      {qc_report_hash}")
print(f"  Freeze manifest:                {FREEZE_MANIFEST_PATH}")
print(f"    SHA-256:                      {freeze_manifest_hash}")
print()

print("NEXT AUTHORIZED WORK")
print("  Stage 4B — reconstruct and freeze the original T0-only weak-label")
print("  rules before fitting the full and no-star GES models.")
print("=" * 120)

STAGE 4A — T0-ONLY BASELINE FEATURE CONSTRUCTION AND FREEZE
Created at UTC:                    2026-07-20T21:39:47.594893+00:00
Validation decision:              PASS_STAGE4A_T0_FEATURES_ACCEPTED_AND_FROZEN

T0 FEATURE COHORT
  Rows:                           71,659
  Columns:                        30
  Unique RCVs:                    71,659
  Embedded cutoff:                2022-12-31

RECENCY
  Available dates:                65,770
  Missing dates:                  5,889
  Post-cutoff dates:              0

SUBMITTER STRUCTURE
  Total nested SCVs:              100,633
  Minimum unique submitters:      1
  Maximum unique submitters:      30

REVIEW AND CONFLICT
  Review-star distribution:      {0: 3878, 1: 50399, 2: 9224, 3: 8158}
  Aggregate conflict-positive:    1,484
  SCV disagreement-positive:      2,465

ENTROPY
  Normalized entropy range:       0.000000 to 1.000000
  Entropy missing:                0

SCIENTIFIC BOUNDARY
  T1 evidence used as a feature:  False
  Future outcom

In [27]:
# ==================================================================================================
# STAGE 4B — MANUSCRIPT-SPECIFIED FEATURE TRANSFORMATIONS,
#            FULL WEAK LABELS, AND NO-STAR WEAK LABELS
#
# Reconstructs:
#   1. Original recency score
#   2. Original normalized submitter-diversity score
#   3. LF-Recency
#   4. LF-ReviewConf
#   5. LF-Conflict with conflict veto
#   6. Majority-vote full weak label
#   7. No-star weak-label ablation
#
# Scientific boundary:
#   - Uses only the frozen Stage 4A T0 feature artifact.
#   - Does not use T1 evidence.
#   - Does not construct future-instability outcomes.
#   - Does not fit logistic regression.
#   - Does not select model coefficients or performance thresholds.
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import os
import platform
import shutil
import sys

import numpy as np
import pandas as pd


# --------------------------------------------------------------------------------------------------
# 1. Google Drive
# --------------------------------------------------------------------------------------------------

DRIVE_ROOT = Path("/content/drive/MyDrive")

if not DRIVE_ROOT.exists():
    from google.colab import drive
    drive.mount("/content/drive")

assert DRIVE_ROOT.exists(), "Google Drive is not mounted."


# --------------------------------------------------------------------------------------------------
# 2. Paths
# --------------------------------------------------------------------------------------------------

BASE_DIR = DRIVE_ROOT / "GES_RAG_Temporal_Study"

STAGE4_DATA_DIR = BASE_DIR / "data_processed" / "stage4_ges"
STAGE4_CONFIG_DIR = BASE_DIR / "configs" / "stage4_ges"

STAGE4_DATA_DIR.mkdir(parents=True, exist_ok=True)
STAGE4_CONFIG_DIR.mkdir(parents=True, exist_ok=True)


# Stage 4A frozen inputs
FEATURE_INPUT_PATH = (
    STAGE4_DATA_DIR
    / "stage4a_t0_ges_baseline_features_v1.parquet"
)

FEATURE_SPEC_INPUT_PATH = (
    STAGE4_CONFIG_DIR
    / "stage4a_t0_feature_specification_v1.json"
)

FEATURE_QC_INPUT_PATH = (
    STAGE4_CONFIG_DIR
    / "stage4a_t0_feature_qc_report_v1.json"
)

STAGE4A_MANIFEST_PATH = (
    STAGE4_CONFIG_DIR
    / "stage4a_t0_feature_freeze_manifest_v1.json"
)


# Stage 4B outputs
WEAK_LABEL_TABLE_PATH = (
    STAGE4_DATA_DIR
    / "stage4b_t0_feature_transforms_and_weak_labels_v1.parquet"
)

TRANSFORM_PARAMETERS_PATH = (
    STAGE4_CONFIG_DIR
    / "stage4b_t0_feature_transform_parameters_v1.json"
)

WEAK_LABEL_RULES_PATH = (
    STAGE4_CONFIG_DIR
    / "stage4b_weak_label_rules_v1.json"
)

WEAK_LABEL_QC_PATH = (
    STAGE4_CONFIG_DIR
    / "stage4b_weak_label_qc_report_v1.json"
)

STAGE4B_MANIFEST_PATH = (
    STAGE4_CONFIG_DIR
    / "stage4b_weak_label_freeze_manifest_v1.json"
)

OUTPUT_PATHS = [
    WEAK_LABEL_TABLE_PATH,
    TRANSFORM_PARAMETERS_PATH,
    WEAK_LABEL_RULES_PATH,
    WEAK_LABEL_QC_PATH,
    STAGE4B_MANIFEST_PATH,
]

existing_outputs = [
    str(path)
    for path in OUTPUT_PATHS
    if path.exists()
]

if existing_outputs:
    raise FileExistsError(
        "Stage 4B outputs already exist and were not overwritten:\n"
        + "\n".join(existing_outputs)
    )


# --------------------------------------------------------------------------------------------------
# 3. Expected Stage 4A hashes
# --------------------------------------------------------------------------------------------------

EXPECTED_INPUT_HASHES = {
    str(FEATURE_INPUT_PATH): (
        "c100b3781e6801425f622f5d091376abfe0939e48c0a792af32f6eebe6401f16"
    ),
    str(FEATURE_SPEC_INPUT_PATH): (
        "fc00146efe5da9b3fbe740bb42ca99d157252cdefc045650f9e88d54d8fcfa8b"
    ),
    str(FEATURE_QC_INPUT_PATH): (
        "4bf72af66aa800fa9f12fdc8598bd06ebc859bb31c97ad97cf7fc11ef1614cf1"
    ),
    str(STAGE4A_MANIFEST_PATH): (
        "2b844ef2dbc0c3e5e57493886a7533587350e1b4b4c76fc9d445ecda8a528fa0"
    ),
}


# --------------------------------------------------------------------------------------------------
# 4. Constants
# --------------------------------------------------------------------------------------------------

STABLE = 1
ABSTAIN = 0
UNSTABLE = -1

RECENCY_POSITIVE_THRESHOLD = 0.70
RECENCY_NEGATIVE_THRESHOLD = 0.30

RANDOM_SEED = 42


# --------------------------------------------------------------------------------------------------
# 5. Utility functions
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path: Path,
    chunk_size: int = 8 * 1024 * 1024,
) -> str:
    """Calculate SHA-256 without loading the complete file into memory."""

    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def write_json(path: Path, payload: dict) -> None:
    """Write deterministic, human-readable JSON."""

    text = json.dumps(
        payload,
        indent=2,
        sort_keys=True,
        ensure_ascii=False,
    )

    path.write_text(text + "\n", encoding="utf-8")


def copy_and_verify(source: Path, destination: Path) -> None:
    """Copy a file to Drive and verify its checksum before renaming."""

    partial_destination = Path(str(destination) + ".partial")

    if partial_destination.exists():
        partial_destination.unlink()

    shutil.copy2(source, partial_destination)

    source_hash = sha256_file(source)
    copied_hash = sha256_file(partial_destination)

    assert source_hash == copied_hash, (
        f"Checksum mismatch while copying {destination.name}"
    )

    os.replace(partial_destination, destination)


def normalize_rcv(series: pd.Series) -> pd.Series:
    """Normalize RCV accession formatting."""

    return (
        series.astype("string")
        .str.strip()
        .str.upper()
    )


def aggregate_votes(
    vote_matrix: np.ndarray,
    conflict_veto: np.ndarray,
) -> dict:
    """
    Aggregate labeling-function outputs.

    Label coding:
      +1 = stable
       0 = abstain
      -1 = unstable

    The conflict flag is a veto. If it is active, the final label is unstable
    and the weak stability probability is zero regardless of other votes.
    """

    if vote_matrix.ndim != 2:
        raise ValueError("vote_matrix must be two-dimensional.")

    row_count = vote_matrix.shape[0]

    if len(conflict_veto) != row_count:
        raise ValueError("Conflict-veto length does not match vote matrix.")

    positive_votes = (vote_matrix == STABLE).sum(axis=1)
    negative_votes = (vote_matrix == UNSTABLE).sum(axis=1)
    abstain_votes = (vote_matrix == ABSTAIN).sum(axis=1)
    fired_votes = positive_votes + negative_votes

    probability = np.full(
        row_count,
        np.nan,
        dtype=float,
    )

    fired_mask = fired_votes > 0

    probability[fired_mask] = (
        positive_votes[fired_mask]
        / fired_votes[fired_mask]
    )

    signed_label = np.full(
        row_count,
        np.nan,
        dtype=float,
    )

    signed_label[positive_votes > negative_votes] = STABLE
    signed_label[negative_votes > positive_votes] = UNSTABLE

    # Conflict veto
    signed_label[conflict_veto] = UNSTABLE
    probability[conflict_veto] = 0.0

    binary_label = np.full(
        row_count,
        np.nan,
        dtype=float,
    )

    binary_label[signed_label == STABLE] = 1.0
    binary_label[signed_label == UNSTABLE] = 0.0

    disagreement_flag = (
        (positive_votes > 0)
        & (negative_votes > 0)
    )

    tie_flag = (
        fired_mask
        & (positive_votes == negative_votes)
        & (~conflict_veto)
    )

    all_abstain_flag = fired_votes == 0

    unanimous_among_fired = (
        fired_mask
        & (
            (positive_votes == fired_votes)
            | (negative_votes == fired_votes)
        )
    )

    return {
        "positive_votes": positive_votes.astype(np.int8),
        "negative_votes": negative_votes.astype(np.int8),
        "abstain_votes": abstain_votes.astype(np.int8),
        "fired_votes": fired_votes.astype(np.int8),
        "weak_stability_probability": probability,
        "weak_label_signed": signed_label,
        "weak_label_binary": binary_label,
        "training_eligible": ~np.isnan(binary_label),
        "disagreement_flag": disagreement_flag,
        "tie_flag": tie_flag,
        "all_abstain_flag": all_abstain_flag,
        "unanimous_among_fired": unanimous_among_fired,
    }


def nullable_int_series(values: np.ndarray) -> pd.Series:
    """Convert numeric values containing NaN to a nullable Int8 Series."""

    return pd.Series(values).round().astype("Int8")


# --------------------------------------------------------------------------------------------------
# 6. Verify all Stage 4A inputs
# --------------------------------------------------------------------------------------------------

required_paths = [
    FEATURE_INPUT_PATH,
    FEATURE_SPEC_INPUT_PATH,
    FEATURE_QC_INPUT_PATH,
    STAGE4A_MANIFEST_PATH,
]

missing_paths = [
    str(path)
    for path in required_paths
    if not path.exists()
]

if missing_paths:
    raise FileNotFoundError(
        "Required Stage 4A files are missing:\n"
        + "\n".join(missing_paths)
    )

verified_input_hashes = {}

for path_string, expected_hash in EXPECTED_INPUT_HASHES.items():
    path = Path(path_string)
    observed_hash = sha256_file(path)

    assert observed_hash == expected_hash, (
        f"Stage 4A checksum mismatch:\n"
        f"Path:     {path}\n"
        f"Expected: {expected_hash}\n"
        f"Observed: {observed_hash}"
    )

    verified_input_hashes[str(path)] = observed_hash


# --------------------------------------------------------------------------------------------------
# 7. Read Stage 4A feature table
# --------------------------------------------------------------------------------------------------

features = pd.read_parquet(FEATURE_INPUT_PATH)

assert features.shape == (71_659, 30), (
    f"Unexpected Stage 4A shape: {features.shape}"
)

required_columns = [
    "t0_row_order",
    "rcv_accession",
    "variation_id",
    "vcv_accession",
    "target_genes_json",
    "study_scope",
    "recency_days",
    "recency_missing_flag",
    "unique_submitter_count",
    "log1p_unique_submitter_count",
    "aggregate_review_stars",
    "aggregate_conflict_flag",
    "scv_group_entropy_normalized",
]

missing_columns = [
    column
    for column in required_columns
    if column not in features.columns
]

if missing_columns:
    raise KeyError(
        "Required Stage 4A columns are missing:\n"
        + "\n".join(missing_columns)
    )

features = features.copy()

features["rcv_accession"] = normalize_rcv(
    features["rcv_accession"]
)

assert len(features) == 71_659
assert features["rcv_accession"].is_unique
assert features["rcv_accession"].notna().all()


# --------------------------------------------------------------------------------------------------
# 8. Reconstruct manuscript-compatible feature transformations
# --------------------------------------------------------------------------------------------------

recency_days = pd.to_numeric(
    features["recency_days"],
    errors="coerce",
).to_numpy(dtype=float)

recency_missing = np.isnan(recency_days)

assert int(recency_missing.sum()) == 5_889

available_recency_days = recency_days[~recency_missing]

assert len(available_recency_days) == 65_770
assert np.all(available_recency_days >= 0)

maximum_observation_window_days = float(
    np.max(available_recency_days)
)

minimum_observed_recency_days = float(
    np.min(available_recency_days)
)

assert maximum_observation_window_days > 0


# Manuscript-compatible formula:
# recency_score = 1 - days_since_evaluation / maximum_observation_window
recency_score = np.full(
    len(features),
    np.nan,
    dtype=float,
)

recency_score[~recency_missing] = (
    1.0
    - (
        recency_days[~recency_missing]
        / maximum_observation_window_days
    )
)

recency_score = np.clip(
    recency_score,
    0.0,
    1.0,
)

assert np.nanmin(recency_score) >= 0.0
assert np.nanmax(recency_score) <= 1.0


# Submitter diversity:
# log-transform followed by min-max normalization
unique_submitter_count = pd.to_numeric(
    features["unique_submitter_count"],
    errors="raise",
).to_numpy(dtype=float)

assert np.all(unique_submitter_count >= 1)

log_submitter_count = np.log1p(
    unique_submitter_count
)

saved_log_submitter_count = pd.to_numeric(
    features["log1p_unique_submitter_count"],
    errors="raise",
).to_numpy(dtype=float)

assert np.allclose(
    log_submitter_count,
    saved_log_submitter_count,
    atol=1e-12,
    rtol=1e-12,
)

minimum_log_submitters = float(
    np.min(log_submitter_count)
)

maximum_log_submitters = float(
    np.max(log_submitter_count)
)

log_submitter_range = (
    maximum_log_submitters
    - minimum_log_submitters
)

assert log_submitter_range > 0

submitter_diversity_score = (
    log_submitter_count
    - minimum_log_submitters
) / log_submitter_range

submitter_diversity_score = np.clip(
    submitter_diversity_score,
    0.0,
    1.0,
)

assert np.min(submitter_diversity_score) == 0.0
assert np.max(submitter_diversity_score) == 1.0


# Review confidence
review_confidence = pd.to_numeric(
    features["aggregate_review_stars"],
    errors="raise",
).to_numpy(dtype=int)

observed_review_levels = set(
    np.unique(review_confidence).tolist()
)

assert observed_review_levels == {0, 1, 2, 3}, (
    f"Unexpected review-star values: {observed_review_levels}"
)


# Conflict
aggregate_conflict = (
    features["aggregate_conflict_flag"]
    .astype(bool)
    .to_numpy()
)

assert int(aggregate_conflict.sum()) == 1_484


# --------------------------------------------------------------------------------------------------
# 9. Full-model labeling functions
# --------------------------------------------------------------------------------------------------

# LF-Recency
lf_recency = np.full(
    len(features),
    ABSTAIN,
    dtype=np.int8,
)

lf_recency[
    (~recency_missing)
    & (recency_score > RECENCY_POSITIVE_THRESHOLD)
] = STABLE

lf_recency[
    (~recency_missing)
    & (recency_score < RECENCY_NEGATIVE_THRESHOLD)
] = UNSTABLE


# LF-ReviewConf
lf_review_confidence = np.full(
    len(features),
    ABSTAIN,
    dtype=np.int8,
)

lf_review_confidence[
    np.isin(review_confidence, [2, 3])
] = STABLE

lf_review_confidence[
    review_confidence == 0
] = UNSTABLE

# Review level 1 explicitly abstains.


# LF-Conflict
lf_conflict = np.full(
    len(features),
    ABSTAIN,
    dtype=np.int8,
)

lf_conflict[
    aggregate_conflict
] = UNSTABLE


# --------------------------------------------------------------------------------------------------
# 10. Aggregate full weak labels
# --------------------------------------------------------------------------------------------------

full_vote_matrix = np.column_stack(
    [
        lf_recency,
        lf_review_confidence,
        lf_conflict,
    ]
)

full_results = aggregate_votes(
    vote_matrix=full_vote_matrix,
    conflict_veto=aggregate_conflict,
)


# --------------------------------------------------------------------------------------------------
# 11. No-star weak-label ablation
#
# Review status is removed from:
#   - predictor construction;
#   - labeling-function aggregation.
#
# This prevents review status from entering indirectly through the target label.
# --------------------------------------------------------------------------------------------------

no_star_vote_matrix = np.column_stack(
    [
        lf_recency,
        lf_conflict,
    ]
)

no_star_results = aggregate_votes(
    vote_matrix=no_star_vote_matrix,
    conflict_veto=aggregate_conflict,
)


# --------------------------------------------------------------------------------------------------
# 12. Assemble Stage 4B table
# --------------------------------------------------------------------------------------------------

weak_label_table = pd.DataFrame(
    {
        # Identifiers
        "t0_row_order": features["t0_row_order"],
        "rcv_accession": features["rcv_accession"],
        "variation_id": features["variation_id"],
        "vcv_accession": features["vcv_accession"],
        "target_genes_json": features["target_genes_json"],
        "study_scope": features["study_scope"],

        # Reconstructed model features
        "recency_score": recency_score,
        "recency_missing_flag": recency_missing,
        "submitter_diversity_score": submitter_diversity_score,
        "review_confidence": review_confidence.astype(np.int8),
        "aggregate_conflict_flag": aggregate_conflict,
        "scv_group_entropy_normalized": (
            features["scv_group_entropy_normalized"]
        ),

        # Full-model labeling functions
        "lf_recency": lf_recency,
        "lf_review_confidence": lf_review_confidence,
        "lf_conflict": lf_conflict,

        # Full weak-label aggregation
        "full_positive_votes": full_results["positive_votes"],
        "full_negative_votes": full_results["negative_votes"],
        "full_abstain_votes": full_results["abstain_votes"],
        "full_fired_votes": full_results["fired_votes"],
        "full_weak_stability_probability": (
            full_results["weak_stability_probability"]
        ),
        "full_weak_label_signed": nullable_int_series(
            full_results["weak_label_signed"]
        ),
        "full_weak_label_binary": nullable_int_series(
            full_results["weak_label_binary"]
        ),
        "full_training_eligible": (
            full_results["training_eligible"]
        ),
        "full_disagreement_flag": (
            full_results["disagreement_flag"]
        ),
        "full_tie_flag": full_results["tie_flag"],
        "full_all_abstain_flag": (
            full_results["all_abstain_flag"]
        ),
        "full_unanimous_among_fired": (
            full_results["unanimous_among_fired"]
        ),

        # No-star weak-label aggregation
        "no_star_positive_votes": (
            no_star_results["positive_votes"]
        ),
        "no_star_negative_votes": (
            no_star_results["negative_votes"]
        ),
        "no_star_abstain_votes": (
            no_star_results["abstain_votes"]
        ),
        "no_star_fired_votes": (
            no_star_results["fired_votes"]
        ),
        "no_star_weak_stability_probability": (
            no_star_results["weak_stability_probability"]
        ),
        "no_star_weak_label_signed": nullable_int_series(
            no_star_results["weak_label_signed"]
        ),
        "no_star_weak_label_binary": nullable_int_series(
            no_star_results["weak_label_binary"]
        ),
        "no_star_training_eligible": (
            no_star_results["training_eligible"]
        ),
        "no_star_disagreement_flag": (
            no_star_results["disagreement_flag"]
        ),
        "no_star_tie_flag": no_star_results["tie_flag"],
        "no_star_all_abstain_flag": (
            no_star_results["all_abstain_flag"]
        ),
        "no_star_unanimous_among_fired": (
            no_star_results["unanimous_among_fired"]
        ),

        # Scientific boundary
        "t1_information_used": False,
        "future_instability_outcome_created": False,
        "ges_model_fitted": False,
        "stage4b_version": "1.0.0",
    }
)

assert len(weak_label_table) == 71_659
assert weak_label_table["rcv_accession"].is_unique


# --------------------------------------------------------------------------------------------------
# 13. Scientific and logical validation
# --------------------------------------------------------------------------------------------------

# Conflict veto must always produce an unstable label.
conflict_mask = weak_label_table[
    "aggregate_conflict_flag"
].astype(bool)

conflict_rows = weak_label_table.loc[
    conflict_mask
].copy()

assert (
    conflict_rows["full_weak_label_binary"]
    .astype("Int8")
    .eq(0)
    .all()
)

assert (
    conflict_rows["full_weak_stability_probability"]
    .eq(0.0)
    .all()
)

assert (
    conflict_rows["no_star_weak_label_binary"]
    .astype("Int8")
    .eq(0)
    .all()
)

assert (
    conflict_rows["no_star_weak_stability_probability"]
    .eq(0.0)
    .all()
)


# LF-ReviewConf exact rule checks
assert np.all(
    lf_review_confidence[
        review_confidence == 0
    ] == UNSTABLE
)

assert np.all(
    lf_review_confidence[
        review_confidence == 1
    ] == ABSTAIN
)

assert np.all(
    lf_review_confidence[
        np.isin(review_confidence, [2, 3])
    ] == STABLE
)


# Missing recency must abstain
assert np.all(
    lf_recency[recency_missing] == ABSTAIN
)


# No forbidden T1 or temporal-outcome fields may contain scientific values.
assert weak_label_table["t1_information_used"].eq(False).all()

assert (
    weak_label_table[
        "future_instability_outcome_created"
    ]
    .eq(False)
    .all()
)

assert weak_label_table["ges_model_fitted"].eq(False).all()


# Labels must be limited to expected values.
for column in [
    "lf_recency",
    "lf_review_confidence",
    "lf_conflict",
]:
    assert set(
        weak_label_table[column]
        .dropna()
        .astype(int)
        .unique()
        .tolist()
    ).issubset({UNSTABLE, ABSTAIN, STABLE})


# --------------------------------------------------------------------------------------------------
# 14. Freeze transformation parameters
# --------------------------------------------------------------------------------------------------

created_at_utc = datetime.now(timezone.utc).isoformat()

transform_parameters = {
    "artifact_name": (
        "Stage 4B T0 feature transformation parameters"
    ),
    "version": "1.0.0",
    "created_at_utc": created_at_utc,
    "status": "FROZEN_BEFORE_MODEL_FITTING",
    "source": (
        "Manuscript-specified reconstruction applied to the "
        "accepted Stage 4A RCV-level T0 cohort."
    ),

    "recency_transformation": {
        "input": "recency_days",
        "formula": (
            "1 - recency_days / maximum_observation_window_days"
        ),
        "maximum_observation_window_days": (
            maximum_observation_window_days
        ),
        "minimum_observed_recency_days": (
            minimum_observed_recency_days
        ),
        "clipping_interval": [0.0, 1.0],
        "missingness_policy": (
            "Missing aggregate evaluation dates remain missing; "
            "LF-Recency abstains."
        ),
    },

    "submitter_diversity_transformation": {
        "input": "unique_submitter_count",
        "log_formula": "log1p(unique_submitter_count)",
        "normalization_formula": (
            "(log_count - minimum_log_count) / "
            "(maximum_log_count - minimum_log_count)"
        ),
        "minimum_log_count": minimum_log_submitters,
        "maximum_log_count": maximum_log_submitters,
        "clipping_interval": [0.0, 1.0],
    },

    "review_confidence": {
        "input": "aggregate_review_stars",
        "observed_levels": sorted(
            observed_review_levels
        ),
        "coding": {
            "0": "No assertion criteria or equivalent zero-star status",
            "1": "Single-submitter/one-star level",
            "2": "Multiple submitters without conflict",
            "3": "Expert panel",
        },
    },

    "conflict": {
        "input": "aggregate_conflict_flag",
        "positive_records": int(
            aggregate_conflict.sum()
        ),
        "semantic_status": (
            "Uses the corrected exact-semantic T0 conflict field."
        ),
    },

    "random_seed_reserved_for_model_fitting": RANDOM_SEED,

    "software": {
        "python": sys.version,
        "platform": platform.platform(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
    },

    "scientific_boundary": {
        "t1_information_used": False,
        "future_outcome_created": False,
        "weak_labels_created": True,
        "model_fitted": False,
        "coefficients_selected": False,
        "risk_thresholds_applied": False,
    },
}


# --------------------------------------------------------------------------------------------------
# 15. Freeze weak-label rules
# --------------------------------------------------------------------------------------------------

weak_label_rules = {
    "artifact_name": "Stage 4B weak-label rules",
    "version": "1.0.0",
    "created_at_utc": created_at_utc,
    "status": "FROZEN_BEFORE_MODEL_FITTING",

    "label_encoding": {
        "stable": STABLE,
        "abstain": ABSTAIN,
        "unstable": UNSTABLE,
    },

    "full_model_labeling_functions": {
        "LF_Recency": {
            "stable_rule": (
                "recency_score > 0.70"
            ),
            "unstable_rule": (
                "recency_score < 0.30"
            ),
            "abstain_rule": (
                "0.30 <= recency_score <= 0.70 "
                "or recency date missing"
            ),
        },

        "LF_ReviewConf": {
            "stable_rule": (
                "review_confidence in {2, 3}"
            ),
            "unstable_rule": (
                "review_confidence == 0"
            ),
            "abstain_rule": (
                "review_confidence == 1"
            ),
        },

        "LF_Conflict": {
            "unstable_rule": (
                "aggregate_conflict_flag is True"
            ),
            "abstain_rule": (
                "aggregate_conflict_flag is False"
            ),
            "veto": True,
        },
    },

    "full_aggregation": {
        "method": (
            "Majority vote across non-abstaining labeling functions"
        ),
        "probability": (
            "Number of positive votes divided by number of "
            "non-abstaining votes"
        ),
        "tie_policy": (
            "Remain unlabeled unless the conflict veto is active"
        ),
        "all_abstain_policy": (
            "Remain unlabeled and excluded from model training"
        ),
        "conflict_veto": (
            "Any active LF-Conflict assigns unstable label and "
            "weak stability probability 0.0"
        ),
    },

    "no_star_ablation": {
        "labeling_functions": [
            "LF_Recency",
            "LF_Conflict",
        ],
        "review_status_used_in_label": False,
        "review_status_used_as_predictor": False,
        "purpose": (
            "Assess signal from recency, submitter structure, "
            "conflict, and optional entropy without direct or "
            "indirect review-star information."
        ),
    },

    "not_performed": [
        "No logistic-regression model was fitted.",
        "No T1 outcome was created.",
        "No weak-label threshold was optimized against T1.",
        "No model coefficient was selected.",
        "No P(stable) model score was generated.",
    ],
}


# --------------------------------------------------------------------------------------------------
# 16. QC summaries
# --------------------------------------------------------------------------------------------------

def nullable_label_counts(
    series: pd.Series,
) -> dict:
    """Return stable, unstable, and missing counts."""

    return {
        "stable_1": int(series.eq(1).sum()),
        "unstable_0": int(series.eq(0).sum()),
        "unlabeled_missing": int(series.isna().sum()),
    }


lf_recency_counts = {
    "stable": int((lf_recency == STABLE).sum()),
    "unstable": int((lf_recency == UNSTABLE).sum()),
    "abstain": int((lf_recency == ABSTAIN).sum()),
}

lf_review_counts = {
    "stable": int(
        (lf_review_confidence == STABLE).sum()
    ),
    "unstable": int(
        (lf_review_confidence == UNSTABLE).sum()
    ),
    "abstain": int(
        (lf_review_confidence == ABSTAIN).sum()
    ),
}

lf_conflict_counts = {
    "stable": int(
        (lf_conflict == STABLE).sum()
    ),
    "unstable": int(
        (lf_conflict == UNSTABLE).sum()
    ),
    "abstain": int(
        (lf_conflict == ABSTAIN).sum()
    ),
}

qc_report = {
    "report_name": "Stage 4B weak-label QC report",
    "version": "1.0.0",
    "created_at_utc": created_at_utc,
    "validation_decision": "PASS",

    "cohort": {
        "rows": int(len(weak_label_table)),
        "columns": int(
            weak_label_table.shape[1]
        ),
        "unique_rcvs": int(
            weak_label_table[
                "rcv_accession"
            ].nunique()
        ),
    },

    "transformed_features": {
        "recency_score_available": int(
            np.sum(~np.isnan(recency_score))
        ),
        "recency_score_missing": int(
            np.sum(np.isnan(recency_score))
        ),
        "recency_score_minimum": float(
            np.nanmin(recency_score)
        ),
        "recency_score_maximum": float(
            np.nanmax(recency_score)
        ),
        "submitter_diversity_minimum": float(
            np.min(submitter_diversity_score)
        ),
        "submitter_diversity_maximum": float(
            np.max(submitter_diversity_score)
        ),
    },

    "labeling_function_counts": {
        "LF_Recency": lf_recency_counts,
        "LF_ReviewConf": lf_review_counts,
        "LF_Conflict": lf_conflict_counts,
    },

    "full_weak_labels": {
        **nullable_label_counts(
            weak_label_table[
                "full_weak_label_binary"
            ]
        ),
        "training_eligible": int(
            weak_label_table[
                "full_training_eligible"
            ].sum()
        ),
        "all_abstain": int(
            weak_label_table[
                "full_all_abstain_flag"
            ].sum()
        ),
        "ties": int(
            weak_label_table[
                "full_tie_flag"
            ].sum()
        ),
        "disagreement": int(
            weak_label_table[
                "full_disagreement_flag"
            ].sum()
        ),
        "unanimous_among_fired": int(
            weak_label_table[
                "full_unanimous_among_fired"
            ].sum()
        ),
    },

    "no_star_weak_labels": {
        **nullable_label_counts(
            weak_label_table[
                "no_star_weak_label_binary"
            ]
        ),
        "training_eligible": int(
            weak_label_table[
                "no_star_training_eligible"
            ].sum()
        ),
        "all_abstain": int(
            weak_label_table[
                "no_star_all_abstain_flag"
            ].sum()
        ),
        "ties": int(
            weak_label_table[
                "no_star_tie_flag"
            ].sum()
        ),
        "disagreement": int(
            weak_label_table[
                "no_star_disagreement_flag"
            ].sum()
        ),
    },

    "conflict_veto": {
        "conflict_positive_records": int(
            aggregate_conflict.sum()
        ),
        "full_conflict_records_labeled_unstable": int(
            conflict_rows[
                "full_weak_label_binary"
            ].eq(0).sum()
        ),
        "no_star_conflict_records_labeled_unstable": int(
            conflict_rows[
                "no_star_weak_label_binary"
            ].eq(0).sum()
        ),
        "violations": 0,
    },

    "leakage_checks": {
        "t1_information_used": False,
        "future_instability_outcome_created": False,
        "model_fitted": False,
        "threshold_tuned_against_t1": False,
    },
}


# --------------------------------------------------------------------------------------------------
# 17. Build temporary local artifacts
# --------------------------------------------------------------------------------------------------

BUILD_DIR = Path(
    "/content/stage4b_weak_label_build_v1"
)

if BUILD_DIR.exists():
    shutil.rmtree(BUILD_DIR)

BUILD_DIR.mkdir(
    parents=True,
    exist_ok=False,
)

temporary_table_path = (
    BUILD_DIR
    / WEAK_LABEL_TABLE_PATH.name
)

temporary_transform_path = (
    BUILD_DIR
    / TRANSFORM_PARAMETERS_PATH.name
)

temporary_rules_path = (
    BUILD_DIR
    / WEAK_LABEL_RULES_PATH.name
)

temporary_qc_path = (
    BUILD_DIR
    / WEAK_LABEL_QC_PATH.name
)

temporary_manifest_path = (
    BUILD_DIR
    / STAGE4B_MANIFEST_PATH.name
)


weak_label_table.to_parquet(
    temporary_table_path,
    index=False,
    engine="pyarrow",
    compression="zstd",
)

write_json(
    temporary_transform_path,
    transform_parameters,
)

write_json(
    temporary_rules_path,
    weak_label_rules,
)

write_json(
    temporary_qc_path,
    qc_report,
)


# --------------------------------------------------------------------------------------------------
# 18. Readback validation
# --------------------------------------------------------------------------------------------------

table_readback = pd.read_parquet(
    temporary_table_path
)

assert table_readback.shape == weak_label_table.shape
assert len(table_readback) == 71_659
assert table_readback["rcv_accession"].is_unique

assert int(
    table_readback[
        "aggregate_conflict_flag"
    ].sum()
) == 1_484

assert table_readback[
    "t1_information_used"
].eq(False).all()

assert table_readback[
    "future_instability_outcome_created"
].eq(False).all()

assert table_readback[
    "ges_model_fitted"
].eq(False).all()


# --------------------------------------------------------------------------------------------------
# 19. Calculate artifact hashes
# --------------------------------------------------------------------------------------------------

weak_label_table_hash = sha256_file(
    temporary_table_path
)

transform_parameters_hash = sha256_file(
    temporary_transform_path
)

weak_label_rules_hash = sha256_file(
    temporary_rules_path
)

weak_label_qc_hash = sha256_file(
    temporary_qc_path
)


# --------------------------------------------------------------------------------------------------
# 20. Freeze manifest
# --------------------------------------------------------------------------------------------------

freeze_manifest = {
    "manifest_name": "Stage 4B weak-label freeze manifest",
    "version": "1.0.0",
    "created_at_utc": created_at_utc,
    "manifest_status": (
        "STAGE4B_FEATURE_TRANSFORMS_AND_WEAK_LABELS_FROZEN"
    ),
    "validation_decision": (
        "PASS_STAGE4B_WEAK_LABELS_ACCEPTED_AND_FROZEN"
    ),

    "source_artifacts": {
        path: {
            "sha256": observed_hash,
        }
        for path, observed_hash in sorted(
            verified_input_hashes.items()
        )
    },

    "frozen_outputs": {
        "weak_label_table": {
            "path": str(
                WEAK_LABEL_TABLE_PATH
            ),
            "sha256": weak_label_table_hash,
            "rows": int(
                weak_label_table.shape[0]
            ),
            "columns": int(
                weak_label_table.shape[1]
            ),
        },

        "transformation_parameters": {
            "path": str(
                TRANSFORM_PARAMETERS_PATH
            ),
            "sha256": (
                transform_parameters_hash
            ),
        },

        "weak_label_rules": {
            "path": str(
                WEAK_LABEL_RULES_PATH
            ),
            "sha256": weak_label_rules_hash,
        },

        "quality_control_report": {
            "path": str(
                WEAK_LABEL_QC_PATH
            ),
            "sha256": weak_label_qc_hash,
        },
    },

    "full_weak_label_summary": (
        qc_report["full_weak_labels"]
    ),

    "no_star_weak_label_summary": (
        qc_report["no_star_weak_labels"]
    ),

    "scientific_boundary": {
        "t1_information_used": False,
        "future_outcome_created": False,
        "feature_transformations_frozen": True,
        "weak_labels_frozen": True,
        "full_model_fitted": False,
        "no_star_model_fitted": False,
        "ges_probabilities_generated": False,
    },

    "next_authorized_work": (
        "Stage 4C: fit and freeze the full GES logistic-regression "
        "model and the mandatory no-star logistic-regression "
        "model using only the frozen Stage 4B T0 artifacts."
    ),
}

write_json(
    temporary_manifest_path,
    freeze_manifest,
)

stage4b_manifest_hash = sha256_file(
    temporary_manifest_path
)


# --------------------------------------------------------------------------------------------------
# 21. Copy artifacts to Drive
# --------------------------------------------------------------------------------------------------

copy_and_verify(
    temporary_table_path,
    WEAK_LABEL_TABLE_PATH,
)

copy_and_verify(
    temporary_transform_path,
    TRANSFORM_PARAMETERS_PATH,
)

copy_and_verify(
    temporary_rules_path,
    WEAK_LABEL_RULES_PATH,
)

copy_and_verify(
    temporary_qc_path,
    WEAK_LABEL_QC_PATH,
)

# Manifest copied last
copy_and_verify(
    temporary_manifest_path,
    STAGE4B_MANIFEST_PATH,
)


# --------------------------------------------------------------------------------------------------
# 22. Persistent checksum verification
# --------------------------------------------------------------------------------------------------

assert (
    sha256_file(WEAK_LABEL_TABLE_PATH)
    == weak_label_table_hash
)

assert (
    sha256_file(TRANSFORM_PARAMETERS_PATH)
    == transform_parameters_hash
)

assert (
    sha256_file(WEAK_LABEL_RULES_PATH)
    == weak_label_rules_hash
)

assert (
    sha256_file(WEAK_LABEL_QC_PATH)
    == weak_label_qc_hash
)

assert (
    sha256_file(STAGE4B_MANIFEST_PATH)
    == stage4b_manifest_hash
)


# --------------------------------------------------------------------------------------------------
# 23. Persistent readback
# --------------------------------------------------------------------------------------------------

persistent_table = pd.read_parquet(
    WEAK_LABEL_TABLE_PATH
)

assert persistent_table.shape == weak_label_table.shape
assert persistent_table["rcv_accession"].is_unique


# --------------------------------------------------------------------------------------------------
# 24. Final output
# --------------------------------------------------------------------------------------------------

full_label_counts = nullable_label_counts(
    persistent_table[
        "full_weak_label_binary"
    ]
)

no_star_label_counts = nullable_label_counts(
    persistent_table[
        "no_star_weak_label_binary"
    ]
)

print("=" * 120)
print("STAGE 4B — FEATURE TRANSFORMATIONS AND WEAK-LABEL FREEZE")
print("=" * 120)

print(f"Created at UTC:                    {created_at_utc}")
print(
    "Validation decision:              "
    "PASS_STAGE4B_WEAK_LABELS_ACCEPTED_AND_FROZEN"
)
print()

print("T0 COHORT")
print(f"  Rows:                           {len(persistent_table):,}")
print(f"  Columns:                        {persistent_table.shape[1]:,}")
print(f"  Unique RCVs:                    {persistent_table['rcv_accession'].nunique():,}")
print()

print("TRANSFORMATIONS")
print(
    "  Maximum observation window:     "
    f"{maximum_observation_window_days:,.0f} days"
)
print(
    "  Recency score range:             "
    f"{np.nanmin(recency_score):.6f} to "
    f"{np.nanmax(recency_score):.6f}"
)
print(
    "  Submitter-diversity range:       "
    f"{np.min(submitter_diversity_score):.6f} to "
    f"{np.max(submitter_diversity_score):.6f}"
)
print()

print("LABELING FUNCTIONS")
print(f"  LF-Recency:                      {lf_recency_counts}")
print(f"  LF-ReviewConf:                   {lf_review_counts}")
print(f"  LF-Conflict:                     {lf_conflict_counts}")
print()

print("FULL WEAK LABEL")
print(f"  Stable:                          {full_label_counts['stable_1']:,}")
print(f"  Unstable:                        {full_label_counts['unstable_0']:,}")
print(f"  Unlabeled:                       {full_label_counts['unlabeled_missing']:,}")
print(
    "  Training eligible:               "
    f"{persistent_table['full_training_eligible'].sum():,}"
)
print(
    "  Disagreement cases:              "
    f"{persistent_table['full_disagreement_flag'].sum():,}"
)
print(
    "  Tie cases:                       "
    f"{persistent_table['full_tie_flag'].sum():,}"
)
print()

print("NO-STAR WEAK LABEL")
print(f"  Stable:                          {no_star_label_counts['stable_1']:,}")
print(f"  Unstable:                        {no_star_label_counts['unstable_0']:,}")
print(f"  Unlabeled:                       {no_star_label_counts['unlabeled_missing']:,}")
print(
    "  Training eligible:               "
    f"{persistent_table['no_star_training_eligible'].sum():,}"
)
print()

print("CONFLICT VETO")
print(f"  Conflict-positive records:       {aggregate_conflict.sum():,}")
print("  Full-label veto violations:      0")
print("  No-star-label veto violations:   0")
print()

print("SCIENTIFIC BOUNDARY")
print("  T1 information used:             False")
print("  Future outcome created:          False")
print("  Full GES model fitted:           False")
print("  No-star GES model fitted:        False")
print("  Risk thresholds applied:         False")
print()

print("FROZEN OUTPUTS")
print(f"  Weak-label table:                {WEAK_LABEL_TABLE_PATH}")
print(f"    SHA-256:                       {weak_label_table_hash}")
print(f"  Transform parameters:            {TRANSFORM_PARAMETERS_PATH}")
print(f"    SHA-256:                       {transform_parameters_hash}")
print(f"  Weak-label rules:                {WEAK_LABEL_RULES_PATH}")
print(f"    SHA-256:                       {weak_label_rules_hash}")
print(f"  QC report:                       {WEAK_LABEL_QC_PATH}")
print(f"    SHA-256:                       {weak_label_qc_hash}")
print(f"  Freeze manifest:                 {STAGE4B_MANIFEST_PATH}")
print(f"    SHA-256:                       {stage4b_manifest_hash}")
print()

print("NEXT AUTHORIZED WORK")
print("  Stage 4C — fit and freeze the full GES and no-star GES")
print("  logistic-regression models using only these frozen T0 labels.")
print("=" * 120)

STAGE 4B — FEATURE TRANSFORMATIONS AND WEAK-LABEL FREEZE
Created at UTC:                    2026-07-20T22:06:01.208650+00:00
Validation decision:              PASS_STAGE4B_WEAK_LABELS_ACCEPTED_AND_FROZEN

T0 COHORT
  Rows:                           71,659
  Columns:                        43
  Unique RCVs:                    71,659

TRANSFORMATIONS
  Maximum observation window:     10,516 days
  Recency score range:             0.000000 to 0.999239
  Submitter-diversity range:       0.000000 to 1.000000

LABELING FUNCTIONS
  LF-Recency:                      {'stable': 62696, 'unstable': 368, 'abstain': 8595}
  LF-ReviewConf:                   {'stable': 17382, 'unstable': 3878, 'abstain': 50399}
  LF-Conflict:                     {'stable': 0, 'unstable': 1484, 'abstain': 70175}

FULL WEAK LABEL
  Stable:                          61,842
  Unstable:                        5,723
  Unlabeled:                       4,094
  Training eligible:               67,565
  Disagreement cases:      

In [28]:
# ==================================================================================================
# STAGE 4C — FIT AND FREEZE FULL GES AND NO-STAR GES MODELS
#
# Models:
#   1. Full GES:
#        recency_score
#        recency_missing_flag
#        submitter_diversity_score
#        review_confidence
#        aggregate_conflict_flag
#        scv_group_entropy_normalized
#
#   2. No-star GES:
#        recency_score
#        recency_missing_flag
#        submitter_diversity_score
#        aggregate_conflict_flag
#        scv_group_entropy_normalized
#
# Scientific boundary:
#   - Uses only frozen Stage 4B T0 features and weak labels.
#   - Does not use T1 classifications or linkage outcomes.
#   - Does not construct future-instability outcomes.
#   - Does not tune hyperparameters against T1.
#   - Uses fixed logistic-regression settings.
#   - Generates baseline T0 P(stable) for every T0 RCV.
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import os
import platform
import shutil
import sys

import joblib
import numpy as np
import pandas as pd
import sklearn

from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    log_loss,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


# --------------------------------------------------------------------------------------------------
# 1. Google Drive
# --------------------------------------------------------------------------------------------------

DRIVE_ROOT = Path("/content/drive/MyDrive")

if not DRIVE_ROOT.exists():
    from google.colab import drive
    drive.mount("/content/drive")

assert DRIVE_ROOT.exists(), "Google Drive is not mounted."


# --------------------------------------------------------------------------------------------------
# 2. Paths
# --------------------------------------------------------------------------------------------------

BASE_DIR = DRIVE_ROOT / "GES_RAG_Temporal_Study"

STAGE4_DATA_DIR = BASE_DIR / "data_processed" / "stage4_ges"
STAGE4_CONFIG_DIR = BASE_DIR / "configs" / "stage4_ges"
STAGE4_MODEL_DIR = BASE_DIR / "models" / "stage4_ges"

STAGE4_DATA_DIR.mkdir(parents=True, exist_ok=True)
STAGE4_CONFIG_DIR.mkdir(parents=True, exist_ok=True)
STAGE4_MODEL_DIR.mkdir(parents=True, exist_ok=True)


# Frozen Stage 4B inputs
WEAK_LABEL_TABLE_PATH = (
    STAGE4_DATA_DIR
    / "stage4b_t0_feature_transforms_and_weak_labels_v1.parquet"
)

TRANSFORM_PARAMETERS_PATH = (
    STAGE4_CONFIG_DIR
    / "stage4b_t0_feature_transform_parameters_v1.json"
)

WEAK_LABEL_RULES_PATH = (
    STAGE4_CONFIG_DIR
    / "stage4b_weak_label_rules_v1.json"
)

WEAK_LABEL_QC_PATH = (
    STAGE4_CONFIG_DIR
    / "stage4b_weak_label_qc_report_v1.json"
)

STAGE4B_MANIFEST_PATH = (
    STAGE4_CONFIG_DIR
    / "stage4b_weak_label_freeze_manifest_v1.json"
)


# Stage 4C outputs
FULL_MODEL_PATH = (
    STAGE4_MODEL_DIR
    / "stage4c_full_ges_logistic_model_v1.joblib"
)

NO_STAR_MODEL_PATH = (
    STAGE4_MODEL_DIR
    / "stage4c_no_star_ges_logistic_model_v1.joblib"
)

SCORE_TABLE_PATH = (
    STAGE4_DATA_DIR
    / "stage4c_t0_full_and_no_star_ges_scores_v1.parquet"
)

COEFFICIENT_TABLE_PATH = (
    STAGE4_DATA_DIR
    / "stage4c_ges_model_coefficients_v1.parquet"
)

MODEL_SPECIFICATION_PATH = (
    STAGE4_CONFIG_DIR
    / "stage4c_ges_model_specification_v1.json"
)

MODEL_QC_PATH = (
    STAGE4_CONFIG_DIR
    / "stage4c_ges_model_qc_report_v1.json"
)

STAGE4C_MANIFEST_PATH = (
    STAGE4_CONFIG_DIR
    / "stage4c_ges_model_freeze_manifest_v1.json"
)

OUTPUT_PATHS = [
    FULL_MODEL_PATH,
    NO_STAR_MODEL_PATH,
    SCORE_TABLE_PATH,
    COEFFICIENT_TABLE_PATH,
    MODEL_SPECIFICATION_PATH,
    MODEL_QC_PATH,
    STAGE4C_MANIFEST_PATH,
]

existing_outputs = [
    str(path)
    for path in OUTPUT_PATHS
    if path.exists()
]

if existing_outputs:
    raise FileExistsError(
        "Stage 4C outputs already exist and were not overwritten:\n"
        + "\n".join(existing_outputs)
    )


# --------------------------------------------------------------------------------------------------
# 3. Expected Stage 4B hashes
# --------------------------------------------------------------------------------------------------

EXPECTED_INPUT_HASHES = {
    str(WEAK_LABEL_TABLE_PATH): (
        "c2e9e4f96cd1f61d3962e88d28557dfa1221729b8f79f8fd4d2dafb34847bdb8"
    ),
    str(TRANSFORM_PARAMETERS_PATH): (
        "baeab167e19381138f93c51ba2fa00e4eeed684b36122c80cd2bf9fc4fe4b08d"
    ),
    str(WEAK_LABEL_RULES_PATH): (
        "3d78e66cea1fed5c75ef1cab1b7cf44d3d3d7bfae50909173bae6c3a0e0bff61"
    ),
    str(WEAK_LABEL_QC_PATH): (
        "03b14efb6d297fd517043c8ff952977715415723f191d079d968c111368d31a9"
    ),
    str(STAGE4B_MANIFEST_PATH): (
        "e766061442e6d4610f661f44a619e41b45dc6363b28f6176d3f9a71f8215c63f"
    ),
}


# --------------------------------------------------------------------------------------------------
# 4. Frozen model settings
# --------------------------------------------------------------------------------------------------

RANDOM_SEED = 42
DECISION_THRESHOLD = 0.50

LOGISTIC_SETTINGS = {
    "penalty": "l2",
    "C": 1.0,
    "solver": "lbfgs",
    "class_weight": None,
    "max_iter": 2_000,
    "tol": 1e-6,
    "fit_intercept": True,
    "random_state": RANDOM_SEED,
}

FULL_FEATURES = [
    "recency_score",
    "recency_missing_flag",
    "submitter_diversity_score",
    "review_confidence",
    "aggregate_conflict_flag",
    "scv_group_entropy_normalized",
]

NO_STAR_FEATURES = [
    "recency_score",
    "recency_missing_flag",
    "submitter_diversity_score",
    "aggregate_conflict_flag",
    "scv_group_entropy_normalized",
]

assert "review_confidence" in FULL_FEATURES
assert "review_confidence" not in NO_STAR_FEATURES


# --------------------------------------------------------------------------------------------------
# 5. Utility functions
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path: Path,
    chunk_size: int = 8 * 1024 * 1024,
) -> str:
    """Calculate SHA-256 without loading the complete file into memory."""

    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def write_json(path: Path, payload: dict) -> None:
    """Write deterministic human-readable JSON."""

    text = json.dumps(
        payload,
        indent=2,
        sort_keys=True,
        ensure_ascii=False,
    )

    path.write_text(text + "\n", encoding="utf-8")


def copy_and_verify(source: Path, destination: Path) -> None:
    """Copy an artifact to Drive and verify its checksum before renaming."""

    partial_destination = Path(str(destination) + ".partial")

    if partial_destination.exists():
        partial_destination.unlink()

    shutil.copy2(source, partial_destination)

    source_hash = sha256_file(source)
    copied_hash = sha256_file(partial_destination)

    assert source_hash == copied_hash, (
        f"Checksum mismatch while copying {destination.name}"
    )

    os.replace(partial_destination, destination)


def normalize_rcv(series: pd.Series) -> pd.Series:
    """Normalize RCV accession formatting."""

    return (
        series.astype("string")
        .str.strip()
        .str.upper()
    )


def probability_summary(values: np.ndarray) -> dict:
    """Return a deterministic score-distribution summary."""

    values = np.asarray(values, dtype=float)

    assert np.isfinite(values).all()

    percentiles = np.percentile(
        values,
        [0, 1, 5, 10, 25, 50, 75, 90, 95, 99, 100],
    )

    percentile_names = [
        "p00_min",
        "p01",
        "p05",
        "p10",
        "p25",
        "p50_median",
        "p75",
        "p90",
        "p95",
        "p99",
        "p100_max",
    ]

    return {
        "mean": float(np.mean(values)),
        "standard_deviation": float(np.std(values)),
        **{
            name: float(value)
            for name, value in zip(
                percentile_names,
                percentiles,
            )
        },
    }


def classification_diagnostics(
    y_true: np.ndarray,
    probabilities: np.ndarray,
    threshold: float,
) -> dict:
    """
    Calculate reconstruction diagnostics against the T0 weak label.

    These are not temporal-validation results.
    """

    y_true = np.asarray(y_true, dtype=int)
    probabilities = np.asarray(probabilities, dtype=float)

    predictions = (
        probabilities >= threshold
    ).astype(int)

    matrix = confusion_matrix(
        y_true,
        predictions,
        labels=[0, 1],
    )

    true_negative = int(matrix[0, 0])
    false_positive = int(matrix[0, 1])
    false_negative = int(matrix[1, 0])
    true_positive = int(matrix[1, 1])

    return {
        "diagnostic_scope": (
            "In-sample reconstruction of the frozen T0 weak label; "
            "not evidence of temporal predictive validity."
        ),
        "rows": int(len(y_true)),
        "positive_stable_labels": int((y_true == 1).sum()),
        "negative_unstable_labels": int((y_true == 0).sum()),
        "stable_prevalence": float(np.mean(y_true)),
        "auroc": float(
            roc_auc_score(
                y_true,
                probabilities,
            )
        ),
        "auprc": float(
            average_precision_score(
                y_true,
                probabilities,
            )
        ),
        "brier_score": float(
            brier_score_loss(
                y_true,
                probabilities,
            )
        ),
        "log_loss": float(
            log_loss(
                y_true,
                probabilities,
                labels=[0, 1],
            )
        ),
        "accuracy_at_0_5": float(
            accuracy_score(
                y_true,
                predictions,
            )
        ),
        "confusion_matrix": {
            "true_negative": true_negative,
            "false_positive": false_positive,
            "false_negative": false_negative,
            "true_positive": true_positive,
        },
    }


def fit_fixed_ges_model(
    dataframe: pd.DataFrame,
    model_name: str,
    feature_names: list[str],
    label_column: str,
    eligibility_column: str,
) -> dict:
    """
    Fit one fixed logistic-regression pipeline.

    Pipeline:
      median imputation
      standard scaling
      fixed L2 logistic regression
    """

    missing_features = [
        feature
        for feature in feature_names
        if feature not in dataframe.columns
    ]

    if missing_features:
        raise KeyError(
            f"{model_name}: missing features: {missing_features}"
        )

    training_mask = (
        dataframe[eligibility_column].astype(bool)
        & dataframe[label_column].notna()
    )

    training_frame = dataframe.loc[
        training_mask,
        feature_names,
    ].astype(float)

    training_label = (
        dataframe.loc[
            training_mask,
            label_column,
        ]
        .astype(int)
        .to_numpy()
    )

    all_features = dataframe[
        feature_names
    ].astype(float)

    assert len(training_frame) > 0
    assert set(np.unique(training_label)) == {0, 1}

    pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="median",
                    add_indicator=False,
                ),
            ),
            (
                "scaler",
                StandardScaler(
                    with_mean=True,
                    with_std=True,
                ),
            ),
            (
                "logistic_regression",
                LogisticRegression(
                    penalty=LOGISTIC_SETTINGS["penalty"],
                    C=LOGISTIC_SETTINGS["C"],
                    solver=LOGISTIC_SETTINGS["solver"],
                    class_weight=LOGISTIC_SETTINGS["class_weight"],
                    max_iter=LOGISTIC_SETTINGS["max_iter"],
                    tol=LOGISTIC_SETTINGS["tol"],
                    fit_intercept=LOGISTIC_SETTINGS["fit_intercept"],
                    random_state=LOGISTIC_SETTINGS["random_state"],
                ),
            ),
        ]
    )

    pipeline.fit(
        training_frame,
        training_label,
    )

    all_probabilities = pipeline.predict_proba(
        all_features
    )[:, 1]

    training_probabilities = pipeline.predict_proba(
        training_frame
    )[:, 1]

    assert np.isfinite(all_probabilities).all()
    assert np.all(all_probabilities >= 0.0)
    assert np.all(all_probabilities <= 1.0)
    assert float(np.std(all_probabilities)) > 0.0

    imputer = pipeline.named_steps["imputer"]
    scaler = pipeline.named_steps["scaler"]
    logistic_model = pipeline.named_steps[
        "logistic_regression"
    ]

    assert logistic_model.n_iter_[0] < LOGISTIC_SETTINGS["max_iter"]

    standardized_coefficients = (
        logistic_model.coef_[0].astype(float)
    )

    standardized_intercept = float(
        logistic_model.intercept_[0]
    )

    imputation_values = (
        imputer.statistics_.astype(float)
    )

    scaler_means = scaler.mean_.astype(float)
    scaler_scales = scaler.scale_.astype(float)

    assert len(feature_names) == len(
        standardized_coefficients
    )

    assert np.all(scaler_scales > 0)

    raw_coefficients = (
        standardized_coefficients
        / scaler_scales
    )

    raw_intercept = float(
        standardized_intercept
        - np.sum(
            standardized_coefficients
            * scaler_means
            / scaler_scales
        )
    )

    coefficient_records = []

    for index, feature_name in enumerate(feature_names):
        coefficient_records.append(
            {
                "model_name": model_name,
                "feature_order": int(index),
                "feature_name": feature_name,
                "imputation_value": float(
                    imputation_values[index]
                ),
                "scaler_mean": float(
                    scaler_means[index]
                ),
                "scaler_scale": float(
                    scaler_scales[index]
                ),
                "standardized_coefficient": float(
                    standardized_coefficients[index]
                ),
                "raw_space_coefficient_after_imputation": float(
                    raw_coefficients[index]
                ),
                "absolute_standardized_coefficient": float(
                    abs(
                        standardized_coefficients[index]
                    )
                ),
            }
        )

    diagnostics = classification_diagnostics(
        y_true=training_label,
        probabilities=training_probabilities,
        threshold=DECISION_THRESHOLD,
    )

    stable_training_probabilities = training_probabilities[
        training_label == 1
    ]

    unstable_training_probabilities = training_probabilities[
        training_label == 0
    ]

    diagnostics[
        "mean_probability_for_stable_weak_labels"
    ] = float(
        np.mean(stable_training_probabilities)
    )

    diagnostics[
        "mean_probability_for_unstable_weak_labels"
    ] = float(
        np.mean(unstable_training_probabilities)
    )

    diagnostics[
        "median_probability_for_stable_weak_labels"
    ] = float(
        np.median(stable_training_probabilities)
    )

    diagnostics[
        "median_probability_for_unstable_weak_labels"
    ] = float(
        np.median(unstable_training_probabilities)
    )

    assert (
        diagnostics[
            "mean_probability_for_stable_weak_labels"
        ]
        >
        diagnostics[
            "mean_probability_for_unstable_weak_labels"
        ]
    )

    return {
        "model_name": model_name,
        "pipeline": pipeline,
        "feature_names": feature_names,
        "label_column": label_column,
        "eligibility_column": eligibility_column,
        "training_mask": training_mask.to_numpy(),
        "training_rows": int(training_mask.sum()),
        "training_label": training_label,
        "all_probabilities": all_probabilities,
        "training_probabilities": training_probabilities,
        "coefficient_records": coefficient_records,
        "standardized_intercept": standardized_intercept,
        "raw_space_intercept_after_imputation": raw_intercept,
        "imputation_values": {
            feature_name: float(imputation_values[index])
            for index, feature_name in enumerate(feature_names)
        },
        "scaler_means": {
            feature_name: float(scaler_means[index])
            for index, feature_name in enumerate(feature_names)
        },
        "scaler_scales": {
            feature_name: float(scaler_scales[index])
            for index, feature_name in enumerate(feature_names)
        },
        "n_iterations": int(logistic_model.n_iter_[0]),
        "diagnostics": diagnostics,
        "all_score_summary": probability_summary(
            all_probabilities
        ),
    }


# --------------------------------------------------------------------------------------------------
# 6. Verify Stage 4B artifacts
# --------------------------------------------------------------------------------------------------

required_paths = [
    WEAK_LABEL_TABLE_PATH,
    TRANSFORM_PARAMETERS_PATH,
    WEAK_LABEL_RULES_PATH,
    WEAK_LABEL_QC_PATH,
    STAGE4B_MANIFEST_PATH,
]

missing_paths = [
    str(path)
    for path in required_paths
    if not path.exists()
]

if missing_paths:
    raise FileNotFoundError(
        "Required Stage 4B artifacts are missing:\n"
        + "\n".join(missing_paths)
    )

verified_input_hashes = {}

for path_string, expected_hash in EXPECTED_INPUT_HASHES.items():
    path = Path(path_string)
    observed_hash = sha256_file(path)

    assert observed_hash == expected_hash, (
        f"Stage 4B checksum mismatch:\n"
        f"Path:     {path}\n"
        f"Expected: {expected_hash}\n"
        f"Observed: {observed_hash}"
    )

    verified_input_hashes[str(path)] = observed_hash


# --------------------------------------------------------------------------------------------------
# 7. Semantic check of the Stage 4B freeze manifest
# --------------------------------------------------------------------------------------------------

with STAGE4B_MANIFEST_PATH.open(
    "r",
    encoding="utf-8",
) as handle:
    stage4b_manifest = json.load(handle)

assert (
    stage4b_manifest["manifest_status"]
    == "STAGE4B_FEATURE_TRANSFORMS_AND_WEAK_LABELS_FROZEN"
)

assert (
    stage4b_manifest["validation_decision"]
    == "PASS_STAGE4B_WEAK_LABELS_ACCEPTED_AND_FROZEN"
)

stage4b_boundary = stage4b_manifest[
    "scientific_boundary"
]

assert stage4b_boundary["t1_information_used"] is False
assert stage4b_boundary["future_outcome_created"] is False
assert stage4b_boundary["weak_labels_frozen"] is True
assert stage4b_boundary["full_model_fitted"] is False
assert stage4b_boundary["no_star_model_fitted"] is False


# --------------------------------------------------------------------------------------------------
# 8. Read frozen Stage 4B table
# --------------------------------------------------------------------------------------------------

data = pd.read_parquet(
    WEAK_LABEL_TABLE_PATH
)

assert data.shape == (71_659, 43), (
    f"Unexpected Stage 4B shape: {data.shape}"
)

required_columns = [
    "t0_row_order",
    "rcv_accession",
    "variation_id",
    "vcv_accession",
    "target_genes_json",
    "study_scope",
    "recency_score",
    "recency_missing_flag",
    "submitter_diversity_score",
    "review_confidence",
    "aggregate_conflict_flag",
    "scv_group_entropy_normalized",
    "full_weak_label_binary",
    "full_training_eligible",
    "full_weak_stability_probability",
    "no_star_weak_label_binary",
    "no_star_training_eligible",
    "no_star_weak_stability_probability",
    "t1_information_used",
    "future_instability_outcome_created",
    "ges_model_fitted",
]

missing_columns = [
    column
    for column in required_columns
    if column not in data.columns
]

if missing_columns:
    raise KeyError(
        "Required Stage 4B columns are missing:\n"
        + "\n".join(missing_columns)
    )

data = data.copy()

data["rcv_accession"] = normalize_rcv(
    data["rcv_accession"]
)

assert data["rcv_accession"].notna().all()
assert data["rcv_accession"].is_unique
assert data["t1_information_used"].eq(False).all()
assert (
    data["future_instability_outcome_created"]
    .eq(False)
    .all()
)
assert data["ges_model_fitted"].eq(False).all()

assert int(data["full_training_eligible"].sum()) == 67_565
assert int(data["no_star_training_eligible"].sum()) == 63_148


# --------------------------------------------------------------------------------------------------
# 9. Fit the fixed full GES model
# --------------------------------------------------------------------------------------------------

full_result = fit_fixed_ges_model(
    dataframe=data,
    model_name="full_ges",
    feature_names=FULL_FEATURES,
    label_column="full_weak_label_binary",
    eligibility_column="full_training_eligible",
)

assert full_result["training_rows"] == 67_565


# --------------------------------------------------------------------------------------------------
# 10. Fit the fixed no-star GES model
# --------------------------------------------------------------------------------------------------

no_star_result = fit_fixed_ges_model(
    dataframe=data,
    model_name="no_star_ges",
    feature_names=NO_STAR_FEATURES,
    label_column="no_star_weak_label_binary",
    eligibility_column="no_star_training_eligible",
)

assert no_star_result["training_rows"] == 63_148


# --------------------------------------------------------------------------------------------------
# 11. Construct the frozen T0 GES score table
# --------------------------------------------------------------------------------------------------

full_probabilities = full_result[
    "all_probabilities"
]

no_star_probabilities = no_star_result[
    "all_probabilities"
]

score_table = pd.DataFrame(
    {
        "t0_row_order": data["t0_row_order"],
        "rcv_accession": data["rcv_accession"],
        "variation_id": data["variation_id"],
        "vcv_accession": data["vcv_accession"],
        "target_genes_json": data["target_genes_json"],
        "study_scope": data["study_scope"],

        "full_training_eligible": (
            data["full_training_eligible"].astype(bool)
        ),
        "full_weak_label_binary": (
            data["full_weak_label_binary"]
        ),
        "full_weak_stability_probability": (
            data["full_weak_stability_probability"]
        ),

        "no_star_training_eligible": (
            data["no_star_training_eligible"].astype(bool)
        ),
        "no_star_weak_label_binary": (
            data["no_star_weak_label_binary"]
        ),
        "no_star_weak_stability_probability": (
            data["no_star_weak_stability_probability"]
        ),

        "full_ges_p_stable_t0": full_probabilities,
        "full_ges_predicted_stable_at_0_5": (
            full_probabilities >= DECISION_THRESHOLD
        ),

        "no_star_ges_p_stable_t0": no_star_probabilities,
        "no_star_ges_predicted_stable_at_0_5": (
            no_star_probabilities >= DECISION_THRESHOLD
        ),

        "model_decision_threshold": DECISION_THRESHOLD,

        "t1_information_used": False,
        "future_instability_outcome_created": False,
        "temporal_performance_evaluated": False,
        "stage4c_version": "1.0.0",
    }
)

assert len(score_table) == 71_659
assert score_table["rcv_accession"].is_unique

for score_column in [
    "full_ges_p_stable_t0",
    "no_star_ges_p_stable_t0",
]:
    assert score_table[score_column].notna().all()
    assert score_table[score_column].between(0, 1).all()
    assert float(score_table[score_column].std()) > 0

assert score_table["t1_information_used"].eq(False).all()

assert (
    score_table["future_instability_outcome_created"]
    .eq(False)
    .all()
)

assert (
    score_table["temporal_performance_evaluated"]
    .eq(False)
    .all()
)


# --------------------------------------------------------------------------------------------------
# 12. Coefficient table
# --------------------------------------------------------------------------------------------------

coefficient_records = (
    full_result["coefficient_records"]
    + no_star_result["coefficient_records"]
)

coefficient_table = pd.DataFrame(
    coefficient_records
)

intercept_rows = pd.DataFrame(
    [
        {
            "model_name": "full_ges",
            "feature_order": -1,
            "feature_name": "__INTERCEPT__",
            "imputation_value": np.nan,
            "scaler_mean": np.nan,
            "scaler_scale": np.nan,
            "standardized_coefficient": (
                full_result["standardized_intercept"]
            ),
            "raw_space_coefficient_after_imputation": (
                full_result[
                    "raw_space_intercept_after_imputation"
                ]
            ),
            "absolute_standardized_coefficient": abs(
                full_result["standardized_intercept"]
            ),
        },
        {
            "model_name": "no_star_ges",
            "feature_order": -1,
            "feature_name": "__INTERCEPT__",
            "imputation_value": np.nan,
            "scaler_mean": np.nan,
            "scaler_scale": np.nan,
            "standardized_coefficient": (
                no_star_result["standardized_intercept"]
            ),
            "raw_space_coefficient_after_imputation": (
                no_star_result[
                    "raw_space_intercept_after_imputation"
                ]
            ),
            "absolute_standardized_coefficient": abs(
                no_star_result["standardized_intercept"]
            ),
        },
    ]
)

coefficient_table = pd.concat(
    [
        intercept_rows,
        coefficient_table,
    ],
    ignore_index=True,
)

assert (
    set(
        coefficient_table.loc[
            coefficient_table["model_name"] == "full_ges",
            "feature_name",
        ]
    )
    ==
    set(FULL_FEATURES + ["__INTERCEPT__"])
)

assert (
    set(
        coefficient_table.loc[
            coefficient_table["model_name"] == "no_star_ges",
            "feature_name",
        ]
    )
    ==
    set(NO_STAR_FEATURES + ["__INTERCEPT__"])
)


# --------------------------------------------------------------------------------------------------
# 13. Freeze model specification
# --------------------------------------------------------------------------------------------------

created_at_utc = datetime.now(
    timezone.utc
).isoformat()

model_specification = {
    "specification_name": (
        "Stage 4C full and no-star GES logistic-regression specification"
    ),
    "version": "1.0.0",
    "created_at_utc": created_at_utc,
    "status": "FROZEN_BEFORE_TEMPORAL_OUTCOME_CONSTRUCTION",

    "scientific_unit": (
        "RCV-level variant-condition aggregate"
    ),

    "target_definition": {
        "full_model": (
            "Frozen Stage 4B full majority-vote weak label "
            "with conflict veto."
        ),
        "no_star_model": (
            "Frozen Stage 4B no-star weak label using only "
            "LF-Recency and LF-Conflict."
        ),
        "hard_label_training": True,
        "unlabeled_records_excluded_from_model_fitting": True,
        "all_t0_records_scored_after_fitting": True,
    },

    "full_model": {
        "model_name": "full_ges",
        "features_in_order": FULL_FEATURES,
        "training_rows": full_result["training_rows"],
        "positive_stable_labels": int(
            (
                full_result["training_label"] == 1
            ).sum()
        ),
        "negative_unstable_labels": int(
            (
                full_result["training_label"] == 0
            ).sum()
        ),
    },

    "no_star_model": {
        "model_name": "no_star_ges",
        "features_in_order": NO_STAR_FEATURES,
        "training_rows": no_star_result["training_rows"],
        "positive_stable_labels": int(
            (
                no_star_result["training_label"] == 1
            ).sum()
        ),
        "negative_unstable_labels": int(
            (
                no_star_result["training_label"] == 0
            ).sum()
        ),
        "review_status_used_as_predictor": False,
        "review_status_used_in_weak_label": False,
    },

    "preprocessing": {
        "missing_value_handling": (
            "Median imputation fitted only on eligible T0 training rows."
        ),
        "missingness_indicator": (
            "recency_missing_flag is retained as an explicit feature."
        ),
        "scaling": (
            "StandardScaler fitted only on eligible T0 training rows."
        ),
        "feature_order_frozen": True,
    },

    "logistic_regression": {
        **LOGISTIC_SETTINGS,
        "model_family": "binary logistic regression",
        "probability_output": (
            "Probability of weak-label stability, P(stable)."
        ),
    },

    "decision_threshold": {
        "value": DECISION_THRESHOLD,
        "purpose": (
            "Fixed reconstruction decision threshold only."
        ),
        "selected_using_t1": False,
        "temporal_evaluation_primary_use": (
            "Continuous P(stable), not the binary threshold."
        ),
        "additional_risk_tiers_selected": False,
    },

    "model_fitting_scope": {
        "uses_t0_information_only": True,
        "uses_t1_information": False,
        "uses_future_instability_outcome": False,
        "uses_temporal_performance_for_tuning": False,
        "hyperparameter_search_performed": False,
        "class_weighting_applied": False,
    },

    "software": {
        "python": sys.version,
        "platform": platform.platform(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scikit_learn": sklearn.__version__,
        "joblib": joblib.__version__,
    },

    "random_seed": RANDOM_SEED,
}


# --------------------------------------------------------------------------------------------------
# 14. QC report
# --------------------------------------------------------------------------------------------------

full_coefficient_summary = {
    row["feature_name"]: row["standardized_coefficient"]
    for row in full_result["coefficient_records"]
}

no_star_coefficient_summary = {
    row["feature_name"]: row["standardized_coefficient"]
    for row in no_star_result["coefficient_records"]
}

model_qc_report = {
    "report_name": "Stage 4C GES model QC report",
    "version": "1.0.0",
    "created_at_utc": created_at_utc,
    "validation_decision": "PASS",

    "cohort": {
        "total_t0_rows_scored": int(
            len(score_table)
        ),
        "unique_rcvs_scored": int(
            score_table["rcv_accession"].nunique()
        ),
    },

    "full_model": {
        "training_rows": full_result["training_rows"],
        "features": FULL_FEATURES,
        "n_iterations": full_result["n_iterations"],
        "converged": True,
        "standardized_intercept": (
            full_result["standardized_intercept"]
        ),
        "standardized_coefficients": (
            full_coefficient_summary
        ),
        "all_t0_score_distribution": (
            full_result["all_score_summary"]
        ),
        "weak_label_reconstruction_diagnostics": (
            full_result["diagnostics"]
        ),
    },

    "no_star_model": {
        "training_rows": no_star_result["training_rows"],
        "features": NO_STAR_FEATURES,
        "n_iterations": no_star_result["n_iterations"],
        "converged": True,
        "standardized_intercept": (
            no_star_result["standardized_intercept"]
        ),
        "standardized_coefficients": (
            no_star_coefficient_summary
        ),
        "all_t0_score_distribution": (
            no_star_result["all_score_summary"]
        ),
        "weak_label_reconstruction_diagnostics": (
            no_star_result["diagnostics"]
        ),
        "review_status_used": False,
    },

    "score_comparison": {
        "pearson_correlation_full_vs_no_star": float(
            np.corrcoef(
                full_probabilities,
                no_star_probabilities,
            )[0, 1]
        ),
        "mean_absolute_score_difference": float(
            np.mean(
                np.abs(
                    full_probabilities
                    - no_star_probabilities
                )
            )
        ),
    },

    "scientific_boundary": {
        "t1_information_used": False,
        "future_instability_outcome_created": False,
        "temporal_discrimination_evaluated": False,
        "temporal_calibration_evaluated": False,
        "model_hyperparameters_tuned": False,
        "full_model_fitted": True,
        "no_star_model_fitted": True,
    },

    "interpretation_warning": (
        "Weak-label reconstruction diagnostics measure agreement "
        "with the labels used for fitting. They are not estimates "
        "of future-instability prediction performance."
    ),
}


# --------------------------------------------------------------------------------------------------
# 15. Build local output artifacts
# --------------------------------------------------------------------------------------------------

BUILD_DIR = Path(
    "/content/stage4c_ges_model_build_v1"
)

if BUILD_DIR.exists():
    shutil.rmtree(BUILD_DIR)

BUILD_DIR.mkdir(
    parents=True,
    exist_ok=False,
)

temporary_full_model_path = (
    BUILD_DIR / FULL_MODEL_PATH.name
)

temporary_no_star_model_path = (
    BUILD_DIR / NO_STAR_MODEL_PATH.name
)

temporary_score_table_path = (
    BUILD_DIR / SCORE_TABLE_PATH.name
)

temporary_coefficient_table_path = (
    BUILD_DIR / COEFFICIENT_TABLE_PATH.name
)

temporary_specification_path = (
    BUILD_DIR / MODEL_SPECIFICATION_PATH.name
)

temporary_qc_path = (
    BUILD_DIR / MODEL_QC_PATH.name
)

temporary_manifest_path = (
    BUILD_DIR / STAGE4C_MANIFEST_PATH.name
)


# --------------------------------------------------------------------------------------------------
# 16. Save model artifacts
# --------------------------------------------------------------------------------------------------

full_model_artifact = {
    "artifact_name": "Full GES logistic-regression model",
    "artifact_version": "1.0.0",
    "created_at_utc": created_at_utc,
    "model_name": "full_ges",
    "pipeline": full_result["pipeline"],
    "feature_names": FULL_FEATURES,
    "label_column": "full_weak_label_binary",
    "eligibility_column": "full_training_eligible",
    "decision_threshold": DECISION_THRESHOLD,
    "training_rows": full_result["training_rows"],
    "imputation_values": full_result["imputation_values"],
    "scaler_means": full_result["scaler_means"],
    "scaler_scales": full_result["scaler_scales"],
    "standardized_intercept": (
        full_result["standardized_intercept"]
    ),
    "raw_space_intercept_after_imputation": (
        full_result[
            "raw_space_intercept_after_imputation"
        ]
    ),
    "random_seed": RANDOM_SEED,
    "uses_t1_information": False,
    "future_outcome_created": False,
}

no_star_model_artifact = {
    "artifact_name": "No-star GES logistic-regression model",
    "artifact_version": "1.0.0",
    "created_at_utc": created_at_utc,
    "model_name": "no_star_ges",
    "pipeline": no_star_result["pipeline"],
    "feature_names": NO_STAR_FEATURES,
    "label_column": "no_star_weak_label_binary",
    "eligibility_column": "no_star_training_eligible",
    "decision_threshold": DECISION_THRESHOLD,
    "training_rows": no_star_result["training_rows"],
    "imputation_values": no_star_result["imputation_values"],
    "scaler_means": no_star_result["scaler_means"],
    "scaler_scales": no_star_result["scaler_scales"],
    "standardized_intercept": (
        no_star_result["standardized_intercept"]
    ),
    "raw_space_intercept_after_imputation": (
        no_star_result[
            "raw_space_intercept_after_imputation"
        ]
    ),
    "random_seed": RANDOM_SEED,
    "review_status_used_as_predictor": False,
    "review_status_used_in_weak_label": False,
    "uses_t1_information": False,
    "future_outcome_created": False,
}

joblib.dump(
    full_model_artifact,
    temporary_full_model_path,
    compress=3,
)

joblib.dump(
    no_star_model_artifact,
    temporary_no_star_model_path,
    compress=3,
)


# --------------------------------------------------------------------------------------------------
# 17. Save score, coefficient, specification, and QC artifacts
# --------------------------------------------------------------------------------------------------

score_table.to_parquet(
    temporary_score_table_path,
    index=False,
    engine="pyarrow",
    compression="zstd",
)

coefficient_table.to_parquet(
    temporary_coefficient_table_path,
    index=False,
    engine="pyarrow",
    compression="zstd",
)

write_json(
    temporary_specification_path,
    model_specification,
)

write_json(
    temporary_qc_path,
    model_qc_report,
)


# --------------------------------------------------------------------------------------------------
# 18. Readback validation
# --------------------------------------------------------------------------------------------------

full_model_readback = joblib.load(
    temporary_full_model_path
)

no_star_model_readback = joblib.load(
    temporary_no_star_model_path
)

score_readback = pd.read_parquet(
    temporary_score_table_path
)

coefficient_readback = pd.read_parquet(
    temporary_coefficient_table_path
)

assert full_model_readback["model_name"] == "full_ges"
assert (
    no_star_model_readback["model_name"]
    == "no_star_ges"
)

assert full_model_readback["feature_names"] == FULL_FEATURES
assert (
    no_star_model_readback["feature_names"]
    == NO_STAR_FEATURES
)

assert score_readback.shape == score_table.shape
assert len(score_readback) == 71_659
assert score_readback["rcv_accession"].is_unique

assert (
    coefficient_readback.shape
    == coefficient_table.shape
)

# Re-score all rows from reloaded models.
full_readback_probabilities = (
    full_model_readback["pipeline"]
    .predict_proba(
        data[FULL_FEATURES].astype(float)
    )[:, 1]
)

no_star_readback_probabilities = (
    no_star_model_readback["pipeline"]
    .predict_proba(
        data[NO_STAR_FEATURES].astype(float)
    )[:, 1]
)

assert np.allclose(
    full_readback_probabilities,
    full_probabilities,
    atol=1e-12,
    rtol=1e-12,
)

assert np.allclose(
    no_star_readback_probabilities,
    no_star_probabilities,
    atol=1e-12,
    rtol=1e-12,
)


# --------------------------------------------------------------------------------------------------
# 19. Calculate output hashes
# --------------------------------------------------------------------------------------------------

full_model_hash = sha256_file(
    temporary_full_model_path
)

no_star_model_hash = sha256_file(
    temporary_no_star_model_path
)

score_table_hash = sha256_file(
    temporary_score_table_path
)

coefficient_table_hash = sha256_file(
    temporary_coefficient_table_path
)

model_specification_hash = sha256_file(
    temporary_specification_path
)

model_qc_hash = sha256_file(
    temporary_qc_path
)


# --------------------------------------------------------------------------------------------------
# 20. Freeze manifest
# --------------------------------------------------------------------------------------------------

freeze_manifest = {
    "manifest_name": "Stage 4C GES model freeze manifest",
    "version": "1.0.0",
    "created_at_utc": created_at_utc,

    "manifest_status": (
        "STAGE4C_FULL_AND_NO_STAR_GES_MODELS_FROZEN"
    ),

    "validation_decision": (
        "PASS_STAGE4C_GES_MODELS_ACCEPTED_AND_FROZEN"
    ),

    "source_artifacts": {
        path: {
            "sha256": observed_hash,
        }
        for path, observed_hash in sorted(
            verified_input_hashes.items()
        )
    },

    "frozen_outputs": {
        "full_ges_model": {
            "path": str(FULL_MODEL_PATH),
            "sha256": full_model_hash,
            "features": FULL_FEATURES,
            "training_rows": full_result["training_rows"],
        },

        "no_star_ges_model": {
            "path": str(NO_STAR_MODEL_PATH),
            "sha256": no_star_model_hash,
            "features": NO_STAR_FEATURES,
            "training_rows": no_star_result["training_rows"],
            "review_status_used": False,
        },

        "t0_ges_score_table": {
            "path": str(SCORE_TABLE_PATH),
            "sha256": score_table_hash,
            "rows": int(score_table.shape[0]),
            "columns": int(score_table.shape[1]),
        },

        "coefficient_table": {
            "path": str(COEFFICIENT_TABLE_PATH),
            "sha256": coefficient_table_hash,
            "rows": int(coefficient_table.shape[0]),
            "columns": int(coefficient_table.shape[1]),
        },

        "model_specification": {
            "path": str(MODEL_SPECIFICATION_PATH),
            "sha256": model_specification_hash,
        },

        "model_qc_report": {
            "path": str(MODEL_QC_PATH),
            "sha256": model_qc_hash,
        },
    },

    "frozen_model_settings": {
        "random_seed": RANDOM_SEED,
        "decision_threshold": DECISION_THRESHOLD,
        "logistic_regression": LOGISTIC_SETTINGS,
        "median_imputation": True,
        "standard_scaling": True,
        "hyperparameter_search_performed": False,
    },

    "model_accounting": {
        "total_t0_records_scored": 71_659,
        "full_model_training_rows": (
            full_result["training_rows"]
        ),
        "no_star_model_training_rows": (
            no_star_result["training_rows"]
        ),
        "full_model_converged": True,
        "no_star_model_converged": True,
    },

    "scientific_boundary": {
        "t1_information_used": False,
        "future_instability_outcome_created": False,
        "temporal_performance_examined": False,
        "full_ges_fitted_and_frozen": True,
        "no_star_ges_fitted_and_frozen": True,
        "t0_baseline_probabilities_generated": True,
        "rag_experiment_started": False,
    },

    "blueprint_stage_4_decision": "COMPLETE",

    "next_authorized_work": (
        "Blueprint Stage 5: construct the prespecified "
        "future-instability outcomes from T1 using only the "
        "frozen Stage 3H linkage and without modifying either "
        "frozen GES model."
    ),
}

write_json(
    temporary_manifest_path,
    freeze_manifest,
)

stage4c_manifest_hash = sha256_file(
    temporary_manifest_path
)


# --------------------------------------------------------------------------------------------------
# 21. Copy outputs to Drive
# --------------------------------------------------------------------------------------------------

copy_and_verify(
    temporary_full_model_path,
    FULL_MODEL_PATH,
)

copy_and_verify(
    temporary_no_star_model_path,
    NO_STAR_MODEL_PATH,
)

copy_and_verify(
    temporary_score_table_path,
    SCORE_TABLE_PATH,
)

copy_and_verify(
    temporary_coefficient_table_path,
    COEFFICIENT_TABLE_PATH,
)

copy_and_verify(
    temporary_specification_path,
    MODEL_SPECIFICATION_PATH,
)

copy_and_verify(
    temporary_qc_path,
    MODEL_QC_PATH,
)

# Freeze manifest copied last.
copy_and_verify(
    temporary_manifest_path,
    STAGE4C_MANIFEST_PATH,
)


# --------------------------------------------------------------------------------------------------
# 22. Persistent checksum verification
# --------------------------------------------------------------------------------------------------

assert sha256_file(FULL_MODEL_PATH) == full_model_hash
assert sha256_file(NO_STAR_MODEL_PATH) == no_star_model_hash
assert sha256_file(SCORE_TABLE_PATH) == score_table_hash
assert (
    sha256_file(COEFFICIENT_TABLE_PATH)
    == coefficient_table_hash
)
assert (
    sha256_file(MODEL_SPECIFICATION_PATH)
    == model_specification_hash
)
assert sha256_file(MODEL_QC_PATH) == model_qc_hash
assert (
    sha256_file(STAGE4C_MANIFEST_PATH)
    == stage4c_manifest_hash
)


# --------------------------------------------------------------------------------------------------
# 23. Persistent readback
# --------------------------------------------------------------------------------------------------

persistent_scores = pd.read_parquet(
    SCORE_TABLE_PATH
)

persistent_full_model = joblib.load(
    FULL_MODEL_PATH
)

persistent_no_star_model = joblib.load(
    NO_STAR_MODEL_PATH
)

assert len(persistent_scores) == 71_659
assert persistent_scores["rcv_accession"].is_unique

assert (
    persistent_full_model["feature_names"]
    == FULL_FEATURES
)

assert (
    persistent_no_star_model["feature_names"]
    == NO_STAR_FEATURES
)


# --------------------------------------------------------------------------------------------------
# 24. Final output
# --------------------------------------------------------------------------------------------------

print("=" * 120)
print("STAGE 4C — FULL GES AND NO-STAR GES MODEL FREEZE")
print("=" * 120)

print(f"Created at UTC:                    {created_at_utc}")
print(
    "Validation decision:              "
    "PASS_STAGE4C_GES_MODELS_ACCEPTED_AND_FROZEN"
)
print()

print("T0 SCORING COHORT")
print(f"  Total rows scored:               {len(persistent_scores):,}")
print(f"  Unique RCVs scored:              {persistent_scores['rcv_accession'].nunique():,}")
print()

print("FULL GES MODEL")
print(f"  Training rows:                   {full_result['training_rows']:,}")
print(
    "  Stable weak labels:              "
    f"{(full_result['training_label'] == 1).sum():,}"
)
print(
    "  Unstable weak labels:            "
    f"{(full_result['training_label'] == 0).sum():,}"
)
print(f"  Features:                        {FULL_FEATURES}")
print(f"  Solver iterations:               {full_result['n_iterations']:,}")
print("  Converged:                       True")
print(
    "  P(stable) range:                 "
    f"{full_probabilities.min():.6f} to "
    f"{full_probabilities.max():.6f}"
)
print(
    "  P(stable) mean:                  "
    f"{full_probabilities.mean():.6f}"
)
print(
    "  Weak-label reconstruction AUPRC: "
    f"{full_result['diagnostics']['auprc']:.6f}"
)
print()

print("NO-STAR GES MODEL")
print(f"  Training rows:                   {no_star_result['training_rows']:,}")
print(
    "  Stable weak labels:              "
    f"{(no_star_result['training_label'] == 1).sum():,}"
)
print(
    "  Unstable weak labels:            "
    f"{(no_star_result['training_label'] == 0).sum():,}"
)
print(f"  Features:                        {NO_STAR_FEATURES}")
print("  Review status used:              False")
print(f"  Solver iterations:               {no_star_result['n_iterations']:,}")
print("  Converged:                       True")
print(
    "  P(stable) range:                 "
    f"{no_star_probabilities.min():.6f} to "
    f"{no_star_probabilities.max():.6f}"
)
print(
    "  P(stable) mean:                  "
    f"{no_star_probabilities.mean():.6f}"
)
print(
    "  Weak-label reconstruction AUPRC: "
    f"{no_star_result['diagnostics']['auprc']:.6f}"
)
print()

print("MODEL COMPARISON")
print(
    "  Full/no-star score correlation:  "
    f"{np.corrcoef(full_probabilities, no_star_probabilities)[0, 1]:.6f}"
)
print(
    "  Mean absolute score difference:  "
    f"{np.mean(np.abs(full_probabilities - no_star_probabilities)):.6f}"
)
print()

print("SCIENTIFIC BOUNDARY")
print("  T1 information used:             False")
print("  Future outcome created:          False")
print("  Temporal performance evaluated:  False")
print("  Hyperparameter search performed: False")
print("  Full GES fitted and frozen:      True")
print("  No-star GES fitted and frozen:   True")
print()

print("FROZEN OUTPUTS")
print(f"  Full GES model:                  {FULL_MODEL_PATH}")
print(f"    SHA-256:                       {full_model_hash}")
print(f"  No-star GES model:               {NO_STAR_MODEL_PATH}")
print(f"    SHA-256:                       {no_star_model_hash}")
print(f"  T0 GES score table:              {SCORE_TABLE_PATH}")
print(f"    Rows / columns:                {score_table.shape[0]:,} / {score_table.shape[1]:,}")
print(f"    SHA-256:                       {score_table_hash}")
print(f"  Coefficient table:               {COEFFICIENT_TABLE_PATH}")
print(f"    SHA-256:                       {coefficient_table_hash}")
print(f"  Model specification:             {MODEL_SPECIFICATION_PATH}")
print(f"    SHA-256:                       {model_specification_hash}")
print(f"  Model QC report:                 {MODEL_QC_PATH}")
print(f"    SHA-256:                       {model_qc_hash}")
print(f"  Freeze manifest:                 {STAGE4C_MANIFEST_PATH}")
print(f"    SHA-256:                       {stage4c_manifest_hash}")
print()

print("BLUEPRINT STATUS")
print("  Stage 4 — reconstruct and freeze T0 GES: COMPLETE")
print()

print("NEXT AUTHORIZED WORK")
print("  Stage 5 — construct the prespecified future-instability")
print("  outcomes from T1 using the frozen Stage 3H linkage.")
print("=" * 120)

STAGE 4C — FULL GES AND NO-STAR GES MODEL FREEZE
Created at UTC:                    2026-07-20T22:15:55.211732+00:00
Validation decision:              PASS_STAGE4C_GES_MODELS_ACCEPTED_AND_FROZEN

T0 SCORING COHORT
  Total rows scored:               71,659
  Unique RCVs scored:              71,659

FULL GES MODEL
  Training rows:                   67,565
  Stable weak labels:              61,842
  Unstable weak labels:            5,723
  Features:                        ['recency_score', 'recency_missing_flag', 'submitter_diversity_score', 'review_confidence', 'aggregate_conflict_flag', 'scv_group_entropy_normalized']
  Solver iterations:               23
  Converged:                       True
  P(stable) range:                 0.000000 to 1.000000
  P(stable) mean:                  0.889359
  Weak-label reconstruction AUPRC: 1.000000

NO-STAR GES MODEL
  Training rows:                   63,148
  Stable weak labels:              61,298
  Unstable weak labels:            1,850
  Feature